   ## 4.1 Import libraries

In [ ]:
import numpy as np
import pandas as pd
import datetime
from datetime import timedelta
import os
from typing import Dict
import random
import gurobipy as gp

import logging
import oemof.solph as solph
from pyomo import environ as po
from pyomo.opt import TerminationCondition
import gzip
import pickle
import json
import re
from pathlib import Path
import glob
import gc
from collections import defaultdict
import hashlib

#Plot charts
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from tqdm import tqdm
from matplotlib import cm
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import math
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


# debug
from time import perf_counter
from contextlib import contextmanager
from collections import Counter


# Get the start time
start_time = datetime.datetime.now()
print(f"Start time : {start_time}")

   ## 4.2 User configuration

### 4.2.1 Main Actions

In [ ]:
# ------------------------------------------------------------
#   Sec 4.2 -- 1.1) MAIN NOTEBOOK ACTIONS
# ------------------------------------------------------------
# These are the only top-level flags that usually need to be modified.
# Each main section from 4.7 to 4.14 should depend on one of these flags.

RUN_BUILD_10MIN_PROFILES                = True         # 4.8: rebuilds 10-minute price/PV profiles
RUN_BUILD_RB_REFERENCE                  = True          # 4.10: builds the rule-based reference profiles used by mixed configurations
RUN_SOLVE_MIXED_SCENARIOS               = True          # 4.11: solves the mixed configurations (operation optimization problem)

RUN_PLOT_SCENARIO_GRAPHS                = True         # 4.12: detailed charts for a single configuration
RUN_COMPARE_MULTIRUN                    = True         # 4.13: multi-run comparison
RUN_COMPARE_DHW_SOC_TWO_CONFIGS         = True         # 4.14: plots the SOC chart for two configurations with ESWH (electric storage water heater)


# ------------------------------------------------------------
# Sec 4.2 -- 1.2) MAIN OUTPUT OPTIONS
# ------------------------------------------------------------

WRITE_RUN_MANIFEST_XLSX                 = True          # Save a list with all the tested configuration
WRITE_EXPERIMENT_INDEX_XLSX             = True          # With the related indexes
WRITE_EXPERIMENT_ECONOMIC_SUMMARY       = True          # Save a summary including economic info of the experiment set of tested configurations

### 4.2.2 Study design

In [ ]:
# ------------------------------------------------------------
#   Sec 4.2 -- 2.1) PHYSICAL/ELECTRIFICATION EXPERIMENT CHOICE
# ------------------------------------------------------------

# Choose one of the possible configuration included in the list "ELECTRIFICATION_EXPERIMENTS" (Section 4.5.4)
ACTIVE_ELECTRIFICATION_EXPERIMENT = "FULL_ELECTRIFICATION"      # e.g., "FULL_ELECTRIFICATION" | "HP_SWEEP" | "PV_SWEEP" | ... | "REF_NO_ELECTRIFICATION"
ACTIVE_ELECTRIFICATION_SCENARIO_IDS_OVERRIDE = None             # None = use every scenario belonging to the selected experiment. Otherwise select only specific scenario IDs (i.e. example below)

# Example: (instead all shares from 10%-20%.... 90%-100% --> just 20% and 50%)
#
# ACTIVE_ELECTRIFICATION_SCENARIO_IDS_OVERRIDE = [
#     "HP020",
#     "HP050",
# ]


# ------------------------------------------------------------
# Sec 4.2 -- 2.2) CHOICE OF WHICH FLEXIBILITY/CONTROL STRATEGY
# Choose one of the possible configuration included in the list "CONTROL_GROUPS" (Section 4.5.4)
ACTIVE_CONTROL_GROUP = "ALL_CASES"                              # e.g., "ALL_CASES" | "BASE_ONLY" | "DHW_SMART_ONLY" | "EV_SMART_ONLY" | ... | "BASE_VS_DHW_SMART"   |   "CASES_2_TO_7"   | "CASES_5_TO_7"  |  "BESS_ONLY"  |  "BESS_EV_ONLY"


# Run revision
RUN_REVISION = "r12_ESWH_89-100_min_10"                         # Synthetic name/description of the main changes/characteristics of the run 



# ------------------------------------------------------------
# Optional RB-cache revision override
# ------------------------------------------------------------
# None:
#     use the RB cache associated with RUN_REVISION.
#
# Set explicitly only when a new revision should reuse RB
# profiles generated under an older revision.
#
# Example:
# RB_CACHE_RUN_REVISION_OVERRIDE = "r09_EV_80_longtrip"
#
RB_CACHE_RUN_REVISION_OVERRIDE = None


### 4.2.3 Network / population

In [ ]:
# ---------------------------------------
#       Sec 4.2 -- 3) LV GRID ARCHETYPE SELECTION
# ---------------------------------------

# Grid type (see the commented table below to choose the grid archetype to select), capacity of the transformer, number of households/end-users
LV_ARCHETYPE                                = "kerber_urban_grid_B"		# "kerber_rural_grid_A"  |  "kerber_urban_grid_B"
secondary_substation_apparent_power_kva     = 630               		# 125 | 400 | 630 [kVA] Apparent power (typical values for residential districts are around 250-400-630-1000 kVA for secondary substations) 
num_households                              = 146               		# number or households/end-users                     

# General parameters (geographical area)
selected_region                             = 'Cen'             		# Choose among (N-O, N-E, Cen, Sud, Isl)


"""
Useful characteristics of the available Kerber network archetypes:

--------------------------------------------------------------------------------------------------------------------------------------------------------------------

		            Lines	Total Length	Loads/end-users	Installed Power (= no. loads * single power)    Transmission power [kVA]    Name

RURAL GRID ---------------------------------------------------------------------------------------------------------------------------------------------------------
Kerber Landnetze				
Freileitung 1	    13	    0.273 km	    13	            0.104 MW    (8 kW each user)                    0.125 MVA                   # kerber_rural_grid_D
Freileitung 2	    8   	0.390 km	    8	            0.064 MW     (8 kW each user)                   0.125 MVA                   # kerber_rural_grid_B
Kabel 1	            16	    1.046 km	    8	            0.064 MW     (8 kW each user)                   0.125 MVA                   # kerber_rural_grid_C
Kabel 2	            28	    1.343 km	    14	            0.112 MW    (8 kW each user)                    0.125 MVA                   # kerber_rural_grid_A
--------------------------------------------------------------------------------------------------------------------------------------------------------------------
	
MIXED GRID ---------------------------------------------------------------------------------------------------------------------------------------------------------			
Kerber Dorfnetz	    114	    3.412 km	    57	            0.342 MW    (6 kW each user)                    0.4 MVA                     # kerber_mixed_grid
--------------------------------------------------------------------------------------------------------------------------------------------------------------------

URBAN GRID ---------------------------------------------------------------------------------------------------------------------------------------------------------				
Kerber Vorstadtnetze				
Kabel 1	            292	    4.476 km	    146	            0.292 MW    (2 kW each user)                    0.63 MVA                    # kerber_urban_grid_B
Kabel 2	            288	    4.689 km	    144	            0.288 MW    (2 kW each user)                    0.63 MVA                    # kerber_urban_grid_A
--------------------------------------------------------------------------------------------------------------------------------------------------------------------

info can be found at: https://pandapower.readthedocs.io/en/latest/networks/kerber.html
"""
#----------------------------------------------------------------------------------------


### 4.2.4 Default / maximum technology pool

In [ ]:
# ------------------------------------------------------------
#   Sec 4.2 -- 4) DEFAULT / MAXIMUM TECHNOLOGY POOL
# ------------------------------------------------------------
# These shares do NOT necessarily describe the active run.
#
# They define:
#   1) fallback shares for technologies not explicitly changed
#      by an electrification scenario;
#   2) the maximum technology-assignment pool from which
#      run-specific subsets are extracted.
#
# The actual physical shares of each run are stored in RUN_SHARES
# and may be lower, including zero.

# These values define the default physical scenario and the
# maximum household pool from which scenario-specific penetration
# subsets are constructed.
#
# Electrification experiments may override one or more shares.
#
# V2G_share is conditional on BEV ownership:
#     V2G_share = % of BEV households enabling V2G.
#
# Other shares are relative to all households.


# Technology penetration
PV_share = 100                                  # households with PV
induction_share = 100                           # households with induction cooktops
heat_pump_share = 100                           # households with heat pumps
cooler_share = 100                              # households with cooling systems

# Coupling assumptions
standalone_ratio = 0.01                         # At most 1% of all BESS units may be standalone, i.e. installed without a coupled PV system.


# Flexibility options
ESWH_share = 100                                # share of households with Electric storage water heater
BESS_share = 100                                # share of households with battery (BESS) 
EV_share = 100                                  # share of households with electric vehicles (EV)
V2G_share = 100                                 # share of households with EV that also allow vehicle-to-grid (V2G) option



# ------------------------------------------------------------
#   Sec 4.2 -- 5) Main technology sizes
# ------------------------------------------------------------
# power ratings and capacities PV, BESS, technology diffusion
PV_power_kwp = 3                                # [kWp], installed PV capacity for each dwelling

BESS_power_kw = 3                               # [kWp], 
BESS_capacity_kwh = 12                          # [kWh],


## 4.3 Technical and advanced input parameters

### 4.3.1 Technology operation parameters

In [ ]:
# Time discretization
timestep_minutes                = 10
number_timesteps                = 1008

# ------------------------------
# BESS
# ------------------------------
eta_BESS                        = 0.95
initial_SoC_BESS                = 0.0                                  # state of charge at the beginning of the simulated period
BESS_loss_rate                  = 0.0                                  # loss at each timestep [%]


# ------------------------------
# BEV / charger / V2G
# ------------------------------
EV_charge_discharge_efficiency  = 0.90
eta_charger                     = 0.99
eta_V2G                         = 0.99
initial_SoC_BEV                 = 0.8                                   # state of charge at the beginning of the simulated period
BEV_loss_rate                   = 0.0                                   # loss at each timestep [%]
min_storage_level_BEV           = 0.05                                  # [%] *100 minimum state of charge to guarantee to enable EV charging outside the home
home_charging_power_kw          = 3                                     # kW with which the EV can charge and feed energy back to the grid (via charger and V2G)
default_ev_battery_capacity_kwh = 50                                    # fallback value for standard EV battery capacity (55 kWh) if individual EV values are not read
SOC_BEV_TARGET_frac             = 0.8                                   # minimum threshold (80%) to reach before stopping EV charging

P_outdoor_charger               = 22                                    # kW for charging at public outdoor chargers (supermarket - parkings) 
SOC_BEV_OUTDOOR_TRIGGER_frac    = 0.20                                  # outdoor alarm threshold - trigger outdoor backup charging - (if outdoor and SOC below treshhold, look for an outdoor charger when available)
SOC_BEV_OUTDOOR_TARGET_frac     = 0.50                                  # outdoor charging target (if outdoor and treshhold activated, SOC cap to be reached through outdoor charger)


ENABLE_BEV_PLANNED_FORESIGHT_RB = True                                  # RB limited foresight for scheduled long trips
BEV_PLANNED_TRIP_LOOKAHEAD_H    = 12.0                                  # n° of hours to check for planned long trips with EV (i.e., it's highly probable to know in advance if long trips are planned for that day; if so, charging EV will be adjusted)


BEV_LONG_TRIP_CONSUMPTION_frac  = 0.30                                  # Expected battery consumption defining a "long" trip. 30% captures the rare, relatively demanding mobility blocks without reacting to ordinary short trips.
BEV_VERY_LONG_TRIP_CONSUMPTION_frac = (                                 # Value above which a trip is considered very long and so EV charge is adapted
    0.70 - min_storage_level_BEV
)

# Simple behavioural targets
SOC_BEV_LONG_TRIP_TARGET_frac       = 0.70                              # desiderable SOC target to be reached in preparation to "long trips"
SOC_BEV_VERY_LONG_TRIP_TARGET_frac  = 1.00                              # desiderable SOC target to be reached in preparation to "very long trips"




# ------------------------------
# DHW / electric storage water heater (ESWH)
# ------------------------------
initial_SoC_DHW                 = 0.90                                 # state of charge at the beginning of the period (Assumed, close to p10 of average SOC at midnight in conventional setup)
eta_DHW                         = 0.98                                 # nearly all electricity is converted into heat (95%-99%)
DHW_loss_rate                   = 0.005                                # [%_loss/hour] relative HOURLY thermal loss of the ESWH (0.5%/h) --> 0.5 kWh every 24h ≈0.02 kWh/h (0.5% every hour --> almost 0.1% every 10 minutes) --> loss rate in GenericStorage must be in [%/h] then automatically converted to %/timestep
dhw_power_kw                    = 1.5                                  # [kW], electrical power for an 80L ESWH (heating elements approx. 1.5-2 kW) (for comparison, apartment ESWH = 75L, 1.2 kW power)
cap_kWh_DHW                     = 4.2                                  # [kWh], for a 100-liter ESWH (2.4 kWh if 50L ESWH, 4.2 kWh for 100L ESWH). Sized for 2-3 people
P_boost_DHW                     = 10                                   # 10 [kW] power of an additional auxiliary "boost" heating element to meet DHW demand when the ESWH alone is insufficient
# standing_loss_default         = 1.35                                 # [kWh/24h] standing loss for an average 100-liter ESWH
SOC_HYST_DEFAULT                = (0.89, 1)                         # (lower % threshold to start, upper % threshold to stop)
min_dwell_steps_home = 1                                               # no. consecutive timesteps at home required to "enable EV charging" (i.e., if home break is only 50', do not charge the EV)
# SOC_MIN_DEP = 0.50                                                   # Enforces the minimum state of charge (SOC) before the EV departs from home (e.g., 50% before leaving)
min_storage_level_ESWH           = 0.1                                 # [%] *100 minimum state of charge to avoid complete depletion of the ESWH in smart management


# DHW/ESWH Dictionaries: household typology → (n_people, ESWH capacity [kWh_th], power [kW], loss rate [%/h])
# >>> WARNING dictionary also defined in Module III, need to be consistent <<<<<<<
dhw_specs = {
    'single':                   (1,  4.2,     1.5,    0.013),             # 1.35),     # ~80 L, 4.2 kWh, 1.5 kW, 1.3%/h static thermal losses,   1.35 kWh/24h (standing loss @T_hot=65°C and T_amb=20°C),
    'couple_wo_children':       (2,  5.2,     2,      0.013),             # 1.56),     # ~100 L, 5.2 kWh, 2 kW, 1.3%/h,                   1.56 kWh/24h,
    'couple_w_children_3':      (3,  10.5,    3,      0.007),             # 1.88),     # ~200 L, 10.5 kWh, 3 kW, 0.7%/h,                   1.88 kWh/24h
    'couple_w_children_4':      (4,  15.7,    4,      0.007),             # 2.6),      # ~300 L, 15-7 kWh, 4 kW, 0.7%/h,                 2.6 kWh/24h
    'couple_w_children_5':      (5,  15.7,    4,      0.007),             # 2.6),      # ~300 L, 15.7 kWh, 4 kW, 0.7%/h,                 2.6 kWh/24h
    'couple_w_children_6':      (6,  20.9,    6,      0.006),             # 3.6),      # 400 L, 20.9 kWh, 4 kW, 0.6%/h,                  3.6 kWh/24h,   The same size is used for large and very large households.
    'single_parent_M_2':        (3,  5.2,     2,      0.013),             # 1.56),     # 2-member single-parent household -> use the size assumed for 3-person households
    'single_parent_F_2':        (3,  5.2,     2,      0.013),             # 1.56),
}

# ------------
# DHW initial-SOC population initialization
DHW_INITIAL_SOC_DISTRIBUTED     = True
DHW_INITIAL_SOC_SEED            = 48371

# Mean initial SOC by household type.
DHW_INITIAL_SOC_MEAN_BY_TYPE = {
    "single":                  0.94,
    "couple_wo_children":      0.93,
    "couple_w_children_3":     0.97,
    "couple_w_children_4":     0.97,
    "couple_w_children_5":     0.97,
    "couple_w_children_6":     0.96,
    "single_parent_M_2":       0.95,
    "single_parent_F_2":       0.95,
}

# Half-width of the within-type initial-SOC distribution.
# Ideally estimated from the dispersion of internal-midnight SOC.

DHW_INITIAL_SOC_HALF_RANGE_BY_TYPE = {
    "single":                  0.03,
    "couple_wo_children":      0.03,
    "couple_w_children_3":     0.02,
    "couple_w_children_4":     0.015,
    "couple_w_children_5":     0.015,
    "couple_w_children_6":     0.02,
    "single_parent_M_2":       0.03,
    "single_parent_F_2":       0.03,
}

### 4.3.2 Economic and time-series assumptions

In [ ]:
#-----------------------------------------------
# Sec 4.3 -- 2) Parameters for creating TIMESERIES profiles (with 10' resolution) 
#                   and ECONOMIC parameters
#-----------------------------------------------
hourly_prices_file          = 'GME_2023_electricity_prices_hourly'

# To import hourly timeseries for PV generation with 3 different PV panel orientations 
hourly_pv_file              = 'PV_2023_historical_hourly_profile_example_Rome'          # WARNING: changing the selected geographical area and the observed period (i.e., the 4 selected weeks) requires to adjust also input timeseries for prices/PV generation/demand profiles
treat_as_energy             = True                                                      # False if hourly timeseries values are expressed in kW (power); True if energy kWh/ora
# Flag: do the files contain power values (kW), or already energy per timestep (kWh)?
data_is_power               = True                                                      # TRUE if profiles are power [kW]

# Assumptions on electricity selling price (injection into LV grid excesses) and on electricity purchase price (withdrawal from the grid)
additional_fees             = 0.045                                                     # [€/kWh_el] additional fees to be charged to residential end-users to cover also distribution and metering charges, in addition to the national single price (i.e., the Italian PUN); This value is obtained from historical values derived from Electricity Market Operator (i.e., the Italian GME), based on the most recent available years (2025)
feed_in_revenue_ratio_to_zonal_price = 0.50                                             # Export remuneration is assumed equal to 50% of the hourly zonal electricity price. The positive economic remuneration is converted into a negative objective coefficient before being assigned to the household-to-LV export flow.


# Economic parameters for shortage/excess components (to limit potential undesired utilization by the model solver)
PV_curtail_cost             = 0                                                         # [€/kWh] cost for curtail the excess PV generation that cannot be injected into the LV grid (if already saturated due to line capacity constraints) -- Assumed to be cost-free (curtailment is not a cost)
cost_outdoor_charger        = 0.5                                                       # [€/kWh] Representative pre-VAT proxy for public AC charging. Italian consumer tariffs considered are approximately 0.53-0.69 EUR/kWh incl. VAT (~0.43-0.57 EUR/kWh excl. 22% VAT). A central value of 0.50 EUR/kWh is adopted. [E.g., source: Enel X Way public AC charging tariff: https://www.enelxway.com/content/dam/enelxmobility/italia/documenti/tariffe-abbonamenti/piani-di-ricarica-enel-x-way.pdf]
cost_boost_DHW              = 10                                                       # [€/kWh] cost for additional charging power inside electric water heater (through an external resistance)



### 4.3.3 Grid and PF assumptions

In [ ]:
#-----------------------------------------------
# Sec 4.3 -- 3.1) Parameters for LV GRID ARCHETYPE
#-----------------------------------------------

# Parameters to represent line capacity constraints in oemof.solph framework  as a bidirectional Converters 
KERBER_LINE_ETA = 0.999                                 # efficiency of the line (proxy for grid losses)
KERBER_LINE_COST = 1e-6                                 # [€/kWh] cost to penalize loops in meshed grids
KERBER_DEFAULT_PMAX_KW = 250.0                          # fallback if in the blueprint for Kerber archetypes Pmax is missing for some lines 

MAX_LV_BUSES_TO_EXPORT = 80                             # if grid Kerber has >N bus, exported results limited to root+bus with POD (the line connected to the transformer + lines connected to end-users)


# Grid techno-economic parameters
eta_grid = 1.0
secondary_substation_power_factor = 0.95                # i.e., cos(Phi); ARERA regulations require it to be at least greater than or equal to 0.95 [AEEG Resolution 180/2013/R/EEL: Cosφ at 0.95 as of January 1, 2016]

# Grid topology
lv_efficiency   = 1.0                                   # efficiency on LV exchange
eta_line = 1.0                                          # or 1.0 or a custom value for line losses


# ------------------------------------------------------------
# OPTIONAL POWER-FLOW FEEDBACK (UNDER DEVELOPMENT)
# ------------------------------------------------------------
# This functionality is intended for a subsequent optimization
# run using network limits derived from a separate Power-Flow
# analysis (Module 05, not included in this release).
#
# False: use the standard simplified LV line-capacity constraints.
# True: attempt to load externally generated PF-calibrated limits.
#
# Keep False for the current public release unless compatible
# PF-calibrated input files have been explicitly provided.

USE_PF_LINE_LIMITS_V1 = False

PF_LIMITS_PF_MODE = "balanced"                          # it has to coincide with PF_ANALYSIS_MODE parameter used in the Power-Flow module (under development)
PF_LIMITS_SHEET = "pf_line_limits_v1"
PF_INTERFACE_SHEET = "pf_interface_limits_v1"

KERBER_LV_LABEL_TABLE_MAX_ROWS = 120                    # Limits the n° of rows to be printed



# Optional override of transmission line capacity limits with PF-calibrated outputs (currently UNDER DEVELOPMENT)
""" 
This section defines any new transmission power limits on the grid 
to address the critical issues identified in the Power Flow analysis (e.g., voltage 
drops or overvoltages, line loading/saturation exceeding thermal limits, etc.).

These new values are then used in a new run of the operational optimization problem, 
which will incorporate the new PF-derived constraints to output a new configuration 
closer to the physical limits of the grid.

To do this, it should be enabled such option, changing the variable:

        USE_PF_LINE_LIMITS_V1 = True

This would allow to run a new operation optimization problem while incorporating such new
PF-derived constraints.

This optional workflow is intended to support a subsequent optimization run using network constraints 
derived from a separate power-flow analysis. The integration is currently under development and is not 
part of the validated Module 04 workflow distributed in this release.
"""


# ------------------------------------------------------------
# Sec 4.3 -- 3.2) Household export assumptions for Power-Flow module (UNDER DEVELOPMENT)
# ------------------------------------------------------------

EXPORT_PF_COMPONENT_PROFILES            = True          # Export the active-power breakdown at each household bus.
EXPORT_ASSUMED_Q_FOR_PF                 = True          # Also export an assumed reactive-power profile that module for power-flow (i.e., Module 05, under development) can load directly as pf_q_mvar.

VALIDATE_PF_EXPORT_RECONSTRUCTION       = True          # Validate the reconstruction of household active-power flows.
PF_EXPORT_VALIDATION_STRICT             = True          # Raise an exception instead of printing only a warning.
SAVE_PF_EXPORT_VALIDATION_DETAILS       = True          # Save detailed residual profiles and a validation summary.
PF_EXPORT_BALANCE_TOLERANCE_KW          = 1e-6          # Numerical tolerance in the original MODULE_04 power unit [kW].

# Assumed power factor by household-absorption category.
#
# These values are modelling assumptions. They do not affect
# the MODULE_04 active-power optimization; they are used only
# to build a first Q profile for MODULE_05.
PF_EXPORT_COSPHI_BY_COMPONENT = {
    "base": 0.95,
    "cooking": 0.95,
    "cooling": 0.95,
    "heating": 0.95,

    # Initial simplifying assumptions:
    # electric resistance and inverter-based charging at unity PF.
    "dhw_charge": 1.00,
    "bev_home_charge": 1.00,
    "bess_charge": 1.00,
}

# PV, BESS discharge, and V2G are initially assumed to operate
# at unity power factor, therefore they exchange no Q.
PF_EXPORT_GENERATION_Q_ASSUMPTION = (
    "unity_power_factor"
)

# ------------------------------------------------------------------





### 4.3.4 Secondary / export / debug / legacy flags

In [ ]:
# ---------------------------
# Sec 4.3 -- 4.1) Parameters for topology analysis and LV-grid information
# ---------------------------
PRINT_KERBER_LV_LABEL_TABLE                 = True              # If True, plot grid mapping tables (Kerber only) for readability          
plt_load_duration_LV_grid_lines             = False             # If True, plot duration curves for NET flows along grid sections


# ---------------------------
# Sec 4.3 -- 4.2) Parameters for ADDITIONAL COMPONENTS "shortage-based" for critical scenarios/cases
# ---------------------------
# Additional options to make RB and OPT comparable
ENABLE_DHW_BOOST_RB                         = True               # if True, RB allows an additional DHW boost when the boiler alone is insufficient
ENABLE_BEV_OUTDOOR_RB                       = True               # if True, RB accounts for outdoor EV charging when SoC is insufficient
ENABLE_PV_CURTAILMENT                       = True               # if True, include a sink for PV excess that cannot be exported to the grid


# ---------------------------
# Sec 4.3 -- 4.3) Parameters for the level of detail of COST INFORMATION TO SAVE
# ---------------------------

# -----------
# Sec 4.3 -- 4.3.1) Economic reporting options

# DHW boost is always penalized inside the optimization objective through
# cost_boost_DHW. This flag affects ONLY ex-post economic reporting.
#
# False:
#   the artificial DHW-boost penalty is excluded from household/system
#   effective costs, while it remains active in the optimization objective.
#
# True:
#   the DHW-boost penalty is also counted as an effective economic cost.
#
INCLUDE_DHW_BOOST_IN_EFFECTIVE_COSTS = False
# -----------


# Export detailed objective-cost sheets
SAVE_OBJECTIVE_TERMS_BY_FLOW_XLSX           = False
SAVE_OBJECTIVE_TERMS_BY_HOUSE_XLSX          = True

# If True, save a clean version in the ris_OPT / ris_RB file
# of objective_terms_ts / summary, without hard-to-read technical entries
FILTER_OBJECTIVE_TERMS_TS_FOR_EXPORT        = True
FILTER_OBJECTIVE_TERMS_SUMMARY_FOR_EXPORT   = True


#-----------------------------------------------
# Sec 4.3 -- 4.4) Parameters for DEBUG INFEASIBLE
#-----------------------------------------------
DEBUG_INFEASIBLE_ON_SOLVE                   = True               # if True, save Linear Programming LP file when non-optimal
COMPUTE_IIS_WITH_GUROBI                     = True               # if True, attempt to compute IIS via gurobipy by reading the .lp file
SAVE_IIS_JSON                               = True               # save a "clean" JSON with IIS constraint names and bounds


# IMPORTANT:
# A non-optimal configuration/season does NOT stop the whole experiment.
# Its post-processing is skipped and the solver continues with the next season/run.
STOP_ON_INFEASIBLE                          = False               # if True, abort loops after saving (without raising an exception)

# Ex-post detailed analysis of one failed solve
RUN_DEBUG_INFEASIBLE                        = False              # if True, display unsatisfied conditions/constraints for ex-post analysis  
PRINT_LP_PREVIEW                            = True               # if True, provides a summary of the infeasible conditions saved in the run


# Optional selectors for Sec. 4.15.
# If both are None, the last failed solve is selected automatically.
DEBUG_SELECT_RUN_UID                        = None               # e.g. None   or specific ID | (example) "R_12345678"
DEBUG_SELECT_SEASON                         = None               # e.g., "winter"; if None, uses LAST_INFEASIBLE_SEASON


# Manual paths always have priority, if specified
DEBUG_LP_PATH                               = None               # manual override (path to the debug_infeasible_*.lp file)
DEBUG_IIS_JSON_PATH                         = None               #  manual override (path to the iis_*.json file)

LP_PREVIEW_HEAD                             = 80
LP_PREVIEW_TAIL                             = 60
MAX_LIST_NAMES                              = 2000


# Parameters to study outdoor/domestic charging profiles (defined in CAN)
CAN_SCAN_PRINT_LIMIT                        = 40                 # max number of rows to print/plot


# Parameter to check if profiles for conventional case are consistent
RUN_RB_PROFILE_DIAGNOSTICS = True
#----------------------------------------------------------------------------------




# ------------------------------------------------------------
#   Sec 4.3 -- 4.5) old LEGACY SECTIONS
# ------------------------------------------------------------
# Kept as an archive, but outside the main workflow.
# If False, section 4.1X must not run.
RUN_ENABLE_LEGACY_SECTIONS                  = False


# These flags should no longer drive the new logic.
# They remain only for backward compatibility with very old functions.
RUN_RULE_BASED                              = False         # rule-based-only configuration case
ALLOW_LEGACY_HAS_FROM_OPT_PICKLE            = False         # If True, allow reading old configurations without rerunning the optimization problem (warning, risk of naming inconsistency)






# ------------------------------------------------------------
#   Sec 4.3 -- 4.6) DEBUG / LOG
# ------------------------------------------------------------

DEBUG_RB = False
DEBUG_CAN_SCAN = False

### 4.3.5 Plotting / comparison parameters

In [ ]:
# ------------------------------------------------------------
#   Sec 4.3 -- 5.1) Chart parameters and dictionaries for JOURNAL / TWO-COLUMNS FIGURES
# ------------------------------------------------------------
SHOW_MULTIRUN_FIGURE_TITLE = False
SHOW_SOC_COMPARE_TITLE = False

SAVE_JOURNAL_PDF = True

# -----------------------
# Journal / two-column chart parameters
# -----------------------
JOURNAL_FONT_SIZE = 9.0
JOURNAL_AXIS_LABEL_SIZE = 10.0
JOURNAL_TICK_LABEL_SIZE = 8.5
JOURNAL_LEGEND_FONT_SIZE = 8.0
JOURNAL_LEGEND_TITLE_SIZE = 8.5
JOURNAL_TITLE_SIZE = 10.5


# -----------------------
# Table font
# -----------------------
# Note: the table has 8-9 columns, so using the same font size as the axes
# would make it inevitably too compressed.
# Use a slightly smaller size while keeping the visual style consistent.
JOURNAL_TABLE_FONT_SIZE = 6.9
JOURNAL_TABLE_SYMBOL_FONT_SIZE = 7.8
JOURNAL_TABLE_HEADER_FONT_SIZE = 6.9
JOURNAL_TABLE_NOTE_FONT_SIZE = 6.9

# -----------------------
# Two-column figure sizes
# -----------------------
# Cost percentile: legend inside, so almost the full width can be used.
# Demand coverage: legend on the right, so the figure is slightly wider.
JOURNAL_TWO_COLUMN_FIGSIZE_COST = (7.20, 4.95)
JOURNAL_TWO_COLUMN_FIGSIZE_DEMAND = (7.80, 5.00)
JOURNAL_TWO_COLUMN_FIGSIZE_SOC = (7.20, 3.95)

# Legends differentiated by chart type
MULTIRUN_COST_LEGEND_MODE = "upper_right_inside"
MULTIRUN_DEMAND_LEGEND_MODE = "right_outside"

# Remove the symbol note from the figure; it is better placed in the caption.
MULTIRUN_TABLE_SHOW_SYMBOL_NOTE = False

# Fixed palette consistent with the stacked consumption charts.
STACKED_CONSUMPTION_COLOR_MAP = {
    "Conventional":              "#B3B3B3",
    "DHW":                       "#0072B2",
    "Cooking":                   "#E69F00",
    "Cooling":                   "#009E73",
    "EV":                        "#D55E00",
    "Heating":                   "#7E57C2",
}
# ------------------------------------------------------------




# ------------------------------------------------------------
#   Sec 4.3 -- 5.2) PLOT OPTIONS FOR SINGLE-CONFIGURATION CHARTS (derived from the manifest), Sec. 4.12
# ------------------------------------------------------------

# ------------------------------------------------------------
# Optional source override for single-run plots.
#
# None = use the experiment/revision currently selected in 4.2.
# Set explicitly only when plotting previously saved runs.
# ------------------------------------------------------------
PLOT_EXPERIMENT_ID_OVERRIDE = None
PLOT_RUN_REVISION_OVERRIDE = None




# Most precise selection.
# None = no filtering by UID.
PLOT_ONLY_RUN_UIDS = None


# Human-readable selection.
#
# Example:
# PLOT_ONLY_RUN_LABELS = [
#     "HP020__BASE",
#     "HP100__BASE",
# ]
PLOT_ONLY_RUN_LABELS = None


# Additional metadata filters.
#
# Examples:
#
# {"case_label": ["Base case"]}
#
# {"share_HP_target": [20, 40, 60, 80, 100]}
#
# Multiple filters are AND-combined.
PLOT_MANIFEST_FILTERS = {
    "case_label": [
        "Base case",
    ],
}


PLOT_ONLY_SEASONS = None

PLOT_ONLY_SOC_TECHS = [
    "BESS",
    "DHW",
    "BEV",
]
# ------------------------------------------------------------




# ------------------------------------------------------------
# Sec 4.3 -- 5.3) Parameters for ESWH (electric storage water heater) SOC (state of charge) comparison charts between two configurations
#               (check the manifest to select which configuration to print)
# ------------------------------------------------------------

# ------------------------------------------------------------
# Optional physical-scenario selector for SOC comparison.
#
# None:
#   - if the active experiment contains exactly one active
#     electrification scenario, use it automatically;
#   - otherwise stop and require an explicit selection.
#
# Example for a sweep:
# SOC_COMPARE_ELECTRIFICATION_ID_OVERRIDE = "HP020"
# ------------------------------------------------------------
SOC_COMPARE_ELECTRIFICATION_ID_OVERRIDE     = None

SOC_COMPARE_EXPERIMENT_ID_OVERRIDE          = None
SOC_COMPARE_RUN_REVISION_OVERRIDE           = None


# Example:
# same physical scenario, different ESWH control.

SOC_COMPARE_SELECTOR_A = {
    "case_label": "Base case",
}

SOC_COMPARE_SELECTOR_B = {
    "case_label": "Case 1",
}
# -----

SOC_COMPARE_SEASONS                 = None                          # None = all seasons; otherwise ["spring"]
SOC_COMPARE_PV_SOURCE               = "total_installed"             # "total_installed" = sum of PV capacity of households with PV in the configuration; "single_house" = use one household only
SOC_COMPARE_SINGLE_PV_HOUSE         = None                          # None = all households; otherwise "c_1" .. "c_100"


SOC_COMPARE_COLOR_MODE              = "case_contrast"               # use "tech_control" to apply the technology/control logic (DHW OPT red, DHW RB orange, etc.)
SOC_COMPARE_USE_HATCHES             = False                         # If True, adds hatching to the bands of case B (useful when bands overlap)

# Median lines: restored to a lighter linewidth suitable for a journal figure.
SOC_COMPARE_MEDIAN_LINEWIDTH        = 1.5                           # median value line
SOC_COMPARE_MIN_LINEWIDTH           = 1.0                           # minimum value line


# Secondary profile line, e.g., price or PV.
SOC_COMPARE_SECONDARY_LINEWIDTH     = 1.2

# More compact SOC legend.
SOC_COMPARE_LEGEND_FONT_SIZE        = 6.8
SOC_COMPARE_LEGEND_TITLE_SIZE       = 7.2
SOC_COMPARE_LEGEND_TITLE            = None
SOC_COMPARE_LEGEND_NCOL             = 2
SOC_COMPARE_LEGEND_LOC              = "lower left"

# Short labels in the legend.
# Base case -> Base
# Case 1 -> C1
# Case 4 -> C4, etc.
SOC_COMPARE_SHORT_CASE_LABELS       = True

# Short labels for the legend and secondary axis.
SOC_COMPARE_PRICE_LEGEND_LABEL      = "El. price"
SOC_COMPARE_PRICE_AXIS_LABEL        = "El. price [EUR/kWh]"


SOC_COMPARE_PV_LEGEND_LABEL         = "PV gen."
SOC_COMPARE_PV_AXIS_LABEL           = "PV generation [kW]"

# Secondary price/PV line marker.
SOC_COMPARE_SECONDARY_MARKERSIZE    = 1.6


# ------------------------------------------------------------
# Optional source override for multi-run comparisons.
# ------------------------------------------------------------

MULTIRUN_EXPERIMENT_ID_OVERRIDE = None
MULTIRUN_RUN_REVISION_OVERRIDE = None
# ------------------------------------------------------------
#   Sec 4.3 -- 5.4) Manifest-based multi-run selection
# ------------------------------------------------------------

# Example: compare the electrification sweep under Base control.
MULTIRUN_MANIFEST_FILTERS = {
    "case_label": [
        "Base case",
        "Case 1",
        "Case 2",
        "Case 3",
        "Case 4",
        "Case 5",
        "Case 6",
        "Case 7",
    ],
}

# Examples:
#
# Base + Case 1 for every electrification level:
#
# MULTIRUN_MANIFEST_FILTERS = {
#     "case_label": ["Base case", "Case 1"],
# }
#
# Base + Case 1 only at HP=100:
#
# MULTIRUN_MANIFEST_FILTERS = {
#     "case_label": ["Base case", "Case 1"],
#     "share_HP_target": [100],
# }





# ------------------------------------------------------------
#   Sec 4.3 -- 5.5) Duration curves parameters
# ------------------------------------------------------------
thresholds = (3, 4.5, 6, 9, 12, 15, 20)                 # power thresholds drawn from the LV grid [kW] (for duration curves for single household peaks)

USE_ABS_FOR_LINES_DURCURVE = False                      # duration curves in absolute value |X|




# ------------------------------------------------------------
#   Sec 4.3 -- 5.6) MULTI-RUN OUTPUTS TO GENERATE
# ------------------------------------------------------------
RUN_PLOT_MULTIRUN_COST_PERCENTILES          = True  # 4.13: plots cost-percentile charts 
RUN_PLOT_MULTIRUN_COST_DELTAS               = True  # 4.13: calculate cost differences for each household between baseline v. other setups
RUN_SAVE_MULTIRUN_HOUSEHOLD_COST_DETAILS    = True  # 4.13: to save cost details for each household
RUN_PLOT_MULTIRUN_DEMAND_COVERAGE           = True  # 4.13: plots internal self-sufficiency charts
RUN_SAVE_MULTIRUN_SUMMARY_CACHE             = True  # 4.13: writes the multi-run summary file



# ------------------------------------------------------------
#   Sec 4.3 -- 5.7) Economic savings table from ALREADY SAVED optimization runs
# ------------------------------------------------------------

RUN_BUILD_ECONOMIC_SAVINGS_TABLE            = True
ECON_SAVINGS_REFERENCE_CASE                 = "Base case"

# Each representative seasonal week represents 13 weeks/year.
ECON_SAVINGS_WEEKS_PER_SEASON               = 13.0

# Household percentiles to report.
ECON_SAVINGS_PERCENTILES = [
    10, 20, 30, 40, 50, 60, 70, 80, 90
]

# Require exactly the same households when Base and Case are paired.
ECON_SAVINGS_STRICT_HOUSEHOLD_PAIRING       = True

# Cost metric already used by the multirun economic plots.
ECON_SAVINGS_SYSTEM_COST_COLUMN = (
    "total_system_cost_effective_eur"
)

# Also append the new sheets to the experiment-level
# economic_summary.xlsx.
ECON_SAVINGS_APPEND_TO_EXPERIMENT_SUMMARY = True







# ------------------------------------------------------------
#   Sec 4.3 -- 5.8) Data policy for mixed-configuration charts
# ------------------------------------------------------------
# Simple logic:
# 1. try to use the existing summary/cache;
# 2. if it does not contain the required columns, rebuild from pickle files;
# 3. save the updated summary/cache.

MULTIRUN_USE_CACHE_IF_COMPLETE              = False
MULTIRUN_REBUILD_FROM_PICKLE_IF_NEEDED      = True



# ------------------------------------------------------------
#   Sec 4.3 -- 5.9) Common plotting options
# ------------------------------------------------------------

MULTIRUN_USE_COMMON_YLIMS = True                    # Same y-axis limits across seasons

MULTIRUN_COST_PERCENTILE_YLIM_FLOOR_ZERO = False    # Use zero as the lower limit for cost percentile charts
MULTIRUN_DEMAND_ENERGY_YLIM_FLOOR_ZERO = True       # Use zero as the lower limit for internal self-sufficiency charts
MULTIRUN_TOTAL_SYSTEM_COST_YLIM_FLOOR_ZERO = True  # Use zero as the lower limit for the secondary cost axis  (WARNING: if total cost is negative, "True" would graphically cut it)

MULTIRUN_SYSTEM_COST_MARKER_SIZE = 3.8


# ------------------------------------------------------------
# Sec 4.3 -- 5.10) Optional extra outputs / safety analysis for Net import - Demand - Internal self-sufficiency balances
# ------------------------------------------------------------
# Default: disabled. Enable them only when they are actually needed.
MULTIRUN_EXPORT_MARKET_TS = False                   # If True, save a table with market cost time series for each configuration and each season
MULTIRUN_SAVE_SELF_SUFFICIENCY_TABLE = True         # Save self_sufficiency_<tag>.xlsx with df_plot + skipped
MULTIRUN_SAVE_ADDITIONAL_COST_AUDIT = False         # If True, save a summary table of the weight of the different "additional" costs, e.g. shortage, PV curtailment, DHW boost, borne by the system
MULTIRUN_RUN_SELF_SUFFICIENCY_AUDIT = True
MULTIRUN_PLOT_SYSTEM_COST_DURATION_CURVES = False   # If True, also print/plot system cost duration curves per timestep across configurations, to inspect the average system cost over the observed horizon

# Self-sufficiency audit parameters
MULTIRUN_SELF_SUFF_GAP_WARN_PCT_DEMAND = 2.0
MULTIRUN_SELF_SUFF_GAP_WARN_ABS_KWH = 100.0

# Demand coverage control parameters in charts
MULTIRUN_DEMAND_MISMATCH_ACTION = "warn"
MULTIRUN_PRINT_DEMAND_COVERAGE_CHECKS = False

# Main metric for cost duration curves.
# It must be a column exportable from system_cost_ts.
MULTIRUN_DURATION_CURVE_COST_COLUMN = "total_system_effective_cost_per_timestep_eur"

# If True and MULTIRUN_PLOT_SYSTEM_COST_DURATION_CURVES=True,
# automatically builds df_market_ts_wide even if MULTIRUN_EXPORT_MARKET_TS=False.
MULTIRUN_BUILD_MARKET_TS_FOR_DURATION_CURVES = True


# ------------------------------------------------------------
# Household-level paired cost differences
# ------------------------------------------------------------

# Configuration used as the reference for paired household deltas.
MULTIRUN_DELTA_REFERENCE_CASE_LABEL = "Base case"

# Runs are paired only when they belong to the same physical
# electrification scenario.
MULTIRUN_DELTA_PAIR_KEY = "electrification_id"

# Delta convention:
# configuration cost - reference cost
# negative = saving; positive = household penalization.
MULTIRUN_DELTA_ZERO_TOL_EUR = 1e-9

# Use a vertical axis symmetric around zero for delta charts.
MULTIRUN_DELTA_SYMMETRIC_YLIM = True




# ------------------------------------------------------------
# Sec 4.3 -- 5.11) Options for comparison between specific configuration (name labelling fix when too long)
# ------------------------------------------------------------
# Optional compact output-folder tag for multi-run comparisons.
MULTIRUN_OUTPUT_TAG_OVERRIDE = "ALL_CASES"  # "ALL_CASES" | "BASE_VS_EV" (es specific config comparison) | None (if you want to use the automatically derived from MULTIRUN_MANIFEST_FILTERS)


### 4.3.6 Advanced reproducibility / run identity

In [ ]:
# Short deterministic UID appended to every run folder.
RUN_UID_HASH_LENGTH                     = 8
RUN_IDENTITY_SCHEMA                     = "HAGRID_M04_RUN_V1"

RB_CACHE_UID_HASH_LENGTH                = 8
RB_CACHE_IDENTITY_SCHEMA                = "HAGRID_M04_RB_CACHE_V1"


# Important parameters that should automatically alter run_uid.
#
# Add further parameters here if they become an explicit
# experimental dimension.
RUN_IDENTITY_PARAMETER_NAMES = (
    "PV_power_kwp",
    "BESS_power_kw",
    "BESS_capacity_kwh",

    "RANDOM_SEED_HAS_TECH",
    "RANDOM_SEED_SHARE_SUBSETS",

    "USE_PF_LINE_LIMITS_V1",
)

# Seeds affecting the physical assignment of technologies.
RANDOM_SEED_HAS_TECH                    = 12345
RANDOM_SEED_SHARE_SUBSETS               = 24680


# ------------------------------------------------------------
# Sec 4.3 -- 6.2) Physical penetration variables forming the run identity.
# ------------------------------------------------------------
RUN_SHARE_KEYS = (
    "PV",
    "BESS",
    "BEV",
    "DHW",
    "HP",
    "COOL",
    "INDUCTION",
    "V2G",
)


RUN_SHARE_FOLDER_TOKEN = {
    "PV": "PV",
    "BESS": "B",
    "BEV": "EV",
    "DHW": "DHW",
    "HP": "HP",
    "COOL": "C",
    "INDUCTION": "I",
    "V2G": "V2G",
}


MANIFEST_PHYSICAL_TECH_KEYS = (
    "PV",
    "BESS",
    "BEV",
    "V2G",
    "DHW",
    "HP",
    "COOL",
    "INDUCTION",
)

   ## 4.4 Directories / Static paths

In [ ]:
# Generic directories
# input_folder = 'input' 
# output_folder = 'output'
PROJECT_ROOT = os.path.abspath(os.getcwd())

input_folder = os.path.join(PROJECT_ROOT, "input",)
output_folder = os.path.join(PROJECT_ROOT,"output",)

print("[PROJECT ROOT]",PROJECT_ROOT,)
print("[OUTPUT ROOT]",output_folder,)

if not os.path.isdir(
    input_folder
):
    raise RuntimeError(
        "The notebook is not being executed "
        "from the expected HAGRID project root.\n"
        f"Current working directory:\n{PROJECT_ROOT}"
    )



#-----------------------------------------------
# Sec 4.4 -- 0) 10-minute time-series paths
#-----------------------------------------------
#  --- INPUT ---
prices_input_folder = os.path.join(input_folder, "04_PV_EE_prices", "hourly_EE_prices_GME")
pv_input_folder = os.path.join(input_folder, "04_PV_EE_prices", "hourly_PV")

# --- OUTPUT ---
profiles_output_folder = os.path.join(output_folder, "04_PV_EE_profiles_Xls", "GME_EE_prices_10min")
pv10_output_folder = os.path.join(output_folder, "04_PV_EE_profiles_Xls", "PV_10min_weeks")
#-------------------------------------------------------------------------------------



#-----------------------------------------------
# Sec 4.4 -- 1) Grid-archetype paths LV (from PANDAPOWER - Kerber)
#-----------------------------------------------
# --- INPUT ---
grid_archetypes_input_folder = os.path.join(output_folder, "05_Kerber_grid_archetypes_Py")



#-----------------------------------------------
# Sec 4.4 -- 2) Input-file paths required by the tool
#-----------------------------------------------

# --- INPUT ---
additional_loads_input_folder = os.path.join(output_folder, "03_additional_consumption_Xls")
base_loads_input_folder = os.path.join(output_folder, "02_base_consumption_Xls")

# Additional loads
dhw_demand_file = os.path.join(additional_loads_input_folder, f"{num_households}_households_{selected_region}_DHW_consumption_kWh.xlsx")
cooling_demand_file = os.path.join(additional_loads_input_folder, f"{num_households}_households_{selected_region}_Cooling_consumption_kWh.xlsx")
heating_demand_file = os.path.join(additional_loads_input_folder, f"{num_households}_households_{selected_region}_Heating_consumption_kWh.xlsx")
cooking_demand_file = os.path.join(additional_loads_input_folder, f"{num_households}_households_{selected_region}_induction_consumption_kWh.xlsx")

# 1) EV demand profile (the effective transfer and the related electricity consumption): add the occupancy profile to identify when they can recharge at home.
ev_demand_file = os.path.join(additional_loads_input_folder, f"{num_households}_households_{selected_region}_EV_electricity_demand_kWh.xlsx")     
# 2) EV "greedy/dumb" charging profile, i.e., when available EV domestic charging while also considering dwelling occupancy
file_charge_BEV_home_greedy = os.path.join(additional_loads_input_folder, f"{num_households}_households_{selected_region}_EV_home_greedy_charging_kWh.xlsx")
# 3) Individual household-occupancy profile
occupancy_file = os.path.join(additional_loads_input_folder, f"{num_households}_households_{selected_region}_occupancy.xlsx") 


# Base loads (households - appliances + lighting)
base_demand_file = os.path.join(output_folder,"02_base_consumption_Xls", f"{num_households}_households_{selected_region}_ts_kWh_aggregated_households_base.xlsx")

# PV
pv_10min_file_path = os.path.join(pv10_output_folder, f"{num_households}_households_{selected_region}_PV_10min_seasons_1kWp.xlsx") 

# Prices for purchase and selling electricity (withdrawal from and injection into the LV grid)
prices_10min_path = os.path.join(profiles_output_folder, "GME_2023_electricity_prices_10min.xlsx")

# --- BEV metadata by household: battery capacity, charging power, etc. ---
ev_metadata_file = os.path.join(output_folder, "03_additional_consumption_Xls", f"{num_households}_households_{selected_region}_ev_metadata_by_person.xlsx")







# ------------------------------------------------------------
# Create only stable root directories.
# Configuration-specific folders are created on demand.
# ------------------------------------------------------------
STATIC_FOLDERS_TO_CREATE = [
    input_folder,
    output_folder,

    # Profiles and time series
    prices_input_folder,
    pv_input_folder,
    profiles_output_folder,
    pv10_output_folder,

    # Electrical loads
    additional_loads_input_folder,
    base_loads_input_folder,
]


   ## 4.5 Catalogs and dictionaries

   ### 4.5.0 Dictionaries for time-series creation, optimization, rule-based profiles, and grid topology

In [ ]:
# ------------------------------------------------------------
# Sec 4.5 -- 000) VARIOUS DICTIONARIES
# ------------------------------------------------------------

# --------- 10-minute TIME-SERIES dictionaries
# Seasons (WARNING: if not already defined earlier during seasonal-profile construction)
season = [
    'inverno SUM',
    'primavera SUM',
    'estate SUM',
    'autunno SUM'
    ]

# WARNING here as well if periods are changed - season mapping
season_periods = {
    'winter':  ("2023-02-15 00:00:00", "2023-02-21 23:00:00"),
    'spring':("2023-04-19 00:00:00", "2023-04-25 23:00:00"),
    'summer':   ("2023-07-12 00:00:00", "2023-07-18 23:00:00"),
    'autumn':  ("2023-10-11 00:00:00", "2023-10-17 23:00:00"),
}

# Columns of price from import
cols_required = [
    'PUN [€/kWh_e]', 
    'NORD [€/kWh_e]' ,
    'CNOR [€/kWh_e]', 
    'CSUD [€/kWh_e]', 
    'SUD [€/kWh_e]', 
    'SICI [€/kWh_e]', 
    'SARD [€/kWh_e]'
    ]



#--------- Dictionaries for GRID-ARCHETYPE construction (PandaPower) ------------
KERBER_BLUEPRINT_FILES = {
    # rural (linear, few customers)
    "kerber_rural_grid_A": "kerber_rural_grid_kabel_2.json",           
    "kerber_rural_grid_B": "kerber_rural_grid_freileitung_2.json",

    "kerber_rural_grid_C": "kerber_rural_grid_kabel_1.json",           
    "kerber_rural_grid_D": "kerber_rural_grid_freileitung_1.json",

    # "mixed" / small-town (linear + open ring)
    "kerber_mixed_grid": "kerber_mixed_grid.json",

    # urban/suburban (open ring / mesh network)
    "kerber_urban_grid_A": "kerber_urban_grid_kabel_2.json",    
    "kerber_urban_grid_B": "kerber_urban_grid_kabel_1.json" ,
}
# -------------------------------------------------------------------



#--------- Dictionaries for ENG-ITA conversion / seasons / geographical area / flow names
seasons_dict = {
    'inverno SUM': 'winter',
    'primavera SUM': 'spring',
    'estate SUM': 'summer',
    'autunno SUM': 'autumn',
    }


# It can be modified if needed different geographical allocation between macro-areas (e.g., Islands not Sicily but Sardinia instead)
# WARNING: the current ISTAT classification does not include all the current electrical bidding zones, as Calabria
region_to_price_col = {
    'N-O': 'NORD',
    'N-E': 'NORD',   
    'Cen' : 'CSUD',  # Centre -> CSUD (Centre-South)
    'Sud' : 'SUD',
    'Isl' : 'SICI'   
}


flow_names = [
    "demand_kwh", 
    "pv_kwh",
    "pv_to_load_kwh",
    "excess_pv_kwh", 
    "pv_to_batt_input_kwh",
    "energy_added_to_soc_kwh", 
    "batt_draw_from_soc_kwh",
    "batt_delivered_to_load_kwh",
    "soc_kwh", 
    "grid_export_kwh", 
    "grid_import_kwh"
]

# for track of the flows to storage ESWH and EV (households conventional/RB setup)
extra_flow_names = [
    # --- ESWH - Electric Storage Water Heater for DHW domestic hot water ---
    "SOC_DHW_prev_kWh",
    "SOC_DHW_after_loss_kWh",
    "SOC_DHW_raw_kWh",
    "SOC_DHW_kWh",
    "SOC_DHW_%",
    "P_el_to_DHW_kw",
    "DHW_boost_th_kWh",
    "DHW_boost_el_kw",
    "DHW_shortage_th_kWh",

    # --- EV ---
    "P_el_to_BEV_kw",
    "SOC_BEV_kWh",
    "SOC_BEV_%",
    "BEV_deficit_kWh",
    "BEV_outdoor_el_kWh",
    "BEV_outdoor_el_kw",
    "BEV_shortage_kWh",

    # --- economics ---
    "econ_€"
]


TECH_TAGS = {
    'PV':        ["PV"],
    "BESS":      ["BESS"],
    "BEV":       ["CHARGER", "V2G"],          # EV = charger/V2G
    "DHW":       ["DHW_BOOST", "DHW_STORAGE"],# ESWH electric storage water heater for DHW
    "HP":        ["DMD_ELC_HEAT"],            # HP = Heating by heat pumps
    "COOL":      ["DMD_ELC_COOL"],            # Cooling
    "INDUCTION": ["DMD_ELC_COOK"],            # Induction = Cook
}
TECH_KEYS = list(TECH_TAGS.keys())


REQUIRE_NONZERO_BY_TECH = {
    'PV': True,
    "BESS": False,
    "BEV": False,
    "DHW": False,
    "HP": False,
    "COOL": False,
    "INDUCTION": False,
}


#----------- Dictionaries PLOTTING State of charge (SOC) -----------------------------
SOC_FILE_STEMS = {
    "BESS": "soc_bess",
    "DHW": "soc_dhw",
    "BEV": "soc_bev",
}
# ------------------------------------------------------------------

   ### 4.5.1 Dictionaries for chart plotting with different scenarios


   Grouped considered categories for LV grid and household perspectives 

In [ ]:
#---------- Dictionaries for plotting figures (optimization) -----------------------
seasons = ['winter', 'spring', 'summer', 'autumn']
season_english_dict = {
    'winter': 'Winter', 
    'spring': 'Spring',
    'summer': 'Summer', 
    'autumn':'Fall'
    }

percentile_cols = [
    "p10_eur", "p20_eur", "p30_eur", "p40_eur", "p50_eur",
    "p60_eur", "p70_eur", "p80_eur", "p90_eur",
]


# ================================
# Sec 4.5 -- 1.1) DEFINITION OF FLEXIBILITY CATEGORIES
# ================================

FLEX_CATEGORIES = [
    'case_0',   # no flexibility
    'case_B',   # BESS only
    'case_V',   # BEV only
    'case_H',   # ESWH only (electric storage water heater)
    'case_BV',  # BESS + BEV
    'case_BH',  # BESS + ESWH
    'case_VH',  # BEV + ESWH
    'case_BVH'  # BESS + BEV + ESWH
]

# Short description for labels in English
flex_category_labels = {
    'case_0':   'NO flex',      # no flexibility
    'case_B':   'B',            # BESS only
    'case_V':   'V',            # BEV only
    'case_H':   'W',            # ESWH only
    'case_BV':  'B + V',        # BESS + BEV
    'case_BH':  'B + W',        # BESS + ESWH
    'case_VH':  'V + W',        # BEV + ESWH
    'case_BVH': 'B + V + W'     # BESS + BEV + ESWH
}

# ==================================================
# Sec 4.5 -- 1.2) LIST OF RELEVANT COLUMNS (AGGREGATED RESULTS)
# ==================================================

# Household side: main aggregated flows for each category
# - Export to LV
# - PV, useful to the household and net of curtailment, to bus_el
# - BESS charging
# - BEV charging
# - V2G
# - DHW storage
house_relevant_columns = []
for pref in FLEX_CATEGORIES:
    house_relevant_columns.extend([
        (f'{pref}_bus_el', f'{pref}_to_the_grid_BT', 'flow'),
        (f'{pref}_PV', f'{pref}_bus_el', 'flow'),
        (f'{pref}_bus_el', f'{pref}_BESS', 'flow'),
        (f'{pref}_bus_el', f'{pref}_charger', 'flow'),
        (f'{pref}_V2G', f'{pref}_bus_el', 'flow'),
        (f'{pref}_bus_el', f'{pref}_DHW_storage', 'flow'),
    ])



# =====================================
# Sec 4.5 -- 1.3) HOUSEHOLD SIDE: COLUMNS FOR LINES AND AREAS
# =====================================

# Here I select the following line series:
# - base demand
# - BEV charging
# - DHW charging
# - BESS charging
line_columns = []
for pref in FLEX_CATEGORIES:
    line_columns.extend([
        (f'{pref}_bus_el', f'{pref}_dmd_elc_base',    'flow'),  # base demand
        (f'{pref}_bus_el', f'{pref}_charger',         'flow'),  # BEV charging
        (f'{pref}_bus_el', f'{pref}_DHW_storage',     'flow'),  # DHW charging
        (f'{pref}_bus_el', f'{pref}_BESS',            'flow'),  # BESS charging
    ])

# AREAS, i.e. GENERATION, in the following order:
# 1) all PV series, case_0, case_B, ..., case_BVH
# 2) all imports from the LV grid
# 3) all BESS discharges
area_columns = []

# 1) PV for all categories
for pref in FLEX_CATEGORIES:
    area_columns.append((f'{pref}_PV', f'{pref}_bus_el', 'flow'))

# 2) Import from LV for all categories
for pref in FLEX_CATEGORIES:
    area_columns.append((f'from_the_grid_BT_to_{pref}', f'{pref}_bus_el', 'flow'))

# 3) BESS discharge for all categories
for pref in FLEX_CATEGORIES:
    area_columns.append((f'{pref}_BESS', f'{pref}_bus_el', 'flow'))


# =====================================
# Sec 4.5 -- 1.4) HOUSEHOLD-SIDE COLUMN NAMES
# =====================================

column_names = {}

# ------------------------------------------------------------------
# AGGREGATED DEMAND NAMES ON THE HOUSEHOLD SIDE
# ------------------------------------------------------------------
column_names[
    ('ALL_bus_el', 'ALL_dmd_elc_base', 'flow')
] = 'El base demand'

column_names[
    ('ALL_bus_el', 'ALL_dmd_elc_Cool', 'flow')
] = 'El cooling demand'

column_names[
    ('ALL_bus_el', 'ALL_dmd_elc_Heat', 'flow')
] = 'El heating demand'

column_names[
    ('ALL_bus_el', 'ALL_dmd_elc_Cook', 'flow')
] = 'El cooking demand'

# Additional electric demand associated with the DHW boost,
# when this series is available.
column_names[
    ('ALL_bus_el', 'ALL_DHW_boost', 'flow')
] = 'El to DHW boost'


# ------------------------------------------------------------------
# TECHNOLOGY- AND FLEXIBILITY-CATEGORY-SPECIFIC FLOW NAMES
# ------------------------------------------------------------------

for pref, descr in flex_category_labels.items():

    # --------------------------------------------------------------
    # Demand series split by household flexibility category
    # --------------------------------------------------------------

    # Base electric demand
    column_names[
        (f'{pref}_bus_el', f'{pref}_dmd_elc_base', 'flow')
    ] = f'El base demand ({descr})'

    # Cooling demand
    column_names[
        (f'{pref}_bus_el', f'{pref}_dmd_elc_Cool', 'flow')
    ] = f'El cooling demand ({descr})'

    # Heating demand
    column_names[
        (f'{pref}_bus_el', f'{pref}_dmd_elc_Heat', 'flow')
    ] = f'El heating demand ({descr})'

    # Cooking demand
    column_names[
        (f'{pref}_bus_el', f'{pref}_dmd_elc_Cook', 'flow')
    ] = f'El cooking demand ({descr})'

    # --------------------------------------------------------------
    # Flexible-technology exchanges
    # --------------------------------------------------------------

    # BEV charging
    column_names[
        (f'{pref}_bus_el', f'{pref}_charger', 'flow')
    ] = f'El to charge BEV ({descr})'

    # Electric storage water heater charging
    column_names[
        (f'{pref}_bus_el', f'{pref}_DHW_storage', 'flow')
    ] = f'El to charge ESWH ({descr})'

    # BESS charging
    column_names[
        (f'{pref}_bus_el', f'{pref}_BESS', 'flow')
    ] = f'El to charge BESS ({descr})'

    # BESS discharging
    column_names[
        (f'{pref}_BESS', f'{pref}_bus_el', 'flow')
    ] = f'El from BESS ({descr})'

    # PV generation
    column_names[
        (f'{pref}_PV', f'{pref}_bus_el', 'flow')
    ] = f'PV generation ({descr})'

    # Export to the LV grid
    column_names[
        (f'{pref}_bus_el', f'{pref}_to_the_grid_BT', 'flow')
    ] = f'El export to LV grid ({descr})'

    # Import from the LV grid
    column_names[
        (f'from_the_grid_BT_to_{pref}', f'{pref}_bus_el', 'flow')
    ] = f'El import from LV grid ({descr})'

### 4.5.2 Dictionaries for comparison of different configurations

In [ ]:
# ------------------------------------------------------------
#   Sec 4.5.2 -- 0) Main KPI and cost metric used
# ------------------------------------------------------------

TOTAL_COST_COL_FOR_MULTIRUN_PLOT = (
    "total_system_cost_effective_eur"
)

pv_share_to_compare = PV_share



# ------------------------------------------------------------
#   Sec 4.5.2 -- 5) Market time-series columns / optional KPIs
# ------------------------------------------------------------
# Used only if MULTIRUN_EXPORT_MARKET_TS=True.

MULTIRUN_MARKET_TS_COLUMNS = [
    # Effective economic cost used for reporting/comparison.
    "total_system_effective_cost_per_timestep_eur",

    # Complete simulation/objective-related cost.
    # Retained for technical diagnostics.
    "total_system_cost_per_timestep_eur"

    # Version normalized by timestep demand.
    "total_system_cost_per_kwh_demand_timestep",

    # Energy settlement consistent with the objective.
    "objective_energy_settlement_eur",

    # Market alias/fallback.
    "market_mt_net_cost_eur",
    "market_mt_purchase_cost_eur",
    "market_mt_sale_revenues_eur",
    "market_mt_net_cost_eur_per_kwh_demand",

    # Energy KPIs.
    "gross_mv_import_share_of_demand_ts",
    "gross_mv_export_share_of_demand_ts",
    "gross_mv_exchange_share_of_demand_ts",
]



# ------------------------------------------------------------
#   Sec 4.5.2 -- 6) Results pickle dictionary
# ------------------------------------------------------------
# Used by the flow_pickle_path / resolve_flow_pickle_path functions.

FLOW_PICKLE_STEMS = {
    "flow_el_df": "flow_el",
    "flow_grid_df": "flow_grid",
    "flow_grid_root": "flow_grid_root",

    "flow_PV_curtail_df": "flow_pv_curtail",
    
    "flow_DHW_use_df": "flow_dhw_use",
    "flow_charge_el_BEV_df": "flow_bev_charge",
    "flow_discharge_el_BEV_df": "flow_bev_discharge",
    "flow_grid_lines_only_df": "flow_grid_lines",
}


# ------------------------------------------------------------
# Sec 4.5.2 -- 7) Scenario table symbols
# ------------------------------------------------------------
# Options:
#   "check_x_no"      -> OPT=✓, RB=X,  NO=no
#   "letters"         -> OPT=OPT, RB=RB, NO=NO
#   "minimal_shapes"  -> OPT=●, RB=○,  NO=–
#   "diamonds"        -> OPT=◆, RB=◇,  NO=–
#   "icons"           -> OPT=★, RB=↻,  NO=–
MULTIRUN_CONTROL_SYMBOL_STYLE = "diamonds"

MULTIRUN_CONTROL_SYMBOL_MAPS = {
    # Option with checkmark - X - no
    "check_x_no": {
        "OPT": "✓",
        "RB": "X",
        "NO": "no",
    },

    # Baseline option with names
    "letters": {
        "OPT": "OPT",
        "RB": "RB",
        "NO": "NO",
    },

    # Filled/empty symbol option: circle
    "minimal_shapes": {
        "OPT": "●",
        "RB": "○",
        "NO": "–",
    },

    # Filled/empty symbol option: diamond
    "diamonds": {
        "OPT": "◆",
        "RB": "◇",
        "NO": "–",
    },

    # Icon option; pay attention to which fonts are available in the paper
    "icons": {
        "OPT": "★",
        "RB": "↻",
        "NO": "–",
    },
}




### 4.5.3 Catalogs for management setup of flexibility options

In [ ]:
# ------------------------------------------------------------
# Sec 4.5 -- 3) MIXED OPT/RB CONFIGURATIONS BY FLEXIBILITY TECHNOLOGY
# ------------------------------------------------------------

# To select the configurations that should be tested in the multi-run refer to the following list "ACTIVE_CONTROL_CONFIG_KEYS " below
CONTROL_CONFIG_CATALOG  = [
    {
        # Case 0/base case: base load + additional load,
        # DHW and EV rule-based, no BESS.
        "mode": "MIX_BNO_VRB_HRB",
        "label": "BESS NO | EV RB | DHW RB",
        "case_label": "Base case",
        "control": {
            "BESS": "NO",
            "BEV": "RB",
            "DHW": "RB",                        # Here and below: DHW = Domestic Hot Water; when present, there is a corresponding electric storage water heater (ESWH)
        },
    },

    {
        # Case 1: no BESS, EV rule-based, ESWH smart.
        "mode": "MIX_BNO_VRB_HOPT",
        "label": "BESS NO | EV RB | DHW OPT",
        "case_label": "Case 1",
        "control": {
            "BESS": "NO",
            "BEV": "RB",
            "DHW": "OPT",
        },
    },

    {
        # Case 2: no BESS, EV smart, ESWH rule-based.
        "mode": "MIX_BNO_VOPT_HRB",
        "label": "BESS NO | EV OPT | DHW RB",
        "case_label": "Case 2",
        "control": {
            "BESS": "NO",
            "BEV": "OPT",
            "DHW": "RB",
        },
    },

    {
        # Case 3: no BESS, EV smart, ESWH smart.
        "mode": "MIX_BNO_VOPT_HOPT",
        "label": "BESS NO | EV OPT | DHW OPT",
        "case_label": "Case 3",
        "control": {
            "BESS": "NO",
            "BEV": "OPT",
            "DHW": "OPT",
        },
    },

    {
        # Case 4: BESS smart, EV rule-based, ESWH rule-based.
        "mode": "MIX_BOPT_VRB_HRB",
        "label": "BESS OPT | EV RB | DHW RB",
        "case_label": "Case 4",
        "control": {
            "BESS": "OPT",
            "BEV": "RB",
            "DHW": "RB",
        },
    },

    {
        # Case 5: BESS smart, EV rule-based, ESWH smart.
        "mode": "MIX_BOPT_VRB_HOPT",
        "label": "BESS OPT | EV RB | DHW OPT",
        "case_label": "Case 5",
        "control": {
            "BESS": "OPT",
            "BEV": "RB",
            "DHW": "OPT",
        },
    },

    {
        # Case 6: BESS smart, EV smart, ESWH rule-based.
        "mode": "MIX_BOPT_VOPT_HRB",
        "label": "BESS OPT | EV OPT | DHW RB",
        "case_label": "Case 6",
        "control": {
            "BESS": "OPT",
            "BEV": "OPT",
            "DHW": "RB",
        },
    },

    {
        # Case 7: BESS smart, EV smart, ESWH smart.
        "mode": "MIX_BOPT_VOPT_HOPT",
        "label": "BESS OPT | EV OPT | DHW OPT",
        "case_label": "Case 7",
        "control": {
            "BESS": "OPT",
            "BEV": "OPT",
            "DHW": "OPT",
        },
    },

    # {
    #     # Possible additional case: BESS rule-based, EV smart, ESWH smart.
    #     "mode": "MIX_BRB_VOPT_HOPT",
    #     "label": "BESS RB | EV OPT | DHW OPT",
    #     "case_label": "Case 8",
    #     "control": {
    #         "BESS": "RB",
    #         "BEV": "OPT",
    #         "DHW": "OPT",
    #     },
    # },
]



# ------------------------------------------------------------
# Short human-readable tags used only in run-folder labels.
#
# They do NOT replace MODE and do NOT affect optimization logic.
# ------------------------------------------------------------

CONTROL_SHORT_TAG_BY_CASE_LABEL = {
    "Base case": "BASE",
    "Case 1": "DHWOPT",
    "Case 2": "EVOPT",
    "Case 3": "EV_DHW_OPT",
    "Case 4": "BESSOPT",
    "Case 5": "BESS_DHW_OPT",
    "Case 6": "BESS_EV_OPT",
    "Case 7": "ALLOPT",
}

### 4.5.4 List of possible configurations/experiments

In [ ]:
# ============================================================
#   Sec 4.5 -- 4.1) ELECTRIFICATION EXPERIMENT LIBRARY
# ============================================================

ELECTRIFICATION_EXPERIMENTS = {

    # --------------------------------------------------------
    # Reference scenario
    # --------------------------------------------------------

    "REF_NO_ELECTRIFICATION": {
        "varying_shares": (),
        "scenarios": [
            {
                "scenario_id": "NOEL",
                "shares": {
                    "PV": 0,
                    "BESS": 0,
                    "BEV": 0,
                    "V2G": 0,
                    "DHW": 0,
                    "HP": 0,
                    "COOL": 0,
                    "INDUCTION": 0,
                },
            }
        ],
    },



    # --------------------------------------------------------
    # Fully electrified scenario (both demand and distributed generation)
    # --------------------------------------------------------
    "FULL_ELECTRIFICATION": {
        "varying_shares": (),
        "scenarios": [
            {
                "scenario_id": "FULL100",
                "shares": {
                    "PV": 100,
                    "BESS": 100,
                    "BEV": 100,
                    "V2G": 100,
                    "DHW": 100,
                    "HP": 100,
                    "COOL": 100,
                    "INDUCTION": 100,
                },
            },
        ],
    },


    # --------------------------------------------------------
    # Single-technology penetration sweeps
    # --------------------------------------------------------

    "HP_SWEEP": {
        "builder": "share_sweep",
        "varying_shares": ("HP",),
        "tech": "HP",
        "values": tuple(range(10, 101, 10)),
    },


    "PV_SWEEP": {
        "builder": "share_sweep",
        "varying_shares": ("PV",),
        "tech": "PV",
        "values": tuple(range(10, 101, 10)),
    },


    "COOL_SWEEP": {
        "builder": "share_sweep",
        "varying_shares": ("COOL",),
        "tech": "COOL",
        "values": tuple(range(10, 101, 10)),
    },


    "EV_SWEEP": {
        "builder": "share_sweep",
        "varying_shares": ("BEV",),
        "tech": "BEV",
        "prefix": "EV",
        "values": tuple(range(10, 101, 10)),
    },


    "DHW_SWEEP": {
        "builder": "share_sweep",
        "varying_shares": ("DHW",),
        "tech": "DHW",
        "values": tuple(range(10, 101, 10)),
    },



    # --------------------------------------------------------
    # Independent two-dimensional grids
    # --------------------------------------------------------

    "HP_PV_GRID": {
        "builder": "share_grid",
        "varying_shares": ("HP", "PV"),
        "axes": {
            "HP": (20, 50, 100),
            "PV": (20, 50, 100),
        },
    },


    "PV_COOL_GRID": {
        "builder": "share_grid",
        "varying_shares": ("PV", "COOL"),
        "axes": {
            "PV": (20, 50, 100),
            "COOL": (20, 50, 100),
        },
    },


    # --------------------------------------------------------
    # Dependent technologies
    # --------------------------------------------------------


    "EV_V2G_GRID": {
        "builder": "ev_v2g_grid",
        "varying_shares": ("BEV", "V2G"),
        "ev_values": (20, 50, 100),
        "v2g_values": (0, 50, 100),
    },


    "PV_BESS_GRID": {
        "builder": "pv_bess_coupled_grid",
        "varying_shares": ("PV", "BESS"),
        "pv_values": (20, 50, 100),
        "bess_on_pv_values": (0, 25, 50, 100),
    },


    # --------------------------------------------------------
    # User-defined electrification pathway
    # --------------------------------------------------------

    "ELECTRIFICATION_PATH": {
        "varying_shares": ("PV", "HP", "BEV", "COOL"),
        "scenarios": [
            {
                "scenario_id": "E20",
                "shares": {
                    "PV": 30,
                    "HP": 20,
                    "BEV": 20,
                    "COOL": 10,
                },
            },
            {
                "scenario_id": "E60",
                "shares": {
                    "PV": 70,
                    "HP": 60,
                    "BEV": 60,
                    "COOL": 50,
                },
            },
            {
                "scenario_id": "E100",
                "shares": {
                    "PV": 100,
                    "HP": 100,
                    "BEV": 100,
                    "COOL": 100,
                },
            },
        ],
    },
}


# ============================================================
#   Sec 4.5 -- 4.2) COMPARISON BETWEEN DIFFERENT MANAGEMENT SETUPS OF FLEXIBILITY OPTIONS
# ============================================================

# List of potential comparison between different flexibility management setups
CONTROL_GROUPS = {
    "BASE_ONLY": [
        "Base case",
    ],

    "EV_SMART_ONLY": [
        "Case 2",
    ],
    
    "DHW_SMART_ONLY": [
        "Case 1",
    ],

    "BESS_ONLY": [
        "Case 4",
    ],

    "BESS_EV_ONLY": [
        "Case 6",
    ],

    "BASE_VS_DHW_SMART": [
        "Base case",
        "Case 1",
    ],

    "BASE_VS_EV_SMART": [
        "Base case",
        "Case 2",
    ],

    "BASE_VS_BESS_SMART": [
        "Base case",
        "Case 4",
    ],

    "BASE_VS_ALL_SMART": [
        "Base case",
        "Case 7",
    ],

    "ALL_CASES": [
        "Base case",
        "Case 1",
        "Case 2",
        "Case 3",
        "Case 4",
        "Case 5",
        "Case 6",
        "Case 7",
    ],

    # If rerun with just some cases is required
    "CASES_2_TO_7": [
        "Case 2",
        "Case 3",
        "Case 4",
        "Case 5",
        "Case 6",
        "Case 7",
    ],

    # If rerun with just some cases is required
    "EXCEPT_case_1": [
        "Base case",
        "Case 2",
        "Case 3",
        "Case 4",
        "Case 5",
        "Case 6",
        "Case 7",
    ],


    # If specific run are missing
    "CASES_5_TO_7": [
        "Case 5",
        "Case 6",
        "Case 7",
    ],


    # If specific run are missing
    "CASES_6_AND_7": [
        "Case 6",
        "Case 7",
    ],
}

   ## 4.6 Functions

   ### 4.6.1 F(x)  I/O, pickle, metadata, path

In [ ]:
# -------------------------------------------------------------------
#       Sec 4.6 -- 1.0) FUNCTIONS FOR Pickle files, loading and saving
#-------------------------------------------------------------------

# Function to save a dataframe with pickle and gzip
def save_compressed_pickle(dataframe, filename):
    filename = os.fspath(filename)

    parent = os.path.dirname(os.path.abspath(filename))
    if parent:
        os.makedirs(parent, exist_ok=True)

    abs_path = os.path.abspath(filename)
    if os.name == "nt" and len(abs_path) > 230:
        print(
            f"[WARN][PATH LENGTH] Long path with {len(abs_path)} characters. "
            f"Possible Windows/OneDrive issue:\n{abs_path}"
        )

    with gzip.open(filename, 'wb') as f:
        pickle.dump(dataframe, f, protocol=pickle.HIGHEST_PROTOCOL)



# Function to load dataframes
def load_compressed_pickle(filename):
    """
    Loads a gzip-compressed pickle object.
    """
    with gzip.open(filename, "rb") as file_obj:
        return pickle.load(file_obj)



# -------------------------------------------------------------------
#       Sec 4.6 -- 1.1) FUNCTIONS FOR run metadata and JSON
#-------------------------------------------------------------------

# Function to convert objects so that configuration information/metadata can be saved in JSON files
def _json_safe(x):
    """
    Converts Python/numpy/pandas objects that are not directly serializable
    into forms compatible with json.dump().
    """
    if x is None:
        return None

    if isinstance(x, (str, int, float, bool)):
        return x

    if isinstance(x, (np.integer,)):
        return int(x)

    if isinstance(x, (np.floating,)):
        return float(x)

    if isinstance(x, (set, tuple, list)):
        return [_json_safe(v) for v in x]

    if isinstance(x, dict):
        return {str(k): _json_safe(v) for k, v in x.items()}

    if isinstance(x, Path):
        return str(x)

    if isinstance(x, pd.Timestamp):
        return x.isoformat()

    return str(x)


# Function to save the metadata of each observed configuration
def save_run_metadata(
    *,
    dirs,
    run_cfg,
    has_potential=None,
    has_run=None,
    extra=None,
):
    """Save the detailed passport of one specific run."""

    shares = _shares_from_cfg_or_global(run_cfg)

    meta = {
        "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "experiment_id": run_cfg["experiment_id"],
        "run_revision": run_cfg["run_revision"],
        "run_uid": run_cfg["run_uid"],
        "run_label": run_cfg["run_label"],
        "run_folder_name": run_cfg["run_folder_name"],
        "electrification_id": run_cfg.get("electrification_id"),
        "mode": run_cfg["mode"],
        "mode_label": run_cfg.get("label"),
        "case_label": run_cfg.get("case_label"),
        "control": dict(run_cfg["control"]),
        "shares_target": dict(shares),
        "LV_ARCHETYPE": LV_ARCHETYPE,
        "HAGRID_GRID_GROUP": HAGRID_GRID_GROUP,
        "num_households": int(num_households),
        "selected_region": selected_region,
        "PV_power_kwp": PV_power_kwp,
        "BESS_power_kw": BESS_power_kw,
        "BESS_capacity_kwh": BESS_capacity_kwh,
        "dirs_relative_to_output": {
            key: _path_relative_to_output(dirs[key])
            for key in ("xls", "py", "pics", "pflow_py", "pf_limits")
        },
    }

    if has_potential is not None:
        meta["physical_technology_counts"] = {
            tech: len(has_potential.get(tech, set()) or set())
            for tech in MANIFEST_PHYSICAL_TECH_KEYS
        }

    if has_run is not None:
        meta["active_technology_counts"] = {
            tech: len(has_run.get(tech, set()) or set())
            for tech in MANIFEST_PHYSICAL_TECH_KEYS
        }

    if extra:
        meta.update(extra)

    meta = _json_safe(meta)

    os.makedirs(dirs["xls"], exist_ok=True)

    with open(
        os.path.join(dirs["xls"], "run_meta.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    return meta



# -------------------------------------------------------------------
#       Sec 4.6 -- 1.2) FUNCTIONS for RESULT PICKLE PATHS
#-------------------------------------------------------------------

# Function to return the result-file directory of the optimization problem
def flow_pickle_path(metric_name, season, base_dir=None):
    """
    Returns the compact path for a flow pickle file.

    The configuration/MODE is no longer included in the filename:
    it is already encoded in the folder_output_py_based directory.
    """
    if base_dir is None:
        base_dir = py_based_output_folder

    if metric_name not in FLOW_PICKLE_STEMS:
        raise KeyError(
            f"Unrecognized metric name: {metric_name}. "
            f"Allowed values: {list(FLOW_PICKLE_STEMS)}"
        )

    stem = FLOW_PICKLE_STEMS[metric_name]
    fname = f"{stem}_{season}.pkl.gz"

    return os.path.join(base_dir, fname)


# Function to retrieve old files from which charts can be reconstructed
def flow_pickle_path_legacy(metric_name, season, mode, base_dir=None):
    """
    Temporary fallback to read old files, if everything has not yet been
    re-saved with the short names.
    """
    if base_dir is None:
        base_dir = py_based_output_folder

    legacy_names = {
        "flow_el_df": f"flow_el_df_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz",
        "flow_grid_df": f"flow_grid_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz",
        "flow_grid_root": f"flow_grid_root_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz",

        "flow_PV_curtail_df": (f"flow_PV_curtail_df_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz"),

        "flow_DHW_use_df": f"flow_DHW_use_df_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz",
        "flow_charge_el_BEV_df": f"flow_charge_el_BEV_df_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz",
        "flow_discharge_el_BEV_df": f"flow_discharge_el_BEV_df_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz",
        "flow_grid_lines_only_df": f"flow_grid_lines_{mode}_{num_households}_nuc_{selected_region}_{season}.pkl.gz",
    }

    return os.path.join(base_dir, legacy_names[metric_name])


# Function to reconstruct the path used to retrieve files for configuration comparison and chart print/plot routines
def resolve_flow_pickle_path(metric_name, season, mode, base_dir=None, required=True):
    """
    First searches for the short name.
    If it does not exist, tries the old name with MODE.
    """
    if base_dir is None:
        base_dir = py_based_output_folder

    p_short = flow_pickle_path(metric_name, season, base_dir=base_dir)
    if os.path.exists(p_short):
        return p_short

    p_legacy = flow_pickle_path_legacy(metric_name, season, mode, base_dir=base_dir)
    if os.path.exists(p_legacy):
        print(f"[WARN][PLOT] Using legacy filename for {metric_name}: {p_legacy}")
        return p_legacy

    if required:
        raise FileNotFoundError(
            f"[ERROR][PLOT] Pickle file not found for metric={metric_name}, "
            f"season={season}, mode={mode}.\n"
            f"Tried short name:\n  {p_short}\n"
            f"Tried legacy name:\n  {p_legacy}"
        )

    return None



# -------------------------------------------------------------------
#       Sec 4.6 -- 1.4) FUNCTIONS for TIME INDICES, timing and safe aggregators
#-------------------------------------------------------------------
# Function to associate a time series referring to a specific period, e.g. autumn, with a canonical time index, e.g. 00:00 1/1/2000
def to_canonical(series_or_df, idx_reale, idx_can):
    """
    Takes a Series/DataFrame indexed with idx_reale and returns
    the same object indexed on idx_can, with the same values and a new index.
    """
    x = series_or_df.reindex(idx_reale)  # safety
    x = x.copy()
    x.index = idx_can
    return x


# Function that reassigns the real time index to a series that has a canonical index
def set_real_index(df, idx_real):
    df2 = df.copy()
    df2.index = idx_real
    return df2


# Function to return an hours:minutes:seconds string format
def _format_hms(seconds: float) -> str:
    seconds = int(round(seconds))
    td = timedelta(seconds=seconds)
    h = td.seconds // 3600
    m = (td.seconds % 3600) // 60
    s = td.seconds % 60
    if td.days > 0:
        h += 24 * td.days
    if h > 0:
        return f"{h}h{m:02d}'{s:02d}\""
    elif m > 0:
        return f"{m}m{s:02d}s"
    else:
        return f"{s}s"



# Function to evaluate when and how long a code section takes to run
@contextmanager
def timed(label, store=None):
    t0 = perf_counter()
    yield
    dt = perf_counter() - t0
    print(f"[TIMER] {label}: {dt:8.3f} s")
    if store is not None:
        store[label] = store.get(label, 0.0) + dt




# Function to save a summary table with computational timing information for the different sections
def save_timings_txt(
    timings: dict,
    total_run_seconds: float,
    out_dir: str,
    filename: str = "timings_summary.txt"
):
    """
    timings: dict {label: seconds}
    """
    # sort by decreasing time
    sorted_items = sorted(timings.items(), key=lambda x: x[1], reverse=True)

    lines = []
    lines.append(f"Total run: {_format_hms(total_run_seconds)}")
    lines.append("")
    lines.append("=== TIMINGS SUMMARY (sorted) ===")

    for label, sec in sorted_items:
        lines.append(f"{sec:10.2f} s  {label}")

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"[INFO] Timings saved in: {path}")


# Function to sum values when creating economic-results dataframes
def _sum_if_present(df, col):
    return df[col].sum() if col in df.columns else np.nan

# Function to identify/order values when creating economic-results dataframes
def _first_if_present(df, col):
    return df[col].iloc[0] if (col in df.columns and len(df) > 0) else np.nan

# Function to identify the maximum value when creating economic-results dataframes
def _max_if_present(df, col):
    return df[col].max() if col in df.columns else np.nan

# --------------------------------------------------------------





#-------------------------------------------------------------------
#       Sec 4.6 -- 1.5) FUNCTIONS FOR CORRECT CONSTRUCTION OF DIRECTORIES/FOLDERS where files for the various configurations are saved
#-------------------------------------------------------------------
# Function to return a filesystem-safe string for file/folder names
def _safe_fs_tag(x):
    """
    Makes a string safe for file/folder names.
    """
    s = str(x)
    s = s.replace(" ", "_")
    s = s.replace("|", "_")
    s = s.replace("/", "_")
    s = s.replace("\\", "_")
    s = s.replace(":", "_")
    s = s.replace(";", "_")
    s = s.replace(",", "_")
    s = re.sub(r"_+", "_", s)
    return s.strip("_")


# Function to group grid archetypes (Kerber) into readable categories (urban-mixed-rural)
def _grid_family_from_lv_archetype(lv_archetype):
    """
    Groups Kerber archetypes into readable folders.
    """
    s = str(lv_archetype).lower()

    if "urban" in s:
        return "urban"
    if "mixed" in s:
        return "mixed"
    if "rural" in s:
        return "rural"

    return _safe_fs_tag(lv_archetype)




# Function to format technology shares for use in folder paths
def _fmt_share_tag_value(x):
    """
    Formats a share value for the folder name:
      100.0 -> 100
      25.0  -> 25
      12.5  -> 12p5
    """
    x = float(x)

    if x.is_integer():
        return str(int(x))

    return str(x).replace(".", "p")



# Functions to reconstruct directory of folders in a dinamic way that follows the defined tested cases/configurations
def get_hagrid_output_dirs_for_run(
    run_cfg,
):
    """
    Return deterministic directories for one fully identified run.

    Directory structure:

        <root>/
            EXP_<experiment_id>/
                <run_label>__<run_uid>/
    """

    required = {
        "run_uid",
        "run_label",
        "run_folder_name",
        "experiment_id",
    }

    missing = (
        required
        - set(run_cfg)
    )

    if missing:
        raise KeyError(
            "RUN_CFG is missing run-identity fields: "
            f"{sorted(missing)}"
        )

    experiment_tag = (
        "EXP_"
        + _safe_fs_tag(
            run_cfg[
                "experiment_id"
            ]
        )
    )

    run_folder = str(
        run_cfg[
            "run_folder_name"
        ]
    )

    return {
        "tag": run_folder,
        "experiment_tag": (experiment_tag),
        "run_uid": (run_cfg["run_uid"]),
        "run_label": (run_cfg["run_label"]),
        "xls": os.path.join(hagrid_results_xls_root_folder, experiment_tag, run_folder,),
        "py": os.path.join(hagrid_results_py_root_folder, experiment_tag, run_folder,),
        "pics": os.path.join(hagrid_results_pictures_root_folder, experiment_tag, run_folder,),
        "pflow_py": os.path.join(hagrid_powerflow_py_root_folder, experiment_tag, run_folder,),
        "pf_limits": os.path.join(hagrid_pf_limits_root_folder, experiment_tag, run_folder,),
    }


# Function to update variable for save directories folders
def set_hagrid_output_dirs_for_run(
    run_cfg,
    create_keys=(),
):
    """
    Activate the directories of one fully identified run.
    """

    global optimization_output_folder
    global py_based_output_folder
    global optimization_graph_output_folder
    global pf_py_based_output_folder
    global pf_limits_input_folder

    dirs = (
        get_hagrid_output_dirs_for_run(
            run_cfg
        )
    )

    optimization_output_folder = (
        dirs["xls"]
    )

    py_based_output_folder = (
        dirs["py"]
    )

    optimization_graph_output_folder = (
        dirs["pics"]
    )

    pf_py_based_output_folder = (
        dirs["pflow_py"]
    )

    pf_limits_input_folder = (
        dirs["pf_limits"]
    )

    allowed_keys = {
        "xls",
        "py",
        "pics",
        "pflow_py",
        "pf_limits",
    }

    unknown = (
        set(create_keys)
        - allowed_keys
    )

    if unknown:
        raise KeyError(
            f"Unknown directory keys: "
            f"{sorted(unknown)}"
        )

    for key in create_keys:

        os.makedirs(
            dirs[key],
            exist_ok=True,
        )

    return dirs


# Function to assign tags describing technology availability in file names
def _compact_share_tag_from_shares(shares):
    items = (
        ("PV", "PV"),
        ("BESS", "B"),
        ("BEV", "EV"),
        ("V2G", "V2G"),
        ("DHW", "DHW"),
        ("HP", "HP"),
        ("COOL", "C"),
        ("INDUCTION", "I"),
    )

    return "_".join(
        f"{tag}{_fmt_share_tag_value(shares[tech])}"
        for tech, tag in items
    )


### 4.6.1bis F(x) for manifest and configuration structure

In [ ]:
# Function to return path
def _path_relative_to_output(
    path,
):
    return os.path.relpath(
        os.path.abspath(path),
        os.path.abspath(output_folder),
    )


# Function to resolve paths saved and assign in the manifest(?)
def manifest_path_to_absolute(
    relative_path,
):
    """
    Resolve paths saved relative to the HAGRID output root.

    This makes the project portable if the whole project folder
    is moved to another computer/location.
    """

    if pd.isna(relative_path):
        return None

    relative_path = str(
        relative_path
    )

    if os.path.isabs(
        relative_path
    ):
        return relative_path

    return os.path.normpath(
        os.path.join(
            output_folder,
            relative_path,
        )
    )


# Function to build proper directory output folders
def _has_counts_and_realized_shares(
    has_dict,
):
    out = {}

    has_dict = (
        has_dict
        if isinstance(has_dict, dict)
        else {}
    )

    for tech in MANIFEST_PHYSICAL_TECH_KEYS:

        n_active = len(
            has_dict.get(
                tech,
                set(),
            )
            or set()
        )

        out[
            f"n_{tech}"
        ] = int(
            n_active
        )

        out[
            f"share_{tech}_realized"
        ] = (
            100.0
            * n_active
            / float(num_households)
        )

    return out


# Function to assign the tested configuration in the manifest
def build_run_manifest_row(
    *,
    run_cfg,
    dirs,
    has_potential=None,
    has_run=None,
    status="configured",
    completed_seasons=None,
):
    """Flatten the complete run identity into one manifest row."""

    now = datetime.datetime.now().isoformat(timespec="seconds")
    shares = _shares_from_cfg_or_global(run_cfg)
    potential = _has_counts_and_realized_shares(has_potential)
    active = _has_counts_and_realized_shares(has_run)

    row = {
        "run_uid": run_cfg["run_uid"],
        "run_label": run_cfg["run_label"],
        "run_folder_name": run_cfg["run_folder_name"],
        "experiment_id": run_cfg["experiment_id"],
        "run_revision": run_cfg["run_revision"],
        "electrification_id": run_cfg.get("electrification_id"),
        "scenario_order": int(run_cfg.get("scenario_order", 0)),
        "control_order": int(run_cfg.get("control_order", 0)),
        "status": str(status),
        "completed_seasons": "|".join(completed_seasons or []),
        "updated_at": now,
        "grid_group": HAGRID_GRID_GROUP,
        "lv_archetype": LV_ARCHETYPE,
        "region": selected_region,
        "n_households": int(num_households),
        "mode": run_cfg["mode"],
        "mode_label": run_cfg.get("label", run_cfg["mode"]),
        "case_label": run_cfg.get("case_label"),
        "control_BESS": run_cfg["control"]["BESS"],
        "control_BEV": run_cfg["control"]["BEV"],
        "control_DHW": run_cfg["control"]["DHW"],
    }

    for tech in RUN_SHARE_KEYS:
        row[f"share_{tech}_target"] = float(shares[tech])

    for tech in MANIFEST_PHYSICAL_TECH_KEYS:
        row[f"n_{tech}_physical"] = potential.get(f"n_{tech}", np.nan)
        row[f"share_{tech}_realized"] = potential.get(
            f"share_{tech}_realized", np.nan
        )
        row[f"n_{tech}_active"] = active.get(f"n_{tech}", np.nan)

    # -------
    # Conditional V2G metrics.
    potential_sets = has_potential if isinstance(has_potential, dict) else {}
    active_sets = has_run if isinstance(has_run, dict) else {}

    n_bev_physical = len(potential_sets.get("BEV", set()) or set())
    n_v2g_physical = len(potential_sets.get("V2G", set()) or set())

    n_bev_active = len(active_sets.get("BEV", set()) or set())
    n_v2g_active = len(active_sets.get("V2G", set()) or set())

    row["share_V2G_target_of_BEV"] = float(shares["V2G"])

    row["share_V2G_realized_of_BEV"] = (
        100.0 * n_v2g_physical / n_bev_physical
        if n_bev_physical > 0
        else 0.0
    )

    row["share_V2G_active_of_BEV"] = (
        100.0 * n_v2g_active / n_bev_active
        if n_bev_active > 0
        else 0.0
    )
    # -------

    scenario_meta = run_cfg.get("scenario_meta", {}) or {}

    row["share_BESS_target_of_PV"] = scenario_meta.get(
        "BESS_share_of_PV_target",
        np.nan,
    )

    n_pv_physical = len(
        potential_sets.get("PV", set()) or set()
    )

    n_bess_on_pv_physical = len(
        (potential_sets.get("BESS", set()) or set())
        & (potential_sets.get("PV", set()) or set())
    )

    row["share_BESS_realized_of_PV"] = (
        100.0 * n_bess_on_pv_physical / n_pv_physical
        if n_pv_physical > 0
        else 0.0
    )


    for key in ("xls", "py", "pics", "pflow_py", "pf_limits"):
        row[f"{key}_relpath"] = _path_relative_to_output(dirs[key])


    row["rb_cache_uid"] = (
        RB_CACHE_UID
        if "RB_CACHE_UID" in globals()
        else None
    )

    row["rb_cache_parameter_signature"] = (
        RB_CACHE_PARAMETER_SIGNATURE
        if "RB_CACHE_PARAMETER_SIGNATURE" in globals()
        else None
    )

    row["rb_cache_revision"] = (
        RB_CACHE_RUN_REVISION
        if "RB_CACHE_RUN_REVISION" in globals()
        else None
    )

    row["rb_cache_relpath"] = (
        _path_relative_to_output(
            rb_cache_folder
        )
        if (
            "rb_cache_folder" in globals()
            and rb_cache_folder is not None
        )
        else None
    )

    return row



# Function to load the central run manifest
def load_hagrid_run_manifest():
    """
    Load the central run manifest.
    """

    if not os.path.isfile(
        hagrid_run_manifest_csv
    ):
        return pd.DataFrame()

    return pd.read_csv(
        hagrid_run_manifest_csv
    )


# Function to write human-readable local index
def _write_experiment_index(
    manifest_df,
    experiment_id,
):
    """
    Write a human-readable local index inside the experiment folder.
    """

    experiment_tag = (
        "EXP_"
        + _safe_fs_tag(
            experiment_id
        )
    )

    experiment_dir = os.path.join(
        hagrid_powerflow_py_root_folder,
        experiment_tag,
    )

    os.makedirs(
        experiment_dir,
        exist_ok=True,
    )

    df = manifest_df.loc[
        manifest_df[
            "experiment_id"
        ].astype(str)
        == str(experiment_id)
    ].copy()

    sort_cols = [
        c
        for c in (
            "run_revision",
            "scenario_order",
            "control_order",
            "run_uid",
        )
        if c in df.columns
    ]

    if sort_cols:
        df = df.sort_values(
            sort_cols
        )

    preferred_cols = [
        "run_uid",
        "run_label",
        "run_revision",
        "electrification_id",
        "case_label",
        "mode",
        "status",

        "share_HP_target",
        "share_HP_realized",
        "share_PV_target",
        "share_PV_realized",
        "share_COOL_target",
        "share_COOL_realized",
        "share_BEV_target",
        "share_BEV_realized",
        "share_DHW_target",
        "share_DHW_realized",
        "share_BESS_target",
        "share_BESS_realized",

        "completed_seasons",
        "updated_at",

        "xls_relpath",
        "py_relpath",
        "pics_relpath",
        "pflow_py_relpath",
    ]

    preferred_cols = [
        c
        for c in preferred_cols
        if c in df.columns
    ]

    df_index = df[
        preferred_cols
    ].copy()

    df_index.to_csv(
        os.path.join(
            experiment_dir,
            "INDEX.csv",
        ),
        index=False,
    )

    if WRITE_EXPERIMENT_INDEX_XLSX:

        df_index.to_excel(
            os.path.join(
                experiment_dir,
                "INDEX.xlsx",
            ),
            index=False,
        )


# Function to update or add the run to the manifest file
def upsert_hagrid_run_manifest(
    row,
):
    """
    Add or update exactly one run.

    run_uid is the primary key.

    The CSV is rewritten atomically, but previous run rows are retained.
    """

    os.makedirs(
        hagrid_run_catalog_root_folder,
        exist_ok=True,
    )

    existing = (
        load_hagrid_run_manifest()
    )

    row = dict(row)

    now = (
        datetime.datetime.now()
        .isoformat(
            timespec="seconds"
        )
    )

    row["updated_at"] = now

    if not existing.empty:

        existing[
            "run_uid"
        ] = existing[
            "run_uid"
        ].astype(str)

        old = existing.loc[
            existing["run_uid"]
            == str(
                row["run_uid"]
            )
        ]

        if (
            not old.empty
            and "created_at"
            in old.columns
        ):

            row["created_at"] = (
                old.iloc[0][
                    "created_at"
                ]
            )

        else:
            row["created_at"] = now

        existing = existing.loc[
            existing["run_uid"]
            != str(
                row["run_uid"]
            )
        ].copy()

    else:

        row["created_at"] = now

    updated = pd.concat(
        [
            existing,
            pd.DataFrame(
                [row]
            ),
        ],
        ignore_index=True,
        sort=False,
    )

    sort_cols = [
        c
        for c in (
            "experiment_id",
            "run_revision",
            "scenario_order",
            "control_order",
            "run_uid",
        )
        if c in updated.columns
    ]

    if sort_cols:
        updated = (
            updated.sort_values(
                sort_cols
            )
            .reset_index(
                drop=True
            )
        )

    # Atomic CSV replacement.
    tmp_csv = (
        hagrid_run_manifest_csv
        + ".tmp"
    )

    updated.to_csv(
        tmp_csv,
        index=False,
    )

    os.replace(
        tmp_csv,
        hagrid_run_manifest_csv,
    )

    if WRITE_RUN_MANIFEST_XLSX:

        tmp_xlsx = (
            hagrid_run_manifest_xlsx
            + ".tmp.xlsx"
        )

        updated.to_excel(
            tmp_xlsx,
            index=False,
        )

        os.replace(
            tmp_xlsx,
            hagrid_run_manifest_xlsx,
        )

    _write_experiment_index(
        updated,
        row["experiment_id"],
    )

    return updated


# Function to update the execution status of an existing manifest row (i.e., if the simulation is a "repetition")
def update_hagrid_run_manifest_status(
    run_uid,
    *,
    status,
    completed_seasons=None,
):
    """
    Update only execution status/seasons of an existing manifest row.
    """

    df = (
        load_hagrid_run_manifest()
    )

    if df.empty:
        raise FileNotFoundError(
            "HAGRID run manifest is empty."
        )

    match = df.loc[
        df["run_uid"].astype(str)
        == str(run_uid)
    ]

    if len(match) != 1:
        raise KeyError(
            f"Expected exactly one manifest row "
            f"for run_uid={run_uid}, "
            f"found {len(match)}."
        )

    row = (
        match.iloc[0]
        .to_dict()
    )

    row["status"] = str(
        status
    )

    row[
        "completed_seasons"
    ] = "|".join(
        completed_seasons
        or []
    )

    return upsert_hagrid_run_manifest(
        row
    )


# Function to add the selecting of the manifest run
def select_hagrid_manifest_runs(
    *,
    experiment_id=None,
    run_uids=None,
    run_labels=None,
    filters=None,
    allowed_statuses=(
        "completed",
        "partial",
    ),
):
    """
    Select already-generated runs without reconstructing their
    configuration from filenames or folder names.
    """

    df = (
        load_hagrid_run_manifest()
    )

    if df.empty:
        return df

    out = df.copy()

    if experiment_id is not None:

        out = out.loc[
            out[
                "experiment_id"
            ].astype(str)
            == str(
                experiment_id
            )
        ]

    if run_uids is not None:

        allowed = {
            str(x)
            for x in run_uids
        }

        out = out.loc[
            out[
                "run_uid"
            ].astype(str)
            .isin(
                allowed
            )
        ]

    if run_labels is not None:

        allowed = {
            str(x)
            for x in run_labels
        }

        out = out.loc[
            out[
                "run_label"
            ].astype(str)
            .isin(
                allowed
            )
        ]

    if filters:

        for column, values in filters.items():

            if column not in out.columns:
                raise KeyError(
                    f"Manifest has no column "
                    f"{column!r}."
                )

            if not isinstance(
                values,
                (
                    list,
                    tuple,
                    set,
                ),
            ):
                values = [
                    values
                ]

            out = out.loc[
                out[column].isin(
                    list(values)
                )
            ]

    if allowed_statuses is not None:

        allowed = {
            str(x)
            for x
            in allowed_statuses
        }

        out = out.loc[
            out["status"]
            .astype(str)
            .isin(
                allowed
            )
        ]

    sort_cols = [
        c
        for c in (
            "scenario_order",
            "control_order",
            "run_uid",
        )
        if c in out.columns
    ]

    if sort_cols:
        out = out.sort_values(
            sort_cols
        )

    return out.reset_index(
        drop=True
    )



# Function to activate the specific directory of a saved run
def activate_hagrid_dirs_from_manifest_row(
    manifest_row,
    *,
    create_pics=False,
):
    """
    Activate exactly the directories stored for a saved run.

    No directory reconstruction from MODE/share values occurs.
    """

    global optimization_output_folder
    global py_based_output_folder
    global optimization_graph_output_folder
    global pf_py_based_output_folder
    global pf_limits_input_folder

    optimization_output_folder = (
        manifest_path_to_absolute(
            manifest_row[
                "xls_relpath"
            ]
        )
    )

    py_based_output_folder = (
        manifest_path_to_absolute(
            manifest_row[
                "py_relpath"
            ]
        )
    )

    optimization_graph_output_folder = (
        manifest_path_to_absolute(
            manifest_row[
                "pics_relpath"
            ]
        )
    )

    pf_py_based_output_folder = (
        manifest_path_to_absolute(
            manifest_row[
                "pflow_py_relpath"
            ]
        )
    )

    pf_limits_input_folder = (
        manifest_path_to_absolute(
            manifest_row[
                "pf_limits_relpath"
            ]
        )
    )

    if create_pics:

        os.makedirs(
            optimization_graph_output_folder,
            exist_ok=True,
        )


# Function to resolve human-readable selector
def resolve_unique_manifest_run(
    *,
    experiment_id,
    selector,
    run_revision=None,
    allowed_statuses=("completed",),
):
    """Resolve one human-readable selector to exactly one saved run."""

    filters = dict(selector or {})

    if run_revision is not None:
        filters["run_revision"] = [run_revision]

    selected = select_hagrid_manifest_runs(
        experiment_id=experiment_id,
        filters=filters,
        allowed_statuses=allowed_statuses,
    )

    if len(selected) != 1:
        cols = [
            c for c in (
                "run_uid",
                "run_label",
                "run_revision",
                "electrification_id",
                "case_label",
                "share_HP_target",
                "share_PV_target",
                "share_BEV_target",
            )
            if c in selected.columns
        ]

        candidates = (
            selected[cols].to_string(index=False)
            if not selected.empty
            else "<no matching runs>"
        )

        raise ValueError(
            "The manifest selector must identify exactly one run. "
            f"Found {len(selected)}.\n"
            f"Selector: {selector}\n"
            f"Candidates:\n{candidates}"
        )

    return selected.iloc[0].to_dict()





# -------------------------------------------------------------------
#   Sec 4.6 -- 1bis.2) FUNCTIONS TO RUN IDENTITY / CONFIGURATION HELPERS
# -------------------------------------------------------------------
# Function that retrieves the shares of the different flexibility technologies (it defines what is default share for all technologies)
def _global_shares_dict():
    """
    Return the complete default physical penetration scenario.
    """

    return {
        "PV":        float(PV_share),
        "BESS":      float(BESS_share),
        "BEV":       float(EV_share),
        "DHW":       float(ESWH_share),
        "HP":        float(heat_pump_share),
        "COOL":      float(cooler_share),
        "INDUCTION": float(induction_share),
        "V2G":       float(V2G_share),
    }



# Function that defines the shares of the specific configuration, if the global ones are not used
def _shares_from_cfg_or_global(cfg=None):
    """
    Return the complete target physical shares of one run.

    Technologies not explicitly specified by the configuration
    inherit the global/default share.
    """

    out = _global_shares_dict()

    if cfg is None:
        return out

    cfg_shares = cfg.get("shares", {}) or {}

    for tech in RUN_SHARE_KEYS:
        if tech in cfg_shares:
            out[tech] = float(cfg_shares[tech])

    return out


# Function to select dictionary in the tested configuration
def _run_identity_parameter_dict():
    out = {}

    for name in RUN_IDENTITY_PARAMETER_NAMES:

        if name not in globals():
            continue

        value = globals()[name]

        if isinstance(
            value,
            (np.integer,),
        ):
            value = int(value)

        elif isinstance(
            value,
            (np.floating,),
        ):
            value = float(value)

        out[name] = value

    return out


# Function to assign unique definition to each run
def _canonical_run_identity_payload(
    run_cfg,
):
    """
    Information that uniquely defines one simulated run.

    RUN_REVISION is intentionally part of the identity.
    """

    shares = _shares_from_cfg_or_global(
        run_cfg
    )

    payload = {
        "identity_schema": (
            RUN_IDENTITY_SCHEMA
        ),

        "experiment_id": str(
            run_cfg.get(
                "experiment_id",
                EXPERIMENT_ID,
            )
        ),

        "run_revision": str(
            run_cfg.get(
                "run_revision",
                RUN_REVISION,
            )
        ),

        "grid_group": str(
            HAGRID_GRID_GROUP
        ),

        "lv_archetype": str(
            LV_ARCHETYPE
        ),

        "region": str(
            selected_region
        ),

        "n_households": int(
            num_households
        ),

        "mode": str(
            run_cfg["mode"]
        ),

        "control": {
            str(k): str(v)
            for k, v
            in sorted(
                run_cfg["control"].items()
            )
        },

        "shares_target": {
            tech: float(
                shares[tech]
            )
            for tech
            in RUN_SHARE_KEYS
        },

        "main_parameters": (
            _run_identity_parameter_dict()
        ),
    }

    return payload


# Function to assign a specific unique ID to each run
def _run_uid_from_config(
    run_cfg,
):
    """
    Deterministic short UID generated from the complete canonical
    run identity.
    """

    payload = (
        _canonical_run_identity_payload(
            run_cfg
        )
    )

    encoded = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")

    digest = hashlib.sha256(
        encoded
    ).hexdigest()

    return (
        "R_"
        + digest[
            :int(RUN_UID_HASH_LENGTH)
        ]
    )


def _canonical_rb_cache_context_payload():
    """
    Physical/data context uniquely identifying an RB cache.

    RUN_REVISION is intentionally NOT included here:
    it is represented explicitly by the REV_<RUN_REVISION>
    directory.

    The same RB context UID may therefore appear under different
    revisions, while the full path remains unique.
    """

    global_shares = _global_shares_dict()

    return {
        "identity_schema":
            RB_CACHE_IDENTITY_SCHEMA,

        "grid_group":
            str(HAGRID_GRID_GROUP),

        "lv_archetype":
            str(LV_ARCHETYPE),

        "region":
            str(selected_region),

        "n_households":
            int(num_households),

        "shares_global": {
            str(k): float(v)
            for k, v
            in sorted(global_shares.items())
        },

        "PV_power_kwp":
            float(PV_power_kwp),

        "BESS_power_kw":
            float(BESS_power_kw),

        "BESS_capacity_kwh":
            float(BESS_capacity_kwh),

        "random_seed_has_tech":
            int(RANDOM_SEED_HAS_TECH),

        "random_seed_share_subsets":
            int(RANDOM_SEED_SHARE_SUBSETS),

        "timestep_minutes":
            int(timestep_minutes),

        "number_timesteps":
            int(number_timesteps),
    }


# Function to assign specific UID for folders to RB cached profiles
def _rb_cache_uid():
    """
    Deterministic short UID identifying the physical/data context
    of the RB reference profiles.
    """

    payload = (
        _canonical_rb_cache_context_payload()
    )

    encoded = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")

    digest = hashlib.sha256(
        encoded
    ).hexdigest()

    return (
        "RB_"
        + digest[
            :int(RB_CACHE_UID_HASH_LENGTH)
        ]
    )


# Function to check snapshot of RB parameters
def _rb_cache_parameter_snapshot():
    """
    Parameters affecting the construction of RB profiles.

    Stored in rb_meta.json for traceability and consistency checks.
    """

    parameter_names = (
        # BESS
        "eta_BESS",
        "initial_SoC_BESS",
        "BESS_loss_rate",
        "BESS_power_kw",
        "BESS_capacity_kwh",

        # BEV
        "EV_charge_discharge_efficiency",
        "eta_charger",
        "initial_SoC_BEV",
        "BEV_loss_rate",
        "min_storage_level_BEV",
        "home_charging_power_kw",
        "default_ev_battery_capacity_kwh",
        "SOC_BEV_TARGET_frac",

        "P_outdoor_charger",
        "SOC_BEV_OUTDOOR_TRIGGER_frac",
        "SOC_BEV_OUTDOOR_TARGET_frac",

        "ENABLE_BEV_OUTDOOR_RB",
        "ENABLE_BEV_PLANNED_FORESIGHT_RB",
        "BEV_PLANNED_TRIP_LOOKAHEAD_H",
        "BEV_LONG_TRIP_CONSUMPTION_frac",
        "BEV_VERY_LONG_TRIP_CONSUMPTION_frac",
        "SOC_BEV_LONG_TRIP_TARGET_frac",
        "SOC_BEV_VERY_LONG_TRIP_TARGET_frac",
        "min_dwell_steps_home",

        # DHW
        "initial_SoC_DHW",
        "eta_DHW",
        "DHW_loss_rate",
        "dhw_power_kw",
        "cap_kWh_DHW",
        "P_boost_DHW",
        "SOC_HYST_DEFAULT",
        "ENABLE_DHW_BOOST_RB",

        "DHW_INITIAL_SOC_DISTRIBUTED",
        "DHW_INITIAL_SOC_SEED",
        "DHW_INITIAL_SOC_MEAN_BY_TYPE",
        "DHW_INITIAL_SOC_HALF_RANGE_BY_TYPE",


        # temporal resolution
        "timestep_minutes",
        "number_timesteps",
    )

    snapshot = {
        name: _json_safe(globals()[name])
        for name in parameter_names
        if name in globals()
    }

    if "dhw_specs" in globals():
        snapshot["dhw_specs"] = (
            _json_safe(dhw_specs)
        )

    return snapshot


# Function to define the cache for parameter signature
def _rb_cache_parameter_signature():

    snapshot = (
        _rb_cache_parameter_snapshot()
    )

    encoded = json.dumps(
        snapshot,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()[:12]




# Function to define compact tag for RB-profile cache
def _short_hash_tag(
    value,
    prefix,
    n=8,
):
    """
    Compact deterministic filesystem tag.
    Human-readable information remains in metadata;
    directories stay short.
    """

    encoded = str(value).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        encoded
    ).hexdigest()

    return (
        str(prefix)
        + digest[:int(n)]
    )


def build_rb_cache_paths(
    *,
    revision,
):
    """
    Single source of truth for RB-cache paths.

    IMPORTANT:
    the same function is used both when BUILDING
    and when LOADING the RB cache.
    """

    revision = str(revision)

    rb_uid = _rb_cache_uid()

    rb_parameter_signature = (
        _rb_cache_parameter_signature()
    )

    # Short tags avoid reproducing long human-readable
    # EXPERIMENT_ID and RUN_REVISION strings in paths.
    experiment_uid = _short_hash_tag(
        EXPERIMENT_ID,
        prefix="E_",
        n=6,
    )

    revision_uid = _short_hash_tag(
        revision,
        prefix="V_",
        n=8,
    )

    root = os.path.abspath(
        os.path.join(
            output_folder,
            "04_HGD_rb_cache",
            HAGRID_GRID_GROUP,
        )
    )

    experiment_folder = os.path.join(
        root,
        experiment_uid,
    )

    revision_folder = os.path.join(
        experiment_folder,
        revision_uid,
    )

    # Include BOTH:
    # - physical/data context UID
    # - RB parameter signature
    #
    # so changing hysteresis, initial SOC, DHW specs, etc.
    # automatically produces a different cache folder.
    cache_folder = os.path.join(
        revision_folder,
        (
            f"{rb_uid}_"
            f"{rb_parameter_signature}"
        ),
    )

    return {
        "root":
            root,

        "experiment_folder":
            experiment_folder,

        "revision_folder":
            revision_folder,

        "cache_folder":
            cache_folder,

        "pickle":
            os.path.join(
                cache_folder,
                "rb_profiles.pkl.gz",
            ),

        "meta":
            os.path.join(
                cache_folder,
                "rb_meta.json",
            ),

        "audit":
            os.path.join(
                cache_folder,
                "audit",
            ),

        "rb_cache_uid":
            rb_uid,

        "rb_parameter_signature":
            rb_parameter_signature,

        "experiment_uid":
            experiment_uid,

        "revision_uid":
            revision_uid,
    }






# Function to define compressed folder name
def _fmt_folder_share(
    value,
):
    """
    Compact percentage representation for readable folder labels.

    20   -> 020
    100  -> 100
    7.5  -> 7p5
    """

    value = float(value)

    if value.is_integer():
        return f"{int(value):03d}"

    return (
        f"{value:g}"
        .replace(".", "p")
    )


# Function to assign/link a human-readable name to the run folder
def _run_folder_label(
    run_cfg,
):
    """
    Human-readable part of the run folder.

    Only shares varying inside the current experiment are shown.
    """

    shares = _shares_from_cfg_or_global(
        run_cfg
    )

    tokens = []

    for tech in EXPERIMENT_VARYING_SHARES:

        if tech not in RUN_SHARE_KEYS:
            raise KeyError(
                f"Unknown experiment share: "
                f"{tech}"
            )

        prefix = (
            RUN_SHARE_FOLDER_TOKEN[
                tech
            ]
        )

        tokens.append(
            f"{prefix}"
            f"{_fmt_folder_share(shares[tech])}"
        )

    control_tag = str(
        run_cfg.get(
            "control_short_tag",
            run_cfg.get(
                "case_label",
                "RUN",
            ),
        )
    )

    control_tag = _safe_fs_tag(
        control_tag
    )

    if tokens:
        return (
            "_".join(tokens)
            + "__"
            + control_tag
        )

    return control_tag


# Function to return a copy of the run identity information
def attach_run_identity(
    run_cfg,
):
    """
    Return a copy of RUN_CFG enriched with experiment/run identity.
    """

    out = dict(run_cfg)

    out["experiment_id"] = str(
        out.get(
            "experiment_id",
            EXPERIMENT_ID,
        )
    )

    out["run_revision"] = str(
        out.get(
            "run_revision",
            RUN_REVISION,
        )
    )

    out["run_uid"] = (
        _run_uid_from_config(out)
    )

    out["run_label"] = (
        _run_folder_label(out)
    )

    out["run_folder_name"] = (
        f"{out['run_label']}"
        f"__{out['run_uid']}"
    )

    return out



#-------------------------------------------------------------------
#       Sec 4.6 -- 1bis.3) FUNCTIONS FOR DEFINING CONFIGURATIONS TO BE TESTED for flexibility technologies
#-------------------------------------------------------------------
# Function to normalize the configuration definitions for the flexibility technology options
def _norm_control_value(x):
    """
    Normalizes technology control values.

    Allowed values:
      - OPT: technology available and optimized
      - RB : technology available with a conventional/rule-based profile
      - NO : technology absent in the run, even if the global share is > 0
    """
    s = str(x).upper().strip()

    if s in {"OPT", "SMART", "OPTIMIZED", "OTTIMIZZATO"}:
        return "OPT"

    if s in {"RB", "RB_FIXED", "RULE_BASED", "RULE-BASED", "CONVENTIONAL"}:
        return "RB"

    if s in {"NO", "NONE", "OFF", "ABSENT", "NOT_PRESENT", "0", "FALSE"}:
        return "NO"

    raise ValueError(f"Unrecognized technology control value: {x}")



# Function to define which configuration is associated with each flexibility technology (OPT-RB-NONE)
def _normalize_run_control_config(cfg):
    """Normalize one OPT/RB/NO control configuration."""

    control = cfg.get("control", {}) or {}

    out_control = {
        "BESS": _norm_control_value(control.get("BESS", "OPT")),
        "BEV": _norm_control_value(control.get("BEV", "OPT")),
        "DHW": _norm_control_value(control.get("DHW", "OPT")),
    }

    mode = str(cfg.get("mode", "")).strip()
    if not mode:
        mode = (
            f"MIX_B{out_control['BESS']}_"
            f"V{out_control['BEV']}_"
            f"H{out_control['DHW']}"
        )

    mode = _safe_fs_tag(mode)
    case_label = cfg.get("case_label", cfg.get("label", mode))

    return {
        "mode": mode,
        "label": cfg.get("label", mode),
        "case_label": case_label,
        "control": out_control,
        "control_short_tag": CONTROL_SHORT_TAG_BY_CASE_LABEL.get(
            case_label,
            _safe_fs_tag(case_label),
        ),
    }


# Function for cases with at least one technology in RB mode
def _run_uses_any_rb(control):
    """
    True if at least one available technology uses an RB profile.
    Technologies set to NO do not require RB profiles.
    """
    return any(str(v).upper() == "RB" for v in control.values())


def _tech_control_mode(control, tech):
    return _norm_control_value(control.get(tech, "OPT"))


# Function to check whether the selected technology is in OPT mode in the observed configuration
def _tech_is_opt(control, tech):
    return _tech_control_mode(control, tech) == "OPT"


# Function to check whether the selected technology is in RB mode in the observed configuration
def _tech_is_rb(control, tech):
    return _tech_control_mode(control, tech) == "RB"


# Function to check whether the selected technology is NOT available in the observed configuration
def _tech_is_no(control, tech):
    return _tech_control_mode(control, tech) == "NO"


### 4.6.2 F(x) Utility columns, ID households, dataframe

In [ ]:
# -------------------------------------------------------------------
#       Sec 4.6 -- 2.0) FUNCTIONS for CONVERTING and NORMALIZING column names
#-------------------------------------------------------------------
# Function that extracts the string from the "results naming convention" and converts it to the format required by the chart print/plot section
def _col_to_arrow_string(col) -> str:
    """
    Extracts a string like 'A -> B' from a column label:
    - string columns
    - MultiIndex: (group, 'A -> B') or even more levels
    Returns a representative string; if '->' is not found, returns a "flat" string.
    """
    if isinstance(col, str):
        return col

    if isinstance(col, tuple):
        # try to find a level containing '->'
        for x in reversed(col):
            if isinstance(x, str) and "->" in x:
                return x
        # fallback: flatten
        return " | ".join(str(x) for x in col if x is not None)

    return str(col)


# Function to flatten MultiIndex result-column tuples into a flat string and extract the column name
def _col_to_text(col) -> str:
    """Transforms a MultiIndex/nested-tuple column into a 'flat' string."""
    parts = []

    def _flatten(x):
        if x is None:
            return
        if isinstance(x, tuple):
            for y in x:
                _flatten(y)
        else:
            parts.append(str(x))

    _flatten(col)
    return " | ".join(parts)




# Function to convert a MultiIndex column into a single string
def col_to_str(col):
    """Transforms a column, either MultiIndex or string, into a single representative string."""
    if isinstance(col, tuple):
        return '|'.join([str(x) for x in col])
    return str(col)



# Function to correctly search column names in a dataframe
def _col_to_searchable_string(col):
    """
    Makes a MultiIndex/string column searchable with simple substrings.
    """
    if isinstance(col, tuple):
        return " | ".join(str(x) for x in col if x is not None)
    return str(col)


# Function that rebuilds the "result-column naming convention" structure required by the chart print/plot section
def convert_busseq_to_flowtuple_df(df):
    """
    Converts df columns:
      - 'A -> B'                     -> (A, B, 'flow')
      - MultiIndex (..., 'A -> B')   -> (A, B, 'flow')
      - ((A,B),'flow')               -> (A, B, 'flow')
      - (A,B,'flow')                 -> unchanged
    If a column cannot be parsed, it is left unchanged.
    """
    if df is None or not hasattr(df, "columns"):
        return df

    cols_new = []
    changed = False

    for col in df.columns:

        # --- already in the target format (A,B,'flow')
        if isinstance(col, tuple) and len(col) == 3 and str(col[2]).lower() == "flow":
            cols_new.append((str(col[0]), str(col[1]), "flow"))
            continue

        # --- legacy: ((A,B),'flow')
        if (
            isinstance(col, tuple) and len(col) == 2
            and isinstance(col[0], tuple) and len(col[0]) >= 2
            and str(col[1]).lower() == "flow"
        ):
            cols_new.append((str(col[0][0]), str(col[0][1]), "flow"))
            changed = True
            continue

        # --- bus-sequences: 'A -> B', also inside MultiIndex
        s = _col_to_arrow_string(col)
        if "->" in s:
            left, right = s.split("->", 1)
            cols_new.append((left.strip(), right.strip(), "flow"))
            changed = True
        else:
            cols_new.append(col)

    if changed:
        out = df.copy()
        out.columns = cols_new
        return out

    return df






# -------------------------------------------------------------------
#       Sec 4.6 -- 2.2) FUNCTIONS for HOUSEHOLD/CASE NAME MAPPING
#-------------------------------------------------------------------
# Function to map columns such as "Nucleo_1"/"Nucleo 10" -> "c_1"/"c_10"
def map_colname_to_c(colname, prefix_out='c'):
    """
    Finds the first number in the string and returns prefix_out_<number>.
    If no number is found, returns None.
    """
    if colname is None:
        return None
    m = re.search(r'(\d+)', str(colname))
    if m:
        return f"{prefix_out}_{int(m.group(1))}"
    return None

# Function to rename the columns of a dataframe from Nucleo_i to c_i; works ONLY for dataframes with aggregated households, not individual members
def rename_axis_to_c(df, axis='columns', prefix_out='c'):
    """
    Renames columns (axis='columns') or index (axis='index') by replacing
    all labels containing a number with prefix_out_<number>.
    Returns df_renamed, mapping_dict.
    """
    if axis == 'columns':
        labels = df.columns
    else:
        labels = df.index

    mapping = {}
    for lab in labels:
        new = map_colname_to_c(lab, prefix_out=prefix_out)
        if new:
            mapping[lab] = new

    if axis == 'columns':
        return df.rename(columns=mapping), mapping
    else:
        return df.rename(index=mapping), mapping
    


# Function to rename and aggregate dataframe columns belonging to the same household, summing individual member loads into one column
def rename_and_aggregate_households(
    df,
    prefix_out='c',
):
    """
    Starting from English columns such as:

        Household_1_Member_1_winter_single
        Household_1_Member_2_winter_single

    creates aggregated household columns:

        c_1, c_2, ...

    by summing all members of the same household.

    For backward compatibility, the legacy Italian prefix
    Nucleo_i is accepted as well.

    Returns:
        df_agg:
            DataFrame with columns c_1, c_2, ...

        mapping:
            dictionary {original_column_name: new_column}
    """
    original_cols = list(df.columns)
    new_name_by_original = {}

    for col in original_cols:
        # Accepted input prefixes:
        #   Household_1_Member_1_...
        #   Nucleo_1_Membro_1_...  (legacy)
        match = re.search(
            r'(?:Household|Nucleo)_(\d+)',
            str(col),
            flags=re.IGNORECASE,
        )

        if not match:
            raise ValueError(
                "No household identifier found in column name "
                f"'{col}'. Expected 'Household_i_...' "
                "or legacy 'Nucleo_i_...'."
            )

        household_number = int(match.group(1))

        new_name_by_original[col] = (
            f"{prefix_out}_{household_number}"
        )

    # Multiple member columns may become the same c_i.
    df_tmp = df.rename(
        columns=new_name_by_original
    )

    # Aggregate all members belonging to the same household.
    df_agg = (
        df_tmp
        .T
        .groupby(level=0)
        .sum()
        .T
    )

    return df_agg, new_name_by_original



CASE_RE = re.compile(r"\bc_\d+\b", flags=re.IGNORECASE)

# Function to identify the ID of a household with a given technology
def extract_case_id(col) -> str | None:
    """
    Returns 'c_i' in lowercase.
    Priority:
    1) if MultiIndex and level 0 is 'c_1', use it
    2) otherwise apply regex on the flattened string
    """
    # typical case: col = ('c_1', (('c_1_PV','c_1_bus_el'),'flow'))
    if isinstance(col, tuple) and len(col) >= 1:
        lvl0 = col[0]
        if isinstance(lvl0, str) and CASE_RE.search(lvl0):
            return CASE_RE.search(lvl0).group(0).lower()

    s = _col_to_text(col)
    m = CASE_RE.search(s)
    return m.group(0).lower() if m else None


# Function to retrieve the name of the observed household/customer (c_i)
def extract_house_ids(col_strings):
    """Extracts household identifiers such as c_1, c_2, ... from column names."""
    ids = set()
    for s in col_strings:
        found = re.findall(r'c_\d+', s)
        for f in found:
            ids.add(f)
    return sorted(list(ids), key=lambda x: int(x.split('_')[1]))


# Function to correctly extract information for saving costs, e.g. outdoor charger in the RB case
def extract_house_id_from_labels(from_label, to_label):
    for lab in [from_label, to_label]:
        m = re.match(r"^(c_\d+)_", str(lab))
        if m:
            return m.group(1)
    return None


# Function to convert names from c_# -> Nucleo_#
def to_household_name(c):
    """
    Normalizes the household name:
    - 'c_1', 'c_01', 'Nucleo_1', ' household 1 '  -> 'Nucleo_1'
    - otherwise returns the string as-is.
    """
    s = str(c).strip()
    m = re.search(r"(\d+)", s)
    if m:
        return f"Nucleo_{int(m.group(1))}"
    return s
# -------------------------------------------------------------------



# -------------------------------------------------------------------
#       Sec 4.6 -- 2.2) NUMERIC CONVERSIONS FOR TIME SERIES
#-------------------------------------------------------------------
# Function to convert input time series to power, both PV and base+additional electric loads, expressed as energy in kWh/10'
def convert_kwh10_to_kw(df):
    """
    Converts a DataFrame containing energy per 10-minute timestep (kWh/10')
    into average power (kW), multiplying all numeric columns by 6.
    The datetime index is not modified.
    """
    numeric_cols = df.select_dtypes(include=["number"]).columns
    df.loc[:, numeric_cols] = df[numeric_cols] * FACTOR_KWH_10MIN_TO_KW
    return df

# Function to normalize time series in kW with respect to the maximum value recorded in each series
def normalize_ts_dict(ts_dict_by_case):
    """
    ts_dict_by_case: dict {c: Series in kW}
    Returns:
      - fix_dict:   {c: normalized Series 0–1}
      - p_nom_dict: {c: float nominal power (kW)}
    """
    fix_dict = {}
    p_nom_dict = {}

    for c, s in ts_dict_by_case.items():
        s = s.astype(float)
        max_val = s.max()

        # handle zero or NaN series
        if pd.isna(max_val) or max_val <= 0:
            p_nom = 0.001   # small dummy value to avoid nominal_value=0
            fix = s * 0.0
        else:
            p_nom = float(max_val)
            fix = s / p_nom

        p_nom_dict[c] = p_nom
        fix_dict[c] = fix

    return fix_dict, p_nom_dict

### 4.6.3 F(x) Mixed setups and assignment of technologies

In [ ]:
# -------------------------------------------------------------------
#       Sec 4.6 -- 3.0) FUNCTIONS FOR CONFIGURATIONS TO BE PLOTTED
#-------------------------------------------------------------------
# Function to define which configurations should be printed/plotted in the charts
def build_plot_run_configs_from_manifest():
    """Return exact saved runs requested for Sec. 4.12."""

    filters = dict(PLOT_MANIFEST_FILTERS or {})
    filters.setdefault("run_revision", [PLOT_RUN_REVISION])

    selected = select_hagrid_manifest_runs(
        experiment_id=PLOT_EXPERIMENT_ID,
        run_uids=PLOT_ONLY_RUN_UIDS,
        run_labels=PLOT_ONLY_RUN_LABELS,
        filters=filters,
        allowed_statuses=("completed",),
    )

    return selected.to_dict(orient="records")



#-------------------------------------------------------------------
#       Sec 4.6 -- 3.1) FUNCTIONS FOR RECONSTRUCTING IMPORT DIRECTORIES in the different hybrid configurations
#-------------------------------------------------------------------

# Function to update the aliases of technology assignments in the different configurations
def _set_has_aliases_from_has_dict(has_dict):
    global has
    global has_PV, has_BESS, has_BEV, has_V2G, has_el_boiler
    global has_HP, has_cooler, has_induction

    has = has_dict

    has_PV = set(has.get("PV", set()) or set())
    has_BESS = set(has.get("BESS", set()) or set())
    has_BEV = set(has.get("BEV", set()) or set())
    has_V2G = set(has.get("V2G", set()) or set())
    has_el_boiler = set(has.get("DHW", set()) or set())
    has_HP = set(has.get("HP", set()) or set())
    has_cooler = set(has.get("COOL", set()) or set())
    has_induction = set(has.get("INDUCTION", set()) or set())


def _has_global_is_valid(has_candidate):
    """
    Minimum check that has is already available in memory.
    """
    if not isinstance(has_candidate, dict):
        return False

    required_keys = ['PV', "BESS", "BEV", "DHW", "HP", "COOL", "INDUCTION"]

    return all(k in has_candidate for k in required_keys)



# -------------------------------------------------------------------
#       Sec 4.6 -- 3.2) ACTIVE AND POTENTIAL TECHNOLOGY ENDOWMENTS FOR EACH RUN
#-------------------------------------------------------------------
# Function to show which technologies/storage systems are currently available in the configuration being observed
def _active_has_for_run(has_global, control):
    """
    Return technology sets actually active in the run.
    """

    if has_global is None:
        raise ValueError(
            "has_global is None."
        )

    out = {}

    # Non-controlled physical technologies.
    for tech in ("PV", "HP", "COOL", "INDUCTION"):
        out[tech] = set(
            has_global.get(tech, set()) or set()
        )

    # Controlled flexibility technologies.
    for tech in ("BESS", "BEV", "DHW"):
        if _tech_is_no(control, tech):
            out[tech] = set()
        else:
            out[tech] = set(
                has_global.get(tech, set()) or set()
            )

    # V2G exists only for BEVs active in this run.
    out["V2G"] = (
        set(has_global.get("V2G", set()) or set())
        & out["BEV"]
    )

    return out


# Function to sort household IDs
def _sort_house_ids(ids):
    """
    Sorts c_1, c_2, ... numerically.
    """
    return sorted(
        list(ids),
        key=lambda x: int(str(x).split("_")[1])
    )


# Function to group households consistently with the technology shares assigned to the configuration
def _subset_house_set_by_share(house_set, target_share, n_total, tech):
    """
    Deterministic nested subset for one technology.

    Increasing penetration keeps previously adopted households and
    adds new ones according to a reproducible pseudo-random ranking.
    """

    target_n = int(round(n_total * float(target_share) / 100.0))
    pool = list(set(house_set))

    if target_n > len(pool):
        raise ValueError(
            f"{tech}: requested {target_n} households, "
            f"but the maximum pool contains only {len(pool)}."
        )

    def ranking_key(house):
        key = f"{RANDOM_SEED_SHARE_SUBSETS}|{tech}|{house}"
        return hashlib.sha256(key.encode("utf-8")).hexdigest()

    ordered = sorted(pool, key=ranking_key)
    return set(ordered[:target_n])


# Function that defines the effective share of a technology for the observed configuration
def _potential_has_for_config_shares(has_global, cfg_shares):
    """
    Build the physical technology assignment of one run.

    Dependency rules:
      - V2G is always a subset of BEV.
      - BESS is preferentially a subset of PV.
      - BESS outside PV is allowed only up to standalone_ratio.

    Share convention:
      - PV/BESS/BEV/DHW/HP/COOL/INDUCTION:
          % of all households.
      - V2G:
          % of households WITH BEV.
    """

    if has_global is None:
        raise ValueError("has_global is required.")

    out = {}

    # --------------------------------------------------------
    # 1) Independent technologies
    # --------------------------------------------------------

    independent_techs = (
        "PV",
        "BEV",
        "DHW",
        "HP",
        "COOL",
        "INDUCTION",
    )

    for tech in independent_techs:
        out[tech] = _subset_house_set_by_share(
            house_set=has_global.get(tech, set()),
            target_share=cfg_shares[tech],
            n_total=num_households,
            tech=tech,
        )

    # --------------------------------------------------------
    # 2) BESS: coupled to PV
    # --------------------------------------------------------

    n_bess_target = int(
        round(
            num_households
            * float(cfg_shares["BESS"])
            / 100.0
        )
    )

    pv_houses = set(out["PV"])
    bess_global_pool = set(
        has_global.get("BESS", set()) or set()
    )

    # Potential BESS households that currently also have PV.
    bess_on_pv_pool = pv_houses & bess_global_pool

    # Potential standalone BESS households.
    bess_standalone_pool = (
        bess_global_pool - pv_houses
    )

    # Minimum number of standalone BESS units necessary to
    # achieve the requested total BESS penetration.
    n_standalone_required = max(
        0,
        n_bess_target - len(bess_on_pv_pool),
    )

    # At most this fraction of installed BESS can be standalone.
    n_standalone_allowed = int(
        round(
            n_bess_target
            * float(standalone_ratio)
        )
    )

    if n_standalone_required > n_standalone_allowed:
        raise ValueError(
            "Incompatible PV/BESS penetration scenario.\n"
            f"Requested PV share   = {cfg_shares['PV']}%\n"
            f"Requested BESS share = {cfg_shares['BESS']}%\n"
            f"Target BESS units    = {n_bess_target}\n"
            f"PV-coupled BESS pool = {len(bess_on_pv_pool)}\n"
            f"Standalone required  = {n_standalone_required}\n"
            f"Standalone allowed   = {n_standalone_allowed}\n"
            "Increase PV penetration, reduce BESS penetration, "
            "or explicitly increase standalone_ratio."
        )

    # Stable ordering.
    bess_on_pv_ordered = sorted(
        bess_on_pv_pool,
        key=lambda house: hashlib.sha256(
            f"{RANDOM_SEED_SHARE_SUBSETS}|PV|{house}".encode("utf-8")
        ).hexdigest(),
    )

    bess_standalone_ordered = sorted(
        bess_standalone_pool,
        key=lambda house: hashlib.sha256(
            f"{RANDOM_SEED_SHARE_SUBSETS}|BESS_STANDALONE|{house}".encode(
                "utf-8"
            )
        ).hexdigest(),
    )

    n_bess_on_pv = (
        n_bess_target
        - n_standalone_required
    )

    out["BESS"] = set(
        bess_on_pv_ordered[:n_bess_on_pv]
    )

    if n_standalone_required > 0:
        out["BESS"].update(
            bess_standalone_ordered[
                :n_standalone_required
            ]
        )

    # --------------------------------------------------------
    # 3) V2G: conditional share of the current BEV population
    # --------------------------------------------------------

    bev_houses = set(out["BEV"])

    n_v2g_target = int(
        round(
            len(bev_houses)
            * float(cfg_shares["V2G"])
            / 100.0
        )
    )

    # Use the same BEV adoption priority so that an increase
    # in penetration tends to preserve previously selected V2G users.
    bev_ordered = sorted(
        bev_houses,
        key=lambda house: hashlib.sha256(
            f"{RANDOM_SEED_SHARE_SUBSETS}|BEV|{house}".encode("utf-8")
        ).hexdigest(),
    )

    out["V2G"] = set(
        bev_ordered[:n_v2g_target]
    )

    # --------------------------------------------------------
    # 4) Consistency checks
    # --------------------------------------------------------

    if not out["V2G"].issubset(out["BEV"]):
        raise RuntimeError(
            "Internal inconsistency: V2G must be a subset of BEV."
        )

    n_bess_without_pv = len(
        out["BESS"] - out["PV"]
    )

    if n_bess_without_pv > n_standalone_allowed:
        raise RuntimeError(
            "Internal inconsistency: too many standalone BESS units."
        )

    return out




# -------------------------------------------------------------------
#       Sec 4.6 -- 3.3) ELECTRIFICATION SCENARIO BUILDERS
#-------------------------------------------------------------------


# Function to define the set of configuration to be tested (e.g., increasing share of technology)
def make_share_sweep(
    tech,
    values,
    prefix=None,
):
    """
    One-dimensional penetration sweep.

    Example:
        HP = 10, 20, ..., 100
    """

    prefix = prefix or tech

    return [
        {
            "scenario_id": (
                f"{prefix}{int(value):03d}"
            ),
            "shares": {
                tech: float(value),
            },
        }
        for value in values
    ]


# Function to create combination of different penetration share for combined configurations
def make_share_grid(axes):
    """
    Cartesian product of independent household penetration shares.

    Example:
        HP x PV.
    """

    from itertools import product

    techs = list(axes.keys())
    value_sets = [
        axes[tech]
        for tech in techs
    ]

    scenarios = []

    for values in product(*value_sets):

        shares = dict(
            zip(techs, values)
        )

        scenario_id = "_".join(
            f"{tech}{int(value):03d}"
            for tech, value
            in shares.items()
        )

        scenarios.append({
            "scenario_id": scenario_id,
            "shares": {
                tech: float(value)
                for tech, value
                in shares.items()
            },
        })

    return scenarios



# Function to define share of EV with V2G
def make_ev_v2g_grid(
    ev_values,
    v2g_values,
):
    """
    EV = % of all households with BEV.
    V2G = % of BEV households with V2G.
    """

    scenarios = []

    for ev_share in ev_values:
        for v2g_share in v2g_values:

            scenarios.append({
                "scenario_id": (
                    f"EV{int(ev_share):03d}_"
                    f"V2G{int(v2g_share):03d}"
                ),
                "shares": {
                    "BEV": float(ev_share),
                    "V2G": float(v2g_share),
                },
            })

    return scenarios


# Function to create physically coupled PV/BESS scenarios
def make_pv_bess_coupled_grid(
    pv_values,
    bess_on_pv_values,
):
    """
    Build physically coupled PV/BESS scenarios.

    pv_share:
        % of all households with PV.

    bess_on_pv:
        % of PV households also equipped with BESS.

    BESS_share stored in the run remains the equivalent
    penetration over all households.
    """

    scenarios = []

    for pv_share in pv_values:
        for bess_on_pv in bess_on_pv_values:

            bess_share_total = (
                float(pv_share)
                * float(bess_on_pv)
                / 100.0
            )

            scenarios.append({
                "scenario_id": (
                    f"PV{int(pv_share):03d}_"
                    f"BPV{int(bess_on_pv):03d}"
                ),

                "shares": {
                    "PV": float(pv_share),
                    "BESS": bess_share_total,
                },

                # Extra metadata, not directly used by the solver.
                "scenario_meta": {
                    "BESS_share_of_PV_target": (
                        float(bess_on_pv)
                    ),
                },
            })

    return scenarios



# Function to build the run catalog as a combination of electrification scenario and control scenario
def build_experiment_run_configs(
    control_configs,
    electrification_configs,
):
    """
    Build the complete run catalog as:

        electrification scenario
        x
        control scenario

    Each returned dictionary already contains its deterministic
    run identity.
    """

    selected_electrification = [
        cfg
        for cfg in electrification_configs
        if cfg["scenario_id"]
        in ACTIVE_ELECTRIFICATION_SCENARIO_IDS
    ]

    unknown_ids = (
        set(
            ACTIVE_ELECTRIFICATION_SCENARIO_IDS
        )
        - {
            cfg["scenario_id"]
            for cfg
            in electrification_configs
        }
    )

    if unknown_ids:
        raise KeyError(
            "Unknown electrification scenario IDs: "
            f"{sorted(unknown_ids)}"
        )

    runs = []

    for scenario_order, e_cfg in enumerate(
        selected_electrification,
        start=1,
    ):

        # Start from global shares.
        scenario_shares = (
            _global_shares_dict()
        )

        # Apply the physical scenario.
        scenario_shares.update({
            tech: float(value)
            for tech, value
            in (
                e_cfg.get(
                    "shares",
                    {},
                )
                or {}
            ).items()
        })

        for control_order, ctrl_cfg in enumerate(
            control_configs,
            start=1,
        ):

            run_cfg = dict(
                ctrl_cfg
            )

            run_cfg["shares"] = dict(
                scenario_shares
            )

            run_cfg["electrification_id"] = e_cfg["scenario_id"]

            run_cfg["scenario_meta"] = dict(e_cfg.get("scenario_meta", {}) or {})

            run_cfg[
                "experiment_id"
            ] = EXPERIMENT_ID

            run_cfg[
                "run_revision"
            ] = RUN_REVISION

            run_cfg[
                "scenario_order"
            ] = int(
                scenario_order
            )

            run_cfg[
                "control_order"
            ] = int(
                control_order
            )

            runs.append(
                attach_run_identity(
                    run_cfg
                )
            )

    return runs



# Function to build explicit electrification scenario catalog
def build_electrification_scenarios_from_definition(experiment_def):
    """
    Build the explicit electrification-scenario catalog from one
    declarative experiment definition.

    Supported forms:
      - explicit "scenarios"
      - builder="share_sweep"
      - builder="share_grid"
      - builder="ev_v2g_grid"
      - builder="pv_bess_coupled_grid"
    """

    if not isinstance(experiment_def, dict):
        raise TypeError(
            "experiment_def must be a dictionary."
        )

    # --------------------------------------------------------
    # 1) Explicit scenario list
    # --------------------------------------------------------

    if "scenarios" in experiment_def:

        raw_scenarios = experiment_def.get("scenarios") or []

        scenarios = []

        for scenario in raw_scenarios:

            if not isinstance(scenario, dict):
                raise TypeError(
                    "Each explicit electrification scenario "
                    "must be a dictionary."
                )

            out = dict(scenario)

            out["shares"] = dict(
                scenario.get("shares", {}) or {}
            )

            out["scenario_meta"] = dict(
                scenario.get("scenario_meta", {}) or {}
            )

            scenarios.append(out)

    # --------------------------------------------------------
    # 2) Builder-based experiment
    # --------------------------------------------------------

    else:

        builder = str(
            experiment_def.get("builder", "")
        ).strip().lower()

        if builder == "share_sweep":

            scenarios = make_share_sweep(
                tech=experiment_def["tech"],
                values=experiment_def["values"],
                prefix=experiment_def.get("prefix"),
            )

        elif builder == "share_grid":

            scenarios = make_share_grid(
                axes=experiment_def["axes"],
            )

        elif builder == "ev_v2g_grid":

            scenarios = make_ev_v2g_grid(
                ev_values=experiment_def["ev_values"],
                v2g_values=experiment_def["v2g_values"],
            )

        elif builder == "pv_bess_coupled_grid":

            scenarios = make_pv_bess_coupled_grid(
                pv_values=experiment_def["pv_values"],
                bess_on_pv_values=experiment_def[
                    "bess_on_pv_values"
                ],
            )

        else:

            raise ValueError(
                "Unknown electrification experiment builder: "
                f"{builder!r}"
            )

    # --------------------------------------------------------
    # 3) Validate generated scenarios
    # --------------------------------------------------------

    if not scenarios:
        raise ValueError(
            "The selected electrification experiment "
            "generated no scenarios."
        )

    scenario_ids = []

    for scenario in scenarios:

        if "scenario_id" not in scenario:
            raise KeyError(
                "Every electrification scenario must define "
                "'scenario_id'."
            )

        scenario_id = str(
            scenario["scenario_id"]
        ).strip()

        if not scenario_id:
            raise ValueError(
                "scenario_id cannot be empty."
            )

        scenario["scenario_id"] = scenario_id
        scenario_ids.append(scenario_id)

        shares = scenario.get("shares", {}) or {}

        unknown_techs = (
            set(shares)
            - set(RUN_SHARE_KEYS)
        )

        if unknown_techs:
            raise KeyError(
                f"Scenario {scenario_id} contains unknown "
                f"technology shares: {sorted(unknown_techs)}"
            )

        for tech, value in shares.items():

            value = float(value)

            if not (0.0 <= value <= 100.0):
                raise ValueError(
                    f"Scenario {scenario_id}: "
                    f"{tech} share must be in [0, 100], "
                    f"found {value}."
                )

            shares[tech] = value

        scenario["shares"] = shares

    if len(scenario_ids) != len(set(scenario_ids)):
        raise ValueError(
            "Duplicate scenario_id values in the "
            "electrification experiment."
        )

    return scenarios

### 4.6.4 F(x) Technology inference and diagnostic for "has" assignement

In [ ]:
# -------------------------------------------------------------------
#       Sec 4.6 -- 4.1) TECHNOLOGY INFERENCE FROM RESULT DATAFRAMES
#-------------------------------------------------------------------
# Generic function to identify which households c_i actually have the different technologies/storage systems, e.g. BESS, BEV, HP
def infer_has_from_flows(df, tech_tags, require_nonzero=True):
    """
    df: flow dataframe, with MultiIndex columns as in df_raw_case
    tech_tags: list of tags, e.g. ["PV"] or ["DMD_ELC_HEAT"]
    require_nonzero: if True, requires at least one value != 0
    """
    if df is None or df.empty:
        return set()

    tags = [t.upper() for t in tech_tags]
    out = set()

    for col in df.columns:
        s = _col_to_text(col).upper()

        # match technology tag
        if not any(tag in s for tag in tags):
            continue

        # extract household
        cid = extract_case_id(col)
        if not cid:
            continue

        if require_nonzero:
            try:
                arr = df[col].to_numpy()
                if np.any(arr != 0):
                    out.add(cid)
            except Exception:
                # fallback: if the series is not numeric, consider the column as evidence of presence
                out.add(cid)
        else:
            out.add(cid)

    return out



# Function to identify which households c_i actually have BESS storage, looking for a BESS charge/discharge column
def infer_has_bess_from_flows(df_raw_case):
    """
    Returns the set of c_i households that have at least one column with the *_BESS pattern,
    typically bus_el<->BESS flows.
    """
    out = set()
    for col in df_raw_case.columns:
        s = _col_to_text(col)
        # take c_i only if "_BESS" also appears
        if "_BESS" in s:
            out.update(re.findall(r"(c_\d+)", s))
    return out



# Function to identify which households c_i actually have DHW storage, looking for a DHW charge/discharge column
def infer_has_dhw_from_flows(dhw_house_flows_df):
    """
    Returns the set of c_i households that have at least one column with the *_DHW_storage pattern
    considering both charge and discharge, from df_raw_case + df_dhw_use.
    """
    out = set()
    for col in dhw_house_flows_df.columns:
        s = _col_to_text(col)
        if "_DHW_storage" in s:
            out.update(re.findall(r"(c_\d+)", s))
    return out


# Function to identify which households c_i actually have a BEV, looking for a BEV charge/discharge column
def infer_has_bev_from_flows(df_bev_charge, df_bev_dis):
    """
    Returns the set of c_i households that have at least one column with *_BEV_storage or bus_el_*BEV.
    """
    out = set()
    for df in [df_bev_charge, df_bev_dis]:
        if df is None:
            continue
        for col in df.columns:
            s = _col_to_text(col)
            if ("_BEV_storage" in s) or ("_bus_el_charge_BEV" in s) or ("_bus_el_discharge_BEV" in s):
                out.update(re.findall(r"(c_\d+)", s))
    return out




# Function to infer the "has_" dictionaries identifying which households c_i have which technologies
def build_has_from_dfraw(df_raw_case):
    """Infers ALL has_* dictionaries from df_raw_case using TECH_TAGS and infer_has_from_flows."""
    has_local = {}
    for k in TECH_KEYS:
        req = REQUIRE_NONZERO_BY_TECH.get(k, True)
        has_local[k] = infer_has_from_flows(df_raw_case, TECH_TAGS[k], require_nonzero=req)
    return has_local

# -----------------------------------------------------------------------


# -------------------------------------------------------------------
#       Sec 4.6 -- 4.2) VALIDATION AND NORMALIZATION OF "has" DICTIONARIES
#-------------------------------------------------------------------
# The current logic uses:
#   - the global has dictionary created/loaded in 4.3/4.9
#   - _set_has_aliases_from_has_dict(...)
#   - build_effective_has_for_run(...)
#   - print_has_summary(...) for diagnostics.
# Function to save summary information on which customers have each flexibility option
def print_has_summary(has: dict, ID=None, max_show=30, title="HAS_TECH SUMMARY"):
    """
    Prints what is about to be saved in 'has'.
    - ID: list of all valid IDs, e.g. ['c_1','c_2',...], for sanity checks.
    - max_show: how many elements to show for each technology; 0 = do not show lists.
    """
    print("\n" + "="*90)
    print(f"[DEBUG] {title}")
    if not isinstance(has, dict):
        print(f"[ERROR] 'has' is not a dict: type={type(has)}")
        print("="*90)
        return

    keys = list(has.keys())
    print(f"[DEBUG] keys ({len(keys)}): {keys}")

    valid = set(ID) if ID is not None else None

    # robust sorting for c_i
    def _sort_key(ci):
        try:
            return int(str(ci).split("_")[1])
        except Exception:
            return 10**9

    for k in sorted(keys):
        s = has.get(k, set())
        # normalize to set
        if s is None:
            s = set()
        elif not isinstance(s, set):
            try:
                s = set(s)
            except Exception:
                s = {str(s)}

        # sort
        s_sorted = sorted(list(s), key=_sort_key)
        n = len(s_sorted)

        # sanity check: elements outside ID
        if valid is not None:
            extras = sorted([x for x in s_sorted if x not in valid], key=_sort_key)
            missing = []  # "missing" is not meaningful here; only extras are checked
        else:
            extras = []

        print(f"\n[DEBUG] {k}: len={n}")
        if valid is not None and len(extras) > 0:
            print(f"  [WARN] elements NOT in ID ({len(extras)}): {extras[:max_show]}")

        if max_show > 0:
            if n <= max_show:
                print("  IDs:", s_sorted)
            else:
                print("  IDs (head):", s_sorted[:max_show])
                print("  ...")
                print("  IDs (tail):", s_sorted[-min(10, max_show):])

    print("="*90 + "\n")
#----------------------------------------------




# -------------------------------------------------------------------
#       Sec 4.6 -- 4.3) EV METADATA
#-------------------------------------------------------------------
# Function to identify the household type for each family, e.g. single vs couple_SF
def extract_household_type_by_household_from_ev_metadata(
    ev_metadata_df: pd.DataFrame,
) -> Dict[str, str]:
    """
    Extracts a dictionary:

        {'Household_i': 'household_type'}

    from the English EV metadata structure.

    Expected examples:
        Household_1_Member_1_autumn_single
        Household_7_Member_1_autumn_couple_w_children_4
        Household_14_Member_1_autumn_single_parent_M_2

    The function also checks that the household identifier embedded in
    'person' is consistent with the separate 'household' column.

    For backward compatibility, legacy Italian labels such as
    Nucleo_1_Membro_1_autunno_single are accepted as well.
    """
    required_columns = {'person', 'household'}
    missing_columns = required_columns.difference(
        ev_metadata_df.columns
    )

    if missing_columns:
        raise KeyError(
            "The EV metadata file is missing the required column(s): "
            f"{sorted(missing_columns)}. "
            f"Available columns: {ev_metadata_df.columns.tolist()}"
        )

    metadata = ev_metadata_df.loc[
        :,
        ['person', 'household'],
    ].copy()

    metadata['person'] = (
        metadata['person']
        .astype('string')
        .str.strip()
    )

    metadata['household'] = (
        metadata['household']
        .astype('string')
        .str.strip()
    )

    # Accepted formats:
    #   Household_i_Member_j_season_household_type
    #   Nucleo_i_Membro_j_season_household_type  (legacy)
    parsed = metadata['person'].str.extract(
        r'^(?P<household_prefix>Household|Nucleo)_'
        r'(?P<household_number>\d+)_'
        r'(?P<member_prefix>Member|Membro)_'
        r'(?P<member_number>\d+)_'
        r'(?P<season>'
        r'winter|spring|summer|autumn|'
        r'inverno|primavera|estate|autunno'
        r')_'
        r'(?P<household_type>.+)$',
        flags=re.IGNORECASE,
    )

    required_parsed_columns = [
        'household_prefix',
        'household_number',
        'member_prefix',
        'member_number',
        'season',
        'household_type',
    ]

    bad_mask = (
        parsed[required_parsed_columns]
        .isna()
        .any(axis=1)
    )

    if bad_mask.any():
        bad_rows = metadata.loc[
            bad_mask,
            ['person', 'household'],
        ]

        raise ValueError(
            "Could not parse some rows in the EV metadata "
            "'person' column. Expected format: "
            "Household_i_Member_j_season_household_type.\n"
            "Unrecognized rows:\n"
            + bad_rows.to_string(index=True)
        )

    # Normalize the household identifier parsed from person
    # to the English convention.
    parsed['household_from_person'] = (
        'Household_'
        + parsed['household_number'].astype(str)
    )

    # Normalize the separate household column.
    household_number_from_column = (
        metadata['household']
        .str.extract(
            r'^(?:Household|Nucleo)_(\d+)$',
            flags=re.IGNORECASE,
            expand=False,
        )
    )

    bad_household_column_mask = (
        household_number_from_column.isna()
    )

    if bad_household_column_mask.any():
        bad_rows = metadata.loc[
            bad_household_column_mask,
            ['person', 'household'],
        ]

        raise ValueError(
            "Could not parse some values in the EV metadata "
            "'household' column. Expected 'Household_i'.\n"
            "Unrecognized rows:\n"
            + bad_rows.to_string(index=True)
        )

    metadata['household_normalized'] = (
        'Household_'
        + household_number_from_column.astype(str)
    )

    # Check consistency between:
    # person = Household_3_Member_1_...
    # household = Household_3
    mismatch_mask = parsed['household_from_person'].ne(
        metadata['household_normalized']
    )

    if mismatch_mask.any():
        mismatch_rows = pd.DataFrame({
            'person':
                metadata.loc[mismatch_mask, 'person'],

            'household_column':
                metadata.loc[mismatch_mask, 'household'],

            'household_from_person':
                parsed.loc[
                    mismatch_mask,
                    'household_from_person',
                ],
        })

        raise ValueError(
            "Inconsistent household identifiers between "
            "the 'person' and 'household' columns:\n"
            + mismatch_rows.to_string(index=True)
        )

    parsed['household'] = (
        parsed['household_from_person']
    )

    parsed['household_type'] = (
        parsed['household_type']
        .astype('string')
        .str.strip()
    )

    # All members belonging to the same household must have
    # the same household type.
    type_count_by_household = (
        parsed
        .groupby('household')['household_type']
        .nunique(dropna=True)
    )

    inconsistent_households = (
        type_count_by_household[
            type_count_by_household > 1
        ]
    )

    if not inconsistent_households.empty:
        raise ValueError(
            "Different household types were found for members "
            "of the same household:\n"
            + inconsistent_households.to_string()
        )

    household_type_by_household = (
        parsed
        .groupby('household')['household_type']
        .first()
        .to_dict()
    )

    return household_type_by_household

   ### 4.6.5 F(x) Debugging, infeasibility, and linear-programming diagnostics (LP/IIS)

In [ ]:
# -------------------------------------------------------------------
#       Sec 4.6 -- 5.1) DEBUG infeasible LP/IIS
#-------------------------------------------------------------------
# Function to retrieve the directory where files with infeasibility information were saved
def auto_pick_paths():
    """
    Select one failed solve for Sec. 4.15 post-mortem analysis.

    Selection priority:
      1) manually specified DEBUG_LP_PATH / DEBUG_IIS_JSON_PATH;
      2) DEBUG_SELECT_RUN_UID + DEBUG_SELECT_SEASON;
      3) DEBUG_SELECT_RUN_UID only -> most recent failure of that run;
      4) DEBUG_SELECT_SEASON only -> most recent failure in that season;
      5) LAST_INFEASIBLE_KEY -> last failure observed.
    """

    lp_path = DEBUG_LP_PATH
    iis_json_path = DEBUG_IIS_JSON_PATH

    selected_key = None


    # --------------------------------------------------------
    # Exact RUN + SEASON selection
    # --------------------------------------------------------

    if (
        DEBUG_SELECT_RUN_UID is not None
        and DEBUG_SELECT_SEASON is not None
    ):

        candidate_key = (
            str(DEBUG_SELECT_RUN_UID),
            str(DEBUG_SELECT_SEASON),
        )

        if (
            candidate_key
            in INFEASIBLE_DEBUG_REGISTRY
        ):

            selected_key = candidate_key

        else:

            print(
                "[WARN][DEBUG] Requested failure "
                f"{candidate_key} not found in "
                "INFEASIBLE_DEBUG_REGISTRY."
            )


    # --------------------------------------------------------
    # RUN only -> last failure for this run
    # --------------------------------------------------------

    elif DEBUG_SELECT_RUN_UID is not None:

        candidates = [
            key
            for key
            in INFEASIBLE_DEBUG_REGISTRY.keys()
            if key[0]
            == str(DEBUG_SELECT_RUN_UID)
        ]

        if candidates:
            selected_key = candidates[-1]


    # --------------------------------------------------------
    # SEASON only -> last failure in this season
    # --------------------------------------------------------

    elif DEBUG_SELECT_SEASON is not None:

        candidates = [
            key
            for key
            in INFEASIBLE_DEBUG_REGISTRY.keys()
            if key[1]
            == str(DEBUG_SELECT_SEASON)
        ]

        if candidates:

            selected_key = candidates[-1]

            if len(candidates) > 1:

                print(
                    "[WARN][DEBUG] Multiple failed runs "
                    f"found for season "
                    f"{DEBUG_SELECT_SEASON}. "
                    "Selecting the latest one. "
                    "Set DEBUG_SELECT_RUN_UID for "
                    "an exact selection."
                )


    # --------------------------------------------------------
    # Default -> last failure observed
    # --------------------------------------------------------

    else:

        selected_key = LAST_INFEASIBLE_KEY


    # --------------------------------------------------------
    # Retrieve stored paths
    # --------------------------------------------------------

    season = None

    if (
        selected_key is not None
        and selected_key
        in INFEASIBLE_DEBUG_REGISTRY
    ):

        info = (
            INFEASIBLE_DEBUG_REGISTRY[
                selected_key
            ]
        )

        season = info.get(
            "season"
        )

        lp_path = (
            lp_path
            or info.get("lp_path")
        )

        iis_json_path = (
            iis_json_path
            or info.get("iis_json_path")
        )

        print(
            "[DEBUG SELECT] "
            f"run_uid={info.get('run_uid')} | "
            f"case={info.get('case_label')} | "
            f"season={season} | "
            f"status={info.get('solve_status')}"
        )


    # --------------------------------------------------------
    # Manual-path fallback
    # --------------------------------------------------------

    if season is None:
        season = DEBUG_SELECT_SEASON


    return (
        season,
        lp_path,
        iis_json_path,
    )


# Function to preview a file containing a summary of infeasibility reasons
def preview_text_file(path, head=80, tail=60, encoding="utf-8"):
    path = str(path)
    with open(path, "r", encoding=encoding, errors="ignore") as f:
        lines = f.readlines()
    print(f"\n--- PREVIEW: {path} (n_lines={len(lines)}) ---")
    print("".join(lines[:head]))
    if tail > 0:
        print("\n... [SNIP] ...\n")
        print("".join(lines[-tail:]))


# Function to identify and extract the names of constraints violated during optimization
def lp_constraint_names(lp_path, contains=None, max_names=2000):
    """
    Extracts constraint names from the 'Subject To' section of the LP file.
    """
    names = []
    in_subj = False
    with open(lp_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not in_subj:
                if s.lower().startswith("subject to"):
                    in_subj = True
                continue

            # end of constraint section
            if s.lower().startswith(("bounds", "binary", "binaries", "general", "generals", "end")):
                break

            # constraint-start line: "<name>: ..."
            m = re.match(r"^\s*([^\s:]+)\s*:\s*", line)
            if m:
                name = m.group(1)
                if (contains is None) or (contains.lower() in name.lower()):
                    names.append(name)
                    if len(names) >= max_names:
                        break
    return names



# Function to group constraint families
def _prefix(name: str):
    # group constraint families: cut at [, (, :
    for sep in ("[", "(", ":"):
        if sep in name:
            return name.split(sep)[0]
    return name


# Function to summarize names
def summarize_names(names, top=30):
    c = Counter(_prefix(n) for n in names)
    print("\n--- TOP families (prefixes) ---")
    for k, v in c.most_common(top):
        print(f"{k:45s}  {v}")


# Function to load the file containing infeasibility information for the run
def load_iis_json(iis_json_path):
    with open(iis_json_path, "r", encoding="utf-8") as f:
        return json.load(f)





# Function to analyze BEV home/outdoor charging availability profiles
def _scan_print(msg):
    global can_scan_hits
    if can_scan_hits < CAN_SCAN_PRINT_LIMIT:
        print(msg)
    can_scan_hits += 1

   ### 4.6.6 F(x) Input 10min timeseries (PV, prices), dwelling occupancy

In [ ]:
#-------------------------------------------------------------------
#       Sec 4.6 -- 6.1) FUNCTIONS FOR CREATING 10-MINUTE PROFILES (PV, prices)
#-------------------------------------------------------------------
# Function to randomly select and assign a specific profile to each household, e.g. PV orientation for each dwelling
def select_random_profile(num_households, df):
    """
    Randomly assigns one source profile to each household.

    The output keeps the legacy 'Nucleo_i' (i.e., 'Household_i') column convention because
    downstream input-processing functions still support that schema.
    """
    profile_columns = df.columns[1:]

    if profile_columns.empty:
        raise ValueError(
            "No PV profiles were found in the source DataFrame."
        )

    selected_columns = [
        random.choice(profile_columns)
        for _ in tqdm(
            range(num_households),
            desc=(
                "Random assignment of PV profiles to "
                f"{num_households} households"
            ),
        )
    ]

    output_data = {
        "Datetime": df["Datetime"]
    }

    for household_index, source_column in enumerate(
        selected_columns,
        start=1,
    ):
        output_data[f"Nucleo_{household_index}"] = df[source_column]

    selected_profiles_df = pd.DataFrame(output_data)

    return selected_profiles_df

#-----------------------------------------------------------------------------------------------------------------------------



#-------------------------------------------------------------------
#       Sec 4.6 -- 6.2) FUNCTIONS FOR OCCUPANCY AND HOME PRESENCE
#-------------------------------------------------------------------
# Function to define the BEV home-charging availability profile, i.e. when the vehicle is at home and can charge, but charging is not mandatory
def build_profile_max_occupancy(
    occupancy_df: pd.DataFrame,
    idx: pd.DatetimeIndex,
    c: str,
    current_season: str | None = None,
    min_dwell_steps: int = 1,
    member: int = 1,
) -> pd.Series:
    """
    Returns a Series with index=idx and values in [0, 1],
    usable as Flow(max=...).

    Rule:
    - use only Member_{member};
    - enable charging only after min_dwell_steps consecutive
      timesteps with occupancy equal to 1.

    Legacy Italian labels Nucleo_i_Membro_j are accepted
    as well.
    """
    match = re.match(
        r"c_(\d+)",
        str(c),
        flags=re.IGNORECASE,
    )

    if not match:
        raise ValueError(
            f"Unrecognized household format: {c}"
        )

    household_number = int(match.group(1))

    household_tags = (
        f"Household_{household_number}_Member_{member}_",
        f"Nucleo_{household_number}_Membro_{member}_",
    )

    # Candidate columns usually also include season
    # and household type.
    candidate_columns = [
        col
        for col in occupancy_df.columns
        if any(
            tag.casefold() in str(col).casefold()
            for tag in household_tags
        )
    ]

    if current_season is not None:
        season_token = (
            f"_{str(current_season).strip().lower()}_"
        )

        season_columns = [
            col
            for col in candidate_columns
            if season_token in str(col).lower()
        ]

        if season_columns:
            candidate_columns = season_columns

    if not candidate_columns:
        print(
            f"[WARN] Occupancy not found for {c} -> "
            "using max=1, i.e. no charging constraint."
        )

        return pd.Series(
            1.0,
            index=idx,
            name=f"max_occ_{c}",
        )

    # Logical OR across possible matching columns.
    occupancy_raw = (
        occupancy_df[candidate_columns]
        .max(axis=1)
    )

    occupancy = (
        occupancy_raw
        .reindex(idx)
        .fillna(0)
        .astype(int)
    )

    output = np.zeros(
        len(idx),
        dtype=float,
    )

    dwell_steps = 0

    for timestep in range(len(idx)):
        if occupancy.iat[timestep] == 1:
            dwell_steps += 1
        else:
            dwell_steps = 0

        output[timestep] = (
            1.0
            if dwell_steps >= min_dwell_steps
            else 0.0
        )

    return pd.Series(
        output,
        index=idx,
        name=f"max_occ_{c}",
    )


# Function to define whether the household member is at home or not, used to define the outdoor charging profile
def build_home_presence_profile(
    occupancy_df: pd.DataFrame,
    idx: pd.DatetimeIndex,
    c: str,
    current_season: str | None = None,
    member: int = 1,
) -> pd.Series:
    """
    Returns a 0/1 Series indicating whether the EV owner
    is at home.

    It does not apply dwell logic.

    English labels Household_i_Member_j are expected.
    Legacy Italian labels Nucleo_i_Membro_j are accepted
    as well.
    """
    match = re.match(
        r"c_(\d+)",
        str(c),
        flags=re.IGNORECASE,
    )

    if not match:
        raise ValueError(
            f"Unrecognized household format: {c}"
        )

    household_number = int(match.group(1))

    household_tags = (
        f"Household_{household_number}_Member_{member}_",
        f"Nucleo_{household_number}_Membro_{member}_",
    )

    candidate_columns = [
        col
        for col in occupancy_df.columns
        if any(
            tag.casefold() in str(col).casefold()
            for tag in household_tags
        )
    ]

    if current_season is not None:
        season_token = (
            f"_{str(current_season).strip().lower()}_"
        )

        season_columns = [
            col
            for col in candidate_columns
            if season_token in str(col).lower()
        ]

        if season_columns:
            candidate_columns = season_columns

    if not candidate_columns:
        print(
            f"[WARN] Raw occupancy not found for {c} -> "
            "assuming always away from home = 0."
        )

        return pd.Series(
            0.0,
            index=idx,
            name=f"home_presence_{c}",
        )

    occupancy_raw = (
        occupancy_df[candidate_columns]
        .max(axis=1)
    )

    occupancy = (
        occupancy_raw
        .reindex(idx)
        .fillna(0)
        .astype(float)
        .clip(0.0, 1.0)
    )

    return pd.Series(
        occupancy,
        index=idx,
        name=f"home_presence_{c}",
    )

### 4.6.7 F(x) Conventional/Rule-based profiles

In [ ]:
#-------------------------------------------------------------------
#       Sec 4.6 -- 7.1) FUNCTIONS FOR SIMULATING BESS PROFILES (conventional/rule-based case)
#-------------------------------------------------------------------
# ---------- Function to simulate a domestic customer with a 10-minute timestep ----------
def simulate_bess_series(
    demand_sr,                   # single-household domestic demand series (values in kWh/timestep)
    pv_1kwp_sr,                  # single-household domestic PV production series (values in kWh/timestep)
    PV_power_kwp,                # =3, PV power (kWp)
    capacity_kwh,                # =12, battery capacity (kWh)
    eta_ch, eta_dis,             # =0.9, BESS efficiencies
    soc0_kwh,                    # =0.0, initial SoC in kWh
    p_charge_max_kw=None,        # BESS charge limit in kW (optional)
    p_discharge_max_kw=None,     # BESS discharge limit in kW (optional)
    timestep_minutes=10,
    data_is_power = True,        # True if the series are in kW; False if already in kWh per timestep
):
    """
    Returns a DataFrame with flows in kWh per timestep, using the same index as demand_sr.
    """
    # ---------- INDEX REALIGNMENT ----------
    # ensure they are pandas.Series
    if not isinstance(demand_sr, pd.Series) or not isinstance(pv_1kwp_sr, pd.Series):
        raise ValueError("demand_sr and pv_1kwp_sr must be pandas.Series")

    # take the index intersection, avoiding errors if some rows are missing
    idx_common = demand_sr.index.intersection(pv_1kwp_sr.index)
    if len(idx_common) == 0:
        raise ValueError("Empty index intersection: check date/file formats for PV and demand")
    demand_sr = demand_sr.reindex(idx_common)
    pv_1kwp_sr = pv_1kwp_sr.reindex(idx_common)

    
    # unit conversion
    timestep_hours = timestep_minutes / 60.0
    
    # If data are in power (kW), convert them into energy per timestep (kWh)
    if data_is_power:
        demand_kwh = demand_sr * timestep_hours
        pv_kwh_per_1kwp = pv_1kwp_sr * timestep_hours
    else:
        # data are already energy per timestep (kWh)
        demand_kwh = demand_sr.astype(float)
        pv_kwh_per_1kwp = pv_1kwp_sr.astype(float)
    
    # scale the PV profile according to installed power (kWp)
    pv_kwh = pv_kwh_per_1kwp * float(PV_power_kwp)
    
    # convert power limits into energy per timestep (kWh per timestep)
    p_charge_max_kwh = None if p_charge_max_kw is None else float(p_charge_max_kw) * timestep_hours
    p_discharge_max_kwh = None if p_discharge_max_kw is None else float(p_discharge_max_kw) * timestep_hours


    # ---------- simulation loop ----------
    n = len(demand_kwh)
    pv = pv_kwh.values                             
    demand = demand_kwh.values
    
    pv_to_load = np.minimum(pv, demand)             # self-consumed PV share
    excess_pv = np.maximum(pv - pv_to_load, 0.0)    # PV share not consumed, sent either to BESS or to the LV grid
    pv_to_batt_input = np.zeros(n)                  # PV share sent to BESS, before eta_charge
    energy_added_to_soc = np.zeros(n)               # energy actually charged into the BESS, net of eta_charge
    batt_draw_from_soc = np.zeros(n)                # energy drawn from the BESS, before eta_discharge
    batt_delivered_to_load = np.zeros(n)            # energy delivered by the BESS, used by the domestic customer, net of eta_discharge
    soc = np.zeros(n)                               # State of Charge
    grid_export = np.zeros(n)                       # PV share delivered to the LV grid
    grid_import = np.zeros(n)                       # energy withdrawn from the LV grid, used by the customer
    
    soc_prev = float(soc0_kwh)
    capacity = float(capacity_kwh)
    
    for t in range(n):
        demand_remaining = demand[t] - pv_to_load[t]
        # CHARGE from surplus PV
        max_storable = capacity - soc_prev
        if max_storable <= 1e-12:
            pv_input_allowed = 0.0
        else:
            charge_limit = np.inf if p_charge_max_kwh is None else p_charge_max_kwh
            pv_input_allowed = min(excess_pv[t], charge_limit, max_storable / eta_ch)
        
        pv_to_batt_input[t] = pv_input_allowed
        energy_added_to_soc[t] = pv_input_allowed * eta_ch
        soc_after_charge = soc_prev + energy_added_to_soc[t]
        grid_export[t] = max(excess_pv[t] - pv_input_allowed, 0.0)
        
        # DISCHARGE to cover demand_remaining
        discharge_limit = np.inf if p_discharge_max_kwh is None else p_discharge_max_kwh
        max_deliverable_to_load = soc_after_charge * eta_dis
        # respect the power limit converted into energy per timestep
        if np.isfinite(discharge_limit):
            max_deliverable_to_load = min(max_deliverable_to_load, discharge_limit * eta_dis)
        energy_from_batt_to_load = min(max_deliverable_to_load, demand_remaining)
        batt_delivered_to_load[t] = energy_from_batt_to_load
        batt_draw = (energy_from_batt_to_load / eta_dis) if energy_from_batt_to_load > 0 else 0.0
        batt_draw_from_soc[t] = batt_draw
        soc_after_discharge = soc_after_charge - batt_draw
        grid_import[t] = max(demand_remaining - energy_from_batt_to_load, 0.0)
        
        soc[t] = soc_after_discharge
        soc_prev = soc[t]
            
    # data are in energy [kWh]
    df_out = pd.DataFrame({
        "demand_kwh": demand,
        "pv_kwh": pv,
        "pv_to_load_kwh": pv_to_load,                               # self-consumption
        "excess_pv_kwh": excess_pv,                                 # PV share not immediately consumed; can go to BESS or LV grid
        "pv_to_batt_input_kwh": pv_to_batt_input,                   # PV share sent to BESS
        "energy_added_to_soc_kwh": energy_added_to_soc,             # energy actually added to BESS SoC, previously called "energy_stored" and potentially ambiguous
        "batt_draw_from_soc_kwh": batt_draw_from_soc,               # energy drawn from BESS
        "batt_delivered_to_load_kwh": batt_delivered_to_load,       # energy actually delivered by BESS, after eta_discharge
        "soc_kwh": soc,
        "grid_export_kwh": grid_export,                             # PV share not self-consumed nor sent to BESS, but exported to the LV grid
        "grid_import_kwh": grid_import                              # energy required from the LV grid to cover remaining demand after BESS and PV
    }, index=idx_common)
    
    return df_out
# ------------------------------------------------------------------




#-------------------------------------------------------------------
#       Sec 4.6 -- 7.2) FUNCTIONS FOR GENERIC SOC CALCULATION AND TIMESTEP LOSS
#-------------------------------------------------------------------


# Function for correctly calculating boiler losses, from hourly loss to timestep loss in the RB case
def effective_loss_fraction_over_timestep(loss_rate_per_hour, dt_hours):
    """
    Converts a loss_rate defined per HOUR into the effective loss over the timestep.
    Consistent with oemof-solph GenericStorage.
    """
    if loss_rate_per_hour is None:
        return 0.0

    try:
        lr = float(loss_rate_per_hour)
    except (TypeError, ValueError):
        return 0.0

    lr = min(max(lr, 0.0), 1.0)
    dt_h = max(float(dt_hours or 0.0), 0.0)

    if (lr <= 0.0) or (dt_h <= 0.0):
        return 0.0

    return 1.0 - (1.0 - lr) ** dt_h
#-----------------------------------------------------------------------------------------------------------------------------


#-------------------------------------------------------------------
#       Sec 4.6 -- 7.3) FUNCTIONS FOR SIMULATING DHW AND BEV PROFILES (rule-based case)
#-------------------------------------------------------------------
# Function to calculate DHW boiler input flows in the RB case. If DHW demand cannot be fully met by the RB boiler, a DHW_boost profile is defined for the missing share
def rb_simulate_dhw_soc_and_charge(
    idx,
    dhw_draw_kwh_ts: pd.Series,     # DHW demand, required thermal energy, kWh/timestep
    cap_kwh: float,
    pmax_kw: float,
    eta: float,
    loss_rate: float,               # interpreted as an HOURLY relative loss
    soc0_kwh: float,
    hyst_on: float,
    hyst_off: float,
    timestep_minutes: int = 10,
    pboost_kw: float = None,        # electric power limit of the boost; if None -> unlimited boost
):
    dt_h = timestep_minutes / 60.0
    eta_eff = float(eta if eta is not None else 1.0)
    eta_eff = max(eta_eff, 1e-9)

    cap_kwh = float(max(0.0, cap_kwh))
    pmax_kw = float(max(0.0, pmax_kw))
    soc_prev = float(min(max(0.0, soc0_kwh), cap_kwh))
    step_loss = effective_loss_fraction_over_timestep(loss_rate, dt_h)

    soc_prev_arr = np.zeros(len(idx))
    soc_after_loss_arr = np.zeros(len(idx))
    soc_raw_arr = np.zeros(len(idx))          # can be negative
    soc_arr = np.zeros(len(idx))              # physical boiler SoC, non-negative
    soc_pct_arr = np.zeros(len(idx))

    p_charge_el_arr = np.zeros(len(idx))      # electric kW for boiler charging
    dhw_boost_th_arr = np.zeros(len(idx))     # missing kWh_th required to supply demand
    dhw_boost_el_arr = np.zeros(len(idx))     # equivalent electric kW of the boost
    dhw_shortage_th_arr = np.zeros(len(idx))  # possible uncovered share if pboost is limited

    is_on = False

    for t in range(len(idx)):
        soc_prev_arr[t] = soc_prev

        # 1) losses
        soc_after_loss = soc_prev * (1.0 - step_loss)
        soc_after_loss_arr[t] = soc_after_loss

        # 2) DHW draw
        e_out = float(max(0.0, dhw_draw_kwh_ts.iloc[t]))
        soc_raw = soc_after_loss - e_out
        soc_raw_arr[t] = soc_raw

        # thermal deficit that the boiler alone cannot cover
        e_boost_th_req = max(0.0, -soc_raw)

        # equivalent electric boost
        p_boost_el_req = (e_boost_th_req / (dt_h * eta_eff)) if dt_h > 0 else 0.0

        if (pboost_kw is not None) and (pboost_kw > 0):
            p_boost_el = min(p_boost_el_req, float(pboost_kw))
        else:
            p_boost_el = p_boost_el_req

        e_boost_th_served = p_boost_el * dt_h * eta_eff
        e_shortage_th = max(0.0, e_boost_th_req - e_boost_th_served)

        dhw_boost_th_arr[t] = e_boost_th_served
        dhw_boost_el_arr[t] = p_boost_el
        dhw_shortage_th_arr[t] = e_shortage_th

        # physical boiler SoC after the draw: cannot go below zero
        soc_after_draw = max(0.0, soc_raw)

        # 3) hysteresis on the physical boiler
        soc_pct = soc_after_draw / cap_kwh if cap_kwh > 0 else 0.0
        
        # If OFF, switch ON when the lower threshold is reached
        if (not is_on) and (soc_pct <= hyst_on):
            is_on = True

        # Safety / consistency:
        # if already ON but the upper threshold is already reached,
        # switch OFF before charging
        if is_on and (soc_pct >= hyst_off):
            is_on = False

        # 4) standard boiler charging
        if is_on and (pmax_kw > 0.0) and (dt_h > 0.0):
            e_in_th_max = pmax_kw * dt_h* eta_eff

            # Thermal content corresponding to thermostat cut-off
            soc_off_kwh = (hyst_off * cap_kwh)

            # Do not charge beyond the thermostat OFF threshold
            e_room_to_off = max(0.0, soc_off_kwh - soc_after_draw,)

            e_in_th = min(e_in_th_max, e_room_to_off,)

            p_el = (e_in_th / (dt_h * eta_eff) if dt_h > 0 else 0.0)

            p_charge_el_arr[t] = p_el

            soc_new = (
                soc_after_draw
                + e_in_th
            )

            # Once the upper thermostat threshold is reached,
            # switch the resistance OFF for the following timestep
            if soc_new >= soc_off_kwh - 1e-9:
                is_on = False

        else:

            soc_new = soc_after_draw

        soc_new = min(max(soc_new, 0.0), cap_kwh)

        soc_arr[t] = soc_new
        soc_pct_arr[t] = (soc_new / cap_kwh * 100.0) if cap_kwh > 0 else 0.0
        soc_prev = soc_new

    return {
        "SOC_DHW_prev_kWh": pd.Series(soc_prev_arr, index=idx, name="SOC_DHW_prev_kWh"),
        "SOC_DHW_after_loss_kWh": pd.Series(soc_after_loss_arr, index=idx, name="SOC_DHW_after_loss_kWh"),
        "SOC_DHW_raw_kWh": pd.Series(soc_raw_arr, index=idx, name="SOC_DHW_raw_kWh"),
        "SOC_DHW_kWh": pd.Series(soc_arr, index=idx, name="SOC_DHW_kWh"),
        "SOC_DHW_%": pd.Series(soc_pct_arr, index=idx, name="SOC_DHW_%"),
        "P_el_to_DHW_kw": pd.Series(p_charge_el_arr, index=idx, name="P_el_to_DHW_kw"),
        "DHW_boost_th_kWh": pd.Series(dhw_boost_th_arr, index=idx, name="DHW_boost_th_kWh"),
        "DHW_boost_el_kw": pd.Series(dhw_boost_el_arr, index=idx, name="DHW_boost_el_kw"),
        "DHW_shortage_th_kWh": pd.Series(dhw_shortage_th_arr, index=idx, name="DHW_shortage_th_kWh"),
    }


# -------- BEV SoC, RB case --------------
# Function to identify the next continuous mobility block and evaluate if it is a normal - long - very long trip (to eventually adjust the EV SOC charge)
def rb_get_next_bev_trip_requirement(
    use,
    current_t,
    lookahead_steps,
    eta_dis,
    cap_kwh,
    eps=1e-12,
):
    """
    Identify the next continuous mobility block whose start
    lies within the selected look-ahead horizon.

    Only the next trip is considered.
    The complete trip energy is evaluated once its start
    has been identified.

    Returns the battery-side fraction of capacity required
    by that trip.
    """

    n = len(use)

    if (
        n == 0
        or cap_kwh <= 0.0
        or lookahead_steps <= 0
    ):
        return None


    search_end = min(
        n - 1,
        current_t + int(lookahead_steps),
    )


    # Find start of next trip
    trip_start = None

    for j in range(
        current_t + 1,
        search_end + 1,
    ):
        if float(use.iat[j]) > eps:
            trip_start = j
            break


    if trip_start is None:
        return None


    # Extend to the end of that continuous mobility block
    trip_end = trip_start

    while (
        trip_end < n - 1
        and float(use.iat[trip_end + 1]) > eps
    ):
        trip_end += 1


    trip_use_kwh = float(
        use.iloc[
            trip_start:
            trip_end + 1
        ].sum()
    )


    # Battery-side energy required by mobility
    trip_battery_required_kwh = (
        trip_use_kwh
        / float(eta_dis)
    )


    trip_required_frac = (
        trip_battery_required_kwh
        / float(cap_kwh)
    )


    return {
        "start": trip_start,
        "end": trip_end,
        "battery_required_kwh":
            trip_battery_required_kwh,
        "required_frac":
            trip_required_frac,
    }



# Function to calculate BEV SoC in a "greedy" way: as soon as the vehicle is at home, charge as much as possible up to the threshold, with the possibility of using "charger_outdoor" when needed
def rb_simulate_bev_soc_and_charge_greedy_to_target(
    idx,
    bev_use_kwh_ts,          # kWh/timestep mobility demand on the "use" side
    max_profile,             # 0..1 home-charging availability
    outdoor_profile,         # 0..1 outdoor-charging availability
    cap_kwh,
    pmax_kw,
    pmax_outdoor_kw,
    eta_ch,                  # total charging efficiency
    eta_dis,                 # discharge efficiency from BEV content to mobility demand
    loss_rate,
    soc0_kwh,

    # Standard RB behaviour
    soc_target_pct=0.80,     # charging target to be reached when connected at home
    outdoor_trigger_pct=0.20,   # warning trigger SOC when outdoor
    outdoor_target_pct=0.50,    # charging target to be reached when connected to outdoor charger and trigger SOC activated
    
    # Minimum SOC, consistent with OPT
    min_soc_pct=0.05,

    # Limited next-trip foresight
    enable_planned_foresight=True,
    planned_lookahead_hours=12.0,

    long_trip_consumption_frac=0.30,
    very_long_trip_consumption_frac=0.65,

    long_trip_target_pct=0.70,
    very_long_trip_target_pct=1.00,
    
    
    min_home_ts=1,           # min. number of consecutive timesteps required to enable home charging
    timestep_minutes=10,
    enable_outdoor=True,
):
    idx = pd.DatetimeIndex(idx)
    n = len(idx)
    dt_h = float(timestep_minutes) / 60.0

    use = pd.to_numeric(pd.Series(bev_use_kwh_ts, index=idx), errors="coerce").fillna(0.0).clip(lower=0.0)
    mp = pd.to_numeric(pd.Series(max_profile, index=idx), errors="coerce").fillna(0.0).clip(0.0, 1.0)
    op = pd.to_numeric(pd.Series(outdoor_profile, index=idx), errors="coerce").fillna(0.0).clip(0.0, 1.0)

    cap_kwh = float(max(0.0, cap_kwh))
    if cap_kwh <= 0.0:
        z = pd.Series(0.0, index=idx)
        return z, z, z, z, z, z, z

    pmax_kw = float(max(0.0, pmax_kw))
    pmax_outdoor_kw = float(max(0.0, pmax_outdoor_kw))
    eta_ch = float(max(1e-6, eta_ch))
    eta_dis = float(max(1e-6, eta_dis))
    loss_rate = float(max(0.0, loss_rate))

    soc_target_kwh = float(soc_target_pct) * cap_kwh

    outdoor_trigger_kwh = (
        float(outdoor_trigger_pct)
        * cap_kwh
    )

    outdoor_target_kwh = (
        float(outdoor_target_pct)
        * cap_kwh
    )

    min_soc_kwh = (
        float(min_soc_pct)
        * cap_kwh
    )

    long_trip_target_kwh = (
        float(long_trip_target_pct)
        * cap_kwh
    )

    very_long_trip_target_kwh = (
        float(very_long_trip_target_pct)
        * cap_kwh
    )

    planned_lookahead_steps = int(
        np.ceil(
            float(planned_lookahead_hours)
            / dt_h
        )
    )




    if not (
        0.0
        <= float(min_soc_pct)
        <= float(outdoor_trigger_pct)
        <= float(outdoor_target_pct)
        <= float(soc_target_pct)
        <= float(very_long_trip_target_pct)
        <= 1.0
    ):
        raise ValueError(
            "Invalid BEV SOC thresholds."
        )

    if not (
        0.0
        <= float(long_trip_consumption_frac)
        < float(very_long_trip_consumption_frac)
        <= 1.0
    ):
        raise ValueError(
            "Invalid long-trip thresholds."
        )


    soc = np.zeros(n, dtype=float)
    p_charge = np.zeros(n, dtype=float)
    deficit_storage = np.zeros(n, dtype=float)    # deficit on the battery-content side, e.g. if the battery is insufficient to meet mobility demand
    outdoor_el_kwh = np.zeros(n, dtype=float)     # electric energy used from the outdoor charger
    outdoor_el_kw = np.zeros(n, dtype=float)      # equivalent outdoor charger power
    bev_shortage_kwh = np.zeros(n, dtype=float)   # Unserved mobility energy when the battery cannot fully supply the mobility demand of the current timestep. Outdoor charging cannot repair this shortage retroactively.

    soc_now = float(min(max(0.0, soc0_kwh), cap_kwh))
    home_run = 0
    # Becomes True when SOC falls below the outdoor alarm threshold.
    # It remains active until SOC reaches the outdoor target
    # or home charging has sufficiently restored the battery.
    outdoor_recharge_pending = False

    for t in range(n):
        can_charge_home = mp.iat[t] > 0.0
        can_charge_outdoor = op.iat[t] > 0.0

        if can_charge_home:
            home_run += 1
        else:
            home_run = 0

        # storage losses
        soc_now *= (1.0 - loss_rate)


        # ------------------------------------------------------------
        # 1) MOBILITY
        #
        # During mobility the BEV can only discharge.
        # No home/outdoor charging is allowed while moving.
        # ------------------------------------------------------------
        draw = float(
            use.iat[t]
        )

        is_stationary = (
            draw <= 1e-12
        )

        if draw > 0.0:

            need_from_batt = (
                draw
                / eta_dis
            )

            available_for_mobility = max(
                0.0,
                soc_now - min_soc_kwh,
            )


            if (
                available_for_mobility
                >= need_from_batt
                - 1e-12
            ):

                soc_now -= need_from_batt


            else:

                deficit_storage_t = max(
                    0.0,
                    need_from_batt
                    - available_for_mobility,
                )

                deficit_storage[t] = (
                    deficit_storage_t
                )

                # Preserve the same minimum SOC used in OPT.
                soc_now = min_soc_kwh

                bev_shortage_kwh[t] = (
                    deficit_storage_t
                    * eta_dis
                )

        # ------------------------------------------------------------
        # 2) OUTDOOR-CHARGING ALARM
        #
        # Once SOC falls below the trigger, remember the need
        # for public charging until the first available
        # away-and-stationary period.
        # ------------------------------------------------------------
        if (
            enable_outdoor
            and soc_now <= outdoor_trigger_kwh
        ):
            outdoor_recharge_pending = True


        # ------------------------------------------------------------
        # 3) LIMITED FORESIGHT: NEXT SCHEDULED TRIP
        #
        # Behavioural simplification:
        # - normal trip      -> no additional planned target
        # - long trip        -> target 70%
        # - very long trip   -> target 100%
        # ------------------------------------------------------------

        is_stationary = (
            draw <= 1e-12
        )

        planned_target_kwh = 0.0
        planned_trip_class = None


        if (
            enable_planned_foresight
            and is_stationary
        ):

            trip_info = (
                rb_get_next_bev_trip_requirement(
                    use=use,
                    current_t=t,
                    lookahead_steps=(
                        planned_lookahead_steps
                    ),
                    eta_dis=eta_dis,
                    cap_kwh=cap_kwh,
                )
            )


            if trip_info is not None:

                trip_required_frac = float(
                    trip_info["required_frac"]
                )


                # Very long trip:
                # prepare the EV to 100%.
                if (
                    trip_required_frac
                    > float(
                        very_long_trip_consumption_frac
                    )
                ):

                    planned_target_kwh = (
                        very_long_trip_target_kwh
                    )

                    planned_trip_class = (
                        "very_long"
                    )


                # Long trip:
                # use a coarse 70% preparation target.
                elif (
                    trip_required_frac
                    >= float(
                        long_trip_consumption_frac
                    )
                ):

                    planned_target_kwh = (
                        long_trip_target_kwh
                    )

                    planned_trip_class = (
                        "long"
                    )

        # ------------------------------------------------------------
        # HOME TARGET PRIORITY
        #
        # Normal home charging target = 80%.
        #
        # Planned-trip targets can override it only if higher.
        #
        # long trip:
        #     max(80%, 70%) = 80%
        #
        # very long trip:
        #     max(80%, 100%) = 100%
        # ------------------------------------------------------------

        home_target_this_ts_kwh = max(
            soc_target_kwh,
            planned_target_kwh,
        )


        # ------------------------------------------------------------
        # 4) HOME CHARGING
        #
        # Conventional/RB behaviour:
        # as soon as the vehicle can charge at home,
        # charge greedily up to the home SOC target (80%).
        # ------------------------------------------------------------



        if (
            is_stationary
            and can_charge_home
            and home_run >= int(min_home_ts)
            and pmax_kw > 0.0
            and soc_now < home_target_this_ts_kwh
        ):

            pmax_t_kw = (
                pmax_kw
                * float(mp.iat[t])
            )

            e_max_kwh = (
                pmax_t_kw
                * dt_h
            )

            room_to_target = max(
                0.0,
                home_target_this_ts_kwh
                - soc_now,
            )

            e_need_kwh = (
                room_to_target
                / eta_ch
            )

            e_charge_kwh = min(
                e_max_kwh,
                e_need_kwh,
            )

            if e_charge_kwh > 0.0:

                soc_now += (
                    e_charge_kwh
                    * eta_ch
                )

                soc_now = min(
                    soc_now,
                    cap_kwh,
                )

                p_charge[t] = (
                    e_charge_kwh
                    / dt_h
                )

        # Home charging has priority over outdoor charging.
        # If SOC has already recovered above the outdoor target,
        # the previous outdoor alarm can be cleared.
        if (
            can_charge_home
            and soc_now >= outdoor_target_kwh
        ):
            outdoor_recharge_pending = False


        # --------------------------------
        # 5.0) OUTDOOR TARGET PRIORITY
        outdoor_target_this_ts_kwh = 0.0


        # Emergency 20% -> 50%
        if outdoor_recharge_pending:

            outdoor_target_this_ts_kwh = max(
                outdoor_target_this_ts_kwh,
                outdoor_target_kwh,
            )


        # Planned long / very-long trip.
        # This does NOT require SOC <= 20%.
        if planned_target_kwh > 0.0:

            outdoor_target_this_ts_kwh = max(
                outdoor_target_this_ts_kwh,
                planned_target_kwh,
            )


        # ------------------------------------------------------------
        # 5) OUTDOOR CHARGING
        #
        # Public charging is allowed only when:
        #   - the SOC alarm was previously triggered;
        #   - the vehicle is away from home;
        #   - the vehicle is stationary;
        #   - an outdoor charger is available.
        #
        # Charging continues until outdoor_target_kwh is reached.
        # ------------------------------------------------------------

        is_stationary = (
            draw <= 1e-12
        )

        if (
            enable_outdoor
            and is_stationary
            and can_charge_outdoor
            and (not can_charge_home)
            and pmax_outdoor_kw > 0.0
            and outdoor_target_this_ts_kwh > 0.0
            and soc_now < outdoor_target_this_ts_kwh
        ):

            pmax_outdoor_t_kw = (
                pmax_outdoor_kw
                * float(op.iat[t])
            )

            e_outdoor_max_kwh = (
                pmax_outdoor_t_kw
                * dt_h
            )

            room_to_outdoor_target = max(
                0.0,
                outdoor_target_this_ts_kwh
                - soc_now,
            )

            e_outdoor_need_kwh = (
                room_to_outdoor_target
                / eta_ch
            )

            e_outdoor_el = min(
                e_outdoor_max_kwh,
                e_outdoor_need_kwh,
            )

            if e_outdoor_el > 0.0:

                soc_now += (
                    e_outdoor_el
                    * eta_ch
                )

                soc_now = min(
                    soc_now,
                    cap_kwh,
                )

                outdoor_el_kwh[t] = (
                    e_outdoor_el
                )

                outdoor_el_kw[t] = (
                    e_outdoor_el / dt_h
                    if dt_h > 0.0
                    else 0.0
                )

            # Keep the alarm active until the 50% target
            # has actually been reached.
            if (
                soc_now
                >= outdoor_target_kwh - 1e-9
            ):
                outdoor_recharge_pending = False


        # ------------------------------------------------------------
        # 6) STORE END-OF-TIMESTEP SOC
        # ------------------------------------------------------------

        soc[t] = soc_now





    soc_sr = pd.Series(soc, index=idx, name="SOC_BEV_kWh")
    soc_pct_sr = (soc_sr / cap_kwh) * 100.0
    p_charge_sr = pd.Series(p_charge, index=idx, name="P_el_to_BEV_kw")
    deficit_sr = pd.Series(deficit_storage, index=idx, name="BEV_deficit_kWh")
    outdoor_el_kwh_sr = pd.Series(outdoor_el_kwh, index=idx, name="BEV_outdoor_el_kWh")
    outdoor_el_kw_sr = pd.Series(outdoor_el_kw, index=idx, name="BEV_outdoor_el_kw")
    bev_shortage_kwh_sr = pd.Series(bev_shortage_kwh, index=idx, name="BEV_shortage_kWh")

    return (
        soc_sr,
        soc_pct_sr,
        p_charge_sr,
        deficit_sr,
        outdoor_el_kwh_sr,
        outdoor_el_kw_sr,
        bev_shortage_kwh_sr,
    )

   ### 4.6.8 F(x) Energy system modeling: solph, objective and adapter profiles

In [ ]:
#-------------------------------------------------------------------
#       Sec 4.6 --8.1) LV BUS/TOPOLOGY & NETWORK LINKS
#-------------------------------------------------------------------
# Function to generalize bus construction while accounting for topology
def _lv_node_name(feeder_code: str, node_index: int) -> str:
    """
    Builds the label of an LV feeder node.
    """
    return f"bus_el_BT_{feeder_code}_{node_index}"


# Function to build the first bus at the secondary-substation head
def _lv_root_name() -> str:
    """
    Returns the LV root-bus label.
    """
    return "bus_el_BT_0"


# Function that identifies the grid-bus indices located between households
def lv_bus_name(
    node_index: int,
    feeder_code: str,
    lv_root_name_func,
    lv_node_name_func,
) -> str:
    """
    Returns the root-bus label for node 0 and a feeder-node label
    for all downstream nodes.
    """
    if node_index == 0:
        return lv_root_name_func()

    return lv_node_name_func(
        feeder_code,
        node_index,
    )


# Function to assign names to grid components located between households
def lv_link_labels(
    upstream_index: int,
    downstream_index: int,
    feeder_code: str = "Da",
) -> tuple[str, str]:
    """
    Returns the forward and backward labels of an LV-grid link.
    """
    forward_label = (
        f"bt_link_forw_{feeder_code}"
        f"{upstream_index}_{downstream_index}"
    )

    backward_label = (
        f"bt_link_back_{feeder_code}"
        f"{downstream_index}_{upstream_index}"
    )

    return forward_label, backward_label


# Function to group all indices and labels for LV-grid components
def build_lv_link_recipes(num_households: int, feeder_code: str = "Da") -> list[dict]:
    """
    Returns a list of link "recipes" for the feeder:
    0<->1, 1<->2, ..., (N-1)<->N

    Each recipe contains:
    - idx_a, idx_b
    - lab_forw, lab_back
    """
    recipes = []

    # 0 <-> 1
    lab_forw, lab_back = lv_link_labels(0, 1, feeder_code=feeder_code)
    recipes.append({"idx_a": 0, "idx_b": 1, "lab_forw": lab_forw, "lab_back": lab_back})

    # 1 <-> 2, ..., (N-1) <-> N
    for k in range(1, num_households):
        lab_forw, lab_back = lv_link_labels(k, k + 1, feeder_code=feeder_code)
        recipes.append({"idx_a": k, "idx_b": k + 1, "lab_forw": lab_forw, "lab_back": lab_back})

    return recipes



# Function that maps households to the LV-grid bus
def _lv_bus_for_house(i_house: int, num_households: int, feeder_code: str = "Da") -> str:
    """
    Maps household c_i to the LV bus of the feeder.

    Default: c_1 -> bus_el_BT_Da_1, ..., c_N -> bus_el_BT_Da_N
    """
    i = int(i_house)
    if i < 1 or i > int(num_households):
        raise ValueError(f"Household index out of range: i={i}, expected 1..{num_households}")
    return _lv_node_name(feeder_code, i)
# --------------------------------------------------------------------------



#-------------------------------------------------------------------
#       Sec 4.6 --8.2) FLOW EXTRACTION FROM SOLPH RESULTS
#-------------------------------------------------------------------
# Function that extracts the series of a specific flow from "results"
def _get_flow_series_from_sequences(seq_df):
    """Extracts the flow series from data['sequences'] robustly."""
    if seq_df is None or len(seq_df) == 0:
        return None
    # solph typically uses 'flow' or ('flow',)
    if "flow" in seq_df.columns:
        return seq_df["flow"]
    if ("flow",) in seq_df.columns:
        return seq_df[("flow",)]
    return None


# Function that extracts the full result time series without losing possible "empty/null" timesteps
def _full_ts_series(s, timeindex_canonico):
    """Forces length n_ts on timeindex_canonico + safety fill."""
    if s is None:
        return pd.Series(0.0, index=timeindex_canonico)
    # if already aligned, reindexing is almost free; if something is missing, fill with 0
    return s.reindex(timeindex_canonico).fillna(0.0)


# Function to read a specific flow from optimization results, e.g. to reconstruct MV-grid import/export
def _get_edge_flow_series(results, from_label, to_label, timeindex_canonico):
    """
    Extracts the time series of the flow results[(from_label, to_label)].
    If not found, returns a zero series on timeindex_canonico.
    """
    for (frm, to), data in results.items():
        if not isinstance(data, dict):
            continue
        if "sequences" not in data:
            continue

        if getattr(frm, "label", None) == from_label and getattr(to, "label", None) == to_label:
            s = _get_flow_series_from_sequences(data["sequences"])
            return _full_ts_series(s, timeindex_canonico).astype(float)

    return pd.Series(0.0, index=timeindex_canonico, dtype=float)



# Function that returns all flows exchanged with respect to a specific bus label
def collect_bus_sequences_from_results(results, bus_label, timeindex_canonico):
    """
    Returns a DataFrame with all incoming/outgoing flow columns associated with the bus.
    Columns: "X -> bus" (in) and "bus -> Y" (out)
    Index: timeindex_canonico, with guaranteed length n_ts
    """
    cols = {}

    for (frm, to), data in results.items():
        if not isinstance(data, dict):
            continue
        if "sequences" not in data:
            continue

        flow_s = _get_flow_series_from_sequences(data["sequences"])
        if flow_s is None:
            continue

        # incoming to the bus
        if hasattr(to, "label") and to.label == bus_label:
            key = f"{frm.label} -> {bus_label}"
            cols[key] = _full_ts_series(flow_s, timeindex_canonico)

        # outgoing from the bus
        if hasattr(frm, "label") and frm.label == bus_label:
            key = f"{bus_label} -> {to.label}"
            cols[key] = _full_ts_series(flow_s, timeindex_canonico)

    if not cols:
        return pd.DataFrame(index=timeindex_canonico)

    return pd.DataFrame(cols, index=timeindex_canonico)

# --------------------------------------------------------------------------



#-------------------------------------------------------------------
#       Sec 4.6 --8.3) OBJECTIVE/COST BREAKDOWN
#-------------------------------------------------------------------
# Function to read the value of the objective function
def _safe_objective_value(om):
    """
    Robustly retrieves the value of the objective function.
    """
    try:
        return float(po.value(om.objective))
    except Exception:
        pass

    try:
        meta = solph.processing.meta_results(om, undefined=True)
        obj = meta.get("objective", np.nan)
        return float(obj) if pd.notna(obj) else np.nan
    except Exception:
        return np.nan
    

# Function to align costs to the real temporal index
def _align_cost_parameter_to_real_index(
    param,
    idx_real,
    timeindex_canonico,
    default: float = 0.0,
) -> pd.Series:
    """
    Align an oemof.solph cost parameter to idx_real.

    Supports:
      - None;
      - plain scalar values;
      - oemof.solph scalar wrappers (_FakeSequence);
      - pandas Series;
      - numpy/list-like time series.

    Important:
    oemof.solph may internally wrap scalar Flow parameters such as
    variable_costs=10 or variable_costs=0.5 into a _FakeSequence.
    Such objects must be interpreted as a constant value over all
    timesteps, not converted to zero through the fallback.
    """

    # ------------------------------------------------------------
    # 1) Missing parameter
    # ------------------------------------------------------------

    if param is None:
        return pd.Series(
            float(default),
            index=idx_real,
            dtype=float,
        )


    # ------------------------------------------------------------
    # 2) Plain Python / NumPy scalar
    # ------------------------------------------------------------

    if np.isscalar(param):

        try:
            value = float(param)

        except Exception:
            value = float(default)

        return pd.Series(
            value,
            index=idx_real,
            dtype=float,
        )


    # ------------------------------------------------------------
    # 3) oemof.solph scalar sequence wrapper
    #
    # Flow(variable_costs=<scalar>) can internally store the
    # scalar as a _FakeSequence rather than as a plain float.
    #
    # Example:
    #   cost_boost_DHW       = 10
    #   cost_outdoor_charger = 0.5
    #
    # param[0] returns the constant scalar value.
    # ------------------------------------------------------------

    if type(param).__name__ == "_FakeSequence":

        try:
            value = float(
                param[0]
            )

        except Exception:

            try:
                value = float(
                    param
                )

            except Exception:
                value = float(
                    default
                )

        return pd.Series(
            value,
            index=idx_real,
            dtype=float,
        )


    # ------------------------------------------------------------
    # 4) pandas Series
    # ------------------------------------------------------------

    if isinstance(
        param,
        pd.Series,
    ):

        s = param.copy()

        try:
            s = pd.to_numeric(
                s,
                errors="coerce",
            )

        except Exception:
            s = s.astype(float)


        if len(s) == len(idx_real):

            if not s.index.equals(
                idx_real
            ):
                s.index = idx_real

            return (
                s
                .reindex(idx_real)
                .fillna(float(default))
                .astype(float)
            )


        if len(s) == len(
            timeindex_canonico
        ):

            if not s.index.equals(
                timeindex_canonico
            ):
                s.index = (
                    timeindex_canonico
                )

            return (
                set_real_index(
                    pd.DataFrame({
                        "v": _full_ts_series(
                            s,
                            timeindex_canonico,
                        )
                    }),
                    idx_real,
                )["v"]
                .astype(float)
            )


        try:

            return (
                s
                .reindex(idx_real)
                .fillna(float(default))
                .astype(float)
            )

        except Exception:
            pass


    # ------------------------------------------------------------
    # 5) Generic list / numpy-array-like parameter
    # ------------------------------------------------------------

    try:

        arr = np.asarray(
            param,
            dtype=float,
        ).reshape(-1)


        if arr.size == 1:

            return pd.Series(
                float(arr[0]),
                index=idx_real,
                dtype=float,
            )


        if arr.size == len(
            idx_real
        ):

            return pd.Series(
                arr,
                index=idx_real,
                dtype=float,
            )


        if arr.size == len(
            timeindex_canonico
        ):

            s = pd.Series(
                arr,
                index=timeindex_canonico,
                dtype=float,
            )

            return (
                set_real_index(
                    pd.DataFrame({
                        "v": _full_ts_series(
                            s,
                            timeindex_canonico,
                        )
                    }),
                    idx_real,
                )["v"]
                .astype(float)
            )

    except Exception:
        pass


    # ------------------------------------------------------------
    # 6) Last-resort fallback
    # ------------------------------------------------------------

    return pd.Series(
        float(default),
        index=idx_real,
        dtype=float,
    )


# Function to classify the different cost terms of the objective function
def _classify_objective_flow_term(from_label: str | None, to_label: str | None) -> tuple[str, str, str]:
    fl = str(from_label or "")
    tl = str(to_label or "")

    if tl.startswith("from_the_grid_BT_to_"):
        return (
            "bt_energy_import_settlement_eur",
            "energy_settlement_bt",
            "Energy purchase cost on LV -> household converters",
        )

    if fl.endswith("_to_the_grid_BT"):
        return (
            "bt_energy_export_settlement_eur",
            "energy_settlement_bt",
            "Energy sale/feed-in term on household -> LV converters",
        )

    if fl.startswith("line_forw_") or fl.startswith("line_back_") or fl.startswith("bt_link_forw") or fl.startswith("bt_link_back"):
        return (
            "kerber_line_transfer_cost_eur",
            "network_transfer",
            "Microscopic transfer penalty on LV lines, used as anti-loop term, not a business cost",
        )

    if tl.endswith("_DHW_boost"):
        return (
            "dhw_boost_cost_eur",
            "flexibility_penalty",
            "Cost/penalty for DHW boost use",
        )

    if fl.endswith("_charger_outdoor"):
        return (
            "bev_outdoor_charge_cost_eur",
            "flexibility_penalty",
            "Cost/penalty for outdoor BEV charging",
        )

    if tl.endswith("_PV_curtail"):
        return (
            "pv_curtail_cost_eur",
            "flexibility_penalty",
            "Cost/penalty for PV curtailment",
        )

    if "shortage" in fl.lower() or "shortage" in tl.lower():
        return (
            "shortage_cost_eur",
            "flexibility_penalty",
            "Cost/penalty for shortage",
        )

    return (
        "other_variable_cost_terms_eur",
        "other",
        "Other variable objective terms",
    )



# Function to unpack the objective-function terms
def build_objective_cost_breakdown(
    om,
    results,
    idx_real,
    timeindex_canonico,
    timestep_minutes,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, float]:
    dt_h = float(timestep_minutes) / 60.0

    ts_by_category: dict[str, pd.Series] = {}
    flow_rows: list[dict] = []

    for (frm, to), flow_obj in getattr(om, "flows", {}).items():
        from_label = getattr(frm, "label", None)
        to_label = getattr(to, "label", None)
        var_costs = getattr(flow_obj, "variable_costs", None)

        if var_costs is None:
            continue

        s_flow_kw = _get_edge_flow_series(results, from_label, to_label, timeindex_canonico)
        s_flow_kw = set_real_index(pd.DataFrame({"flow_kw": s_flow_kw}), idx_real)["flow_kw"].astype(float)
        s_var_cost = _align_cost_parameter_to_real_index(var_costs, idx_real, timeindex_canonico, default=0.0)
        s_term_eur = s_flow_kw * dt_h * s_var_cost

        if not np.isfinite(s_term_eur.to_numpy(dtype=float)).any():
            continue

        category, cost_group, description = _classify_objective_flow_term(from_label, to_label)
        ts_by_category.setdefault(category, pd.Series(0.0, index=idx_real, dtype=float))
        ts_by_category[category] = ts_by_category[category].add(s_term_eur, fill_value=0.0)

        flow_rows.append({
            "from_label": from_label,
            "to_label": to_label,
            "house_id": extract_house_id_from_labels(from_label, to_label),
            "category": category,
            "cost_group": cost_group,
            "description": description,
            "flow_total_kwh": float((s_flow_kw * dt_h).sum()),
            "avg_variable_cost_eur_per_kwh": float(s_var_cost.mean()),
            "term_total_eur": float(s_term_eur.sum()),
            "term_abs_total_eur": float(s_term_eur.abs().sum()),
            "n_nonzero_ts": int((s_term_eur.abs() > 1e-12).sum()),
        })

    df_terms_ts = pd.DataFrame(index=idx_real)
    for cat, s in sorted(ts_by_category.items()):
        df_terms_ts[cat] = s.astype(float)

    if not df_terms_ts.empty:
        df_terms_ts["objective_variable_cost_terms_eur"] = df_terms_ts.sum(axis=1)

    df_terms_by_flow = pd.DataFrame(flow_rows)

    if df_terms_by_flow.empty:
        df_terms_summary = pd.DataFrame(columns=["category", "cost_group", "description", "term_total_eur", "term_abs_total_eur", "n_flows", "n_nonzero_ts_max"])
        objective_variable_cost_terms_eur = 0.0
    else:
        df_terms_summary = (
            df_terms_by_flow.groupby(["category", "cost_group", "description"], dropna=False, as_index=False)
            .agg(
                term_total_eur=("term_total_eur", "sum"),
                term_abs_total_eur=("term_abs_total_eur", "sum"),
                n_flows=("from_label", "count"),
                n_nonzero_ts_max=("n_nonzero_ts", "max"),
            )
            .sort_values(["cost_group", "category"])
            .reset_index(drop=True)
        )
        objective_variable_cost_terms_eur = float(df_terms_summary["term_total_eur"].sum())

    return df_terms_ts, df_terms_by_flow, df_terms_summary, objective_variable_cost_terms_eur



# Function to apply ex-post economic reporting rules, to distinguish between effective cost and simulation costs (e.g., including or not costs for DHW_boost)
def apply_effective_system_cost_reporting(
    df_cost_ts,
    include_dhw_boost_in_effective_costs=False,
):
    """
    Apply ex-post economic reporting rules.

    This function DOES NOT modify the optimization objective
    and DOES NOT modify the dispatch.

    It only distinguishes:
      - DHW boost penalty used by the optimization;
      - DHW boost cost considered in effective economic reporting.
    """

    out = df_cost_ts.copy()

    required_columns = [
        "bt_purchase_cost_eur",
        "bt_sale_revenues_eur",
        "dhw_boost_cost_eur",
        "bev_outdoor_charge_cost_eur",
        "pv_curtail_cost_eur",
    ]

    missing = [
        col for col in required_columns
        if col not in out.columns
    ]

    if missing:
        raise KeyError(
            "Missing columns required for effective-cost reporting: "
            f"{missing}"
        )

    # ------------------------------------------------------------
    # DHW BOOST
    #
    # dhw_boost_cost_eur:
    #     artificial penalty used by the optimization.
    #
    # dhw_boost_cost_effective_eur:
    #     part actually included in economic reporting.
    # ------------------------------------------------------------

    if include_dhw_boost_in_effective_costs:

        out["dhw_boost_cost_effective_eur"] = (
            out["dhw_boost_cost_eur"]
        )

        out["dhw_boost_cost_excluded_eur"] = 0.0

    else:

        out["dhw_boost_cost_effective_eur"] = 0.0

        out["dhw_boost_cost_excluded_eur"] = (
            out["dhw_boost_cost_eur"]
        )

    # ------------------------------------------------------------
    # EFFECTIVE SYSTEM COST
    #
    # Outdoor EV charging remains a real cost.
    # DHW boost penalty is optional only in reporting.
    # ------------------------------------------------------------

    out["total_system_cost_effective_eur"] = (
        out["bt_purchase_cost_eur"]
        + out["bt_sale_revenues_eur"]
        + out["dhw_boost_cost_effective_eur"]
        + out["bev_outdoor_charge_cost_eur"]
        + out["pv_curtail_cost_eur"]
    )

    out["total_system_effective_cost_per_timestep_eur"] = (
        out["total_system_cost_effective_eur"]
    )

    return out


# Function to collect the economic information obtained by solving the optimization problem, i.e. min-cost operation
def build_opt_system_cost_outputs(
    om,
    results,
    lv_root,
    idx_real,
    timeindex_canonico,
    current_season,
    MODE,
    ee_cost_series,
    ee_rev_series,
    timestep_minutes,
    LV_ARCHETYPE=None,
    num_households=None,
    selected_region=None,
    USE_PF_LINE_LIMITS_V1=False,
    PF_LINE_LIMITS_MAP=None,
    PF_INTERFACE_LIMITS_MAP=None,
    cap_imp_mt_kw=None,
    cap_exp_mt_kw=None,
    cost_boost_dhw_eur_per_kwh=None,
    cost_outdoor_charger_eur_per_kwh=None,
    include_dhw_boost_in_effective_costs=False,
    shortage_cost_eur_per_kwh=0.0,
    eps_active_kw=1e-9,
):
    """
    Builds:
      1) df_cost_ts: time series of the economic terms used in the summary
      2) df_cost_summary: one-row summary for scenario comparison
      3) df_objective_terms_ts / by_flow / summary: complete breakdown of the objective variable_costs
      4) df_objective_reconciliation: technical objective-reconciliation table

    NOTES:
      - 'bt_sale_settlement_eur' remains a technical item, positive, for reconciliation
      - 'bt_sale_revenues_eur' is the business reporting item, negative because it is revenue
      - 'bt_energy_net_settlement_eur' is the net business balance: purchases + negative revenues
      - shortage terms remain commented/ready, but are currently excluded from the summary
    """
    dt_h = float(timestep_minutes) / 60.0

    if cost_boost_dhw_eur_per_kwh is None:
        cost_boost_dhw_eur_per_kwh = float(cost_boost_DHW)

    if cost_outdoor_charger_eur_per_kwh is None:
        cost_outdoor_charger_eur_per_kwh = float(cost_outdoor_charger)

    def _to_real_index(series, colname):
        return set_real_index(pd.DataFrame({colname: series}), idx_real)[colname].astype(float)

    def _sum_result_flows_by_match_local(match_fn, colname):
        s_tot = pd.Series(0.0, index=timeindex_canonico, dtype=float)
        matched = []

        for (frm, to), data in results.items():
            if not isinstance(data, dict):
                continue
            if "sequences" not in data:
                continue

            frm_label = getattr(frm, "label", str(frm))
            to_label = getattr(to, "label", str(to))

            if not match_fn(frm_label, to_label):
                continue

            s = _get_flow_series_from_sequences(data["sequences"])
            if s is None:
                continue

            s = _full_ts_series(s, timeindex_canonico).astype(float)
            s_tot = s_tot.add(s, fill_value=0.0)
            matched.append((frm_label, to_label))

        return _to_real_index(s_tot, colname), matched

    def _active_houses_for_match(match_fn):
        active = set()

        for (frm, to), data in results.items():
            if not isinstance(data, dict):
                continue
            if "sequences" not in data:
                continue

            frm_label = getattr(frm, "label", str(frm))
            to_label = getattr(to, "label", str(to))

            if not match_fn(frm_label, to_label):
                continue

            s = _get_flow_series_from_sequences(data["sequences"])
            if s is None:
                continue

            s = _full_ts_series(s, timeindex_canonico).astype(float)

            # consider the household active only if the total flow is actually > 0
            if float((s.abs() * dt_h).sum()) <= eps_active_kw:
                continue

            house_id = extract_house_id_from_labels(frm_label, to_label)
            if house_id is not None:
                active.add(house_id)

        return active

    # ------------------------------------------------------------------
    # 1) Complete breakdown of objective variable_costs
    # ------------------------------------------------------------------
    (
        df_objective_terms_ts,
        df_objective_terms_by_flow,
        df_objective_terms_summary,
        objective_variable_cost_terms_eur,
    ) = build_objective_cost_breakdown(
        om=om,
        results=results,
        idx_real=idx_real,
        timeindex_canonico=timeindex_canonico,
        timestep_minutes=timestep_minutes,
    )

    # ------------------------------------------------------------------
    # 2) Flows to expose in the business summary
    # ------------------------------------------------------------------
    s_lv_import_kw, matched_lv_import = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).startswith("from_the_grid_BT_to_"),
        "bt_import_house_kw",
    )
    s_lv_export_kw, matched_lv_export = _sum_result_flows_by_match_local(
        lambda frm, to: str(frm).endswith("_to_the_grid_BT"),
        "bt_export_house_kw",
    )
    s_dhw_boost_kw, matched_dhw_boost = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_DHW_boost"),
        "dhw_boost_kw",
    )
    s_bev_outdoor_kw, matched_bev_outdoor = _sum_result_flows_by_match_local(
        lambda frm, to: str(frm).endswith("_charger_outdoor"),
        "bev_outdoor_kw",
    )
    s_pv_curtail_kw, matched_pv_curtail = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_PV_curtail"),
        "pv_curtail_kw",
    )

    # ------------------------------------------------------------
    # Energy flows for multi-run charts:
    # - total electric demand of customers
    # - import from the MV grid
    # - share covered locally/net self-consumption
    # ------------------------------------------------------------

    s_total_local_demand_kw, matched_total_local_demand = _sum_result_flows_by_match_local(
        lambda frm, to: (
            str(to).endswith("_dmd_elc_base")
            or str(to).endswith("_dmd_elc_Cool")
            or str(to).endswith("_dmd_elc_Heat")
            or str(to).endswith("_dmd_elc_Cook")
            or str(to).endswith("_charger")
            or str(to).endswith("_DHW_storage")
            or str(to).endswith("_DHW_boost")
        ),
        "total_local_demand_kw",
    )


    # ------------------------------------------------------------
    # Energy demand breakdown: base + cooling + heating + cooking + EV charge + DHW
    # ------------------------------------------------------------

    s_demand_base_kw, matched_demand_base = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_dmd_elc_base"),"demand_base_kw",)

    s_demand_cooling_kw, matched_demand_cooling = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_dmd_elc_Cool"), "demand_cooling_kw",)

    s_demand_heating_kw, matched_demand_heating = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_dmd_elc_Heat"), "demand_heating_kw",)

    s_demand_cooking_kw, matched_demand_cooking = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_dmd_elc_Cook"), "demand_cooking_kw",)

    s_demand_ev_charging_kw, matched_demand_ev_charging = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_charger"), "demand_ev_charging_kw",)

    s_demand_dhw_storage_kw, matched_demand_dhw_storage = _sum_result_flows_by_match_local(
        lambda frm, to: str(to).endswith("_DHW_storage"), "demand_dhw_storage_kw",)




    s_mt_import_kw, matched_mt_import = _sum_result_flows_by_match_local(
        lambda frm, to: (
            str(frm) == "supply_from_grid_MT"
            and str(to) == str(lv_root)
        ),
        "mt_import_kw",
    )

    # Gross export to the MV grid
    s_mt_export_kw, matched_mt_export = _sum_result_flows_by_match_local(
        lambda frm, to: (
            str(to).endswith("excess_to_grid_MT")
        ),
        "mt_export_kw",
    )

    # Kerber / legacy LV lines:
    # microscopic KERBER_LINE_COST used in the objective to penalize loops.
    # Reconstruct it directly from results because the breakdown via om.flows
    # may not always capture it robustly.
    s_kerber_line_kw, matched_kerber_line = _sum_result_flows_by_match_local(
        lambda frm, to: (
            str(frm).startswith("line_forw_")
            or str(frm).startswith("line_back_")
            or str(frm).startswith("bt_link_forw")
            or str(frm).startswith("bt_link_back")
        ),
        "kerber_line_transfer_kw",
    )

    # --- SHORTAGE: kept ready but currently excluded from the summary ---
    houses_dhw_boost_active = _active_houses_for_match(
        lambda frm, to: str(to).endswith("_DHW_boost")
    )
    houses_bev_outdoor_active = _active_houses_for_match(
        lambda frm, to: str(frm).endswith("_charger_outdoor")
    )
    houses_pv_curtail_active = _active_houses_for_match(
        lambda frm, to: str(to).endswith("_PV_curtail")
    )

    buy_price = ee_cost_series[current_season].reindex(idx_real).astype(float)
    sell_price = ee_rev_series[current_season].reindex(idx_real).astype(float)


    df_cost_ts = pd.DataFrame(index=idx_real)
    df_cost_ts["bt_import_house_kw"] = s_lv_import_kw
    df_cost_ts["bt_export_house_kw"] = s_lv_export_kw
    df_cost_ts["dhw_boost_kw"] = s_dhw_boost_kw
    df_cost_ts["bev_outdoor_kw"] = s_bev_outdoor_kw
    df_cost_ts["pv_curtail_kw"] = s_pv_curtail_kw
    df_cost_ts["kerber_line_transfer_kw"] = s_kerber_line_kw
    # df_cost_ts["shortage_kw"] = s_shortage_kw

    # Energy quantities for multi-run charts
    df_cost_ts["total_local_demand_kw"] = s_total_local_demand_kw

    # Gross exchanges with the MV grid
    df_cost_ts["mt_import_kw"] = s_mt_import_kw
    df_cost_ts["mt_export_kw"] = s_mt_export_kw
    df_cost_ts["mt_net_import_kw"] = (
        df_cost_ts["mt_import_kw"] - df_cost_ts["mt_export_kw"]
    )

    # Final electric-demand breakdown by component
    df_cost_ts["demand_base_kw"] = s_demand_base_kw
    df_cost_ts["demand_cooling_kw"] = s_demand_cooling_kw
    df_cost_ts["demand_heating_kw"] = s_demand_heating_kw
    df_cost_ts["demand_cooking_kw"] = s_demand_cooking_kw
    df_cost_ts["demand_ev_charging_kw"] = s_demand_ev_charging_kw
    df_cost_ts["demand_dhw_storage_kw"] = s_demand_dhw_storage_kw

    # Total DHW = storage charge + boost
    df_cost_ts["demand_dhw_total_kw"] = (
        df_cost_ts["demand_dhw_storage_kw"] + df_cost_ts["dhw_boost_kw"]
    )


    df_cost_ts["bt_import_house_kwh"] = df_cost_ts["bt_import_house_kw"] * dt_h
    df_cost_ts["bt_export_house_kwh"] = df_cost_ts["bt_export_house_kw"] * dt_h
    df_cost_ts["dhw_boost_kwh"] = df_cost_ts["dhw_boost_kw"] * dt_h
    df_cost_ts["bev_outdoor_kwh"] = df_cost_ts["bev_outdoor_kw"] * dt_h
    df_cost_ts["pv_curtail_kwh"] = df_cost_ts["pv_curtail_kw"] * dt_h
    df_cost_ts["kerber_line_transfer_kwh"] = df_cost_ts["kerber_line_transfer_kw"] * dt_h
    # df_cost_ts["shortage_kwh"] = df_cost_ts["shortage_kw"] * dt_h


    # Energy quantities for multi-run charts
    df_cost_ts["total_local_demand_kwh"] = df_cost_ts["total_local_demand_kw"] * dt_h

    # Gross and net exchanges with the MV grid
    df_cost_ts["mt_import_kwh"] = df_cost_ts["mt_import_kw"] * dt_h
    df_cost_ts["mt_export_kwh"] = df_cost_ts["mt_export_kw"] * dt_h
    df_cost_ts["mt_net_import_kwh"] = (
        df_cost_ts["mt_import_kwh"] - df_cost_ts["mt_export_kwh"]
    )

    # ratio between gross import and final demand
    df_cost_ts["gross_mv_import_share_of_demand_ts"] = np.where(
        df_cost_ts["total_local_demand_kwh"] > 1e-12,
        df_cost_ts["mt_import_kwh"] / df_cost_ts["total_local_demand_kwh"],
        np.nan,
    )

    # ratio between gross feed-in to the MV grid and final demand
    df_cost_ts["gross_mv_export_share_of_demand_ts"] = np.where(
        df_cost_ts["total_local_demand_kwh"] > 1e-12,
        df_cost_ts["mt_export_kwh"] / df_cost_ts["total_local_demand_kwh"],
        np.nan,
    )


    # ratio between NET import and final demand
    df_cost_ts["net_mv_import_share_of_demand_ts"] = np.where(
        df_cost_ts["total_local_demand_kwh"] > 1e-12,
        df_cost_ts["mt_net_import_kwh"] / df_cost_ts["total_local_demand_kwh"],
        np.nan,
    )




    # Final electric-demand breakdown by component [kWh]
    df_cost_ts["demand_base_kwh"] = df_cost_ts["demand_base_kw"] * dt_h
    df_cost_ts["demand_cooling_kwh"] = df_cost_ts["demand_cooling_kw"] * dt_h
    df_cost_ts["demand_heating_kwh"] = df_cost_ts["demand_heating_kw"] * dt_h
    df_cost_ts["demand_cooking_kwh"] = df_cost_ts["demand_cooking_kw"] * dt_h
    df_cost_ts["demand_ev_charging_kwh"] = df_cost_ts["demand_ev_charging_kw"] * dt_h
    df_cost_ts["demand_dhw_storage_kwh"] = df_cost_ts["demand_dhw_storage_kw"] * dt_h
    df_cost_ts["demand_dhw_total_kwh"] = df_cost_ts["demand_dhw_total_kw"] * dt_h

    df_cost_ts["demand_components_total_kwh"] = (
        df_cost_ts["demand_base_kwh"]
        + df_cost_ts["demand_cooling_kwh"]
        + df_cost_ts["demand_heating_kwh"]
        + df_cost_ts["demand_cooking_kwh"]
        + df_cost_ts["demand_ev_charging_kwh"]
        + df_cost_ts["demand_dhw_total_kwh"]
    )

    # Check: should be approximately zero.
    # If different from zero, total_local_demand_kwh includes some flow
    # that has not been mapped in the component breakdown.
    df_cost_ts["demand_components_gap_kwh"] = (
        df_cost_ts["total_local_demand_kwh"]
        - df_cost_ts["demand_components_total_kwh"]
    )

    # Demand share covered by local generation/self-consumption.
    # Use clip lower=0 to avoid small negative numerical values.
    df_cost_ts["local_self_consumption_kwh"] = (
        df_cost_ts["total_local_demand_kwh"] - df_cost_ts["mt_import_kwh"]
    ).clip(lower=0.0)

    df_cost_ts["mt_import_share_of_demand"] = np.where(
        df_cost_ts["total_local_demand_kwh"] > 1e-12,
        df_cost_ts["mt_import_kwh"] / df_cost_ts["total_local_demand_kwh"],
        np.nan,
    )

    df_cost_ts["local_self_consumption_share_of_demand"] = np.where(
        df_cost_ts["total_local_demand_kwh"] > 1e-12,
        df_cost_ts["local_self_consumption_kwh"] / df_cost_ts["total_local_demand_kwh"],
        np.nan,
    )

    df_cost_ts["buy_price_eur_per_kwh"] = buy_price
    df_cost_ts["sell_price_eur_per_kwh"] = sell_price

    # -----------------------------
    # Settlement / revenues
    # -----------------------------
    # Adopted convention:
    # - buy_price_eur_per_kwh  > 0
    # - sell_price_eur_per_kwh < 0
    #
    # Therefore, the export term is already a negative revenue:
    # export_kWh * negative_selling_price = negative value.
    # No additional minus sign should be applied.
    # --------------------
    df_cost_ts["bt_purchase_cost_eur"] = (
        df_cost_ts["bt_import_house_kwh"] * df_cost_ts["buy_price_eur_per_kwh"]
    )

    # local technical series: NOT saved in system_cost_ts
    lv_sale_settlement_series = (
        df_cost_ts["bt_export_house_kwh"] * df_cost_ts["sell_price_eur_per_kwh"]
    )

    # business reporting item: negative because it is revenue
    df_cost_ts["bt_sale_revenues_eur"] = lv_sale_settlement_series

    # correct net business balance:
    # positive purchase costs + negative revenues
    df_cost_ts["bt_energy_net_settlement_eur"] = (
        df_cost_ts["bt_purchase_cost_eur"] + df_cost_ts["bt_sale_revenues_eur"]
    )

    # energy settlement consistent with the objective function
    # if the household -> LV converter uses variable_costs = ee_rev_can negative,
    # then this is exactly the objective settlement for import/export.
    df_cost_ts["bt_energy_objective_settlement_eur"] = (
        df_cost_ts["bt_purchase_cost_eur"] + lv_sale_settlement_series
    )


    # ------------------------------------------------------------
    # Market cost/revenue on gross MV exchanges
    # ------------------------------------------------------------
    # These columns are a post-processing KPI on MV exchanges.
    # The objective function, instead, in the current model applies costs
    # on the LV -> household and household -> LV converters.
    # ------------------------------------------------------------

    df_cost_ts["market_mt_purchase_cost_eur"] = (
        df_cost_ts["mt_import_kwh"] * df_cost_ts["buy_price_eur_per_kwh"]
    )

    df_cost_ts["market_mt_sale_revenues_eur"] = (
        df_cost_ts["mt_export_kwh"] * df_cost_ts["sell_price_eur_per_kwh"]
    )

    df_cost_ts["market_mt_net_cost_eur"] = (
        df_cost_ts["market_mt_purchase_cost_eur"]
        + df_cost_ts["market_mt_sale_revenues_eur"]
    )

    df_cost_ts["market_mt_gross_absolute_value_eur"] = (
        df_cost_ts["market_mt_purchase_cost_eur"].abs()
        + df_cost_ts["market_mt_sale_revenues_eur"].abs()
    )

    df_cost_ts["market_mt_net_cost_eur_per_kwh_demand"] = np.where(
        df_cost_ts["total_local_demand_kwh"] > 1e-12,
        df_cost_ts["market_mt_net_cost_eur"] / df_cost_ts["total_local_demand_kwh"],
        np.nan,
    )

    # Readable alias: market settlement actually used in the objective
    # on the LV-household side.
    df_cost_ts["market_bt_objective_settlement_eur"] = (
        df_cost_ts["bt_energy_objective_settlement_eur"]
    )



    # -----------------------------
    # Flexibility / penalties
    # -----------------------------
    df_cost_ts["dhw_boost_cost_eur"] = (
        df_cost_ts["dhw_boost_kwh"] * float(cost_boost_dhw_eur_per_kwh)
    )
    df_cost_ts["bev_outdoor_charge_cost_eur"] = (
        df_cost_ts["bev_outdoor_kwh"] * float(cost_outdoor_charger_eur_per_kwh)
    )

    df_cost_ts["pv_curtail_cost_eur"] = (
        df_cost_ts["pv_curtail_kwh"] * float(PV_curtail_cost)
    )

    # Technical transfer cost on Kerber LV lines.
    # This cost actually enters the objective function through variable_costs=KERBER_LINE_COST.
    df_cost_ts["kerber_line_transfer_cost_eur"] = (
        df_cost_ts["kerber_line_transfer_kwh"] * float(KERBER_LINE_COST)
    )




    # ------------------------------------------------------------
    # LEGACY / DIAGNOSTIC business-formula cost.
    # It always includes the DHW boost penalty.
    # Do NOT use this column as the main economic reporting KPI.
    
    # costs + negative revenues + BEV outdoor + DHW boost + PV curtailment
    # ------------------------------------------------------------
    df_cost_ts["total_system_cost_business_eur"] = (
        df_cost_ts["bt_purchase_cost_eur"]
        + df_cost_ts["bt_sale_revenues_eur"]
        + df_cost_ts["dhw_boost_cost_eur"]
        + df_cost_ts["bev_outdoor_charge_cost_eur"]
        + df_cost_ts["pv_curtail_cost_eur"]
    )


    # ------------------------------------------------------------
    # EFFECTIVE ECONOMIC REPORTING
    # ------------------------------------------------------------

    df_cost_ts = apply_effective_system_cost_reporting(
        df_cost_ts=df_cost_ts,
        include_dhw_boost_in_effective_costs=(
            include_dhw_boost_in_effective_costs
        ),
    )



    # ------------------------------------------------------------
    # Extra terms available in the objective function
    # ------------------------------------------------------------
    # The complete breakdown of variable_costs has already been calculated in:
    # df_objective_terms_ts
    #
    # Here I retrieve some technical items, for example:
    # - KERBER_LINE_COST
    # - other unclassified variable costs
    # - shortage, if activated in the future
    # ------------------------------------------------------------

    obj_terms_aligned = df_objective_terms_ts.reindex(index=idx_real).fillna(0.0).copy()

    objective_extra_cols = [
        "other_variable_cost_terms_eur",
        "shortage_cost_eur",
    ]

    for col in objective_extra_cols:
        if col in obj_terms_aligned.columns:
            df_cost_ts[col] = obj_terms_aligned[col].astype(float)
        else:
            df_cost_ts[col] = 0.0

    df_cost_ts["objective_extra_variable_cost_terms_eur"] = (
        df_cost_ts["kerber_line_transfer_cost_eur"]
        + df_cost_ts[objective_extra_cols].sum(axis=1)
    )

    # System cost = FM formula + technical items actually included
    # among the objective variable_costs.
    df_cost_ts["total_system_cost_including_objective_variable_terms_eur"] = (
        df_cost_ts["total_system_cost_business_eur"]
        + df_cost_ts["objective_extra_variable_cost_terms_eur"]
    )

    # Compact alias for time-series analysis:
    # total cost rebuilt in the single timestep.
    df_cost_ts["total_system_cost_per_timestep_eur"] = (
        df_cost_ts["total_system_cost_including_objective_variable_terms_eur"]
    )

    df_cost_ts["total_system_cost_per_kwh_demand_timestep"] = np.where(
        df_cost_ts["total_local_demand_kwh"] > 1e-12,
        df_cost_ts["total_system_cost_per_timestep_eur"]
        / df_cost_ts["total_local_demand_kwh"],
        np.nan,
    )

    # ------------------------------------------------------------
    # Technical part: objective reconciliation
    # ------------------------------------------------------------
    df_cost_ts["explained_objective_terms_eur"] = (
        df_cost_ts["bt_energy_objective_settlement_eur"]
        + df_cost_ts["dhw_boost_cost_eur"]
        + df_cost_ts["bev_outdoor_charge_cost_eur"]
        + df_cost_ts["pv_curtail_cost_eur"]
        # + df_cost_ts["shortage_cost_eur"]
        + df_cost_ts["objective_extra_variable_cost_terms_eur"]
    )

    objective_total_eur = _safe_objective_value(om)
    explained_terms_eur = float(df_cost_ts["explained_objective_terms_eur"].sum())
    reconstructed_total_system_cost_eur = float(
        df_cost_ts["total_system_cost_including_objective_variable_terms_eur"].sum()
    )

    # After including Kerber line cost, this should be approximately zero,
    # except for very small numerical tolerances.
    objective_gap_vs_reconstructed_total_eur = (
        float(objective_total_eur - reconstructed_total_system_cost_eur)
        if np.isfinite(objective_total_eur) else np.nan
    )

    # This is instead a technical diagnostic:
    # it indicates how much of the objective function is NOT captured by the automatic breakdown
    # of om.flows.variable_costs only.
    # It may include DHW boost, BEV outdoor, PV curtailment or manually reconstructed terms.
    objective_gap_vs_variable_cost_breakdown_eur = (
        float(objective_total_eur - objective_variable_cost_terms_eur)
        if np.isfinite(objective_total_eur) else np.nan
    )

    terms_not_captured_by_variable_cost_breakdown_eur = (
        reconstructed_total_system_cost_eur - objective_variable_cost_terms_eur
    )

    objective_match_tolerance_eur = 1e-4
    objective_match_ok = (
        abs(objective_gap_vs_reconstructed_total_eur) <= objective_match_tolerance_eur
        if np.isfinite(objective_gap_vs_reconstructed_total_eur) else False
    )


    # ------------------------------------------------------------
    # Energy/economic aggregates used in the summary
    # ------------------------------------------------------------
    _total_demand_kwh_sum = float(df_cost_ts["total_local_demand_kwh"].sum())
    _mt_import_gross_kwh_sum = float(df_cost_ts["mt_import_kwh"].sum())
    _mt_export_gross_kwh_sum = float(df_cost_ts["mt_export_kwh"].sum())
    _mt_net_import_kwh_sum = float(df_cost_ts["mt_net_import_kwh"].sum())

    _market_mt_purchase_cost_eur_sum = float(df_cost_ts["market_mt_purchase_cost_eur"].sum())
    _market_mt_sale_revenues_eur_sum = float(df_cost_ts["market_mt_sale_revenues_eur"].sum())
    _market_mt_net_cost_eur_sum = float(df_cost_ts["market_mt_net_cost_eur"].sum())
    _market_mt_gross_absolute_value_eur_sum = float(
        df_cost_ts["market_mt_gross_absolute_value_eur"].sum()
    )

    _average_market_mt_net_cost_eur_per_timestep = float(
        df_cost_ts["market_mt_net_cost_eur"].mean()
    )

    _average_total_system_cost_eur_per_timestep = float(
        df_cost_ts["total_system_cost_per_timestep_eur"].mean()
    )

    _average_market_mt_net_cost_eur_per_kwh_demand = (
        _market_mt_net_cost_eur_sum / _total_demand_kwh_sum
        if _total_demand_kwh_sum > 1e-12
        else np.nan
    )

    _average_market_mt_purchase_price_eur_per_kwh = (
        _market_mt_purchase_cost_eur_sum / _mt_import_gross_kwh_sum
        if _mt_import_gross_kwh_sum > 1e-12
        else np.nan
    )

    _average_market_mt_sale_price_eur_per_kwh = (
        _market_mt_sale_revenues_eur_sum / _mt_export_gross_kwh_sum
        if _mt_export_gross_kwh_sum > 1e-12
        else np.nan
    )

    _gross_mv_import_share_of_demand = (
        _mt_import_gross_kwh_sum / _total_demand_kwh_sum
        if _total_demand_kwh_sum > 1e-12
        else np.nan
    )

    _gross_mv_export_share_of_demand = (
        _mt_export_gross_kwh_sum / _total_demand_kwh_sum
        if _total_demand_kwh_sum > 1e-12
        else np.nan
    )

    _net_mv_import_share_of_demand = (
        _mt_net_import_kwh_sum / _total_demand_kwh_sum
        if _total_demand_kwh_sum > 1e-12
        else np.nan
    )


    # ------------------------------------------------------------
    # Main summary to export in system_cost_summary_*.xlsx files
    # ------------------------------------------------------------
    df_cost_summary = pd.DataFrame([{
        "mode": MODE,
        "season": current_season,
        "bt_archetype": LV_ARCHETYPE,
        "region": selected_region,
        "n_houses": num_households,
        "timesteps": len(df_cost_ts),
        "dt_minutes": float(timestep_minutes),

        # Synthetic scenario metadata
        "use_pf_limits": bool(USE_PF_LINE_LIMITS_V1),
        "mt_import_cap_kw": float(cap_imp_mt_kw) if cap_imp_mt_kw is not None else np.nan,
        "mt_export_cap_kw": float(cap_exp_mt_kw) if cap_exp_mt_kw is not None else np.nan,

        # LV exchanges
        "bt_import_total_kwh": float(df_cost_ts["bt_import_house_kwh"].sum()),
        "bt_export_total_kwh": float(df_cost_ts["bt_export_house_kwh"].sum()),
        "bt_purchase_cost_eur": float(df_cost_ts["bt_purchase_cost_eur"].sum()),
        "bt_sale_revenues_eur": float(df_cost_ts["bt_sale_revenues_eur"].sum()),
        "bt_energy_net_settlement_eur": float(df_cost_ts["bt_energy_net_settlement_eur"].sum()),
        
        # Demand and energy coverage
        "total_local_demand_kwh": _total_demand_kwh_sum,

        # Gross/net exchanges with the MV grid
        "mt_import_total_kwh": _mt_import_gross_kwh_sum,
        "mt_export_total_kwh": _mt_export_gross_kwh_sum,
        "mt_net_import_total_kwh": _mt_net_import_kwh_sum,

        # Explicit aliases for consistency with multi-run functions
        "mv_import_total_kwh": _mt_import_gross_kwh_sum,
        "mv_export_total_kwh": _mt_export_gross_kwh_sum,
        "net_mv_import_kwh_raw": _mt_net_import_kwh_sum,

        # Required KPI: gross import / demand
        "gross_mv_import_share_of_demand": _gross_mv_import_share_of_demand,
        "gross_mv_export_share_of_demand": _gross_mv_export_share_of_demand,
        "net_mv_import_share_of_demand": _net_mv_import_share_of_demand,

        "gross_mv_import_exceeds_demand": bool(
            _mt_import_gross_kwh_sum > _total_demand_kwh_sum + 1e-6
        ),

        "local_self_consumption_kwh": float(df_cost_ts["local_self_consumption_kwh"].sum()),

        # Final electric-demand breakdown by component
        "demand_base_kwh": float(df_cost_ts["demand_base_kwh"].sum()),
        "demand_cooling_kwh": float(df_cost_ts["demand_cooling_kwh"].sum()),
        "demand_heating_kwh": float(df_cost_ts["demand_heating_kwh"].sum()),
        "demand_cooking_kwh": float(df_cost_ts["demand_cooking_kwh"].sum()),
        "demand_ev_charging_kwh": float(df_cost_ts["demand_ev_charging_kwh"].sum()),
        "demand_dhw_storage_kwh": float(df_cost_ts["demand_dhw_storage_kwh"].sum()),
        "demand_dhw_total_kwh": float(df_cost_ts["demand_dhw_total_kwh"].sum()),
        "demand_components_total_kwh": float(df_cost_ts["demand_components_total_kwh"].sum()),
        "demand_components_gap_kwh": float(df_cost_ts["demand_components_gap_kwh"].sum()),

        "n_edges_demand_base_matched": int(len(matched_demand_base)),
        "n_edges_demand_cooling_matched": int(len(matched_demand_cooling)),
        "n_edges_demand_heating_matched": int(len(matched_demand_heating)),
        "n_edges_demand_cooking_matched": int(len(matched_demand_cooking)),
        "n_edges_demand_ev_charging_matched": int(len(matched_demand_ev_charging)),
        "n_edges_demand_dhw_storage_matched": int(len(matched_demand_dhw_storage)),


        "mt_import_share_of_demand": (
            float(df_cost_ts["mt_import_kwh"].sum() / df_cost_ts["total_local_demand_kwh"].sum())
            if float(df_cost_ts["total_local_demand_kwh"].sum()) > 1e-12
            else np.nan
        ),
        "local_self_consumption_share_of_demand": (
            float(df_cost_ts["local_self_consumption_kwh"].sum() / df_cost_ts["total_local_demand_kwh"].sum())
            if float(df_cost_ts["total_local_demand_kwh"].sum()) > 1e-12
            else np.nan
        ),

        "n_edges_total_local_demand_matched": int(len(matched_total_local_demand)),
        "n_edges_mt_import_matched": int(len(matched_mt_import)),
        "n_edges_mt_export_matched": int(len(matched_mt_export)),

        # Costs/revenues
        "total_system_cost_business_eur": float(df_cost_ts["total_system_cost_business_eur"].sum()),
        # Market cost/revenues on gross MV exchanges
        "market_mt_purchase_cost_eur": _market_mt_purchase_cost_eur_sum,
        "market_mt_sale_revenues_eur": _market_mt_sale_revenues_eur_sum,
        "market_mt_net_cost_eur": _market_mt_net_cost_eur_sum,
        "market_mt_gross_absolute_value_eur": _market_mt_gross_absolute_value_eur_sum,

        "average_market_mt_net_cost_eur_per_timestep": _average_market_mt_net_cost_eur_per_timestep,
        "average_total_system_cost_eur_per_timestep": _average_total_system_cost_eur_per_timestep,
        "average_market_mt_net_cost_eur_per_kwh_demand": _average_market_mt_net_cost_eur_per_kwh_demand,
        "average_market_mt_purchase_price_eur_per_kwh": _average_market_mt_purchase_price_eur_per_kwh,
        "average_market_mt_sale_price_eur_per_kwh": _average_market_mt_sale_price_eur_per_kwh,

        # LV-household settlement, i.e. the one closer to the current objective
        "market_bt_objective_settlement_eur": float(
            df_cost_ts["market_bt_objective_settlement_eur"].sum()
        ),




        # DHW boost
        "dhw_boost_total_kwh": float(df_cost_ts["dhw_boost_kwh"].sum()),
        "dhw_boost_active_timesteps": int((df_cost_ts["dhw_boost_kw"] > eps_active_kw).sum()),
        "dhw_boost_cost_eur": float(df_cost_ts["dhw_boost_cost_eur"].sum()),
        "n_houses_dhw_boost_active": int(len(houses_dhw_boost_active)),
        "n_edges_dhw_boost_matched": int(len(matched_dhw_boost)),
        
        "include_dhw_boost_in_effective_costs": bool(include_dhw_boost_in_effective_costs),
        "dhw_boost_cost_effective_eur": float(df_cost_ts["dhw_boost_cost_effective_eur"].sum()),
        "dhw_boost_cost_excluded_eur": float(df_cost_ts["dhw_boost_cost_excluded_eur"].sum()),

        # BEV outdoor
        "bev_outdoor_total_kwh": float(df_cost_ts["bev_outdoor_kwh"].sum()),
        "bev_outdoor_active_timesteps": int((df_cost_ts["bev_outdoor_kw"] > eps_active_kw).sum()),
        "bev_outdoor_charge_cost_eur": float(df_cost_ts["bev_outdoor_charge_cost_eur"].sum()),
        "n_houses_bev_outdoor_active": int(len(houses_bev_outdoor_active)),
        "n_edges_bev_outdoor_matched": int(len(matched_bev_outdoor)),

        # PV curtailment
        "pv_curtail_total_kwh": float(df_cost_ts["pv_curtail_kwh"].sum()),
        "pv_curtail_active_timesteps": int((df_cost_ts["pv_curtail_kw"] > eps_active_kw).sum()),
        "pv_curtail_cost_eur": float(df_cost_ts["pv_curtail_cost_eur"].sum()),
        "n_houses_pv_curtail_active": int(len(houses_pv_curtail_active)),
        "n_edges_pv_curtail_matched": int(len(matched_pv_curtail)),


        # Technical extras actually available among the objective variable_costs
        "kerber_line_transfer_total_kwh": float(df_cost_ts["kerber_line_transfer_kwh"].sum()),
        "kerber_line_transfer_cost_eur": float(df_cost_ts["kerber_line_transfer_cost_eur"].sum()),
        "n_edges_kerber_line_matched": int(len(matched_kerber_line)),

        "other_variable_cost_terms_eur": float(df_cost_ts["other_variable_cost_terms_eur"].sum()),
        "shortage_cost_eur": float(df_cost_ts["shortage_cost_eur"].sum()),
        "objective_extra_variable_cost_terms_eur": float(
            df_cost_ts["objective_extra_variable_cost_terms_eur"].sum()
        ),

        # Rebuilt formula + technical variable objective terms
        "total_system_cost_including_objective_variable_terms_eur": reconstructed_total_system_cost_eur,

        # Values read directly from the objective function
        "objective_variable_cost_terms_eur": objective_variable_cost_terms_eur,

        # Reconciliation diagnostics
        "terms_not_captured_by_variable_cost_breakdown_eur": terms_not_captured_by_variable_cost_breakdown_eur,
        "objective_gap_vs_variable_cost_breakdown_eur": objective_gap_vs_variable_cost_breakdown_eur,
        "objective_gap_vs_reconstructed_total_eur": objective_gap_vs_reconstructed_total_eur,
        "objective_match_ok": bool(objective_match_ok),

        # This is the complete Pyomo/oemof objective function
        "total_system_cost_including_all_objective_terms_eur": objective_total_eur,
        "objective_total_eur": objective_total_eur,

        "total_system_cost_effective_eur": float(df_cost_ts["total_system_cost_effective_eur"].sum()),          # Cost of the system WITHOUT expenses due to the use of DHW_boost
        "total_system_cost_simulation_eur": objective_total_eur,


    }])


    # ------------------------------------------------------------
    # Separate technical sheet: kept for debugging/reconciliation
    # ------------------------------------------------------------
    df_objective_reconciliation = pd.DataFrame([{
        # Value read directly from the Pyomo/oemof objective
        "objective_total_eur": objective_total_eur,

        # Automatic breakdown of om.flows with variable_costs only
        "objective_variable_cost_terms_eur": objective_variable_cost_terms_eur,

        # Complete manual reconstruction: business cost + extra objective terms
        "reconstructed_total_system_cost_eur": reconstructed_total_system_cost_eur,

        # Terms explained in df_cost_ts
        "explained_objective_terms_eur": explained_terms_eur,

        # Difference between the Pyomo/oemof objective and the complete reconstruction.
        # This is the main diagnostic: it should tend to zero.
        "objective_gap_vs_reconstructed_total_eur": objective_gap_vs_reconstructed_total_eur,

        # Difference between the Pyomo/oemof objective and the automatic variable-cost breakdown.
        # It may differ from zero if some terms are reconstructed manually.
        "objective_gap_vs_variable_cost_breakdown_eur": objective_gap_vs_variable_cost_breakdown_eur,

        # Terms included in the manual reconstruction but not captured by the automatic breakdown.
        "terms_not_captured_by_variable_cost_breakdown_eur": terms_not_captured_by_variable_cost_breakdown_eur,

        # Synthetic boolean check
        "objective_match_ok": bool(objective_match_ok),

        # Main component details
        "bt_energy_objective_settlement_eur": float(df_cost_ts["bt_energy_objective_settlement_eur"].sum()),
        "dhw_boost_cost_eur": float(df_cost_ts["dhw_boost_cost_eur"].sum()),
        "bev_outdoor_charge_cost_eur": float(df_cost_ts["bev_outdoor_charge_cost_eur"].sum()),
        "pv_curtail_cost_eur": float(df_cost_ts["pv_curtail_cost_eur"].sum()),
        "kerber_line_transfer_cost_eur": float(df_cost_ts["kerber_line_transfer_cost_eur"].sum()),
        "other_variable_cost_terms_eur": float(df_cost_ts["other_variable_cost_terms_eur"].sum()),
        "shortage_cost_eur": float(df_cost_ts["shortage_cost_eur"].sum()),
    }])

    return (
        df_cost_ts,
        df_cost_summary,
        df_objective_terms_ts,
        df_objective_terms_by_flow,
        df_objective_terms_summary,
        df_objective_reconciliation,
    )
# -------------------------------------------------------------------------------------------------




#-------------------------------------------------------------------
#       Sec 4.6 --8.4) FIXED/RB PROFILE ADAPTERS
#-------------------------------------------------------------------
# Function to export DHW-boiler and BEV profiles to compare the RB configuration with the OPT case, since the built components are different
def _rb_profile_series_kw(rb_prof, key, c, idx_real, timeindex_canonico):
    """
    Extracts from rb_prof[key][c] a kW Series aligned to timeindex_canonico.
    If the profile does not exist, returns zeros.
    """
    if (
        rb_prof is not None
        and key in rb_prof
        and isinstance(rb_prof[key], pd.DataFrame)
        and c in rb_prof[key].columns
    ):
        s = to_canonical(rb_prof[key][c], idx_real, timeindex_canonico)
        return pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)

    return pd.Series(0.0, index=timeindex_canonico, dtype=float)



# Function to export the same exogenous demand used in the two RB and OPT configurations, i.e. DHW and BEV demand
def _fixed_input_series_kw(fix_dicts, pnom_dicts, current_season, c, idx_real, timeindex_canonico):
    """
    Rebuilds the model input profile as fix * P_nom, in kW.
    Useful for comparing RB vs OPT on exogenous demands.
    """
    if (
        current_season in fix_dicts
        and current_season in pnom_dicts
        and c in fix_dicts[current_season]
        and c in pnom_dicts[current_season]
    ):
        s_fix = pd.to_numeric(fix_dicts[current_season][c], errors="coerce").fillna(0.0).astype(float)
        p_nom = float(pnom_dicts[current_season][c])
        s_kw = s_fix * p_nom
        s_kw = to_canonical(s_kw, idx_real, timeindex_canonico)
        return pd.to_numeric(s_kw, errors="coerce").fillna(0.0).astype(float)

    return pd.Series(0.0, index=timeindex_canonico, dtype=float)

   ### 4.6.9 F(x) SOC (BESS, EV, ESWHs) and comparison SOC between two setups

In [ ]:
#-------------------------------------------------------------------
#       Sec 4.6 -- 9.1) SOC PATHS AND SAVING
#-------------------------------------------------------------------
# *** NEW VERSIONS FOR MIXED CONFIGURATIONS
# Function to define the directory from which SOC files are retrieved
def soc_output_path(tech, season, base_dir=None):
    """
    Compact path for SOC files.

    The configuration is in the MODE folder.
    The file contains only technology + season.
    """
    if base_dir is None:
        base_dir = optimization_output_folder

    tech = str(tech).upper()
    if tech not in SOC_FILE_STEMS:
        raise KeyError(f"Unrecognized SOC technology: {tech}")

    return os.path.join(base_dir, f"{SOC_FILE_STEMS[tech]}_{season}.xlsx")


# Function to save SOC files in .xlsx format
def save_soc_excel(
    soc_kwh,
    soc_pct,
    output_path,
    meta=None,
    boundary_df=None,
    extra_sheets=None,
):
    """
    Saves SOC in kWh and SOC percentage to Excel.

    extra_sheets is an optional mapping:
        {sheet_name: DataFrame}

    It is used only for diagnostic time series
    (e.g. intermediate RB-DHW storage states).
    """

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True,
    )

    if soc_kwh is None:
        soc_kwh = pd.DataFrame()

    if soc_pct is None:
        soc_pct = pd.DataFrame()

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
    ) as writer:

        # Standard SOC outputs
        soc_kwh.to_excel(
            writer,
            sheet_name="SOC_kWh",
            index=True,
        )

        soc_pct.to_excel(
            writer,
            sheet_name="SOC_pct",
            index=True,
        )

        # Optional initial/final storage boundary information
        if boundary_df is not None:

            boundary_df.to_excel(
                writer,
                sheet_name="SOC_boundary",
                index=False,
            )

        # Optional diagnostic time series
        if extra_sheets:

            reserved = {
                "SOC_kWh",
                "SOC_pct",
                "SOC_boundary",
                "meta",
            }

            for sheet_name, df_extra in extra_sheets.items():

                if df_extra is None:
                    continue

                sheet_name = str(sheet_name)[:31]

                if sheet_name in reserved:
                    raise ValueError(
                        "Reserved SOC sheet name used "
                        f"in extra_sheets: {sheet_name}"
                    )

                if isinstance(df_extra, pd.Series):
                    df_extra = df_extra.to_frame()

                if not isinstance(
                    df_extra,
                    pd.DataFrame,
                ):
                    raise TypeError(
                        f"extra_sheets[{sheet_name!r}] "
                        "must be a pandas DataFrame/Series, "
                        f"got {type(df_extra).__name__}."
                    )

                df_extra.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    index=True,
                )

        # Metadata
        if meta is not None:

            pd.DataFrame(
                [meta]
            ).to_excel(
                writer,
                sheet_name="meta",
                index=False,
            )

    print(
        f"[INFO][SOC] Saved: {output_path}"
    )
    

# Function to extract SOC for the conventional/RB configuration of the technology
def rb_soc_df_from_profile(rb_prof, key, idx_real, columns=None):
    """
    Extracts an RB SOC dataframe from rb_prof and safely realigns it.
    """
    if rb_prof is None or key not in rb_prof:
        return pd.DataFrame(index=idx_real)

    df = rb_prof[key]

    if isinstance(df, pd.Series):
        df = df.to_frame()

    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame(index=idx_real)

    out = df.reindex(idx_real).fillna(0.0).astype(float)

    if columns is not None:
        cols = [c for c in columns if c in out.columns]
        out = out.loc[:, cols] if cols else out.iloc[:, 0:0]

    return out
# -----------------------------------------------------------


#-------------------------------------------------------------------
#       Sec 4.6 -- 9.2) STORAGE COLUMN SEARCH
#-------------------------------------------------------------------
# Function to identify flows in the list of oemof results
def find_flow_columns_for_house_by_text(df, house_id):
    """
    Searches the columns of `df` for flow columns related to house_id.
    Returns (charge_col, discharge_col), where:
      - charge_col    -> flow (bus_el -> BESS)  [charge]
      - discharge_col -> flow (BESS -> bus_el)  [discharge]
    Works with MultiIndex columns with any number of levels.
    """
    charge_col = None
    discharge_col = None

    patt_bess = re.compile(re.escape(f"{house_id}_BESS"), re.I)
    patt_bus  = re.compile(re.escape(f"{house_id}_bus_el"), re.I)

    for col in df.columns:
        if isinstance(col, tuple):
            s = ' | '.join([str(x) for x in col if x is not None])
        else:
            s = str(col)

        found_bess = patt_bess.search(s)
        found_bus  = patt_bus.search(s)

        if found_bess and found_bus:
            if found_bess.start() < found_bus.start():
                # BESS appears first -> (BESS, bus_el) => discharge
                if discharge_col is None:
                    discharge_col = col
            else:
                # bus_el appears first -> (bus_el, BESS) => charge
                if charge_col is None:
                    charge_col = col

    return charge_col, discharge_col


# Function to identify BEV charge/discharge flows
def find_flow_columns_for_house_bev(df, house_id):
    """
    Finds BEV flow columns for a given household.

    Expected flows, as from oemof results:

      - charge:     (('c_i_bus_el_charge_BEV', 'c_i_BEV_storage'), 'flow')
      - discharge:  (('c_i_BEV_storage', 'c_i_bus_el_discharge_BEV'), 'flow')

    `df` is the flow dataframe, e.g. df_raw_case, with MultiIndex columns.
    Returns:
      - charge_col: column (bus_el_charge_BEV -> BEV_storage)
      - discharge_col: column (BEV_storage -> bus_el_discharge_BEV)
    """

    charge_col = None
    discharge_col = None

    patt_storage       = re.escape(f"{house_id}_BEV_storage")
    patt_bus_charge    = re.escape(f"{house_id}_bus_el_charge_BEV")
    patt_bus_discharge = re.escape(f"{house_id}_bus_el_discharge_BEV")

    for col in df.columns:
        if isinstance(col, tuple):
            s = ' | '.join([str(x) for x in col if x is not None])
        else:
            s = str(col)

        # charge: storage node + bus_el_charge_BEV
        if (re.search(patt_storage, s) and re.search(patt_bus_charge, s)) and (charge_col is None):
            charge_col = col

        # discharge: storage node + bus_el_discharge_BEV
        if (re.search(patt_storage, s) and re.search(patt_bus_discharge, s)) and (discharge_col is None):
            discharge_col = col

    return charge_col, discharge_col


# Function to identify charge/discharge flows for DHW thermal storage
def find_dhw_flow_columns_for_house(df, house_id):
    """
    Searches the columns of `df` for DHW flows related to house_id.
    Returns (charge_col, discharge_col), where:
      - charge_col    -> (c_i_bus_el -> c_i_DHW_storage)
      - discharge_col -> (c_i_DHW_storage -> c_i_bus_DHW_use)
    Works with MultiIndex columns and nested tuples such as:
        (('c_1_DHW_storage', 'c_1_bus_DHW_use'), 'flow')
    """

    def flatten_tuple(x):
        """Flattens nested tuples into a list of strings."""
        out = []
        if isinstance(x, tuple):
            for el in x:
                out.extend(flatten_tuple(el))
        elif x is not None:
            out.append(str(x))
        return out

    charge_col = None
    discharge_col = None

    storage_tag = f"{house_id}_DHW_storage"
    bus_el_tag  = f"{house_id}_bus_el"
    bus_dhw_tag = f"{house_id}_bus_DHW_use"

    for col in df.columns:
        # col may be:
        #   ('something', ( 'c_1_DHW_storage', 'c_1_bus_DHW_use'), 'flow')
        #   (('c_1_DHW_storage', 'c_1_bus_DHW_use'), 'flow')
        #   '((c_1_DHW_storage, c_1_bus_DHW_use), flow)'
        tokens = flatten_tuple(col)
        s = " | ".join(tokens)

        has_storage = storage_tag in s
        has_bus_el  = bus_el_tag  in s
        has_bus_dhw = bus_dhw_tag in s

        # bus_el <-> DHW_storage  -> charge
        if has_storage and has_bus_el and not has_bus_dhw:
            if charge_col is None:
                charge_col = col

        # DHW_storage <-> bus_DHW_use -> discharge
        if has_storage and has_bus_dhw and not has_bus_el:
            if discharge_col is None:
                discharge_col = col

    # Safety fallback: if discharge was not found,
    # search for ANY column with storage + bus_DHW_use.
    if discharge_col is None:
        for col in df.columns:
            tokens = flatten_tuple(col)
            s = " | ".join(tokens)
            if (storage_tag in s) and (bus_dhw_tag in s):
                discharge_col = col
                break
    
    return charge_col, discharge_col
# -------------------------------------------------------------


# -------------------------------------------------------------------
# Sec 4.6 -- 9.3) SOC EXTRACTION DIRECTLY FROM OEMOF RESULTS
# -------------------------------------------------------------------

def extract_opt_storage_soc(
    results,
    storage_node,
    capacity_kwh,
    idx_real,
    storage_label=None,
    initial_storage_level=None,
    tolerance=1e-6,
):
    """
    Extracts the optimized storage content directly from oemof.solph
    results for a GenericStorage.

    Parameters
    ----------
    results : dict
        Output of solph.processing.results(om).

    storage_node : oemof GenericStorage
        The actual storage object contained in noded[].

    capacity_kwh : float
        Nominal storage capacity [kWh].

    idx_real : pd.DatetimeIndex
        Real time index used for Excel output / plotting.

    storage_label : str, optional
        Human-readable label used in warnings/errors.

    initial_storage_level : float or None
        Relative initial storage level used in GenericStorage.
        - If numeric, initial content is known directly.
        - If None, initial content is optimized by oemof and is
          retrieved from the result scalars when available.

    tolerance : float
        Numerical tolerance used for the balanced-storage check.

    Returns
    -------
    soc_kwh : pd.Series
        Storage content [kWh], one value per plotting timestep.

    soc_pct : pd.Series
        Storage content [%].

    boundary_info : dict
        Initial/final SOC information used for checks/export.
    """

    if storage_label is None:
        storage_label = str(getattr(storage_node, "label", storage_node))

    capacity_kwh = float(capacity_kwh)

    if capacity_kwh <= 0:
        raise ValueError(
            f"[SOC] {storage_label}: capacity must be > 0, "
            f"got {capacity_kwh}."
        )

    # --------------------------------------------------------------
    # 1) Locate the GenericStorage result
    #
    # In processing.results(), storage state variables are normally
    # stored under:
    #     results[(storage_node, None)]
    # --------------------------------------------------------------

    result_key = (storage_node, None)

    if result_key not in results:
        raise KeyError(
            f"[SOC] {storage_label}: storage result key "
            f"(storage_node, None) not found."
        )

    storage_result = results[result_key]

    if "sequences" not in storage_result:
        raise KeyError(
            f"[SOC] {storage_label}: no 'sequences' found "
            f"in optimization results."
        )

    seq = storage_result["sequences"]


    # --------------------------------------------------------------
    # 2) Find storage_content
    # --------------------------------------------------------------

    storage_content_col = None

    for col in seq.columns:

        # works for:
        #   "storage_content"
        #   ("storage_content",)
        #   other MultiIndex-like variants
        if "storage_content" in str(col):

            storage_content_col = col
            break


    if storage_content_col is None:
        raise KeyError(
            f"[SOC] {storage_label}: 'storage_content' not found. "
            f"Available columns: {list(seq.columns)}"
        )


    soc_points = pd.to_numeric(
        seq[storage_content_col],
        errors="coerce",
    ).dropna().astype(float)


    if soc_points.empty:
        raise ValueError(
            f"[SOC] {storage_label}: storage_content is empty."
        )


    # --------------------------------------------------------------
    # 3) Retrieve initial storage content
    #
    # If initial_storage_level was fixed in GenericStorage, use it.
    #
    # If initial_storage_level=None, oemof optimizes init_content.
    # In that case try to retrieve the scalar directly.
    # --------------------------------------------------------------

    if initial_storage_level is not None:

        initial_content_kwh = (
            float(initial_storage_level)
            * capacity_kwh
        )

    else:

        initial_content_kwh = np.nan

        scalars = storage_result.get(
            "scalars",
            pd.Series(dtype=float),
        )

        if scalars is not None and len(scalars) > 0:

            for idx in scalars.index:

                if "init_content" in str(idx):

                    try:
                        initial_content_kwh = float(
                            scalars.loc[idx]
                        )
                    except Exception:
                        pass

                    break


        # Fallback:
        # in many result structures the first storage_content point
        # is already the initial boundary state.
        if pd.isna(initial_content_kwh):

            initial_content_kwh = float(
                soc_points.iloc[0]
            )


    # --------------------------------------------------------------
    # 4) Identify final boundary value and construct the series
    #    used for plots / Excel.
    #
    # Storage content is defined at points in time, whereas flows
    # are defined over intervals. Therefore oemof may return one
    # more storage point than flow timesteps.
    # --------------------------------------------------------------

    n_plot = len(idx_real)
    n_soc = len(soc_points)


    if n_soc == n_plot + 1:

        # Standard case:
        #
        # boundary states:
        # E_0, E_1, ..., E_N
        #
        # plotting series:
        # E_0, ..., E_(N-1)
        #
        # final boundary:
        # E_N

        final_content_kwh = float(
            soc_points.iloc[-1]
        )

        soc_plot_values = (
            soc_points.iloc[:-1]
            .to_numpy(dtype=float)
        )


    elif n_soc == n_plot:

        # Depending on solph version / result processing,
        # storage_content can already have the same number
        # of entries as the requested plotting index.

        soc_plot_values = (
            soc_points
            .to_numpy(dtype=float)
        )

        final_content_kwh = float(
            soc_points.iloc[-1]
        )


    else:

        raise ValueError(
            f"[SOC] {storage_label}: unexpected number of "
            f"storage_content values.\n"
            f"storage_content = {n_soc}\n"
            f"idx_real        = {n_plot}\n"
            f"Expected either n or n+1 values."
        )


    # --------------------------------------------------------------
    # 5) Build kWh and percentage series
    # --------------------------------------------------------------

    soc_kwh = pd.Series(
        soc_plot_values,
        index=idx_real,
        name=f"{storage_label}_SOC_kWh",
        dtype=float,
    )

    soc_pct = (
        soc_kwh
        / capacity_kwh
        * 100.0
    )

    soc_pct.name = (
        f"{storage_label}_SOC_pct"
    )


    # --------------------------------------------------------------
    # 6) Balanced-storage consistency check
    # --------------------------------------------------------------

    delta_kwh = (
        final_content_kwh
        - initial_content_kwh
    )

    balanced_ok = (
        abs(delta_kwh) <= tolerance
    )


    if not balanced_ok:

        print(
            f"[WARN SOC BALANCE] {storage_label}: "
            f"SOC_initial={initial_content_kwh:.8f} kWh | "
            f"SOC_final={final_content_kwh:.8f} kWh | "
            f"delta={delta_kwh:.3e} kWh"
        )


    boundary_info = {
        "storage": storage_label,
        "capacity_kWh": capacity_kwh,
        "SOC_initial_kWh": initial_content_kwh,
        "SOC_initial_pct": (
            initial_content_kwh
            / capacity_kwh
            * 100.0
        ),
        "SOC_final_kWh": final_content_kwh,
        "SOC_final_pct": (
            final_content_kwh
            / capacity_kwh
            * 100.0
        ),
        "delta_SOC_kWh": delta_kwh,
        "balanced_check": balanced_ok,
        "n_storage_content_points": n_soc,
        "n_exported_points": n_plot,
    }

    return (
        soc_kwh,
        soc_pct,
        boundary_info,
    )





# Function for the actual calculation of BESS state of charge for individual customers
def calculate_bess_soc_by_season(
        house_flows_df,
        season,
        soc_output_folder,
        capacity_kwh_default,
        power_limit_kw_default,
        eta_charge_default,
        eta_discharge_default,
        initial_soc_kwh_default=0.0,
        capacity_kwh_by_house=None,
        power_limit_kw_by_house=None,
        eta_charge_by_house=None,
        eta_discharge_by_house=None,
        initial_soc_kwh_by_house=None,
    ):
    """
    Calculates the SoC (kWh and %) of BESS batteries for each household (c_1, c_2, ...)
    starting from the electric-flow dataframe 'df_flows_case' (Flow Electric).

    Considers only households that actually have a BESS:
      - if capacity_kwh_by_house is a dict, use only the households available there;
      - otherwise use the default value for all households.
    """

    # 1) timestep in hours
    if isinstance(house_flows_df.index, pd.DatetimeIndex) and len(house_flows_df.index) > 1:
        dt_hours = (house_flows_df.index[1] - house_flows_df.index[0]).total_seconds() / 3600.0
    else:
        dt_hours = timestep_minutes / 60.0  # fallback

    # 2) normalize column names to strings and identify households
    cols = list(house_flows_df.columns)
    cols_str_map = {orig: col_to_str(orig) for orig in cols}
    houses = extract_house_ids(cols_str_map.values())

    if len(houses) == 0:
        print(f"[WARN BESS] Season {season}: no 'c_i' found in flows.")
        return

    # 3) Output DataFrame (SoC in kWh) + memory of the capacity actually used
    soc_kwh = pd.DataFrame(index=house_flows_df.index)
    cap_used = {}

    # 4) loop over households
    for h in houses:
        # BESS capacity for this household
        if capacity_kwh_by_house is not None:
            cap_kwh = capacity_kwh_by_house.get(h, None)
        else:
            cap_kwh = capacity_kwh_default

        # if there is no capacity > 0, no BESS for this household
        if (cap_kwh is None) or pd.isna(cap_kwh) or (cap_kwh <= 0):
            continue

        cap_kwh = float(cap_kwh)
        cap_used[h] = cap_kwh

        # other household-specific parameters
        p_lim_kw = (power_limit_kw_by_house or {}).get(h, power_limit_kw_default)
        eta_ch   = (eta_charge_by_house   or {}).get(h, eta_charge_default)
        eta_dis  = (eta_discharge_by_house or {}).get(h, eta_discharge_default)

        # initial SoC
        soc0 = (initial_soc_kwh_by_house or {}).get(h, initial_soc_kwh_default)
        if soc0 is None:
            soc0 = 0.5 * cap_kwh  # fallback: half capacity
        soc0 = float(soc0)

        # charge/discharge columns
        charge_col, discharge_col = find_flow_columns_for_house_by_text(house_flows_df, h)
        if charge_col is None and discharge_col is None:
            continue


        if charge_col is None:
            charge_series = pd.Series(0.0, index=house_flows_df.index)
        else:
            charge_series = house_flows_df[charge_col].astype(float).fillna(0.0)

        if discharge_col is None:
            discharge_series = pd.Series(0.0, index=house_flows_df.index)
        else:
            discharge_series = house_flows_df[discharge_col].astype(float).fillna(0.0)

        # time integration of the SoC
        soc_list = []
        soc_prev = soc0

        for t in range(len(house_flows_df)):
            p_charge    = max(0.0, charge_series.iloc[t])
            p_discharge = max(0.0, discharge_series.iloc[t])

            # power limits, if defined
            if (p_lim_kw is not None) and (p_lim_kw > 0):
                p_charge    = min(p_charge, p_lim_kw)
                p_discharge = min(p_discharge, p_lim_kw)

            # "safe" efficiencies
            eta_ch_eff  = eta_ch  if (eta_ch  is not None and eta_ch  > 0) else 1.0
            eta_dis_eff = eta_dis if (eta_dis is not None and eta_dis > 0) else 1.0

            # energy (kWh)
            energy_in  = p_charge    * dt_hours * eta_ch_eff      # stored
            energy_out = p_discharge * dt_hours / eta_dis_eff     # extracted

            # capacity / SoC limits
            actual_energy_in  = min(energy_in,  cap_kwh - soc_prev)
            actual_energy_out = min(energy_out, soc_prev)

            soc_new = soc_prev + actual_energy_in - actual_energy_out
            soc_new = min(max(soc_new, 0.0), cap_kwh)

            soc_list.append(soc_new)
            soc_prev = soc_new

        soc_kwh[h + '_SOC_kWh'] = soc_list

    if soc_kwh.empty:
        print(f"[WARN BESS] Season {season}: no BESS SoC calculated, no household with capacity > 0.")
        return

    # 5) percentages by household, using the effective capacity of that household
    soc_pct = pd.DataFrame(index=soc_kwh.index)
    for h, cap_kwh in cap_used.items():
        col_kwh = h + '_SOC_kWh'
        if col_kwh in soc_kwh.columns and cap_kwh > 0:
            soc_pct[h + '_SOC_pct'] = soc_kwh[col_kwh] / cap_kwh * 100.0

    # 6) save to Excel
    if not os.path.exists(soc_output_folder):
        os.makedirs(soc_output_folder, exist_ok=True)

    output_path = soc_output_path("BESS", season, base_dir=soc_output_folder)

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        soc_kwh.to_excel(writer, sheet_name='SOC_kWh', index=True)
        soc_pct.to_excel(writer, sheet_name='SOC_pct', index=True)

    print(f"[INFO] BESS SoC saved in: {output_path}")


# Function for BEV SOC calculation for each season
def calculate_ev_soc_by_season(
        house_flows_df,
        season,
        soc_output_folder,
        capacity_kwh_default,
        power_limit_kw_default,
        eta_charge_default,
        eta_discharge_default,
        loss_rate_default,
        initial_soc_kwh_default=None,
        capacity_kwh_by_house=None,
        power_limit_kw_by_house=None,
        eta_charge_by_house=None,
        eta_discharge_by_house=None,
        loss_rate_by_house=None,
        initial_soc_kwh_by_house=None,
    ):
    """
    Calculates the SoC (kWh and %) of BEV batteries for each household (c_1, c_2, ...).

    Considers only households with BEV capacity > 0, via capacity_kwh_by_house or default.
    """

    # 1) timestep in hours
    if isinstance(house_flows_df.index, pd.DatetimeIndex) and len(house_flows_df.index) > 1:
        dt_hours = (house_flows_df.index[1] - house_flows_df.index[0]).total_seconds() / 3600.0
    else:
        dt_hours = timestep_minutes / 60.0

    # 2) households
    cols = list(house_flows_df.columns)
    cols_str_map = {orig: col_to_str(orig) for orig in cols}
    houses = extract_house_ids(cols_str_map.values())

    if len(houses) == 0:
        print(f"[WARN BEV] Season {season}: no 'c_i' found in BEV flows.")
        return

    soc_kwh = pd.DataFrame(index=house_flows_df.index)
    houses_effettive = []

    # 4) loop over households
    for h in houses:
        if capacity_kwh_by_house is not None:
            cap_kwh = capacity_kwh_by_house.get(h, None)
        else:
            cap_kwh = capacity_kwh_default

        if (cap_kwh is None) or pd.isna(cap_kwh) or (cap_kwh <= 0):
            # household without BEV
            continue

        cap_kwh   = float(cap_kwh)
        p_lim_kw  = (power_limit_kw_by_house or {}).get(h, power_limit_kw_default)
        eta_ch    = (eta_charge_by_house    or {}).get(h, eta_charge_default)
        eta_dis   = (eta_discharge_by_house or {}).get(h, eta_discharge_default)
        loss_rate = (loss_rate_by_house     or {}).get(h, loss_rate_default)

        # initial SoC in kWh, default 0 kWh
        soc0 = (initial_soc_kwh_by_house or {}).get(h, initial_soc_kwh_default)
        if soc0 is None:
            soc0 = 0.0
        soc0 = float(soc0)

        # BEV charge/discharge columns
        charge_col, discharge_col = find_flow_columns_for_house_bev(house_flows_df, h)
        if charge_col is None and discharge_col is None:
            continue


        if charge_col is None:
            charge_series = pd.Series(0.0, index=house_flows_df.index)
        else:
            charge_series = house_flows_df[charge_col].astype(float).fillna(0.0)

        if discharge_col is None:
            discharge_series = pd.Series(0.0, index=house_flows_df.index)
        else:
            discharge_series = house_flows_df[discharge_col].astype(float).fillna(0.0)

        # time integration
        soc_list = []
        soc_prev = soc0

        loss_rate_eff = 0.0 if loss_rate is None else loss_rate
        eta_ch_eff    = eta_ch  if (eta_ch  is not None and eta_ch  > 0) else 1.0
        eta_dis_eff   = eta_dis if (eta_dis is not None and eta_dis > 0) else 1.0

        for t in range(len(house_flows_df)):
            p_charge    = max(0.0, charge_series.iloc[t])     # kW entering storage
            p_discharge = max(0.0, discharge_series.iloc[t])  # kW leaving storage

            # if p_lim_kw is not None and p_lim_kw > 0:
            #     p_charge    = min(p_charge,    p_lim_kw)
            #     p_discharge = min(p_discharge, p_lim_kw)

            # step 1: losses on stored content
            soc_after_losses = soc_prev * (1.0 - loss_rate_eff)

            # step 2: charged/discharged energy
            energy_in  = p_charge    * dt_hours * eta_ch_eff
            energy_out = p_discharge * dt_hours / eta_dis_eff

            # step 3: capacity limits
            actual_energy_in  = min(energy_in,  cap_kwh - soc_after_losses)
            actual_energy_out = min(energy_out, soc_after_losses)

            soc_new = soc_after_losses + actual_energy_in - actual_energy_out
            soc_new = min(max(soc_new, 0.0), cap_kwh)

            soc_list.append(soc_new)
            soc_prev = soc_new

        soc_kwh[h + '_SOC_BEV_kWh'] = soc_list
        houses_effettive.append(h)

    if len(houses_effettive) == 0:
        print(f"[WARN BEV] Season {season}: no BEV with valid capacity found.")
        return

    # 5) percentages for each household
    soc_pct = pd.DataFrame(index=house_flows_df.index)
    for h in houses_effettive:
        if capacity_kwh_by_house is not None:
            cap_kwh = capacity_kwh_by_house.get(h, capacity_kwh_default)
        else:
            cap_kwh = capacity_kwh_default

        col_kwh = h + '_SOC_BEV_kWh'
        col_pct = h + '_SOC_BEV_pct'

        if col_kwh in soc_kwh.columns and cap_kwh not in (None, 0):
            soc_pct[col_pct] = soc_kwh[col_kwh] / cap_kwh * 100.0

    # 6) saving
    if not os.path.exists(soc_output_folder):
        os.makedirs(soc_output_folder, exist_ok=True)

    output_path = soc_output_path("BEV", season, base_dir=soc_output_folder)

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        soc_kwh.to_excel(writer, sheet_name='SOC_kWh', index=True)
        soc_pct.to_excel(writer, sheet_name='SOC_pct', index=True)

    print(f"[INFO] BEV SoC saved in: {output_path}")



# Function for DHW boiler SOC calculation for each season
def calculate_dhw_soc_by_season(
        house_flows_df,
        season,
        soc_output_folder,
        capacity_kwh_default,
        power_limit_kw_default,
        eta_charge_default,
        eta_discharge_default,
        loss_rate_default        = 0.0,
        initial_soc_kwh_default  = None,
        capacity_kwh_by_house    = None,
        power_limit_kw_by_house  = None,
        power_limit_out_kw_default = None,
        power_limit_out_kw_by_house = None,
        eta_charge_by_house      = None,
        eta_discharge_by_house   = None,
        loss_rate_by_house       = None,
        initial_soc_kwh_by_house = None,
    ):
    """
    Calculates the SoC (kWh and %) of the DHW electric boiler for each household (c_1, c_2, ...).
    Uses the flows:
        - charge:     (c_i_bus_el -> c_i_DHW_storage)
        - discharge:  (c_i_DHW_storage -> c_i_bus_DHW_use)

    Consistency with GenericStorage:
      - loss_rate is interpreted as an HOURLY relative loss
      - the input power limit can be different from the output one
      - if no discharge limit is provided, discharge is NOT clamped
    """

    # 1) dt in hours
    if isinstance(house_flows_df.index, pd.DatetimeIndex) and len(house_flows_df.index) > 1:
        dt_hours = (house_flows_df.index[1] - house_flows_df.index[0]).total_seconds() / 3600.0
    else:
        dt_hours = timestep_minutes / 60.0

    # 2) which households (c_i) appear in the flows?
    cols_str_map = {col: col_to_str(col) for col in house_flows_df.columns}
    houses = extract_house_ids(cols_str_map.values())

    if len(houses) == 0:
        print(f"[WARN DHW] Season {season}: no household found in flows.")
        return

    soc_kwh = pd.DataFrame(index=house_flows_df.index)
    cap_used = {}

    for h in houses:

        if capacity_kwh_by_house is not None:
            cap_kwh = capacity_kwh_by_house.get(h, None)
        else:
            cap_kwh = capacity_kwh_default

        if (cap_kwh is None) or pd.isna(cap_kwh) or (cap_kwh <= 0):
            continue

        cap_kwh = float(cap_kwh)
        cap_used[h] = cap_kwh

        p_lim_in_kw  = (power_limit_kw_by_house or {}).get(h, power_limit_kw_default)
        p_lim_out_kw = (power_limit_out_kw_by_house or {}).get(h, power_limit_out_kw_default)
        eta_ch       = (eta_charge_by_house or {}).get(h, eta_charge_default)
        eta_dis      = (eta_discharge_by_house or {}).get(h, eta_discharge_default)
        loss_rate    = (loss_rate_by_house or {}).get(h, loss_rate_default)

        eta_in  = float(eta_ch if eta_ch is not None else 1.0)
        eta_out = float(eta_dis if (eta_dis is not None and eta_dis > 0) else 1.0)

        # effective timestep loss, consistent with GenericStorage
        step_loss = effective_loss_fraction_over_timestep(loss_rate, dt_hours)

        # initial SoC
        if initial_soc_kwh_by_house and (h in initial_soc_kwh_by_house):
            soc0 = float(initial_soc_kwh_by_house[h])
        else:
            if initial_soc_kwh_default is not None:
                soc0 = float(initial_soc_kwh_default)
            else:
                soc0 = cap_kwh

        soc_prev = float(min(max(0.0, soc0), cap_kwh))

        charge_col, discharge_col = find_dhw_flow_columns_for_house(house_flows_df, h)
        if charge_col is None and discharge_col is None:
            continue

        if charge_col is None:
            charge_series = pd.Series(0.0, index=house_flows_df.index)
        else:
            charge_series = house_flows_df[charge_col].astype(float).fillna(0.0)

        if discharge_col is None:
            discharge_series = pd.Series(0.0, index=house_flows_df.index)
        else:
            discharge_series = house_flows_df[discharge_col].astype(float).fillna(0.0)

        soc_list = []

        for t in range(len(house_flows_df)):
            p_charge = max(0.0, charge_series.iloc[t])
            p_discharge = max(0.0, discharge_series.iloc[t])

            # charge limit only
            if (p_lim_in_kw is not None) and (p_lim_in_kw > 0):
                p_charge = min(p_charge, p_lim_in_kw)

            # discharge limit only if explicitly provided
            if (p_lim_out_kw is not None) and (p_lim_out_kw > 0):
                p_discharge = min(p_discharge, p_lim_out_kw)

            soc_after_loss = soc_prev * (1.0 - step_loss)

            energy_in  = p_charge    * dt_hours * eta_in
            energy_out = p_discharge * dt_hours / eta_out

            actual_energy_in  = min(energy_in,  max(0.0, cap_kwh - soc_after_loss))
            actual_energy_out = min(energy_out, max(0.0, soc_after_loss))

            soc_new = soc_after_loss + actual_energy_in - actual_energy_out
            soc_new = min(max(soc_new, 0.0), cap_kwh)

            soc_list.append(soc_new)
            soc_prev = soc_new

        soc_kwh[h + '_SOC_DHW_kWh'] = soc_list

    if soc_kwh.empty:
        print(f"[WARN DHW] Season {season}: no DHW SoC calculated, no household with boiler.")
        return

    soc_pct = pd.DataFrame(index=soc_kwh.index)
    for h, cap_kwh in cap_used.items():
        col = h + '_SOC_DHW_kWh'
        if col in soc_kwh.columns and cap_kwh > 0:
            soc_pct[h + '_SOC_DHW_pct'] = soc_kwh[col] / cap_kwh * 100.0

    if not os.path.exists(soc_output_folder):
        os.makedirs(soc_output_folder, exist_ok=True)

    output_path = soc_output_path("DHW", season, base_dir=soc_output_folder)

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        soc_kwh.to_excel(writer, sheet_name='SOC_kWh', index=True)
        soc_pct.to_excel(writer, sheet_name='SOC_pct', index=True)

    print(f"[INFO] DHW SoC saved in: {output_path}")
#-------------------------------------------------------------------




# ------------------------------------------------------------
# Sec 4.6 -- 9.4) SOC color style by technology and control
# ------------------------------------------------------------
# Function to identify the technology setup in the analyzed scenario
def _control_for_mode_and_tech(mode, tech):
    """
    Retrieves OPT/RB/NO for a given technology and MODE.
    """
    mode = str(mode)
    tech = str(tech).upper()

    for cfg in RUN_CONTROL_CONFIGS:
        if str(cfg.get("mode", "")) == mode:
            return _norm_control_value(cfg.get("control", {}).get(tech, "OPT"))

    if mode == "OPT":
        return "OPT"

    if mode in {"RB", "RB_FIXED"}:
        return "RB"

    return "OPT"


# Function to assign the color gradient to each technology in charts
def _soc_plot_style(tech, control, variant=0):
    """
    Returns colors for SOC percentile charts.

    Required mapping:
      - BESS OPT: blue
      - BESS RB : light blue
      - DHW OPT : red
      - DHW RB  : orange
      - BEV OPT : dark green
      - BEV RB  : light green

    variant is used in comparison charts between two configurations:
      variant=0 and variant=1 use the same color family but
      with slightly different intensities.
    """
    tech = str(tech).upper()
    control = _norm_control_value(control)

    cmap_name = {
        ("BESS", "OPT"): "Blues",
        ("BESS", "RB"):  "GnBu",
        ("DHW",  "OPT"): "Reds",
        ("DHW",  "RB"):  "Oranges",
        ("BEV",  "OPT"): "Greens",
        ("BEV",  "RB"):  "YlGn",
    }.get((tech, control), "Greys")

    cmap = plt.get_cmap(cmap_name)

    if int(variant) == 0:
        levels = [0.25, 0.40, 0.58, 0.76]
        median_level = 0.92
    else:
        levels = [0.18, 0.34, 0.52, 0.70]
        median_level = 0.86

    return {
        "cmap_name": cmap_name,
        "band_colors": [cmap(v) for v in levels],
        "median_color": cmap(median_level),
        "alpha_bands": [0.22, 0.30, 0.42, 0.56],
    }


# Function to print/plot SOC percentile charts
def plot_soc_percentiles_from_xlsx(
    tech,
    season,
    out_dir_graph,
    mode,
    mode_label=None,
    base_dir_soc=None,
    control_mode=None,
):
    """
    Reads soc_<tech>_<season>.xlsx produced in 4.11
    and saves the SOC percentile chart in 4.12.

    It now uses different color scales by technology and control type:
      - BESS OPT: blue
      - BESS RB : light blue
      - DHW OPT : red
      - DHW RB  : orange
      - BEV OPT : dark green
      - BEV RB  : light green
    """
    if base_dir_soc is None:
        base_dir_soc = optimization_output_folder

    tech_u = str(tech).upper()

    if control_mode is None:
        control_mode = _control_for_mode_and_tech(mode, tech_u)

    control_mode = _norm_control_value(control_mode)

    soc_path = soc_output_path(
        tech_u,
        season,
        base_dir=base_dir_soc,
    )

    if not os.path.exists(soc_path):
        print(f"[INFO][SOC PLOT] SOC file not found, skipping {tech_u}: {soc_path}")
        return None

    xls = pd.ExcelFile(soc_path)

    if "SOC_pct" in xls.sheet_names:
        df_soc_pct = pd.read_excel(soc_path, sheet_name="SOC_pct", index_col=0)
    elif "SOC_%" in xls.sheet_names:
        df_soc_pct = pd.read_excel(soc_path, sheet_name="SOC_%", index_col=0)
    else:
        print(f"[WARN][SOC PLOT] No SOC_pct sheet in {soc_path}")
        return None

    if df_soc_pct.empty:
        print(f"[INFO][SOC PLOT] Empty SOC, skipping {tech_u}: {soc_path}")
        return None

    try:
        df_soc_pct.index = pd.to_datetime(df_soc_pct.index)
    except Exception:
        pass

    df_soc_pct = df_soc_pct.apply(pd.to_numeric, errors="coerce")
    df_soc_pct = df_soc_pct.dropna(axis=1, how="all")

    if df_soc_pct.empty:
        print(f"[INFO][SOC PLOT] No valid numeric SOC column, skipping {tech_u}: {soc_path}")
        return None

    qs = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

    df_pct = pd.DataFrame(
        {
            f"p{int(q * 100)}": df_soc_pct.quantile(q, axis=1)
            for q in qs
        },
        index=df_soc_pct.index,
    )

    style = _soc_plot_style(
        tech=tech_u,
        control=control_mode,
        variant=0,
    )

    fig, ax = plt.subplots(figsize=(12.5, 5.4))

    bands = [
        ("p10", "p90", "10-90%"),
        ("p20", "p80", "20-80%"),
        ("p30", "p70", "30-70%"),
        ("p40", "p60", "40-60%"),
    ]

    for (lo, hi, lab), color, alpha in zip(
        bands,
        style["band_colors"],
        style["alpha_bands"],
    ):
        ax.fill_between(
            df_pct.index,
            df_pct[lo],
            df_pct[hi],
            alpha=alpha,
            color=color,
            label=lab,
        )

    ax.plot(
        df_pct.index,
        df_pct["p50"],
        linewidth=2.2,
        color=style["median_color"],
        label="Median p50",
    )

    ax.set_title(
        f"SOC {tech_u} percentiles - {season_english_dict.get(season, season)} "
        f"({mode_label or mode}; {tech_u}={control_mode})"
    )

    ax.set_ylabel("SOC [%]")
    ax.set_xlabel("Time")
    ax.set_ylim(0.0, 100.0)
    ax.grid(True, axis="both", linestyle=":", alpha=0.35)

    try:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
    except Exception:
        pass

    leg = ax.legend(
        loc="best",
        frameon=True,
        fontsize=8,
        title=f"{tech_u} {control_mode}",
        title_fontsize=9,
    )
    leg.get_frame().set_alpha(0.90)

    os.makedirs(out_dir_graph, exist_ok=True)

    out_png = os.path.join(
        out_dir_graph,
        f"soc_{tech_u.lower()}_percentiles_{control_mode.lower()}_{season}.png",
    )

    fig.tight_layout()
    fig.savefig(out_png, dpi=250, bbox_inches="tight", pad_inches=0.04)
    plt.close(fig)

    print(f"[INFO][SOC PLOT] Saved: {out_png}")

    return out_png



# ------------------------------------------------------------
# Sec 4.6 -- 9.4quater) Strong style for SOC comparison between two cases
# ------------------------------------------------------------
# Function that converts colors
def _rgba(color, alpha):
    """
    Converts a Matplotlib color into RGBA with the desired alpha.
    """
    r, g, b, _ = mcolors.to_rgba(color)
    return (r, g, b, alpha)


# Function that assigns colors to the selected case/configuration
def _soc_compare_case_style(tech, control, case_index):
    """
    Style for direct comparison between two configurations.

    If SOC_COMPARE_COLOR_MODE = "case_contrast":
      - case A: orange/red
      - case B: blue/purple
      - optional hatch for case B
      - highly distinguishable medians

    If SOC_COMPARE_COLOR_MODE = "tech_control":
      use the technology/control palette from _soc_plot_style.
    """
    color_mode = globals().get(
        "SOC_COMPARE_COLOR_MODE",
        "case_contrast",
    )

    use_hatches = globals().get(
        "SOC_COMPARE_USE_HATCHES",
        True,
    )

    tech = str(tech).upper()
    control = _norm_control_value(control)

    if color_mode == "tech_control":
        base = _soc_plot_style(
            tech=tech,
            control=control,
            variant=case_index,
        )

        return {
            "outer_face": base["band_colors"][0],
            "inner_face": base["band_colors"][2],
            "outer_alpha": 0.22,
            "inner_alpha": 0.36,
            "edge_color": base["median_color"],
            "median_color": base["median_color"],
            "median_linestyle": "-" if case_index == 0 else "-",
            "hatch_outer": "" if case_index == 0 else ("///" if use_hatches else ""),
            "hatch_inner": "" if case_index == 0 else ("..." if use_hatches else ""),
        }

    # Default: strong contrast between case A and case B
    if int(case_index) == 0:
        main = "#D35400"      # dark orange
        inner = "#E67E22"     # orange
        hatch_outer = ""
        hatch_inner = ""
        linestyle = "-"
    else:
        main = "#2C3E50"      # dark blue/purple
        inner = "#5D6DFF"     # blue/purple
        hatch_outer = "///" if use_hatches else ""
        hatch_inner = "..." if use_hatches else ""
        linestyle = "--"

    return {
        "outer_face": _rgba(inner, 0.18),
        "inner_face": _rgba(inner, 0.34),
        "outer_alpha": None,
        "inner_alpha": None,
        "edge_color": main,
        "median_color": main,
        "median_linestyle": linestyle,
        "hatch_outer": hatch_outer,
        "hatch_inner": hatch_inner,
    }






# ------------------------------------------------------------
# Sec 4.6 -- 9.5) DHW SOC comparison between two configurations + price/PV
# ------------------------------------------------------------
# Function to define the SOC-file directory for the selected configuration
def _soc_path_for_manifest_row(run_row, season, tech="DHW"):
    xls_dir = manifest_path_to_absolute(
        run_row["xls_relpath"]
    )

    return soc_output_path(
        tech=tech,
        season=season,
        base_dir=xls_dir,
    )


# Function to read the SOC percentile bands file for the selected configuration
def _read_soc_pct_for_manifest_row(run_row, season, tech="DHW"):
    soc_path = _soc_path_for_manifest_row(
        run_row,
        season=season,
        tech=tech,
    )

    if not os.path.isfile(soc_path):
        raise FileNotFoundError(
            f"SOC file not found for {run_row['run_label']}, "
            f"tech={tech}, season={season}:\n{soc_path}"
        )

    xls = pd.ExcelFile(soc_path)

    if "SOC_pct" in xls.sheet_names:
        sheet = "SOC_pct"
    elif "SOC_%" in xls.sheet_names:
        sheet = "SOC_%"
    else:
        raise KeyError(
            f"No SOC_pct/SOC_% sheet in {soc_path}."
        )

    df = pd.read_excel(
        soc_path,
        sheet_name=sheet,
        index_col=0,
    )

    try:
        df.index = pd.to_datetime(df.index)
    except Exception:
        pass

    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=1, how="all")

    if df.empty:
        raise ValueError(
            f"No valid SOC data in {soc_path}."
        )

    return df


# Function that computes percentile bands for each timestep
def _soc_percentiles_dataframe(df_soc_pct):
    """
    Calculates minimum and p10...p90 across households
    at each timestep.
    """
    qs = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

    out = pd.DataFrame(
        {
            f"p{int(q * 100)}": df_soc_pct.quantile(q, axis=1)
            for q in qs
        },
        index=df_soc_pct.index,
    )

    # Minimum SOC across households at each timestep
    out["min"] = df_soc_pct.min(axis=1)

    return out


# Function that aligns a series to the required index
def _align_series_to_index(s, target_index):
    """
    Aligns a series to target_index.
    """
    s = pd.to_numeric(s, errors="coerce")

    try:
        s.index = pd.to_datetime(s.index)
    except Exception:
        pass

    target_index = pd.Index(target_index)

    try:
        out = (
            s.reindex(s.index.union(target_index))
             .sort_index()
             .interpolate(method="time")
             .ffill()
             .bfill()
             .reindex(target_index)
        )
    except Exception:
        out = s.reindex(target_index).ffill().bfill()

    return pd.to_numeric(out, errors="coerce")


# Function that retrieves the price series to include in the SOC percentile chart
def _get_purchase_price_series_for_soc_compare(
    season,
    target_index,
):
    """
    Purchase price [EUR/kWh] for SOC secondary axis.

    Order:
      1) ee_cost_series, if already loaded from 4.9;
      2) lazy loading from the 10-minute price file.
    """
    if (
        "ee_cost_series" in globals()
        and isinstance(ee_cost_series, dict)
        and season in ee_cost_series
    ):
        return _align_series_to_index(
            ee_cost_series[season],
            target_index,
        )

    try:
        prices_df = pd.read_excel(
            prices_10min_path,
            sheet_name=season,
            index_col=0,
            parse_dates=True,
        )
    except ValueError:
        prices_df = pd.read_excel(
            prices_10min_path,
            sheet_name=f"{season} SUM",
            index_col=0,
            parse_dates=True,
        )

    try:
        prices_df.index = pd.to_datetime(prices_df.index)
    except Exception:
        pass

    if (
        "Prezzo acquisto finale [€/kWh_e]" not in prices_df.columns
        and "PUN [€/kWh_e]" in prices_df.columns
    ):
        prices_df["Prezzo acquisto finale [€/kWh_e]"] = (
            pd.to_numeric(prices_df["PUN [€/kWh_e]"], errors="coerce")
            + float(additional_fees)
        )

    price_candidates = [
        "Prezzo acquisto finale [€/kWh_e]",
        "buy_price_eur_per_kwh",
        "PUN [€/kWh_e]",
    ]

    zone_col = region_to_price_col.get(selected_region, None)
    if zone_col is not None:
        price_candidates.append(f"{zone_col} [€/kWh_e]")
        price_candidates.append(zone_col)

    for col in price_candidates:
        if col in prices_df.columns:
            return _align_series_to_index(
                prices_df[col],
                target_index,
            )

    raise KeyError(
        f"Purchase price not found for season={season}. "
        f"Available columns: {prices_df.columns.tolist()}"
    )


# Function that retrieves the PV generation series to include in the SOC percentile chart
def _get_pv_generation_series_for_soc_compare(
    season,
    target_index,
    run_row,
    pv_source="total_installed",
    single_house=None,
):
    """
    Return the PV-generation series associated with one exact saved run.

    The PV time-series profile is a common input.
    The set of households actually equipped with PV is read from the
    run-specific has_active.pkl.gz referenced by the manifest.
    """

    # --------------------------------------------------------
    # 1) Load common PV profiles
    # --------------------------------------------------------

    if (
        "pv_raw_dicts" in globals()
        and isinstance(pv_raw_dicts, dict)
        and season in pv_raw_dicts
        and isinstance(pv_raw_dicts[season], dict)
    ):
        pv_source_dict = pv_raw_dicts[season]

    else:
        try:
            pv_df = pd.read_excel(
                pv_10min_file_path,
                sheet_name=season,
                index_col=0,
                parse_dates=True,
            )
        except ValueError:
            pv_df = pd.read_excel(
                pv_10min_file_path,
                sheet_name=f"{season} SUM",
                index_col=0,
                parse_dates=True,
            )

        try:
            pv_df.index = pd.to_datetime(pv_df.index)
        except Exception:
            pass


        # ----------------------------------------------------
        # Standardize household column names.
        #
        # The raw PV workbook preserves the original household
        # labels (e.g. Nucleo_1, Nucleo_2, ...), whereas
        # Module 04 internally uses c_1, c_2, ...
        #
        # This is the same normalization already performed
        # during the standard profile loading in Sec. 4.9.
        # ----------------------------------------------------

        pv_df, pv_map_soc_compare = (
            rename_axis_to_c(
                pv_df,
                axis="columns",
                prefix_out="c",
            )
        )


        # Defensive check against accidental duplicated
        # household IDs after normalization.
        duplicated_pv_cols = (
            pv_df.columns[
                pv_df.columns.duplicated()
            ]
            .astype(str)
            .tolist()
        )

        if duplicated_pv_cols:
            raise ValueError(
                "[SOC COMPARE][PV] Duplicated household "
                "columns after normalization: "
                f"{duplicated_pv_cols[:10]}"
            )


        missing_pv_cols = [
            c for c in ID
            if c not in pv_df.columns
        ]

        if missing_pv_cols:
            raise KeyError(
                f"Missing PV columns for season={season}. "
                f"First missing: {missing_pv_cols[:10]}"
            )

        pv_source_dict = {
            c: pd.to_numeric(
                pv_df[c],
                errors="coerce",
            ).fillna(0.0)
            for c in ID
        }

    # --------------------------------------------------------
    # 2) Load the exact technology assignment of this run
    # --------------------------------------------------------

    py_dir = manifest_path_to_absolute(
        run_row["py_relpath"]
    )

    has_active_path = os.path.join(
        py_dir,
        "has_active.pkl.gz",
    )

    if not os.path.isfile(has_active_path):
        raise FileNotFoundError(
            f"has_active.pkl.gz not found for "
            f"{run_row['run_label']}:\n"
            f"{has_active_path}"
        )

    has_active = load_compressed_pickle(
        has_active_path
    )

    pv_houses = set(
        has_active.get("PV", set())
    )

    available_pv_houses = sorted(
        [
            c
            for c in pv_houses
            if c in pv_source_dict
        ],
        key=lambda x: int(str(x).split("_")[1]),
    )

    pv_source = str(
        pv_source
    ).lower().strip()

    # --------------------------------------------------------
    # 3) Select PV households
    # --------------------------------------------------------

    if pv_source == "single_house":

        if single_house is None:
            if not available_pv_houses:
                raise ValueError(
                    f"No household with PV in "
                    f"{run_row['run_label']}."
                )

            selected_houses = [
                available_pv_houses[0]
            ]

        elif single_house not in pv_houses:
            raise ValueError(
                f"{single_house} has no PV in "
                f"{run_row['run_label']}."
            )

        elif single_house not in pv_source_dict:
            raise KeyError(
                f"No PV profile available for {single_house}."
            )

        else:
            selected_houses = [
                single_house
            ]

    elif pv_source == "total_installed":

        selected_houses = (
            available_pv_houses
        )

    else:
        raise ValueError(
            "pv_source must be "
            "'total_installed' or 'single_house'."
        )

    # --------------------------------------------------------
    # 4) Zero-PV configuration
    # --------------------------------------------------------

    if not selected_houses:
        return pd.Series(
            0.0,
            index=target_index,
            name="PV_generation_kW",
        )

    # --------------------------------------------------------
    # 5) Aggregate installed PV generation
    # --------------------------------------------------------

    pieces = []

    for c in selected_houses:

        profile = pd.to_numeric(
            pv_source_dict[c],
            errors="coerce",
        ).fillna(0.0)

        pieces.append(
            profile * float(PV_power_kwp)
        )

    pv_total = pd.concat(
        pieces,
        axis=1,
    ).sum(axis=1)

    pv_total.name = (
        "PV_generation_kW"
    )

    return _align_series_to_index(
        pv_total,
        target_index,
    )


# Function to return the common output directory for one experiment revision
def build_experiment_revision_output_dir(
    *,
    experiment_id,
    run_revision,
):
    """
    Return the common output directory for one experiment revision.

    Directory hierarchy:

        04_HGD_cmp_multi_runs/
            <grid_group>/
                EXP_<experiment_id>/
                    REV_<run_revision>/

    This directory is the common parent for:
      - experiment-level economic summaries;
      - multi-run comparisons;
      - SOC comparisons;
      - other experiment-level post-processing outputs.
    """

    experiment_folder = (
        "EXP_"
        + _safe_fs_tag(
            experiment_id
        )
    )

    revision_folder = (
        "REV_"
        + _safe_fs_tag(
            run_revision
        )
    )

    out_dir = os.path.join(
        hagrid_multirun_root_folder,
        experiment_folder,
        revision_folder,
    )

    os.makedirs(
        out_dir,
        exist_ok=True,
    )

    return out_dir




# Function that builds the path/directory for comparing multiple simulations.
def build_compare_output_dir(
    *,
    experiment_id,
    run_revision,
    comparison_type,
    comparison_tag=None,
):
    """
    Return the output directory for post-processing comparisons
    involving multiple saved HAGRID runs.

    Directory hierarchy:

        04_HGD_cmp_multi_runs/
            <grid_group>/
                EXP_<experiment_id>/
                    REV_<run_revision>/
                        <comparison_type>/
                            [<comparison_tag>/]
    """

    experiment_revision_dir = (
        build_experiment_revision_output_dir(
            experiment_id=experiment_id,
            run_revision=run_revision,
        )
    )

    out_dir = os.path.join(
        experiment_revision_dir,
        _safe_fs_tag(
            comparison_type
        ),
    )

    if comparison_tag:

        out_dir = os.path.join(
            out_dir,
            _safe_fs_tag(
                comparison_tag
            ),
        )

    os.makedirs(
        out_dir,
        exist_ok=True,
    )

    return out_dir



# Function to build a readable filesystem tag describing the selection of tested configurations
def _manifest_filters_folder_tag(
    filters,
):
    """
    Build a readable filesystem tag describing a manifest-based
    multi-run selection.

    Examples
    --------
    {"case_label": ["Base case"]}
        -> "case_label-Base_case"

    {"case_label": ["Base case", "Case 1"]}
        -> "case_label-Base_case+Case_1"
    """

    if not filters:
        return "all_selected_runs"

    tokens = []

    for key in sorted(filters):

        values = filters[key]

        if not isinstance(
            values,
            (
                list,
                tuple,
                set,
            ),
        ):
            values = [values]

        value_tag = "+".join(
            _safe_fs_tag(v)
            for v in values
        )

        tokens.append(
            f"{_safe_fs_tag(key)}-"
            f"{value_tag}"
        )

    return "__".join(
        tokens
    )





# Function to define the chart-saving folder for the configuration
def _soc_compare_output_dir(
    *,
    experiment_id,
    run_revision,
    run_a,
    run_b,
):
    """
    Return the output directory for a two-run SOC comparison.
    """

    label_a = (
        run_a.get(
            "run_label",
            run_a["run_uid"],
        )
    )

    label_b = (
        run_b.get(
            "run_label",
            run_b["run_uid"],
        )
    )

    pair_tag = (
        f"{_safe_fs_tag(label_a)}"
        "__VS__"
        f"{_safe_fs_tag(label_b)}"
    )

    return build_compare_output_dir(
        experiment_id=experiment_id,
        run_revision=run_revision,
        comparison_type="soc_compare",
        comparison_tag=pair_tag,
    )


# Function to assign short labels in the ESWH SOC comparison chart
def _soc_compare_short_case_label(label):
    """
    Short label for SOC legend:
    - 'Base case' -> 'Base'
    - 'Case 1'    -> 'C1'
    - 'Case 4'    -> 'C4'
    """
    if not globals().get("SOC_COMPARE_SHORT_CASE_LABELS", True):
        return str(label)

    s = str(label).strip()

    if s.lower() == "base case":
        return "Base"

    m = re.match(r"Case\s+(\d+)", s, flags=re.IGNORECASE)

    if m:
        return f"C{m.group(1)}"

    return s


# Function to identify a lighter color compared to another one
def _lighten_color(color, amount=0.55):
    """
    Lightens a matplotlib color by mixing it with white.

    amount = 0.0 -> original color
    amount = 1.0 -> white
    """
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(rgb + (1.0 - rgb) * amount)


# Function to print/plot a chart with SOC percentile bands for two configurations + secondary axis, price or PV
def plot_compare_dhw_soc_two_configs_with_secondary(
    season,
    run_a,
    run_b,
    secondary_kind="pv",
    out_dir=None,
    pv_source=None,
    single_house=None,
    save_xlsx=True,
    show_title=None,
    save_pdf=None,
):
    """
    Compares DHW SOC percentiles for two configurations and adds:
      - secondary_kind='pv'    -> PV profile on the secondary axis
      - secondary_kind='price' -> purchase price on the secondary axis

    Uses short labels:
      Base case, Case 1, Case 4, ...
    both in file names and in the legend.
    """
    tech = "DHW"

    if show_title is None:
        show_title = globals().get("SHOW_SOC_COMPARE_TITLE", False)

    if save_pdf is None:
        save_pdf = globals().get("SAVE_JOURNAL_PDF", True)


    secondary_kind = str(secondary_kind).lower().strip()

    label_a = run_a.get(
        "case_label",
        run_a.get("run_label", run_a["run_uid"]),
    )

    label_b = run_b.get(
        "case_label",
        run_b.get("run_label", run_b["run_uid"]),
    )

    # ------------------------------------------------------------
    # Short case labels for the chart legend.
    #
    # The physical/electrification scenario is intentionally NOT
    # shown in the legend because the comparison already refers to
    # a uniquely selected physical scenario.
    #
    # Examples:
    #   Base case -> Base
    #   Case 1    -> C1
    # ------------------------------------------------------------

    legend_label_a = _soc_compare_short_case_label(
        label_a
    )

    legend_label_b = _soc_compare_short_case_label(
        label_b
    )


    ctrl_a = _norm_control_value(
        run_a.get("control_DHW", "OPT")
    )

    ctrl_b = _norm_control_value(
        run_b.get("control_DHW", "OPT")
    )

    df_soc_a = _read_soc_pct_for_manifest_row(
        run_a,
        season=season,
        tech=tech,
    )

    df_soc_b = _read_soc_pct_for_manifest_row(
        run_b,
        season=season,
        tech=tech,
    )

    pct_a = _soc_percentiles_dataframe(df_soc_a)
    pct_b = _soc_percentiles_dataframe(df_soc_b)

    idx_common = pct_a.index.intersection(pct_b.index)

    pct_a = pct_a.loc[idx_common].copy()
    pct_b = pct_b.loc[idx_common].copy()

    if pct_a.empty or pct_b.empty:
        raise ValueError(
            f"No common time index between {label_a} and {label_b}, "
            f"season={season}."
        )

    if secondary_kind == "price":
        secondary_series = _get_purchase_price_series_for_soc_compare(
            season=season,
            target_index=idx_common,
        )
        secondary_label = "Electricity purchase price [EUR/kWh]"
        secondary_axis_label = globals().get(
            "SOC_COMPARE_PRICE_AXIS_LABEL",
            "El. price [EUR/kWh]",
        )
        secondary_legend_label = globals().get(
            "SOC_COMPARE_PRICE_LEGEND_LABEL",
            "El. price",
        )
        secondary_tag = "price"

    elif secondary_kind == "pv":
        if pv_source is None:
            pv_source = globals().get(
                "SOC_COMPARE_PV_SOURCE",
                "total_installed",
            )

        if single_house is None:
            single_house = globals().get(
                "SOC_COMPARE_SINGLE_PV_HOUSE",
                None,
            )

        secondary_series = _get_pv_generation_series_for_soc_compare(
            season=season,
            target_index=idx_common,
            run_row=run_a,
            pv_source=pv_source,
            single_house=single_house,
        )

        if str(pv_source).lower().strip() == "single_house":
            secondary_label = f"PV generation {single_house or 'single household'} [kW]"
        else:
            secondary_label = "Total installed PV generation [kW]"

        secondary_axis_label = globals().get(
            "SOC_COMPARE_PV_AXIS_LABEL",
            "PV generation [kW]",
        )
        secondary_legend_label = globals().get(
            "SOC_COMPARE_PV_LEGEND_LABEL",
            "PV gen.",
        )


        secondary_tag = "pv_generation"

    else:
        raise ValueError("secondary_kind must be 'pv' or 'price'.")

    if out_dir is None:
        out_dir = _soc_compare_output_dir()

    os.makedirs(out_dir, exist_ok=True)

    case_a_tag = _safe_fs_tag(f"{run_a['run_label']}__{run_a['run_uid']}")
    case_b_tag = _safe_fs_tag(f"{run_b['run_label']}__{run_b['run_uid']}")


    # ------------------------------------------------------------
    # Windows/OneDrive-safe output path
    #
    # The output directory already identifies the compared runs,
    # therefore the compact filename does not need run UIDs.
    # ------------------------------------------------------------
    # ------------------------------------------------------------
    # Very compact tags for Windows/OneDrive-safe filenames
    # ------------------------------------------------------------

    season_short_tag = {
        "winter": "win",
        "spring": "spr",
        "summer": "sum",
        "autumn": "aut",
    }.get(
        str(season).lower(),
        _safe_fs_tag(season)[:3],
    )


    secondary_short_tag = {
        "pv": "pv",
        "price": "pr",
    }.get(
        str(secondary_kind).lower(),
        _safe_fs_tag(secondary_kind)[:2],
    )


    out_xlsx = _safe_output_path(
        folder=out_dir,

        descriptive_filename=(
            f"compare_soc_dhw_{secondary_tag}_"
            f"{case_a_tag}_vs_{case_b_tag}_"
            f"{season}.xlsx"
        ),

        compact_filename=(
            f"soc_eswh_{secondary_short_tag}_"
            f"{season_short_tag}.xlsx"
        ),
    )

    if save_xlsx:
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            pct_a.to_excel(writer, sheet_name="A_percentiles")
            pct_b.to_excel(writer, sheet_name="B_percentiles")

            pd.DataFrame({
                secondary_label: secondary_series,
            }).to_excel(writer, sheet_name="secondary_series")

            pd.DataFrame([
                {
                    "season": season,

                    "run_uid_a": run_a["run_uid"],
                    "run_label_a": run_a["run_label"],
                    "mode_a": run_a["mode"],
                    "case_label_a": label_a,
                    "electrification_id_a": run_a.get("electrification_id"),
                    "control_DHW_a": ctrl_a,

                    "run_uid_b": run_b["run_uid"],
                    "run_label_b": run_b["run_label"],
                    "mode_b": run_b["mode"],
                    "case_label_b": label_b,
                    "electrification_id_b": run_b.get("electrification_id"),
                    "control_DHW_b": ctrl_b,

                    "secondary_kind": secondary_kind,
                    "secondary_label": secondary_label,
                }
            ]).to_excel(writer, sheet_name="meta", index=False)

        print(f"[OK] Saved DHW SOC comparison data: {out_xlsx}")

    style_a = _soc_compare_case_style(
        tech="DHW",
        control=ctrl_a,
        case_index=0,
    )

    style_b = _soc_compare_case_style(
        tech="DHW",
        control=ctrl_b,
        case_index=1,
    )


    # Lighter color for area between "minimum value" and 10th percentile
    tail_face_a = _lighten_color(
        style_a["outer_face"],
        amount=0.50,
    )

    tail_face_b = _lighten_color(
        style_b["outer_face"],
        amount=0.50,
    )




    with plt.rc_context({
        "font.size": globals().get("JOURNAL_FONT_SIZE", 9.5),
        "axes.labelsize": globals().get("JOURNAL_AXIS_LABEL_SIZE", 10.5),
        "xtick.labelsize": globals().get("JOURNAL_TICK_LABEL_SIZE", 9.0),
        "ytick.labelsize": globals().get("JOURNAL_TICK_LABEL_SIZE", 9.0),
        "legend.fontsize": globals().get("JOURNAL_LEGEND_FONT_SIZE", 8.5),
        "axes.titlesize": globals().get("JOURNAL_TITLE_SIZE", 11.0),
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }):

        fig, ax1 = plt.subplots(
            figsize=globals().get("JOURNAL_TWO_COLUMN_FIGSIZE_SOC", (7.20, 3.95)),
            constrained_layout=False,
        )
        ax2 = ax1.twinx()

        x = pct_a.index

        # Robust conversions for matplotlib/fill_between.
        # They avoid silent errors or TypeError with DatetimeIndex + pandas Series.
        if isinstance(x, pd.DatetimeIndex):
            x_plot = x.to_pydatetime()
        else:
            x_plot = np.asarray(x)

        min_a = pd.to_numeric(pct_a["min"], errors="coerce").to_numpy(dtype=float)
        p10_a = pd.to_numeric(pct_a["p10"], errors="coerce").to_numpy(dtype=float)
        p30_a = pd.to_numeric(pct_a["p30"], errors="coerce").to_numpy(dtype=float)
        p50_a = pd.to_numeric(pct_a["p50"], errors="coerce").to_numpy(dtype=float)
        p70_a = pd.to_numeric(pct_a["p70"], errors="coerce").to_numpy(dtype=float)
        p90_a = pd.to_numeric(pct_a["p90"], errors="coerce").to_numpy(dtype=float)

        min_b = pd.to_numeric(pct_b["min"], errors="coerce").to_numpy(dtype=float)
        p10_b = pd.to_numeric(pct_b["p10"], errors="coerce").to_numpy(dtype=float)
        p30_b = pd.to_numeric(pct_b["p30"], errors="coerce").to_numpy(dtype=float)
        p50_b = pd.to_numeric(pct_b["p50"], errors="coerce").to_numpy(dtype=float)
        p70_b = pd.to_numeric(pct_b["p70"], errors="coerce").to_numpy(dtype=float)
        p90_b = pd.to_numeric(pct_b["p90"], errors="coerce").to_numpy(dtype=float)

        secondary_y = (
            pd.to_numeric(secondary_series.reindex(idx_common), errors="coerce")
            .to_numpy(dtype=float)
        )



        band_linewidth = 0.0
        median_linestyle = "-"
        secondary_linestyle = "-"

        # Secondary-axis markers: few markers, not one per timestep.
        markevery_secondary = max(1, len(x_plot) // 48)

        if secondary_kind == "price":
            secondary_marker = "o" #"o"
        else:
            secondary_marker = "^" #"^"

        # ------------------------------------------------------------
        # Case A: percentile bands
        # ------------------------------------------------------------
        # Minimum - p10
        ax1.fill_between(
            x_plot,
            min_a,
            p10_a,
            facecolor=tail_face_a,
            edgecolor="none",
            alpha=0.18,
            linewidth=band_linewidth,
            label=f"{legend_label_a} min-p10",
            zorder=1,
        )

        # p10 - p90
        ax1.fill_between(
            x_plot,
            p10_a,
            p90_a,
            facecolor=style_a["outer_face"],
            edgecolor="none",
            alpha=0.2,
            linewidth=band_linewidth,
            label=f"{legend_label_a} p10-p90",
            zorder=2,
        )

        # p30 - p70
        ax1.fill_between(
            x_plot,
            p30_a,
            p70_a,
            facecolor=style_a["inner_face"],
            edgecolor="none",
            alpha=0.28,
            linewidth=band_linewidth,
            label=f"{legend_label_a} p30-p70",
            zorder=3,
        )

        # median
        ax1.plot(
            x_plot,
            p50_a,
            color=style_a["median_color"],
            linewidth=globals().get("SOC_COMPARE_MEDIAN_LINEWIDTH", 2.6),
            linestyle=median_linestyle,
            label=f"{legend_label_a} median",
            zorder=6,
        )



        # ------------------------------------------------------------
        # Case B: percentile bands
        # ------------------------------------------------------------
        # Lower-tail band: minimum to p10
        ax1.fill_between(
            x_plot,
            min_b,
            p10_b,
            facecolor=tail_face_b,
            edgecolor="none",
            alpha=0.12,
            linewidth=band_linewidth,
            label=f"{legend_label_b} min-p10",
            zorder=4,
        )
        
        # p10 - p90
        ax1.fill_between(
            x_plot,
            p10_b,
            p90_b,
            facecolor=style_b["outer_face"],
            edgecolor="none",
            alpha=0.2,
            linewidth=band_linewidth,
            label=f"{legend_label_b} p10-p90",
            zorder=2,
        )


        # p30 - p70
        ax1.fill_between(
            x_plot,
            p30_b,
            p70_b,
            facecolor=style_b["inner_face"],
            edgecolor="none",
            alpha=0.28,
            linewidth=band_linewidth,
            label=f"{legend_label_b} p30-p70",
            zorder=4,
        )

        # median
        ax1.plot(
            x_plot,
            p50_b,
            color=style_b["median_color"],
            linewidth=globals().get("SOC_COMPARE_MEDIAN_LINEWIDTH", 2.6),
            linestyle=median_linestyle,
            label=f"{legend_label_b} median",
            zorder=8,
        )

    
        

        # ------------------------------------------------------------
        # Secondary axis
        # ------------------------------------------------------------
        if secondary_kind == "price":
            sec_color = "darkgreen"
            sec_style = "-"
            title_secondary = "purchase price"
        else:
            sec_color = "darkgreen"
            sec_style = "-"
            title_secondary = "PV generation"

        ax2.plot(
            x_plot,
            secondary_y,
            linewidth=0.7,
            linestyle=secondary_linestyle,
            marker=secondary_marker,
            markersize=globals().get("SOC_COMPARE_SECONDARY_MARKERSIZE", 1.6),
            markevery=markevery_secondary,
            color=sec_color,
            alpha=0.85,
            label=secondary_legend_label,
            zorder=10,
        )

        ax1.set_ylim(0.0, 100.0)
        ax1.set_ylabel("ESWH state of charge [%]")
        ax2.set_ylabel(secondary_axis_label)
        ax1.set_xlabel("Time")

        if show_title:
            ax1.set_title(
                f"ESWH SOC percentiles comparison + {title_secondary}\n"
                f"{label_a} v. {label_b} - Season: {season_english_dict.get(season, season)}",
                pad=8,
            )

        ax1.grid(True, axis="both", linestyle=":", alpha=0.35)
        # ax1.grid(True, axis="both", linestyle="-", alpha=0.35)
        try:
            ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
        except Exception:
            pass

        ax1.set_zorder(2)
        ax1.patch.set_visible(False)
        ax2.set_zorder(1)

        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()

        leg = ax1.legend(
            h1 + h2,
            l1 + l2,
            loc=globals().get("SOC_COMPARE_LEGEND_LOC", "lower left"),
            frameon=True,
            fontsize=globals().get("SOC_COMPARE_LEGEND_FONT_SIZE", 6.8),
            title=globals().get("SOC_COMPARE_LEGEND_TITLE", None),
            title_fontsize=globals().get("SOC_COMPARE_LEGEND_TITLE_SIZE", 7.2),
            ncol=globals().get("SOC_COMPARE_LEGEND_NCOL", 2),
            labelspacing=0.22,
            handlelength=1.25,
            handletextpad=0.35,
            columnspacing=0.55,
            borderpad=0.35,
        )

        # Legend above the rest of the graph
        leg.set_zorder(100)

        leg.get_frame().set_alpha(0.95)
        leg.get_frame().set_linewidth(0.6)

        fig.subplots_adjust(
            left=0.080,
            right=0.895,
            top=0.92 if not show_title else 0.84,
            bottom=0.16,
        )

        out_png = _safe_output_path(
            folder=out_dir,

            descriptive_filename=(
                f"compare_soc_dhw_{secondary_tag}_"
                f"{case_a_tag}_vs_{case_b_tag}_"
                f"{season}.png"
            ),

            compact_filename=(
                f"soc_eswh_{secondary_short_tag}_"
                f"{season_short_tag}.png"
            ),
        )


        try:
            fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.04)

            if save_pdf:
                out_pdf = os.path.splitext(out_png)[0] + ".pdf"
                fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.04)
            else:
                out_pdf = None

            if not os.path.exists(out_png):
                raise RuntimeError(f"fig.savefig did not create the expected PNG file: {out_png}")

            print(f"[OK] Saved DHW SOC comparison chart: {out_png}")

            if save_pdf and out_pdf is not None:
                print(f"[OK] Saved DHW SOC comparison chart PDF: {out_pdf}")

        finally:
            plt.close(fig)
            
        return {
            "out_png": out_png,
            "out_xlsx": out_xlsx if save_xlsx else None,
            "pct_a": pct_a,
            "pct_b": pct_b,
            "secondary_series": secondary_series,
        }

   ### 4.6.10 F(x) Costs for household

In [ ]:
#---------------------------------------------------------------------
#           Sec 4.6 -- 10.1) PRICE SERIES AND SERIES UTILITIES
#---------------------------------------------------------------------
# Function to sum selected dataframe columns
def _sum_flow_columns_by_tokens(df, required_tokens):
    """
    Sums all dataframe columns that contain all required tokens.

    Returns:
      - summed Series
      - list of used columns
    """
    if df is None or df.empty:
        return pd.Series(dtype=float), []

    cols = []
    for col in df.columns:
        s = _col_to_searchable_string(col)
        if all(tok in s for tok in required_tokens):
            cols.append(col)

    if not cols:
        return pd.Series(0.0, index=df.index, dtype=float), []

    out = (
        df.loc[:, cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .sum(axis=1)
    )

    return out, cols
# -------------------------------------------------------------------


#---------------------------------------------------------------------
#           Sec 4.6 -- 10.2) TOTAL COSTS PER HOUSEHOLD FROM FLOW/OBJECTIVE
#---------------------------------------------------------------------
# Function to reconstruct household-level cost information across the different configurations
def build_total_costs_per_household_from_el_flows(
    flow_el_df,
    df_objective_terms_by_house,
    house_ids,
    ee_cost_real,
    ee_rev_real,
    timestep_minutes,
    values_are_power_kw=True,
    include_dhw_boost_in_effective_costs=False,
):
    """
    Rebuild household-level economic costs.

    Two different totals are returned:

    simulation_total_eur
        Household-attributable cost terms seen by the optimization,
        including the artificial DHW-boost penalty.

    effective_total_eur
        Effective economic cost used for reporting.
        The DHW-boost penalty is included only if
        include_dhw_boost_in_effective_costs=True.

    Notes
    -----
    - LV export remuneration must already be negative.
    - BEV outdoor charging is always treated as a real cost.
    - total_eur is retained as a backward-compatible alias of
      effective_total_eur so existing comparison plots use the
      effective economic metric.
    """

    output_columns = [
        "house_id",

        "bt_import_kwh",
        "bt_export_kwh",

        "bt_purchase_cost_eur",
        "bt_sale_revenues_eur",
        "bt_energy_net_settlement_eur",

        "dhw_boost_cost_eur",
        "dhw_boost_cost_effective_eur",
        "dhw_boost_cost_excluded_eur",

        "bev_outdoor_charge_cost_eur",
        "pv_curtail_cost_eur",

        "other_objective_terms_eur",
        "extra_objective_terms_eur",
        "effective_extra_terms_eur",

        "simulation_total_eur",
        "effective_total_eur",

        # backward-compatible alias used by existing plots
        "total_eur",

        "n_cols_import_matched",
        "n_cols_export_matched",
    ]

    if flow_el_df is None or flow_el_df.empty:
        return pd.DataFrame(columns=output_columns)

    dt_h = float(timestep_minutes) / 60.0

    price_buy = (
        pd.Series(ee_cost_real)
        .reindex(flow_el_df.index)
        .astype(float)
        .ffill()
        .bfill()
        .fillna(0.0)
    )

    price_sell = (
        pd.Series(ee_rev_real)
        .reindex(flow_el_df.index)
        .astype(float)
        .ffill()
        .bfill()
        .fillna(0.0)
    )


    # ------------------------------------------------------------
    # Household-specific objective terms
    # ------------------------------------------------------------

    extra_by_house = pd.Series(dtype=float)
    category_by_house = pd.DataFrame()

    if (
        df_objective_terms_by_house is not None
        and not df_objective_terms_by_house.empty
        and "house_id" in df_objective_terms_by_house.columns
        and "term_total_eur" in df_objective_terms_by_house.columns
    ):

        tmp_extra = df_objective_terms_by_house.copy()

        # LV import/export settlement is reconstructed directly
        # from flow_el_df below -> avoid double counting.
        if "category" in tmp_extra.columns:

            mask_not_lv = ~tmp_extra[
                "category"
            ].astype(str).str.contains(
                "bt_energy",
                case=False,
                na=False,
            )

            tmp_extra = tmp_extra.loc[
                mask_not_lv
            ].copy()

        extra_by_house = (
            tmp_extra
            .groupby("house_id")["term_total_eur"]
            .sum()
        )

        if "category" in tmp_extra.columns:

            category_by_house = (
                tmp_extra
                .pivot_table(
                    index="house_id",
                    columns="category",
                    values="term_total_eur",
                    aggfunc="sum",
                    fill_value=0.0,
                )
            )


    def _house_category_cost(house_id, category):

        if (
            category_by_house is None
            or category_by_house.empty
            or category not in category_by_house.columns
            or house_id not in category_by_house.index
        ):
            return 0.0

        return float(
            category_by_house.loc[
                house_id,
                category,
            ]
        )


    rows = []

    for c in house_ids:

        # Household LV import
        s_imp_kw, cols_imp = _sum_flow_columns_by_tokens(
            flow_el_df,
            required_tokens=[
                f"from_the_grid_BT_to_{c}",
                f"{c}_bus_el",
            ],
        )

        # Household LV export
        s_exp_kw, cols_exp = _sum_flow_columns_by_tokens(
            flow_el_df,
            required_tokens=[
                f"{c}_bus_el",
                f"{c}_to_the_grid_BT",
            ],
        )

        if values_are_power_kw:

            s_imp_kwh = s_imp_kw * dt_h
            s_exp_kwh = s_exp_kw * dt_h

        else:

            s_imp_kwh = s_imp_kw
            s_exp_kwh = s_exp_kw


        lv_import_kwh = float(
            s_imp_kwh.sum()
        )

        lv_export_kwh = float(
            s_exp_kwh.sum()
        )


        lv_purchase_cost_eur = float(
            (s_imp_kwh * price_buy).sum()
        )

        # price_sell is already negative.
        lv_sale_revenues_eur = float(
            (s_exp_kwh * price_sell).sum()
        )

        lv_energy_net_settlement_eur = (
            lv_purchase_cost_eur
            + lv_sale_revenues_eur
        )


        # --------------------------------------------------------
        # Explicit additional costs
        # --------------------------------------------------------

        dhw_boost_cost_eur = _house_category_cost(
            c,
            "dhw_boost_cost_eur",
        )

        bev_outdoor_charge_cost_eur = _house_category_cost(
            c,
            "bev_outdoor_charge_cost_eur",
        )

        pv_curtail_cost_eur = _house_category_cost(
            c,
            "pv_curtail_cost_eur",
        )


        extra_terms_eur = float(
            extra_by_house.get(c, 0.0)
        )


        known_explicit_terms_eur = (
            dhw_boost_cost_eur
            + bev_outdoor_charge_cost_eur
            + pv_curtail_cost_eur
        )

        other_objective_terms_eur = (
            extra_terms_eur
            - known_explicit_terms_eur
        )


        # --------------------------------------------------------
        # DHW boost reporting rule
        # --------------------------------------------------------

        if include_dhw_boost_in_effective_costs:

            dhw_boost_cost_effective_eur = (
                dhw_boost_cost_eur
            )

            dhw_boost_cost_excluded_eur = 0.0

        else:

            dhw_boost_cost_effective_eur = 0.0

            dhw_boost_cost_excluded_eur = (
                dhw_boost_cost_eur
            )


        effective_extra_terms_eur = (
            extra_terms_eur
            - dhw_boost_cost_excluded_eur
        )


        # --------------------------------------------------------
        # Two economic totals
        # --------------------------------------------------------

        simulation_total_eur = (
            lv_energy_net_settlement_eur
            + extra_terms_eur
        )

        effective_total_eur = (
            lv_energy_net_settlement_eur
            + effective_extra_terms_eur
        )


        rows.append({

            "house_id": c,

            "bt_import_kwh": lv_import_kwh,
            "bt_export_kwh": lv_export_kwh,

            "bt_purchase_cost_eur": lv_purchase_cost_eur,
            "bt_sale_revenues_eur": lv_sale_revenues_eur,
            "bt_energy_net_settlement_eur":
                lv_energy_net_settlement_eur,

            "dhw_boost_cost_eur":
                dhw_boost_cost_eur,

            "dhw_boost_cost_effective_eur":
                dhw_boost_cost_effective_eur,

            "dhw_boost_cost_excluded_eur":
                dhw_boost_cost_excluded_eur,

            "bev_outdoor_charge_cost_eur":
                bev_outdoor_charge_cost_eur,

            "pv_curtail_cost_eur":
                pv_curtail_cost_eur,

            "other_objective_terms_eur":
                other_objective_terms_eur,

            "extra_objective_terms_eur":
                extra_terms_eur,

            "effective_extra_terms_eur":
                effective_extra_terms_eur,

            "simulation_total_eur":
                simulation_total_eur,

            "effective_total_eur":
                effective_total_eur,

            # Existing multi-run functions expect this column.
            # From now on it represents EFFECTIVE cost.
            "total_eur":
                effective_total_eur,

            "n_cols_import_matched":
                len(cols_imp),

            "n_cols_export_matched":
                len(cols_exp),
        })


    return pd.DataFrame(
        rows,
        columns=output_columns,
    )



### 4.6.11 F(x) Post-processing LV grid, PowerFlow and duration curves

In [ ]:
#---------------------------------------------------------------------
#           Sec 4.6 -- 11.0) KERBER BLUEPRINT LOADING AND GRID CONSTRUCTION
#---------------------------------------------------------------------
# Function that loads the files containing the grid archetype information from the Kerber model
def _load_kerber_blueprint(archetype: str) -> dict:
    fn = KERBER_BLUEPRINT_FILES.get(archetype, None)
    if fn is None:
        raise ValueError(f"[LV] Unrecognized archetype: {archetype}. Available choices: {list(KERBER_BLUEPRINT_FILES)}")
    fp = Path(grid_archetypes_input_folder) / fn
    if not fp.exists():
        raise FileNotFoundError(
            f"[LV] Kerber blueprint not found: {fp}.\n"
            "Place the required blueprint JSON in "
            f"'{grid_archetypes_input_folder}'. "
            "See docs/OUTPUT_STRUCTURE.md."
        )
    with open(fp, "r", encoding="utf-8") as f:
        return json.load(f)



# Function to assign a naming convention to the grid buses/nodes, distinguishing household nodes from infrastructure nodes
def _build_kerber_lv_labels(bp: dict, id_list, pad: int = 3):
    """
    Builds a readable naming convention for the Kerber LV buses.

    ------------------------------------------------------------
        Naming convention
    - bus_root : LV node on the transformer/MV side (root_lv_bus)
    - bus_n00X: "grid" nodes, such as junctions/branches, without a POD
    - bus_h00Y: household POD nodes (c_Y)
    ------------------------------------------------------------

    Returns:
      - bus_root_label (str)
      - bt_bus_labels_all (list[str])     in an order consistent with bp["lv_buses"]
      - house_to_bt_bus_label (dict)      c_i -> bus_h00i, or bus_root if the POD is on the root
      - bus_id_to_label (dict[int,str])   pandapower bus_id -> label
      - label_to_bus_id (dict[str,int])   label -> pandapower bus_id
      - bus_role (dict[str,str])          label -> "root" | "net" | "house"

    """
    root_bus = int(bp["root_lv_bus"])
    lv_buses = [int(b) for b in bp.get("lv_buses", [])]
    if not lv_buses:
        raise ValueError("[LV] Kerber blueprint does not contain 'lv_buses'.")



    # -----------------------
    # Func 4.6.11 - a) household -> bus_id mapping via load_id
    # -----------------------
    # sort households c_1, c_2, ...
    id_sorted = sorted(id_list, key=lambda x: int(str(x).split("_")[1]))

    loads = bp.get("loads_lv", None)
    if not loads:
        raise ValueError(
            "[LV] Kerber blueprint is missing 'loads_lv'. "
            "Provide a compatible blueprint JSON containing "
            "the required load-to-bus mapping. "
            "See docs/OUTPUT_STRUCTURE.md."
        )

    load_records_by_id = {}


    for load_record in loads:
        if "load_id" not in load_record:
            raise ValueError(
                "[LV] A loads_lv record does not contain "
                "'load_id'.\n"
                f"Invalid record: {load_record}"
            )

        load_id = int(
            load_record["load_id"]
        )

        if load_id in load_records_by_id:
            raise ValueError(
                "[LV] Duplicate load_id values were found "
                "in the Kerber blueprint.\n"
                f"Duplicate load_id: {load_id}"
            )

        load_records_by_id[load_id] = (
            load_record
        )


    expected_household_ids = {
        int(
            household_id.split("_")[1]
        )
        for household_id in id_sorted
    }

    blueprint_load_ids = set(
        load_records_by_id
    )


    missing_load_ids = sorted(
        expected_household_ids
        - blueprint_load_ids
    )

    unexpected_load_ids = sorted(
        blueprint_load_ids
        - expected_household_ids
    )


    if missing_load_ids or unexpected_load_ids:
        raise ValueError(
            "[LV] The load_id values stored in the "
            "Kerber blueprint do not match the household "
            "IDs used in MODULE_04.\n"
            f"Missing load IDs: {missing_load_ids}\n"
            f"Unexpected load IDs: {unexpected_load_ids}"
        )


    house_to_bus_id = {}

    bus_to_house_ids = defaultdict(
        list
    )


    for household_id in id_sorted:
        household_number = int(
            household_id.split("_")[1]
        )

        load_record = (
            load_records_by_id[
                household_number
            ]
        )

        bus_id = int(
            load_record["bus"]
        )

        house_to_bus_id[
            household_id
        ] = bus_id

        bus_to_house_ids[
            bus_id
        ].append(
            household_number
        )



    # -----------------------
    # Func 4.6.11 - b)  BFS from the root to number grid nodes in a topological order
    # -----------------------
    adj = defaultdict(set)
    for e in bp.get("lines_lv", []) or []:
        a = int(e["from_bus"])
        b = int(e["to_bus"])
        adj[a].add(b)
        adj[b].add(a)

    # deterministic BFS: neighbors sorted by bus_id
    bfs = []
    seen = {root_bus}
    q = [root_bus]
    while q:
        u = q.pop(0)
        for v in sorted(adj.get(u, [])):
            if v in seen:
                continue
            seen.add(v)
            bfs.append(v)
            q.append(v)

    # if any bus is not reachable, append it at the end in ascending order
    for b in sorted(set(lv_buses) - set(seen)):
        bfs.append(b)

    # -----------------------
    # Func 4.6.11 - c)  Label assignment
    # -----------------------
    bus_id_to_label = {}
    label_to_bus_id = {}
    bus_role = {}

    # root
    bus_id_to_label[root_bus] = "bus_root"
    label_to_bus_id["bus_root"] = root_bus
    bus_role["bus_root"] = "root"

    # household buses: dedicated label; if multiple households are connected to the same bus, use bus_h<first>_mK
    # namely --- bus_h (household i)_m (multiple users) K (number of multiple users) ---
    # all households connected to the same bus point to the same label
    for bus_id, house_ids in sorted(bus_to_house_ids.items(), key=lambda x: min(x[1])):
        if bus_id == root_bus:
            # POD on the root: keep bus_root
            continue
        house_ids_sorted = sorted(house_ids)
        if len(house_ids_sorted) == 1:
            lbl = f"bus_h{house_ids_sorted[0]:0{pad}d}"
        else:
            lbl = f"bus_h{house_ids_sorted[0]:0{pad}d}_m{len(house_ids_sorted)}"     # "_mK" = multiple K-users
        bus_id_to_label[bus_id] = lbl
        label_to_bus_id[lbl] = bus_id
        bus_role[lbl] = "house"

    # grid nodes: all remaining nodes, excluding root and household buses
    net_bus_ids = [b for b in bfs if b != root_bus and b not in bus_id_to_label]
    for k, b in enumerate(net_bus_ids, start=1):
        lbl = f"bus_n{k:0{pad}d}"
        bus_id_to_label[b] = lbl
        label_to_bus_id[lbl] = b
        bus_role[lbl] = "net"

    # safety: any buses still without a label, should not happen
    for b in lv_buses:
        if b not in bus_id_to_label:
            k = len([x for x in bus_role.values() if x == "net"]) + 1
            lbl = f"bus_n{k:0{pad}d}"
            bus_id_to_label[b] = lbl
            label_to_bus_id[lbl] = b
            bus_role[lbl] = "net"

    # -----------------------
    # Func 4.6.11 - d) Output
    # -----------------------
    lv_root_label = bus_id_to_label[root_bus]
    all_lv_bus_labels = [bus_id_to_label[int(b)] for b in lv_buses]

    # c_i -> label of the associated bus
    house_to_lv_bus_label = {}
    for c in id_sorted:
        b = house_to_bus_id[c]
        house_to_lv_bus_label[c] = bus_id_to_label[b]  # may be bus_root if the POD is on the root

    return lv_root_label, all_lv_bus_labels, house_to_lv_bus_label, bus_id_to_label, label_to_bus_id, bus_role


# Function to validate the consistency between the Kerber blueprint
# and the household/bus mappings reconstructed in MODULE_04.
def _validate_kerber_blueprint_mapping(
    bp: dict,
    house_to_lv_bus_label: dict,
    label_to_bus_id: dict,
    *,
    archetype: str = "",
    require_connection_kind: bool = False,
) -> dict:
    """
    Validate the structural consistency of a Kerber blueprint.

    The validation checks that:

    1. the root bus belongs to the exported LV-bus set;
    2. all household/load buses belong to the LV-bus set;
    3. all MODULE_04 household mappings point to valid pandapower buses;
    4. the load buses reconstructed in MODULE_04 coincide with the
       load buses stored in the MODULE_05 blueprint;
    5. all LV-line endpoints belong to the exported LV-bus set;
    6. line IDs are unique;
    7. when available, connection_kind is consistent with loads_lv.

    Returns
    -------
    dict
        Compact validation summary.
    """
    if not isinstance(bp, dict):
        raise TypeError(
            "[LV] The Kerber blueprint must be a dictionary."
        )

    lv_bus_ids = {
        int(bus_id)
        for bus_id in bp.get(
            "lv_buses",
            [],
        )
    }

    if not lv_bus_ids:
        raise ValueError(
            "[LV] The Kerber blueprint does not contain "
            "a valid 'lv_buses' list."
        )


    root_bus_id = int(
        bp["root_lv_bus"]
    )

    if root_bus_id not in lv_bus_ids:
        raise ValueError(
            "[LV] The root LV bus stored in the blueprint "
            "is not included in lv_buses.\n"
            f"Archetype: {archetype}\n"
            f"root_lv_bus: {root_bus_id}"
        )


    load_records = (
        bp.get(
            "loads_lv",
            [],
        )
        or []
    )

    if not load_records:
        raise ValueError(
            "[LV] The Kerber blueprint does not contain "
            "'loads_lv'."
        )


    blueprint_load_bus_ids = {
        int(load_record["bus"])
        for load_record in load_records
    }


    load_buses_outside_lv_grid = sorted(
        blueprint_load_bus_ids
        - lv_bus_ids
    )

    if load_buses_outside_lv_grid:
        raise ValueError(
            "[LV] Some loads_lv records refer to buses "
            "that are not included in lv_buses.\n"
            f"Archetype: {archetype}\n"
            "Invalid load-bus IDs: "
            f"{load_buses_outside_lv_grid}"
        )


    # Translate MODULE_04 labels back into original
    # pandapower/Kerber bus IDs.
    missing_household_labels = sorted(
        {
            str(lv_bus_label)
            for lv_bus_label
            in house_to_lv_bus_label.values()
            if lv_bus_label
            not in label_to_bus_id
        }
    )

    if missing_household_labels:
        raise ValueError(
            "[LV] Some household bus labels generated in "
            "MODULE_04 cannot be translated back into "
            "pandapower bus IDs.\n"
            f"Archetype: {archetype}\n"
            f"Missing labels: {missing_household_labels}"
        )


    mapped_household_bus_ids = {
        int(
            label_to_bus_id[
                lv_bus_label
            ]
        )
        for lv_bus_label
        in house_to_lv_bus_label.values()
    }


    if (
        blueprint_load_bus_ids
        != mapped_household_bus_ids
    ):
        missing_in_module_04 = sorted(
            blueprint_load_bus_ids
            - mapped_household_bus_ids
        )

        extra_in_module_04 = sorted(
            mapped_household_bus_ids
            - blueprint_load_bus_ids
        )

        raise ValueError(
            "[LV] The household-to-bus mapping reconstructed "
            "in MODULE_04 does not match the load buses stored "
            "in the MODULE_05 Kerber blueprint.\n"
            f"Archetype: {archetype}\n"
            "Blueprint load buses: "
            f"{sorted(blueprint_load_bus_ids)}\n"
            "MODULE_04 mapped buses: "
            f"{sorted(mapped_household_bus_ids)}\n"
            "Missing in MODULE_04: "
            f"{missing_in_module_04}\n"
            "Unexpected in MODULE_04: "
            f"{extra_in_module_04}"
        )


    line_records = (
        bp.get(
            "lines_lv",
            [],
        )
        or []
    )

    if not line_records:
        raise ValueError(
            "[LV] The Kerber blueprint does not contain "
            "'lines_lv'."
        )


    invalid_line_endpoints = []

    for line_record in line_records:
        from_bus = int(
            line_record["from_bus"]
        )

        to_bus = int(
            line_record["to_bus"]
        )

        if (
            from_bus not in lv_bus_ids
            or to_bus not in lv_bus_ids
        ):
            invalid_line_endpoints.append({
                "line_id": line_record.get(
                    "line_id"
                ),
                "from_bus": from_bus,
                "to_bus": to_bus,
            })


    if invalid_line_endpoints:
        raise ValueError(
            "[LV] Some blueprint lines refer to buses "
            "outside the exported LV-bus set.\n"
            f"Archetype: {archetype}\n"
            "Invalid lines, first 10 records:\n"
            f"{invalid_line_endpoints[:10]}"
        )


    line_ids = [
        int(line_record["line_id"])
        for line_record in line_records
    ]

    duplicate_line_ids = sorted({
        line_id
        for line_id in line_ids
        if line_ids.count(line_id) > 1
    })

    if duplicate_line_ids:
        raise ValueError(
            "[LV] Duplicate line IDs were found in the "
            "Kerber blueprint.\n"
            f"Archetype: {archetype}\n"
            f"Duplicate line IDs: {duplicate_line_ids}"
        )


    # Validate the MODULE_05 classification (from which grid archetypes are derived), when available.
    missing_connection_kind_line_ids = []

    inconsistent_connection_kind = []

    valid_connection_kinds = {
        "branch_house",
        "network_internal",
    }


    for line_record in line_records:
        line_id = int(
            line_record["line_id"]
        )

        from_bus = int(
            line_record["from_bus"]
        )

        to_bus = int(
            line_record["to_bus"]
        )

        expected_connection_kind = (
            "branch_house"
            if (
                from_bus in blueprint_load_bus_ids
                or to_bus in blueprint_load_bus_ids
            )
            else "network_internal"
        )

        exported_connection_kind = str(
            line_record.get(
                "connection_kind",
                "",
            )
        ).strip().lower()


        if not exported_connection_kind:
            missing_connection_kind_line_ids.append(
                line_id
            )
            continue


        if (
            exported_connection_kind
            not in valid_connection_kinds
        ):
            inconsistent_connection_kind.append({
                "line_id": line_id,
                "exported": exported_connection_kind,
                "expected": expected_connection_kind,
                "reason": "unknown value",
            })

        elif (
            exported_connection_kind
            != expected_connection_kind
        ):
            inconsistent_connection_kind.append({
                "line_id": line_id,
                "exported": exported_connection_kind,
                "expected": expected_connection_kind,
                "reason": (
                    "classification does not match "
                    "loads_lv"
                ),
            })


    if inconsistent_connection_kind:
        raise ValueError(
            "[LV] Some line connection_kind values are "
            "inconsistent with loads_lv.\n"
            f"Archetype: {archetype}\n"
            "Inconsistent lines, first 10 records:\n"
            f"{inconsistent_connection_kind[:10]}"
        )


    if (
        require_connection_kind
        and missing_connection_kind_line_ids
    ):
        raise ValueError(
            "[LV] The blueprint does not contain "
            "connection_kind for all LV lines.\n"
            f"Archetype: {archetype}\n"
            "Lines without connection_kind, first 20 IDs: "
            f"{missing_connection_kind_line_ids[:20]}\n"
            "Regenerate the blueprint with the updated "
            "MODULE_05."
        )


    if missing_connection_kind_line_ids:
        print(
            "[WARN][LV] The blueprint is structurally valid, "
            "but connection_kind is missing for "
            f"{len(missing_connection_kind_line_ids)} lines. "
            "MODULE_04 will use its backward-compatible "
            "fallback classification."
        )


    validation_summary = {
        "archetype": archetype,
        "n_lv_buses": len(lv_bus_ids),
        "n_load_records": len(load_records),
        "n_distinct_load_buses": len(
            blueprint_load_bus_ids
        ),
        "n_households_mapped": len(
            house_to_lv_bus_label
        ),
        "n_lv_lines": len(line_records),
        "n_lines_without_connection_kind": len(
            missing_connection_kind_line_ids
        ),
        "status": "valid",
    }


    print(
        "[OK][LV] Kerber blueprint/mapping validation "
        f"completed for '{archetype}': "
        f"{validation_summary['n_lv_buses']} buses, "
        f"{validation_summary['n_lv_lines']} lines, "
        f"{validation_summary['n_households_mapped']} "
        "households."
    )


    return validation_summary



# Function that returns a summary table of the grid bus/node naming convention, distinguishing infrastructure and household nodes
def _build_kerber_lv_label_table(bp: dict,
                                 bus_id_to_label: dict,
                                 bus_role: dict,
                                 house_to_lv_bus_label: dict):
    """Creates a DataFrame table to quickly inspect the Kerber LV topology.

    Columns:
      - bus_id_pp: LV bus id in the pandapower archetype blueprint
      - label:     label used in oemof (bus_root/bus_nXXX/bus_hYYY[_mK])
                   transformer - infrastructure node - household node [with multiple users]
      - role:      root | net | house
      - dist_edges_from_root: distance from the root as number of lines, computed through BFS
      - houses:    list of households (c_i) connected to that bus, if available
      - n_houses:  number of households connected to the bus
    """
    root_bus = int(bp.get("root_lv_bus", 0))
    lv_buses = [int(b) for b in bp.get("lv_buses", [])]

    # adjacency from lines_lv
    adj = defaultdict(set)
    for e in bp.get("lines_lv", []) or []:
        a = int(e["from_bus"])
        b = int(e["to_bus"])
        adj[a].add(b)
        adj[b].add(a)

    # BFS distances
    dist = {root_bus: 0}
    q = [root_bus]
    while q:
        u = q.pop(0)
        for v in sorted(adj.get(u, [])):
            if v in dist:
                continue
            dist[v] = dist[u] + 1
            q.append(v)

    # invert: label -> [households]
    label_to_houses = defaultdict(list)
    for c, lbl in (house_to_lv_bus_label or {}).items():
        label_to_houses[lbl].append(c)

    rows = []
    for b in lv_buses:
        lbl = bus_id_to_label.get(b, f"bus_n???")
        role = bus_role.get(lbl, "net")
        houses = sorted(label_to_houses.get(lbl, []), key=lambda x: int(str(x).split("_")[1]))
        rows.append({
            "bus_id_pp": b,
            "label": lbl,
            "role": role,
            "dist_edges_from_root": dist.get(b, None),
            "houses": ", ".join(houses),
            "n_houses": len(houses),
        })

    df = pd.DataFrame(rows)

    # Order: first by distance, then root/net/house
    role_order = {"root": 0, "net": 1, "house": 2}
    df["role_order"] = df["role"].map(role_order).fillna(9).astype(int)
    df["dist_sort"] = df["dist_edges_from_root"].fillna(10**9).astype(int)
    df = (df.sort_values(["dist_sort", "role_order", "bus_id_pp"], ascending=[True, True, True])
            .drop(columns=["role_order", "dist_sort"]))
    return df



# Function to print the summary table of the naming convention used for grid topology construction
def _print_kerber_lv_label_table(df: "pd.DataFrame", archetype: str = "", max_rows: int = 120):
    """Prints the LV mapping table in compact form, avoiding excessively large outputs."""
    if df is None:
        return
    n = len(df)
    header = f"\n[KERBER] LV bus table (bus_root/bus_n/bus_h) - '{archetype}' - {n} rows"
    print(header)
    if n <= max_rows:
        print(df.to_string(index=False))
    else:
        print(df.head(max_rows).to_string(index=False))
        print(f"... (showing {max_rows} rows out of {n}). Set KERBER_LV_LABEL_TABLE_MAX_ROWS to change the limit.")



# Function to extract the compact grid bus name (bus_h, bus_n, bus_root) to be used for grid converter/line construction
def _short_lv_bus_label(x: str) -> str:
    """
    Makes bus labels compact:
      - bt_root  -> root
      - bt_n001  -> n001
      - bt_h001  -> h001
      - bus_n001 -> n001   if ever available
    """
    s = str(x)
    for pref in ("bus_", "bt_"):
        if s.startswith(pref):
            s = s[len(pref):]
    return s



def initialize_lv_topology(id_list):
    """
    Resolve the selected LV topology after household IDs
    and user configuration are available.

    Returns a dictionary containing the complete topology state
    required by the optimization, post-processing and PF export.
    """

    state = {
        "KERBER_BP": None,
        "KERBER_BLUEPRINT_VALIDATION": None,
        "KERBER_BUS_ID_TO_LABEL": {},
        "KERBER_LABEL_TO_BUS_ID": {},
        "KERBER_BUS_ROLE": {},
        "KERBER_LV_LABEL_TABLE": None,
        "HOUSE_TO_LV_BUS_LABEL": {},
        "HOUSE_TO_PP_BUS_ID": {},
        "LV_ROOT_LABEL": None,
        "LV_BUS_LABELS_ALL": [],
        "LV_BUS_LABELS_EXPORT": [],
    }

    # --------------------------------------------------------
    # 1) Single fictitious LV bus
    # --------------------------------------------------------

    if LV_ARCHETYPE == "single_bus":

        lv_root = "bus_el_BT"

        state["LV_ROOT_LABEL"] = lv_root
        state["LV_BUS_LABELS_ALL"] = [lv_root]
        state["LV_BUS_LABELS_EXPORT"] = [lv_root]

        state["HOUSE_TO_LV_BUS_LABEL"] = {
            c: lv_root
            for c in id_list
        }

    # --------------------------------------------------------
    # 2) Legacy rural chain
    # --------------------------------------------------------

    elif LV_ARCHETYPE == "legacy_rural_chain":

        lv_root = "bus_el_BT_0"

        lv_bus_labels = [
            lv_root
        ] + [
            f"bus_el_BT_Da_{k}"
            for k in range(
                1,
                int(num_households) + 1,
            )
        ]

        state["LV_ROOT_LABEL"] = lv_root
        state["LV_BUS_LABELS_ALL"] = (
            lv_bus_labels.copy()
        )
        state["LV_BUS_LABELS_EXPORT"] = (
            lv_bus_labels.copy()
        )

        state["HOUSE_TO_LV_BUS_LABEL"] = {
            f"c_{i}": f"bus_el_BT_Da_{i}"
            for i in range(
                1,
                int(num_households) + 1,
            )
        }

    # --------------------------------------------------------
    # 3) Kerber archetype
    # --------------------------------------------------------

    elif LV_ARCHETYPE in KERBER_BLUEPRINT_FILES:

        bp = _load_kerber_blueprint(
            LV_ARCHETYPE
        )

        (
            lv_root_label,
            lv_bus_labels_all,
            house_to_lv_bus_label,
            bus_id_to_label,
            label_to_bus_id,
            bus_role,
        ) = _build_kerber_lv_labels(
            bp,
            id_list,
            pad=3,
        )

        validation = (
            _validate_kerber_blueprint_mapping(
                bp=bp,
                house_to_lv_bus_label=(
                    house_to_lv_bus_label
                ),
                label_to_bus_id=(
                    label_to_bus_id
                ),
                archetype=LV_ARCHETYPE,
                require_connection_kind=False,
            )
        )

        house_to_pp_bus_id = {
            household_id: int(
                label_to_bus_id[
                    lv_bus_label
                ]
            )
            for household_id, lv_bus_label
            in house_to_lv_bus_label.items()
        }

        label_table = (
            _build_kerber_lv_label_table(
                bp,
                bus_id_to_label,
                bus_role,
                house_to_lv_bus_label,
            )
        )

        if PRINT_KERBER_LV_LABEL_TABLE:
            _print_kerber_lv_label_table(
                label_table,
                archetype=LV_ARCHETYPE,
                max_rows=(
                    KERBER_LV_LABEL_TABLE_MAX_ROWS
                ),
            )

        lv_bus_labels_export = (
            lv_bus_labels_all.copy()
        )

        if len(lv_bus_labels_all) > int(
            MAX_LV_BUSES_TO_EXPORT
        ):

            pod_buses = sorted(
                set(
                    house_to_lv_bus_label.values()
                )
            )

            lv_bus_labels_export = (
                [lv_root_label]
                + [
                    b
                    for b in pod_buses
                    if b != lv_root_label
                ]
            )

            print(
                f"[INFO] Kerber '{LV_ARCHETYPE}': "
                f"{len(lv_bus_labels_all)} LV buses. "
                "For output/export, only root+POD buses "
                f"will be used ({len(lv_bus_labels_export)})."
            )

        state.update({
            "KERBER_BP": bp,
            "KERBER_BLUEPRINT_VALIDATION": validation,
            "KERBER_BUS_ID_TO_LABEL": bus_id_to_label,
            "KERBER_LABEL_TO_BUS_ID": label_to_bus_id,
            "KERBER_BUS_ROLE": bus_role,
            "KERBER_LV_LABEL_TABLE": label_table,
            "HOUSE_TO_LV_BUS_LABEL": (
                house_to_lv_bus_label
            ),
            "HOUSE_TO_PP_BUS_ID": (
                house_to_pp_bus_id
            ),
            "LV_ROOT_LABEL": lv_root_label,
            "LV_BUS_LABELS_ALL": (
                lv_bus_labels_all.copy()
            ),
            "LV_BUS_LABELS_EXPORT": (
                lv_bus_labels_export.copy()
            ),
        })

    else:

        raise ValueError(
            f"[LV] LV_ARCHETYPE='{LV_ARCHETYPE}' "
            "is not valid. Use: "
            "single_bus | legacy_rural_chain | "
            f"{list(KERBER_BLUEPRINT_FILES.keys())}"
        )

    return state








#---------------------------------------------------------------------
#           Sec 4.6 -- 11.1) EXCEL AND TIME UTILITIES
#---------------------------------------------------------------------
# Function to add multiple information sheets to an existing Excel file, e.g. LV-grid duration curves and DHW boost information
def write_many_sheets(xlsx_path: str, sheets: dict, overwrite: bool = True):
    """
    Writes multiple sheets to an Excel file.
    - overwrite=True: creates/rewrites the file from scratch (mode='w') -> much faster and does NOT load the existing workbook.
    - overwrite=False: if the file exists, appends and replaces only the provided sheets.
    """
    def _safe_sheetname(name: str) -> str:
        name = str(name)[:31]  # Excel limit
        return name

    if overwrite or (not os.path.exists(xlsx_path)):
        # NEW FILE, does not open/read anything -> faster
        with pd.ExcelWriter(xlsx_path, engine="openpyxl", mode="w") as w:
            for name, df in sheets.items():
                if df is None:
                    continue
                df.to_excel(w, sheet_name=_safe_sheetname(name), index=True)
    else:
        # APPEND, loads the existing workbook -> slower
        with pd.ExcelWriter(xlsx_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
            for name, df in sheets.items():
                if df is None:
                    continue
                df.to_excel(w, sheet_name=_safe_sheetname(name), index=True)


# Function to return specific text messages
def _message_df(message: str, meta: dict | None = None) -> pd.DataFrame:
    rows = [{"Messaggio": message}]
    if meta:
        rows += [{"Chiave": k, "Valore": v} for k, v in meta.items()]
    return pd.DataFrame(rows)


# Function to identify the current timestep duration
def _dt_hours_from_index(df_index, timestep_minutes_fallback=10):
    if isinstance(df_index, pd.DatetimeIndex) and len(df_index) > 1:
        return (df_index[1] - df_index[0]).total_seconds() / 3600.0
    return timestep_minutes_fallback / 60.


# Function to correctly connect the temporal indices exported with net flows for PF, avoiding the canonical 01-01-2000 index
def ensure_datetime_index(idx, name="index"):
    """
    Robustly converts idx into a DatetimeIndex.
    Supports strings such as '2023.02.18' or '2023-02-18' and variants with hourly data.
    """
    if isinstance(idx, pd.DatetimeIndex):
        return idx

    # Try direct parsing
    dt = pd.to_datetime(idx, errors="coerce", dayfirst=False, infer_datetime_format=True)

    # If all values are NaT, try replacing dots with hyphens, e.g. 2023.02.18
    if dt.isna().all():
        idx2 = pd.Index(idx).astype(str).str.replace(".", "-", regex=False)
        dt = pd.to_datetime(idx2, errors="coerce", dayfirst=False, infer_datetime_format=True)

    if dt.isna().any():
        bad = pd.Index(idx)[pd.isna(dt)].astype(str).unique()[:10]
        raise ValueError(f"[{name}] Non-parseable dates, first 10: {bad}")

    return pd.DatetimeIndex(dt)
#---------------------------------------------------------------------



#---------------------------------------------------------------------
#           Sec 4.6 -- 11.2) LV IMPORT/EXPORT/NET BY HOUSE
#---------------------------------------------------------------------
# Function to extract the flow withdrawn from the LV grid by each house c_i
def extract_lv_import_series_by_house(df_raw_case, ID, fill_zeros_if_missing=True):
    """
    Extracts, for each c_i, the import series from the LV grid to house c_i.
    Logical target: from_the_grid_BT_to_c_i  ->  c_i_bus_el   (flow)

    Works even if columns are:
      - nested MultiIndex: ((('from...', 'c_bus'), 'flow'))  [your case]
      - 3-level MultiIndex: ('from...', 'c_bus', 'flow')
      - MultiIndex with a different order
      - flattened string columns, fallback

    Returns dict: { 'c_i': pd.Series(kW) }.
    """
    out = {}

    cols = df_raw_case.columns

    # helper: "normalize" a column, tuple/string, into a list of string tokens
    def _tokens(col):
        # col can be: nested tuple, simple tuple, string
        tokens = []

        def rec(x):
            if isinstance(x, tuple):
                for y in x:
                    rec(y)
            else:
                s = str(x)
                tokens.append(s)

                # If 'A -> B' is found, also add 'A' and 'B' as separate tokens
                if "->" in s:
                    parts = [p.strip() for p in s.split("->") if p.strip()]
                    tokens.extend(parts)

        rec(col)
        return tokens

    # precompute tokens for all columns, low cost and faster lookup
    col_tokens = {col: _tokens(col) for col in cols}

    for c in ID:
        src = f"from_the_grid_BT_to_{c}"
        dst = f"{c}_bus_el"

        # 1) attempt: exact match, your format
        exact = ((src, dst), "flow")
        if exact in cols:
            s = df_raw_case[exact]
            out[c] = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
            continue

        # 2) attempt: other common formats
        cand_list = [
            (src, dst, "flow"),
            ("flow", src, dst),
            (src, "flow", dst),
            (dst, src, "flow"),
            (src, dst),  # in case there is no 'flow' level
        ]
        found = None
        for cand in cand_list:
            if cand in cols:
                found = cand
                break

        if found is not None:
            s = df_raw_case[found]
            out[c] = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
            continue

        # 3) robust fallback: search all columns containing src + dst (+ flow)
        matches = []
        for col, toks in col_tokens.items():
            has_src = any(t == src for t in toks)
            has_dst = any(t == dst for t in toks)
            has_flow = any(t == "flow" for t in toks)
            if has_src and has_dst and has_flow:
                matches.append(col)

        # if not found with 'flow', try without the flow constraint
        if not matches:
            for col, toks in col_tokens.items():
                has_src = any(t == src for t in toks)
                has_dst = any(t == dst for t in toks)
                if has_src and has_dst:
                    matches.append(col)

        if len(matches) == 1:
            s = df_raw_case[matches[0]]
            out[c] = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
        elif len(matches) > 1:
            # ambiguity: sum them, usually rare but safer than choosing one case
            ssum = None
            for m in matches:
                s = pd.to_numeric(df_raw_case[m], errors="coerce").fillna(0.0).astype(float)
                ssum = s if ssum is None else (ssum + s)
            out[c] = ssum
            print(f"[WARN] Ambiguous LV import for {c}: found {len(matches)} columns, summing them.")
        else:
            if fill_zeros_if_missing:
                out[c] = pd.Series(0.0, index=df_raw_case.index, dtype=float)
            # otherwise: skip

    return out


# Function to extract the flow injected/sold to the LV grid by each house c_i, i.e. export
def extract_lv_export_series_by_house(df_raw_case, ID, fill_zeros_if_missing=True):
    """
    Extracts, for each household c_i, the series (kW) fed into the LV grid (export):
        (f'{c_i}_bus_el' -> f'{c_i}_to_the_grid_BT')

    Works with:
    - tuple columns (from,to,'flow')
    - legacy ((from,to),'flow')
    - strings 'A -> B', also inside MultiIndex
    """
    out = {}
    cols = df_raw_case.columns

    def _tokens(col):
        tokens = []

        def rec(x):
            if isinstance(x, tuple):
                for y in x:
                    rec(y)
            else:
                s = str(x)
                tokens.append(s)
                if "->" in s:
                    parts = [p.strip() for p in s.split("->") if p.strip()]
                    tokens.extend(parts)

        rec(col)
        return tokens

    def _pick(require_flow: bool):
        for col in cols:
            tks = _tokens(col)
            # match, substrings are accepted
            has_bus = any(f"{case}_bus_el" in tk for tk in tks)
            has_exp = any(f"{case}_to_the_grid_BT" in tk for tk in tks)
            if not (has_bus and has_exp):
                continue

            if require_flow:
                if any(tk == "flow" or "flow" in tk for tk in tks):
                    return col
            else:
                return col
        return None

    for case in ID:
        chosen = _pick(require_flow=True)
        if chosen is None:
            chosen = _pick(require_flow=False)

        if chosen is None:
            out[case] = pd.Series(0.0, index=df_raw_case.index, dtype=float) if fill_zeros_if_missing else None
        else:
            out[case] = pd.to_numeric(df_raw_case[chosen], errors="coerce").fillna(0.0).astype(float)

    return out


# Function to calculate net import from the LV grid, import - export, for each house
def compute_net_lv_by_house(import_dict, export_dict):
    """
    net = import - export
    values < 0 => the house has excess and sells it to the LV grid
    """
    out = {}
    for c, s_imp in import_dict.items():
        s_exp = export_dict.get(c, None)
        if s_exp is None:
            s_exp = pd.Series(0.0, index=s_imp.index)
        out[c] = pd.to_numeric(s_imp, errors="coerce").fillna(0.0) - pd.to_numeric(s_exp, errors="coerce").fillna(0.0)
    return out
#---------------------------------------------------------------------







#---------------------------------------------------------------------
#           Sec 4.6 -- 11.3) DURATION CURVES AND EXCEEDANCE
#---------------------------------------------------------------------
# Function to calculate the actual LV grid-house c_i exchange duration curve
def build_duration_curves(import_dict):
    """
    Sorts each series in decreasing order -> duration curve.
    Returns df: index=rank (0..N-1), columns=c_i, values=kW sorted descending.
    """
    if not import_dict:
        return pd.DataFrame()

    N = len(next(iter(import_dict.values())))
    df = pd.DataFrame(index=np.arange(N))

    for c, s in import_dict.items():
        arr = np.asarray(s.values, dtype=float)
        arr = np.nan_to_num(arr, nan=0.0)
        arr_sorted = np.sort(arr)[::-1]  # decreasing
        df[c] = arr_sorted

    return df



# Function to create a duration curve from an input series
def duration_curve_from_series(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    return s.sort_values(ascending=False).reset_index(drop=True)


# Function to create a summary table of how many timesteps exceed a given power threshold, e.g. P > 6 kW
def exceedance_table_ts(
    import_dict: dict,
    thresholds_kw=(3, 4.5, 6, 9, 12, 15, 20),
    n_ts_week=1008,
    strict_gt=True,
) -> pd.DataFrame:
    """
    For each house c, counts how many timesteps (10') exceed each threshold (kW)
    and the percentage over the total weekly timesteps.
    
    Output:
    index = c_i
    columns:
        "max import power [kW]"
        ts_gt_{thr}kW
        pct_ts_gt_{thr}kW
    """
    if not import_dict:
        return pd.DataFrame()

    rows = []
    for c, s in import_dict.items():
        v = pd.to_numeric(s, errors="coerce").fillna(0.0).to_numpy(dtype=float)

        rec = {"c": c, "max import power [kW]": float(v.max()) if v.size else 0.0}

        for thr in thresholds_kw:
            if strict_gt:
                n_ts = int((v > thr).sum())
                col_ts  = f"timesteps with P>{thr} kW"
                col_pct = f"[% of the week] with P>{thr} kW"

            else:
                n_ts = int((v >= thr).sum())
                col_ts  = f"timesteps with P≥{thr} kW"
                col_pct = f"[% of the week] with P≥{thr} kW"


            rec[col_ts] = n_ts
            rec[col_pct] = float(n_ts) / float(n_ts_week) if n_ts_week else np.nan

        rows.append(rec)

    df = pd.DataFrame(rows).set_index("c")

    # sort c_1, c_2, ...
    try:
        df = df.sort_index(key=lambda idx: idx.str.split("_").str[1].astype(int))
    except Exception:
        df = df.sort_index()

    return df


# Function that calculates percentiles on timesteps where an LV-grid import power threshold is exceeded
def exceedance_summary_percentiles_ts(
        df_exceed: pd.DataFrame,
        thresholds_kw=(3, 4.5, 6, 9, 12, 15, 20),
        strict_gt=True,
    ) -> pd.DataFrame:
        """
        Percentiles (p10/p50/p90) and mean:
        - on timesteps above threshold (ts)
        - on the share of timesteps above threshold (%)
        aggregated across all houses.
        
        n_houses_with_exceedance = number of houses with at least 1 timestep above threshold
        """
        if df_exceed is None or df_exceed.empty:
            return pd.DataFrame()

        out = []
        for thr in thresholds_kw:
            if strict_gt:
                col_ts = f"timesteps with P>{thr} kW"
                col_pct = f"[% of the week] with P>{thr} kW"
            else:
                col_ts = f"ts_ge_{thr}kW"
                col_pct = f"pct_ts_ge_{thr}kW"

            if col_ts not in df_exceed.columns or col_pct not in df_exceed.columns:
                continue

            x_ts = pd.to_numeric(df_exceed[col_ts], errors="coerce").fillna(0.0).to_numpy(dtype=float)
            x_pct = pd.to_numeric(df_exceed[col_pct], errors="coerce").fillna(0.0).to_numpy(dtype=float)

            out.append({
                "Domestic import power threshold [kW]": thr,

                "p10 timesteps with P>threshold [no.]": float(np.percentile(x_ts, 10)),
                "p50 timesteps with P>threshold [no.]": float(np.percentile(x_ts, 50)),
                "p90 timesteps with P>threshold [no.]": float(np.percentile(x_ts, 90)),
                "mean timesteps with P>threshold [no.]": float(np.mean(x_ts)),

                "p10 share of week with P>threshold [%]": float(np.percentile(x_pct, 10)),
                "p50 share of week with P>threshold [%]": float(np.percentile(x_pct, 50)),
                "p90 share of week with P>threshold [%]": float(np.percentile(x_pct, 90)),
                "mean share of week with P>threshold [%]": float(np.mean(x_pct)),

                "no. houses with P>threshold [no.]": int(np.sum(x_ts > 0)),
            })

        return pd.DataFrame(out)
#---------------------------------------------------------------------







#---------------------------------------------------------------------
#           Sec 4.6 -- 11.4) DHW BOOST POST-PROCESSING
#---------------------------------------------------------------------
# Function to identify in which houses the additional DHW "boost" technology is used
def extract_dhw_boost_series_by_house(df_flows, ID):
    """
    Extracts, for each house c_i, the DHW_boost flow series:
    Converter c_i_DHW_boost : (c_i_bus_el -> c_i_bus_DHW_use)

    Returns dict: { 'c_i': pd.Series(kW) }.
    """
    out = {}

    for c in ID:
        # case 1: columns like ((from,to),'flow')
        col1 = ((f"{c}_bus_el", f"{c}_DHW_boost"), "flow")          # WARNING: depends on how oemof saves labels
        col2 = ((f"{c}_DHW_boost", f"{c}_bus_DHW_use"), "flow")     # more typical for converter: (label_converter, bus_out)

        # case 2: flattened columns (from,to,'flow')
        col1b = (f"{c}_bus_el", f"{c}_DHW_boost", "flow")
        col2b = (f"{c}_DHW_boost", f"{c}_bus_DHW_use", "flow")

        # Try all possibilities and take the existing one
        chosen = None
        for col in (col2, col2b, col1, col1b):
            if col in df_flows.columns:
                chosen = col
                break

        if chosen is None:
            continue

        out[c] = pd.to_numeric(df_flows[chosen], errors="coerce").fillna(0.0)

    return out


# Function to calculate the usage level of the DHW "boost" technology
def dhw_boost_usage_table(
    boost_dict: dict,
    n_ts_week: int,
    timestep_min: int = 10,
    eps_kw: float = 1e-6
) -> pd.DataFrame:
    """
    boost_dict: {c: Series(kW)}
    Output per house:
      - ts_active: # timesteps with P>0
      - pct_ts_active: ts_active / n_ts_week
      - n_starts: # activations (0 -> >0)
      - energy_kwh: sum kW * dt(h)
      - max_kw: peak kW
    """
    if not boost_dict:
        return pd.DataFrame()

    dt_h = timestep_min / 60.0

    rows = []
    for c, s in boost_dict.items():
        v = pd.to_numeric(s, errors="coerce").fillna(0.0).to_numpy(dtype=float)

        active = v > eps_kw
        ts_active = int(active.sum())

        # activations: how many times it switches from 0 to >0
        # active[0] True counts as start
        starts = int(np.sum(active & ~np.r_[False, active[:-1]]))

        energy_kwh = float(v.sum() * dt_h)
        max_kw = float(v.max()) if v.size else 0.0

        rows.append({
            "c": c,
            "ts_active (10min)": ts_active,
            "share_week [%]": 100.0 * ts_active / float(n_ts_week) if n_ts_week else np.nan,
            "n_starts": starts,
            "energy_week [kWh]": energy_kwh,
            "max [kW]": max_kw,
        })

    df = pd.DataFrame(rows).set_index("c")

    # sort c_1, c_2, ...
    try:
        df = df.sort_index(key=lambda idx: idx.str.split("_").str[1].astype(int))
    except Exception:
        df = df.sort_index()

    return df


# Function for calculating percentiles and summarizing DHW "boost" technology usage information
def dhw_boost_usage_summary_percentiles(df_usage: pd.DataFrame, qs=(10,50,90)) -> pd.DataFrame:
    """
    Percentile summary across all houses, for the main numeric columns.
    """
    if df_usage is None or df_usage.empty:
        return pd.DataFrame()

    cols = ["ts_active (10min)", "share_week [%]", "n_starts", "energy_week [kWh]", "max [kW]"]
    cols = [c for c in cols if c in df_usage.columns]

    out = {}
    for c in cols:
        x = pd.to_numeric(df_usage[c], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        for q in qs:
            out[f"{c} - p{q}"] = float(np.percentile(x, q))
        out[f"{c} - mean"] = float(np.mean(x))

    # number of houses using boost at least once
    if "ts_active (10min)" in df_usage.columns:
        out["n_houses_with_boost"] = int((df_usage["ts_active (10min)"] > 0).sum())

    return pd.DataFrame([out])


# Function to check whether the DHW "boost" technology was used and print/plot a warning message
def dhw_boost_is_used(boost_dict: dict, eps_kw: float = 1e-6) -> bool:
    """
    True if at least one house has at least one timestep with P_boost > eps_kw.
    """
    if not boost_dict:
        return False
    for s in boost_dict.values():
        v = pd.to_numeric(s, errors="coerce").fillna(0.0).to_numpy(dtype=float)
        if np.any(v > eps_kw):
            return True
    return False
#---------------------------------------------------------------------







#---------------------------------------------------------------------
#           Sec 4.6 -- 11.5) SEASON POST-PROCESSING EXPORT
#---------------------------------------------------------------------
# Function to save the post-processing of the observed configuration
def export_postproc_xlsx_for_season(
    flow_el_df,
    flow_grid_root,
    season,
    out_dir,
    mode,
    mode_label=None,
):
    """
    Saves the technical post-processing of the configuration/season.

    Replaces the old:
      postproc_OPT_<n>_nuc_<selected_region>_<season>.xlsx

    New:
      postproc_<season>.xlsx

    The configuration is already identified by the MODE folder.
    """
    os.makedirs(out_dir, exist_ok=True)

    df_result_el = convert_busseq_to_flowtuple_df(flow_el_df)
    lv_grid_root_results_df = convert_busseq_to_flowtuple_df(flow_grid_root)

    dt_hours = _dt_hours_from_index(
        df_result_el.index,
        timestep_minutes_fallback=timestep_minutes,
    )

    # ------------------------------------------------------------
    # LV import/export by house and duration curves
    # ------------------------------------------------------------
    lv_import_dict = extract_lv_import_series_by_house(df_result_el, ID)
    lv_export_dict = extract_lv_export_series_by_house(df_result_el, ID)
    net_lv_dict = compute_net_lv_by_house(lv_import_dict, lv_export_dict)

    lv_import_duration_df = (
        build_duration_curves(lv_import_dict)
        if lv_import_dict else pd.DataFrame()
    )

    lv_net_duration_df = (
        build_duration_curves(net_lv_dict)
        if net_lv_dict else pd.DataFrame()
    )

    n_ts_week = len(next(iter(lv_import_dict.values()))) if lv_import_dict else len(df_result_el.index)

    # ------------------------------------------------------------
    # LV import duration-curve percentiles
    # ------------------------------------------------------------
    if lv_import_duration_df is not None and not lv_import_duration_df.empty:
        qs_pct = (5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95)
        df_pct = pd.DataFrame(
            {
                f"p{q}": lv_import_duration_df.quantile(q / 100.0, axis=1)
                for q in qs_pct
            },
            index=lv_import_duration_df.index,
        )
    else:
        df_pct = pd.DataFrame()

    # ------------------------------------------------------------
    # LV import threshold-exceedance table
    # ------------------------------------------------------------
    df_exc = exceedance_table_ts(
        import_dict=lv_import_dict,
        thresholds_kw=thresholds,
        n_ts_week=n_ts_week,
        strict_gt=True,
    )

    df_exceed_pct = exceedance_summary_percentiles_ts(
        df_exceed=df_exc,
        thresholds_kw=thresholds,
        strict_gt=True,
    )

    df_exceed_pct_round = df_exceed_pct.copy()

    for col in df_exceed_pct_round.columns:
        if "ts" in col and "share" not in col:
            df_exceed_pct_round[col] = (
                pd.to_numeric(df_exceed_pct_round[col], errors="coerce")
                .round(0)
                .astype("Int64")
            )

        if "share settimana" in col or "share" in col:
            df_exceed_pct_round[col] = (
                100 * pd.to_numeric(df_exceed_pct_round[col], errors="coerce")
            ).round(1)

    # ------------------------------------------------------------
    # DHW boost usage
    # ------------------------------------------------------------
    dhw_boost_dict = extract_dhw_boost_series_by_house(df_result_el, ID)
    n_ts_week_boost = len(next(iter(dhw_boost_dict.values()))) if dhw_boost_dict else len(df_result_el.index)

    if not dhw_boost_is_used(dhw_boost_dict, eps_kw=1e-6):
        msg = (
            "The DHW_boost technology was not used for any "
            "of the observed houses."
        )
        meta = {
            "mode": mode,
            "mode_label": mode_label or mode,
            "season": season,
            "n_case_totali": len(ID),
            "n_ts_settimana": n_ts_week_boost,
            "nota": (
                "If you expected usage >0, check that DHW_boost is included "
                "in the model and that the column is available in the results."
            ),
        }
        df_dhw_boost_usage = _message_df(msg, meta)
        df_dhw_boost_summary = _message_df(msg, meta)
    else:
        df_dhw_boost_usage = dhw_boost_usage_table(
            boost_dict=dhw_boost_dict,
            n_ts_week=n_ts_week_boost,
            timestep_min=timestep_minutes,
            eps_kw=1e-6,
        )
        df_dhw_boost_summary = dhw_boost_usage_summary_percentiles(
            df_dhw_boost_usage
        )

    # ------------------------------------------------------------
    # Excel export
    # ------------------------------------------------------------
    xlsx_postproc = os.path.join(out_dir, f"postproc_{season}.xlsx")

    lv_import_duration_df_to_save = lv_import_duration_df.copy()
    if not lv_import_duration_df_to_save.empty:
        lv_import_duration_df_to_save.index.name = "Duration [ts=10min]"

    lv_net_duration_df_to_save = lv_net_duration_df.copy()
    if not lv_net_duration_df_to_save.empty:
        lv_net_duration_df_to_save.index.name = "Duration [ts=10min]"

    df_pct_to_save = df_pct.copy()
    if not df_pct_to_save.empty:
        df_pct_to_save.index.name = "Rank [10min timestep]"

    sheets_out = {
        "DurCurve_ImportBT": lv_import_duration_df_to_save,
        "DurCurve_ImportBT_PCT": df_pct_to_save,
        "DurCurve_NetBT": lv_net_duration_df_to_save,
        "ImportBT_Exceed": df_exc,
        "ImportBT_Exceed_summary": df_exceed_pct_round,
        "DHW_Boost_Usage": df_dhw_boost_usage,
        "DHW_Boost_Usage_Summary": df_dhw_boost_summary,
    }

    write_many_sheets(xlsx_postproc, sheets_out)

    print(f"[INFO][POSTPROC] Saved: {xlsx_postproc}")

    return {
        "postproc_xlsx": xlsx_postproc,
        "df_dur_import_bt": lv_import_duration_df,
        "df_dur_net_bt": lv_net_duration_df,
        "df_import_bt_exceed": df_exc,
        "df_dhw_boost_usage": df_dhw_boost_usage,
    }
#---------------------------------------------------------------------







#---------------------------------------------------------------------
#           Sec 4.6 -- 11.6) POWER-FLOW INPUT EXPORT by LV house/bus
#---------------------------------------------------------------------
# Function to collect the identifier for each customer, later used to calculate flows exchanged with the LV grid
def _get_lv_bus_key_for_house(c: str, i: int,
                             LV_ARCHETYPE: str,
                             num_households: int,
                             KERBER_BLUEPRINT_FILES,
                             HOUSE_TO_LV_BUS_LABEL: dict,
                             LV_ROOT_LABEL: str,
                             _lv_bus_for_house):
    """
    Returns the LV-bus label to which house c is connected,
    consistent with the logic used in 4.11c.
    """
    if LV_ARCHETYPE == "single_bus":                         # no topology
        return "bus_el_BT"
    elif LV_ARCHETYPE == "legacy_rural_chain":               # old single feeder
        return _lv_bus_for_house(i, num_households, feeder_code="Da")
    elif LV_ARCHETYPE in KERBER_BLUEPRINT_FILES:             # new Kerber topologies
        return HOUSE_TO_LV_BUS_LABEL.get(c, LV_ROOT_LABEL)
    else:
        raise ValueError(f"[LV] LV_ARCHETYPE='{LV_ARCHETYPE}' is not valid.")
    



# Function to sum a list of flows
def _sum_pf_result_edges(
    results,
    edge_labels,
    timeindex_canonico,
) -> pd.Series:
    """
    Sum a list of oemof.solph result edges.

    Parameters
    ----------
    edge_labels:
        Iterable of (from_label, to_label) pairs.

    Missing edges are interpreted as zero. This allows the same
    function to handle OPT, RB, and disabled technologies.
    """
    total = pd.Series(
        0.0,
        index=timeindex_canonico,
        dtype=float,
    )

    for from_label, to_label in edge_labels:
        edge_series = _get_edge_flow_series(
            results=results,
            from_label=from_label,
            to_label=to_label,
            timeindex_canonico=timeindex_canonico,
        )

        total = total.add(
            edge_series,
            fill_value=0.0,
        )

    return total.astype(float)


# Function to reconstruct all the components/technologies present at each household
def build_pf_household_component_profiles(
    results,
    house_ids,
    timeindex_canonico,
) -> dict:
    """
    Reconstruct active-power components at each household
    electric bus.

    All returned values use the original MODULE_04 power unit
    (currently kW). Missing edges are interpreted as zero.
    """
    house_ids = list(house_ids)

    component_names = [
        "base",
        "cooking",
        "cooling",
        "heating",

        "dhw_charge",
        "bev_home_charge",
        "bess_charge",

        "pv_generation",
        "pv_curtailment",

        "bess_discharge",
        "v2g_generation",

        "grid_import_house_side",
        "grid_export_house_side",
    ]

    component_profiles = {
        component: pd.DataFrame(
            0.0,
            index=timeindex_canonico,
            columns=house_ids,
            dtype=float,
        )
        for component in component_names
    }

    for household_id in house_ids:
        household_bus = f"{household_id}_bus_el"

        edge_groups = {
            "base": [
                (
                    household_bus,
                    f"{household_id}_dmd_elc_base",
                ),
            ],
            "cooking": [
                (
                    household_bus,
                    f"{household_id}_dmd_elc_Cook",
                ),
            ],
            "cooling": [
                (
                    household_bus,
                    f"{household_id}_dmd_elc_Cool",
                ),
            ],
            "heating": [
                (
                    household_bus,
                    f"{household_id}_dmd_elc_Heat",
                ),
            ],
            "dhw_charge": [
                (
                    household_bus,
                    f"{household_id}_DHW_storage",
                ),
                (
                    household_bus,
                    f"{household_id}_DHW_charge_RB_sink",
                ),
                (
                    household_bus,
                    f"{household_id}_DHW_boost",
                ),
            ],
            "bev_home_charge": [
                (
                    household_bus,
                    f"{household_id}_charger",
                ),
                (
                    household_bus,
                    f"{household_id}_BEV_charge_RB_sink",
                ),
            ],
            "bess_charge": [
                (
                    household_bus,
                    f"{household_id}_BESS_Opt",
                ),
                (
                    household_bus,
                    f"{household_id}_BESS_charge_RB",
                ),
            ],
            "pv_generation": [
                (
                    f"{household_id}_PV",
                    household_bus,
                ),
                (
                    f"{household_id}_PV_to_house",
                    household_bus,
                ),
            ],

            "pv_curtailment": [
                (
                    f"{household_id}_bus_el_PV",
                    f"{household_id}_PV_curtail",
                ),
            ],


            "bess_discharge": [
                (
                    f"{household_id}_BESS_Opt",
                    household_bus,
                ),
                (
                    f"{household_id}_BESS_discharge_RB",
                    household_bus,
                ),
            ],
            "v2g_generation": [
                (
                    f"{household_id}_V2G",
                    household_bus,
                ),
            ],
            "grid_import_house_side": [
                (
                    f"from_the_grid_BT_to_{household_id}",
                    household_bus,
                ),
            ],
            "grid_export_house_side": [
                (
                    household_bus,
                    f"{household_id}_to_the_grid_BT",
                ),
            ],
        }

        for component_name, edge_labels in edge_groups.items():
            component_profiles[component_name][household_id] = (
                _sum_pf_result_edges(
                    results=results,
                    edge_labels=edge_labels,
                    timeindex_canonico=timeindex_canonico,
                )
            )

    zero_df = pd.DataFrame(
        0.0,
        index=timeindex_canonico,
        columns=house_ids,
        dtype=float,
    )

    end_use_components = [
        "base",
        "cooking",
        "cooling",
        "heating",
        "dhw_charge",
        "bev_home_charge",
    ]

    generation_components = [
        "pv_generation",
        "bess_discharge",
        "v2g_generation",
    ]

    p_gross_end_use_kw = sum(
        (
            component_profiles[component]
            for component in end_use_components
        ),
        start=zero_df.copy(),
    )

    p_storage_charge_kw = component_profiles[
        "bess_charge"
    ].copy()

    p_gross_absorption_kw = (
        p_gross_end_use_kw
        + p_storage_charge_kw
    )

    p_local_generation_kw = sum(
        (
            component_profiles[component]
            for component in generation_components
        ),
        start=zero_df.copy(),
    )

    p_pv_curtailment_kw = component_profiles[
        "pv_curtailment"
    ]

    p_pv_available_kw = (
        component_profiles["pv_generation"]
        + p_pv_curtailment_kw
    )

    p_household_balance_residual_kw = (
        component_profiles["grid_import_house_side"]
        + p_local_generation_kw
        - p_gross_absorption_kw
        - component_profiles["grid_export_house_side"]
    )

    return {
        "components_kw": component_profiles,
        "p_gross_end_use_kw": p_gross_end_use_kw,
        "p_storage_charge_kw": p_storage_charge_kw,
        "p_gross_absorption_kw": p_gross_absorption_kw,
        "p_local_generation_kw": p_local_generation_kw,
        "p_household_balance_residual_kw": (p_household_balance_residual_kw),
        
        "p_pv_curtailment_kw": (p_pv_curtailment_kw),
        "p_pv_available_kw": (p_pv_available_kw),
    }



# Function to validate active power reconstruction within the household (end-uses v. exchanged power with LV grid)
def validate_pf_household_reconstruction(
    p_import_network_kw,
    p_export_network_kw,
    p_gross_absorption_kw,
    p_local_generation_kw,
    component_profiles_kw,
    eta_grid: float = 1.0,
    tolerance_kw: float = 1e-6,
    strict: bool = True,
) -> tuple[pd.DataFrame, dict, dict]:
    """
    Validate the household active-power reconstruction.

    Checks:
      1. household-bus active-power balance;
      2. import-converter consistency;
      3. export-converter consistency;
      4. network-side net-exchange consistency;
      5. direct internal-vs-network net balance when eta_grid=1.
    """
    eta_grid = float(eta_grid)
    tolerance_kw = float(tolerance_kw)

    if eta_grid <= 0.0:
        raise ValueError(f"eta_grid must be positive. Received: {eta_grid}")

    required_components = {"grid_import_house_side", "grid_export_house_side"}
    missing_components = required_components - set(component_profiles_kw)

    if missing_components:
        raise KeyError(
            "Missing household-side grid-exchange components: "
            f"{sorted(missing_components)}"
        )

    reference_index = p_import_network_kw.index
    reference_columns = p_import_network_kw.columns

    def _align(df):
        return (
            df.reindex(index=reference_index, columns=reference_columns)
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
            .astype(float)
        )

    p_import_network_kw = _align(p_import_network_kw)
    p_export_network_kw = _align(p_export_network_kw)
    p_gross_absorption_kw = _align(p_gross_absorption_kw)
    p_local_generation_kw = _align(p_local_generation_kw)
    p_import_house_kw = _align(component_profiles_kw["grid_import_house_side"])
    p_export_house_kw = _align(component_profiles_kw["grid_export_house_side"])

    p_net_network_kw = p_import_network_kw - p_export_network_kw
    p_net_internal_kw = p_gross_absorption_kw - p_local_generation_kw
    p_net_house_exchange_kw = p_import_house_kw - p_export_house_kw

    p_import_network_reconstructed_kw = p_import_house_kw / eta_grid
    p_export_network_reconstructed_kw = p_export_house_kw * eta_grid
    p_net_network_reconstructed_kw = (
        p_import_network_reconstructed_kw - p_export_network_reconstructed_kw
    )

    residuals_kw = {
        "house_bus_balance": p_net_house_exchange_kw - p_net_internal_kw,
        "import_converter": (
            p_import_network_kw - p_import_network_reconstructed_kw
        ),
        "export_converter": (
            p_export_network_kw - p_export_network_reconstructed_kw
        ),
        "network_net_exchange": (
            p_net_network_kw - p_net_network_reconstructed_kw
        ),
    }

    if np.isclose(eta_grid, 1.0):
        residuals_kw["direct_internal_vs_network_net"] = (
            p_net_network_kw - p_net_internal_kw
        )

    summary_rows = []

    for check_name, residual_df in residuals_kw.items():
        residual_abs = residual_df.abs()
        stacked_abs = residual_abs.stack(dropna=True)

        if stacked_abs.empty:
            max_abs_residual_kw = np.nan
            rmse_kw = np.nan
            worst_timestamp = None
            worst_household = None
            passed = False
        else:
            worst_timestamp, worst_household = stacked_abs.idxmax()
            max_abs_residual_kw = float(stacked_abs.max())
            rmse_kw = float(
                np.sqrt(np.mean(residual_df.to_numpy(dtype=float) ** 2))
            )
            passed = bool(max_abs_residual_kw <= tolerance_kw)

        summary_rows.append({
            "check": check_name,
            "max_abs_residual_kw": max_abs_residual_kw,
            "rmse_kw": rmse_kw,
            "worst_timestamp": (
                str(worst_timestamp) if worst_timestamp is not None else None
            ),
            "worst_household": worst_household,
            "tolerance_kw": tolerance_kw,
            "passed": passed,
        })

    validation_summary = pd.DataFrame(summary_rows)

    simultaneous_exchange_mask = (
        (p_import_network_kw > tolerance_kw)
        & (p_export_network_kw > tolerance_kw)
    )
    n_simultaneous_exchange_values = int(
        simultaneous_exchange_mask.to_numpy(dtype=bool).sum()
    )

    print("\n[PF EXPORT VALIDATION]")
    print(
        validation_summary[
            ["check", "max_abs_residual_kw", "rmse_kw", "passed"]
        ].to_string(index=False)
    )

    if n_simultaneous_exchange_values:
        print(
            "[WARN][PF EXPORT] Simultaneous positive import and export "
            f"were found in {n_simultaneous_exchange_values} "
            "household-timestep combinations."
        )

    failed_checks = validation_summary.loc[~validation_summary["passed"]]

    if not failed_checks.empty:
        message = (
            "[PF EXPORT] Household-flow reconstruction failed.\n"
            + failed_checks.to_string(index=False)
        )

        if strict:
            raise ValueError(message)

        print(f"[WARN] {message}")
    else:
        print(
            "[OK][PF EXPORT] Reconstructed household flows are coherent "
            "with the previously exported import, export and net profiles."
        )

    validation_info = {
        "eta_grid": eta_grid,
        "tolerance_kw": tolerance_kw,
        "strict": bool(strict),
        "all_checks_passed": bool(failed_checks.empty),
        "n_simultaneous_exchange_values": n_simultaneous_exchange_values,
    }

    return validation_summary, residuals_kw, validation_info



# Function to build a reactive-power profile for each household
def build_assumed_pf_reactive_power(
    component_profiles_kw: dict,
    cosphi_by_component: dict,
) -> tuple[pd.DataFrame, dict]:
    """
    Build assumed household reactive-power profiles.

    Sign convention:
        Q > 0 = reactive-power absorption.

    Local generation is assumed to operate at unity power
    factor and therefore contributes no reactive power.
    """
    if not component_profiles_kw:
        return pd.DataFrame(), {}

    first_df = next(iter(component_profiles_kw.values()))
    q_total_kvar = pd.DataFrame(
        0.0,
        index=first_df.index,
        columns=first_df.columns,
        dtype=float,
    )
    q_components_kvar = {}

    for component_name, cos_phi in cosphi_by_component.items():
        if component_name not in component_profiles_kw:
            raise KeyError(
                f"No active-power component named '{component_name}' "
                "was reconstructed."
            )

        cos_phi = float(cos_phi)

        if not 0.0 < cos_phi <= 1.0:
            raise ValueError(
                "Power factor must belong to (0, 1]. "
                f"Component={component_name}, value={cos_phi}"
            )

        tan_phi = (
            0.0
            if np.isclose(cos_phi, 1.0)
            else float(np.tan(np.arccos(cos_phi)))
        )

        q_component_kvar = (
            component_profiles_kw[component_name]
            .clip(lower=0.0)
            .mul(tan_phi)
        )

        q_components_kvar[component_name] = q_component_kvar
        q_total_kvar = q_total_kvar.add(q_component_kvar, fill_value=0.0)

    return q_total_kvar, q_components_kvar



# Function to save net flows exchanged between LV grid and customers as output.py
def export_pf_inputs_house_lv(
    results,
    noded,
    LV_ARCHETYPE: str,
    num_households: int,
    KERBER_BLUEPRINT_FILES,
    HOUSE_TO_LV_BUS_LABEL: dict,
    LV_ROOT_LABEL: str,
    _lv_bus_for_house,
    out_dir: Path,
    season_tag: str,
    mode_tag: str,
    timeindex_canonico,
    to_mw_factor: float = 1 / 1000,
    idx_real=None,
    export_component_profiles: bool = True,
    export_assumed_q: bool = True,
    cosphi_by_component: dict | None = None,
    balance_tolerance_kw: float = 1e-6,
    eta_grid: float = 1.0,
    validate_reconstruction: bool = True,
    validation_strict: bool = True,
    save_validation_details: bool = True,
    run_context=None,
):
    """
    Export household active- and reactive-power profiles for MODULE_05.

    Active-power convention:
        P > 0 = net demand from the LV grid;
        P < 0 = net export to the LV grid.

    Reactive-power convention:
        Q > 0 = reactive-power absorption;
        Q < 0 = reactive-power injection.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    houses = [f"c_{i}" for i in range(1, num_households + 1)]
    empty_df = pd.DataFrame(
        0.0,
        index=timeindex_canonico,
        columns=houses,
        dtype=float,
    )

    p_import_network_kw = empty_df.copy()
    p_export_network_kw = empty_df.copy()
    bus_key_by_house = {}

    run_context = (
        run_context
        if isinstance(
            run_context,
            dict,
        )
        else {}
    )


    for household_number, household_id in enumerate(houses, start=1):
        lv_bus_key = _get_lv_bus_key_for_house(
            household_id,
            household_number,
            LV_ARCHETYPE=LV_ARCHETYPE,
            num_households=num_households,
            KERBER_BLUEPRINT_FILES=KERBER_BLUEPRINT_FILES,
            HOUSE_TO_LV_BUS_LABEL=HOUSE_TO_LV_BUS_LABEL,
            LV_ROOT_LABEL=LV_ROOT_LABEL,
            _lv_bus_for_house=_lv_bus_for_house,
        )

        bus_key_by_house[household_id] = lv_bus_key

        p_import_network_kw[household_id] = _get_edge_flow_series(
            results=results,
            from_label=lv_bus_key,
            to_label=f"from_the_grid_BT_to_{household_id}",
            timeindex_canonico=timeindex_canonico,
        )

        p_export_network_kw[household_id] = _get_edge_flow_series(
            results=results,
            from_label=f"{household_id}_to_the_grid_BT",
            to_label=lv_bus_key,
            timeindex_canonico=timeindex_canonico,
        )

    p_net_network_kw = p_import_network_kw - p_export_network_kw

    component_output = build_pf_household_component_profiles(
        results=results,
        house_ids=houses,
        timeindex_canonico=timeindex_canonico,
    )

    component_profiles_kw = component_output["components_kw"]
    p_gross_end_use_kw = component_output["p_gross_end_use_kw"]
    p_storage_charge_kw = component_output["p_storage_charge_kw"]
    p_gross_absorption_kw = component_output["p_gross_absorption_kw"]
    p_local_generation_kw = component_output["p_local_generation_kw"]
    p_balance_residual_kw = component_output[
        "p_household_balance_residual_kw"
    ]

    max_abs_balance_residual_kw = float(
        p_balance_residual_kw.abs().to_numpy(dtype=float).max()
    )

    validation_summary = pd.DataFrame()
    validation_residuals_kw = {}
    validation_info = {
        "enabled": bool(validate_reconstruction),
        "all_checks_passed": None,
    }

    if validate_reconstruction:
        (
            validation_summary,
            validation_residuals_kw,
            validation_info,
        ) = validate_pf_household_reconstruction(
            p_import_network_kw=p_import_network_kw,
            p_export_network_kw=p_export_network_kw,
            p_gross_absorption_kw=p_gross_absorption_kw,
            p_local_generation_kw=p_local_generation_kw,
            component_profiles_kw=component_profiles_kw,
            eta_grid=eta_grid,
            tolerance_kw=balance_tolerance_kw,
            strict=validation_strict,
        )



    cosphi_by_component = cosphi_by_component or {}

    if export_assumed_q:
        q_total_kvar, q_components_kvar = build_assumed_pf_reactive_power(
            component_profiles_kw=component_profiles_kw,
            cosphi_by_component=cosphi_by_component,
        )
    else:
        q_total_kvar = empty_df.copy()
        q_components_kvar = {}

    factor = float(to_mw_factor)

    frames = {
        "p_import_mw": p_import_network_kw * factor,
        "p_export_mw": p_export_network_kw * factor,
        "p_net_mw": p_net_network_kw * factor,
        "p_gross_end_use_mw": p_gross_end_use_kw * factor,
        "p_storage_charge_mw": p_storage_charge_kw * factor,
        "p_gross_absorption_mw": p_gross_absorption_kw * factor,
        "p_local_generation_mw": p_local_generation_kw * factor,
        "p_balance_residual_mw": p_balance_residual_kw * factor,
        "q_total_mvar": q_total_kvar * factor,
    }

    if validate_reconstruction and validation_residuals_kw:
        frames["validation_residuals_mw"] = pd.concat(
            {
                check_name: residual_df * factor
                for check_name, residual_df in validation_residuals_kw.items()
            },
            axis=1,
        )
        frames["validation_residuals_mw"].columns.names = [
            "validation_check",
            "household",
        ]


    frames["p_components_mw"] = pd.concat(
        {
            component: component_df * factor
            for component, component_df in component_profiles_kw.items()
            if component not in {
                "grid_import_house_side",
                "grid_export_house_side",
            }
        },
        axis=1,
    )
    frames["p_components_mw"].columns.names = ["component", "household"]

    if q_components_kvar:
        frames["q_components_mvar"] = pd.concat(
            {
                component: component_df * factor
                for component, component_df in q_components_kvar.items()
            },
            axis=1,
        )
        frames["q_components_mvar"].columns.names = [
            "component",
            "household",
        ]
    else:
        frames["q_components_mvar"] = pd.DataFrame(
            index=timeindex_canonico
        )

    if idx_real is not None:
        idx_real = ensure_datetime_index(idx_real, name="idx_real")

        for frame_name, frame in frames.items():
            if len(frame) != len(idx_real):
                raise ValueError(
                    f"[PF EXPORT] Length mismatch for {frame_name}: "
                    f"{len(frame)} vs {len(idx_real)} timestamps."
                )

            frame.index = idx_real


    stem = _safe_fs_tag(season_tag)

    output_files = {
        "p_import_mw": f"p_import_mw_{stem}.pkl.gz",
        "p_export_mw": f"p_export_mw_{stem}.pkl.gz",

        "p_net_mw": f"p_net_mw_{stem}.pkl.gz",
        "p_gross_end_use_mw": f"p_gross_end_use_mw_{stem}.pkl.gz",

        "p_storage_charge_mw": f"p_storage_charge_mw_{stem}.pkl.gz",
        "p_gross_absorption_mw": f"p_gross_absorption_mw_{stem}.pkl.gz",
        "p_local_generation_mw": f"p_local_generation_mw_{stem}.pkl.gz",
        "p_components_mw": f"p_components_mw_{stem}.pkl.gz",

        "q_total_mvar": f"q_total_mvar_{stem}.pkl.gz",
        "q_components_mvar": f"q_components_mvar_{stem}.pkl.gz",

        "p_balance_residual_mw": f"p_balance_residual_mw_{stem}.pkl.gz",
        "validation_residuals_mw": f"validation_residuals_mw_{stem}.pkl.gz",
        "validation_summary": f"validation_summary_{stem}.csv",
    }

    frames_to_save = {
        "p_import_mw": frames["p_import_mw"],
        "p_export_mw": frames["p_export_mw"],
        "p_net_mw": frames["p_net_mw"],
        "p_gross_end_use_mw": frames["p_gross_end_use_mw"],
        "p_storage_charge_mw": frames["p_storage_charge_mw"],
        "p_gross_absorption_mw": frames["p_gross_absorption_mw"],
        "p_local_generation_mw": frames["p_local_generation_mw"],
        "p_balance_residual_mw": frames["p_balance_residual_mw"],
    }

    if export_component_profiles:
        frames_to_save["p_components_mw"] = frames["p_components_mw"]

    if export_assumed_q:
        frames_to_save["q_total_mvar"] = frames["q_total_mvar"]
        frames_to_save["q_components_mvar"] = frames[
            "q_components_mvar"
        ]

    for frame_name, frame in frames_to_save.items():
        frame.to_pickle(
            out_dir / output_files[frame_name],
            compression="gzip",
        )

    if (
        validate_reconstruction
        and save_validation_details
        and not validation_summary.empty
    ):


        # Check for path length
        validation_path = (
            out_dir
            / output_files["validation_residuals_mw"]
        )

        print(
            "[DEBUG][PF PATH]",
            validation_path
        )

        print(
            "[DEBUG][PF PATH] absolute length =",
            len(str(validation_path.resolve()))
        )

        print(
            "[DEBUG][PF PATH] parent exists =",
            validation_path.parent.exists()
        )
        # -----


        frames["validation_residuals_mw"].to_pickle(
            out_dir / output_files["validation_residuals_mw"],
            compression="gzip",
        )

        validation_summary.to_csv(
            out_dir / output_files["validation_summary"],
            index=False,
        )



    meta = {
        "schema_version": 2,
        "bt_archetype": LV_ARCHETYPE,
        "season": season_tag,
        "mode": mode_tag,
        "to_mw_factor": factor,
        "active_power_unit": "MW",
        "reactive_power_unit": "Mvar",
        "p_net_sign_convention": (
            "positive = net demand from LV grid; "
            "negative = net export to LV grid"
        ),
        "q_sign_convention": (
            "positive = reactive-power absorption; "
            "negative = reactive-power injection"
        ),
        "houses": houses,
        "bt_bus_key_by_house": bus_key_by_house,
        "reactive_power_method": (
            "component_active_power_times_tan_arccos_cosphi"
            if export_assumed_q
            else "not_exported"
        ),
        "cosphi_by_component": {
            key: float(value)
            for key, value in cosphi_by_component.items()
        },
        "generation_q_assumption": (
            PF_EXPORT_GENERATION_Q_ASSUMPTION
        ),
        "max_household_balance_residual_kw": (
            max_abs_balance_residual_kw
        ),
        "balance_tolerance_kw": float(balance_tolerance_kw),
        "files": output_files,
        "component_definitions": {
            "gross_end_use": [
                "base",
                "cooking",
                "cooling",
                "heating",
                "dhw_charge",
                "bev_home_charge",
            ],
            "storage_charge": ["bess_charge"],
            "gross_absorption": "gross_end_use + storage_charge",
            "local_generation": [
                "pv_generation",
                "bess_discharge",
                "v2g_generation",
            ],
            "outdoor_bev_charging": (
                "excluded because it does not occur at the "
                "household LV connection"
            ),
        },

        "active_power_validation": {
            **validation_info,
            "summary": (
                validation_summary.to_dict(orient="records")
                if not validation_summary.empty
                else []
            ),
        },

        "run_uid": (run_context.get("run_uid")),
        "run_label": (run_context.get("run_label")),
        "experiment_id": (run_context.get("experiment_id")),
        "run_revision": (run_context.get("run_revision")),
        "electrification_id": (run_context.get("electrification_id")),
        "case_label": (run_context.get("case_label")),
        "control": (run_context.get("control")),
        "shares_target": (run_context.get("shares")),

    }


    with open(
        out_dir / f"pf_meta_{stem}.json",
        "w",
        encoding="utf-8",
    ) as file_obj:
        json.dump(meta, file_obj, indent=2, ensure_ascii=False)

    print(f"[OK][PF EXPORT] Saved MODULE_05 inputs in:\n  {out_dir}")
    print(
        "[PF EXPORT] P net range [MW]: "
        f"{frames['p_net_mw'].min().min():.6f} to "
        f"{frames['p_net_mw'].max().max():.6f}"
    )
    print(
        "[PF EXPORT] Gross absorption peak [MW]: "
        f"{frames['p_gross_absorption_mw'].max().max():.6f}"
    )
    print(
        "[PF EXPORT] Local generation peak [MW]: "
        f"{frames['p_local_generation_mw'].max().max():.6f}"
    )
    print(
        "[PF EXPORT] Q absorption peak [Mvar]: "
        f"{frames['q_total_mvar'].max().max():.6f}"
    )

    return frames["p_net_mw"], meta
#---------------------------------------------------------------------







#---------------------------------------------------------------------
#           Sec 4.6 -- 11.7) PF LIMITS AND KERBER LINE RECIPES
#---------------------------------------------------------------------
#   Functions for importing grid limits from PF analysis
# Function that loads transmission limits defined by PF analysis for the new "operation-optimization" run
def _load_pf_line_limits_for_season(
    current_season: str,
    lv_archetype: str | None = None,
    hagrid_mode: str | None = None,
) -> dict[int, float]:
    """Reads the PF-calibrated line limits produced by notebook 05."""
    if not USE_PF_LINE_LIMITS_V1:
        return {}

    lv = lv_archetype
    if lv is None:
        lv = globals().get("BT_ARCHETYPE", globals().get("HAGRID_BT_ARCHETYPE", None))

    mode = hagrid_mode
    if mode is None:
        mode = globals().get("MODE", globals().get("HAGRID_MODE", None))

    if lv is None or mode is None:
        print(
            f"[WARN][PF LIMITS] Could not build the filename: "
            f"BT_ARCHETYPE={lv}, MODE/HAGRID_MODE={mode}"
        )
        return {}

    fp = os.path.join(
        pf_limits_input_folder,
        f"pf_{str(PF_LIMITS_PF_MODE).strip().lower()}_{lv}_{mode}_{current_season}.xlsx"
    )

    if not os.path.exists(fp):
        print(f"[WARN][PF LIMITS] File not found for season {current_season}: {fp}")
        return {}

    try:
        df_lim = pd.read_excel(fp, sheet_name=PF_LIMITS_SHEET)
    except Exception as e:
        print(f"[WARN][PF LIMITS] Could not read sheet '{PF_LIMITS_SHEET}' from {fp}: {type(e).__name__}: {e}")
        return {}

    req = {"line_id", "p_limit_kw_feas"}
    if not req.issubset(set(df_lim.columns)):
        print(f"[WARN][PF LIMITS] Missing columns in {fp}: expected {req}, found {set(df_lim.columns)}")
        return {}

    out = {}
    for _, r in df_lim.iterrows():
        try:
            lid = int(r["line_id"])
            val = float(r["p_limit_kw_feas"])
        except Exception:
            continue
        if np.isfinite(val) and val > 0.0:
            out[lid] = val
    return out


# Function that identifies the new grid transmission power limits to assign to the Kerber grid
def _get_kerber_line_p_limit_kw(e: dict, pf_limits_map: dict[int, float] | None = None) -> float:
    lid = e.get("line_id", None)
    lid = int(lid) if lid is not None and str(lid) != "" else None

    p_base = None
    raw = e.get("pmax_kw")
    if raw is not None:
        try:
            val = float(raw)
            if np.isfinite(val) and val > 0.0:
                p_base = val
        except Exception:
            pass

    p_pf = None
    if pf_limits_map and lid is not None and lid in pf_limits_map:
        try:
            val = float(pf_limits_map[lid])
            if np.isfinite(val) and val > 0.0:
                p_pf = val
        except Exception:
            pass

    if p_base is not None and p_pf is not None:
        return min(p_base, p_pf)
    if p_pf is not None:
        return p_pf
    if p_base is not None:
        return p_base

    return float(KERBER_DEFAULT_PMAX_KW)


# Function to import transformer/trafo limits
def _load_pf_interface_limits_for_season(
    current_season: str,
    lv_archetype: str | None = None,
    hagrid_mode: str | None = None,
) -> dict[str, float]:

    if not USE_PF_LINE_LIMITS_V1:
        return {}

    lv = lv_archetype if lv_archetype is not None else globals().get("BT_ARCHETYPE", globals().get("HAGRID_BT_ARCHETYPE", None))
    mode = hagrid_mode if hagrid_mode is not None else globals().get("MODE", globals().get("HAGRID_MODE", None))

    if lv is None or mode is None:
        return {}

    fp = os.path.join(
        pf_limits_input_folder,
        f"pf_{str(PF_LIMITS_PF_MODE).strip().lower()}_{lv}_{mode}_{current_season}.xlsx"
    )

    if not os.path.exists(fp):
        return {}

    try:
        df_lim = pd.read_excel(fp, sheet_name=PF_INTERFACE_SHEET)
    except Exception:
        return {}

    req = {"direction", "p_limit_kw_feas"}
    if not req.issubset(set(df_lim.columns)):
        return {}

    out = {}
    for _, r in df_lim.iterrows():
        try:
            direction = str(r["direction"]).strip().lower()
            val = float(r["p_limit_kw_feas"])
        except Exception:
            continue
        if np.isfinite(val) and val > 0.0:
            out[direction] = val
    return out


# Function that identifies the new transformer/trafo transmission power limits to assign to the Kerber grid
def _get_pf_interface_limit_kw(
    pf_limits_map: dict[str, float] | None,
    direction: str,
    fallback_kw: float,
) -> float:
    p_pf = None
    if pf_limits_map:
        for k in (str(direction).lower(), "both"):
            if k in pf_limits_map:
                try:
                    val = float(pf_limits_map[k])
                    if np.isfinite(val) and val > 0.0:
                        p_pf = val
                        break
                except Exception:
                    pass

    p_base = float(fallback_kw)

    if p_pf is not None and np.isfinite(p_base) and p_base > 0.0:
        return min(p_pf, p_base)
    if p_pf is not None:
        return p_pf
    return p_base



# ---------------------
#   Functions for output creation and grid topology
# Function to define grid topology in order to define line saturation limits
def build_kerber_line_recipes_for_current_limits(PF_LINE_LIMITS_MAP=None):
    """
    Builds Kerber line recipes for calculating net flows and saturations.
    """
    recipes = []

    if LV_ARCHETYPE not in KERBER_BLUEPRINT_FILES:
        return recipes

    if PF_LINE_LIMITS_MAP is None:
        PF_LINE_LIMITS_MAP = {}

    for e in KERBER_BP["lines_lv"]:
        fb = int(e["from_bus"])
        tb = int(e["to_bus"])

        a_key = KERBER_BUS_ID_TO_LABEL[fb]
        b_key = KERBER_BUS_ID_TO_LABEL[tb]

        a_s = _short_lv_bus_label(a_key)
        b_s = _short_lv_bus_label(b_key)

        role_a = KERBER_BUS_ROLE.get(a_key, "unknown")
        role_b = KERBER_BUS_ROLE.get(b_key, "unknown")

        pmax_kw = _get_kerber_line_p_limit_kw(e, PF_LINE_LIMITS_MAP)


        connection_kind = str(
            e.get(
                "connection_kind",
                "",
            )
        ).strip().lower()


        if connection_kind == "branch_house":
            kind = "branch_house"

        elif connection_kind == "network_internal":
            # Retain the current plotting category for backward compatibility.
            kind = "backbone"

        else:
            # Backward-compatible fallback for old blueprints that do not
            # contain connection_kind.
            touches_household_bus = (
                "house" in (
                    role_a,
                    role_b,
                )
                or a_s.startswith("h")
                or b_s.startswith("h")
            )

            connection_kind = (
                "branch_house"
                if touches_household_bus
                else "network_internal"
            )

            kind = (
                "branch_house"
                if touches_household_bus
                else "backbone"
            )


        recipes.append({
            "line_id": int(e["line_id"]),

            "a_key": a_key,
            "b_key": b_key,
            "a_s": a_s,
            "b_s": b_s,
            "lab_forw": f"line_forw_{a_s}_to_{b_s}",
            "lab_back": f"line_back_{b_s}_to_{a_s}",
            "role_a": role_a,
            "role_b": role_b,
            "pmax_kw": pmax_kw,

            "connection_kind": connection_kind,

            # Retained for compatibility with the current plots.
            "kind": kind,
        })

    return recipes
#---------------------------------------------------------------------







#---------------------------------------------------------------------
#           Sec 4.6 -- 11.8) NET LINE OUTPUTS FROM MIXED operation RESULTS
#---------------------------------------------------------------------
# Function to find columns even if they contain oemof objects in the dataframes loaded for chart print/plot routines
def _find_col_by_str(cols, target_3tuple):
    if target_3tuple in cols:
        return target_3tuple
    t0, t1, t2 = target_3tuple
    for c in cols:
        if isinstance(c, tuple) and len(c) == 3:
            if (str(c[0]) == str(t0)) and (str(c[1]) == str(t1)) and (str(c[2]) == str(t2)):
                return c
    return None


# Function that rebuilds the flows circulating along grid lines and the related information
def build_net_lines_outputs(lv_grid_line_results_df, kerber_line_recipes):
    """
    From flow_grid_lines_only_df builds:
      - signed net flows per line
      - net flows for duration curves
      - duration curve
      - saturation %
      - saturation % duration curve
      - line metadata
    """
    if lv_grid_line_results_df is None or lv_grid_line_results_df.empty:
        return {}

    src_lines = convert_busseq_to_flowtuple_df(lv_grid_line_results_df)
    idx = src_lines.index

    net_lines_dict_signed = {}
    missing_forw = 0
    missing_back = 0

    for r in kerber_line_recipes:
        col_forw = (r["lab_forw"], r["b_key"], "flow")
        col_back = (r["lab_back"], r["a_key"], "flow")

        c_forw = _find_col_by_str(src_lines.columns, col_forw)
        if c_forw is not None:
            s_forw = pd.to_numeric(src_lines[c_forw], errors="coerce").fillna(0.0)
        else:
            missing_forw += 1
            s_forw = pd.Series(0.0, index=idx, dtype=float)

        c_back = _find_col_by_str(src_lines.columns, col_back)
        if c_back is not None:
            s_back = pd.to_numeric(src_lines[c_back], errors="coerce").fillna(0.0)
        else:
            missing_back += 1
            s_back = pd.Series(0.0, index=idx, dtype=float)

        net_lines_dict_signed[f"net_line_{r['a_s']}_to_{r['b_s']}"] = s_forw - s_back

    df_net_lines_ts_signed = pd.DataFrame(net_lines_dict_signed, index=idx)

    df_net_lines_ts_for_dur = (
        df_net_lines_ts_signed.abs()
        if USE_ABS_FOR_LINES_DURCURVE
        else df_net_lines_ts_signed
    )

    df_dur_net_lines = build_duration_curves(
        {c: df_net_lines_ts_for_dur[c] for c in df_net_lines_ts_for_dur.columns}
    )

    pmax_map = {
        f"net_line_{r['a_s']}_to_{r['b_s']}": r["pmax_kw"]
        for r in kerber_line_recipes
    }
    kind_map = {
        f"net_line_{r['a_s']}_to_{r['b_s']}": r["kind"]
        for r in kerber_line_recipes
    }

    s_pmax = pd.Series(pmax_map).reindex(df_net_lines_ts_signed.columns)
    s_kind = pd.Series(kind_map).reindex(df_net_lines_ts_signed.columns)

    df_sat_lines_ts_pct = (
        100.0
        * df_net_lines_ts_signed.abs()
        .divide(s_pmax, axis=1)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    ).clip(lower=0.0)

    df_dur_sat_lines = build_duration_curves(
        {c: df_sat_lines_ts_pct[c] for c in df_sat_lines_ts_pct.columns}
    )

    df_lines_meta = pd.DataFrame({
        "pmax_kw": s_pmax,
        "kind": s_kind,
    })

    return {
        "net_lines_ts_signed": df_net_lines_ts_signed,
        "net_lines_ts_for_dur": df_net_lines_ts_for_dur,
        "durcurve_net_lines": df_dur_net_lines,
        "lines_meta": df_lines_meta,
        "sat_lines_ts_pct": df_sat_lines_ts_pct,
        "durcurve_sat_lines_pct": df_dur_sat_lines,
        "missing_forw": missing_forw,
        "missing_back": missing_back,
    }


# Function to read grid files for chart creation
def load_net_lines_outputs_for_plot(season, base_dir=None):
    """
    Reads the net_lines_<season>.xlsx files produced in 4.11 and
    creates charts without recalculating everything.
    """
    if base_dir is None:
        base_dir = optimization_output_folder

    fp = os.path.join(base_dir, f"net_lines_{season}.xlsx")

    if not os.path.exists(fp):
        raise FileNotFoundError(f"net_lines file not found: {fp}")

    out = {}

    sheet_map = {
        "net_lines_ts_signed": "net_lines_ts_signed",
        "net_lines_ts_for_dur": "net_lines_ts_for_dur",
        "durcurve_net_lines": "durcurve_net_lines",
        "lines_meta": "lines_meta",
        "sat_lines_ts_pct": "sat_lines_ts_pct",
        "durcurve_sat_lines_pct": "durcurve_sat_lines_pct",
    }

    for key, sheet in sheet_map.items():
        out[key] = pd.read_excel(fp, sheet_name=sheet, index_col=0)

    return out
#---------------------------------------------------------------------

   ### 4.6.12 F(x) Plots for a single setup: mapping, flow aggregation and duration curves

In [ ]:
#-------------------------------------------------------------------
#       Sec 4.6 -- 12.1) CHART STYLE AND COLOR/NAME MAPS
#-------------------------------------------------------------------
# Function to assign a gradient of X colors in charts with X household classes, e.g. without flexibility, with BESS, with BESS+BEV, etc.
def make_gradient(cmap_name, n):
    """
    Returns n colors from a matplotlib colormap
    as rgba tuples, uniformly distributed.
    """
    cmap = cm.get_cmap(cmap_name)
    if n == 1:
        return [cmap(0.5)]
    return [cmap(i / (n - 1)) for i in range(n)]


# Function to assign a darker edge color to areas, e.g. generation-demand profile charts
def darken_color(color, amount=0.7):
    """
    Makes a color slightly darker.
    color: any format accepted by matplotlib, e.g. name, hex, rgba...
    amount < 1 -> darker, >1 -> lighter, in theory.
    """
    try:
        c = mcolors.to_rgb(color)
    except ValueError:
        # if matplotlib cannot interpret it, return the original color
        return color
    return (c[0]*amount, c[1]*amount, c[2]*amount)


# Function that defines the structure of the generation-demand chart, both RB and OPT
def init_gen_dmd_plot_maps(FLEX_CATEGORIES, make_gradient):
    """
    Initializes shared structures between OPT and RB charts:
    - area_columns / line_columns
    - generation_column_colors / demand_column_colors
    - column_names, if it already exists, do not overwrite it: update/merge
    Returns: area_columns, line_columns, generation_column_colors, demand_column_colors, column_names
    """

    # -----------------------
    # Sec 4.6 -- 6.3.1) Column lists
    # -----------------------
    area_columns = []
    # 1) Useful PV to the household electric bus, by category
    #    The aggregated column always remains:
    #    (pref_PV -> pref_bus_el)
    #    but upstream it may come from:
    #    - directly from c_i_PV -> c_i_bus_el
    #    - or from c_i_PV_to_house -> c_i_bus_el if curtailment is active
    for pref in FLEX_CATEGORIES:
        area_columns.append((f'{pref}_PV', f'{pref}_bus_el', 'flow'))
    # 2) LV import, by category
    for pref in FLEX_CATEGORIES:
        area_columns.append((f'from_the_grid_BT_to_{pref}', f'{pref}_bus_el', 'flow'))
    # 3) BESS discharge, by category
    for pref in FLEX_CATEGORIES:
        area_columns.append((f'{pref}_BESS', f'{pref}_bus_el', 'flow'))

    line_columns = []
    # aggregated demands
    line_columns.extend([
        ('ALL_bus_el', 'ALL_dmd_elc_base', 'flow'),
        ('ALL_bus_el', 'ALL_dmd_elc_Cool', 'flow'),
        ('ALL_bus_el', 'ALL_dmd_elc_Heat', 'flow'),
        ('ALL_bus_el', 'ALL_dmd_elc_Cook', 'flow'),
    ])
    # BEV charger, by category
    for pref in FLEX_CATEGORIES:
        line_columns.append((f'{pref}_bus_el', f'{pref}_charger', 'flow'))
    # DHW storage, by category
    for pref in FLEX_CATEGORIES:
        line_columns.append((f'{pref}_bus_el', f'{pref}_DHW_storage', 'flow'))
    # BESS charge, by category
    for pref in FLEX_CATEGORIES:
        line_columns.append((f'{pref}_bus_el', f'{pref}_BESS', 'flow'))

    # Note: DHW_boost is always added here so it is available for both OPT and RB
    col_boost_all = ('ALL_bus_el', 'ALL_DHW_boost', 'flow')
    if col_boost_all not in line_columns:
        line_columns.append(col_boost_all)

    # -----------------------
    # Sec 4.6 -- 6.3.2) Color dictionaries
    # -----------------------
    generation_column_colors = {}
    pv_colors = make_gradient('YlOrBr', len(FLEX_CATEGORIES))
    for pref, col in zip(FLEX_CATEGORIES, pv_colors):
        generation_column_colors[(f'{pref}_PV', f'{pref}_bus_el', 'flow')] = col

    imp_colors = make_gradient('Blues', len(FLEX_CATEGORIES))
    for pref, col in zip(FLEX_CATEGORIES, imp_colors):
        generation_column_colors[(f'from_the_grid_BT_to_{pref}', f'{pref}_bus_el', 'flow')] = col

    bess_dis_colors = make_gradient('Greens', len(FLEX_CATEGORIES))
    for pref, col in zip(FLEX_CATEGORIES, bess_dis_colors):
        generation_column_colors[(f'{pref}_BESS', f'{pref}_bus_el', 'flow')] = col

    demand_column_colors = {}
    # aggregated demand colors, one color per type
    demand_column_colors[('ALL_bus_el', 'ALL_dmd_elc_base', 'flow')] = '#bdbdbd'
    demand_column_colors[('ALL_bus_el', 'ALL_dmd_elc_Cool', 'flow')] = '#1f77b4'
    demand_column_colors[('ALL_bus_el', 'ALL_dmd_elc_Heat', 'flow')] = "#d515cf88"
    demand_column_colors[('ALL_bus_el', 'ALL_dmd_elc_Cook', 'flow')] = "#0ef7ffc7"
    demand_column_colors[('ALL_bus_el', 'ALL_DHW_boost', 'flow')]    = 'tab:olive'

    # BEV charger, by category
    charger_colors = make_gradient('Reds', len(FLEX_CATEGORIES))
    for pref, col in zip(FLEX_CATEGORIES, charger_colors):
        demand_column_colors[(f'{pref}_bus_el', f'{pref}_charger', 'flow')] = col

    # DHW storage, by category
    dhw_colors = make_gradient('Blues', len(FLEX_CATEGORIES))
    for pref, col in zip(FLEX_CATEGORIES, dhw_colors):
        demand_column_colors[(f'{pref}_bus_el', f'{pref}_DHW_storage', 'flow')] = col

    # BESS charge, by category
    bess_chg_colors = make_gradient('Greens', len(FLEX_CATEGORIES))
    for pref, col in zip(FLEX_CATEGORIES, bess_chg_colors):
        demand_column_colors[(f'{pref}_bus_el', f'{pref}_BESS', 'flow')] = col

    # -----------------------
    # Sec 4.6 -- 6.3.3) Legend names, merge-safe
    # -----------------------
    try:
        column_names = dict(globals().get('column_names', {}) or {})
    except Exception:
        column_names = {}

    # Aggregated demand names
    column_names.update({
        ('ALL_bus_el', 'ALL_dmd_elc_base', 'flow'): 'El base demand',
        ('ALL_bus_el', 'ALL_dmd_elc_Cool', 'flow'): 'El cooling demand',
        ('ALL_bus_el', 'ALL_dmd_elc_Heat', 'flow'): 'El heating demand',
        ('ALL_bus_el', 'ALL_dmd_elc_Cook', 'flow'): 'El cooking demand',
        ('ALL_bus_el', 'ALL_DHW_boost', 'flow'): 'El to DHW boost',
    })

    # Technology flow names by flexibility category
    flex_labels = globals().get('flex_category_labels', {}) or {}

    for pref in FLEX_CATEGORIES:
        descr = flex_labels.get(pref, pref)

        column_names.update({
            (f'{pref}_PV', f'{pref}_bus_el', 'flow'):
                f'PV generation ({descr})',

            (f'from_the_grid_BT_to_{pref}', f'{pref}_bus_el', 'flow'):
                f'El import from LV grid ({descr})',

            (f'{pref}_BESS', f'{pref}_bus_el', 'flow'):
                f'El from BESS ({descr})',

            (f'{pref}_bus_el', f'{pref}_charger', 'flow'):
                f'El to charge BEV ({descr})',

            (f'{pref}_bus_el', f'{pref}_DHW_storage', 'flow'):
                f'El to charge ESWH ({descr})',

            (f'{pref}_bus_el', f'{pref}_BESS', 'flow'):
                f'El to charge BESS ({descr})',
        })

    return (
        area_columns,
        line_columns,
        generation_column_colors,
        demand_column_colors,
        column_names,
    )
# ---------------------------------------




#-------------------------------------------------------------------
#       Sec 4.6 -- 12.1) STRUCTURAL FILTERS FOR COLUMNS AND TECHNOLOGIES
#-------------------------------------------------------------------
# Function to identify the flexibility class to which a single house/household belongs
def _extract_pref_from_node(node, flex_categories):
    """Returns 'case_BV' etc. if node starts with 'case_BV_'."""
    s = str(node)
    for pref in flex_categories:
        if s.startswith(pref + "_"):
            return pref
    return None


# Function to identify which storage technology is assigned to the house
def _category_has_tech(pref, tech_letter):
    """
    pref: 'case_BVH' etc.
    tech_letter: 'B' (BESS), 'V' (BEV), 'H' (DHW storage)
    """
    if pref is None or pref == "case_0":
        return False
    suffix = pref.replace("case_", "")  # e.g. 'BVH'
    return tech_letter in suffix


# Function to identify which flows should be searched for and associated with flexibility classes, i.e. storage presence
def _flow_required_tech(col):
    """
    col is a 3-tuple such as (from, to, 'flow').
    If the column refers to BESS/BEV/DHW, returns 'B'/'V'/'H'; otherwise None.
    """
    if not isinstance(col, tuple) or len(col) < 2:
        return None

    a = str(col[0])
    b = str(col[1])
    s = a + " | " + b

    # BESS
    if "_BESS" in s:
        return "B"

    # BEV, charger / V2G
    if "_charger" in s or "V2G" in s or "_BEV" in s:
        return "V"

    # DHW storage / boost
    if "_DHW_storage" in s or "_DHW_boost" in s or "_bus_DHW_use" in s:
        return "H"

    return None


# Function to keep only the columns that are structurally possible in the charts, e.g. a B+V house cannot have electric flow for DHW charging
def filter_cols_structural(cols, flex_categories):
    """
    Removes columns that are structurally impossible for the category.
    It does not use numerical values, only category-technology consistency.
    """
    out = []
    for col in cols:
        # if it is an ALL_* column, always keep it
        from_node = col[0] if isinstance(col, tuple) and len(col) > 0 else ""
        to_node   = col[1] if isinstance(col, tuple) and len(col) > 1 else ""

        pref = _extract_pref_from_node(from_node, flex_categories) or _extract_pref_from_node(to_node, flex_categories)
        if pref is None:
            out.append(col)
            continue

        tech = _flow_required_tech(col)
        if tech is None:
            out.append(col)
            continue

        if _category_has_tech(pref, tech):
            out.append(col)
        # else: discard, impossible, e.g. BESS in case_V

    return out
# -----------------------------------------------





#-------------------------------------------------------------------
#       Sec 4.6 -- 12.3) HOUSE CLASSIFICATION AND FLOW AGGREGATION
#-------------------------------------------------------------------
# Function that returns the flexibility-category keys to which households can be assigned
def classify_flex_case(
    c: str,
    has_BESS: set,
    has_BEV: set,
    has_el_boiler: set,
) -> str:
    """
    Returns one of the keys in FLEX_CATEGORIES based on the flexibility options
    available for household 'c'.

    Uses the sets passed as arguments, so it works with both:
    - global has sets, used in 4.10 to build potential RB profiles;
    - has_*_run sets, used in 4.11c for mixed OPT/RB/NO configurations.
    """
    hasB = c in set(has_BESS or set())
    hasV = c in set(has_BEV or set())
    hasH = c in set(has_el_boiler or set())

    mapping = {
        (False, False, False): "case_0",
        (True,  False, False): "case_B",
        (False, True,  False): "case_V",
        (False, False, True ): "case_H",
        (True,  True,  False): "case_BV",
        (True,  False, True ): "case_BH",
        (False, True,  True ): "case_VH",
        (True,  True,  True ): "case_BVH",
    }

    return mapping[(hasB, hasV, hasH)]



# Function to sum the relevant columns on the house side
def house_flows_sum(df, case_list, col_origin_keyword, col_dest_keyword, col_sum_name, df_target):    # FM_explain: respectively, dataframe to search, list of cases to consider, flow origin and destination, name of the cumulative "exchange" to build
    df_target[(col_sum_name, 'flow')] = 0                                                               # Initialize the aggregated column
    for case in case_list:
        if col_origin_keyword == 'from_the_grid_BT_to':
            col_origin = f'{col_origin_keyword}_{case}'
        else:
            col_origin = f'{case}_{col_origin_keyword}'
        
        if col_dest_keyword == 'from_the_grid_BT_to':
            col_dest = f'{col_dest_keyword}_{case}'
        else:
            col_dest = f'{case}_{col_dest_keyword}'
        relevant_columns = [
            col for col in df.columns 
            if col[0] == col_origin and col[1] == col_dest # and col[2] == 'flow'                       # FM_explain: the column name is a tuple of two elements: first element is the flow origin-destination sub-tuple, second element is the string "flow"
        ]
        if relevant_columns:
            df_target[(col_sum_name, 'flow')] += df[relevant_columns].sum(axis=1)
        # otherwise: no column, but this is normal for houses without that technology



# Specific function to sum house-side PV flows,
# handling both the case WITHOUT curtailment and the case WITH curtailment.
def house_flows_from_pv_sum(df, case_list, col_sum_name, df_target):
    """
    Aggregates useful PV flows to the households' electric bus.

    Handles both cases:
    1) without PV curtailment:
        c_i_PV -> c_i_bus_el

    2) with PV curtailment:
        c_i_PV_to_house -> c_i_bus_el

    The result is always saved in the same aggregated column:
        (prefix_PV, prefix_bus_el, 'flow')
    so the charts remain compatible.
    """
    df_target[(col_sum_name, 'flow')] = 0.0

    for case in case_list:
        relevant_columns = []

        # Legacy case / without curtailment
        col_origin_direct = f'{case}_PV'
        col_dest_direct = f'{case}_bus_el'

        # Case with curtailment: PV passes through the PV_to_house converter
        col_origin_curt = f'{case}_PV_to_house'
        col_dest_curt = f'{case}_bus_el'

        for col in df.columns:
            if (
                (col[0] == col_origin_direct and col[1] == col_dest_direct)
                or
                (col[0] == col_origin_curt and col[1] == col_dest_curt)
            ):
                relevant_columns.append(col)

        if relevant_columns:
            df_target[(col_sum_name, 'flow')] += df[relevant_columns].sum(axis=1)


# Function that sums the contributions exchanged with the LV grid, along the feeder, across the different flexibility household categories, see Sec. 4.12, part 2.3.2 for charts
def _sum_cols_safe(df: pd.DataFrame, cols):
    """
    Robustly sums a set of columns, even if missing / non-numeric.
    Returns a Series indexed as df.index.
    """
    if df is None or len(df) == 0:
        return pd.Series(dtype=float)

    cols_ok = [c for c in cols if c in df.columns]
    if not cols_ok:
        return pd.Series(0.0, index=df.index, dtype=float)

    sub = df.loc[:, cols_ok]

    # sub may be a DataFrame: to_numeric does NOT accept 2D -> apply column by column
    sub_num = sub.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return sub_num.sum(axis=1)
#--------------------------------------------------------------------------------------    





#-------------------------------------------------------------------
#       Sec 4.6 -- 12.4) LEGEND LAYOUT AND DURATION-CURVE PLOTS
#-------------------------------------------------------------------
# Function to split very long legends into multiple columns
def auto_ncol_legend(labels, fig, max_cols=4, min_cols=1, char_px=7):
    """
    Automatically chooses ncol for the legend based on
    figure width and average string length.

    labels : list of strings, legend labels
    fig    : Figure object
    max_cols : maximum number of columns
    min_cols : minimum number of columns
    char_px  : average character width in pixels, heuristic
    """
    if not labels:
        return 1

    # Ensure that the figure has a renderer
    fig.canvas.draw()

    fig_width_px = fig.get_size_inches()[0] * fig.dpi
    avg_len = np.mean([len(s) for s in labels])

    if avg_len <= 0:
        # fallback
        return max(min_cols, min(len(labels), max_cols))

    approx_label_width_px = avg_len * char_px
    max_labels_per_row = max(1, int(fig_width_px // approx_label_width_px))

    ncol = min(len(labels), max_labels_per_row, max_cols)
    ncol = max(min_cols, ncol)

    return ncol


# Function to create duration-curve charts for ALL LV grid-house c_i flows
def plot_duration_curves(
        duration_df, 
        dt_hours, 
        season, 
        out_png,
        timestep_min=10, 
        title_prefix="Duration curve - Import from LV \n (Time-resolution: 10-minute)",
        ylim=None,
        ):
    
    """
    Plots all houses: x = cumulative hours, y = sorted kW.
    """
    if duration_df.empty:
        print(f"[WARN] No LV import duration curve for {season_english_dict[season]}.")
        return

    # If I want an axis with the legend in hours, while still using 10-min series
    # x_hours = np.arange(len(df_duration)) * dt_hours
    # Otherwise, axis with 10' detail
    x = duration_df.index.values # * (timestep_min/60.0)  # 1008 points

    fig, ax = plt.subplots(figsize=(13, 6), dpi=150)

    # many houses -> thin and transparent lines
    for c in duration_df.columns:
        # ax.plot(x_hours, df_duration[c].values, linewidth=0.8, alpha=0.35)        # X-axis shows labels in hours
        ax.plot(x, duration_df[c].values, linewidth=0.8, alpha=0.35)                # X-axis shows 10-min labels, 1008 weekly timesteps
    ax.set_xlabel("Cumulative duration (10-min resolution)")
    ax.set_ylabel("Import power from LV [kW]")
    ax.set_title(f"{title_prefix} ({season_english_dict[season]})")
    ax.grid(True, linestyle=":", alpha=0.35)
    if ylim is not None:
        ax.set_ylim(ylim[0], ylim[1])

    fig.tight_layout()
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close(fig)



# Function to print/plot duration-curve charts with percentile bands       
def plot_duration_curve_percentile_bands(
    duration_df: pd.DataFrame,
    dt_hours: float,
    season: str,
    out_png: str,
    x_mode: str = "rank",   # "hours" or "rank"
    qs=(5,10,20,30,40,50,60,70,80,90,95),
    bands=((5,95), (10,90), (20,80), (30,70), (40,60)),
    cmap_name="Greens",
    y_label="Import power [kW]",
    x_label="Duration [10-min resolution]",
    ylim=None,
):

    """
    Plots percentile bands + median for LV import duration curves.
    df_pct: output of duration_curve_percentiles, index=rank, columns=p10..p90
    """
    if duration_df is None or duration_df.empty:
        print(f"[WARN] empty df_duration: skipping plot {season}")
        return

    
    # Calculate percentiles for each rank
    arr = duration_df.to_numpy(dtype=float)
    arr = np.nan_to_num(arr, nan=0.0)

    pct = np.percentile(arr, qs, axis=1)  # shape (len(qs), N)
    pct_df = pd.DataFrame(pct.T, index=duration_df.index, columns=[f"p{q}" for q in qs])

    rank = duration_df.index.to_numpy()

    if x_mode == "hours":
        x = rank * dt_hours
        xlabel = "Duration [h]"
    else:
        x = rank
        xlabel = "Duration rank (10-min timesteps)"

    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)

    
    # band colors from cmap, light -> dark
    cmap = plt.get_cmap(cmap_name)
    colors_from_cmap = [cmap(v) for v in np.linspace(0.15, 0.75, len(bands))]


    # percentile bands
    alpha_bases = [ 0.28, 0.38, 0.52, 0.65, 0.80]  # similar to SOC: inner bands are darker
    for (lo, hi), alpha_base, col in zip(bands, alpha_bases, colors_from_cmap):
        lo_col = f"p{lo}"
        hi_col = f"p{hi}"
        if lo_col not in pct_df.columns or hi_col not in pct_df.columns:
            continue
            
        ax.fill_between(
            x,
            pct_df[lo_col].to_numpy(dtype=float),
            pct_df[hi_col].to_numpy(dtype=float),
            alpha=float(alpha_base),
            color=col,
            label=f"{lo}-{hi}%"
        )

    # median
    ax.plot(
        x,
        pct_df["p50"].to_numpy(dtype=float),
        linewidth=1.6,
        color="k",
        label="Median (p50)"
    )
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(f"Duration curve percentiles – LV import – {season_english_dict[season]}")
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.legend(loc="best", fontsize=9)

    # Y-axis limits, if required, used to standardize across seasons
    if ylim is not None:
        ax.set_ylim(ylim[0], ylim[1])
    else:
        y_max = np.nanmax(duration_df.values)
        if np.isfinite(y_max) and y_max > 0:
            ax.set_ylim(0, y_max * 1.05)


    ax.legend(loc="best", fontsize=9, framealpha=0.9)
    fig.tight_layout()
    fig.savefig(out_png, dpi=200, bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)

    return pct_df 


# Function to print/plot the "net" duration curve, which therefore also includes negative values = exports to the LV grid
def plot_duration_curve_percentile_bands_with_overlays(
    duration_df: pd.DataFrame,
    percentiles=(5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95),
    title="Load duration curve (percentiles)",
    xlabel="Sorted timesteps",
    ylabel="Power [kW]",
    overlays=None,            # dict {label: Series(kW)}
    overlays_style=None,      # dict {label: dict kwargs plt.plot}
    savepath=None,
    cmap_name="Blues",        # <-- colormap for the bands
    cmap_span=(0.18, 0.78),   # <-- cmap range used for the bands
    ylim=None,
):
    # percentiles in [0,100] -> quantiles in [0,1]
    qs = [p / 100.0 for p in percentiles]

    # quantiles for each rank, row, across houses, columns
    df_q = duration_df.quantile(qs, axis=1)
    df_pct = df_q.T.copy()

    # Rename columns from float quantiles to integer percentiles, for easier access
    df_pct.columns = list(percentiles)

    plt.figure(figsize=(9, 5))
    x = df_pct.index

    # Bands
    bands = [(10, 90), (20, 80), (30, 70), (40, 60)]
    alpha_bases = [0.22, 0.30, 0.42, 0.60]

    cmap = plt.get_cmap(cmap_name)
    v0, v1 = cmap_span
    colors_from_cmap = [cmap(v) for v in np.linspace(v0, v1, len(bands))]

    for (lo, hi), a, col in zip(bands, alpha_bases, colors_from_cmap):
        if (lo in df_pct.columns) and (hi in df_pct.columns):
            plt.fill_between(
                x, df_pct[lo], df_pct[hi],
                alpha=a,
                color=col,
                label=f"{lo}-{hi}%",
                zorder=1                                                                # lower zorder to keep the median highlighted on the Z axis
            )
            
    # Median
    if 50 in df_pct.columns:
        plt.plot(x, df_pct[50], linewidth=2.2, color="k", label="Median (p50)", zorder=10 )        # zorder puts the median "higher" on the Z axis
    else:
        # fallback: central percentile
        mid = percentiles[len(percentiles)//2]
        plt.plot(x, df_pct[mid], color="k", linewidth=2.2, label=f"p{mid}")

    # Overlay, e.g. average household on the MV side
    if overlays:
        for lab, s in overlays.items():
            dur = duration_curve_from_series(s)
            xx = np.arange(len(dur))
            style = {}
            if overlays_style and lab in overlays_style:
                style.update(overlays_style[lab])
            plt.plot(xx, dur.values, label=lab, **style)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()


    # before save/show:
    if ylim is not None:
        plt.ylim(ylim[0], ylim[1])


    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")

    plt.show()

    return df_pct

   ### 4.6.13 F(x) Multi-run dataset/cache/audit

In [ ]:
# --------------------------------------------------------------------
# Sec 4.6 -- 14.3) FUNCTIONS FOR NEW PATHS AND MIXED-SCENARIO READING
# --------------------------------------------------------------------
#  MULTI-RUN CHART:
# household cost percentiles + total system cost on the secondary axis

# Function to identify summary files containing economic information for the configurations to compare
def _scenario_summary_file(scenario):
    """Return the saved system-cost summary of one manifest scenario."""

    path = scenario.get("summary_file")

    if path:
        return path

    return os.path.join(
        scenario["xls_dir"],
        f"system_cost_summary_{scenario['summary_mode']}_"
        f"{num_households}_nuc_{selected_region}.xlsx",
    )


# Function to retrieve costs from the economic files of the configurations to compare
def _read_total_system_cost_from_summary(
    summary_file,
    season,
    summary_mode,
    preferred_total_cost_col="total_system_cost_effective_eur",
):
    """
    Reads the total system cost from the system-cost summary
    of one specific saved run.

    preferred_total_cost_col:
    - "total_system_cost_effective_eur":
    effective economic cost used for reporting.
    It includes LV energy settlement and real additional costs
    such as BEV outdoor charging, while the artificial DHW-boost
    penalty is included only if explicitly requested.
    - "total_system_cost_including_objective_variable_terms_eur":
        formula FM + Kerber line cost + other variable costs.
    - "total_system_cost_business_eur":
        FM business formula only.
    """

    if not os.path.exists(summary_file):
        raise FileNotFoundError(f"Summary file not found: {summary_file}")

    xls = pd.ExcelFile(summary_file)
    if "all_modes_by_season" in xls.sheet_names:
        df = pd.read_excel(summary_file, sheet_name="all_modes_by_season")
    elif "by_season" in xls.sheet_names:
        df = pd.read_excel(summary_file, sheet_name="by_season")
    else:
        df = pd.read_excel(summary_file, sheet_name=xls.sheet_names[0])

    d = df.copy()

    if "season" not in d.columns:
        raise KeyError(f"Column 'season' not found in {summary_file}")

    if "mode" not in d.columns:
        raise KeyError(f"Column 'mode' not found in {summary_file}")

    d = d[
        (d["season"].astype(str) == str(season))
        & (d["mode"].astype(str) == str(summary_mode))
    ].copy()

    if d.empty:
        raise ValueError(
            f"No row found in {summary_file} for season={season}, mode={summary_mode}"
        )

    # Column priority:
    # 1) new complete column
    # 2) old objective_total_eur
    # 3) variable costs
    # 4) business formula
    candidate_cols = [
        preferred_total_cost_col,

        # New economic-reporting metric
        "total_system_cost_effective_eur",

        # Legacy / simulation metrics
        "total_system_cost_including_all_objective_terms_eur",
        "objective_total_eur",
        "total_system_cost_including_objective_variable_terms_eur",
        "objective_variable_cost_terms_eur",
        "total_system_cost_business_eur",
    ]

    for col in candidate_cols:
        if col in d.columns:
            return float(pd.to_numeric(d[col], errors="coerce").iloc[0]), col

    # fallback: rebuild the business formula if the file is old
    business_cols = [
        "bt_purchase_cost_eur",
        "bt_sale_revenues_eur",
        "dhw_boost_cost_eur",
        "bev_outdoor_charge_cost_eur",
        "pv_curtail_cost_eur",
    ]
    missing = [c for c in business_cols if c not in d.columns]
    if missing:
        raise KeyError(
            f"No total cost column found and impossible to rebuild the business formula. "
            f"Missing columns: {missing}"
        )

    value = float(d[business_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1).iloc[0])
    return value, "business_formula_rebuilt"



# Function to search for hybrid-configuration results
def _scenario_results_file(scenario, season=None):
    """Return the seasonal result workbook of one saved run."""

    season = season or scenario.get("season")

    if season is None:
        raise ValueError("season is required")

    direct = scenario.get("results_file")

    if direct and os.path.isfile(direct):
        return direct

    run_dir = scenario["xls_dir"]

    candidates = [
        os.path.join(run_dir, f"ris_{season}.xlsx"),
        os.path.join(
            run_dir,
            f"ris_{scenario['summary_mode']}_"
            f"{num_households}_nuc_{selected_region}_{season}.xlsx",
        ),
    ]

    for path in candidates:
        if os.path.isfile(path):
            return path

    matches = glob.glob(
        os.path.join(run_dir, f"*ris*{season}*.xlsx")
    )

    if matches:
        return sorted(matches)[0]

    raise FileNotFoundError(
        f"No results workbook found for "
        f"{scenario['run_uid']}, season={season}."
    )


# Function to read costs from the files of a specific configuration
def _read_house_costs_from_mode_results(
    results_file,
    mode,
):
    """
    Reads household costs directly from the results file of a MODE.

    Tries different sheet/column names for robustness.
    """
    if not os.path.exists(results_file):
        raise FileNotFoundError(f"Results file not found: {results_file}")

    xls = pd.ExcelFile(results_file)

    sheet_candidates = [
        'total_costs_per_household',
        "costs_per_nucleo",
        "household_costs",
        "economics_sum_week",
    ]

    sheet = next((s for s in sheet_candidates if s in xls.sheet_names), None)

    if sheet is None:
        raise KeyError(
            f"No household-cost sheet found in {results_file}. "
            f"Available sheets: {xls.sheet_names}"
        )

    df = pd.read_excel(results_file, sheet_name=sheet)

    col_candidates = [
        "effective_total_eur",
        "total_eur",

        # Legacy fallbacks
        "total_costs_eur",
        "total_cost_eur",
        "objective_total_eur",
        "opt_total_eur",
        "sim_total_eur",
        "cost_total_eur",
        "econ_€",
    ]

    cost_col = next((c for c in col_candidates if c in df.columns), None)

    if cost_col is None:
        # fallback: first meaningful numeric column
        numeric_cols = [
            c for c in df.columns
            if pd.api.types.is_numeric_dtype(df[c])
        ]

        if not numeric_cols:
            raise KeyError(
                f"No numeric cost column found in {results_file}, sheet={sheet}. "
                f"Available columns: {df.columns.tolist()}"
            )

        cost_col = numeric_cols[-1]

    cost_values = pd.to_numeric(
        df[cost_col],
        errors="coerce",
    )

    house_col_candidates = [
        "house_id",
        "household_id",
        "household",
        "house",
        "case",
        "nucleo",
    ]

    house_col = next(
        (
            column
            for column in house_col_candidates
            if column in df.columns
        ),
        None,
    )

    if house_col is not None:
        house_ids = (
            df[house_col]
            .where(df[house_col].notna(), "")
            .astype(str)
            .str.strip()
        )
    else:
        # Fallback for old Excel files where household IDs were saved as index.
        unnamed_cols = [
            column
            for column in df.columns
            if str(column).lower().startswith("unnamed")
        ]

        if unnamed_cols:
            house_ids = (
                df[unnamed_cols[0]]
                .where(df[unnamed_cols[0]].notna(), "")
                .astype(str)
                .str.strip()
            )
        else:
            house_ids = pd.Series(
                [
                    f"row_{i + 1}"
                    for i in range(len(df))
                ],
                index=df.index,
            )

    valid_mask = (
        cost_values.notna()
        & house_ids.ne("")
    )

    s = pd.Series(
        data=cost_values.loc[valid_mask].to_numpy(dtype=float),
        index=house_ids.loc[valid_mask].to_numpy(),
        name=str(mode),
    )

    # Avoid accidental duplicated household IDs.
    if s.index.has_duplicates:
        duplicated_ids = sorted(
            set(
                s.index[
                    s.index.duplicated(keep=False)
                ]
            )
        )

        raise ValueError(
            f"Duplicated household IDs in {results_file}, "
            f"sheet={sheet}: {duplicated_ids}"
        )

    if s.empty:
        raise ValueError(
            f"No valid household cost in {results_file}, "
            f"sheet={sheet}, col={cost_col}"
        )

    return s, sheet, cost_col





# Function to read all required information for system-configuration comparison
def _read_multirun_energy_breakdown_from_summary(
    summary_file,
    season,
    summary_mode,
    allow_missing=False,
):
    """
    Reads from system_cost_summary the energy quantities required
    by the multi-run chart.

    This function does NOT search in pickle files.
    It is only used to read the summary if it has already been updated.
    """

    if not os.path.exists(summary_file):
        raise FileNotFoundError(f"Summary file not found: {summary_file}")

    xls = pd.ExcelFile(summary_file)

    if "all_modes_by_season" in xls.sheet_names:
        df = pd.read_excel(summary_file, sheet_name="all_modes_by_season")
    elif "by_season" in xls.sheet_names:
        df = pd.read_excel(summary_file, sheet_name="by_season")
    else:
        df = pd.read_excel(summary_file, sheet_name=xls.sheet_names[0])

    d = df[
        (df["season"].astype(str) == str(season))
        & (df["mode"].astype(str) == str(summary_mode))
    ].copy()

    if d.empty:
        raise ValueError(
            f"No row found in {summary_file} "
            f"for season={season}, mode={summary_mode}"
        )

    row = d.iloc[0]

    required_cols = [
        "total_local_demand_kwh",
        "mt_import_total_kwh",
        "local_self_consumption_kwh",
        "demand_base_kwh",
        "demand_cooling_kwh",
        "demand_heating_kwh",
        "demand_cooking_kwh",
        "demand_ev_charging_kwh",
        "demand_dhw_total_kwh",
    ]

    missing = [c for c in required_cols if c not in d.columns]

    if missing and not allow_missing:
        raise KeyError(
            f"Energy columns missing from the summary: {missing}. "
            f"File: {summary_file}"
        )

    def _num(col, default=np.nan):
        if col not in d.columns:
            return default
        val = pd.to_numeric(row[col], errors="coerce")
        return float(val) if pd.notna(val) else default

    mv_import_total_kwh = _num("mv_import_total_kwh", _num("mt_import_total_kwh"))
    mv_export_total_kwh = _num("mv_export_total_kwh", _num("mt_export_total_kwh", 0.0))

    net_mv_import_kwh_raw = _num(
        "net_mv_import_kwh_raw",
        mv_import_total_kwh - mv_export_total_kwh
        if pd.notna(mv_import_total_kwh) and pd.notna(mv_export_total_kwh)
        else np.nan
    )

    total_local_demand_kwh = _num("total_local_demand_kwh")

    gross_mv_import_share_of_demand = (
        mv_import_total_kwh / total_local_demand_kwh
        if pd.notna(mv_import_total_kwh)
        and pd.notna(total_local_demand_kwh)
        and total_local_demand_kwh > 1e-12
        else np.nan
    )

    gross_mv_export_share_of_demand = (
        mv_export_total_kwh / total_local_demand_kwh
        if pd.notna(mv_export_total_kwh)
        and pd.notna(total_local_demand_kwh)
        and total_local_demand_kwh > 1e-12
        else np.nan
    )

    net_mv_import_share_of_demand = (
        net_mv_import_kwh_raw / total_local_demand_kwh
        if pd.notna(net_mv_import_kwh_raw)
        and pd.notna(total_local_demand_kwh)
        and total_local_demand_kwh > 1e-12
        else np.nan
    )

    net_mv_import_for_demand_kwh = _num(
        "net_mv_import_for_demand_kwh",
        min(max(net_mv_import_kwh_raw, 0.0), total_local_demand_kwh)
        if pd.notna(net_mv_import_kwh_raw) and pd.notna(total_local_demand_kwh)
        else np.nan
    )

    internal_self_sufficiency_kwh = _num(
        "internal_self_sufficiency_kwh",
        max(0.0, total_local_demand_kwh - net_mv_import_for_demand_kwh)
        if pd.notna(total_local_demand_kwh) and pd.notna(net_mv_import_for_demand_kwh)
        else np.nan
    )

    out = {
        "total_local_demand_kwh": total_local_demand_kwh,
        "total_el_demand_kwh": total_local_demand_kwh,

        "mv_import_total_kwh": mv_import_total_kwh,
        "mv_export_total_kwh": mv_export_total_kwh,
        "net_mv_import_kwh_raw": net_mv_import_kwh_raw,
        "net_mv_import_for_demand_kwh": net_mv_import_for_demand_kwh,

        "gross_mv_import_share_of_demand": gross_mv_import_share_of_demand,
        "gross_mv_export_share_of_demand": gross_mv_export_share_of_demand,
        "net_mv_import_share_of_demand": net_mv_import_share_of_demand,

        # Backward compatibility
        "mt_import_total_kwh": net_mv_import_for_demand_kwh,
        "mt_import_total_kwh_raw": mv_import_total_kwh,
        "mt_export_total_kwh": mv_export_total_kwh,

        "internal_self_sufficiency_kwh": internal_self_sufficiency_kwh,
        "local_self_consumption_kwh": internal_self_sufficiency_kwh,

        "demand_base_kwh": _num("demand_base_kwh"),
        "demand_cooling_kwh": _num("demand_cooling_kwh"),
        "demand_heating_kwh": _num("demand_heating_kwh"),
        "demand_cooking_kwh": _num("demand_cooking_kwh"),
        "demand_ev_charging_kwh": _num("demand_ev_charging_kwh"),
        "demand_dhw_storage_kwh": _num("demand_dhw_storage_kwh"),
        "demand_dhw_boost_kwh": _num("demand_dhw_boost_kwh"),
        "demand_dhw_total_kwh": _num("demand_dhw_total_kwh"),
        "demand_components_total_kwh": _num("demand_components_total_kwh"),
        "demand_components_gap_kwh": _num("demand_components_gap_kwh"),

        "energy_breakdown_source": "summary",
        "flow_el_pickle_used": "",
        "flow_grid_root_pickle_used": "",
        "net_mv_import_exceeds_demand": (
            bool(net_mv_import_kwh_raw > total_local_demand_kwh + 1e-6)
            if pd.notna(net_mv_import_kwh_raw) and pd.notna(total_local_demand_kwh)
            else False
        ),
    }

    return out
#-----------------------------------------------------------------




# --------------------------------------------------------------------
# Sec 4.6 -- 14.4) PICKLE FALLBACK TO REBUILD MULTI-RUN ENERGY METRICS
# --------------------------------------------------------------------

# Function to specify where to search for pickle files for configuration comparison
def _scenario_pickle_search_dirs(scenario):
    """Return exact pickle directories recorded in the manifest."""

    return [
        scenario["py_dir"],
        scenario["pflow_py_dir"],
    ]



# Function to identify the dataframe pickle files from which comparison information is extracted
def _find_pickle_file_for_metric(
    search_dirs,
    metric_name,
    season,
    summary_mode,
    required=True,
):
    """
    Strictly searches for the correct pickle for:
      - metrica
      - mode: OPT / RB_FIXED
      - season

    Silent version:
    - raises an error only if required=True and the pickle is missing
    """

    mode = str(summary_mode).upper()
    season = str(season)

    for d in search_dirs:
        if not os.path.exists(d):
            continue

        try:
            p_short = flow_pickle_path(
                metric_name,
                season,
                base_dir=d,
            )
            if os.path.exists(p_short):
                return p_short
        except Exception:
            pass

    

    if mode == "OPT":
        aliases = ["OPT"]
    elif mode in {"RB_FIXED", "RB", "RULE_BASED"}:
        aliases = ["RB_FIXED"]
    else:
        aliases = [str(summary_mode)]

    extensions = [".pkl.gz", ".pickle.gz", ".pkl", ".pickle"]

    all_candidates = []

    for d in search_dirs:
        if not os.path.exists(d):
            continue

        for ext in extensions:
            pattern = os.path.join(d, f"*{metric_name}*{ext}")
            all_candidates.extend(glob.glob(pattern))

    valid = []

    for p in sorted(set(all_candidates)):
        bn = os.path.basename(p)

        has_metric = metric_name in bn
        has_season = season in bn
        has_mode = any(a in bn for a in aliases)

        if has_metric and has_season and has_mode:
            valid.append(p)

    if len(valid) > 0:
        valid = sorted(valid, key=os.path.getmtime, reverse=True)
        return valid[0]

    if required:
        raise FileNotFoundError(
            f"No pickle found per metric_name={metric_name}, "
            f"mode={summary_mode}, season={season}. "
            f"Searched folders: {search_dirs}"
        )

    return None


# Function to load pickle files
def _load_pickle_any(path):
    """
    Loads compressed or uncompressed pickle files.
    Uses charge_dataframe if available; otherwise manual fallback.
    """

    if path is None:
        return None

    if str(path).endswith(".gz"):
        with gzip.open(path, "rb") as f:
            return pickle.load(f)

    with open(path, "rb") as f:
        return pickle.load(f)


# Function to normalize the flow structure contained in the pickle files used for configuration comparison
def _as_flow_tuple_columns(df):
    """
    Tries to normalize flow dataframe columns into the format:
      (from_label, to_label, "flow")

    Uses convert_busseq_to_flowtuple_df if it is already defined in the notebook.
    """

    if df is None or not hasattr(df, "columns"):
        return df

    try:
        return convert_busseq_to_flowtuple_df(df)
    except Exception:
        return df


# Function to extract flow labels from pickle-file columns
def _flow_col_labels(col):
    """
    Returns from_label and to_label from a flow dataframe column.

    Supports:
    - already-clean tuples: ("from", "to", "flow")
    - tuple legacy: (("from", "to"), "flow")
    - MultiIndex Excel-like/pickle: ("c_1", "c_1_bus_el -> c_1_charger")
    - strings: "from -> to"

    The important point is to FIRST search for a string containing '->'
    inside the tuple, because flow_el_df often has columns such as:
        ("c_1", "c_1_bus_el -> c_1_dmd_elc_base")
    """

    # Tuple case
    if isinstance(col, tuple):

        # Already-correct case: (from, to, flow)
        if len(col) >= 3 and str(col[2]).lower() == "flow":
            return str(col[0]), str(col[1])

        # Legacy case: ((from, to), flow)
        if (
            len(col) >= 2
            and isinstance(col[0], tuple)
            and len(col[0]) >= 2
            and str(col[1]).lower() == "flow"
        ):
            return str(col[0][0]), str(col[0][1])

        # MultiIndex case: search for the level that contains "A -> B"
        for x in reversed(col):
            sx = str(x)
            if "->" in sx:
                a, b = sx.split("->", 1)
                return a.strip(), b.strip()

        # Non-parseable tuple fallback
        if len(col) >= 2:
            return str(col[0]), str(col[1])

    # String case
    s = str(col)
    if "->" in s:
        a, b = s.split("->", 1)
        return a.strip(), b.strip()

    return "", s


# Function to sum the flows of interest, e.g. cooling + heating + cooking, in pickle files for comparison
def _sum_flow_df_kwh_by_match(
    df,
    match_fn,
    timestep_minutes=10,
    values_are_power_kw=True,
):
    """
    Sums all columns of a flow dataframe that satisfy match_fn(frm, to).

    If values_are_power_kw=True:
      values are average kW over the timestep -> multiply by dt_h

    If values_are_power_kw=False:
      values are already in kWh/timestep -> do not multiply
    """

    if df is None or not hasattr(df, "columns") or df.empty:
        return 0.0, []

    d = _as_flow_tuple_columns(df).copy()

    dt_h = float(timestep_minutes) / 60.0 if values_are_power_kw else 1.0

    total = 0.0
    matched_cols = []

    for col in d.columns:
        frm, to = _flow_col_labels(col)

        if not match_fn(frm, to):
            continue

        vals = pd.to_numeric(d[col], errors="coerce").fillna(0.0)
        total += float(vals.sum() * dt_h)
        matched_cols.append(col)

    return total, matched_cols


# Function to quantify the PV share actually injected into the local household bus, net of curtailment
def _is_useful_pv_to_local_bus(frm, to):
    """
    True for PV actually injected into the local electric system.

    Include:
    - c_i_PV -> c_i_bus_el
    - c_i_PV_to_house -> c_i_bus_el

    Excludes:
    - flows to curtailment
    - potential PV not injected
    """

    frm_s = str(frm)
    to_s = str(to)

    s_all = f"{frm_s} | {to_s}".lower()

    if "curtail" in s_all:
        return False

    from_is_pv = (
        frm_s.endswith("_PV")
        or frm_s.endswith("_PV_to_house")
        or "_PV_to_house" in frm_s
    )

    to_is_local_bus = (
        to_s.endswith("_bus_el")
        or "_bus_el" in to_s
    )

    return bool(from_is_pv and to_is_local_bus)


# Function to extract the household identifier from the technology/column under analysis
def _extract_case_id_any(x):
    """
    Extracts c_i from a SOC or flow column name.
    """
    m = re.search(r"\bc_\d+\b", str(x))
    return m.group(0) if m else None


# Function to rebuild the total storage capacity assigned to each household
def _infer_capacity_from_soc_kwh_pct(soc_kwh, soc_pct):
    """
    Rebuilds the capacity per household using:
        capacity = SOC_kWh / (SOC_pct / 100)

    Works if both sheets are available in the SOC file:
        SOC_kWh
        SOC_pct
    """

    if soc_kwh is None or soc_pct is None:
        return {}

    if soc_kwh.empty or soc_pct.empty:
        return {}

    cap_by_case = {}

    for col_kwh in soc_kwh.columns:
        c = _extract_case_id_any(col_kwh)

        if c is None:
            continue

        pct_candidates = [
            col_pct for col_pct in soc_pct.columns
            if _extract_case_id_any(col_pct) == c
        ]

        if not pct_candidates:
            continue

        col_pct = pct_candidates[0]

        kwh = pd.to_numeric(soc_kwh[col_kwh], errors="coerce")
        pct = pd.to_numeric(soc_pct[col_pct], errors="coerce")

        mask = (
            kwh.notna()
            & pct.notna()
            & (pct.abs() > 1e-9)
        )

        if not mask.any():
            continue

        cap_series = kwh[mask] / (pct[mask] / 100.0)
        cap_series = cap_series.replace([np.inf, -np.inf], np.nan).dropna()

        if cap_series.empty:
            continue

        cap_by_case[c] = float(cap_series.median())

    return cap_by_case


# Function to calculate the charge difference between the end and the beginning of the period for each storage technology
def _read_soc_delta_for_tech(
    tech,
    season,
    base_dir_xls,
    initial_soc_fraction=None,
    fallback_use_first_row=True,
):
    """
    Reads soc_<tech>_<season>.xlsx and returns:
        ΔSoC = final SoC - initial SoC

    As far as possible, uses the capacity rebuilt from SOC_kWh/SOC_pct and
    the global initial fraction:
        BESS -> initial_SoC_BESS
        BEV  -> initial_SoC_BEV
        DHW  -> initial_SoC_DHW

    If this is not possible, uses as fallback:
        initial SoC ≈ first row of the SOC_kWh file

    Note:
    the first row of the saved SOC is generally after the first timestep,
    therefore the fallback is approximate. The estimate based on capacity + fraction
    initial fraction is preferable.
    """

    tech_u = str(tech).upper()

    out = {
        f"soc_{tech_u.lower()}_path": "",
        f"soc_{tech_u.lower()}_initial_kwh": np.nan,
        f"soc_{tech_u.lower()}_final_kwh": np.nan,
        f"soc_{tech_u.lower()}_delta_kwh": 0.0,
        f"soc_{tech_u.lower()}_n_cols": 0,
        f"soc_{tech_u.lower()}_source": "missing",
    }

    try:
        soc_path = soc_output_path(
            tech_u,
            season,
            base_dir=base_dir_xls,
        )
    except Exception:
        return out

    out[f"soc_{tech_u.lower()}_path"] = soc_path

    if not os.path.exists(soc_path):
        return out

    try:
        soc_kwh = pd.read_excel(
            soc_path,
            sheet_name="SOC_kWh",
            index_col=0,
        )
    except Exception as e:
        out[f"soc_{tech_u.lower()}_source"] = f"read_error_SOC_kWh_{type(e).__name__}"
        return out

    if soc_kwh is None or soc_kwh.empty:
        out[f"soc_{tech_u.lower()}_source"] = "empty_SOC_kWh"
        return out

    soc_kwh = soc_kwh.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    try:
        soc_pct = pd.read_excel(
            soc_path,
            sheet_name="SOC_pct",
            index_col=0,
        )
        soc_pct = soc_pct.apply(pd.to_numeric, errors="coerce")
    except Exception:
        soc_pct = None

    final_total = float(soc_kwh.iloc[-1].sum())

    initial_total = np.nan
    source = ""

    cap_by_case = _infer_capacity_from_soc_kwh_pct(soc_kwh, soc_pct)

    if initial_soc_fraction is not None and len(cap_by_case) > 0:
        initial_total = float(
            sum(float(cap) * float(initial_soc_fraction)
                for cap in cap_by_case.values())
        )
        source = "initial_from_capacity_and_global_initial_fraction"

    elif initial_soc_fraction is not None and tech_u == "BESS":
        # robust fallback for homogeneous BESS units
        n_cols = len(soc_kwh.columns)
        initial_total = float(n_cols * BESS_capacity_kwh * float(initial_soc_fraction))
        source = "initial_from_global_BESS_capacity"

    elif fallback_use_first_row:
        initial_total = float(soc_kwh.iloc[0].sum())
        source = "initial_approximated_from_first_SOC_row"

    else:
        initial_total = np.nan
        source = "initial_unavailable"

    if pd.notna(initial_total):
        delta = final_total - initial_total
    else:
        delta = np.nan

    out[f"soc_{tech_u.lower()}_initial_kwh"] = initial_total
    out[f"soc_{tech_u.lower()}_final_kwh"] = final_total
    out[f"soc_{tech_u.lower()}_delta_kwh"] = delta
    out[f"soc_{tech_u.lower()}_n_cols"] = len(soc_kwh.columns)
    out[f"soc_{tech_u.lower()}_source"] = source

    return out


# Function to read the storage state of charge and calculate the delta over the period, useful if some storages have balanced=False
def _read_storage_soc_delta_audit(
    season,
    base_dir_xls,
):
    """
    Reads the BESS, BEV and DHW SOC files and returns the total Delta SoC.

    Convention:
        ΔSoC = final SoC - initial SoC

    ΔSoC > 0:
        storages are net sinks over the period.

    ΔSoC < 0:
        storages are net sources over the period.
    """

    pieces = {}

    tech_specs = {
        "BESS": initial_SoC_BESS if "initial_SoC_BESS" in globals() else None,
        "BEV":  initial_SoC_BEV  if "initial_SoC_BEV"  in globals() else None,
        "DHW":  initial_SoC_DHW  if "initial_SoC_DHW"  in globals() else None,
    }

    total_delta = 0.0
    any_valid = False

    for tech, init_frac in tech_specs.items():
        d_tech = _read_soc_delta_for_tech(
            tech=tech,
            season=season,
            base_dir_xls=base_dir_xls,
            initial_soc_fraction=init_frac,
            fallback_use_first_row=True,
        )

        pieces.update(d_tech)

        delta_key = f"soc_{tech.lower()}_delta_kwh"
        delta_val = d_tech.get(delta_key, np.nan)

        if pd.notna(delta_val):
            total_delta += float(delta_val)
            any_valid = True

    pieces["soc_delta_total_kwh"] = total_delta if any_valid else np.nan
    pieces["soc_delta_source"] = "soc_xlsx" if any_valid else "missing_soc_xlsx"

    pieces["storage_net_sink_kwh"] = (
        max(total_delta, 0.0)
        if any_valid else np.nan
    )

    pieces["storage_net_source_kwh"] = (
        max(-total_delta, 0.0)
        if any_valid else np.nan
    )

    return pieces




# Function to rebuild the metrics used to compare internal self-sufficiency across configurations 
def _read_multirun_energy_breakdown_from_pickles(
    *,
    search_dirs,
    xls_dir,
    season,
    summary_mode,
    timestep_minutes=10,
    values_are_power_kw=True,
    control_BEV=None,
    control_DHW=None,
):
    """
    Rebuilds the energy metrics of the multi-run chart from pickle files.

    Correct sources:
    - flow_el_df:
        base, cooling, heating, cooking
    - flow_DHW_use_df:
        final DHW demand already electric/electric-equivalent (do NOT use the boiler/storage charge flow)
        In OPT, use the flow to the final DHW demand sink:
            c_i_bus_DHW_use -> c_i_dmd_th_DHW
        In RB, use the input profile of final demand:
        input_dmd_th_DHW_kw
    - flow_didischarge_el_BEV_df:
        final BEV demand already electric (do NOT use the home/outdoor charging flow)
        In OPT, use the flow to the mobility demand sink:
            c_i_bus_el_didischarge_BEV -> c_i_dmd_elc_BEV
        In RB, use the input profile of final demand:
        input_dmd_elc_BEV_kw
    - flow_grid_root:
        MV-grid import/export
    """

    def _control_is_opt(x):
        return str(x).upper().strip() in {"OPT", "OPTIMIZED", "SMART"}

    def _control_is_rb(x):
        return str(x).upper().strip() in {"RB", "RB_FIXED", "RULE_BASED"}

    def _infer_tech_control_from_summary_mode(summary_mode, tech):
        """
        Fallback only for backward compatibility.

        In mixed scenarios, it is much better to pass explicitly:
            control_BEV = sc["control_BEV"]
            control_DHW = sc["control_DHW"]

        However, this fallback preserves compatibility with old pure runs:
            summary_mode == "OPT"
            summary_mode == "RB_FIXED"
        """
        m = str(summary_mode).upper().strip()

        if m == "OPT":
            return "OPT"

        if m in {"RB", "RB_FIXED", "RULE_BASED"}:
            return "RB"

        # For mixed mode, try to read from the name, but this is only a backup.
        # Examples:
        #   MIX_BNO_VRB_HOPT
        #   MIX_BNO_VOPT_HRB
        #   MIX_BOPT_VOPT_HOPT
        if tech == "BEV":
            if "_VOPT" in m:
                return "OPT"
            if "_VRB" in m:
                return "RB"

        if tech == "DHW":
            if "_HOPT" in m:
                return "OPT"
            if "_HRB" in m:
                return "RB"

        return None

    control_BEV_eff = control_BEV or _infer_tech_control_from_summary_mode(summary_mode, "BEV")
    control_DHW_eff = control_DHW or _infer_tech_control_from_summary_mode(summary_mode, "DHW")


    flow_el_path = _find_pickle_file_for_metric(
        search_dirs=search_dirs,
        metric_name="flow_el_df",
        season=season,
        summary_mode=summary_mode,
        required=True,
    )

    flow_grid_root_path = _find_pickle_file_for_metric(
        search_dirs=search_dirs,
        metric_name="flow_grid_root",
        season=season,
        summary_mode=summary_mode,
        required=True,
    )

    flow_dhw_use_path = _find_pickle_file_for_metric(
        search_dirs=search_dirs,
        metric_name="flow_DHW_use_df",
        season=season,
        summary_mode=summary_mode,
        required=False,
    )

    flow_bev_discharge_path = _find_pickle_file_for_metric(
        search_dirs=search_dirs,
        metric_name="flow_discharge_el_BEV_df",
        season=season,
        summary_mode=summary_mode,
        required=False,
    )

    flow_el_df = _load_pickle_any(flow_el_path)
    flow_grid_root = _load_pickle_any(flow_grid_root_path)
    flow_DHW_use_df = _load_pickle_any(flow_dhw_use_path) if flow_dhw_use_path else None
    flow_discharge_el_BEV_df = _load_pickle_any(flow_bev_discharge_path) if flow_bev_discharge_path else None

    def _is_flow_to_component(frm, to, target_suffix):
        """
        Robust match of a flow to final demand.
        Examples:
          c_1_bus_el -> c_1_dmd_elc_base
          c_8_bus_DHW_use -> c_8_dmd_th_DHW
          c_11_bus_el_didischarge_BEV -> c_11_dmd_elc_BEV
        """
        frm_s = str(frm)
        to_s = str(to)

        if to_s.endswith(target_suffix):
            return True

        if "->" in to_s:
            left, right = to_s.split("->", 1)
            if right.strip().endswith(target_suffix):
                return True

        if "->" in frm_s:
            left, right = frm_s.split("->", 1)
            if right.strip().endswith(target_suffix):
                return True

        return False

    # ------------------------------------------------------------
    # 1) Domestic electric demands from flow_el_df
    # ------------------------------------------------------------
    demand_base_kwh, cols_base = _sum_flow_df_kwh_by_match(
        flow_el_df,
        lambda frm, to: _is_flow_to_component(frm, to, "_dmd_elc_base"),
        timestep_minutes=timestep_minutes,
        values_are_power_kw=values_are_power_kw,
    )

    demand_cooling_kwh, cols_cooling = _sum_flow_df_kwh_by_match(
        flow_el_df,
        lambda frm, to: _is_flow_to_component(frm, to, "_dmd_elc_Cool"),
        timestep_minutes=timestep_minutes,
        values_are_power_kw=values_are_power_kw,
    )

    demand_heating_kwh, cols_heating = _sum_flow_df_kwh_by_match(
        flow_el_df,
        lambda frm, to: _is_flow_to_component(frm, to, "_dmd_elc_Heat"),
        timestep_minutes=timestep_minutes,
        values_are_power_kw=values_are_power_kw,
    )

    demand_cooking_kwh, cols_cooking = _sum_flow_df_kwh_by_match(
        flow_el_df,
        lambda frm, to: _is_flow_to_component(frm, to, "_dmd_elc_Cook"),
        timestep_minutes=timestep_minutes,
        values_are_power_kw=values_are_power_kw,
    )

    # ------------------------------------------------------------
    # 2) Final DHW demand
    # ------------------------------------------------------------
    if flow_DHW_use_df is not None and not flow_DHW_use_df.empty:

        if _control_is_opt(control_DHW_eff):
            # Case OPT:
            # c_i_bus_DHW_use -> c_i_dmd_th_DHW
            demand_dhw_final_kwh, cols_dhw = _sum_flow_df_kwh_by_match(
                flow_DHW_use_df,
                lambda frm, to: _is_flow_to_component(frm, to, "_dmd_th_DHW"),
                timestep_minutes=timestep_minutes,
                values_are_power_kw=values_are_power_kw,
            )

            demand_dhw_source = "OPT_final_DHW_from_flow_DHW_use_df"

        elif _control_is_rb(control_DHW_eff):
            # Case RB_FIXED:
            # (c_i, input_dmd_th_DHW_kw, flow)
            demand_dhw_final_kwh, cols_dhw = _sum_flow_df_kwh_by_match(
                flow_DHW_use_df,
                lambda frm, to: (
                    "input_dmd_th_DHW_kw" in str(frm)
                    or "input_dmd_th_DHW_kw" in str(to)
                ),
                timestep_minutes=timestep_minutes,
                values_are_power_kw=values_are_power_kw,
            )

            demand_dhw_source = "RB_final_DHW_input_dmd_th_DHW_kw"

        else:
            demand_dhw_total_kwh = 0.0
            cols_dhw = []
            demand_dhw_source = f"unknown_DHW_control_{control_DHW_eff}"

    else:
        demand_dhw_final_kwh = 0.0
        cols_dhw = []
        demand_dhw_source = "missing_flow_DHW_use_df"

    # ------------------------------------------------------------
    # 3) Final EV demand
    # ------------------------------------------------------------
    if flow_discharge_el_BEV_df is not None and not flow_discharge_el_BEV_df.empty:

        if _control_is_opt(control_BEV_eff):
            # Case OPT:
            # c_i_bus_el_didischarge_BEV -> c_i_dmd_elc_BEV
            demand_bev_final_kwh, cols_ev = _sum_flow_df_kwh_by_match(
                flow_discharge_el_BEV_df,
                lambda frm, to: _is_flow_to_component(frm, to, "_dmd_elc_BEV"),
                timestep_minutes=timestep_minutes,
                values_are_power_kw=values_are_power_kw,
            )

            demand_ev_source = "OPT_final_EV_from_flow_discharge_el_BEV_df"

        elif _control_is_rb(control_BEV_eff):
            # Case RB_FIXED:
            # (c_i, input_dmd_elc_BEV_kw, flow)
            demand_bev_final_kwh, cols_ev = _sum_flow_df_kwh_by_match(
                flow_discharge_el_BEV_df,
                lambda frm, to: (
                    "input_dmd_elc_BEV_kw" in str(frm)
                    or "input_dmd_elc_BEV_kw" in str(to)
                ),
                timestep_minutes=timestep_minutes,
                values_are_power_kw=values_are_power_kw,
            )

            demand_ev_source = "RB_final_EV_input_dmd_elc_BEV_kw"

        else:
            demand_ev_charging_kwh = 0.0
            cols_ev = []
            demand_ev_source = f"unknown_BEV_control_{control_BEV_eff}"

    else:
        demand_bev_final_kwh = 0.0
        cols_ev = []
        demand_ev_source = "missing_flow_discharge_el_BEV_df"

    # ------------------------------------------------------------
    # 4) Total final demand
    # ------------------------------------------------------------
    demand_components_total_kwh = (
        demand_base_kwh
        + demand_cooling_kwh
        + demand_heating_kwh
        + demand_cooking_kwh
        + demand_bev_final_kwh
        + demand_dhw_final_kwh
    )

    total_final_demand_kwh = demand_components_total_kwh

    # ------------------------------------------------------------
    # 5) MV-grid import/export from flow_grid_root
    # ------------------------------------------------------------
    mv_import_total_kwh, cols_mv_import = _sum_flow_df_kwh_by_match(
        flow_grid_root,
        lambda frm, to: (
            "supply_from_grid_MT" in str(frm)
            or "supply_from_grid_MT" in str(to)
            or "import from mv grid" in str(frm).lower()
            or "import from mv grid" in str(to).lower()
        ),
        timestep_minutes=timestep_minutes,
        values_are_power_kw=values_are_power_kw,
    )

    mv_export_total_kwh, cols_mv_export = _sum_flow_df_kwh_by_match(
        flow_grid_root,
        lambda frm, to: (
            "excess_to_grid_MT" in str(frm)
            or "excess_to_grid_MT" in str(to)
            or "excess to mv grid" in str(frm).lower()
            or "excess to mv grid" in str(to).lower()
        ),
        timestep_minutes=timestep_minutes,
        values_are_power_kw=values_are_power_kw,
    )

    net_mv_import_kwh_raw = mv_import_total_kwh - mv_export_total_kwh

    # ------------------------------------------------------------
    # 5bis) Balance used to correctly track losses and Delta SoC
    # ------------------------------------------------------------

    # Useful PV actually injected into the local system, net of curtailment
    pv_to_local_bus_kwh, cols_pv_to_local_bus = _sum_flow_df_kwh_by_match(
        flow_el_df,
        _is_useful_pv_to_local_bus,
        timestep_minutes=timestep_minutes,
        values_are_power_kw=values_are_power_kw,
    )

    if xls_dir is None:
        raise ValueError(
            "xls_dir is required to read storage-SOC audit files."
        )

    soc_audit = _read_storage_soc_delta_audit(
        season=season,
        base_dir_xls=xls_dir,
    )

    soc_delta_total_kwh = soc_audit.get("soc_delta_total_kwh", np.nan)

    # If net_mv_import_kwh_raw < 0, this may exceed demand.
    internal_self_sufficiency_signed_kwh = (
        total_final_demand_kwh
        - net_mv_import_kwh_raw
    )

    internal_self_sufficiency_signed_share = (
        internal_self_sufficiency_signed_kwh / total_final_demand_kwh
        if abs(total_final_demand_kwh) > 1e-12
        else np.nan
    )

    # Gen = Dmd + losses --> PV + NetImp = Dmd + Delta SoC + losses
    if pd.notna(soc_delta_total_kwh):
        internal_self_sufficiency_soc_corrected_kwh = (
            internal_self_sufficiency_signed_kwh
            + soc_delta_total_kwh
        )
    else:
        internal_self_sufficiency_soc_corrected_kwh = np.nan

    internal_self_sufficiency_soc_corrected_share = (
        internal_self_sufficiency_soc_corrected_kwh / total_final_demand_kwh
        if pd.notna(internal_self_sufficiency_soc_corrected_kwh)
        and abs(total_final_demand_kwh) > 1e-12
        else np.nan
    )

    # The residual should represent losses + untracked mismatches.
    if pd.notna(internal_self_sufficiency_soc_corrected_kwh):
        pv_vs_soc_corrected_self_suff_gap_kwh = (
            pv_to_local_bus_kwh
            - internal_self_sufficiency_soc_corrected_kwh
        )
    else:
        pv_vs_soc_corrected_self_suff_gap_kwh = np.nan

    pv_vs_soc_corrected_self_suff_gap_pct_pv = (
        100.0 * pv_vs_soc_corrected_self_suff_gap_kwh / pv_to_local_bus_kwh
        if pd.notna(pv_vs_soc_corrected_self_suff_gap_kwh)
        and abs(pv_to_local_bus_kwh) > 1e-12
        else np.nan
    )

    pv_vs_soc_corrected_self_suff_gap_pct_demand = (
        100.0 * pv_vs_soc_corrected_self_suff_gap_kwh / total_final_demand_kwh
        if pd.notna(pv_vs_soc_corrected_self_suff_gap_kwh)
        and abs(total_final_demand_kwh) > 1e-12
        else np.nan
    )

    # Simple Dmd - PV vs NetImp check.
    demand_minus_useful_pv_kwh = (
        total_final_demand_kwh
        - pv_to_local_bus_kwh
    )

    simple_pv_mv_gap_kwh = (
        demand_minus_useful_pv_kwh
        - net_mv_import_kwh_raw
    )

    simple_pv_mv_gap_pct_demand = (
        100.0 * simple_pv_mv_gap_kwh / total_final_demand_kwh
        if abs(total_final_demand_kwh) > 1e-12
        else np.nan
    )

    net_mv_import_positive_kwh = max(net_mv_import_kwh_raw, 0.0)
    net_mv_export_positive_kwh = max(-net_mv_import_kwh_raw, 0.0)

    gross_mv_import_share_of_demand = (
        mv_import_total_kwh / total_final_demand_kwh
        if abs(total_final_demand_kwh) > 1e-12
        else np.nan
    )

    gross_mv_export_share_of_demand = (
        mv_export_total_kwh / total_final_demand_kwh
        if abs(total_final_demand_kwh) > 1e-12
        else np.nan
    )

    net_mv_import_share_of_demand = (
        net_mv_import_kwh_raw / total_final_demand_kwh
        if abs(total_final_demand_kwh) > 1e-12
        else np.nan
    )





    if net_mv_import_kwh_raw > 1e-6:
        mv_net_position = "net_importer"
    elif net_mv_import_kwh_raw < -1e-6:
        mv_net_position = "net_exporter"
    else:
        mv_net_position = "balanced"

    # ------------------------------------------------------------
    # Clipped quantities for the coverage chart, i.e. if there is net export to MV, set MV import to zero and do not treat it as negative
    # ------------------------------------------------------------
    net_mv_import_for_demand_kwh = min(
        max(net_mv_import_kwh_raw, 0.0),
        total_final_demand_kwh
    )

    internal_self_sufficiency_kwh = max(
        0.0,
        total_final_demand_kwh - net_mv_import_for_demand_kwh
    )

    net_mv_import_exceeds_demand = bool(
        net_mv_import_kwh_raw > total_final_demand_kwh + 1e-6
    )

    out = {
        "total_local_demand_kwh": total_final_demand_kwh,
        "total_el_demand_kwh": total_final_demand_kwh,
        "total_final_demand_kwh": total_final_demand_kwh,

        "mv_import_total_kwh": mv_import_total_kwh,
        "mv_export_total_kwh": mv_export_total_kwh,

        # Signed MV net.
        # > 0 net import; < 0 net export.
        "net_mv_import_kwh_raw": net_mv_import_kwh_raw,
        "net_mv_import_positive_kwh": net_mv_import_positive_kwh,
        "net_mv_export_positive_kwh": net_mv_export_positive_kwh,
        "mv_net_position": mv_net_position,

        "gross_mv_import_share_of_demand": gross_mv_import_share_of_demand,
        "gross_mv_export_share_of_demand": gross_mv_export_share_of_demand,
        "net_mv_import_share_of_demand": net_mv_import_share_of_demand,

        # Clipped MV net, i.e. MV export set to zero and not negative, used by the coverage chart.
        "net_mv_import_for_demand_kwh": net_mv_import_for_demand_kwh,

        # Useful PV and simple A-B vs E check.
        "pv_to_local_bus_kwh": pv_to_local_bus_kwh,
        "demand_minus_useful_pv_kwh": demand_minus_useful_pv_kwh,
        "simple_pv_mv_gap_kwh": simple_pv_mv_gap_kwh,
        "simple_pv_mv_gap_pct_demand": simple_pv_mv_gap_pct_demand,
        "n_cols_pv_to_local_bus_matched": len(cols_pv_to_local_bus),

        # ΔSoC storage.
        "soc_delta_total_kwh": soc_delta_total_kwh,
        "storage_net_sink_kwh": soc_audit.get("storage_net_sink_kwh", np.nan),
        "storage_net_source_kwh": soc_audit.get("storage_net_source_kwh", np.nan),
        "soc_delta_source": soc_audit.get("soc_delta_source", ""),

        "soc_bess_initial_kwh": soc_audit.get("soc_bess_initial_kwh", np.nan),
        "soc_bess_final_kwh": soc_audit.get("soc_bess_final_kwh", np.nan),
        "soc_bess_delta_kwh": soc_audit.get("soc_bess_delta_kwh", np.nan),
        "soc_bess_source": soc_audit.get("soc_bess_source", ""),

        "soc_bev_initial_kwh": soc_audit.get("soc_bev_initial_kwh", np.nan),
        "soc_bev_final_kwh": soc_audit.get("soc_bev_final_kwh", np.nan),
        "soc_bev_delta_kwh": soc_audit.get("soc_bev_delta_kwh", np.nan),
        "soc_bev_source": soc_audit.get("soc_bev_source", ""),

        "soc_dhw_initial_kwh": soc_audit.get("soc_dhw_initial_kwh", np.nan),
        "soc_dhw_final_kwh": soc_audit.get("soc_dhw_final_kwh", np.nan),
        "soc_dhw_delta_kwh": soc_audit.get("soc_dhw_delta_kwh", np.nan),
        "soc_dhw_source": soc_audit.get("soc_dhw_source", ""),

        # Backward compatibility
        "mt_import_total_kwh": net_mv_import_for_demand_kwh,
        "mt_import_total_kwh_raw": mv_import_total_kwh,
        "mt_export_total_kwh": mv_export_total_kwh,

        # Clipped self-sufficiency, i.e. export to the MV grid set to zero and not negative, used in the coverage chart.
        "internal_self_sufficiency_kwh": internal_self_sufficiency_kwh,
        "local_self_consumption_kwh": internal_self_sufficiency_kwh,

        # Signed self-sufficiency corrected for Delta SoC, i.e. it correctly accounts for import - export with losses.
        "internal_self_sufficiency_signed_kwh": internal_self_sufficiency_signed_kwh,
        "internal_self_sufficiency_signed_share": internal_self_sufficiency_signed_share,

        "internal_self_sufficiency_soc_corrected_kwh": internal_self_sufficiency_soc_corrected_kwh,
        "internal_self_sufficiency_soc_corrected_share": internal_self_sufficiency_soc_corrected_share,

        # Residual interpretable as losses + untracked mismatch.
        "pv_vs_soc_corrected_self_suff_gap_kwh": pv_vs_soc_corrected_self_suff_gap_kwh,
        "pv_vs_soc_corrected_self_suff_gap_pct_pv": pv_vs_soc_corrected_self_suff_gap_pct_pv,
        "pv_vs_soc_corrected_self_suff_gap_pct_demand": pv_vs_soc_corrected_self_suff_gap_pct_demand,

        "demand_base_kwh": demand_base_kwh,
        "demand_cooling_kwh": demand_cooling_kwh,
        "demand_heating_kwh": demand_heating_kwh,
        "demand_cooking_kwh": demand_cooking_kwh,
        "demand_bev_final_kwh": demand_bev_final_kwh,
        "demand_ev_final_kwh": demand_bev_final_kwh,
        "demand_dhw_final_kwh": demand_dhw_final_kwh,

        # Backward-compatible aliases.
        # Warning: demand_ev_charging_kwh no longer represents BEV charging.
        # Kept only to avoid breaking old code.
        "demand_ev_charging_kwh": demand_bev_final_kwh,
        "demand_dhw_total_kwh": demand_dhw_final_kwh,

        "demand_components_total_kwh": demand_components_total_kwh,
        "demand_components_gap_kwh": total_final_demand_kwh - demand_components_total_kwh,

        "energy_breakdown_source": "pickle_fallback_component_specific",
        "flow_el_pickle_used": flow_el_path,
        "flow_grid_root_pickle_used": flow_grid_root_path,
        "flow_dhw_use_pickle_used": flow_dhw_use_path or "",
        "flow_discharge_bev_pickle_used": flow_bev_discharge_path or "",

        "demand_dhw_source": demand_dhw_source,
        "demand_ev_source": demand_ev_source,
        "control_BEV_used_for_energy": control_BEV_eff,
        "control_DHW_used_for_energy": control_DHW_eff,

        "net_mv_import_exceeds_demand": net_mv_import_exceeds_demand,

        "n_cols_demand_base_matched": len(cols_base),
        "n_cols_demand_cooling_matched": len(cols_cooling),
        "n_cols_demand_heating_matched": len(cols_heating),
        "n_cols_demand_cooking_matched": len(cols_cooking),
        "n_cols_demand_ev_matched": len(cols_ev),
        "n_cols_demand_dhw_matched": len(cols_dhw),
        "n_cols_mv_import_matched": len(cols_mv_import),
        "n_cols_mv_export_matched": len(cols_mv_export),
    }
    return out
# ---------------------------------------------------------



# ---------------------------------------------------------
#       Sec 4.6 -- 14.5) FINAL DATASET AND 4.13 CHARTS
# ---------------------------------------------------------
# Function to save household economic percentiles for the configurations to compare into dictionaries
def build_multirun_cost_percentile_summary(
    scenarios,
    PV_share,
    PV_power_kwp,
    preferred_total_cost_col="total_system_cost_effective_eur",
    skip_missing=True,
    allow_energy_pickle_fallback=True,
    pickle_values_are_power_kw=True,
    energy_source="pickle",
):
    """
    Builds the table for the multi-run chart.

    Each scenario is a dictionary, for example:

    {
        "label": "OPT - 75% BESS",
        "season": "estate",
        "share_FV": 100,
        "share_BESS": 75,
        "share_BEV": 0,
        "share_boiler_DHW": 0,
        "pot_FV": 3,
        "summary_mode": "OPT",
        "comparison_kind": "opt",
    }

    summary_mode:
      - "OPT"
      - "RB_FIXED"
    """

    rows = []
    skipped = []

    for i, sc in enumerate(scenarios, start=1):
        label = sc.get("label", f"Scenario {i}")
        season = sc.get("season", sc.get("season", None))
        if season is None:
            raise KeyError(f"Scenario without season: {sc}")

        pv = int(sc.get("share_FV", PV_share))
        b = int(sc.get("share_BESS", BESS_share))
        v = int(sc.get("share_BEV", EV_share))
        h = int(sc.get("share_boiler_DHW", sc.get("share_DHW", ESWH_share)))
        p_pv = sc.get("pot_FV", PV_power_kwp)

        summary_mode = sc.get("summary_mode", sc.get("mode", "OPT"))
        comparison_kind = sc.get(
            "comparison_kind",
            "opt" if str(summary_mode).upper() == "OPT" else "sim"
        )

        summary_file = sc.get("summary_file", None)
        if summary_file is None:
            summary_file = _scenario_summary_file(sc)


        try:
            results_file = _scenario_results_file(
                sc,
                season=season,
            )

            (
                s_costs,
                comparison_sheet,
                comparison_cost_col,
            ) = _read_house_costs_from_mode_results(
                results_file=results_file,
                mode=summary_mode,
            )

            house_cost_source_file = results_file



            total_system_cost, total_cost_col_used = _read_total_system_cost_from_summary(
                summary_file=summary_file,
                season=season,
                summary_mode=summary_mode,
                preferred_total_cost_col=preferred_total_cost_col,
            )
            
            energy_source_l = str(energy_source).lower().strip()

            def _energy_from_pickles():
                return _read_multirun_energy_breakdown_from_pickles(
                    search_dirs=_scenario_pickle_search_dirs(sc),
                    xls_dir=sc["xls_dir"],
                    season=season,
                    summary_mode=summary_mode,
                    timestep_minutes=timestep_minutes,
                    values_are_power_kw=pickle_values_are_power_kw,
                    control_BEV=sc.get("control_BEV"),
                    control_DHW=sc.get("control_DHW"),
                )

            def _energy_from_summary():
                return _read_multirun_energy_breakdown_from_summary(
                    summary_file=summary_file,
                    season=season,
                    summary_mode=summary_mode,
                    allow_missing=False,
                )

            if energy_source_l == "pickle":
                energy_breakdown = _energy_from_pickles()

            elif energy_source_l == "summary":
                energy_breakdown = _energy_from_summary()

            elif energy_source_l == "auto":
                try:
                    energy_breakdown = _energy_from_summary()
                except Exception as e_summary_energy:
                    if not allow_energy_pickle_fallback:
                        raise
                    print(
                        f"[WARN][MULTIRUN ENERGY] Summary cannot be used for "
                        f"scenario '{label}', season={season}. "
                        f"Using pickle. Summary error: {type(e_summary_energy).__name__}: {e_summary_energy}"
                    )
                    energy_breakdown = _energy_from_pickles()

            else:
                raise ValueError(
                    "energy_source must be one of: 'pickle', 'summary', 'auto'."
                )

            # Observed week: use the flow_el_df pickle index if available.
            week_start = ""
            week_end = ""

            if isinstance(energy_breakdown.get("flow_el_pickle_used", ""), str):
                try:
                    _tmp_flow_el = _load_pickle_any(energy_breakdown["flow_el_pickle_used"])
                    if hasattr(_tmp_flow_el, "index") and len(_tmp_flow_el.index) > 0:
                        week_start = str(pd.to_datetime(_tmp_flow_el.index[0]))
                        week_end = str(pd.to_datetime(_tmp_flow_el.index[-1]))
                except Exception:
                    pass


            total_el_demand_kwh = float(
                energy_breakdown.get("total_local_demand_kwh", np.nan)
            )

            total_demand_col_used = energy_breakdown.get(
                "energy_breakdown_source",
                "unknown"
            )

            avg_system_cost_eur_per_kwh = (
                total_system_cost / total_el_demand_kwh
                if pd.notna(total_el_demand_kwh) and total_el_demand_kwh > 1e-12
                else np.nan
            )


            
            # average weekly system operating cost per household, i.e. total cost divided by number of households
            n_households_for_avg_cost = int(s_costs.shape[0])
            avg_household_weekly_system_cost_eur = (
                total_system_cost / n_households_for_avg_cost
                if n_households_for_avg_cost > 0 and pd.notna(total_system_cost)
                else np.nan
            )


        except Exception as e:
            msg = (
                f"[WARN][MULTIRUN COST] Skip scenario '{label}' "
                f"due to error: {type(e).__name__}: {e}"
            )
            print(msg)

            skipped.append({
                "x_order": i,
                "label": label,
                "season": season,
                "share_FV_%": pv,
                "share_BESS_%": b,
                "share_BEV_%": v,
                "share_DHW_%": h,
                "summary_mode": summary_mode,
                "comparison_kind": comparison_kind,
                "summary_file": summary_file,
                "results_file": locals().get("results_file", ""),
                "error": str(e),
                "house_cost_source_file": locals().get("house_cost_source_file", ""),
            })

            if skip_missing:
                continue
            else:
                raise

        pct_values = np.percentile(
            s_costs.values,
            [10, 20, 30, 40, 50, 60, 70, 80, 90]
        ).astype(float)

        rows.append({
            "x_order": i,
            "label": label,
            "scenario_display_label": sc.get(
                "scenario_display_label",
                "Base case" if i == 1 else f"Case {i - 1}"
            ),
            "season": season,
            "summary_mode": summary_mode,
            "comparison_kind": comparison_kind,

            "control_BESS": sc.get("control_BESS", ""),
            "control_BEV": sc.get("control_BEV", ""),
            "control_DHW": sc.get("control_DHW", ""),

            "share_FV_%": pv,
            "share_BESS_%": b,
            "share_BEV_%": v,
            "share_DHW_%": h,

            "p10_eur": pct_values[0],
            "p20_eur": pct_values[1],
            "p30_eur": pct_values[2],
            "p40_eur": pct_values[3],
            "p50_eur": pct_values[4],
            "p60_eur": pct_values[5],
            "p70_eur": pct_values[6],
            "p80_eur": pct_values[7],
            "p90_eur": pct_values[8],

            "total_system_cost_eur": total_system_cost,
            "total_cost_col_used": total_cost_col_used,
            "total_el_demand_kwh": total_el_demand_kwh,
            "total_demand_col_used": total_demand_col_used,
            "avg_system_cost_eur_per_kwh": avg_system_cost_eur_per_kwh,                     # system operating cost per unit of final electricity demand [EUR/kWh]
            "n_households_for_avg_cost": n_households_for_avg_cost,
            "avg_household_weekly_system_cost_eur": avg_household_weekly_system_cost_eur,   # average weekly cost per household, total cost / no. households [EUR/household]


            "scenario_group": sc.get("scenario_group", "other"),

            "week_start": week_start,
            "week_end": week_end,

            # Energy quantities for the chart:
            # column 1 = net MV import + internal self-sufficiency
            # column 2 = final-demand components
            "total_final_demand_kwh": energy_breakdown.get("total_final_demand_kwh", np.nan),
            "mv_import_total_kwh": energy_breakdown.get("mv_import_total_kwh", np.nan),
            "mv_export_total_kwh": energy_breakdown.get("mv_export_total_kwh", np.nan),

            "net_mv_import_kwh_raw": energy_breakdown.get("net_mv_import_kwh_raw", np.nan),
            "net_mv_import_positive_kwh": energy_breakdown.get("net_mv_import_positive_kwh", np.nan),
            "net_mv_export_positive_kwh": energy_breakdown.get("net_mv_export_positive_kwh", np.nan),
            "mv_net_position": energy_breakdown.get("mv_net_position", ""),

            "net_mv_import_for_demand_kwh": energy_breakdown.get("net_mv_import_for_demand_kwh", np.nan),

            "gross_mv_import_share_of_demand": energy_breakdown.get("gross_mv_import_share_of_demand", np.nan),
            "gross_mv_export_share_of_demand": energy_breakdown.get("gross_mv_export_share_of_demand", np.nan),
            "net_mv_import_share_of_demand": energy_breakdown.get("net_mv_import_share_of_demand", np.nan),

            # Useful-PV and MV-balance audit.
            "pv_to_local_bus_kwh": energy_breakdown.get("pv_to_local_bus_kwh", np.nan),
            "demand_minus_useful_pv_kwh": energy_breakdown.get("demand_minus_useful_pv_kwh", np.nan),
            "simple_pv_mv_gap_kwh": energy_breakdown.get("simple_pv_mv_gap_kwh", np.nan),
            "simple_pv_mv_gap_pct_demand": energy_breakdown.get("simple_pv_mv_gap_pct_demand", np.nan),
            "n_cols_pv_to_local_bus_matched": energy_breakdown.get("n_cols_pv_to_local_bus_matched", np.nan),

            # ΔSoC storage.
            "soc_delta_total_kwh": energy_breakdown.get("soc_delta_total_kwh", np.nan),
            "storage_net_sink_kwh": energy_breakdown.get("storage_net_sink_kwh", np.nan),
            "storage_net_source_kwh": energy_breakdown.get("storage_net_source_kwh", np.nan),
            "soc_delta_source": energy_breakdown.get("soc_delta_source", ""),

            "soc_bess_initial_kwh": energy_breakdown.get("soc_bess_initial_kwh", np.nan),
            "soc_bess_final_kwh": energy_breakdown.get("soc_bess_final_kwh", np.nan),
            "soc_bess_delta_kwh": energy_breakdown.get("soc_bess_delta_kwh", np.nan),
            "soc_bess_source": energy_breakdown.get("soc_bess_source", ""),

            "soc_bev_initial_kwh": energy_breakdown.get("soc_bev_initial_kwh", np.nan),
            "soc_bev_final_kwh": energy_breakdown.get("soc_bev_final_kwh", np.nan),
            "soc_bev_delta_kwh": energy_breakdown.get("soc_bev_delta_kwh", np.nan),
            "soc_bev_source": energy_breakdown.get("soc_bev_source", ""),

            "soc_dhw_initial_kwh": energy_breakdown.get("soc_dhw_initial_kwh", np.nan),
            "soc_dhw_final_kwh": energy_breakdown.get("soc_dhw_final_kwh", np.nan),
            "soc_dhw_delta_kwh": energy_breakdown.get("soc_dhw_delta_kwh", np.nan),
            "soc_dhw_source": energy_breakdown.get("soc_dhw_source", ""),

            # Backward compatibility
            "mt_import_total_kwh": energy_breakdown.get("mt_import_total_kwh", np.nan),
            "mt_import_total_kwh_raw": energy_breakdown.get("mt_import_total_kwh_raw", np.nan),
            "mt_export_total_kwh": energy_breakdown.get("mt_export_total_kwh", np.nan),
            "internal_self_sufficiency_kwh": energy_breakdown.get("internal_self_sufficiency_kwh", np.nan),
            "local_self_consumption_kwh": energy_breakdown.get("local_self_consumption_kwh", np.nan),

            "internal_self_sufficiency_signed_kwh": energy_breakdown.get("internal_self_sufficiency_signed_kwh", np.nan),
            "internal_self_sufficiency_signed_share": energy_breakdown.get("internal_self_sufficiency_signed_share", np.nan),

            "internal_self_sufficiency_soc_corrected_kwh": energy_breakdown.get("internal_self_sufficiency_soc_corrected_kwh", np.nan),
            "internal_self_sufficiency_soc_corrected_share": energy_breakdown.get("internal_self_sufficiency_soc_corrected_share", np.nan),

            "pv_vs_soc_corrected_self_suff_gap_kwh": energy_breakdown.get("pv_vs_soc_corrected_self_suff_gap_kwh", np.nan),
            "pv_vs_soc_corrected_self_suff_gap_pct_pv": energy_breakdown.get("pv_vs_soc_corrected_self_suff_gap_pct_pv", np.nan),
            "pv_vs_soc_corrected_self_suff_gap_pct_demand": energy_breakdown.get("pv_vs_soc_corrected_self_suff_gap_pct_demand", np.nan),

            "demand_base_kwh": energy_breakdown.get("demand_base_kwh", np.nan),
            "demand_cooling_kwh": energy_breakdown.get("demand_cooling_kwh", np.nan),
            "demand_heating_kwh": energy_breakdown.get("demand_heating_kwh", np.nan),
            "demand_cooking_kwh": energy_breakdown.get("demand_cooking_kwh", np.nan),
            # "demand_ev_charging_kwh": energy_breakdown.get("demand_ev_charging_kwh", np.nan),
            # "demand_dhw_total_kwh": energy_breakdown.get("demand_dhw_total_kwh", np.nan),
            "demand_bev_final_kwh": energy_breakdown.get("demand_bev_final_kwh", energy_breakdown.get("demand_ev_final_kwh", energy_breakdown.get("demand_ev_charging_kwh", np.nan))),
            "demand_ev_final_kwh": energy_breakdown.get("demand_ev_final_kwh", energy_breakdown.get("demand_bev_final_kwh", energy_breakdown.get("demand_ev_charging_kwh", np.nan))),
            "demand_dhw_final_kwh": energy_breakdown.get("demand_dhw_final_kwh", energy_breakdown.get("demand_dhw_total_kwh", np.nan)),


            # Backward-compatible aliases
            "demand_ev_charging_kwh": energy_breakdown.get("demand_ev_charging_kwh", energy_breakdown.get("demand_ev_final_kwh", energy_breakdown.get("demand_bev_final_kwh", np.nan))),
            "demand_dhw_total_kwh": energy_breakdown.get("demand_dhw_total_kwh", energy_breakdown.get("demand_dhw_final_kwh", np.nan)),     

            
            "demand_components_total_kwh": energy_breakdown.get("demand_components_total_kwh", np.nan),
            "demand_components_gap_kwh": energy_breakdown.get("demand_components_gap_kwh", np.nan),

            "energy_breakdown_source": energy_breakdown.get("energy_breakdown_source", ""),
            "flow_el_pickle_used": energy_breakdown.get("flow_el_pickle_used", ""),
            "flow_grid_root_pickle_used": energy_breakdown.get("flow_grid_root_pickle_used", ""),
            "flow_dhw_use_pickle_used": energy_breakdown.get("flow_dhw_use_pickle_used", ""),
            "flow_discharge_bev_pickle_used": energy_breakdown.get("flow_discharge_bev_pickle_used", ""),


            "demand_dhw_source": energy_breakdown.get("demand_dhw_source", ""),
            "demand_ev_source": energy_breakdown.get("demand_ev_source", ""),
            "net_mv_import_exceeds_demand": energy_breakdown.get("net_mv_import_exceeds_demand", False),

            "control_BEV_used_for_energy": energy_breakdown.get("control_BEV_used_for_energy", ""),
            "control_DHW_used_for_energy": energy_breakdown.get("control_DHW_used_for_energy", ""),

            "house_cost_source_file": house_cost_source_file,


            "n_cols_demand_base_matched": energy_breakdown.get("n_cols_demand_base_matched", np.nan),
            "n_cols_demand_cooling_matched": energy_breakdown.get("n_cols_demand_cooling_matched", np.nan),
            "n_cols_demand_heating_matched": energy_breakdown.get("n_cols_demand_heating_matched", np.nan),
            "n_cols_demand_cooking_matched": energy_breakdown.get("n_cols_demand_cooking_matched", np.nan),
            "n_cols_demand_ev_matched": energy_breakdown.get("n_cols_demand_ev_matched", np.nan),
            "n_cols_demand_dhw_matched": energy_breakdown.get("n_cols_demand_dhw_matched", np.nan),
            "n_cols_mv_import_matched": energy_breakdown.get("n_cols_mv_import_matched", np.nan),
            "n_cols_mv_export_matched": energy_breakdown.get("n_cols_mv_export_matched", np.nan),


            "n_house_costs": int(s_costs.shape[0]),
            "summary_file": summary_file,
            "house_cost_source_file": house_cost_source_file,
            "comparison_sheet": comparison_sheet,
            "comparison_cost_col": comparison_cost_col,
        })

    df_plot = pd.DataFrame(rows)
    df_skipped = pd.DataFrame(skipped)

    if not df_skipped.empty:
        print("\n[INFO][MULTIRUN COST] Skipped scenarios:")
        cols_show = [
            "label",
            "season",
            "summary_mode",
            "comparison_kind",
            "summary_file",
            "error",
        ]
        cols_show = [c for c in cols_show if c in df_skipped.columns]
        print(df_skipped[cols_show].to_string(index=False))

    if df_plot.empty:
        raise ValueError(
            "\nNo valid scenario found for the multi-run chart.\n"
            "See the [INFO][MULTIRUN COST] table above: it contains the precise reason.\n"
            "Typical causes in the new mixed schema:\n"
            "1) system_cost_summary_<MODE>_<n>_nuc_<region>.xlsx file missing in the MODE folder;\n"
            "2) ris_<season>.xlsx file missing in the MODE folder;\n"
            "3) total_costs_per_household sheet missing in ris_<season>.xlsx;\n"
            "4) flow_el_<season>.pkl.gz or flow_grid_root_<season>.pkl.gz pickle missing;\n"
            "5) _scenario_run_dir/_scenario_pickle_search_dirs not aligned with the compact roots 04_HGD_res_*.\n"
        )

    return (
        df_plot.sort_values("x_order").reset_index(drop=True),
        df_skipped.sort_values("x_order").reset_index(drop=True) if not df_skipped.empty else df_skipped,
    )


# Function to build household-level costs and to define delta between baseline and other considered scenarios
def build_multirun_household_cost_delta_summary(
    scenarios,
    reference_case_label="Base case",
    pair_key="electrification_id",
    zero_tolerance_eur=1e-9,
    skip_missing=True,
):
    """
    Compare household costs against the reference control case
    within the same physical electrification scenario.
    """

    percentile_levels = [10, 20, 30, 40, 50, 60, 70, 80, 90]

    data_by_uid = {}
    skipped_rows = []

    # --------------------------------------------------------
    # 1) Read household costs for every selected run
    # --------------------------------------------------------
    for i, scenario in enumerate(scenarios, start=1):

        run_uid = str(scenario["run_uid"])
        season = scenario["season"]

        try:
            results_file = _scenario_results_file(
                scenario,
                season=season,
            )

            household_costs, source_sheet, source_cost_col = (
                _read_house_costs_from_mode_results(
                    results_file=results_file,
                    mode=scenario["summary_mode"],
                )
            )

            household_costs = household_costs.copy()

            household_costs.index = pd.Index(
                [
                    to_household_name(house_id)
                    for house_id in household_costs.index
                ],
                name="house_id",
            )

            if household_costs.index.has_duplicates:
                raise ValueError(
                    f"Duplicated household IDs for {run_uid}."
                )

            household_costs = pd.to_numeric(
                household_costs,
                errors="coerce",
            ).dropna()

            if household_costs.empty:
                raise ValueError(
                    f"No valid household costs for {run_uid}."
                )

            data_by_uid[run_uid] = {
                "costs": household_costs,
                "meta": {
                    "x_order": i,
                    "run_uid": run_uid,
                    "run_label": scenario["label"],
                    "case_label": scenario.get("case_label"),
                    "electrification_id": scenario.get(
                        "electrification_id"
                    ),
                    "season": season,
                    "mode": scenario["summary_mode"],
                    "scenario_group": scenario.get("scenario_group"),
                    "control_BESS": scenario.get("control_BESS"),
                    "control_BEV": scenario.get("control_BEV"),
                    "control_DHW": scenario.get("control_DHW"),
                    "share_FV_%": scenario.get("share_FV"),
                    "share_BESS_%": scenario.get("share_BESS"),
                    "share_BEV_%": scenario.get("share_BEV"),
                    "share_DHW_%": scenario.get("share_boiler_DHW"),
                    "share_HP_%": scenario.get("share_HP"),
                    "share_COOL_%": scenario.get("share_COOL"),
                    "source_file": results_file,
                    "source_sheet": source_sheet,
                    "source_cost_column": source_cost_col,
                },
            }

        except Exception as error:
            skipped_rows.append({
                "x_order": i,
                "run_uid": run_uid,
                "label": scenario.get("label"),
                "season": season,
                "error": f"{type(error).__name__}: {error}",
            })

            if skip_missing:
                continue
            raise

    # --------------------------------------------------------
    # 2) Group runs by physical scenario
    # --------------------------------------------------------
    groups = {}

    for run_uid, item in data_by_uid.items():
        pair_value = item["meta"].get(pair_key)

        if pair_value is None:
            raise KeyError(
                f"Missing pair key '{pair_key}' for {run_uid}."
            )

        groups.setdefault(str(pair_value), []).append(run_uid)

    delta_summary_rows = []
    household_detail_rows = []
    costs_wide = {}
    deltas_wide = {}

    # --------------------------------------------------------
    # 3) Find one Base/reference inside every physical scenario
    # --------------------------------------------------------
    for pair_value, run_uids in groups.items():

        reference_uids = [
            uid
            for uid in run_uids
            if data_by_uid[uid]["meta"]["case_label"]
            == reference_case_label
        ]

        if len(reference_uids) != 1:
            raise ValueError(
                f"{pair_key}={pair_value}: expected exactly one "
                f"'{reference_case_label}' run, found "
                f"{len(reference_uids)}."
            )

        reference_uid = reference_uids[0]
        reference_item = data_by_uid[reference_uid]
        reference_costs = reference_item["costs"]

        for run_uid in run_uids:

            item = data_by_uid[run_uid]
            scenario_costs = item["costs"]
            metadata = item["meta"].copy()

            paired = pd.concat(
                [
                    reference_costs.rename("reference_cost_eur"),
                    scenario_costs.rename("scenario_cost_eur"),
                ],
                axis=1,
                join="inner",
            ).dropna()

            if paired.empty:
                continue

            paired["delta_eur"] = (
                paired["scenario_cost_eur"]
                - paired["reference_cost_eur"]
            )

            delta_series = paired["delta_eur"].astype(float)
            run_label = metadata["run_label"]

            costs_wide[run_label] = scenario_costs
            deltas_wide[run_label] = delta_series

            pct = np.percentile(
                delta_series.to_numpy(dtype=float),
                percentile_levels,
            )

            n_paired = len(delta_series)
            n_saving = int(
                (delta_series < -zero_tolerance_eur).sum()
            )
            n_penalized = int(
                (delta_series > zero_tolerance_eur).sum()
            )
            n_unchanged = int(
                (delta_series.abs() <= zero_tolerance_eur).sum()
            )

            summary_row = {
                **metadata,
                "pair_key": pair_key,
                "pair_value": pair_value,
                "reference_run_uid": reference_uid,
                "reference_run_label": reference_item["meta"]["run_label"],
                "reference_case_label": reference_case_label,
                "n_households_paired": n_paired,
                "mean_delta_eur": float(delta_series.mean()),
                "min_delta_eur": float(delta_series.min()),
                "max_delta_eur": float(delta_series.max()),
                "n_households_saving": n_saving,
                "n_households_unchanged": n_unchanged,
                "n_households_penalized": n_penalized,
                "share_households_saving_pct": 100 * n_saving / n_paired,
                "share_households_unchanged_pct": 100 * n_unchanged / n_paired,
                "share_households_penalized_pct": 100 * n_penalized / n_paired,
            }

            for percentile, value in zip(percentile_levels, pct):
                summary_row[f"p{percentile}_delta_eur"] = float(value)

            delta_summary_rows.append(summary_row)

            for house_id, row in paired.iterrows():
                household_detail_rows.append({
                    "season": metadata["season"],
                    "pair_value": pair_value,
                    "reference_run_uid": reference_uid,
                    "run_uid": run_uid,
                    "run_label": run_label,
                    "house_id": house_id,
                    "reference_cost_eur": float(row["reference_cost_eur"]),
                    "scenario_cost_eur": float(row["scenario_cost_eur"]),
                    "delta_eur": float(row["delta_eur"]),
                })

    df_delta_summary = pd.DataFrame(delta_summary_rows)

    if not df_delta_summary.empty:
        df_delta_summary = (
            df_delta_summary
            .sort_values("x_order")
            .reset_index(drop=True)
        )

    df_household_detail = pd.DataFrame(household_detail_rows)
    df_costs_wide = pd.DataFrame(costs_wide)
    df_deltas_wide = pd.DataFrame(deltas_wide)
    df_skipped = pd.DataFrame(skipped_rows)

    return (
        df_delta_summary,
        df_household_detail,
        df_costs_wide,
        df_deltas_wide,
        df_skipped,
    )



# ------------------------------------------------------------
#       Sec 4.6 -- 14.6) Market KPIs and additional-cost audit
# ------------------------------------------------------------
# Function to read system costs from result files
def _read_system_cost_ts_from_results_file(results_file):
    """
    Legge il foglio system_cost_ts da ris_<season>.xlsx.
    """
    if not os.path.exists(results_file):
        raise FileNotFoundError(f"Results file not found: {results_file}")

    xls = pd.ExcelFile(results_file)

    if "system_cost_ts" not in xls.sheet_names:
        raise KeyError(
            f"Sheet 'system_cost_ts' not found in {results_file}. "
            f"Fogli disponibili: {xls.sheet_names}"
        )

    df = pd.read_excel(results_file, sheet_name="system_cost_ts", index_col=0)

    try:
        df.index = pd.to_datetime(df.index)
    except Exception:
        pass

    return df


# Function to integrate missing/relevant cost terms into result files
def _ensure_market_cost_columns(df_ts):
    """
    Adds missing economic columns, using what is already available
    in the old ris_<season>.xlsx files.

    Does not require the optimisation to be rerun.

    Hierarchy:
      1) use new columns, if already available;
      2) otherwise, use the old columns from system_cost_ts;
      3) if anything is missing, create NaN values or safe fallbacks.
    """
    df = df_ts.copy()

    # ------------------------------------------------------------
    # 1) Alias for total system cost per timestep
    # ------------------------------------------------------------
    if "total_system_cost_per_timestep_eur" not in df.columns:
        for candidate in [
            "total_system_cost_including_objective_variable_terms_eur",
            "total_system_cost_business_eur",
            "objective_variable_cost_terms_eur",
        ]:
            if candidate in df.columns:
                df["total_system_cost_per_timestep_eur"] = pd.to_numeric(
                    df[candidate],
                    errors="coerce",
                )
                break

    if "total_system_cost_per_timestep_eur" not in df.columns:
        df["total_system_cost_per_timestep_eur"] = np.nan

    # ------------------------------------------------------------
    # 2) Local demand for EUR/kWh normalizations
    # ------------------------------------------------------------
    if "total_local_demand_kwh" not in df.columns:
        df["total_local_demand_kwh"] = np.nan

    # ------------------------------------------------------------
    # 3) Energy settlement consistent with the objective function
    # ------------------------------------------------------------
    # In old files this column already exists and is the most consistent one
    # with the way how the objective function was built.
    if "objective_energy_settlement_eur" not in df.columns:
        if "bt_energy_objective_settlement_eur" in df.columns:
            df["objective_energy_settlement_eur"] = pd.to_numeric(
                df["bt_energy_objective_settlement_eur"],
                errors="coerce",
            )

        elif "bt_energy_net_settlement_eur" in df.columns:
            df["objective_energy_settlement_eur"] = pd.to_numeric(
                df["bt_energy_net_settlement_eur"],
                errors="coerce",
            )

        else:
            purchase = pd.to_numeric(
                df.get("bt_purchase_cost_eur", 0.0),
                errors="coerce",
            ).fillna(0.0)

            sale = pd.to_numeric(
                df.get("bt_sale_revenues_eur", 0.0),
                errors="coerce",
            ).fillna(0.0)

            # convention: settlement = purchase + export_settlement,
            df["objective_energy_settlement_eur"] = purchase + sale

    # ------------------------------------------------------------
    # 4) Fallback market_* fields for multi-run compatibility.
    # ------------------------------------------------------------
    # Warning: if mt_export_kwh is not available in system_cost_ts,
    # These values are proxies based on LV-house settlement,
    # not actual gross exchanges at the MV grid interface.
    if "market_mt_purchase_cost_eur" not in df.columns:
        df["market_mt_purchase_cost_eur"] = pd.to_numeric(
            df.get("bt_purchase_cost_eur", np.nan),
            errors="coerce",
        )

    if "market_mt_sale_revenues_eur" not in df.columns:
        df["market_mt_sale_revenues_eur"] = pd.to_numeric(
            df.get("bt_sale_revenues_eur", np.nan),
            errors="coerce",
        )

    if "market_mt_net_cost_eur" not in df.columns:
        # Per old file uso la grandezza objective-consistent.
        # Is more robusta rispetto a ricostruzioni parziali MT.
        df["market_mt_net_cost_eur"] = pd.to_numeric(
            df["objective_energy_settlement_eur"],
            errors="coerce",
        )

    if "market_mt_gross_absolute_value_eur" not in df.columns:
        df["market_mt_gross_absolute_value_eur"] = (
            pd.to_numeric(df["market_mt_purchase_cost_eur"], errors="coerce").abs()
            + pd.to_numeric(df["market_mt_sale_revenues_eur"], errors="coerce").abs()
        )

    # ------------------------------------------------------------
    # 5) Costs normalized by timestep demand
    # ------------------------------------------------------------
    if "market_mt_net_cost_eur_per_kwh_demand" not in df.columns:
        df["market_mt_net_cost_eur_per_kwh_demand"] = np.where(
            pd.to_numeric(df["total_local_demand_kwh"], errors="coerce") > 1e-12,
            pd.to_numeric(df["market_mt_net_cost_eur"], errors="coerce")
            / pd.to_numeric(df["total_local_demand_kwh"], errors="coerce"),
            np.nan,
        )

    if "total_system_cost_per_kwh_demand_timestep" not in df.columns:
        df["total_system_cost_per_kwh_demand_timestep"] = np.where(
            pd.to_numeric(df["total_local_demand_kwh"], errors="coerce") > 1e-12,
            pd.to_numeric(df["total_system_cost_per_timestep_eur"], errors="coerce")
            / pd.to_numeric(df["total_local_demand_kwh"], errors="coerce"),
            np.nan,
        )

    # ------------------------------------------------------------
    # 6) KPI import/export su demand
    # ------------------------------------------------------------
    if "gross_mv_import_share_of_demand_ts" not in df.columns:
        if "mt_import_kwh" in df.columns:
            df["gross_mv_import_share_of_demand_ts"] = np.where(
                pd.to_numeric(df["total_local_demand_kwh"], errors="coerce") > 1e-12,
                pd.to_numeric(df["mt_import_kwh"], errors="coerce")
                / pd.to_numeric(df["total_local_demand_kwh"], errors="coerce"),
                np.nan,
            )
        else:
            df["gross_mv_import_share_of_demand_ts"] = np.nan

    if "gross_mv_export_share_of_demand_ts" not in df.columns:
        if "mt_export_kwh" in df.columns:
            df["gross_mv_export_share_of_demand_ts"] = np.where(
                pd.to_numeric(df["total_local_demand_kwh"], errors="coerce") > 1e-12,
                pd.to_numeric(df["mt_export_kwh"], errors="coerce")
                / pd.to_numeric(df["total_local_demand_kwh"], errors="coerce"),
                np.nan,
            )
        else:
            df["gross_mv_export_share_of_demand_ts"] = np.nan

    if "gross_mv_exchange_share_of_demand_ts" not in df.columns:
        if ("mt_import_kwh" in df.columns) and ("mt_export_kwh" in df.columns):
            df["gross_mv_exchange_share_of_demand_ts"] = np.where(
                pd.to_numeric(df["total_local_demand_kwh"], errors="coerce") > 1e-12,
                (
                    pd.to_numeric(df["mt_import_kwh"], errors="coerce").fillna(0.0)
                    + pd.to_numeric(df["mt_export_kwh"], errors="coerce").fillna(0.0)
                )
                / pd.to_numeric(df["total_local_demand_kwh"], errors="coerce"),
                np.nan,
            )
        else:
            df["gross_mv_exchange_share_of_demand_ts"] = np.nan

    return df


# Function to rebuild the system-cost time series, per timestep, for each configuration and season
def build_multirun_market_cost_timeseries_dataset(
    scenarios,
    PV_share,
    PV_power_kwp,
    columns_to_export=None,
):
    """
    It creates:
      - df_market_ts_wide: wide time series, columns = Scenario | metric
      - df_market_summary: summary KPIs by scenario

    It also works on older results, using a fallback to system_cost_ts.
    """
    if columns_to_export is None:
        columns_to_export = globals().get(
            "MULTIRUN_MARKET_TS_COLUMNS",
            [
                "market_mt_net_cost_eur",
                "market_mt_purchase_cost_eur",
                "market_mt_sale_revenues_eur",
                "market_mt_net_cost_eur_per_kwh_demand",
                "total_system_cost_per_timestep_eur",
            ],
        )

    wide_parts = []
    summary_rows = []
    skipped_rows = []

    for i, sc in enumerate(scenarios, start=1):
        label = sc.get("label", f"Scenario {i}")
        display_label = sc.get(
            "scenario_display_label",
            "Base case" if i == 1 else f"Case {i - 1}",
        )

        season = sc.get("season", sc.get("season", None))
        summary_mode = sc.get("summary_mode", sc.get("mode", "OPT"))

        pv = int(sc.get("share_FV", PV_share))
        b = int(sc.get("share_BESS", BESS_share))
        v = int(sc.get("share_BEV", EV_share))
        h = int(sc.get("share_boiler_DHW", sc.get("share_DHW", ESWH_share)))
        p_pv = sc.get("pot_FV", PV_power_kwp)

        try:
            results_file = _scenario_results_file(
                sc,
                season=season,
            )

            df_ts = _read_system_cost_ts_from_results_file(results_file)
            df_ts = _ensure_market_cost_columns(df_ts)

            # Wide dataframe
            df_piece = pd.DataFrame(index=df_ts.index)

            for col in columns_to_export:
                if col in df_ts.columns:
                    df_piece[f"{display_label} | {col}"] = pd.to_numeric(
                        df_ts[col],
                        errors="coerce",
                    )
                else:
                    df_piece[f"{display_label} | {col}"] = np.nan

            wide_parts.append(df_piece)

            total_demand = float(
                pd.to_numeric(df_ts.get("total_local_demand_kwh", np.nan), errors="coerce").sum()
            )

            mt_import = float(
                pd.to_numeric(df_ts.get("mt_import_kwh", np.nan), errors="coerce").sum()
            ) if "mt_import_kwh" in df_ts.columns else np.nan

            mt_export = float(
                pd.to_numeric(df_ts.get("mt_export_kwh", np.nan), errors="coerce").sum()
            ) if "mt_export_kwh" in df_ts.columns else np.nan

            market_net = float(
                pd.to_numeric(df_ts["market_mt_net_cost_eur"], errors="coerce").sum()
            )

            market_purchase = float(
                pd.to_numeric(df_ts["market_mt_purchase_cost_eur"], errors="coerce").sum()
            )

            market_revenues = float(
                pd.to_numeric(df_ts["market_mt_sale_revenues_eur"], errors="coerce").sum()
            )

            total_system_ts = float(
                pd.to_numeric(df_ts["total_system_cost_per_timestep_eur"], errors="coerce").sum()
            )

            summary_rows.append({
                "x_order": i,
                "label": label,
                "scenario_display_label": display_label,
                "season": season,
                "summary_mode": summary_mode,
                "results_file": results_file,

                "total_local_demand_kwh": total_demand,
                "mv_import_total_kwh": mt_import,
                "mv_export_total_kwh": mt_export,
                "mv_net_import_kwh": (
                    mt_import - mt_export
                    if pd.notna(mt_import) and pd.notna(mt_export)
                    else np.nan
                ),

                "gross_mv_import_share_of_demand": (
                    mt_import / total_demand
                    if pd.notna(mt_import) and total_demand > 1e-12
                    else np.nan
                ),
                "gross_mv_export_share_of_demand": (
                    mt_export / total_demand
                    if pd.notna(mt_export) and total_demand > 1e-12
                    else np.nan
                ),
                "gross_mv_exchange_share_of_demand": (
                    (mt_import + mt_export) / total_demand
                    if pd.notna(mt_import)
                    and pd.notna(mt_export)
                    and total_demand > 1e-12
                    else np.nan
                ),

                "market_mt_purchase_cost_eur": market_purchase,
                "market_mt_sale_revenues_eur": market_revenues,
                "market_mt_net_cost_eur": market_net,
                "total_system_cost_from_ts_eur": total_system_ts,

                "average_market_mt_net_cost_eur_per_timestep": float(
                    pd.to_numeric(df_ts["market_mt_net_cost_eur"], errors="coerce").mean()
                ),
                "average_total_system_cost_eur_per_timestep": float(
                    pd.to_numeric(df_ts["total_system_cost_per_timestep_eur"], errors="coerce").mean()
                ),
                "average_market_mt_net_cost_eur_per_kwh_demand": (
                    market_net / total_demand
                    if total_demand > 1e-12
                    else np.nan
                ),
            })

        except Exception as e:
            skipped_rows.append({
                "x_order": i,
                "label": label,
                "scenario_display_label": display_label,
                "season": season,
                "summary_mode": summary_mode,
                "error": f"{type(e).__name__}: {e}",
            })

    if wide_parts:
        df_market_ts_wide = pd.concat(wide_parts, axis=1)
    else:
        df_market_ts_wide = pd.DataFrame()

    df_market_summary = pd.DataFrame(summary_rows)
    df_market_skipped = pd.DataFrame(skipped_rows)

    return df_market_ts_wide, df_market_summary, df_market_skipped


# ------------------------------------------------------------
#       Sec 4.6 -- 14.6bis) Duration curves of per-timestep costs
# ------------------------------------------------------------
# Function to identify the analyzed configuration name
def _market_ts_scenario_label_from_col(col, metric_col):
    """
    From:
        'Case 1 | total_system_cost_per_timestep_eur'
    returns:
        'Case 1'
    """
    s = str(col)
    suffix = f" | {metric_col}"

    if s.endswith(suffix):
        return s[:-len(suffix)]

    if " | " in s:
        return s.split(" | ", 1)[0]

    return s


# Function to build the dataframe containing system-cost duration curves for the different configurations
def build_multirun_duration_curve_dataframe(
    df_market_ts_wide,
    metric_col="total_system_cost_per_timestep_eur",
    descending=True,
):
    """
    Creates a wide dataframe containing duration curves.

    Rows:
      exceedance_percent [%]

    Columns:
      scenarios

    Values:
      metric sorted from highest to lowest value.
    """
    if df_market_ts_wide is None or df_market_ts_wide.empty:
        return pd.DataFrame()

    suffix = f" | {metric_col}"

    metric_cols = [
        c for c in df_market_ts_wide.columns
        if str(c).endswith(suffix)
    ]

    if not metric_cols:
        print(
            f"[WARN][DURATION CURVE] No column found for metric_col='{metric_col}'."
        )
        print("[INFO] Available columns in df_market_ts_wide:")
        print(list(df_market_ts_wide.columns))
        return pd.DataFrame()

    curves = []

    for col in metric_cols:
        s = (
            pd.to_numeric(df_market_ts_wide[col], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if s.empty:
            continue

        values = np.sort(s.values)

        if descending:
            values = values[::-1]

        exceedance_percent = (
            np.arange(1, len(values) + 1, dtype=float)
            / float(len(values))
            * 100.0
        )

        scenario_label = _market_ts_scenario_label_from_col(col, metric_col)

        curves.append(
            pd.Series(
                data=values,
                index=exceedance_percent,
                name=scenario_label,
            )
        )

    if not curves:
        return pd.DataFrame()

    duration_df = pd.concat(curves, axis=1)
    duration_df.index.name = "exceedance_percent"

    return duration_df


# Function to print/plot the chart with per-timestep system-cost duration curves across configurations
def plot_multirun_system_cost_duration_curves(
    df_market_ts_wide,
    title,
    metric_col="total_system_cost_per_timestep_eur",
    out_png=None,
    out_xlsx=None,
    y_label=None,
    legend_mode=None,
):
    """
    Graph showing the duration curves of the economic metric by time step.

    Interpretation:
      - x = time steps ordered from highest to lowest cost;
      - y = cost in the time step.

    Useful for understanding whether a scenario is cost-effective only over the entire
    period or also in terms of the temporal distribution of costs.
    """
    duration_df = build_multirun_duration_curve_dataframe(
        df_market_ts_wide=df_market_ts_wide,
        metric_col=metric_col,
        descending=True,
    )

    if duration_df is None or duration_df.empty:
        print("[WARN][DURATION CURVE] No valid data: skip chart.")
        return duration_df

    if y_label is None:
        y_label_map = {
            "total_system_cost_per_timestep_eur": "Total system cost per timestep [EUR]",
            "objective_energy_settlement_eur": "Objective energy settlement per timestep [EUR]",
            "market_mt_net_cost_eur": "Market/energy settlement per timestep [EUR]",
            "market_mt_net_cost_eur_per_kwh_demand": "Market/energy settlement [EUR/kWh demand]",
            "total_system_cost_per_kwh_demand_timestep": "Total system cost [EUR/kWh demand]",
        }

        y_label = y_label_map.get(metric_col, metric_col)

    if out_xlsx:
        os.makedirs(os.path.dirname(out_xlsx), exist_ok=True)
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            duration_df.to_excel(writer, sheet_name="duration_curves", index=True)

        print(f"[OK] Salvata tabella curve di durata: {out_xlsx}")

    fig, ax = plt.subplots(
        figsize=(max(11.0, 1.25 * len(duration_df.columns) + 4.0), 6.5),
        constrained_layout=False,
    )

    for col in duration_df.columns:
        ax.plot(
            duration_df.index.values,
            pd.to_numeric(duration_df[col], errors="coerce").values,
            linewidth=1.8,
            label=str(col),
        )

    ax.set_xlim(0.0, 100.0)
    ax.set_xlabel("Share of timesteps sorted from highest to lowest cost [%]")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.grid(axis="both", linestyle=":", alpha=0.4)

    ylo, yhi = ax.get_ylim()
    if ylo < 0.0 < yhi:
        ax.axhline(
            0.0,
            color="black",
            linewidth=0.8,
            alpha=0.45,
        )

    handles, labels = ax.get_legend_handles_labels()

    _add_smart_inside_legend(
        ax=ax,
        handles=handles,
        labels=labels,
        title="Scenario",
        fontsize=8,
        title_fontsize=9,
        ncol=2,
        legend_mode=legend_mode,
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.95,
        top=0.86,
        bottom=0.12,
    )

    if out_png:
        os.makedirs(os.path.dirname(out_png), exist_ok=True)
        fig.savefig(out_png, dpi=300, bbox_inches="tight", pad_inches=0.04)
        print(f"[OK] Saved cost duration-curve chart: {out_png}")

    plt.show()

    return duration_df









# ---------------------------------------------------
#   Sec 4.6 -- 14.6ter) Function to calculate cost-term shares of the total
# ---------------------------------------------------

# Function that checks the weight of additional-cost/shortage terms, e.g. PV curtailment, DHW boost, BEV outdoor, relative to system cost
def build_multirun_additional_cost_weight_audit(
    scenarios,
    PV_share,
    PV_power_kwp,
):
    """
    Check the proportion of the additional cost items relative to the total cost.

    Excludes:
      - bt_energy_import_settlement_eur
      - bt_energy_export_settlement_eur

    Includes:
      - dhw_boost_cost_eur
      - bev_outdoor_charge_cost_eur
      - pv_curtail_cost_eur
      - kerber_line_transfer_cost_eur
      - shortage_cost_eur
      - other_variable_cost_terms_eur
    """
    excluded_categories = {
        "bt_energy_import_settlement_eur",
        "bt_energy_export_settlement_eur",
    }

    detail_rows = []
    summary_rows = []
    skipped_rows = []

    for i, sc in enumerate(scenarios, start=1):
        label = sc.get("label", f"Scenario {i}")
        display_label = sc.get(
            "scenario_display_label",
            "Base case" if i == 1 else f"Case {i - 1}",
        )

        season = sc.get("season", sc.get("season", None))
        summary_mode = sc.get("summary_mode", sc.get("mode", "OPT"))

        pv = int(sc.get("share_FV", PV_share))
        b = int(sc.get("share_BESS", BESS_share))
        v = int(sc.get("share_BEV", EV_share))
        h = int(sc.get("share_boiler_DHW", sc.get("share_DHW", ESWH_share)))
        p_pv = sc.get("pot_FV", PV_power_kwp)

        try:
            results_file = _scenario_results_file(
                sc,
                season=season,
            )

            xls = pd.ExcelFile(results_file)

            if "objective_terms_summary" not in xls.sheet_names:
                raise KeyError("objective_terms_summary sheet missing")

            df_obj = pd.read_excel(results_file, sheet_name="objective_terms_summary")

            if "system_cost_summary" in xls.sheet_names:
                df_sys = pd.read_excel(results_file, sheet_name="system_cost_summary")
                sys_row = df_sys.iloc[0] if not df_sys.empty else pd.Series(dtype=float)
            else:
                sys_row = pd.Series(dtype=float)

            objective_total = pd.to_numeric(
                sys_row.get("objective_total_eur", np.nan),
                errors="coerce",
            )

            total_system_cost = pd.to_numeric(
                sys_row.get(
                    "total_system_cost_including_all_objective_terms_eur",
                    objective_total,
                ),
                errors="coerce",
            )

            if "category" not in df_obj.columns:
                raise KeyError("objective_terms_summary does not contain the category column")

            df_obj = df_obj.copy()
            df_obj["term_total_eur"] = pd.to_numeric(
                df_obj.get("term_total_eur", 0.0),
                errors="coerce",
            ).fillna(0.0)

            df_obj["term_abs_total_eur"] = pd.to_numeric(
                df_obj.get("term_abs_total_eur", df_obj["term_total_eur"].abs()),
                errors="coerce",
            ).fillna(df_obj["term_total_eur"].abs())

            df_obj["is_energy_settlement"] = df_obj["category"].isin(excluded_categories)
            df_obj["is_additional_cost"] = ~df_obj["is_energy_settlement"]

            for _, r in df_obj.iterrows():
                term_total = float(r["term_total_eur"])
                term_abs = float(r["term_abs_total_eur"])

                detail_rows.append({
                    "x_order": i,
                    "label": label,
                    "scenario_display_label": display_label,
                    "season": season,
                    "summary_mode": summary_mode,
                    "category": r.get("category", ""),
                    "cost_group": r.get("cost_group", ""),
                    "description": r.get("description", ""),
                    "term_total_eur": term_total,
                    "term_abs_total_eur": term_abs,
                    "is_energy_settlement": bool(r["is_energy_settlement"]),
                    "is_additional_cost": bool(r["is_additional_cost"]),
                    "share_of_objective_total": (
                        term_total / objective_total
                        if pd.notna(objective_total) and abs(objective_total) > 1e-12
                        else np.nan
                    ),
                    "share_abs_of_abs_objective_total": (
                        term_abs / abs(objective_total)
                        if pd.notna(objective_total) and abs(objective_total) > 1e-12
                        else np.nan
                    ),
                    "results_file": results_file,
                })

            df_add = df_obj[df_obj["is_additional_cost"]].copy()

            additional_total = float(df_add["term_total_eur"].sum())
            additional_abs_total = float(df_add["term_abs_total_eur"].sum())

            summary_rows.append({
                "x_order": i,
                "label": label,
                "scenario_display_label": display_label,
                "season": season,
                "summary_mode": summary_mode,
                "objective_total_eur": float(objective_total) if pd.notna(objective_total) else np.nan,
                "total_system_cost_eur": float(total_system_cost) if pd.notna(total_system_cost) else np.nan,
                "additional_cost_total_eur": additional_total,
                "additional_cost_abs_total_eur": additional_abs_total,
                "additional_cost_share_of_objective": (
                    additional_total / objective_total
                    if pd.notna(objective_total) and abs(objective_total) > 1e-12
                    else np.nan
                ),
                "additional_abs_cost_share_of_abs_objective": (
                    additional_abs_total / abs(objective_total)
                    if pd.notna(objective_total) and abs(objective_total) > 1e-12
                    else np.nan
                ),
                "n_additional_categories": int(df_add["category"].nunique()),
                "results_file": results_file,
            })

        except Exception as e:
            skipped_rows.append({
                "x_order": i,
                "label": label,
                "scenario_display_label": display_label,
                "season": season,
                "summary_mode": summary_mode,
                "error": f"{type(e).__name__}: {e}",
            })

    return (
        pd.DataFrame(detail_rows),
        pd.DataFrame(summary_rows),
        pd.DataFrame(skipped_rows),
    )










# ------------------------------------------------------------
#       Sec 4.6 -- 14.7) Functions to check internal self-sufficiency 
# ------------------------------------------------------------
# Function to distinguish between self-load on the grid side and self-load on the customer side, tracking what is accumulated in storage systems (Delta SoC)
def print_multirun_self_sufficiency_audit(
    df_plot,
    out_xlsx=None,
    gap_warn_pct_demand=2.0,
    gap_warn_abs_kwh=100.0,
):
    """
    Consistency check of the internal share for the period to distinguish between:
    - self-sufficiency clipped from the chart (i.e. exports to MV are treated as zero, not negative);
    - self-sufficiency as reported (i.e. imports and exports are correctly accounted for, including losses);
    - correction for ΔSoC storage;
    - comparison with net PV excluding curtailment.

    Key metric:

        internal_self_sufficiency_soc_corrected
        =
        total_final_demand
        - net_mv_import_raw
        + ΔSoC_storage

    This should be comparable with:
        useful PV – losses

    The gap:
        useful PV – internal_self_sufficiency_soc_corrected

    can be interpreted as:
        losses + unaccounted mismatch
    """

    if df_plot is None or df_plot.empty:
        print("[WARN][SELF-SUFF CHECK] empty df_plot.")
        return pd.DataFrame()

    d = df_plot.copy()

    cols = [
        "label",
        "season",
        "summary_mode",

        "total_final_demand_kwh",

        "mv_import_total_kwh",
        "mv_export_total_kwh",
        "net_mv_import_kwh_raw",
        "net_mv_import_positive_kwh",
        "net_mv_export_positive_kwh",
        "mv_net_position",

        "net_mv_import_for_demand_kwh",

        "internal_self_sufficiency_kwh",
        "internal_self_sufficiency_signed_kwh",
        "internal_self_sufficiency_soc_corrected_kwh",

        "pv_to_local_bus_kwh",

        "soc_delta_total_kwh",
        "storage_net_sink_kwh",
        "storage_net_source_kwh",

        "soc_bess_delta_kwh",
        "soc_bev_delta_kwh",
        "soc_dhw_delta_kwh",

        "pv_vs_soc_corrected_self_suff_gap_kwh",
        "pv_vs_soc_corrected_self_suff_gap_pct_demand",
        "pv_vs_soc_corrected_self_suff_gap_pct_pv",

        "simple_pv_mv_gap_kwh",
        "simple_pv_mv_gap_pct_demand",

        "n_cols_pv_to_local_bus_matched",
        "n_cols_mv_import_matched",
        "n_cols_mv_export_matched",

        "soc_delta_source",
        "soc_bess_source",
        "soc_bev_source",
        "soc_dhw_source",
    ]

    cols = [c for c in cols if c in d.columns]
    audit = d[cols].copy()

    print("\n[CHECK][Consistency check for internal share WITH SOC CORRECTION]")
    print("D = total final demand")
    print("E = MV import - MV export")
    print("PV = useful PV to local buses, excluding curtailment")
    print("ΔSoC = final storage energy - initial storage energy")
    print("signed self-sufficiency = D - E")
    print("SoC-corrected self-sufficiency = D - E + ΔSoC")
    print("PV gap = PV - SoC-corrected self-sufficiency")
    print(audit.to_string(index=False))

    if "pv_vs_soc_corrected_self_suff_gap_pct_demand" in audit.columns:
        gap_pct = pd.to_numeric(
            audit["pv_vs_soc_corrected_self_suff_gap_pct_demand"],
            errors="coerce",
        )

        gap_abs = pd.to_numeric(
            audit["pv_vs_soc_corrected_self_suff_gap_kwh"],
            errors="coerce",
        ).abs()

        bad = audit[
            (gap_pct.abs() > float(gap_warn_pct_demand))
            | (gap_abs > float(gap_warn_abs_kwh))
        ]

        if not bad.empty:
            print(
                "\n[WARN][SELF-SUFF CHECK] Relevant gap between useful PV and "
                "self-sufficiency corretta per ΔSoC. "
                "May indicate high losses, flow mismatch, curtailment not excluded "
                "correctly, or dynamics storage/grid to focus on."
            )

            warn_cols = [
                c for c in [
                    "label",
                    "pv_to_local_bus_kwh",
                    "internal_self_sufficiency_soc_corrected_kwh",
                    "soc_delta_total_kwh",
                    "pv_vs_soc_corrected_self_suff_gap_kwh",
                    "pv_vs_soc_corrected_self_suff_gap_pct_demand",
                    "mv_net_position",
                ]
                if c in bad.columns
            ]

            print(bad[warn_cols].to_string(index=False))

    if out_xlsx:
        os.makedirs(os.path.dirname(out_xlsx), exist_ok=True)
        audit.to_excel(out_xlsx, index=False)
        print(f"[OK][SELF-SUFF AUDIT] Saved audit: {out_xlsx}")

    return audit



# Function to get the gradient of a colour
def _lighten_color(color, amount=0.5):
    """
    Schiarisce un colore.
    amount=0 -> colore originale
    amount=1 -> bianco
    """
    try:
        c = np.array(mcolors.to_rgb(color))
    except Exception:
        c = np.array(mcolors.to_rgb("grey"))
    white = np.array([1.0, 1.0, 1.0])
    return tuple(c + (white - c) * amount)


# Function to return X if the value is empty/None, used for the mini configuration table under the 4.13 charts
def _is_missing_like(x):
    """
    True se x is vuoto/NaN/None.
    """
    if x is None:
        return True

    try:
        if pd.isna(x):
            return True
    except Exception:
        pass

    return str(x).strip() == ""


# Function to define which smart technologies are available in the observed configuration, used for the mini configuration table under the 4.13 charts
def _extract_control_from_label(label, tech):
    """
    Fallback: extracts the control technology from the label type:
        “BESS NO | EV RB | DHW OPT”

    Important note:
      - DHW remains the internal name used by the model.
      - ESWH is merely a graphical/text label.
      - Therefore, ESWH is treated as an alias for DHW.
    """
    label = str(label)
    tech = str(tech).upper().strip()

    tech_aliases = {
        "BEV": "EV",
        "ESWH": "DHW",
        "E-SWH": "DHW",
        "EL STORAGE WATER HEATER": "DHW",
        "ELECTRIC STORAGE WATER HEATER": "DHW",
    }
    tech = tech_aliases.get(tech, tech)

    aliases = {
        "BESS": ["BESS"],
        "EV": ["EV", "BEV"],
        "DHW": ["DHW", "ESWH", "E-SWH"],
    }

    for part in label.split("|"):
        p = part.strip()
        p_up = p.upper()

        if any(p_up.startswith(a) for a in aliases.get(tech, [tech])):
            tokens = p_up.split()
            if len(tokens) >= 2:
                return tokens[-1]

    return ""


# Function to link character to the available flexibility technology in the tested configuration
def _control_state_from_value(control_value):
    """
    Normalized the management of a technology between following states:
      OPT, RB, NO
    """
    if _is_missing_like(control_value):
        return ""

    s = str(control_value).strip().upper()

    if s in {"NO", "NONE", "NAN", "NA", "ASSENTE", "ABSENT", "OFF", "FALSE", "0"}:
        return "NO"

    if s in {"OPT", "SMART", "SPUNTA", "OPTIMIZED", "OTTIMIZZATO"} or "OPT" in s:
        return "OPT"

    if s in {"RB", "RB_FIXED", "RULE_BASED", "RULE-BASED", "X", "CONVENTIONAL"} or "RULE" in s:
        return "RB"

    return s


# Function to assign table symbols in comparative charts
def _normalize_control_for_table(control_value, symbol_style=None):
    """
    Mapping chart for the table under the simulated comparison charts.

    Style selected from:
        MULTIRUN_CONTROL_SYMBOL_STYLE
    """
    state = _control_state_from_value(control_value)

    if state == "":
        return ""

    if symbol_style is None:
        symbol_style = globals().get(
            "MULTIRUN_CONTROL_SYMBOL_STYLE",
            "check_x_no",
        )

    symbol_map = MULTIRUN_CONTROL_SYMBOL_MAPS.get(
        symbol_style,
        MULTIRUN_CONTROL_SYMBOL_MAPS["check_x_no"],
    )

    return symbol_map.get(state, state)


# Function to print/plot the table legend for comparative configuration charts, mapping selected symbols to meanings 
def _control_legend_note(symbol_style=None):
    """
    Dynamic note to print/plot below the configuration table.
    """
    if symbol_style is None:
        symbol_style = globals().get(
            "MULTIRUN_CONTROL_SYMBOL_STYLE",
            "check_x_no",
        )

    symbol_map = MULTIRUN_CONTROL_SYMBOL_MAPS.get(
        symbol_style,
        MULTIRUN_CONTROL_SYMBOL_MAPS["check_x_no"],
    )

    return (
        f"Legend: {symbol_map['RB']} = rule-based/conventional; "
        f"{symbol_map['OPT']} = smart; "
        f"{symbol_map['NO']} = absent"
    )





# Function to define which symbol is associated with each technology in the comparative-chart table
def _scenario_control_symbol(row, tech):
    """
    Returns the print/plotter symbol from the mini-table for a technology.
    First, use the control_BESS/control_BEV/control_DHW columns.
    If these are not present, try reconstructing it from the label.

    Important note:
      - ESWH is only the name shown in the table.
      - Internally ESWH has to read the column control_DHW.
    """
    tech = str(tech).upper().strip()

    tech_aliases = {
        "BEV": "EV",
        "ESWH": "DHW",
        "E-SWH": "DHW",
        "EL STORAGE WATER HEATER": "DHW",
        "ELECTRIC STORAGE WATER HEATER": "DHW",
    }
    tech = tech_aliases.get(tech, tech)

    col_map = {
        "BESS": "control_BESS",
        "EV": "control_BEV",
        "DHW": "control_DHW",
    }

    col = col_map[tech]
    value = row.get(col, "")

    if _is_missing_like(value):
        value = _extract_control_from_label(row.get("label", ""), tech)

    return _normalize_control_for_table(value)


# Function that links compact name to the observed configuration 
def _scenario_display_label_for_plot(row, i_zero_based):
    """
    Label breve per colonne scenario:
      Base case, Case 1, Case 2, ...
    """
    value = row.get("scenario_display_label", "")

    if not _is_missing_like(value):
        return str(value)

    return "Base case" if i_zero_based == 0 else f"Case {i_zero_based}"


# Function to add a table below the comparative-configuration figure axis, identifying which technologies are available
def _add_scenario_config_table_under_axis(
    ax,
    d,
    include_first_column=False,
    first_column_label="",
    table_y=None,
    table_height=None,
    font_size=None,
    symbol_font_size=None,
    header_font_size=None,
    note_font_size=None,
    add_legend_note=None,
    show_column_labels=False,
):
    """
    Adds a BESS / EV / ESWH control strip below the x-axis.

    Paper version:
    - no column headers, as the scenarios are already labelled on the x-axis;
    - a smaller, less prominent font;
    - legible symbols, but not excessive;
    - symbols are optional, but their inclusion in the caption is recommended.
    """
    d = d.reset_index(drop=True).copy()

    if font_size is None:
        font_size = globals().get("JOURNAL_TABLE_FONT_SIZE", 6.9)

    if symbol_font_size is None:
        symbol_font_size = globals().get("JOURNAL_TABLE_SYMBOL_FONT_SIZE", 7.8)

    if header_font_size is None:
        header_font_size = globals().get("JOURNAL_TABLE_HEADER_FONT_SIZE", font_size)

    if note_font_size is None:
        note_font_size = globals().get("JOURNAL_TABLE_NOTE_FONT_SIZE", font_size)

    if add_legend_note is None:
        add_legend_note = globals().get("MULTIRUN_TABLE_SHOW_SYMBOL_NOTE", False)

    row_labels = ["BESS", "EV", "ESWH"]

    cell_text = []

    for tech in row_labels:
        row_values = [
            _scenario_control_symbol(row, tech)
            for _, row in d.iterrows()
        ]

        if include_first_column:
            row_values = [""] + row_values

        cell_text.append(row_values)

    # If the header is still desired, create it. For the paper, I recommend False.
    if show_column_labels:
        col_labels = [
            _scenario_display_label_for_plot(row, i).replace(" ", "\n")
            for i, (_, row) in enumerate(d.iterrows())
        ]

        if include_first_column:
            if first_column_label in {"Demand\ncomponents", "Demand components"}:
                first_column_label = "Demand\ncomp."
            col_labels = [first_column_label] + col_labels
    else:
        col_labels = None

    if table_height is None:
        table_height = 0.205 if not show_column_labels else 0.305

    if table_y is None:
        table_y = -(table_height + 0.125)

    n_cols = len(cell_text[0])
    col_widths = [1.0 / n_cols] * n_cols

    table = ax.table(
        cellText=cell_text,
        rowLabels=row_labels,
        colLabels=col_labels,
        cellLoc="center",
        rowLoc="center",
        loc="bottom",
        bbox=[0.0, table_y, 1.0, table_height],
        colWidths=col_widths,
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)

    for (r, c), cell in table.get_celld().items():
        cell.PAD = 0.020
        cell.set_edgecolor("0.68")
        cell.set_linewidth(0.45)

        # Column header, if available
        if show_column_labels and r == 0:
            cell.set_facecolor("0.93")
            cell.set_text_props(
                weight="bold",
                fontsize=header_font_size,
                va="center",
                ha="center",
            )

        # Label rows
        if c == -1:
            cell.set_facecolor("0.93")
            cell.set_text_props(
                weight="bold",
                fontsize=header_font_size,
                va="center",
                ha="center",
            )

        # Cells with symbols
        if c >= 0 and ((show_column_labels and r > 0) or ((not show_column_labels) and r >= 0)):
            txt = cell.get_text().get_text()

            if txt in {"◆", "◇", "–"}:
                cell.set_text_props(
                    fontsize=symbol_font_size,
                    weight="bold",
                    va="center",
                    ha="center",
                )
            else:
                cell.set_text_props(
                    fontsize=font_size,
                    weight="normal",
                    va="center",
                    ha="center",
                )

    if add_legend_note:
        ax.text(
            0.5,
            table_y - 0.045,
            _control_legend_note(),
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=note_font_size,
        )

    return table


# Function to better build the multi-configuration table and chart
def _multirun_figure_layout(n_columns, plot_type="cost"):
    """
    Layout per figure journal two-column.

    Cost:
    - legenda dentro;
    - area chart more larga;
    - tabella compatta.

    Demand:
    - legenda fuori a destra;
    - area barre ancora leggibile;
    - control strip compatta sotto.
    """
    n_columns = int(n_columns)

    if plot_type == "cost":
        figsize = globals().get(
            "JOURNAL_TWO_COLUMN_FIGSIZE_COST",
            (7.20, 4.95),
        )

        subplots = dict(
            left=0.115,
            right=0.885,
            top=0.92 if not globals().get("SHOW_MULTIRUN_FIGURE_TITLE", False) else 0.84,
            bottom=0.34,
        )

    elif plot_type == "demand":
        figsize = globals().get(
            "JOURNAL_TWO_COLUMN_FIGSIZE_DEMAND",
            (7.80, 5.00),
        )

        subplots = dict(
            left=0.095,
            right=0.70,
            top=0.92 if not globals().get("SHOW_MULTIRUN_FIGURE_TITLE", False) else 0.84,
            bottom=0.34,
        )

    else:
        figsize = (
            max(7.20, 0.75 * n_columns + 2.5),
            4.100,
        )

        subplots = dict(
            left=0.10,
            right=0.88,
            top=0.92,
            bottom=0.30,
        )

    return figsize, subplots


# Function to add the legend intelligently inside the comparative chart across configurations
def _add_smart_inside_legend(
    ax,
    handles,
    labels,
    title=None,
    fontsize=None,
    title_fontsize=None,
    ncol=None,
    legend_mode=None,
):
    """
    Aggiunge una legenda ai charts multi-run.

    Modalita principali:
    - smart_inside
    - upper_left_inside
    - upper_right_inside
    - right_outside: consigliata per figure two-column / journal
    """

    if legend_mode is None:
        legend_mode = globals().get("MULTIRUN_LEGEND_MODE", "smart_inside")

    if ncol is None:
        ncol = 2 if len(labels) > 6 else 1

    if fontsize is None:
        fontsize = globals().get("JOURNAL_LEGEND_FONT_SIZE", 8.5)

    if title_fontsize is None:
        title_fontsize = globals().get("JOURNAL_LEGEND_TITLE_SIZE", 9.2)

    legend_kwargs = dict(
        handles=handles,
        labels=labels,
        title=title,
        frameon=True,
        fontsize=fontsize,
        title_fontsize=title_fontsize,
        borderaxespad=0.0,
        labelspacing=0.22,
        handlelength=1.05,
        handletextpad=0.35,
        columnspacing=0.6,
        ncol=ncol,
    )

    if legend_mode == "right_outside":
        leg = ax.legend(
            loc="center left",
            bbox_to_anchor=(1.16, 0.50),
            bbox_transform=ax.transAxes,
            **legend_kwargs,
        )

    elif legend_mode == "smart_inside":
        leg = ax.legend(
            loc="best",
            bbox_to_anchor=(0.01, 0.03, 0.98, 0.86),
            bbox_transform=ax.transAxes,
            **legend_kwargs,
        )

    elif legend_mode == "upper_left_inside":
        leg = ax.legend(
            loc="upper left",
            bbox_to_anchor=(0.015, 0.985),
            **legend_kwargs,
        )

    elif legend_mode == "upper_right_inside":
        leg = ax.legend(
            loc="upper right",
            bbox_to_anchor=(0.995, 0.995),
            **legend_kwargs,
        )

    else:
        leg = ax.legend(
            loc="best",
            **legend_kwargs,
        )

    leg.get_frame().set_alpha(0.92)
    leg.get_frame().set_linewidth(0.7)

    return leg

# Function to concatenare values numerici
def _finite_numeric_values(*items):
    """
    Concatena values numerici finiti da Series/DataFrame/array/list/scalari.
    """
    chunks = []

    for item in items:
        if item is None:
            continue

        if isinstance(item, pd.DataFrame):
            vals = item.to_numpy().ravel()
        elif isinstance(item, (pd.Series, pd.Index)):
            vals = item.to_numpy()
        elif isinstance(item, (list, tuple, np.ndarray)):
            vals = np.asarray(item).ravel()
        else:
            vals = np.asarray([item]).ravel()

        s = pd.to_numeric(pd.Series(vals), errors="coerce")
        s = s.replace([np.inf, -np.inf], np.nan).dropna()

        if not s.empty:
            chunks.append(s.astype(float).values)

    if not chunks:
        return np.array([], dtype=float)

    return np.concatenate(chunks).astype(float)

# Function to define axis limits for comparative-configuration charts across seasons
def _expanded_ylim_from_values(values, pad_frac=0.08, floor_zero="auto"):
    """
    Returns robust Y limits.

    floor_zero:
      - True   -> minimum forced to 0
      - False  -> unrestricted minimum
      - ‘auto’ -> minimum set to 0 only if all values are >= 0
    """
    arr = _finite_numeric_values(values)

    if arr.size == 0:
        return None

    raw_min = float(np.nanmin(arr))
    raw_max = float(np.nanmax(arr))

    if floor_zero == "auto":
        force_zero = raw_min >= 0.0
    else:
        force_zero = bool(floor_zero)

    ymin = 0.0 if force_zero else raw_min
    ymax = raw_max

    span = ymax - ymin

    if not np.isfinite(span) or span <= 0:
        span = max(abs(ymax), abs(ymin), 1.0) * 0.10

    pad = span * float(pad_frac)

    ylo = 0.0 if force_zero else ymin - pad
    yhi = ymax + pad

    if yhi <= ylo:
        yhi = ylo + max(abs(ylo), 1.0) * 0.10

    return (float(ylo), float(yhi))


# Function to rebuild final demand across seasons
def _multirun_demand_total_series_for_limits(df_plot):
    """
    Ribuilds la demand totale da usare per calcolare limits comuni
    del chart demand coverage.
    """
    if df_plot is None or df_plot.empty:
        return pd.Series(dtype=float)

    d = df_plot.copy()

    if "demand_bev_final_kwh" not in d.columns:
        if "demand_ev_final_kwh" in d.columns:
            d["demand_bev_final_kwh"] = d["demand_ev_final_kwh"]
        else:
            d["demand_bev_final_kwh"] = d.get("demand_ev_charging_kwh", np.nan)

    if "demand_dhw_final_kwh" not in d.columns:
        d["demand_dhw_final_kwh"] = d.get("demand_dhw_total_kwh", np.nan)

    demand_cols = [
        "demand_base_kwh",
        "demand_cooling_kwh",
        "demand_heating_kwh",
        "demand_cooking_kwh",
        "demand_bev_final_kwh",
        "demand_dhw_final_kwh",
    ]

    for col in demand_cols:
        if col not in d.columns:
            d[col] = 0.0

        d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0.0)

    return d[demand_cols].sum(axis=1)


# Function to calculate the average weekly system cost per household, total system cost divided by number of households
def _multirun_average_household_weekly_system_cost_series(df):
    """
Returns a Series [EUR/household/week] for use in the cost_percentiles chart.

    It prefers the column already calculated in the multirun dataset. If it does not exist
    (e.g. cache created with previous versions), it rebuilds it from:
        total_system_cost_eur / n_households_for_avg_cost
    with a fallback to n_house_costs and then to the global n_households.
    """
    if df is None or df.empty:
        return pd.Series(dtype=float)

    if "avg_household_weekly_system_cost_eur" in df.columns:
        return pd.to_numeric(
            df["avg_household_weekly_system_cost_eur"],
            errors="coerce",
        )

    if "total_system_cost_eur" not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)

    total = pd.to_numeric(df["total_system_cost_eur"], errors="coerce")

    if "n_households_for_avg_cost" in df.columns:
        denom = pd.to_numeric(df["n_households_for_avg_cost"], errors="coerce")
    elif "n_house_costs" in df.columns:
        denom = pd.to_numeric(df["n_house_costs"], errors="coerce")
    else:
        denom = pd.Series(
            float(globals().get("n_nuclei_familiari", np.nan)),
            index=df.index,
            dtype=float,
        )

    denom = denom.where(denom > 0.0, np.nan)
    return total / denom



# Function to define common vertical-axis limits for graph cost delta at household level between different setups
def _compute_multirun_delta_common_ylim(
    df_delta_by_season,
    symmetric=True,
):
    """
    Returns common Y limits for household paired-delta charts.
    """
    if isinstance(df_delta_by_season, dict):
        dataframes = list(df_delta_by_season.values())
    else:
        dataframes = list(df_delta_by_season)

    percentile_cols = [
        "p10_delta_eur",
        "p20_delta_eur",
        "p30_delta_eur",
        "p40_delta_eur",
        "p50_delta_eur",
        "p60_delta_eur",
        "p70_delta_eur",
        "p80_delta_eur",
        "p90_delta_eur",
    ]

    values = []

    for df in dataframes:
        if df is None or df.empty:
            continue

        existing_cols = [
            col for col in percentile_cols
            if col in df.columns
        ]

        if not existing_cols:
            continue

        array = (
            df[existing_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=float)
            .ravel()
        )

        array = array[np.isfinite(array)]

        if array.size > 0:
            values.extend(array.tolist())

    if len(values) == 0:
        return None

    values = np.asarray(values, dtype=float)

    if symmetric:
        max_abs = float(np.nanmax(np.abs(values)))

        if max_abs <= 1e-12:
            max_abs = 1.0

        limit = max_abs * 1.10
        return (-limit, limit)

    low = float(np.nanmin(values))
    high = float(np.nanmax(values))

    span = high - low

    if span <= 1e-12:
        span = max(1.0, abs(high))

    return (
        low - 0.10 * span,
        high + 0.10 * span,
    )









# Function to calculate common vertical-axis limits across seasons in comparative charts
def _compute_multirun_common_y_limits(df_plot_by_season):
    """
    Calcola limits Y comuni tra seasons per entrambi i charts 4.13.

    Output:
      - cost_y1_ylim: percentili cost famiglie
      - cost_y2_ylim: average system cost
      - demand_y1_ylim: energy demand coverage
      - demand_y2_ylim: total system cost
    """
    if isinstance(df_plot_by_season, dict):
        dfs = list(df_plot_by_season.values())
    else:
        dfs = list(df_plot_by_season)

    dfs = [
        df for df in dfs
        if df is not None and not df.empty
    ]

    if not dfs:
        return {
            "cost_y1_ylim": None,
            "cost_y2_ylim": None,
            "demand_y1_ylim": None,
            "demand_y2_ylim": None,
        }

    # -----------------------------
    # Cost percentile chart
    # -----------------------------
    percentile_cols = [
        "p10_eur", "p20_eur", "p30_eur", "p40_eur", "p50_eur",
        "p60_eur", "p70_eur", "p80_eur", "p90_eur",
    ]

    cost_y1_values = []
    cost_y2_values = []

    demand_y1_values = []
    demand_y2_values = []

    for df in dfs:
        for col in percentile_cols:
            if col in df.columns:
                cost_y1_values.append(df[col])

        avg_household_cost = _multirun_average_household_weekly_system_cost_series(df)
        if not avg_household_cost.empty:
            cost_y2_values.append(avg_household_cost)
            # Same unit as household percentiles: Include average household costs in the primary-axis limits for a consistent comparison with the median
            cost_y1_values.append(avg_household_cost)

        demand_y1_values.append(_multirun_demand_total_series_for_limits(df))

        if "total_system_cost_eur" in df.columns:
            demand_y2_values.append(df["total_system_cost_eur"])

    return {
        # Primary axis of percentile chart, common minimum and maximum across seasons without forcing the minimum to zero
        "cost_y1_ylim": _expanded_ylim_from_values(
            cost_y1_values,
            pad_frac=0.08,
            floor_zero=globals().get(
                "MULTIRUN_COST_PERCENTILE_YLIM_FLOOR_ZERO",
                False,
            ),
        ),

        # Secondary axis of percentile chart:
        # average weekly cost per household [EUR/household/week].
        # It has the same unit as the primary axis; therefore use the same limits
        # of the percentile chart, which also includes average points per household.
        "cost_y2_ylim": _expanded_ylim_from_values(
            cost_y1_values,
            pad_frac=0.08,
            floor_zero=globals().get(
                "MULTIRUN_COST_PERCENTILE_YLIM_FLOOR_ZERO",
                False,
            ),
        ),

        # Grafico demand coverage: energy always from zero.
        "demand_y1_ylim": _expanded_ylim_from_values(
            demand_y1_values,
            pad_frac=0.10,
            floor_zero=globals().get(
                "MULTIRUN_DEMAND_ENERGY_YLIM_FLOOR_ZERO",
                True,
            ),
        ),

        # Total system cost: it may be negative.
        "demand_y2_ylim": _expanded_ylim_from_values(
            demand_y2_values,
            pad_frac=0.08,
            floor_zero=globals().get(
                "MULTIRUN_TOTAL_SYSTEM_COST_YLIM_FLOOR_ZERO",
                False,
            ),
        ),
    }





# Function to print/plot the comparison chart across different setups, household and total costs
def plot_multirun_cost_percentiles_vs_household_avg_cost(
    df_plot,
    title,
    out_png=None,
    y1_label="Weekly household cost [EUR]",
    y2_label="Mean weekly system cost per household [EUR/household]",
    rotate_xticks=0,
    y1_ylim=None,
    y2_ylim=None,
    legend_mode=None,
    show_title=None,
    save_pdf=None,
):
    """
    Journal-ready plot:
    - X: scenarios defined manually
    - primary Y-axis: household weekly cost percentile bands
    - secondary Y-axis: mean weekly system cost per household [EUR/household]
    - optional title for journal publication
    - enlarged axes, legend, table symbols
    - PNG 600 dpi + optional PDF
    """

    # ------------------------------------------------------------
    # 0) Check available data
    # ------------------------------------------------------------
    if df_plot is None or df_plot.empty:
        print("[WARN] empty df_plot: skip chart.")
        return

    if show_title is None:
        show_title = globals().get("SHOW_MULTIRUN_FIGURE_TITLE", False)

    if save_pdf is None:
        save_pdf = globals().get("SAVE_JOURNAL_PDF", True)

    # ------------------------------------------------------------
    # 1) Journal chart parameters
    # ------------------------------------------------------------
    font_size = globals().get("JOURNAL_FONT_SIZE", 9.5)
    axis_label_size = globals().get("JOURNAL_AXIS_LABEL_SIZE", 10.5)
    tick_label_size = globals().get("JOURNAL_TICK_LABEL_SIZE", 9.0)
    legend_font_size = globals().get("JOURNAL_LEGEND_FONT_SIZE", 8.5)
    legend_title_size = globals().get("JOURNAL_LEGEND_TITLE_SIZE", 9.2)
    title_size = globals().get("JOURNAL_TITLE_SIZE", 11.0)

    plt.rcParams.update({
        "font.size": font_size,
        "axes.labelsize": axis_label_size,
        "xtick.labelsize": tick_label_size,
        "ytick.labelsize": tick_label_size,
        "legend.fontsize": legend_font_size,
        "axes.titlesize": title_size,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

    # ------------------------------------------------------------
    # 2) Preparazione dataframe
    # ------------------------------------------------------------
    d = df_plot.sort_values("x_order").reset_index(drop=True).copy()

    required_cols = [
        "p10_eur", "p20_eur", "p30_eur", "p40_eur", "p50_eur",
        "p60_eur", "p70_eur", "p80_eur", "p90_eur",
        "total_system_cost_eur",
        "scenario_group",
    ]

    missing = [c for c in required_cols if c not in d.columns]

    if missing:
        raise KeyError(
            f"Missing columns for the updated cost chart: {missing}. "
            "Check to have updated build_multirun_cost_percentile_summary."
        )

    # Ensure the new KPI is present even when using a previously produced cache
    # even when using a cache produced by an earlier notebook version..
    d["avg_household_weekly_system_cost_eur"] = (
        _multirun_average_household_weekly_system_cost_series(d)
    )


    x = np.arange(len(d), dtype=float)

    # ------------------------------------------------------------
    # 3) Colors by scenario group
    # ------------------------------------------------------------
    group_base_colors = {
        "no_flex":   "#2ca02c",
        "only_DHW":  "#f2c94c",
        "only_BESS": "#1f77b4",
        "only_EV":   "#d62728",
        "all_flex":  "#7f7f7f",
        "other":     "#9467bd",
    }

    # ------------------------------------------------------------
    # 4) Plot creation
    # ------------------------------------------------------------
    figsize, subplot_margins = _multirun_figure_layout(
        n_columns=len(d),
        plot_type="cost",
    )

    fig, ax1 = plt.subplots(
        figsize=figsize,
        constrained_layout=False,
    )

    ax2 = ax1.twinx()

    ax1.tick_params(
        axis="both",
        labelsize=tick_label_size,
    )

    ax2.tick_params(
        axis="both",
        labelsize=tick_label_size,
    )

    # ------------------------------------------------------------
    # 5) Percentile bands
    # ------------------------------------------------------------
    band_width = 0.64

    band_specs = [
        ("p10_eur", "p90_eur", 0.78, "10-90%"),
        ("p20_eur", "p80_eur", 0.62, "20-80%"),
        ("p30_eur", "p70_eur", 0.46, "30-70%"),
        ("p40_eur", "p60_eur", 0.30, "40-60%"),
    ]

    for j, row in d.iterrows():
        group = str(row.get("scenario_group", "other"))
        base_color = group_base_colors.get(
            group,
            group_base_colors["other"],
        )

        for lo_col, hi_col, lighten_amount, band_label in band_specs:
            band_color = _lighten_color(
                base_color,
                amount=lighten_amount,
            )

            ax1.fill_between(
                [x[j] - band_width / 2, x[j] + band_width / 2],
                [row[lo_col], row[lo_col]],
                [row[hi_col], row[hi_col]],
                color=band_color,
                edgecolor=base_color,
                linewidth=0.45,
                label=None,
                zorder=2,
            )

        # Mediana p50
        ax1.plot(
            [x[j] - band_width / 2, x[j] + band_width / 2],
            [row["p50_eur"], row["p50_eur"]],
            color="black",
            linewidth=2.1,
            solid_capstyle="butt",
            zorder=5,
        )

    # ------------------------------------------------------------
    # 6) Average weekly cost per household on the secondary axis
    # ------------------------------------------------------------
    avg_household_weekly_system_cost = pd.to_numeric(
        d["avg_household_weekly_system_cost_eur"],
        errors="coerce",
    ).values

    ax2.plot(
        x,
        avg_household_weekly_system_cost,
        marker="o",
        markersize=globals().get("MULTIRUN_SYSTEM_COST_MARKER_SIZE", 4.9),
        linewidth=0.0,
        color="black",
        linestyle="None",
        label="Mean weekly system cost per household [EUR/household]",
        zorder=20,
    )

    # ------------------------------------------------------------
    # 7) X-axis: no text labels because there is a table below
    # ------------------------------------------------------------
    scenario_xticklabels = [
        _scenario_display_label_for_plot(row, i).replace(" ", "\n")
        for i, (_, row) in enumerate(d.iterrows())
    ]

    ax1.set_xticks(x)
    ax1.set_xticklabels(
        scenario_xticklabels,
        rotation=0,
        ha="center",
        fontsize=globals().get("JOURNAL_TICK_LABEL_SIZE", 8.5),
    )
    ax1.tick_params(
        axis="x",
        which="both",
        length=0,
        pad=3,
    )

    ax1.set_xlim(-0.75, len(d) - 0.25)

    # ------------------------------------------------------------
    # 8) Configuration table below X-axis
    # ------------------------------------------------------------
    _add_scenario_config_table_under_axis(
        ax=ax1,
        d=d,
        include_first_column=False,
        table_y=None,
        table_height=None,
        font_size=globals().get("JOURNAL_TABLE_FONT_SIZE", 6.9),
        symbol_font_size=globals().get("JOURNAL_TABLE_SYMBOL_FONT_SIZE", 7.8),
        header_font_size=globals().get("JOURNAL_TABLE_HEADER_FONT_SIZE", 6.9),
        note_font_size=globals().get("JOURNAL_TABLE_NOTE_FONT_SIZE", 6.9),
        add_legend_note=False,
        show_column_labels=False,
    )

    # ------------------------------------------------------------
    # 9) Assi, titolo opzionale, griglia
    # ------------------------------------------------------------
    ax1.set_ylabel(
        y1_label,
        fontsize=axis_label_size,
    )

    ax2.set_ylabel(
        y2_label,
        fontsize=axis_label_size,
    )

    if show_title:
        ax1.set_title(
            title,
            pad=8,
            fontsize=title_size,
        )

    ax1.grid(
        axis="y",
        linestyle=":",
        linewidth=0.7,
        alpha=0.42,
    )

    ax1.grid(
        axis="x",
        visible=False,
    )

    ax1.set_axisbelow(True)

    # ------------------------------------------------------------
    # 10) Common axis limits across seasons, if passed from the first pass
    # ------------------------------------------------------------
    # The secondary axis is average household cost in EUR/household/week, so it has the
    # same unit as the percentiles on the primary axis. For a consistent visual comparison with the median,
    # the two cost axes are always aligned.
    if y1_ylim is not None:
        ax1.set_ylim(*y1_ylim)

    if y2_ylim is not None:
        ax2.set_ylim(*y2_ylim)
    else:
        common_cost_ylim = _expanded_ylim_from_values(
            [d[col] for col in percentile_cols] + [d["avg_household_weekly_system_cost_eur"]],
            pad_frac=0.08,
            floor_zero=globals().get(
                "MULTIRUN_COST_PERCENTILE_YLIM_FLOOR_ZERO",
                False,
            ),
        )
        if common_cost_ylim is not None:
            ax1.set_ylim(*common_cost_ylim)
            ax2.set_ylim(*common_cost_ylim)

    # Keep both axes aligned: both use EUR/household/week. The first pass includes average household cost points in the y-axis limits.
    if y1_ylim is not None:
        ax2.set_ylim(ax1.get_ylim())
    elif y2_ylim is not None:
        ax1.set_ylim(ax2.get_ylim())

    # If household costs cross zero, highlight it.
    y1_lo, y1_hi = ax1.get_ylim()

    if y1_lo < 0.0 < y1_hi:
        ax1.axhline(
            0.0,
            color="black",
            linewidth=0.8,
            alpha=0.45,
            zorder=0,
        )

    # ------------------------------------------------------------
    # 11) Legenda
    # ------------------------------------------------------------
    band_handles = [
        Patch(facecolor="grey", alpha=0.20, label="p10-p90"),
        Patch(facecolor="grey", alpha=0.35, label="p20-p80"),
        Patch(facecolor="grey", alpha=0.50, label="p30-p70"),
        Patch(facecolor="grey", alpha=0.65, label="p40-p60"),
        Line2D([0], [0], color="black", lw=2.0, label="Median"),
        Line2D(
            [0], [0],
            color="black",
            lw=0,
            linestyle="None",
            marker="o",
            markersize=3.8,
            label="Mean / HH",
        ),
    ]

    _add_smart_inside_legend(
        ax=ax1,
        handles=band_handles,
        labels=[h.get_label() for h in band_handles],
        title="Bands / points",
        fontsize=globals().get("JOURNAL_LEGEND_FONT_SIZE", 7.4),
        title_fontsize=globals().get("JOURNAL_LEGEND_TITLE_SIZE", 7.8),
        ncol=2,
        legend_mode=globals().get("MULTIRUN_COST_LEGEND_MODE", "upper_right_inside"),
    )

    # ------------------------------------------------------------
    # 12) Margini e savetaggio
    # ------------------------------------------------------------
    fig.subplots_adjust(**subplot_margins)

    if out_png:
        fig.savefig(
            out_png,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.04,
        )

        if save_pdf:
            out_pdf = os.path.splitext(out_png)[0] + ".pdf"

            fig.savefig(
                out_pdf,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        print(f"[OK] Saved multi-run chart of average household costs: {out_png}")

    else:
        plt.show()

    plt.close(fig)



# Function to create internal self-sufficiency graph
def plot_multirun_self_sufficiency_and_demand_components(
    df_plot,
    title=None,
    out_png=None,
    rotate_xticks=0,
    y1_label="Energy [kWh/week]",
    y2_label="Effective system operating cost [EUR/week]",
    demand_tolerance_abs_kwh=10,
    demand_tolerance_rel=5e-4,
    demand_mismatch_action="warn",
    demand_reference_if_within_tolerance="max",
    print_checks=True,
    y1_ylim=None,
    y2_ylim=None,
    legend_mode=None,
    show_title=None,
    save_pdf=None,
):
    """
    Grafico demand coverage
    - uso il bilancio netto over the whole period osservato:

        net_mv_import_period_kwh =
            mv_import_total_kwh - mv_export_total_kwh

    - verifico che la demand finale totale sia uguale in tutti gli scenari.

    - definisco una demand finale comune:

        total_final_demand_reference_kwh

    - per ogni scenario calcolo:

        net_mv_import_plot_kwh =
            clip(net_mv_import_period_kwh, 0, total_final_demand_reference_kwh)

        internal_self_sufficiency_plot_kwh =
            total_final_demand_reference_kwh - net_mv_import_plot_kwh

    In questo modo ogni colonna scenario chiude esattamente sulla stessa
    demand finale comune.
    """

    if df_plot is None or df_plot.empty:
        print("[WARN] empty df_plot: skip chart demand coverage/components.")
        return
    if show_title is None:
        show_title = globals().get("SHOW_MULTIRUN_FIGURE_TITLE", False)

    if save_pdf is None:
        save_pdf = globals().get("SAVE_JOURNAL_PDF", True)
    d = df_plot.sort_values("x_order").reset_index(drop=True).copy()

    # ------------------------------------------------------------
    # Robust aliases for final BEV and DHW
    # ------------------------------------------------------------
    # Prefer explicit names, but keeping fallbacks to the old names.
    # Note: ‘demand_ev_charging_kwh’ here should be interpreted solely as
    # a backward-compatible alias for final BEV demand, not as a charge flow.
    if "demand_bev_final_kwh" not in d.columns:
        if "demand_ev_final_kwh" in d.columns:
            d["demand_bev_final_kwh"] = d["demand_ev_final_kwh"]
        else:
            d["demand_bev_final_kwh"] = d.get("demand_ev_charging_kwh", np.nan)

    if "demand_dhw_final_kwh" not in d.columns:
        d["demand_dhw_final_kwh"] = d.get("demand_dhw_total_kwh", np.nan)

    # ------------------------------------------------------------
    # Period net MV import
    # ------------------------------------------------------------
    # Preferisco net_mv_import_kwh_raw, perché is:
    #     mv_import_total_kwh - mv_export_total_kwh
    # over the whole period.
    if "net_mv_import_kwh_raw" not in d.columns:
        if ("mv_import_total_kwh" in d.columns) and ("mv_export_total_kwh" in d.columns):
            d["net_mv_import_kwh_raw"] = (
                pd.to_numeric(d["mv_import_total_kwh"], errors="coerce")
                - pd.to_numeric(d["mv_export_total_kwh"], errors="coerce")
            )
        elif "net_mv_import_for_demand_kwh" in d.columns:
            d["net_mv_import_kwh_raw"] = d["net_mv_import_for_demand_kwh"]
        elif "mt_import_total_kwh" in d.columns:
            d["net_mv_import_kwh_raw"] = d["mt_import_total_kwh"]
        else:
            raise KeyError(
                "Impossible to rebuild net_mv_import_kwh_raw. "
                "Either net_mv_import_kwh_raw or "
                "mv_import_total_kwh/mv_export_total_kwh is required."
            )

    component_color_map = globals().get(
        "STACKED_CONSUMPTION_COLOR_MAP",
        {
            "Conventional": "#B3B3B3",
            "DHW": "#0072B2",
            "Cooking": "#E69F00",
            "Cooling": "#009E73",
            "EV": "#D55E00",
            "Heating": "#7E57C2",
        },
    )

    component_specs = [
        ("demand_base_kwh", "Conventional", component_color_map["Conventional"]),
        ("demand_dhw_final_kwh", "DHW", component_color_map["DHW"]),                # DHW final demand (electricity to be provided to cover heat dmd)
        ("demand_cooking_kwh", "Cooking", component_color_map["Cooking"]),
        ("demand_cooling_kwh", "Cooling", component_color_map["Cooling"]),
        ("demand_bev_final_kwh", "EV", component_color_map["EV"]),                  # EV final demand (electricity to be provided to cover transport demand)
        ("demand_heating_kwh", "Heating", component_color_map["Heating"]),
    ]

    required_cols = [
        "label",
        "season",
        "net_mv_import_kwh_raw",
        "total_system_cost_eur",
    ] + [c for c, _, _ in component_specs]

    missing = [c for c in required_cols if c not in d.columns]
    if missing:
        raise KeyError(
            f"Colonne missing per chart demand coverage/components: {missing}. "
            "Controlla build_multirun_cost_percentile_summary e fallback pickle."
        )

    numeric_cols = [
        "net_mv_import_kwh_raw",
        "total_system_cost_eur",
    ] + [c for c, _, _ in component_specs]

    for col in numeric_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")

    # ------------------------------------------------------------
    # Total final demand scenario by scenario
    # ------------------------------------------------------------
    demand_cols = [c for c, _, _ in component_specs]

    d["_demand_total_from_components_kwh"] = (
        d[demand_cols].fillna(0.0).sum(axis=1)
    )

    demand_values = (
        d["_demand_total_from_components_kwh"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if demand_values.empty:
        raise ValueError(
            "Final demand cannot be calculated: all rows have NaN demand."
        )

    demand_min = float(demand_values.min())
    demand_max = float(demand_values.max())
    demand_mean = float(demand_values.mean())
    demand_spread_abs = demand_max - demand_min
    demand_spread_rel = demand_spread_abs / max(abs(demand_mean), 1e-12)

    demand_ok = (
        demand_spread_abs <= demand_tolerance_abs_kwh
        or demand_spread_rel <= demand_tolerance_rel
    )

    demand_mismatch_action = str(demand_mismatch_action).lower().strip()

    if demand_mismatch_action not in {"raise", "warn", "ignore"}:
        raise ValueError(
            "demand_mismatch_action must have to be 'raise', 'warn' or 'ignore'."
        )

    if not demand_ok:
        check_cols = [
            "label",
            "energy_breakdown_source",
            "_demand_total_from_components_kwh",
            "demand_base_kwh",
            "demand_cooling_kwh",
            "demand_heating_kwh",
            "demand_cooking_kwh",
            "demand_bev_final_kwh",
            "demand_dhw_final_kwh",
            "n_cols_demand_ev_matched",
            "n_cols_demand_dhw_matched",

            "control_BEV",
            "control_DHW",
            "control_BEV_used_for_energy",
            "control_DHW_used_for_energy",
            "demand_ev_source",
            "demand_dhw_source",
        ]

        check_cols = [c for c in check_cols if c in d.columns]

        if demand_mismatch_action != "ignore":
            print("\n[ERROR][DEMAND CONSISTENCY] La demand finale NON is uguale tra gli scenari.")
            print(f"  min  = {demand_min:.6f} kWh")
            print(f"  max  = {demand_max:.6f} kWh")
            print(f"  diff = {demand_spread_abs:.6f} kWh")
            print(f"  rel  = {100.0 * demand_spread_rel:.6f} %")
            print("\nDettaglio demand per scenario:")
            print(d[check_cols].to_string(index=False))

        if demand_mismatch_action == "raise":
            raise ValueError(
                "Final demand differs across scenarios. "
                "Chart demand coverage not generated because the comparison "
                "would not be coherent. Check EV/ESWH final demand "
                "and the corresponding matches in pickle files."
            )

        elif demand_mismatch_action == "warn":
            print(
                "[WARN] Keep going by using average demand as a reference, but the plotted chart might be confusing."
            )

        elif demand_mismatch_action == "ignore":
            pass

    # If demand is equal within tolerance, or if proceeding was selected
    # When proceeding with a warning, construct a common reference demand.
    # For the demand-coverage chart, it is better to use "max":
    # - avoids underestimating common final demand
    if demand_reference_if_within_tolerance == "max":
        total_final_demand_reference_kwh = demand_max

    elif demand_reference_if_within_tolerance == "mean":
        total_final_demand_reference_kwh = demand_mean

    elif demand_reference_if_within_tolerance == "median":
        total_final_demand_reference_kwh = float(demand_values.median())

    else:
        raise ValueError(
            "demand_reference_if_within_tolerance  must have to be 'max', 'mean' or 'median'."
        )

    # Components of the single "Demand components" column.
    # Use the average per component; if inputs are consistent, it coincides
    # approximately matches the value of each scenario.
    demand_reference_by_component = {}

    for col, _, _ in component_specs:
        vals = (
            pd.to_numeric(d[col], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if vals.empty:
            demand_reference_by_component[col] = 0.0
        else:
            demand_reference_by_component[col] = float(vals.mean())

    # Realign the stacked-column total to the common demand.
    # Needed only to avoid micro-gap from approximations/round.
    component_sum = float(sum(demand_reference_by_component.values()))

    if component_sum > 0:
        scale = total_final_demand_reference_kwh / component_sum

        for col in demand_reference_by_component:
            demand_reference_by_component[col] *= scale


    # ------------------------------------------------------------
    # Net MV imports and internal self-sufficiency for plotting
    # ------------------------------------------------------------
    # Robust choice:
    # 1) if the columns already calculated by the builder exist, use them;
    # 2) otherwise reconstructed from the net MV import raw;
    # 3) if everything is zero, print a diagnostic warning.

    if (
        "net_mv_import_for_demand_kwh" in d.columns
        and "internal_self_sufficiency_kwh" in d.columns
    ):
        net_mv_import_plot = (
            pd.to_numeric(d["net_mv_import_for_demand_kwh"], errors="coerce")
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

        internal_self_plot = (
            pd.to_numeric(d["internal_self_sufficiency_kwh"], errors="coerce")
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

        net_mv_import_period = (
            pd.to_numeric(
                d.get("net_mv_import_kwh_raw", d["net_mv_import_for_demand_kwh"]),
                errors="coerce"
            )
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

    else:
        net_mv_import_period = (
            pd.to_numeric(d["net_mv_import_kwh_raw"], errors="coerce")
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

        net_mv_import_plot = np.minimum(
            np.maximum(net_mv_import_period, 0.0),
            total_final_demand_reference_kwh,
        )

        internal_self_plot = np.maximum(
            0.0,
            total_final_demand_reference_kwh - net_mv_import_plot,
        )

    # Numerical safeguards: prevent negative values or values exceeding total demand.
    net_mv_import_plot = np.minimum(
        np.maximum(net_mv_import_plot, 0.0),
        total_final_demand_reference_kwh,
    )

    internal_self_plot = np.minimum(
        np.maximum(internal_self_plot, 0.0),
        total_final_demand_reference_kwh,
    )

    # If rounding prevents the sum from closing exactly, realign it.
    coverage_total = net_mv_import_plot + internal_self_plot
    coverage_gap = total_final_demand_reference_kwh - coverage_total

    small_gap_mask = np.abs(coverage_gap) <= max(
        float(demand_tolerance_abs_kwh),
        float(demand_tolerance_rel) * float(total_final_demand_reference_kwh)
    )

    internal_self_plot = np.where(
        small_gap_mask,
        internal_self_plot + coverage_gap,
        internal_self_plot,
    )

    internal_self_plot = np.maximum(internal_self_plot, 0.0)

    d["_net_mv_import_period_kwh"] = net_mv_import_period
    d["_net_mv_import_plot_kwh"] = net_mv_import_plot
    d["_internal_self_sufficiency_plot_kwh"] = internal_self_plot

    d["_coverage_total_plot_kwh"] = (
        d["_net_mv_import_plot_kwh"]
        + d["_internal_self_sufficiency_plot_kwh"]
    )

    d["_coverage_gap_vs_reference_kwh"] = (
        d["_coverage_total_plot_kwh"]
        - total_final_demand_reference_kwh
    )

    if np.allclose(internal_self_plot, 0.0, atol=1e-6):
        print(
            "[WARN][DEMAND COVERAGE] Internal self-sufficiency is zero for all scenarios. "
            "Check energy_breakdown_source, net_mv_import_kwh_raw, "
            "net_mv_import_for_demand_kwh and total_final_demand_kwh in the file "
            "self_sufficiency_*.xlsx. If energy_breakdown_source='summary', "
            "rerun with MULTIRUN_FAST_GRAPH_ONLY=False and MULTIRUN_ENERGY_SOURCE_FOR_PLOT='pickle'."
        )

    # ------------------------------------------------------------
    # Flow consistency checks
    # ------------------------------------------------------------
    if print_checks:
        check_cols = [
            "label",
            "energy_breakdown_source",
            "_demand_total_from_components_kwh",
            "_net_mv_import_period_kwh",
            "_net_mv_import_plot_kwh",
            "_internal_self_sufficiency_plot_kwh",
            "_coverage_total_plot_kwh",
            "_coverage_gap_vs_reference_kwh",
            "mv_import_total_kwh",
            "mv_export_total_kwh",
            "net_mv_import_kwh_raw",
            "n_cols_mv_import_matched",
            "n_cols_mv_export_matched",
            "n_cols_demand_ev_matched",
            "n_cols_demand_dhw_matched",
        ]

        check_cols = [c for c in check_cols if c in d.columns]

        print("\n[CHECK][DEMAND CONSISTENCY]")
        print(f"Common final demand used in the plot = {total_final_demand_reference_kwh:.6f} kWh")
        print(f"Demand spread across scenarios = {demand_spread_abs:.6f} kWh")
        print(f"Relative demand spread = {100.0 * demand_spread_rel:.6f} %")

        print("\n[CHECK][DEMAND COVERAGE FLOWS]")
        print(d[check_cols].to_string(index=False))

        max_gap = float(np.nanmax(np.abs(d["_coverage_gap_vs_reference_kwh"].values)))

        if max_gap > max(demand_tolerance_abs_kwh, 1e-6):
            print(
                f"[WARN][DEMAND COVERAGE] Maximum coverage-demand gap = {max_gap:.6f} kWh."
            )

        if "n_cols_mv_import_matched" in d.columns:
            bad_import = d[pd.to_numeric(d["n_cols_mv_import_matched"], errors="coerce").fillna(0) <= 0]
            if not bad_import.empty:
                print(
                    "[WARN][MV IMPORT] In some configurations, the following were not matched "
                    "colonne di import MV."
                )
                print(bad_import[["label", "n_cols_mv_import_matched"]].to_string(index=False))

        if "n_cols_mv_export_matched" in d.columns:
            bad_export = d[pd.to_numeric(d["n_cols_mv_export_matched"], errors="coerce").fillna(0) <= 0]
            if not bad_export.empty:
                print(
                    "[INFO][MV EXPORT] In some configurations, the following were not matched "
                    "MV export columns. This is acceptable only if export is truly zero."
                )
                print(bad_export[["label", "n_cols_mv_export_matched"]].to_string(index=False))

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------
    x_demand = 0.0
    x_scenarios = np.arange(1, len(d) + 1, dtype=float)
    x_all = np.concatenate([[x_demand], x_scenarios])

    labels = ["Demand\ncomponents"] + [
        _scenario_display_label_for_plot(row, i).replace(" ", "\n")
        for i, (_, row) in enumerate(d.iterrows())
    ]

    figsize, subplot_margins = _multirun_figure_layout(
        n_columns=len(d) + 1,
        plot_type="demand",
    )

    fig, ax1 = plt.subplots(
        figsize=figsize,
        constrained_layout=False,
    )

    ax2 = ax1.twinx()

    ax1.tick_params(axis="both", labelsize=globals().get("JOURNAL_TICK_LABEL_SIZE", 9.0))
    ax2.tick_params(axis="both", labelsize=globals().get("JOURNAL_TICK_LABEL_SIZE", 9.0))

    bar_width = 0.62

    # ------------------------------------------------------------
    # Single column: final demand
    # ------------------------------------------------------------
    bottom_demand = 0.0

    for col, comp_label, color in component_specs:
        val = float(demand_reference_by_component[col])

        ax1.bar(
            x_demand,
            val,
            width=bar_width,
            bottom=bottom_demand,
            label=comp_label,
            color=color,
            edgecolor="black",
            linewidth=0.4,
        )

        bottom_demand += val

    # ------------------------------------------------------------
    # Scenario columns: net MV import + internal self sufficiency
    # ------------------------------------------------------------
    ax1.bar(
        x_scenarios,
        net_mv_import_plot,
        width=bar_width,
        label="Net MV import",
        color="#55524f",
        edgecolor="black",
        linewidth=0.4,
    )

    ax1.bar(
        x_scenarios,
        internal_self_plot,
        width=bar_width,
        bottom=net_mv_import_plot,
        label="Int. self sufficiency",
        color="#66c2a5",
        edgecolor="black",
        linewidth=0.4,
    )

    # ------------------------------------------------------------
    # Secondary axis: total system cost
    # ------------------------------------------------------------
    total_cost = pd.to_numeric(
        d["total_system_cost_eur"],
        errors="coerce",
    ).values

    ax2.plot(
        x_scenarios,
        total_cost,
        marker="o",
        markersize=globals().get("MULTIRUN_SYSTEM_COST_MARKER_SIZE", 4.9),
        linewidth=0.0,
        linestyle="None",
        color="black",
        label="System operating cost",
        zorder=20,
    )

    # ------------------------------------------------------------
    # Formattazione
    # ------------------------------------------------------------
    y1_max = max(
        float(total_final_demand_reference_kwh),
        float(np.nanmax(net_mv_import_plot + internal_self_plot)),
    )

    if y1_ylim is not None:
        ax1.set_ylim(*y1_ylim)
    elif y1_max > 0:
        ax1.set_ylim(0.0, y1_max * 1.15)

    if y2_ylim is not None:
        ax2.set_ylim(*y2_ylim)
    else:
        auto_y2_ylim = _expanded_ylim_from_values(
            total_cost,
            pad_frac=0.08,
            floor_zero="auto",
        )
        if auto_y2_ylim is not None:
            ax2.set_ylim(*auto_y2_ylim)



    ax1.axhline(
        total_final_demand_reference_kwh,
        color="0.25",
        linewidth=0.8,
        linestyle="--",
        alpha=0.65,
        zorder=1,
    )

    ax1.text(
        0.995,
        total_final_demand_reference_kwh,
        "Total demand",
        transform=ax1.get_yaxis_transform(),
        ha="right",
        va="bottom",
        fontsize=globals().get("JOURNAL_TABLE_NOTE_FONT_SIZE", 6.9),
        color="0.25",
    )






    scenario_xticklabels = [
        _scenario_display_label_for_plot(row, i).replace(" ", "\n")
        for i, (_, row) in enumerate(d.iterrows())
    ]

    x_all_labels = ["Demand\ncomp."] + scenario_xticklabels

    ax1.set_xticks(x_all)
    ax1.set_xticklabels(
        x_all_labels,
        rotation=0,
        ha="center",
        fontsize=globals().get("JOURNAL_TICK_LABEL_SIZE", 8.5),
    )
    ax1.tick_params(
        axis="x",
        which="both",
        length=0,
        pad=3,
    )

    _add_scenario_config_table_under_axis(
        ax=ax1,
        d=d,
        include_first_column=True,
        first_column_label="",
        table_y=None,
        table_height=None,
        font_size=globals().get("JOURNAL_TABLE_FONT_SIZE", 6.9),
        symbol_font_size=globals().get("JOURNAL_TABLE_SYMBOL_FONT_SIZE", 7.8),
        header_font_size=globals().get("JOURNAL_TABLE_HEADER_FONT_SIZE", 6.9),
        note_font_size=globals().get("JOURNAL_TABLE_NOTE_FONT_SIZE", 6.9),
        add_legend_note=False,
        show_column_labels=False,
    )

    ax1.set_ylabel(y1_label)
    ax2.set_ylabel(y2_label)

    if show_title:
        ax1.set_title(title, pad=8)
    ax1.grid(axis="y", linestyle=":", alpha=0.4)

    # Visual separator between the demand column and scenarios
    ax1.axvline(0.5, color="black", linewidth=0.8, alpha=0.35)

    # Legenda combinata
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()

    seen = set()
    handles = []
    labels_legend = []

    for h, l in zip(h1 + h2, l1 + l2):
        if l not in seen:
            handles.append(h)
            labels_legend.append(l)
            seen.add(l)

    _add_smart_inside_legend(
        ax=ax1,
        handles=handles,
        labels=labels_legend,
        title="Legend",
        fontsize=globals().get("JOURNAL_LEGEND_FONT_SIZE", 8.0),
        title_fontsize=globals().get("JOURNAL_LEGEND_TITLE_SIZE", 8.5),
        ncol=1,
        legend_mode=globals().get("MULTIRUN_DEMAND_LEGEND_MODE", "right_outside"),
    )

    fig.subplots_adjust(**subplot_margins)

    if out_png:
        fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.04)

        if save_pdf:
            out_pdf = os.path.splitext(out_png)[0] + ".pdf"
            fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.04)

        print(f"[OK] Saved demand coverage/components chart: {out_png}")

    plt.show()

### 4.6.14 F(x) Multi-run plotting graphs

In [ ]:
# ----------------------------------------------
#   Sec 4.6 --14) MULTI-RUN CACHE-SUMMARY FUNCTIONS
# ----------------------------------------------

# Function to create a dictionary with the configurations to compare.
# Here I define the function that creates the dictionary for each season;
# the dictionary is defined in the loop of Sec. 4.13.
def build_multirun_cost_scenarios_for_season(season_name):
    """Build Sec. 4.13 scenarios directly from the run manifest."""

    filters = dict(MULTIRUN_MANIFEST_FILTERS or {})
    filters.setdefault("run_revision", [MULTIRUN_RUN_REVISION])

    selected = select_hagrid_manifest_runs(
        experiment_id=MULTIRUN_EXPERIMENT_ID,
        filters=filters,
        allowed_statuses=("completed",),
    )


    # ------------------------------------------------------------
    # Explicit paper-case ordering for Sec. 4.13
    #
    # Do not rely on manifest control_order / run_uid ordering.
    # This affects only the comparative multi-run presentation.
    # ------------------------------------------------------------

    PAPER_CASE_ORDER = [
        "Base case",
        "Case 1",
        "Case 2",
        "Case 3",
        "Case 4",
        "Case 5",
        "Case 6",
        "Case 7",
    ]

    paper_case_order_map = {
        case_label: i
        for i, case_label in enumerate(PAPER_CASE_ORDER)
    }


    if "case_label" not in selected.columns:
        raise KeyError(
            "[MULTIRUN] 'case_label' is missing from the run manifest."
        )


    selected = selected.copy()

    selected["_paper_case_order"] = (
        selected["case_label"]
        .map(paper_case_order_map)
    )


    unknown_cases = selected.loc[
        selected["_paper_case_order"].isna(),
        "case_label",
    ].astype(str).unique().tolist()


    if unknown_cases:
        raise ValueError(
            "[MULTIRUN] Unknown case labels for paper ordering: "
            f"{unknown_cases}"
        )


    selected = (
        selected
        .sort_values(
            "_paper_case_order",
            kind="stable",
        )
        .drop(
            columns="_paper_case_order"
        )
        .reset_index(drop=True)
    )


    scenarios = []

    for x_order, (_, row) in enumerate(selected.iterrows(), start=1,):

        xls_dir = manifest_path_to_absolute(row["xls_relpath"])
        py_dir = manifest_path_to_absolute(row["py_relpath"])
        pflow_py_dir = manifest_path_to_absolute(row["pflow_py_relpath"])
        mode = str(row["mode"])

        scenarios.append({
            "x_order": x_order,
            "run_uid": row["run_uid"],
            "electrification_id": row.get("electrification_id"),
            "run_revision": row.get("run_revision"),
            "label": row["run_label"],
            "scenario_display_label": row.get("case_label", row["run_label"],),         # Display identifier used only by Sec. 4.13 charts/tables: Base case, Case 1, ..., Case 7.
            "case_label": row.get("case_label", row["run_label"]),
            "season": season_name,
            "summary_mode": mode,
            "comparison_kind": "mixed",
            "scenario_group": row["experiment_id"],

            # Exact saved directories.
            "xls_dir": xls_dir,
            "py_dir": py_dir,
            "pflow_py_dir": pflow_py_dir,

            # Exact main files.
            "summary_file": os.path.join(
                xls_dir,
                f"system_cost_summary_{mode}_"
                f"{num_households}_nuc_{selected_region}.xlsx",
            ),
            "results_file": os.path.join(
                xls_dir,
                f"ris_{season_name}.xlsx",
            ),

            # Compatibility with the current 4.13 functions.
            "share_FV": row.get("share_PV_target", np.nan),
            "share_BESS": row.get("share_BESS_target", np.nan),
            "share_BEV": row.get("share_BEV_target", np.nan),
            "share_boiler_DHW": row.get("share_DHW_target", np.nan),
            "share_HP": row.get("share_HP_target", np.nan),
            "share_COOL": row.get("share_COOL_target", np.nan),

            "control_BESS": row["control_BESS"],
            "control_BEV": row["control_BEV"],
            "control_DHW": row["control_DHW"],
        })

    return scenarios


# ============================================================
# Sec 4.6 -- Economic savings from already-saved multirun data
# ============================================================

def _economic_saving_pct(
    reference_cost,
    saving_eur,
    eps=1e-9,
):
    """
    Saving percentage relative to the Base/reference cost.

    Defined only for a strictly positive reference cost.
    For zero/negative reference costs, percentage saving
    is economically ambiguous and NaN is returned.
    """

    reference_cost = float(reference_cost)
    saving_eur = float(saving_eur)

    if reference_cost > eps:
        return (
            100.0
            * saving_eur
            / reference_cost
        )

    return np.nan


def _economic_control_symbol(control):
    """
    Symbols used in the paper-style table.

    NO  -> –
    RB  -> ◊
    OPT -> ⧫
    """

    value = str(control).strip().upper()

    if value in {"NO", "NONE", "OFF"}:
        return "–"

    if "OPT" in value:
        return "⧫"

    if "RB" in value:
        return "◊"

    return str(control)


def build_economic_savings_from_saved_runs(
    seasons_to_use=None,
    reference_case_label="Base case",
    annualization_factor=13.0,
    percentile_levels=None,
    preferred_total_cost_col="total_system_cost_effective_eur",
    strict_household_pairing=True,
):
    """
    Reconstruct economic savings using ONLY already-saved outputs.

    No optimization is solved here.

    Sources
    -------
    System:
        system_cost_summary_<MODE>_*.xlsx
        -> total_system_cost_effective_eur

    Household:
        ris_<season>.xlsx
        -> total_costs_per_household
        -> effective_total_eur

    Convention
    ----------
        saving = Base cost - Case cost

    Positive values therefore mean economic savings.
    """

    if seasons_to_use is None:
        seasons_to_use = list(
            seasons_dict.values()
        )

    seasons_to_use = [
        str(s)
        for s in seasons_to_use
    ]

    if percentile_levels is None:
        percentile_levels = [
            10, 20, 30, 40, 50,
            60, 70, 80, 90,
        ]

    percentile_levels = [
        int(p)
        for p in percentile_levels
    ]


    # --------------------------------------------------------
    # Paper case order
    # --------------------------------------------------------

    case_order = [
        "Base case",
        "Case 1",
        "Case 2",
        "Case 3",
        "Case 4",
        "Case 5",
        "Case 6",
        "Case 7",
    ]


    # --------------------------------------------------------
    # Storage of raw seasonal data
    # --------------------------------------------------------

    raw = {}

    source_rows = []

    metadata_by_case = {}

    physical_scenarios = set()


    # ========================================================
    # 1) READ SAVED OUTPUTS
    # ========================================================

    for season in seasons_to_use:

        scenarios = (
            build_multirun_cost_scenarios_for_season(
                season
            )
        )

        scenarios_by_case = {}

        for scenario in scenarios:

            case_label = str(
                scenario["case_label"]
            )

            if case_label not in case_order:
                continue

            if case_label in scenarios_by_case:
                raise ValueError(
                    "[ECON SAVINGS] More than one saved "
                    f"run found for {case_label}, "
                    f"season={season}. "
                    "Refine MULTIRUN_MANIFEST_FILTERS."
                )

            scenarios_by_case[
                case_label
            ] = scenario


        missing_cases = [
            case
            for case in case_order
            if case not in scenarios_by_case
        ]

        if missing_cases:
            raise ValueError(
                "[ECON SAVINGS] Missing cases for "
                f"season={season}: {missing_cases}"
            )


        for case_label in case_order:

            scenario = (
                scenarios_by_case[
                    case_label
                ]
            )

            physical_id = str(
                scenario.get(
                    "electrification_id",
                    "",
                )
            )

            physical_scenarios.add(
                physical_id
            )


            # --------------------------------------------
            # SYSTEM COST
            # --------------------------------------------

            summary_file = (
                _scenario_summary_file(
                    scenario
                )
            )

            (
                system_cost_eur,
                system_cost_column,
            ) = (
                _read_total_system_cost_from_summary(
                    summary_file=summary_file,
                    season=season,
                    summary_mode=(
                        scenario[
                            "summary_mode"
                        ]
                    ),
                    preferred_total_cost_col=(
                        preferred_total_cost_col
                    ),
                )
            )


            # --------------------------------------------
            # HOUSEHOLD COST
            # --------------------------------------------

            results_file = (
                _scenario_results_file(
                    scenario,
                    season=season,
                )
            )

            (
                household_costs,
                household_sheet,
                household_cost_column,
            ) = (
                _read_house_costs_from_mode_results(
                    results_file=results_file,
                    mode=(
                        scenario[
                            "summary_mode"
                        ]
                    ),
                )
            )


            # Standard household names.
            household_costs = (
                household_costs.copy()
            )

            household_costs.index = pd.Index(
                [
                    to_household_name(
                        house_id
                    )
                    for house_id
                    in household_costs.index
                ],
                name="house_id",
            )

            household_costs = (
                pd.to_numeric(
                    household_costs,
                    errors="coerce",
                )
                .dropna()
                .astype(float)
            )

            if household_costs.index.has_duplicates:
                raise ValueError(
                    "[ECON SAVINGS] Duplicated "
                    f"household IDs for "
                    f"{case_label}/{season}."
                )


            raw[
                (
                    case_label,
                    season,
                )
            ] = {
                "system_cost_eur":
                    float(system_cost_eur),

                "household_costs":
                    household_costs,

                "scenario":
                    scenario,
            }


            metadata_by_case.setdefault(
                case_label,
                scenario,
            )


            source_rows.append({
                "case_label":
                    case_label,

                "season":
                    season,

                "run_uid":
                    scenario["run_uid"],

                "mode":
                    scenario["summary_mode"],

                "electrification_id":
                    scenario.get(
                        "electrification_id"
                    ),

                "system_summary_file":
                    summary_file,

                "system_cost_column":
                    system_cost_column,

                "results_file":
                    results_file,

                "household_cost_sheet":
                    household_sheet,

                "household_cost_column":
                    household_cost_column,

                "n_households":
                    len(household_costs),
            })


    # --------------------------------------------------------
    # This paper table must refer to ONE physical scenario.
    # --------------------------------------------------------

    physical_scenarios.discard("")

    if len(physical_scenarios) > 1:

        raise ValueError(
            "[ECON SAVINGS] More than one "
            "electrification scenario selected:\n"
            f"{sorted(physical_scenarios)}\n"
            "Select one physical scenario before "
            "building the Base-vs-Case 1-7 table."
        )


    # ========================================================
    # 2) HOUSEHOLD POPULATION CHECK
    # ========================================================

    reference_houses = None

    for season in seasons_to_use:

        base_series = raw[
            (
                reference_case_label,
                season,
            )
        ]["household_costs"]

        houses = set(
            base_series.index
        )

        if reference_houses is None:
            reference_houses = houses

        elif (
            strict_household_pairing
            and houses != reference_houses
        ):

            raise ValueError(
                "[ECON SAVINGS] Base-case household "
                "population changes across seasons."
            )


    n_households = len(
        reference_houses
    )


    # ========================================================
    # 3) SEASONAL SYSTEM SAVINGS
    # ========================================================

    system_season_rows = []

    household_season_rows = []

    household_season_detail_rows = []


    for season in seasons_to_use:

        base_system_cost = raw[
            (
                reference_case_label,
                season,
            )
        ]["system_cost_eur"]

        base_household_costs = raw[
            (
                reference_case_label,
                season,
            )
        ]["household_costs"]


        for case_label in case_order:

            case_system_cost = raw[
                (
                    case_label,
                    season,
                )
            ]["system_cost_eur"]

            system_saving = (
                base_system_cost
                - case_system_cost
            )

            system_saving_pct = (
                _economic_saving_pct(
                    base_system_cost,
                    system_saving,
                )
            )


            # --------------------------------------------
            # Pair exactly the same households.
            # --------------------------------------------

            case_household_costs = raw[
                (
                    case_label,
                    season,
                )
            ]["household_costs"]


            if strict_household_pairing:

                if (
                    set(
                        case_household_costs.index
                    )
                    != set(
                        base_household_costs.index
                    )
                ):
                    raise ValueError(
                        "[ECON SAVINGS] Household "
                        "population mismatch between "
                        f"{reference_case_label} and "
                        f"{case_label}, season={season}."
                    )


            paired = pd.concat(
                [
                    base_household_costs.rename(
                        "base_cost_eur"
                    ),
                    case_household_costs.rename(
                        "case_cost_eur"
                    ),
                ],
                axis=1,
                join="inner",
            ).dropna()


            paired["saving_eur"] = (
                paired["base_cost_eur"]
                - paired["case_cost_eur"]
            )


            paired["saving_pct"] = np.where(
                paired["base_cost_eur"]
                > 1e-9,

                (
                    100.0
                    * paired["saving_eur"]
                    / paired["base_cost_eur"]
                ),

                np.nan,
            )


            # --------------------------------------------
            # System seasonal table
            # --------------------------------------------

            system_season_rows.append({

                "case_label":
                    case_label,

                "season":
                    season,

                "base_system_cost_eur_week":
                    base_system_cost,

                "case_system_cost_eur_week":
                    case_system_cost,

                "system_saving_eur_week":
                    system_saving,

                "system_saving_pct":
                    system_saving_pct,

                "system_avg_saving_eur_per_house_week":
                    (
                        system_saving
                        / n_households
                    ),

                "year_equiv_system_saving_from_this_season_eur":
                    (
                        annualization_factor
                        * system_saving
                    ),

                "year_equiv_system_avg_saving_from_this_season_eur_per_house":
                    (
                        annualization_factor
                        * system_saving
                        / n_households
                    ),

                "n_households":
                    n_households,
            })


            # --------------------------------------------
            # Household seasonal statistics
            # --------------------------------------------

            saving_values = (
                paired["saving_eur"]
                .to_numpy(dtype=float)
            )

            valid_pct = (
                pd.to_numeric(
                    paired["saving_pct"],
                    errors="coerce",
                )
                .dropna()
            )


            row = {

                "case_label":
                    case_label,

                "season":
                    season,

                "n_households_paired":
                    len(paired),

                "mean_household_saving_eur_week":
                    float(
                        paired[
                            "saving_eur"
                        ].mean()
                    ),

                "median_household_saving_eur_week":
                    float(
                        paired[
                            "saving_eur"
                        ].median()
                    ),

                "mean_household_saving_pct":
                    float(
                        valid_pct.mean()
                    )
                    if not valid_pct.empty
                    else np.nan,

                "median_household_saving_pct":
                    float(
                        valid_pct.median()
                    )
                    if not valid_pct.empty
                    else np.nan,

                "share_households_saving_pct":
                    (
                        100.0
                        * float(
                            (
                                paired[
                                    "saving_eur"
                                ]
                                > 1e-9
                            ).sum()
                        )
                        / len(paired)
                    ),

                "share_households_penalized_pct":
                    (
                        100.0
                        * float(
                            (
                                paired[
                                    "saving_eur"
                                ]
                                < -1e-9
                            ).sum()
                        )
                        / len(paired)
                    ),
            }


            for p in percentile_levels:

                row[
                    f"p{p}_household_saving_eur_week"
                ] = float(
                    np.percentile(
                        saving_values,
                        p,
                    )
                )

                if not valid_pct.empty:

                    row[
                        f"p{p}_household_saving_pct"
                    ] = float(
                        np.percentile(
                            valid_pct.to_numpy(
                                dtype=float
                            ),
                            p,
                        )
                    )

                else:

                    row[
                        f"p{p}_household_saving_pct"
                    ] = np.nan


            household_season_rows.append(
                row
            )


            # --------------------------------------------
            # Full household seasonal detail
            # --------------------------------------------

            for house_id, hh_row in paired.iterrows():

                household_season_detail_rows.append({

                    "case_label":
                        case_label,

                    "season":
                        season,

                    "house_id":
                        house_id,

                    "base_cost_eur_week":
                        float(
                            hh_row[
                                "base_cost_eur"
                            ]
                        ),

                    "case_cost_eur_week":
                        float(
                            hh_row[
                                "case_cost_eur"
                            ]
                        ),

                    "saving_eur_week":
                        float(
                            hh_row[
                                "saving_eur"
                            ]
                        ),

                    "saving_pct":
                        float(
                            hh_row[
                                "saving_pct"
                            ]
                        )
                        if pd.notna(
                            hh_row[
                                "saving_pct"
                            ]
                        )
                        else np.nan,
                })


    df_system_season = pd.DataFrame(
        system_season_rows
    )

    df_household_season = pd.DataFrame(
        household_season_rows
    )

    df_household_season_detail = (
        pd.DataFrame(
            household_season_detail_rows
        )
    )


    # ========================================================
    # 4) ANNUALIZED HOUSEHOLD COSTS
    #
    # IMPORTANT:
    # first annualize EACH household, then calculate
    # median/percentiles.
    # ========================================================

    annual_household_cost_by_case = {}

    for case_label in case_order:

        season_series = []

        for season in seasons_to_use:

            s = raw[
                (
                    case_label,
                    season,
                )
            ]["household_costs"].rename(
                season
            )

            season_series.append(
                s
            )


        df_case_seasons = pd.concat(
            season_series,
            axis=1,
            join="inner",
        )


        if strict_household_pairing:

            if (
                set(
                    df_case_seasons.index
                )
                != reference_houses
            ):
                raise ValueError(
                    "[ECON SAVINGS] Annual household "
                    f"population mismatch for {case_label}."
                )


        annual_household_cost_by_case[
            case_label
        ] = (
            df_case_seasons
            .sum(axis=1)
            * float(
                annualization_factor
            )
        )


    # ========================================================
    # 5) ANNUAL SYSTEM + HOUSEHOLD SUMMARY
    # ========================================================

    base_system_annual = (
        float(
            annualization_factor
        )
        * sum(
            raw[
                (
                    reference_case_label,
                    season,
                )
            ]["system_cost_eur"]
            for season in seasons_to_use
        )
    )


    base_household_annual = (
        annual_household_cost_by_case[
            reference_case_label
        ]
    )


    annual_summary_rows = []

    annual_household_detail_rows = []


    for case_label in case_order:

        case_system_annual = (
            float(
                annualization_factor
            )
            * sum(
                raw[
                    (
                        case_label,
                        season,
                    )
                ]["system_cost_eur"]
                for season in seasons_to_use
            )
        )


        system_saving_annual = (
            base_system_annual
            - case_system_annual
        )


        system_saving_pct = (
            _economic_saving_pct(
                base_system_annual,
                system_saving_annual,
            )
        )


        case_household_annual = (
            annual_household_cost_by_case[
                case_label
            ]
        )


        paired_annual = pd.concat(
            [
                base_household_annual.rename(
                    "base_cost_annual_eur"
                ),
                case_household_annual.rename(
                    "case_cost_annual_eur"
                ),
            ],
            axis=1,
            join="inner",
        ).dropna()


        paired_annual[
            "saving_annual_eur"
        ] = (
            paired_annual[
                "base_cost_annual_eur"
            ]
            - paired_annual[
                "case_cost_annual_eur"
            ]
        )


        paired_annual[
            "saving_annual_pct"
        ] = np.where(

            paired_annual[
                "base_cost_annual_eur"
            ] > 1e-9,

            (
                100.0
                * paired_annual[
                    "saving_annual_eur"
                ]
                / paired_annual[
                    "base_cost_annual_eur"
                ]
            ),

            np.nan,
        )


        savings = (
            paired_annual[
                "saving_annual_eur"
            ]
            .astype(float)
        )

        saving_pct_valid = (
            paired_annual[
                "saving_annual_pct"
            ]
            .dropna()
            .astype(float)
        )


        household_sum_saving = float(
            savings.sum()
        )


        annual_row = {

            "case_label":
                case_label,

            "base_system_cost_annual_eur":
                base_system_annual,

            "case_system_cost_annual_eur":
                case_system_annual,

            "system_saving_annual_eur":
                system_saving_annual,

            "system_saving_annual_pct":
                system_saving_pct,

            # Your "System average OLD"
            "system_avg_saving_annual_eur_per_house":
                (
                    system_saving_annual
                    / n_households
                ),

            # Mean from household-attributable costs.
            # It may differ from system_total / N if some
            # system-wide costs are not household allocated.
            "mean_household_saving_annual_eur":
                float(
                    savings.mean()
                ),

            # Your "Median household OLD"
            "median_household_saving_annual_eur":
                float(
                    savings.median()
                ),

            "mean_household_saving_annual_pct":
                float(
                    saving_pct_valid.mean()
                )
                if not saving_pct_valid.empty
                else np.nan,

            "median_household_saving_annual_pct":
                float(
                    saving_pct_valid.median()
                )
                if not saving_pct_valid.empty
                else np.nan,

            "n_households":
                len(paired_annual),

            "household_sum_saving_annual_eur":
                household_sum_saving,

            # Audit:
            # difference between system-level saving and
            # the sum of household-attributable savings.
            "system_minus_household_saving_gap_eur":
                (
                    system_saving_annual
                    - household_sum_saving
                ),

            "share_households_saving_pct":
                (
                    100.0
                    * (
                        savings > 1e-9
                    ).sum()
                    / len(savings)
                ),

            "share_households_penalized_pct":
                (
                    100.0
                    * (
                        savings < -1e-9
                    ).sum()
                    / len(savings)
                ),
        }


        for p in percentile_levels:

            annual_row[
                f"p{p}_household_saving_annual_eur"
            ] = float(
                np.percentile(
                    savings.to_numpy(),
                    p,
                )
            )

            if not saving_pct_valid.empty:

                annual_row[
                    f"p{p}_household_saving_annual_pct"
                ] = float(
                    np.percentile(
                        saving_pct_valid.to_numpy(),
                        p,
                    )
                )

            else:

                annual_row[
                    f"p{p}_household_saving_annual_pct"
                ] = np.nan


        annual_summary_rows.append(
            annual_row
        )


        for house_id, hh_row in paired_annual.iterrows():

            annual_household_detail_rows.append({

                "case_label":
                    case_label,

                "house_id":
                    house_id,

                "base_cost_annual_eur":
                    float(
                        hh_row[
                            "base_cost_annual_eur"
                        ]
                    ),

                "case_cost_annual_eur":
                    float(
                        hh_row[
                            "case_cost_annual_eur"
                        ]
                    ),

                "saving_annual_eur":
                    float(
                        hh_row[
                            "saving_annual_eur"
                        ]
                    ),

                "saving_annual_pct":
                    float(
                        hh_row[
                            "saving_annual_pct"
                        ]
                    )
                    if pd.notna(
                        hh_row[
                            "saving_annual_pct"
                        ]
                    )
                    else np.nan,
            })


    df_annual_summary = pd.DataFrame(
        annual_summary_rows
    )

    df_annual_household_detail = (
        pd.DataFrame(
            annual_household_detail_rows
        )
    )


    # Preserve paper order.
    case_rank = {
        case: i
        for i, case in enumerate(
            case_order
        )
    }

    df_annual_summary[
        "_order"
    ] = (
        df_annual_summary[
            "case_label"
        ].map(case_rank)
    )

    df_annual_summary = (
        df_annual_summary
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )


    # ========================================================
    # 6) PAPER-STYLE TABLE
    # ========================================================

    annual_by_case = (
        df_annual_summary
        .set_index("case_label")
    )


    paper_rows = {}


    paper_rows["BESS"] = {
        case:
            _economic_control_symbol(
                metadata_by_case[
                    case
                ].get(
                    "control_BESS"
                )
            )
        for case in case_order
    }

    paper_rows["EV"] = {
        case:
            _economic_control_symbol(
                metadata_by_case[
                    case
                ].get(
                    "control_BEV"
                )
            )
        for case in case_order
    }

    paper_rows["ESWH"] = {
        case:
            _economic_control_symbol(
                metadata_by_case[
                    case
                ].get(
                    "control_DHW"
                )
            )
        for case in case_order
    }


    paper_rows[
        "Yearly-equivalent system savings [EUR]"
    ] = {
        case:
            annual_by_case.loc[
                case,
                "system_saving_annual_eur",
            ]
        for case in case_order
    }


    paper_rows[
        "Yearly-equivalent system savings [%]"
    ] = {
        case:
            annual_by_case.loc[
                case,
                "system_saving_annual_pct",
            ]
        for case in case_order
    }


    paper_rows[
        "Yearly-equivalent savings [EUR/household] - system average"
    ] = {
        case:
            annual_by_case.loc[
                case,
                "system_avg_saving_annual_eur_per_house",
            ]
        for case in case_order
    }


    paper_rows[
        "Median household yearly-equivalent savings [EUR/household]"
    ] = {
        case:
            annual_by_case.loc[
                case,
                "median_household_saving_annual_eur",
            ]
        for case in case_order
    }


    paper_rows[
        "Median household yearly-equivalent savings [%]"
    ] = {
        case:
            annual_by_case.loc[
                case,
                "median_household_saving_annual_pct",
            ]
        for case in case_order
    }


    df_paper_table = (
        pd.DataFrame(
            paper_rows
        )
        .T
        .reindex(
            columns=case_order
        )
    )


    # ========================================================
    # 7) SOURCE / METHODOLOGY AUDIT
    # ========================================================

    df_source_audit = pd.DataFrame(
        source_rows
    )


    methodology = pd.DataFrame([
        {
            "item":
                "Savings sign convention",
            "value":
                "Base cost - Case cost; positive = saving",
        },
        {
            "item":
                "Annualization",
            "value":
                (
                    f"{annualization_factor:g} x each "
                    "representative seasonal week"
                ),
        },
        {
            "item":
                "System metric",
            "value":
                preferred_total_cost_col,
        },
        {
            "item":
                "Household metric",
            "value":
                "effective_total_eur",
        },
        {
            "item":
                "Annual household percentile rule",
            "value":
                (
                    "annualize each paired household first, "
                    "then calculate percentiles"
                ),
        },
        {
            "item":
                "Reference case",
            "value":
                reference_case_label,
        },
    ])


    return {
        "paper_table":
            df_paper_table,

        "annual_summary":
            df_annual_summary,

        "system_seasonal":
            df_system_season,

        "household_seasonal_stats":
            df_household_season,

        "household_seasonal_detail":
            df_household_season_detail,

        "household_annual_detail":
            df_annual_household_detail,

        "source_audit":
            df_source_audit,

        "methodology":
            methodology,
    }



def write_economic_savings_tables(
    output_path,
    tables,
    append=False,
):
    """
    Save/append the economic-savings tables to an Excel workbook.
    """

    os.makedirs(
        os.path.dirname(
            os.path.abspath(
                output_path
            )
        ),
        exist_ok=True,
    )


    if append and os.path.isfile(
        output_path
    ):

        writer_kwargs = {
            "engine": "openpyxl",
            "mode": "a",
            "if_sheet_exists": "replace",
        }

    else:

        writer_kwargs = {
            "engine": "openpyxl",
            "mode": "w",
        }


    with pd.ExcelWriter(
        output_path,
        **writer_kwargs,
    ) as writer:

        tables[
            "paper_table"
        ].to_excel(
            writer,
            sheet_name="savings_paper_table",
            index=True,
        )

        tables[
            "annual_summary"
        ].to_excel(
            writer,
            sheet_name="savings_annual",
            index=False,
        )

        tables[
            "system_seasonal"
        ].to_excel(
            writer,
            sheet_name="savings_system_season",
            index=False,
        )

        tables[
            "household_seasonal_stats"
        ].to_excel(
            writer,
            sheet_name="savings_hh_season",
            index=False,
        )

        tables[
            "household_annual_detail"
        ].to_excel(
            writer,
            sheet_name="savings_hh_annual",
            index=False,
        )

        tables[
            "household_seasonal_detail"
        ].to_excel(
            writer,
            sheet_name="savings_hh_detail",
            index=False,
        )

        tables[
            "source_audit"
        ].to_excel(
            writer,
            sheet_name="savings_sources",
            index=False,
        )

        tables[
            "methodology"
        ].to_excel(
            writer,
            sheet_name="savings_method",
            index=False,
        )















# Function to return the columns required for chart construction from the summary file
def _multirun_required_columns_for_graphs(graphs_to_plot):
    """
    Returns the minimum columns required in the plot_data sheet
    to produce the requested charts.
    """
    graphs_to_plot = set(graphs_to_plot or [])

    required = {
        # Common base columns.
        "scenario_group",
        "x_order",
        "mode",
        "label",
        "case_label",
        "total_system_cost_eur",
    }

    if "cost_percentiles" in graphs_to_plot:
        required.update({
            "p10_eur", "p20_eur", "p30_eur", "p40_eur", "p50_eur",
            "p60_eur", "p70_eur", "p80_eur", "p90_eur",
            "avg_household_weekly_system_cost_eur",
        })

    if "demand_coverage" in graphs_to_plot:
        required.update({
            "total_final_demand_kwh",
            "net_mv_import_for_demand_kwh",
            "internal_self_sufficiency_kwh",
            "pv_direct_to_demand_kwh",
            "bess_to_demand_kwh",
            "grid_import_to_demand_kwh",
        })

    return required


# Function to check whether the summary/cache file contains all required information/columns
def _multirun_cache_is_complete(df_cache, graphs_to_plot, verbose=True):
    """
    Checks whether the summary/cache contains all columns required
    for the requested charts.
    """
    if df_cache is None or df_cache.empty:
        if verbose:
            print("[INFO][MULTIRUN CACHE] Cache empty or missing.")
        return False

    required_cols = _multirun_required_columns_for_graphs(graphs_to_plot)
    missing_cols = sorted(required_cols - set(df_cache.columns))

    if missing_cols:
        if verbose:
            print("[INFO][MULTIRUN CACHE] Incomplete cache. Missing columns:")
            for c in missing_cols:
                print(f"  - {c}")
        return False

    if verbose:
        print("[OK][MULTIRUN CACHE] Cache complete for the requested charts.")

    return True


# Function to read the summary/cache file containing the information for multi-run charts
def _load_multirun_summary_cache_if_complete(cache_xlsx, graphs_to_plot, verbose=True):
    """
    Tries to read the plot_data sheet from the multi-run cache.
    Returns it only if it contains the required columns.
    """
    if not os.path.exists(cache_xlsx):
        if verbose:
            print(f"[INFO][MULTIRUN CACHE] Cache file not found: {cache_xlsx}")
        return None

    try:
        df_cache = pd.read_excel(cache_xlsx, sheet_name="plot_data")
    except Exception as e:
        if verbose:
            print(f"[WARN][MULTIRUN CACHE] Could not read cache: {cache_xlsx}")
            print(f"  Error: {e}")
        return None

    if not _multirun_cache_is_complete(
        df_cache=df_cache,
        graphs_to_plot=graphs_to_plot,
        verbose=verbose,
    ):
        return None

    return df_cache


# Function to use data from summary files; if incomplete, use pickle files to rebuild the data
def _get_or_build_multirun_plot_data(
    *,
    cache_xlsx,
    scenarios,
    graphs_to_plot,
    output_folder,
    num_households,
    selected_region,
    PV_share,
    PV_power_kwp,
    preferred_total_cost_col,
    use_cache_if_complete=True,
    rebuild_from_pickle_if_needed=True,
    save_updated_cache=True,
):
    """
    Unified workflow for 4.13:
      1. try to use summary/cache if complete;
      2. if not enough, rebuild from pickle files;
      3. optionally save the updated cache.
    """
    df_plot = None
    df_skipped = pd.DataFrame()

    if use_cache_if_complete:
        df_plot = _load_multirun_summary_cache_if_complete(
            cache_xlsx=cache_xlsx,
            graphs_to_plot=graphs_to_plot,
            verbose=True,
        )

    if df_plot is not None:
        print("[INFO][MULTIRUN] Using existing summary/cache.")
        return df_plot, df_skipped, "cache"

    if not rebuild_from_pickle_if_needed:
        raise RuntimeError(
            "[ERROR][MULTIRUN] Cache missing/incomplete and "
            "rebuild_from_pickle_if_needed=False."
        )

    print("[INFO][MULTIRUN] Rebuilding multi-run summary from pickle files.")

    df_plot, df_skipped = build_multirun_cost_percentile_summary(
        scenarios=scenarios,
        PV_share=PV_share,
        PV_power_kwp=PV_power_kwp,
        preferred_total_cost_col=preferred_total_cost_col,
        skip_missing=True,
        allow_energy_pickle_fallback=True,
        pickle_values_are_power_kw=True,
        energy_source="pickle",
    )

    if df_plot is None or df_plot.empty:
        raise RuntimeError(
            "[ERROR][MULTIRUN] Rebuilt summary is empty. "
            "Check scenarios, paths and pickle files."
        )

    if save_updated_cache:
        os.makedirs(os.path.dirname(cache_xlsx), exist_ok=True)

        with pd.ExcelWriter(cache_xlsx, engine="openpyxl") as writer:
            df_plot.to_excel(writer, sheet_name="plot_data", index=False)

            if df_skipped is not None and not df_skipped.empty:
                df_skipped.to_excel(writer, sheet_name="skipped", index=False)

            metadata_df = pd.DataFrame([{
                "cache_created_from": "pickle",
                "cache_policy": "use_cache_if_complete_else_rebuild_pickle",
                "graphs_to_plot": ", ".join(graphs_to_plot),
                "preferred_total_cost_col": preferred_total_cost_col,
            }])

            metadata_df.to_excel(writer, sheet_name="metadata", index=False)

        print(f"[OK][MULTIRUN] Updated cache saved: {cache_xlsx}")

    return df_plot, df_skipped, "rebuilt_from_pickle"



# Function to plot graph comparison for household delta cost between scenarios
def plot_multirun_household_cost_delta_percentiles(
    df_delta_plot,
    title=None,
    out_png=None,
    y1_label="Paired household cost difference [EUR/week]",
    y2_label="Households penalized [%]",
    y1_ylim=None,
    legend_mode=None,
    show_title=None,
    save_pdf=None,
):
    """
    Plots paired household-cost differences relative to a reference MODE.

    Primary axis:
      percentile bands of:
          configuration household cost - reference household cost

    Secondary axis:
      share of paired households with a positive delta.

    Positive delta = household penalization.
    Negative delta = household saving.
    """
    if df_delta_plot is None or df_delta_plot.empty:
        print("[WARN][HOUSEHOLD DELTA] Empty dataframe: skip chart.")
        return

    if show_title is None:
        show_title = globals().get(
            "SHOW_MULTIRUN_FIGURE_TITLE",
            False,
        )

    if save_pdf is None:
        save_pdf = globals().get(
            "SAVE_JOURNAL_PDF",
            True,
        )

    if legend_mode is None:
        legend_mode = globals().get(
            "MULTIRUN_COST_LEGEND_MODE",
            "upper_right_inside",
        )

    font_size = globals().get("JOURNAL_FONT_SIZE", 9.0)
    axis_label_size = globals().get(
        "JOURNAL_AXIS_LABEL_SIZE",
        10.0,
    )
    tick_label_size = globals().get(
        "JOURNAL_TICK_LABEL_SIZE",
        8.5,
    )
    title_size = globals().get(
        "JOURNAL_TITLE_SIZE",
        10.5,
    )

    plt.rcParams.update({
        "font.size": font_size,
        "axes.labelsize": axis_label_size,
        "xtick.labelsize": tick_label_size,
        "ytick.labelsize": tick_label_size,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

    d = (
        df_delta_plot
        .sort_values("x_order")
        .reset_index(drop=True)
        .copy()
    )

    percentile_cols = [
        "p10_delta_eur",
        "p20_delta_eur",
        "p30_delta_eur",
        "p40_delta_eur",
        "p50_delta_eur",
        "p60_delta_eur",
        "p70_delta_eur",
        "p80_delta_eur",
        "p90_delta_eur",
    ]

    required_cols = percentile_cols + [
        "share_households_penalized_pct",
        "scenario_group",
    ]

    missing_cols = [
        col for col in required_cols
        if col not in d.columns
    ]

    if missing_cols:
        raise KeyError(
            "Missing columns for household-delta chart: "
            f"{missing_cols}"
        )

    x = np.arange(len(d), dtype=float)

    group_base_colors = {
        "no_flex": "#2ca02c",
        "only_DHW": "#f2c94c",
        "only_BESS": "#1f77b4",
        "only_EV": "#d62728",
        "all_flex": "#7f7f7f",
        "mixed_control": "#9467bd",
        "other": "#9467bd",
    }

    figsize, subplot_margins = _multirun_figure_layout(
        n_columns=len(d),
        plot_type="cost",
    )

    fig, ax1 = plt.subplots(
        figsize=figsize,
        constrained_layout=False,
    )

    ax2 = ax1.twinx()

    band_width = 0.64

    band_specs = [
        (
            "p10_delta_eur",
            "p90_delta_eur",
            0.78,
        ),
        (
            "p20_delta_eur",
            "p80_delta_eur",
            0.62,
        ),
        (
            "p30_delta_eur",
            "p70_delta_eur",
            0.46,
        ),
        (
            "p40_delta_eur",
            "p60_delta_eur",
            0.30,
        ),
    ]

    for j, row in d.iterrows():

        scenario_group = str(
            row.get("scenario_group", "other")
        )

        base_color = group_base_colors.get(
            scenario_group,
            group_base_colors["other"],
        )

        for low_col, high_col, lighten_amount in band_specs:

            band_color = _lighten_color(
                base_color,
                amount=lighten_amount,
            )

            ax1.fill_between(
                [
                    x[j] - band_width / 2,
                    x[j] + band_width / 2,
                ],
                [
                    row[low_col],
                    row[low_col],
                ],
                [
                    row[high_col],
                    row[high_col],
                ],
                color=band_color,
                edgecolor=base_color,
                linewidth=0.45,
                zorder=2,
            )

        # Median paired delta.
        ax1.plot(
            [
                x[j] - band_width / 2,
                x[j] + band_width / 2,
            ],
            [
                row["p50_delta_eur"],
                row["p50_delta_eur"],
            ],
            color="black",
            linewidth=2.1,
            solid_capstyle="butt",
            zorder=5,
        )

    penalized_share = pd.to_numeric(
        d["share_households_penalized_pct"],
        errors="coerce",
    )

    ax2.plot(
        x,
        penalized_share,
        marker="o",
        markersize=globals().get(
            "MULTIRUN_SYSTEM_COST_MARKER_SIZE",
            3.8,
        ),
        linewidth=0.0,
        linestyle="None",
        color="black",
        zorder=20,
    )

    scenario_xticklabels = [
        _scenario_display_label_for_plot(
            row,
            i,
        ).replace(" ", "\n")
        for i, (_, row) in enumerate(d.iterrows())
    ]

    ax1.set_xticks(x)
    ax1.set_xticklabels(
        scenario_xticklabels,
        rotation=0,
        ha="center",
    )

    ax1.tick_params(
        axis="x",
        which="both",
        length=0,
        pad=3,
    )

    ax1.set_xlim(-0.75, len(d) - 0.25)

    _add_scenario_config_table_under_axis(
        ax=ax1,
        d=d,
        include_first_column=False,
        table_y=None,
        table_height=None,
        font_size=globals().get(
            "JOURNAL_TABLE_FONT_SIZE",
            6.9,
        ),
        symbol_font_size=globals().get(
            "JOURNAL_TABLE_SYMBOL_FONT_SIZE",
            7.8,
        ),
        header_font_size=globals().get(
            "JOURNAL_TABLE_HEADER_FONT_SIZE",
            6.9,
        ),
        note_font_size=globals().get(
            "JOURNAL_TABLE_NOTE_FONT_SIZE",
            6.9,
        ),
        add_legend_note=False,
        show_column_labels=False,
    )

    ax1.set_ylabel(
        y1_label,
        fontsize=axis_label_size,
    )

    ax2.set_ylabel(
        y2_label,
        fontsize=axis_label_size,
    )

    ax2.set_ylim(0.0, 100.0)

    # Zero separates savings from penalizations.
    ax1.axhline(
        0.0,
        color="black",
        linewidth=0.9,
        alpha=0.65,
        zorder=1,
    )

    if y1_ylim is not None:
        ax1.set_ylim(*y1_ylim)

    else:
        values = (
            d[percentile_cols]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=float)
            .ravel()
        )

        values = values[np.isfinite(values)]

        if values.size > 0:
            max_abs = float(np.max(np.abs(values)))

            if max_abs <= 1e-12:
                max_abs = 1.0

            if globals().get(
                "MULTIRUN_DELTA_SYMMETRIC_YLIM",
                True,
            ):
                limit = 1.10 * max_abs
                ax1.set_ylim(-limit, limit)

    ax1.grid(
        axis="y",
        linestyle=":",
        linewidth=0.7,
        alpha=0.42,
    )

    ax1.set_axisbelow(True)

    if show_title and title:
        ax1.set_title(
            title,
            pad=8,
            fontsize=title_size,
        )

    legend_handles = [
        Patch(
            facecolor="grey",
            alpha=0.20,
            label="p10-p90",
        ),
        Patch(
            facecolor="grey",
            alpha=0.35,
            label="p20-p80",
        ),
        Patch(
            facecolor="grey",
            alpha=0.50,
            label="p30-p70",
        ),
        Patch(
            facecolor="grey",
            alpha=0.65,
            label="p40-p60",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=2.0,
            label="Median delta",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            marker="o",
            linestyle="None",
            markersize=3.8,
            label="Penalized households",
        ),
    ]

    _add_smart_inside_legend(
        ax=ax1,
        handles=legend_handles,
        labels=[
            handle.get_label()
            for handle in legend_handles
        ],
        title="Bands / points",
        ncol=2,
        legend_mode=legend_mode,
    )

    fig.subplots_adjust(**subplot_margins)

    if out_png:
        fig.savefig(
            out_png,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.04,
        )

        if save_pdf:
            out_pdf = os.path.splitext(out_png)[0] + ".pdf"

            fig.savefig(
                out_pdf,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        print(
            "[OK][HOUSEHOLD DELTA] Saved paired-delta chart: "
            f"{out_png}"
        )

    else:
        plt.show()

    plt.close(fig)



# Function for build the proper directory for input data for graph construction
def _safe_output_path(
    folder,
    descriptive_filename,
    compact_filename,
    windows_safe_length=230,
):
    """
    Build an output path for Sec. 4.13.

    The function also reduces the risk of Windows/OneDrive
    FileNotFoundError exceptions caused by:

    1. a missing parent directory;
    2. an excessively long absolute path.

    The descriptive filename is retained whenever the complete
    path is sufficiently short. On Windows, if the path exceeds
    the selected safety threshold, the compact filename is used.
    """

    # Always make sure that the output directory exists before
    # pandas, openpyxl or Matplotlib attempts to create a file.
    os.makedirs(
        folder,
        exist_ok=True,
    )

    descriptive_path = os.path.join(
        folder,
        descriptive_filename,
    )

    descriptive_abs = os.path.abspath(
        descriptive_path
    )

    # Long paths can still create problems in some Windows,
    # OneDrive, pandas and openpyxl combinations.
    if (
        os.name == 'nt'
        and len(descriptive_abs) > windows_safe_length
    ):

        compact_path = os.path.join(
            folder,
            compact_filename,
        )

        compact_abs = os.path.abspath(
            compact_path
        )

        print(
            '[WARN][MULTIRUN PATH] Descriptive path is too long '
            f'({len(descriptive_abs)} characters). '
            'Using compact name:\n'
            f'  {compact_path}'
        )

        if len(compact_abs) > windows_safe_length:
            print(
                '[WARN][MULTIRUN PATH] The compact path is still '
                f'long ({len(compact_abs)} characters). '
                'Consider moving the project closer to the '
                'drive root.'
            )

        return compact_path

    return descriptive_path

## 4.7 Resolve user configuration

In [ ]:
# ------------------------------------------------------------
#   Sec 4.7 -- 0) Derived constraints
# ------------------------------------------------------------

HAGRID_GRID_GROUP = _grid_family_from_lv_archetype(LV_ARCHETYPE)

pv_power_tag = int(PV_power_kwp) if float(PV_power_kwp).is_integer() else PV_power_kwp

FACTOR_KWH_10MIN_TO_KW = 60.0 / timestep_minutes        # Conversion factor from energy per 10-minute timestep to average power during the timestep: 10 min = 1/6 h, therefore kWh/10 min * 6 = kW.


secondary_substation_active_power_kw = secondary_substation_apparent_power_kva * secondary_substation_power_factor      # [kW] (since kW = kVA * cos (Phi), active power + reactive power)


# ------------------------------------------------------------
#   Sec 4.7 -- 1) Resolve electrification experiment
# ------------------------------------------------------------

if (
    ACTIVE_ELECTRIFICATION_EXPERIMENT
    not in ELECTRIFICATION_EXPERIMENTS
):
    raise KeyError(
        "Unknown electrification experiment: "
        f"{ACTIVE_ELECTRIFICATION_EXPERIMENT}. "
        "Available experiments: "
        f"{list(ELECTRIFICATION_EXPERIMENTS)}"
    )

experiment_def = ELECTRIFICATION_EXPERIMENTS[
    ACTIVE_ELECTRIFICATION_EXPERIMENT
]

EXPERIMENT_ID = str(
    ACTIVE_ELECTRIFICATION_EXPERIMENT
)

EXPERIMENT_VARYING_SHARES = tuple(
    experiment_def.get(
        "varying_shares",
        (),
    )
)

ELECTRIFICATION_SCENARIO_CATALOG = (
    build_electrification_scenarios_from_definition(
        experiment_def
    )
)




# ------------------------------------------------------------
#   Sec 4.7 -- 2) Select physical scenarios
# ------------------------------------------------------------

available_electrification_ids = [
    sc["scenario_id"]
    for sc in ELECTRIFICATION_SCENARIO_CATALOG
]

if ACTIVE_ELECTRIFICATION_SCENARIO_IDS_OVERRIDE is None:

    ACTIVE_ELECTRIFICATION_SCENARIO_IDS = (
        available_electrification_ids.copy()
    )

else:

    ACTIVE_ELECTRIFICATION_SCENARIO_IDS = list(
        ACTIVE_ELECTRIFICATION_SCENARIO_IDS_OVERRIDE
    )


unknown_electrification_ids = sorted(
    set(ACTIVE_ELECTRIFICATION_SCENARIO_IDS)
    - set(available_electrification_ids)
)

if unknown_electrification_ids:
    raise KeyError(
        "Unknown electrification scenario IDs: "
        f"{unknown_electrification_ids}. "
        "Available IDs: "
        f"{available_electrification_ids}"
    )


# ------------------------------------------------------------
#   Sec 4.7 -- 3) Resolve control group
# ------------------------------------------------------------

if ACTIVE_CONTROL_GROUP not in CONTROL_GROUPS:
    raise KeyError(
        f"Unknown control group: {ACTIVE_CONTROL_GROUP}"
    )

ACTIVE_CONTROL_CONFIG_KEYS = list(
    CONTROL_GROUPS[ACTIVE_CONTROL_GROUP]
)

CONTROL_CONFIG_BY_CASE_LABEL = {
    cfg["case_label"]: cfg
    for cfg in CONTROL_CONFIG_CATALOG
}

RUN_CONTROL_CONFIGS = [
    _normalize_run_control_config(
        CONTROL_CONFIG_BY_CASE_LABEL[key]
    )
    for key in ACTIVE_CONTROL_CONFIG_KEYS
]



# ------------------------------------------------------------
#   Sec 4.7 --  4) Build complete physical x control runs
# ------------------------------------------------------------

RUN_CONFIGS = build_experiment_run_configs(
    control_configs=RUN_CONTROL_CONFIGS,
    electrification_configs=ELECTRIFICATION_SCENARIO_CATALOG,
)

SOLVE_RUN_CONFIGS = (
    RUN_CONFIGS.copy()
    if RUN_SOLVE_MIXED_SCENARIOS
    else []
)



# ============================================================
# PAPER CASES - PRE-SOLVE VALIDATION
# ============================================================

EXPECTED_PAPER_CONTROLS = {
    "Base case": {"BESS": "NO",  "BEV": "RB",  "DHW": "RB"},
    "Case 1":    {"BESS": "NO",  "BEV": "RB",  "DHW": "OPT"},
    "Case 2":    {"BESS": "NO",  "BEV": "OPT", "DHW": "RB"},
    "Case 3":    {"BESS": "NO",  "BEV": "OPT", "DHW": "OPT"},
    "Case 4":    {"BESS": "OPT", "BEV": "RB",  "DHW": "RB"},
    "Case 5":    {"BESS": "OPT", "BEV": "RB",  "DHW": "OPT"},
    "Case 6":    {"BESS": "OPT", "BEV": "OPT", "DHW": "RB"},
    "Case 7":    {"BESS": "OPT", "BEV": "OPT", "DHW": "OPT"},
}


if RUN_SOLVE_MIXED_SCENARIOS:

    # --------------------------------------------------------
    # 1) Physical scenario
    # --------------------------------------------------------
    if ACTIVE_ELECTRIFICATION_EXPERIMENT != "FULL_ELECTRIFICATION":
        raise ValueError(
            "[PAPER CASE CHECK] "
            "Expected ACTIVE_ELECTRIFICATION_EXPERIMENT="
            "'FULL_ELECTRIFICATION', found "
            f"{ACTIVE_ELECTRIFICATION_EXPERIMENT!r}."
        )

    if set(ACTIVE_ELECTRIFICATION_SCENARIO_IDS) != {"FULL100"}:
        raise ValueError(
            "[PAPER CASE CHECK] Expected only electrification "
            f"scenario FULL100, found "
            f"{ACTIVE_ELECTRIFICATION_SCENARIO_IDS}."
        )


    # --------------------------------------------------------
    # 2) Number and identity of selected control cases
    # --------------------------------------------------------

    actual_case_labels = [
        cfg["case_label"]
        for cfg in SOLVE_RUN_CONFIGS
    ]

    # The expected cases are exactly those selected through
    # ACTIVE_CONTROL_GROUP.
    expected_case_labels = list(ACTIVE_CONTROL_CONFIG_KEYS)


    if len(SOLVE_RUN_CONFIGS) != len(expected_case_labels):
        raise ValueError(
            "[PAPER CASE CHECK] Wrong number of configurations.\n"
            f"Expected: {len(expected_case_labels)}\n"
            f"Found:    {len(SOLVE_RUN_CONFIGS)}\n"
            f"Cases:    {actual_case_labels}"
        )


    if set(actual_case_labels) != set(expected_case_labels):
        raise ValueError(
            "[PAPER CASE CHECK] Wrong case set.\n"
            f"Expected: {expected_case_labels}\n"
            f"Found:    {actual_case_labels}"
        )


    # --------------------------------------------------------
    # 3) Check BESS / EV / DHW control of every selected case
    # --------------------------------------------------------

    for cfg in SOLVE_RUN_CONFIGS:

        case_label = cfg["case_label"]

        if case_label not in EXPECTED_PAPER_CONTROLS:
            raise ValueError(
                f"[PAPER CASE CHECK] Unknown paper case: {case_label}"
            )

        actual_control = cfg["control"]
        expected_control = EXPECTED_PAPER_CONTROLS[
            case_label
        ]

        if actual_control != expected_control:
            raise ValueError(
                f"[PAPER CASE CHECK] {case_label}: "
                f"expected {expected_control}, "
                f"found {actual_control}."
            )


    # --------------------------------------------------------
    # 4) Print exact run matrix
    # --------------------------------------------------------

    df_preflight_cases = pd.DataFrame([
        {
            "case": cfg["case_label"],
            "mode": cfg["mode"],
            "BESS": cfg["control"]["BESS"],
            "EV": cfg["control"]["BEV"],
            "ESWH": cfg["control"]["DHW"],
            "electrification": cfg["electrification_id"],
            "run_revision": cfg["run_revision"],
            "run_uid": cfg["run_uid"],
        }
        for cfg in SOLVE_RUN_CONFIGS
    ])

    print("\n" + "=" * 90)
    print("[PAPER CASE CHECK] Configurations scheduled for optimization")
    print("=" * 90)
    print(df_preflight_cases.to_string(index=False))


    expected_n_solves = (
        len(SOLVE_RUN_CONFIGS)
        * len(seasons_dict.values())
    )

    print(
        f"\n[PAPER CASE CHECK] "
        f"{len(SOLVE_RUN_CONFIGS)} configurations x "
        f"{len(seasons_dict.values())} seasons = "
        f"{expected_n_solves} optimization problems."
    )



#----------------------------------------------------
# Sec 4.7 -- 5) Derive NEED_* and plotting selectors
#----------------------------------------------------
NEED_RB_PROFILES = (
    RUN_BUILD_RB_REFERENCE
    or RUN_RULE_BASED
    or any(
        _run_uses_any_rb(cfg["control"])
        for cfg in SOLVE_RUN_CONFIGS
    )
)


NEED_RB_CACHE = (
    NEED_RB_PROFILES
    or RUN_RB_PROFILE_DIAGNOSTICS
)


NEED_INPUT_TIMESERIES = (
    RUN_BUILD_10MIN_PROFILES
    or RUN_BUILD_RB_REFERENCE
    or RUN_RULE_BASED
    or RUN_SOLVE_MIXED_SCENARIOS
)

NEED_HAS_TECH = (
    NEED_INPUT_TIMESERIES
    or RUN_PLOT_SCENARIO_GRAPHS
    or RUN_COMPARE_DHW_SOC_TWO_CONFIGS
)

NEED_SOC_SECONDARY_TIMESERIES = (
    RUN_COMPARE_DHW_SOC_TWO_CONFIGS
    and not NEED_INPUT_TIMESERIES
)

NEED_LEGACY_PRICE_SHEETS = (
    RUN_RULE_BASED
    or RUN_ENABLE_LEGACY_SECTIONS
)

NEED_PF_LIMITS_INPUT = (
    RUN_SOLVE_MIXED_SCENARIOS
    and USE_PF_LINE_LIMITS_V1
)


# ------------------------------------------------------------
# By default, plotting and comparisons use the experiment selected
# in Sec. 4.2.
#
# The *_OVERRIDE variables in Sec. 4.3.5 may instead point to
# previously saved experiments/revisions.
# ------------------------------------------------------------

PLOT_EXPERIMENT_ID = (
    EXPERIMENT_ID
    if PLOT_EXPERIMENT_ID_OVERRIDE is None
    else str(PLOT_EXPERIMENT_ID_OVERRIDE)
)

PLOT_RUN_REVISION = (
    RUN_REVISION
    if PLOT_RUN_REVISION_OVERRIDE is None
    else str(PLOT_RUN_REVISION_OVERRIDE)
)


MULTIRUN_EXPERIMENT_ID = (
    EXPERIMENT_ID
    if MULTIRUN_EXPERIMENT_ID_OVERRIDE is None
    else str(MULTIRUN_EXPERIMENT_ID_OVERRIDE)
)

MULTIRUN_RUN_REVISION = (
    RUN_REVISION
    if MULTIRUN_RUN_REVISION_OVERRIDE is None
    else str(MULTIRUN_RUN_REVISION_OVERRIDE)
)


SOC_COMPARE_EXPERIMENT_ID = (
    EXPERIMENT_ID
    if SOC_COMPARE_EXPERIMENT_ID_OVERRIDE is None
    else str(SOC_COMPARE_EXPERIMENT_ID_OVERRIDE)
)

SOC_COMPARE_RUN_REVISION = (
    RUN_REVISION
    if SOC_COMPARE_RUN_REVISION_OVERRIDE is None
    else str(SOC_COMPARE_RUN_REVISION_OVERRIDE)
)


# ------------------------------------------------------------
# Sec 4.7 -- 5.2) Requested multi-run charts
# ------------------------------------------------------------

MULTIRUN_GRAPHS_TO_PLOT = []

if RUN_PLOT_MULTIRUN_COST_PERCENTILES:
    MULTIRUN_GRAPHS_TO_PLOT.append(
        "cost_percentiles"
    )

if RUN_PLOT_MULTIRUN_COST_DELTAS:
    MULTIRUN_GRAPHS_TO_PLOT.append(
        "cost_deltas"
    )

if RUN_PLOT_MULTIRUN_DEMAND_COVERAGE:
    MULTIRUN_GRAPHS_TO_PLOT.append(
        "demand_coverage"
    )

MULTIRUN_SAVE_UPDATED_CACHE = (
    RUN_SAVE_MULTIRUN_SUMMARY_CACHE
)


print(
    "[INFO][FLAGS] "
    f"NEED_INPUT_TIMESERIES={NEED_INPUT_TIMESERIES}, "
    f"NEED_RB_PROFILES={NEED_RB_PROFILES}, "
    f"NEED_HAS_TECH={NEED_HAS_TECH}, "
    f"NEED_SOC_SECONDARY_TIMESERIES="
    f"{NEED_SOC_SECONDARY_TIMESERIES}, "
    f"N_SOLVE_RUN_CONFIGS={len(SOLVE_RUN_CONFIGS)}"
)






#----------------------------------------------------
# Sec 4.7 -- 6) Initialize runtime containers
#----------------------------------------------------

# Household identifiers
ID = [
    f"c_{i}"
    for i in range(
        1,
        int(num_households) + 1,
    )
]

start_household = 1
end_household = int(num_households)
household_list = None


if (
    household_list is not None
    and len(household_list) > 0
):

    global_households = [
        c
        for c in household_list
        if c in ID
    ]

elif (
    start_household is not None
    and end_household is not None
):

    global_households = [
        f"c_{i}"
        for i in range(
            start_household,
            end_household + 1,
        )
        if f"c_{i}" in ID
    ]

else:

    global_households = ID[:]


if not global_households:
    global_households = ID[:]



# -------------------------
# Profile containers
# -------------------------
# Profile pre-loading
pv_fix_dicts = {}                           

ee_rev_series = {}                          
ee_cost_series = {}                                               

# Electric loads (base + additional)
fixed_dhw_electric_demand_dicts = {}
dmd_Cool_el_fix_dicts = {}
dmd_Heat_el_fix_dicts = {}
dmd_Cook_el_fix_dicts = {}
dmd_BEV_el_fix_dicts = {}
dmd_base_el_fix_dicts = {}

# Dictionaries for the "raw" profiles in kW (as exported from the Excel files)
pv_raw_dicts        = {}
dmd_base_raw_dicts  = {}
raw_dhw_demand_dicts   = {}
dmd_Cool_raw_dicts  = {}
dmd_Heat_raw_dicts  = {}
dmd_Cook_raw_dicts  = {}
dmd_BEV_raw_dicts   = {}

# Normalized dictionaries (PV series and electric loads normalized by their respective maximum value)
pv_pnom_perkwp_dicts = {}                   # <--- new: P_nom of PV plant per 1 kWp (kW per household)
dhw_nominal_power_dicts = {}
dmd_Cool_pnom_dicts = {}
dmd_Heat_pnom_dicts = {}
dmd_Cook_pnom_dicts = {}
dmd_BEV_pnom_dicts = {}
dmd_base_pnom_dicts = {}


# -----------------------
# PF / debug runtime state
# -----------------------
PF_LINE_LIMITS_MAP = {}
PF_INTERFACE_LIMITS_MAP = {}


# ------------------------------------------------------------
# Infeasibility / non-optimal solve runtime state
# ------------------------------------------------------------

# Each failure is identified by:
#     (RUN_UID, season)
#
# This is necessary because several configurations may fail
# in the same season without overwriting one another.
INFEASIBLE_DEBUG_REGISTRY = {}

LAST_INFEASIBLE_KEY = None
LAST_INFEASIBLE_SEASON = None

# Legacy variable retained for backward compatibility.
# With the new workflow it should remain False.
ABORT_ALL_RUNS = False

can_scan_hits = 0
can_scan_rows = []



#----------------------------------------------------
# Sec 4.7 -- 7) Initialize LV topology
#----------------------------------------------------

LV_TOPOLOGY_STATE = initialize_lv_topology(
    ID
)

KERBER_BP = (
    LV_TOPOLOGY_STATE["KERBER_BP"]
)

KERBER_BLUEPRINT_VALIDATION = (
    LV_TOPOLOGY_STATE[
        "KERBER_BLUEPRINT_VALIDATION"
    ]
)

KERBER_BUS_ID_TO_LABEL = (
    LV_TOPOLOGY_STATE[
        "KERBER_BUS_ID_TO_LABEL"
    ]
)

KERBER_LABEL_TO_BUS_ID = (
    LV_TOPOLOGY_STATE[
        "KERBER_LABEL_TO_BUS_ID"
    ]
)

KERBER_BUS_ROLE = (
    LV_TOPOLOGY_STATE[
        "KERBER_BUS_ROLE"
    ]
)

KERBER_LV_LABEL_TABLE = (
    LV_TOPOLOGY_STATE[
        "KERBER_LV_LABEL_TABLE"
    ]
)

HOUSE_TO_LV_BUS_LABEL = (
    LV_TOPOLOGY_STATE[
        "HOUSE_TO_LV_BUS_LABEL"
    ]
)

HOUSE_TO_PP_BUS_ID = (
    LV_TOPOLOGY_STATE[
        "HOUSE_TO_PP_BUS_ID"
    ]
)

LV_ROOT_LABEL = (
    LV_TOPOLOGY_STATE[
        "LV_ROOT_LABEL"
    ]
)

LV_BUS_LABELS_ALL = (
    LV_TOPOLOGY_STATE[
        "LV_BUS_LABELS_ALL"
    ]
)

LV_BUS_LABELS_EXPORT = (
    LV_TOPOLOGY_STATE[
        "LV_BUS_LABELS_EXPORT"
    ]
)


# Backward-compatible aliases still used by downstream code.
lv_root = LV_ROOT_LABEL
lv_bus_labels = LV_BUS_LABELS_EXPORT.copy()



#----------------------------------------------------
# Sec 4.7 -- 8) Initialize maximum technology endowment
#----------------------------------------------------
# Create a new has_tech endowment only when solving new configurations.
# If only charts/comparisons are being produced, has must be loaded in 4.9
# from the shared file saved by the original run.
RUN_CREATE_NEW_HAS_TECH = (
    RUN_SOLVE_MIXED_SCENARIOS
    or RUN_BUILD_RB_REFERENCE
)



if RUN_CREATE_NEW_HAS_TECH:

    random.seed(RANDOM_SEED_HAS_TECH)
    np.random.seed(RANDOM_SEED_HAS_TECH)

    # PV
    num_pv_households = int(round(num_households * PV_share / 100))
    has_PV = set(random.sample(ID, k=num_pv_households))

    # BESS
    n_BESS_target = int(round(num_households * BESS_share / 100))
    non_pv_households = list(set(ID) - has_PV)

    n_BESS_standalone_base = int(round(n_BESS_target * standalone_ratio))
    n_BESS_standalone_base = min(n_BESS_standalone_base, len(non_pv_households))

    has_BESS_standalone = set(random.sample(non_pv_households, k=n_BESS_standalone_base))

    remaining_BESS = n_BESS_target - len(has_BESS_standalone)
    num_BESS_in_PV = min(remaining_BESS, len(has_PV))
    has_BESS_in_PV = set(random.sample(list(has_PV), k=num_BESS_in_PV))

    has_BESS = has_BESS_standalone | has_BESS_in_PV

    remaining_BESS = n_BESS_target - len(has_BESS)

    if remaining_BESS > 0:
        non_pv_available = list(set(non_pv_households) - has_BESS_standalone)
        extra_BESS = min(remaining_BESS, len(non_pv_available))

        if extra_BESS > 0:
            extra_BESS_ids = set(random.sample(non_pv_available, k=extra_BESS))
            has_BESS |= extra_BESS_ids


    # HP
    has_HP = set(random.sample(
        ID,
        k=int(round(num_households * heat_pump_share / 100)),
    ))

    # Cooling
    has_cooler = set(random.sample(
        ID,
        k=int(round(num_households * cooler_share / 100)),
    ))

    # Induction
    has_induction = set(random.sample(
        ID,
        k=int(round(num_households * induction_share / 100)),
    ))

    # EV
    has_BEV = set(random.sample(
        ID,
        k=int(round(num_households * EV_share / 100)),
    ))

    # EV & V2G
    n_BEV_tot = len(has_BEV)
    n_V2G = int(round(n_BEV_tot * V2G_share / 100))
    n_V2G = min(n_V2G, n_BEV_tot)

    if n_V2G > 0:
        has_V2G = set(random.sample(list(has_BEV), k=n_V2G))
    else:
        has_V2G = set()

    has_BEV_only = has_BEV - has_V2G


    # ESWH Electric Storage Water Heater
    has_el_boiler = set(random.sample(
        ID,
        k=int(round(num_households * ESWH_share / 100)),
    ))

    has = {
        "PV": has_PV,
        "BESS": has_BESS,
        "BEV": has_BEV,
        "V2G": has_V2G,
        "DHW": has_el_boiler,
        "HP": has_HP,
        "COOL": has_cooler,
        "INDUCTION": has_induction,
    }

    print(
        "[INFO][HAS] Created new maximum "
        "technology endowment."
          )

else:

    has_PV = set()
    has_BESS = set()
    has_BEV = set()
    has_el_boiler = set()
    has_HP = set()
    has_cooler = set()
    has_induction = set()
    has_V2G = set()
    has_BEV_only = set()

    has = None

    print(
        "[INFO][HAS] New has_tech is not created. "
        "If required for previously saved results, "
        "it will be loaded later."
    )




#----------------------------------------------------
# Sec 4.7 -- 9) Dynamic paths
#----------------------------------------------------
# ----------------
# Compact identifiers for files shared across configurations
# Identifies the input dataset/population.
# Suitable for objects that do not depend on technology assignment,
# such as the real temporal indexes.
HAGRID_DATASET_TAG = _safe_fs_tag(
    f"{num_households}n_{selected_region}"
)

# Identifies the common physical technology endowment used by all MODEs.
# It is used in filenames instead of creating a long 00_shared folder.
HAGRID_CONTEXT_TAG = _safe_fs_tag(
    f"{HAGRID_DATASET_TAG}_"
    f"{_compact_share_tag_from_shares(_global_shares_dict())}_"
    f"{pv_power_tag}kWp_"
    f"s{RANDOM_SEED_HAS_TECH}"
)

# --------------------------
# output roots
# --------------------------
# Root new folders (Xls, Py, Pics, Power-Flow)
hagrid_results_xls_root_folder = os.path.join(output_folder, "04_HGD_res_xls", HAGRID_GRID_GROUP,)
hagrid_results_py_root_folder = os.path.join(output_folder, "04_HGD_res_py", HAGRID_GRID_GROUP)
hagrid_results_pictures_root_folder = os.path.join(output_folder, "04_HGD_res_pics", HAGRID_GRID_GROUP)

# Shared objects used across multiple run configurations.
hagrid_shared_py_folder = os.path.join(hagrid_results_py_root_folder, "_shared",)


# for power-flow outputs and new LV grid constraints
hagrid_powerflow_py_root_folder = os.path.join(output_folder, "04_HGD_for_PF_Py", HAGRID_GRID_GROUP,)
hagrid_pf_limits_root_folder = os.path.join(output_folder, "05_Power_Flow_Xlsx", HAGRID_GRID_GROUP)

# for comparison of different configurations
hagrid_multirun_root_folder = os.path.join(output_folder, "04_HGD_cmp_multi_runs", HAGRID_GRID_GROUP)


# -------------------------
# Central run catalog
# -------------------------
hagrid_run_catalog_root_folder = os.path.join(hagrid_powerflow_py_root_folder, "_catalog",)
hagrid_run_manifest_csv = os.path.join(hagrid_run_catalog_root_folder, "HAGRID_run_manifest.csv",)
hagrid_run_manifest_xlsx = os.path.join(hagrid_run_catalog_root_folder, "HAGRID_run_manifest.xlsx",)


# ------------------------
# Shared files
# ------------------------
# Maximum/global technology endowment used to derive the individual MODEs.
has_tech_path = os.path.join(hagrid_shared_py_folder, f"has_tech_{HAGRID_CONTEXT_TAG}.pkl.gz",)

# Human-readable list of households with BEV.
has_bev_path = os.path.join(hagrid_shared_py_folder, f"has_BEV_{HAGRID_CONTEXT_TAG}.txt",)

# Tabular export of the global technology assignment.
has_global_used_path = os.path.join(hagrid_shared_py_folder, f"has_global_used_{HAGRID_CONTEXT_TAG}.csv",)

# Real temporal indexes used by Module IV and potentially by Module V.
# Technology shares are not required in this filename.
idx_real_by_season_path = os.path.join(hagrid_powerflow_py_root_folder, f"IDX_REAL_BY_SEASON_{HAGRID_DATASET_TAG}.pkl.gz",)


# Mapping between original input columns from Modules 01-03
# and the standardized c_i household identifiers used by Module 04.
# The mapping does not depend on the active control configuration.
input_column_mappings_path = os.path.join(hagrid_shared_py_folder,(f"input_column_mappings_{HAGRID_DATASET_TAG}.pkl.gz"),)


# --------------
#   Sec 4.7 -- 9.2) Revision associated with the RB cache
# --------------
#
# When BUILDING:
#     always save under the current RUN_REVISION.
#
# When LOADING:
#     normally use RUN_REVISION;
#     optionally use an explicitly requested older revision.
# ---------------
if RUN_BUILD_RB_REFERENCE:

    if (
        RB_CACHE_RUN_REVISION_OVERRIDE is not None
        and str(RB_CACHE_RUN_REVISION_OVERRIDE)
        != str(RUN_REVISION)
    ):
        raise ValueError(
            "[RB CACHE] RB_CACHE_RUN_REVISION_OVERRIDE "
            "cannot be used while RUN_BUILD_RB_REFERENCE=True.\n"
            "New RB profiles must always be saved under "
            "the current RUN_REVISION."
        )

    RB_CACHE_RUN_REVISION = str(
        RUN_REVISION
    )

else:

    RB_CACHE_RUN_REVISION = (
        str(RUN_REVISION)
        if RB_CACHE_RUN_REVISION_OVERRIDE is None
        else str(RB_CACHE_RUN_REVISION_OVERRIDE)
    )


# --------------------------------------------------------
#   Sec 4.7 -- 9.3) RB cache path resolution
# Single source of truth for BUILD and LOAD
# --------------------------------------------------------

RB_CACHE_PATHS = build_rb_cache_paths(
    revision=RB_CACHE_RUN_REVISION,
)

RB_CACHE_UID = (
    RB_CACHE_PATHS[
        "rb_cache_uid"
    ]
)

RB_CACHE_PARAMETER_SIGNATURE = (
    RB_CACHE_PATHS[
        "rb_parameter_signature"
    ]
)

RB_CACHE_EXPERIMENT_UID = (
    RB_CACHE_PATHS[
        "experiment_uid"
    ]
)

RB_CACHE_REVISION_UID = (
    RB_CACHE_PATHS[
        "revision_uid"
    ]
)


hagrid_rb_cache_root_folder = (
    RB_CACHE_PATHS[
        "root"
    ]
)

rb_cache_experiment_folder = (
    RB_CACHE_PATHS[
        "experiment_folder"
    ]
)

rb_cache_revision_folder = (
    RB_CACHE_PATHS[
        "revision_folder"
    ]
)

rb_cache_folder = (
    RB_CACHE_PATHS[
        "cache_folder"
    ]
)

rb_cache_pickle_path = (
    RB_CACHE_PATHS[
        "pickle"
    ]
)

rb_cache_meta_path = (
    RB_CACHE_PATHS[
        "meta"
    ]
)

rb_cache_audit_folder = (
    RB_CACHE_PATHS[
        "audit"
    ]
)


print("[RB CACHE PATH]")
print("  revision     :", RB_CACHE_RUN_REVISION,)
print("  revision UID :", RB_CACHE_REVISION_UID,)
print("  context UID  :", RB_CACHE_UID,)
print("  param sig    :", RB_CACHE_PARAMETER_SIGNATURE,)
print("  folder       :", rb_cache_folder,)
print("  path length  :", len(os.path.abspath(rb_cache_pickle_path)),)











# Hourly price workbook used only by legacy/RB sections.
prices_file_path = os.path.join(prices_input_folder, f"{hourly_prices_file}.xlsx")


#----------------------------------------------------
# Sec 4.7 -- 10) Create folders
#----------------------------------------------------
folders_to_create = list(
    STATIC_FOLDERS_TO_CREATE
)

folders_to_create.extend([
    hagrid_results_xls_root_folder,
    hagrid_results_py_root_folder,
    hagrid_results_pictures_root_folder,
    hagrid_powerflow_py_root_folder,
    hagrid_pf_limits_root_folder,
    hagrid_multirun_root_folder,
    hagrid_run_catalog_root_folder,
])

if NEED_RB_CACHE:
    folders_to_create.extend([
        hagrid_rb_cache_root_folder,
        rb_cache_experiment_folder,
        rb_cache_revision_folder,
        rb_cache_folder,
        rb_cache_audit_folder,
    ])

# Remove duplicates while preserving order.
folders_to_create = list(
    dict.fromkeys(
        folder
        for folder in folders_to_create
        if folder is not None
    )
)

for folder in folders_to_create:
    os.makedirs(
        folder,
        exist_ok=True,
    )




#----------------------------------------------------
# Sec 4.7 -- 11) Validate input paths
#----------------------------------------------------
# Note:
# - paths_to_check remains empty if only result analyses/plotting charts 
#   of results already saved.
# - adding only necessary files to the active/current workflow.
# - the extended control is provided on an extended form.

paths_to_check = []

# Inputs required to rebuild 10-minute profiles.
if RUN_BUILD_10MIN_PROFILES:
    paths_to_check += [
        os.path.join(prices_input_folder, f"{hourly_prices_file}.xlsx"),
        os.path.join(pv_input_folder, f"{hourly_pv_file}.xlsx"),
    ]


# Complete inputs required for RB reference / optimization.
if NEED_INPUT_TIMESERIES:
    paths_to_check += [
        dhw_demand_file,
        cooling_demand_file,
        heating_demand_file,
        cooking_demand_file,
        ev_demand_file,
        file_charge_BEV_home_greedy,
        occupancy_file,
        base_demand_file,
        pv_10min_file_path,
        prices_10min_path,
        ev_metadata_file,
    ]


# Minimal additional inputs for SOC comparison only.
elif NEED_SOC_SECONDARY_TIMESERIES:
    paths_to_check += [
        pv_10min_file_path,
        prices_10min_path,
    ]


# Optional PF-derived constraints.
pf_limits_precheck_path = os.path.join(
    hagrid_pf_limits_root_folder,
    f"{PF_LIMITS_SHEET}_{HAGRID_DATASET_TAG}.xlsx",
)

if NEED_PF_LIMITS_INPUT:
    paths_to_check += [
        pf_limits_precheck_path,
    ]


# Remove duplicate paths.
paths_to_check = list(dict.fromkeys([
    p for p in paths_to_check
    if p is not None
]))

if paths_to_check:

    missing_paths = []
    existing_paths = []

    for path in paths_to_check:
        if os.path.exists(path):
            existing_paths.append(path)
        else:
            missing_paths.append(path)

    print(
        f"[INFO][PATH] Control path workflow active: "
        f"{len(existing_paths)} found, {len(missing_paths)} missing."
    )

    if missing_paths:
        print("[WARN][PATH] File/folders missing required by the active workflow:")
        for path in missing_paths:
            print(f"  - {path}")

    else:
        print("[INFO][PATH] All the paths required by the workflow are available.")

else:
    print("[INFO][PATH] No input timeseries required by the active workflow.")

  ## 4.8 Creation of 10-minute timeseries profiles (PV-prices)

In [ ]:
if RUN_BUILD_10MIN_PROFILES:
    # ------------------------------
    # Sec 4.8 -- 001) RANDOM SELECTION OF PV PROFILES for num_households households
    # ------------------------------
    
    # Read the hourly Excel file
    hourly_pv_profiles_df = pd.read_excel(os.path.join(pv_input_folder, f"{hourly_pv_file}.xlsx"), sheet_name='PV_profiles')
    hourly_pv_profiles_df['Datetime'] = pd.to_datetime(hourly_pv_profiles_df['Datetime'])

    # Obtain random profiles for each person/household, still hourly
    selected_random_hourly_pv_profiles_df = select_random_profile(num_households, hourly_pv_profiles_df)
    
    # Save the hourly PV profiles randomly assigned to each household
    selected_random_hourly_pv_profiles_df.to_excel(os.path.join(pv_input_folder, f'randomly_assigned_PV_hourly_profiles_per_{num_households}_households.xlsx'), index=False, engine='openpyxl')


    
    # ------------------------------
    # Sec 4.8 -- 002) CREATION OF 10-min PROFILES  (PV, electricity prices)
    # ------------------------------

    # ------------ 2.1 Electricity prices -------------------------------------------
    # Control parameters
    dt_min = 10
    factor = 60 // dt_min   # = 6 for dt_min=10

    # Path and reading of the hourly price file
    
    hourly_prices_df = pd.read_excel(prices_file_path)

    # Set the Datetime column as index, assuming it is exactly named 'Datetime'
    # Prices
    hourly_prices_df['Datetime'] = pd.to_datetime(hourly_prices_df['Datetime'], dayfirst=True, errors='coerce')
    hourly_prices_df = hourly_prices_df.set_index('Datetime')

    missing = [c for c in cols_required if c not in hourly_prices_df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}. Available columns: {list(hourly_prices_df.columns)}")

    
    # subset with copy to avoid warnings
    hourly_prices_df = hourly_prices_df[cols_required].copy()

    # Output
    prices_workbook_name = prices_10min_path
    

    with pd.ExcelWriter(prices_workbook_name, engine='openpyxl') as writer:
        # loop over seasons defined in season_periods (season_name -> start_date)
        for season_name, (start, end) in season_periods.items():
            start_ts = pd.to_datetime(start)
            hourly_index = pd.date_range(start=start_ts, periods=24*7, freq='h')  # 168 hours

            # extract the hourly block; if hours are missing -> fill with ffill/bfill and then residual 0
            blocco = hourly_prices_df.reindex(hourly_index)
            missing = blocco.index[blocco.isna().all(axis=1)]
            if len(missing) > 0:
                print(f"WARNING: for '{season_name}', {len(missing)} hours are missing in the annual hourly data. They will be filled with ffill/bfill and residual 0.")
                blocco = blocco.ffill().bfill().fillna(0.0)

            # value matrix (168, n_customers)
            arr = blocco.values  # shape (168, n)

            # repeat each row factor times (-> 1008 rows)
            arr_10min = np.repeat(arr, factor, axis=0)

            # sub-hourly index (1008 rows)
            index_10min = pd.date_range(start=start_ts, periods=24*7*factor, freq=f'{dt_min}min')

            # rebuild DataFrame
            df_10min = pd.DataFrame(arr_10min, index=index_10min, columns=blocco.columns)

            # clean sheet name, no forbidden characters and max 31 characters
            safe_sheet = season_name.replace('/', '_').replace('\\', '_').replace('*', '_').replace('?', '_')
            safe_sheet = safe_sheet.replace('[', '_').replace(']', '_').replace(':', '_')
            safe_sheet = safe_sheet[:31]

            # write sheet
            df_10min.to_excel(writer, sheet_name=safe_sheet, index=True)
            print(f"Sheet '{safe_sheet}' written: shape {df_10min.shape}")

    print(f"Workbook created: {prices_workbook_name}")
    #-----------------------------------------------------------------------------


    # ------------ 2.2 Photovoltaic PV -------------------------------------------
    # Path and reading of the hourly PV file, for the selected households
    pv_file_path = os.path.join(pv_input_folder, f"randomly_assigned_PV_hourly_profiles_per_{num_households}_households.xlsx")
    hourly_pv_df = pd.read_excel(pv_file_path)

    # PV
    hourly_pv_df['Datetime'] = pd.to_datetime(hourly_pv_df['Datetime'], dayfirst=True, errors='coerce')
    hourly_pv_df = hourly_pv_df.set_index('Datetime')
    hourly_pv_df = hourly_pv_df.loc[hourly_pv_df.index.notna()].copy()

    # Output
    pv_workbook_name = os.path.join(pv10_output_folder, f"{num_households}_households_{selected_region}_PV_10min_seasons_1kWp.xlsx")


    with pd.ExcelWriter(pv_workbook_name, engine='openpyxl') as writer:
        for season_name, (start, end) in season_periods.items():
        # for nome_stag, inizio in settimane_inizio.items():
            start_ts = pd.to_datetime(start)
            hourly_index = pd.date_range(start=start_ts, periods=24*7, freq='h')  # 168 hours

            # extract the hourly block; if hours are missing -> fill with ffill/bfill and then residual 0
            blocco = hourly_pv_df.reindex(hourly_index)
            missing = blocco.index[blocco.isna().all(axis=1)]
            if len(missing) > 0:
                print(f"WARNING: for '{season_name}', {len(missing)} hours are missing in the annual hourly data. They will be filled with ffill/bfill and residual 0.")
                blocco = blocco.ffill().bfill().fillna(0.0)

            # value matrix (168, n_customers)
            arr = blocco.values  # shape (168, n)

            # if data are hourly energy values (kWh), divide them by the number of intervals
            if treat_as_energy:
                arr = arr / factor

            # repeat each row factor times (-> 1008 rows)
            arr_10min = np.repeat(arr, factor, axis=0)

            # sub-hourly index (1008 rows)
            index_10min = pd.date_range(start=start_ts, periods=24*7*factor, freq=f'{dt_min}min')

            # rebuild DataFrame
            df_10min = pd.DataFrame(arr_10min, index=index_10min, columns=blocco.columns)

            # clean sheet name, no forbidden characters and max 31 characters
            safe_sheet = season_name.replace('/', '_').replace('\\', '_').replace('*', '_').replace('?', '_')
            safe_sheet = safe_sheet.replace('[', '_').replace(']', '_').replace(':', '_')
            safe_sheet = safe_sheet[:31]

            # write sheet
            df_10min.to_excel(writer, sheet_name=safe_sheet, index=True)
            print(f"Sheet '{safe_sheet}' written: shape {df_10min.shape}")

    print(f"Workbook created: {pv_workbook_name}")
    #-----------------------------------------------------------------------------

   ## 4.9 Import of dataset and time-series (PV, prices, demand profiles)

In [ ]:
# ---------------------------------------
#   Sec 4.9)    READING HAS_TECH, i.e. which technologies each individual customer/household has
# ---------------------------------------

if NEED_HAS_TECH or NEED_INPUT_TIMESERIES:

    # ---------------------------------------
    #  Sec 4.9 -- Here some datasets are imported/read
    #   that will be used both for optimization
    #   and for the "rule-based" case, e.g. BEV battery capacity
    # -----------------------------------------

    # Index used to avoid issues with the "canonical" index
    IDX_REAL_BY_SEASON = {} 

    # ----------------------------
    # Sec 4.9 -- 000) Define, or retrieve, which dwellings have which technology/storage
    # e.g. PV, BESS, BEV, HP...
    # ----------------------------
    # NOTE:
    # - global has = maximum physical/potential technology endowment, defined by the global shares.
    # - HAS_RUN = effective technology endowment of the single configuration, built later in 4.11c.


    path_has_tech = has_tech_path
    path_has_bev = has_bev_path
    

    # ------------------------------------------------------------------
    # CASE A: has has already been created in Sect. 4.7.
    # e.g. I want to solve the operation-optimization problem
    # ------------------------------------------------------------------
    if _has_global_is_valid(globals().get("has", None)):

        _set_has_aliases_from_has_dict(has)

        save_compressed_pickle(has, path_has_tech)

        print(f"[INFO][4.9] Using 'has' already available in memory and saving it in:")
        print(f"  {path_has_tech}")

        has_sets_initialized = True


    # ------------------------------------------------------------------
    # CASE B: has is not in memory, but a common has_tech file already exists.
    # e.g. I do not want to solve the operation-optimization problem, only print/plot charts
    # ------------------------------------------------------------------
    elif os.path.exists(path_has_tech):

        has_loaded = load_compressed_pickle(path_has_tech)

        if not _has_global_is_valid(has_loaded):
            raise ValueError(
                f"[ERROR][4.9] The has_tech file exists but does not have a valid structure:\n"
                f"  {path_has_tech}\n"
                f"Keys found: {list(has_loaded.keys()) if isinstance(has_loaded, dict) else type(has_loaded)}"
            )

        _set_has_aliases_from_has_dict(has_loaded)

        print(f"[INFO][4.9] Loaded shared has_tech from:")
        print(f"  {path_has_tech}")

        has_sets_initialized = True


    # ------------------------------------------------------------------
    # CASE C: legacy fallback.
    # e.g. I do not want to solve the optimization problem BUT I do not have
    # an old in-memory file to use for print/plot charts
    # This is only for backward compatibility with old runs.
    # ------------------------------------------------------------------
    else:

        allow_legacy_has_from_opt_pickle = (
            ALLOW_LEGACY_HAS_FROM_OPT_PICKLE
            and (not RUN_SOLVE_MIXED_SCENARIOS)
        )

        if not allow_legacy_has_from_opt_pickle:
            raise FileNotFoundError(
                "[ERROR][4.9] 'has' is not available either in memory or in the shared file.\n"
                f"File searched:\n  {path_has_tech}\n\n"
                "With the new mixed structure, I no longer rebuild has from flow_el_df_OPT.\n"
                "To solve this:\n"
                "1) rerun 4.7 so that has is created in memory; or\n"
                "2) create/save has_tech in the common HAGRID result root; or\n"
                "3) if you only want to make charts from old runs, make sure path_has_tech exists."
            )

        season0 = seasons[0]

        filename_pov_case_legacy = os.path.join(
            py_based_output_folder,
            f"flow_el_df_OPT_{num_households}_nuc_{selected_region}_{season0}.pkl.gz"
        )

        if not os.path.exists(filename_pov_case_legacy):
            raise FileNotFoundError(
                "[ERROR][4.9] Legacy fallback requested, but the file is missing:\n"
                f"  {filename_pov_case_legacy}"
            )

        df_raw_case = load_compressed_pickle(filename_pov_case_legacy)

        has_rebuilt = build_has_from_dfraw(df_raw_case)

        _set_has_aliases_from_has_dict(has_rebuilt)

        save_compressed_pickle(has, path_has_tech)

        print(f"[INFO][4.9] Rebuilt has from legacy OPT pickle and saved it in:")
        print(f"  {path_has_tech}")

        has_sets_initialized = True


    # ------------------------------------------------------------------
    # Save global EV list in the common HAGRID result root.
    # ------------------------------------------------------------------
    with open(path_has_bev, "w") as f:
        for c in sorted(has_BEV, key=lambda x: int(x.split("_")[1])):
            f.write(c + "\n")

    print(f"[INFO][4.9] Saved has_BEV in:")
    print(f"  {path_has_bev}")


else:
    print("[INFO][4.9] has_tech not required by the active workflow.")

# ----------------------------------------------------------------------------------------





# ---------------------------------------
#   Sec 4.9)    READING INPUT DATA, i.e. time series, technology parameters and demo charts
# ---------------------------------------

if NEED_INPUT_TIMESERIES:
    # ----------------------------
    # Sec 4.9 -- 001) Import EV metadata, battery capacity, household type
    # ----------------------------

    # 1.1) Import EV metadata: one row per person/EV owner.
    # Expected English columns:
    # person, household, has_ev, bev_capacity_kWh,
    # cons_kWh_per_km, soc_initial, energy_to_full_kWh.
    all_ev_metadata_df = pd.read_excel(
        ev_metadata_file
    )

    # Remove accidental spaces from column names.
    all_ev_metadata_df.columns = [
        str(column).strip()
        for column in all_ev_metadata_df.columns
    ]

    # 1.1.a) Dictionary:
    # Household_i -> household_type
    # Used to assign DHW/ESWH technical parameters.
    household_type_by_household = (
        extract_household_type_by_household_from_ev_metadata(
            all_ev_metadata_df
        )
    )

    # 1.1.b) Map Household_i -> c_i,
    # e.g. Household_3 -> c_3.
    household_type_by_case = {}

    for (
        household_id,
        household_type,
    ) in household_type_by_household.items():

        c_label = map_colname_to_c(
            household_id,
            prefix_out='c',
        )

        if (
            c_label is not None
            and c_label in ID
        ):
            household_type_by_case[c_label] = (
                household_type
            )

    # Check that all metadata household types are available
    # in the DHW specification dictionary.
    unknown_household_types = sorted(
        set(household_type_by_case.values())
        - set(dhw_specs.keys())
    )

    if unknown_household_types:
        raise ValueError(
            "The following household types were read from "
            "the EV metadata but are not defined in dhw_specs: "
            f"{unknown_household_types}"
        )

    # Identify households for which no type was found.
    # The following DHW section will use its fallback values
    # for these households.
    missing_household_types = sorted(
        set(ID) - set(household_type_by_case.keys()),
        key=lambda household: int(
            household.split('_')[1]
        ),
    )

    if missing_household_types:
        print(
            "[WARN][4.9] No household type found in EV "
            "metadata for: "
            f"{missing_household_types}. "
            "The DHW fallback values will be used."
        )

    # 1.1.c) Keep only rows corresponding to an EV.
    # This conversion works both with Boolean values and
    # strings such as True/False.
    if 'has_ev' in all_ev_metadata_df.columns:
        has_ev_mask = (
            all_ev_metadata_df['has_ev']
            .astype('string')
            .str.strip()
            .str.lower()
            .isin({
                'true',
                '1',
                'yes',
                'y',
            })
        )

        ev_metadata_df = (
            all_ev_metadata_df
            .loc[has_ev_mask]
            .copy()
        )

    else:
        print(
            "[WARN][4.9] Column 'has_ev' not found in "
            "EV metadata. All rows will be treated as "
            "EV rows."
        )

        ev_metadata_df = (
            all_ev_metadata_df.copy()
        )

    # 1.1.d) Check and convert EV battery capacity.
    if 'bev_capacity_kWh' not in ev_metadata_df.columns:
        raise KeyError(
            "Column 'bev_capacity_kWh' is missing from "
            "the EV metadata file."
        )

    ev_metadata_df['bev_capacity_kWh'] = (
        pd.to_numeric(
            ev_metadata_df['bev_capacity_kWh'],
            errors='coerce',
        )
    )

    invalid_capacity_mask = (
        ev_metadata_df['bev_capacity_kWh']
        .isna()
    )

    if invalid_capacity_mask.any():
        invalid_rows = ev_metadata_df.loc[
            invalid_capacity_mask,
            [
                'person',
                'household',
                'bev_capacity_kWh',
            ],
        ]

        raise ValueError(
            "Some EV battery capacities are missing "
            "or non-numeric:\n"
            + invalid_rows.to_string(index=True)
        )

    # Aggregate EV capacity by household.
    # If a household has multiple EVs, their capacities
    # are summed.
    ev_capacity_by_household = (
        ev_metadata_df
        .groupby(
            'household',
            sort=False,
        )['bev_capacity_kWh']
        .sum()
    )

    # 1.1.e) Map Household_i -> c_i.
    ev_capacity_by_case = (
        ev_capacity_by_household.copy()
    )

    ev_capacity_by_case.index = [
        map_colname_to_c(
            household_id,
            prefix_out='c',
        )
        for household_id
        in ev_capacity_by_case.index
    ]

    if any(
        household_id is None
        for household_id
        in ev_capacity_by_case.index
    ):
        raise ValueError(
            "At least one value in the EV metadata "
            "'household' column could not be mapped to c_i."
        )

    # 1.1.f) Align to the complete household ID list.
    # Households without an EV remain NaN at this stage.
    ev_capacity_by_case = (
        ev_capacity_by_case
        .reindex(ID)
    )

    print(
        "[INFO][4.9] EV metadata loaded: "
        f"{len(household_type_by_case)} household types "
        "and "
        f"{ev_capacity_by_case.notna().sum()} "
        "EV-equipped households."
    )


    # ----------------------------
    # Sec 4.9 -- 002) DHW parameters by household
    # ----------------------------
    capacity_dhw_by_house = {}
    power_limit_dhw_by_house = {}
    # standing_loss_dhw_by_house = {}
    loss_rate_dhw_by_house = {}

    for c in has_el_boiler:
        household_type = household_type_by_case.get(c, None)

        if (household_type is not None) and (household_type in dhw_specs):
            num_people, cap_kWh_DHW_c, dhw_power_kw_c, DHW_loss_rate_c = dhw_specs[household_type]
        else:
            num_people, cap_kWh_DHW_c, dhw_power_kw_c, DHW_loss_rate_c = (2, cap_kWh_DHW, dhw_power_kw, DHW_loss_rate)  # average fallback
            print(f"[WARN] DHW type not found for {c}: using cap={cap_kWh_DHW_c} kWh, power={dhw_power_kw_c} kW and loss_rate={DHW_loss_rate_c} 1/h")

        capacity_dhw_by_house[c] = float(cap_kWh_DHW_c)
        power_limit_dhw_by_house[c] = float(dhw_power_kw_c)
        # standing_loss_dhw_by_house[c] = float(standing_loss_c)
        loss_rate_dhw_by_house[c] = float(DHW_loss_rate_c)


    # --------------
    # Sec 4.9 -- 002.1) DHW household-specific initial SOC
    # --------------
    initial_soc_dhw_frac_by_house = {}

    # ------------------------------------------------------------
    # Group households by household type
    # ------------------------------------------------------------

    houses_by_dhw_type = {}

    for c in capacity_dhw_by_house:

        household_type = household_type_by_case.get(
            c,
            None,
        )

        houses_by_dhw_type.setdefault(
            household_type,
            [],
        ).append(c)


    # ------------------------------------------------------------
    # Build initial SOC for each household
    # ------------------------------------------------------------

    for household_type, houses in houses_by_dhw_type.items():

        # --------------------------------------------------------
        # Mean initial SOC of this household type
        # --------------------------------------------------------

        soc_mean = float(
            DHW_INITIAL_SOC_MEAN_BY_TYPE.get(
                household_type,
                initial_SoC_DHW,
            )
        )


        # --------------------------------------------------------
        # Physical range compatible with:
        #
        # - boiler initially OFF
        # - SOC above thermostat ON threshold
        # - SOC below full-charge threshold
        # --------------------------------------------------------

        lower_physical = (
            float(SOC_HYST_DEFAULT[0])
            + 1e-3
        )

        upper_physical = (
            float(SOC_HYST_DEFAULT[1])
            - 1e-3
        )


        # Explicit validation of the category mean
        if not (
            lower_physical
            <= soc_mean
            <= upper_physical
        ):
            raise ValueError(
                "[DHW INITIAL SOC] "
                f"Mean initial SOC for "
                f"'{household_type}' = {soc_mean:.4f} "
                "is outside the admissible initially-OFF "
                f"range [{lower_physical:.4f}, "
                f"{upper_physical:.4f}]."
            )


        n = len(houses)


        # ========================================================
        # OPTION A:
        # distributed household-specific initial SOC
        # ========================================================

        if DHW_INITIAL_SOC_DISTRIBUTED:

            half_range = float(
                DHW_INITIAL_SOC_HALF_RANGE_BY_TYPE.get(
                    household_type,
                    0.0,
                )
            )

            if half_range < 0.0:
                raise ValueError(
                    "[DHW INITIAL SOC] "
                    f"Negative half-range for "
                    f"'{household_type}': "
                    f"{half_range}"
                )


            # ----------------------------------------------------
            # Keep a SYMMETRIC interval around soc_mean.
            #
            # If the requested range would cross the thermostat
            # or full-charge boundaries, reduce it symmetrically.
            #
            # Therefore the population mean remains soc_mean.
            # ----------------------------------------------------

            half_range_eff = min(
                half_range,
                soc_mean - lower_physical,
                upper_physical - soc_mean,
            )

            half_range_eff = max(
                0.0,
                half_range_eff,
            )

            soc_low = (
                soc_mean
                - half_range_eff
            )

            soc_high = (
                soc_mean
                + half_range_eff
            )


            # ----------------------------------------------------
            # Deterministic quantile distribution
            #
            # No stochastic change between repeated HAGRID runs.
            # Population mean = soc_mean.
            # ----------------------------------------------------

            if n == 1:

                soc_values = np.array(
                    [soc_mean],
                    dtype=float,
                )

            else:

                quantiles = (
                    np.arange(
                        n,
                        dtype=float,
                    )
                    + 0.5
                ) / float(n)

                soc_values = (
                    soc_low
                    + quantiles
                    * (
                        soc_high
                        - soc_low
                    )
                )


            # ----------------------------------------------------
            # Deterministically assign the SOC values to households
            #
            # The seed changes only the household-value mapping,
            # not the distribution itself.
            # ----------------------------------------------------

            houses_ordered = sorted(
                houses,
                key=lambda c: hashlib.sha256(
                    (
                        f"{DHW_INITIAL_SOC_SEED}"
                        f"|{household_type}"
                        f"|{c}"
                    ).encode("utf-8")
                ).hexdigest(),
            )


            for c, soc_frac in zip(
                houses_ordered,
                soc_values,
            ):

                initial_soc_dhw_frac_by_house[c] = (
                    float(soc_frac)
                )


        # ========================================================
        # OPTION B:
        # no within-type distribution
        #
        # Every household of the same type starts from the
        # category mean.
        # ========================================================

        else:

            for c in houses:

                initial_soc_dhw_frac_by_house[c] = (
                    soc_mean
                )


    # ------------------------------------------------------------
    # Convert fractional SOC into thermal energy [kWh_th]
    # ------------------------------------------------------------

    initial_soc_dhw_by_house = {
        c: (
            initial_soc_dhw_frac_by_house[c]
            * float(
                capacity_dhw_by_house[c]
            )
        )
        for c in capacity_dhw_by_house
    }


    # ------------------------------------------------------------
    # Audit of generated initial SOC population
    # ------------------------------------------------------------

    print(
        "\n[DHW INITIAL SOC POPULATION]"
    )

    print(
        "  distributed:",
        DHW_INITIAL_SOC_DISTRIBUTED,
    )

    for household_type, houses in sorted(
        houses_by_dhw_type.items(),
        key=lambda x: str(x[0]),
    ):

        values = np.array(
            [
                initial_soc_dhw_frac_by_house[c]
                for c in houses
            ],
            dtype=float,
        )

        print(
            f"  {str(household_type):25s} | "
            f"N={len(values):3d} | "
            f"mean={100.0*np.mean(values):6.2f}% | "
            f"min={100.0*np.min(values):6.2f}% | "
            f"max={100.0*np.max(values):6.2f}%"
        )



    # ----------------------------------------
    # Sec 4.9 -- 003) Loading OCCUPANCY and defining EV home-charging availability profile
    # ----------------------------------------
    occ_sheets = pd.read_excel(occupancy_file, sheet_name=None)

    occ_by_season = {}
    for sheet_name, df_occ in occ_sheets.items():
        s_key = str(sheet_name).strip().lower()

        if "Datetime" not in df_occ.columns:
            raise KeyError(f"In occupancy sheet '{sheet_name}', column 'Datetime' is missing.")

        df_occ["Datetime"] = pd.to_datetime(df_occ["Datetime"])
        df_occ = df_occ.set_index("Datetime").sort_index()

        for col in df_occ.columns:
            df_occ[col] = pd.to_numeric(df_occ[col], errors="coerce").fillna(0).clip(0, 1).astype(int)

        occ_by_season[s_key] = df_occ

    # output: max occupancy profiles by season
    max_occupancy_profiles_by_season = {}


    #   3.b) Define OUTDOOR profiles, when not at home, for outdoor BEV charging
    df_outdoor_by_season = {}


    for s_key, df_occ in occ_by_season.items():
        df_temp = df_occ.copy().fillna(0.0)
        df_temp.index = pd.to_datetime(df_temp.index)

        # binarize
        df_bin = (df_temp >= 0.5).astype(float)

        # 1) rename/aggregate to c_i columns, sum or max; for occupancy, max is better
        # use helper already available in the notebook
        df_c, _ = rename_and_aggregate_households(df_bin, prefix_out="c")  # creates c_1, c_2, ...

        # if rename_and_aggregate_households sums members, bring it back to binary, max/clip
        df_c = (df_c >= 0.5).astype(float)

        # 2) outdoor = complement
        df_outdoor = (1.0 - df_c).astype(float)

        # 3) align columns to ID, filling missing values with 0
        df_outdoor = df_outdoor.reindex(columns=list(ID), fill_value=0.0)

        df_outdoor_by_season[s_key] = df_outdoor
    #-------------------------------------------------------------




    #------------------------------------------------------------
    # Sec 4.9 -- 004) READING DEMAND TIME SERIES, DHW, Cool, Heat, EV, Cook, Base
    # and PV TIME SERIES
    #-------------------------------------------------------------

    # 4.0.1) Read all price sheets, more efficient
    # cleaned season values, e.g. ['inverno', 'primavera', 'estate', 'autunno']
    cleaned_seasons = list(seasons_dict.values())

    # effective sheet names in the price file: "inverno SUM", "primavera SUM", etc.
    # sheet_names_prices = [f"{s} SUM" for s in seasons_pulite]
    price_sheet_names = [s for s in cleaned_seasons]


    # read ONLY those sheets
    raw_prices = pd.read_excel(
        prices_10min_path,
        sheet_name=price_sheet_names,
        index_col=0,
        parse_dates=True
    )

    prices_10min_by_season = {}


    if selected_region not in region_to_price_col:
        raise KeyError(
            f"Region '{selected_region}' not present in region_to_price_col. "
            f"Allowed values: {list(region_to_price_col.keys())}"
        )


    price_col = f"{region_to_price_col[selected_region]} [€/kWh_e]"



    for sheet_name, df in raw_prices.items():
        # retrieve the season name by removing " SUM" at the end
        season = sheet_name.replace(" SUM", "").strip()   # e.g. "inverno SUM" -> "inverno"

        # sanity check: must be one of the expected seasons
        if season not in cleaned_seasons:
            print(f"[WARN] Price sheet '{sheet_name}' not recognized as a season, skipping.")
            continue


        missing_cols = [c for c in ['PUN [€/kWh_e]', price_col] if c not in df.columns]
        if missing_cols:
            raise KeyError(
                f"Missing columns in price sheet '{sheet_name}': {missing_cols}. "
                f"Available columns: {df.columns.tolist()}"
            )

        pun = df['PUN [€/kWh_e]'].astype(float)

        # Zonal market price corresponding to the selected geographical area.
        zonal_price = df[price_col].astype(float)

        # Household electricity purchase price:
        # positive coefficient entering the minimization objective.
        df['Prezzo acquisto finale [€/kWh_e]'] = (
            pun + additional_fees
        )
        # Economic remuneration received for electricity exported from the household to the LV grid. (positive remuneration)        export_remuneration = (
        export_remuneration = (
            zonal_price * feed_in_revenue_ratio_to_zonal_price
        )

        # Revenues have to have a negative coefficient (oemof.solph adds variable_costs to the objective function.
        df['Prezzo vendita (revenues) [€/kWh_e]'] = ( -export_remuneration)

        # ------------------------------------------------------------
        # Revenue-sign consistency check
        # ------------------------------------------------------------

        export_objective_coefficient = (
            df['Prezzo vendita (revenues) [€/kWh_e]']
        )

        expected_export_objective_coefficient = (
            -zonal_price * feed_in_revenue_ratio_to_zonal_price
        )

        if not np.allclose(
            export_objective_coefficient.values,
            expected_export_objective_coefficient.values,
            rtol=0.0,
            atol=1e-12,
        ):
            raise ValueError(
                f"[PRICE SIGN ERROR] Export objective coefficient is not "
                f"consistent with the assumed feed-in remuneration in season "
                f"'{season}'."
            )


        print(
            f"[CHECK PRICES] season={season} | "
            f"buy price min/max="
            f"{df['Prezzo acquisto finale [€/kWh_e]'].min():.6f}/"
            f"{df['Prezzo acquisto finale [€/kWh_e]'].max():.6f} | "
            f"export remuneration min/max="
            f"{export_remuneration.min():.6f}/"
            f"{export_remuneration.max():.6f} | "
            f"export objective coefficient min/max="
            f"{export_objective_coefficient.min():.6f}/"
            f"{export_objective_coefficient.max():.6f}"
        )


        # here the key is the "clean" season: 'inverno', 'primavera', ...
        prices_10min_by_season[season] = df



    # --------------
    # 4.9.2) Profile pre-loading, already moved to Section 4.2
    # --------------
    # moved
    # ---------------


    # Store the correspondence between original input columns and
    # standardized Module-04 household identifiers for every season.
    INPUT_COLUMN_MAPPINGS_BY_SEASON = {}

    for s in seasons_dict.values():                                                                # seasons "inverno", "primavera", "autunno", "estate"
        
        s_key = str(s).strip().lower()   
        
        # ----------------------------
        # Sec 4.9 -- 4.1) load PV and loads by season
        # ----------------------------
        
        # read PV and demand: index_col=0 + parse_dates to obtain datetime index
        pv_df = pd.read_excel(pv_10min_file_path, sheet_name=s, index_col=0, parse_dates=True)            # seasons "inverno"
        # dmd_df = pd.read_excel(file_dmd_case_path, sheet_name=s, index_col=0, parse_dates=True) <------ If I have a single fully aggregated electric demand
        dhw_demand_df = pd.read_excel(dhw_demand_file, sheet_name=s, index_col=0, parse_dates=True)       # seasons "inverno"
        dmd_Cool_df = pd.read_excel(cooling_demand_file, sheet_name=s, index_col=0, parse_dates=True)     # seasons "inverno"
        dmd_Heat_df = pd.read_excel(heating_demand_file, sheet_name=s, index_col=0, parse_dates=True)     
        dmd_Cook_df = pd.read_excel(cooking_demand_file, sheet_name=s, index_col=0, parse_dates=True)
        dmd_BEV_df = pd.read_excel(ev_demand_file, sheet_name=s, index_col=0, parse_dates=True)
        
        # the base household demand is the one from activities, members, plus constant loads, refrigerator, standby, cooler
        sheet_base = f"{s} TOT"
        dmd_base_df = pd.read_excel(base_demand_file, sheet_name=sheet_base, index_col=0, parse_dates=True)     # <--- WARNING: individual-member loads do not include constant loads, fridge! It is necessary to import total household loads, NOT "100_nuc_Cen_ts_indiv_base.xlsx" but "100_nuc_Cen_ts_households_aggregati_base.xlsx", which however has sheets to read as "inverno TOT"

        # get the price sheet, if all sheets have already been loaded in all_prices_sheets
        if s not in prices_10min_by_season:
            raise KeyError(
                f"10-min prices not loaded for season '{s}'. "
                f"Available: {list(prices_10min_by_season.keys())}"
            )

        prices_df = prices_10min_by_season[s].copy()



        # use the price index as the "master" for all time series
        timeindex = prices_df.index

        # Associate the index with the real values, to avoid losing track of the canonical index, e.g. 01.01.2000
        # 1) build the real index robustly
        idx_real = ensure_datetime_index(prices_df.index, name=f"prices_df.index[{s_key}]")

        # 2) ALWAYS save the real index, use s_key for consistency
        IDX_REAL_BY_SEASON[s_key] = idx_real


        # ----------------------------
        # Sec 4.9 -- 4.2) realign everything on timeindex
        # ----------------------------

        # reindex EVERYTHING on this index before converting to kW
        pv_df       = pv_df.reindex(timeindex).interpolate().fillna(0)
        dhw_demand_df  = dhw_demand_df.reindex(timeindex).interpolate().fillna(0)
        dmd_Cool_df = dmd_Cool_df.reindex(timeindex).interpolate().fillna(0)
        dmd_Heat_df = dmd_Heat_df.reindex(timeindex).interpolate().fillna(0)
        dmd_Cook_df = dmd_Cook_df.reindex(timeindex).interpolate().fillna(0)
        dmd_BEV_df  = dmd_BEV_df.reindex(timeindex).interpolate().fillna(0)
        dmd_base_df = dmd_base_df.reindex(timeindex).interpolate().fillna(0)
        prices_df   = prices_df.reindex(timeindex).ffill().bfill()

        # ------ rename columns, Nucleo_i --> c_i: households already aggregated, one household = one column
        pv_df, pv_map = rename_axis_to_c(pv_df, axis='columns', prefix_out='c')
        dmd_base_df, dmd_map = rename_axis_to_c(dmd_base_df, axis='columns', prefix_out='c')
        # dmd_df, dmd_map = rename_axis_to_c(dmd_df, axis='columns', prefix_out='c')
        


        # ----------------------------
        # Sec 4.9 -- 4.3) rename household columns -> c_i
        # ----------------------------

        # ------ rename columns, Nucleo_i --> c_i: individual-member case, one member = one column -> to be aggregated
        dhw_demand_df, dhw_demand_map   = rename_and_aggregate_households(dhw_demand_df, prefix_out='c')
        dmd_Cool_df, dmd_Cool_map = rename_and_aggregate_households(dmd_Cool_df, prefix_out='c')
        dmd_Heat_df, dmd_Heat_map = rename_and_aggregate_households(dmd_Heat_df, prefix_out='c')
        dmd_Cook_df, dmd_Cook_map = rename_and_aggregate_households(dmd_Cook_df, prefix_out='c')
        dmd_BEV_df,  dmd_BEV_map  = rename_and_aggregate_households(dmd_BEV_df,  prefix_out='c')
        
        # Preserve input-column provenance for debugging,
        # reproducibility, and cross-module checks.
        INPUT_COLUMN_MAPPINGS_BY_SEASON[s_key] = {
            "PV": dict(pv_map),
            "base_demand": dict(dmd_map),
            "DHW": dict(dhw_demand_map),
            "cooling": dict(dmd_Cool_map),
            "heating": dict(dmd_Heat_map),
            "cooking": dict(dmd_Cook_map),
            "BEV": dict(dmd_BEV_map),
        }

        # ----------------------------
        # Sec 4.9 -- 4.4) conversion kWh/10' -> kW
        # ----------------------------
        pv_df        = convert_kwh10_to_kw(pv_df)
        dmd_base_df  = convert_kwh10_to_kw(dmd_base_df)
        dhw_demand_df   = convert_kwh10_to_kw(dhw_demand_df)
        dmd_Cool_df  = convert_kwh10_to_kw(dmd_Cool_df)
        dmd_Heat_df  = convert_kwh10_to_kw(dmd_Heat_df)
        dmd_Cook_df  = convert_kwh10_to_kw(dmd_Cook_df)
        dmd_BEV_df   = convert_kwh10_to_kw(dmd_BEV_df)
        # =========================================



        # ----------------------------
        # Sec 4.9 -- 4.5) checks, raw dictionaries and normalizations
        # ----------------------------

        # dimension / temporal-alignment check
        if not (
            len(pv_df)
            == len(dhw_demand_df)
            == len(dmd_Cool_df)
            == len(dmd_Heat_df)
            == len(dmd_Cook_df)
            == len(dmd_BEV_df)
            == len(dmd_base_df)
            == len(prices_df)
            == number_timesteps
        ):
            raise ValueError(
                f"Timestep mismatch for '{s}': "
                f"{len(pv_df)}, {len(dhw_demand_df)}, {len(dmd_Cool_df)}, "
                f"{len(dmd_Heat_df)}, {len(dmd_Cook_df)}, {len(dmd_BEV_df)}, "
                f"{len(dmd_base_df)}, {len(prices_df)}"
            )

        # column check
        missing_pv = set(ID) - set(pv_df.columns)
        missing_dhw_demand = set(ID) - set(dhw_demand_df.columns)
        missing_dmd_Cool = set(ID) - set(dmd_Cool_df.columns)
        missing_dmd_Heat = set(ID) - set(dmd_Heat_df.columns)
        missing_dmd_Cook = set(ID) - set(dmd_Cook_df.columns)
        missing_dmd_BEV = set(ID) - set(dmd_BEV_df.columns)
        missing_dmd_base = set(ID) - set(dmd_base_df.columns)

        if (
            missing_pv
            or missing_dhw_demand
            or missing_dmd_Cool
            or missing_dmd_Heat
            or missing_dmd_Cook
            or missing_dmd_BEV
            or missing_dmd_base
        ):
            raise KeyError(
                f"Missing columns for season {s}. "
                f"Missing in PV: {missing_pv}; "
                f"in DHW: {missing_dhw_demand}; "
                f"in Cooling: {missing_dmd_Cool}; "
                f"in Heating: {missing_dmd_Heat}; "
                f"in Cooking: {missing_dmd_Cook}; "
                f"in BEV: {missing_dmd_BEV}; "
                f"in dmd_el_base: {missing_dmd_base};"
            )

        # raw profiles in kW
        pv_raw_dicts[s]       = {c: pv_df[c].astype(float)       for c in ID}
        dmd_base_raw_dicts[s] = {c: dmd_base_df[c].astype(float) for c in ID}
        raw_dhw_demand_dicts[s]  = {c: dhw_demand_df[c].astype(float)  for c in ID}
        dmd_Cool_raw_dicts[s] = {c: dmd_Cool_df[c].astype(float) for c in ID}
        dmd_Heat_raw_dicts[s] = {c: dmd_Heat_df[c].astype(float) for c in ID}
        dmd_Cook_raw_dicts[s] = {c: dmd_Cook_df[c].astype(float) for c in ID}
        dmd_BEV_raw_dicts[s]  = {c: dmd_BEV_df[c].astype(float)  for c in ID}

        # normalized profiles for oemof/solph:
        #   fix 0-1 + separate nominal value
        pv_fix_dicts[s], pv_pnom_perkwp_dicts[s] = normalize_ts_dict(pv_raw_dicts[s])
        dmd_base_el_fix_dicts[s], dmd_base_pnom_dicts[s] = normalize_ts_dict(dmd_base_raw_dicts[s])
        fixed_dhw_electric_demand_dicts[s],  dhw_nominal_power_dicts[s]  = normalize_ts_dict(raw_dhw_demand_dicts[s])
        dmd_Cool_el_fix_dicts[s], dmd_Cool_pnom_dicts[s] = normalize_ts_dict(dmd_Cool_raw_dicts[s])
        dmd_Heat_el_fix_dicts[s], dmd_Heat_pnom_dicts[s] = normalize_ts_dict(dmd_Heat_raw_dicts[s])
        dmd_Cook_el_fix_dicts[s], dmd_Cook_pnom_dicts[s] = normalize_ts_dict(dmd_Cook_raw_dicts[s])
        dmd_BEV_el_fix_dicts[s],  dmd_BEV_pnom_dicts[s]  = normalize_ts_dict(dmd_BEV_raw_dicts[s])



        # price columns — select columns with exact name
        if price_col not in prices_df.columns or 'PUN [€/kWh_e]' not in prices_df.columns:
            raise KeyError(
                f"Price columns not found in sheet {s}. "
                f"Expected: ['PUN [€/kWh_e]', '{price_col}']. "
                f"Available columns: {prices_df.columns.tolist()}"
            )
        # Positive purchase-cost coefficient.
        ee_cost_series[s] = prices_df['Prezzo acquisto finale [€/kWh_e]'].astype(float)   # purchase cost

        # Signed export coefficient used by the minimization objective. Normally negative when exporting electricity generates a positive revenue.
        ee_rev_series[s] = prices_df['Prezzo vendita (revenues) [€/kWh_e]'].astype(float)   # feed-in revenue
        
        # Debug / sanity check.
        print(
            f"[CHECK EE PRICES] season={s} | "
            f"import coeff min/max="
            f"{ee_cost_series[s].min():.6f}/"
            f"{ee_cost_series[s].max():.6f} | "
            f"export coeff min/max="
            f"{ee_rev_series[s].min():.6f}/"
            f"{ee_rev_series[s].max():.6f}"
        )

        # ----------------------------
        # Sec 4.9 -- 4.6) OCCUPANCY profile for each season
        # ----------------------------  
        if s_key not in occ_by_season:
            raise KeyError(f"Occupancy sheet '{s_key}' not found. Available: {list(occ_by_season.keys())}")

        df_occ_sheet = occ_by_season[s_key]
        df_occ_s = df_occ_sheet.reindex(timeindex).fillna(0).astype(int)


        # ----------------------------
        # Sec 4.9 -- 4.7) Create BEV max charger profiles, from occupancy
        # ----------------------------

        # Create home-charging "availability" profile for BEV, from occupancy
        max_occupancy_profiles = {}
        for c in ID:
            max_occupancy_profiles[c] = build_profile_max_occupancy(
                occupancy_df=df_occ_s,
                idx=timeindex,
                c=c,
                current_season= None, #s,
                min_dwell_steps=min_dwell_steps_home,
                member=1
            )

        max_occupancy_profiles_by_season[s_key] = max_occupancy_profiles

        # ----------------------------
        # Sec 4.9 -- 4.9) DEBUG occupancy profiles, max on "charger"
        # ----------------------------
        # max profiles must have the same length as timeindex
        bad = [(c, len(max_occupancy_profiles[c]), len(timeindex)) for c in ID if len(max_occupancy_profiles[c]) != len(timeindex)]
        if bad:
            print("[ERROR] Max profiles not aligned for season", s_key, "->", bad[:10])

        zeros = [c for c in ID if max_occupancy_profiles[c].max() == 0]
        if zeros:
            print(f"[WARN] In season {s}, there are {len(zeros)} households with max profile always 0, examples:", zeros[:10])

        #-------------------------------------------------

    # Make sure values are DatetimeIndex
    for k, v in IDX_REAL_BY_SEASON.items():
        IDX_REAL_BY_SEASON[k] = ensure_datetime_index(v, name=f"IDX_REAL_BY_SEASON[{k}]")

    # Save mapping for debug/reuse
    pd.to_pickle(IDX_REAL_BY_SEASON, idx_real_by_season_path, compression="gzip",)

    print("[INFO][4.9] Saved real seasonal indexes in:\n"
        f"  {idx_real_by_season_path}")


    price_col_for_cost    = 'Prezzo acquisto finale [€/kWh_e]'
    price_col_for_revenue = 'Prezzo vendita (revenues) [€/kWh_e]'



    # Save the correspondence between original input columns
    # and standardized c_i identifiers.
    save_compressed_pickle(
        INPUT_COLUMN_MAPPINGS_BY_SEASON,
        input_column_mappings_path,
    )

    print(
        "[INFO][4.9] Saved input-column mappings in:\n"
        f"  {input_column_mappings_path}"
    )
    #--------------------------------------------------------------------------------------------------------

   ## 4.10 Conventional/Rule-Based profiles construction

In [ ]:
if RUN_RULE_BASED or RUN_BUILD_RB_REFERENCE:
    
    # ------------------------------------------------------------------
    # Sec 4.10c -- A) Loading / construction of the 'has' dictionary, technologies by household
    # ------------------------------------------------------------------
    if ("has" in globals()) and (has is not None) and isinstance(has, dict):
        print("[INFO] (RB) Using 'has' already available in memory.")
    else:
        if os.path.exists(path_has_tech):
            has = load_compressed_pickle(path_has_tech)  # in your notebook this function also loads pickle dictionaries
            print(f"[INFO] (RB) Loaded has_tech from: {path_has_tech}")
        else:
            raise RuntimeError(
                "[ERROR] RUN_RULE_BASED requires 'has', but the has_tech file was not found.\n"
                f"  path_has_tech: {path_has_tech}\n"
                "Solution: run an optimization (RUN_OPT == True), i.e. has_ initialization, before the Rule-Based run,\n"
                "and run again."
            )

    # robust normalization, some loaders may return list/tuple
    if not isinstance(has, dict):
        raise TypeError(f"[ERROR] loaded has_tech is not a dict: type={type(has)}")



    # ------------------------------------------------------------------
    # Sec 4.10c -- B) "Effective" technology sets, i.e. which houses have which technology
    # ------------------------------------------------------------------
    has_PV_set        = set(has.get('PV', set()) or set())
    has_BESS_set      = set(has.get("BESS", set()) or set())
    has_BEV_set       = set(has.get("BEV", set()) or set())
    has_el_boiler_set = set(has.get("DHW", set()) or set())
    has_HP_set        = set(has.get("HP", set()) or set())
    has_cooler_set    = set(has.get("COOL", set()) or set())
    has_induction_set = set(has.get("INDUCTION", set()) or set())

    # Also realign legacy variables if they are used below
    has_PV        = has_PV_set
    has_BESS      = has_BESS_set
    has_BEV       = has_BEV_set
    has_el_boiler = has_el_boiler_set
    has_HP        = has_HP_set
    has_cooler    = has_cooler_set
    has_induction = has_induction_set

    # Global dictionary: c_i -> flexibility category
    flex_category_by_c = {
        c: classify_flex_case(c, has_BESS_set, has_BEV_set, has_el_boiler_set)
        for c in ID
    }

    if DEBUG_RB:
        print(f"[RB] has: PV={len(has_PV_set)} BESS={len(has_BESS_set)} BEV={len(has_BEV_set)} DHW={len(has_el_boiler_set)}")


    # ------------------------------------------------------------
    # Sec 4.10c -- B.1) FLAGS, technologies really available and share > 0
    # ------------------------------------------------------------
    HAS_BEV_non_zero  = (EV_share > 0) and (has_BEV_set is not None) and (len(has_BEV_set) > 0)
    HAS_BESS_non_zero = (BESS_share > 0) and (has_BESS_set is not None) and (len(has_BESS_set) > 0)
    HAS_DHW_non_zero  = (ESWH_share > 0) and (has_el_boiler_set is not None) and (len(has_el_boiler_set) > 0)

    # If share=0, or the set is empty, force the sets to empty so downstream code never enters SOC blocks
    if not HAS_BEV_non_zero:
        has_BEV_set = set()
        has_BEV = has_BEV_set                   # Legacy alias: keep it until downstream functions only use has/HAS_RUN.

    if not HAS_BESS_non_zero:
        has_BESS_set = set()
        has_BESS = has_BESS_set                 # Legacy alias: keep it until downstream functions only use has/HAS_RUN.

    if not HAS_DHW_non_zero:
        has_el_boiler_set = set()
        has_el_boiler = has_el_boiler_set       # Legacy alias: keep it until downstream functions only use has/HAS_RUN.


    # Dynamic keys, excluding keys when that technology is not available
    extra_flow_names_active = ["econ_€"]
    if HAS_DHW_non_zero:
        extra_flow_names_active += [
            "SOC_DHW_prev_kWh",
            "SOC_DHW_after_loss_kWh",
            "SOC_DHW_raw_kWh",
            "SOC_DHW_kWh",
            "SOC_DHW_%",
            "P_el_to_DHW_kw",
            "DHW_boost_th_kWh",
        "DHW_boost_el_kw",
        "DHW_shortage_th_kWh",
        ]

    if HAS_BEV_non_zero:
        extra_flow_names_active += [
            "P_el_to_BEV_kw",
            "SOC_BEV_kWh",
            "SOC_BEV_%",
            "BEV_deficit_kWh",
            "BEV_outdoor_el_kWh",
            "BEV_outdoor_el_kw",
            "BEV_shortage_kWh",
        ]

    PROTECTED_KEYS_ACTIVE = set(extra_flow_names_active)



    # ------------------------------------------------------------------
    # Sec 4.10c -- C) Build RB dataframe for each season: PV, demand, prices, DHW, BEV-use
    # ------------------------------------------------------------------
    all_pv_df = {}
    df_dmd_all = {}
    all_prices_df = {}
    all_dhw_df = {}
    df_bev_use_all = {}


    for s in seasons_dict.values():
        # C.1) PV
        all_pv_df[s] = pd.DataFrame({
            c: pv_raw_dicts[s][c]     # if this is the raw PV in kW used in OPT
            for c in ID
            if c in pv_raw_dicts[s]
        })

        # C.2) base demand + cool/heat/cook, while keeping DHW and BEV separate, in storage
        dmd_tot = {}
        dhw_only_demand = {}
        dmd_bev_only = {}

        for c in ID:
            base = dmd_base_raw_dicts[s][c]    # same series as in the optimization
            cool = dmd_Cool_raw_dicts[s][c]    if c in has_cooler_set       and c in dmd_Cool_raw_dicts[s]  else 0.0
            heat = dmd_Heat_raw_dicts[s][c]    if c in has_HP_set           and c in dmd_Heat_raw_dicts[s]  else 0.0
            cook = dmd_Cook_raw_dicts[s][c]    if c in has_induction_set    and c in dmd_Cook_raw_dicts[s]  else 0.0


            # To keep track in cases where share_DHW or share_BEV = 0
            idx0 = base.index                 # <-- local master index to build "zero" series
            dhw  = (
                raw_dhw_demand_dicts[s][c]
                if HAS_DHW_non_zero and (c in has_el_boiler_set) and (c in raw_dhw_demand_dicts[s])
                else pd.Series(0.0, index=idx0)
            )

            bev  = (
                dmd_BEV_raw_dicts[s][c]
                if HAS_BEV_non_zero and (c in has_BEV_set) and (c in dmd_BEV_raw_dicts[s])
                else pd.Series(0.0, index=idx0)
            )


            # Non-flexible/conventional electric demand.
            # DHW and EV are kept separate because they are managed as ESWH/EV in RB profiles.
            # Do not sum DHW/EV here, otherwise they are double counted.
            dmd_tot[c] = base + cool + heat + cook

            # Save DHW and EV separately: this is needed for ESWH (electric storage water heater) SoC
            dhw_only_demand[c] = dhw
            dmd_bev_only[c] = bev

        df_dmd_all[s] = pd.DataFrame(dmd_tot)
        all_dhw_df[s] = pd.DataFrame(dhw_only_demand)
        df_bev_use_all[s] = pd.DataFrame(dmd_bev_only)

        # C.3) Prices: build a dataframe with the columns used in the rule-based case
        all_prices_df[s] = pd.DataFrame({
            'Prezzo acquisto finale [€/kWh_e]': ee_cost_series[s],
            'Prezzo vendita (revenues) [€/kWh_e]': ee_rev_series[s],
        })

        if all_prices_df[s].shape[0] > 0:
            # reference index = price index
            idx_master = all_prices_df[s].index
        else:
            # fallback: PV/demand intersection
            idx_master = df_dmd_all[s].index.intersection(all_pv_df[s].index)


        # C.4) master index and consistent reindexing
        all_pv_df[s]   = all_pv_df[s].reindex(idx_master).fillna(0.0)
        df_dmd_all[s]  = df_dmd_all[s].reindex(idx_master).fillna(0.0)
        all_prices_df[s] = all_prices_df[s].reindex(idx_master).ffill().bfill()
        all_dhw_df[s] = all_dhw_df[s].reindex(idx_master).fillna(0.0)
        df_bev_use_all[s] = df_bev_use_all[s].reindex(idx_master).fillna(0.0)


        # Debug assertions
        assert all_pv_df[s].index.equals(df_dmd_all[s].index)
        assert all_pv_df[s].index.equals(all_prices_df[s].index)

        # ---------------------------------------------------------
        # C.5) DHW input-profile sanity check
        # ---------------------------------------------------------

        steps_per_day = int(
            round(
                24.0 * 60.0
                / float(timestep_minutes)
            )
        )

        dhw_first_day = (
            all_dhw_df[s]
            .iloc[:steps_per_day]
        )

        eps_dhw = 1e-9

        n_dhw_events_first_day = int(
            (dhw_first_day > eps_dhw)
            .sum()
            .sum()
        )

        n_households_with_dhw_first_day = int(
            (dhw_first_day > eps_dhw)
            .any(axis=0)
            .sum()
        )

        dhw_energy_first_day_kwh = float(
            dhw_first_day
            .sum()
            .sum()
            * float(timestep_minutes)
            / 60.0
        )

        dhw_max_kw_first_day = float(
            dhw_first_day.max().max()
        )

        print(
            f"[CHECK DHW INPUT] {s} | "
            f"positive timesteps={n_dhw_events_first_day} | "
            f"households with DHW={n_households_with_dhw_first_day} | "
            f"E_DHW first day={dhw_energy_first_day_kwh:.2f} kWh_th | "
            f"max={dhw_max_kw_first_day:.2f} kW"
        )

        if n_dhw_events_first_day == 0:
            raise RuntimeError(
                f"[DHW INPUT ERROR] No positive DHW demand "
                f"found during first day of {s}."
            )



    # ------------------------------------------------------------------
    # Sec 4.10c -- D) Seasons and households to simulate
    # ------------------------------------------------------------------
    # check seasons in common
    common_seasons = sorted(set(all_pv_df.keys()).intersection(set(df_dmd_all.keys())))
    if len(common_seasons) == 0:
        print("WARNING: no season in common between PV and DMD files.")
        raise SystemExit("Stop execution: check profile loading.")

    # Choose households to simulate
    sample_st = common_seasons[0]
    cols_season = set(df_dmd_all[sample_st].columns)   # columns actually available, c_1, c_2, ...

    # use the global list, but only houses that actually exist in rule-based data
    households_to_run = [c for c in global_households if c in cols_season]

    # fallback: in the unlikely case where none of global_households is available,
    # use all columns matching ID
    if len(households_to_run) == 0:
        households_to_run = [c for c in cols_season if c in ID]


    # ---------------------------------------------------------
    # Sec 4.10c -- E) DHW: "effective" dictionaries, only houses with DHW
    # ---------------------------------------------------------
    capacity_dhw_by_house_eff = {
        c: capacity_dhw_by_house[c]
        for c in has_el_boiler_set
        if c in capacity_dhw_by_house
    }
    power_limit_dhw_by_house_eff = {
        c: power_limit_dhw_by_house[c]
        for c in has_el_boiler_set
        if c in power_limit_dhw_by_house
    }

    loss_rate_dhw_by_house_eff = {
        c: loss_rate_dhw_by_house[c]
        for c in has_el_boiler_set
        if c in loss_rate_dhw_by_house
    }


    initial_soc_dhw_by_house_eff = {
        c: initial_soc_dhw_by_house[c]
        for c in has_el_boiler_set
        if c in initial_soc_dhw_by_house
    }


    # ------------------------------------------------------------------
    # Sec 4.10c -- F) Loop over season -> loop over household -> DHW/BEV -> PV/BESS -> economics -> export
    # ------------------------------------------------------------------
    agg_per_season = {}
    flows_per_season = {}
    grid_net_ts_per_season = {}
    grid_net_detail_per_season = {}
    grid_net_duration_per_season = {}
    rb_profiles_by_season = {}                      # Container for all profiles determined in the RB case, e.g. BEV-BESS-DHW charge/discharge

    ymax_net_lv_rb_global = -np.inf
    ymin_net_lv_rb_global =  np.inf
    durcurve_rb_cache = {}

    ymax_mt_rb_global = -np.inf
    ymin_mt_rb_global =  np.inf
    durcurve_mt_rb_cache = {}


    for current_season in common_seasons:

        season_pv_df = all_pv_df[current_season]
        df_dmd_season = df_dmd_all[current_season]
        season_prices_df = all_prices_df.get(current_season)
    
        timeindex = df_dmd_season.index        # already equal to PV and prices
        assert season_pv_df.index.equals(timeindex)
        if season_prices_df is not None:
            assert season_prices_df.index.equals(timeindex)


        # initialize dataframes for each flow, same common index
        flow_dfs = {fn: pd.DataFrame(index=timeindex) for fn in flow_names}

        # ------------- F.0) Build economic part: prices and revenues ---------------------
        if season_prices_df is None:
            # fallback: constant zero series, or you may decide to skip the season
            revenue_sr = pd.Series(0.0, index=timeindex)
            cost_sr = pd.Series(0.0, index=timeindex)
            print(f"[WARN] prices for season {current_season} not found -> economics set to 0.")
        else:
            # no reindex is needed anymore: the index is already correct
            revenue_sr = season_prices_df[price_col_for_revenue].astype(float)
            cost_sr    = season_prices_df[price_col_for_cost].astype(float)

            # ---- Revenue sign check consistent with OPT:
            # the revenue series used in RB must have the same convention
            # as the ee_rev_series used to build ee_rev_can in the OPT case.
            if s in ee_rev_series:
                rev_origin = ee_rev_series[s].reindex(revenue_sr.index).ffill().bfill().astype(float)
                max_diff_rev = float((revenue_sr.astype(float) - rev_origin).abs().max())

                print(
                    f"[CHECK RB revenues] season={s} | "
                    f"revenue_sr min={float(revenue_sr.min()):.6f}, max={float(revenue_sr.max()):.6f} | "
                    f"ee_rev_series min={float(rev_origin.min()):.6f}, max={float(rev_origin.max()):.6f} | "
                    f"max_diff={max_diff_rev:.3e}"
                )
            else:
                print(f"[WARN][CHECK RB revenues] season {s} not found in ee_rev_series.")


        
        # --- household loop ---
        for household_id in households_to_run:
            if household_id not in df_dmd_season.columns:
                print(f" - {household_id} not found in demand columns ({current_season}), skipping")
                continue
            demand_sr = df_dmd_season[household_id]      # no reindex needed


            # -------------------- F.1) DHW RB: SoC + electric charging power + boost --------------------
            if HAS_DHW_non_zero and (household_id in has_el_boiler_set):

                # DHW profile
                dhw_kw = all_dhw_df[current_season][household_id].astype(float).reindex(timeindex).fillna(0.0)

                # Treat DHW as thermal demand over the timestep
                dhw_draw_kwh = dhw_kw * (float(timestep_minutes) / 60.0)

                # Boiler capacity, power and loss_rate for this household
                cap_dhw = float(capacity_dhw_by_house_eff.get(household_id, cap_kWh_DHW))
                pmax_dhw = float(power_limit_dhw_by_house_eff.get(household_id, dhw_power_kw))
                loss_rate_dhw = float(loss_rate_dhw_by_house_eff.get(household_id, DHW_loss_rate))


                # Initial SoC consistent with the specific capacity
                soc0_dhw = float(initial_soc_dhw_by_house_eff.get(household_id, initial_SoC_DHW * cap_dhw))

                # Conventional DHW boiler charging profile
                dhw_rb = rb_simulate_dhw_soc_and_charge(
                    idx=timeindex,
                    dhw_draw_kwh_ts=dhw_draw_kwh,
                    cap_kwh=cap_dhw,
                    pmax_kw=pmax_dhw,
                    eta=float(eta_DHW),
                    loss_rate= loss_rate_dhw,
                    soc0_kwh=soc0_dhw,
                    hyst_on=float(SOC_HYST_DEFAULT[0]),
                    hyst_off=float(SOC_HYST_DEFAULT[1]),
                    timestep_minutes=int(timestep_minutes),
                    pboost_kw=float(P_boost_DHW),   # same boost limit as OPT
                )

                # If RB boost is not enabled, zero the DHW boost profiles
                if not ENABLE_DHW_BOOST_RB:
                    dhw_rb["DHW_boost_th_kWh"] = dhw_rb["DHW_boost_th_kWh"] * 0.0
                    dhw_rb["DHW_boost_el_kw"] = dhw_rb["DHW_boost_el_kw"] * 0.0

                # Standard boiler charge is always a domestic electric load
                demand_sr = demand_sr + dhw_rb["P_el_to_DHW_kw"]

                # Additional DHW boost enters only if enabled
                if ENABLE_DHW_BOOST_RB:
                    demand_sr = demand_sr + dhw_rb["DHW_boost_el_kw"]


                for key in [
                    "SOC_DHW_prev_kWh",
                    "SOC_DHW_after_loss_kWh",
                    "SOC_DHW_raw_kWh",
                    "SOC_DHW_kWh",
                    "SOC_DHW_%",
                    "P_el_to_DHW_kw",
                    "DHW_boost_th_kWh",
                    "DHW_boost_el_kw",
                    "DHW_shortage_th_kWh",
                ]:
                    if key not in flow_dfs:
                        flow_dfs[key] = pd.DataFrame(index=timeindex)

                    flow_dfs[key].loc[timeindex, household_id] = (
                        pd.Series(dhw_rb[key])
                        .reindex(timeindex)
                        .ffill()
                        .fillna(0.0)
                        .astype(float)
                        .values
                    )



            # -------------------- F.2) BEV RB: SoC + electric charging power --------------------
            # if nuc in has_BEV_set:
            if HAS_BEV_non_zero and (household_id in has_BEV_set):

                # dt in hours, needed for kW -> kWh
                dt_hours = float(timestep_minutes) / 60.0

                # (F.2.1) BEV mobility profile
                bev_use_kw = df_bev_use_all[current_season][household_id].astype(float).reindex(timeindex).fillna(0.0)
                bev_use_kwh_ts = bev_use_kw * dt_hours

                # (F.2.2) max_profile, occupancy -> charging availability
                season_key = str(current_season).strip().lower()

                # home-charging availability, with dwell logic
                max_occupancy_profile = max_occupancy_profiles_by_season[season_key][household_id]
                max_profile = (
                    max_occupancy_profile
                    .reindex(timeindex)
                    .fillna(0.0)
                    .astype(float)
                    .clip(0.0, 1.0)
                )

                # raw home presence, without dwell logic
                home_presence_profile = build_home_presence_profile(
                    occupancy_df=occ_by_season[season_key],
                    idx=timeindex,
                    c=household_id,
                    current_season=current_season,
                    member=1,
                ).astype(float)

                # outdoor available only when away from home and not moving
                eps_move = 1e-9
                mask_not_moving = (bev_use_kwh_ts <= eps_move).astype(float)

                outdoor_profile = (
                    (1.0 - home_presence_profile) * mask_not_moving
                ).clip(lower=0.0, upper=1.0)


                # (F.2.3) BEV capacity by household
                cap_bev = ev_capacity_by_case.loc[household_id]
                if pd.isna(cap_bev) or float(cap_bev) <= 0:
                    cap_bev = float(default_ev_battery_capacity_kwh)                                    # fallback already defined in the parameters
                cap_bev = float(cap_bev)


                # (F.2.4) Initial SoC and target
                soc0_bev_kwh = float(initial_SoC_BEV) * cap_bev                                         # initial_SoC_BEV is a fraction
                soc_target_frac = float(SOC_BEV_TARGET_frac)                                            # <-- define it once above, e.g. 0.8

                # (F.2.5) efficiencies
                eta_ch_total = float(eta_charger) * float(EV_charge_discharge_efficiency)               # consistent with OPT: charger + storage 


                (
                    soc_bev_kwh_sr,
                    soc_bev_pct_sr,
                    p_el_to_bev_kw_sr,
                    deficit_bev_sr,
                    bev_outdoor_el_kwh_sr,
                    bev_outdoor_el_kw_sr,
                    bev_shortage_kwh_sr,
                ) = rb_simulate_bev_soc_and_charge_greedy_to_target(
                    idx=timeindex,
                    bev_use_kwh_ts=bev_use_kwh_ts,
                    max_profile=max_profile,
                    outdoor_profile=outdoor_profile,
                    cap_kwh=cap_bev,
                    pmax_kw=float(home_charging_power_kw),
                    pmax_outdoor_kw=float(P_outdoor_charger),
                    eta_ch=eta_ch_total,
                    eta_dis=float(EV_charge_discharge_efficiency),
                    loss_rate=float(BEV_loss_rate) if "BEV_loss_rate" in globals() else 0.0,
                    soc0_kwh=soc0_bev_kwh,
                    soc_target_pct=soc_target_frac,
                    outdoor_trigger_pct=float(SOC_BEV_OUTDOOR_TRIGGER_frac),
                    outdoor_target_pct=float(SOC_BEV_OUTDOOR_TARGET_frac),
                    min_soc_pct=float(min_storage_level_BEV),

                    enable_planned_foresight=bool(ENABLE_BEV_PLANNED_FORESIGHT_RB),
                    planned_lookahead_hours=float(BEV_PLANNED_TRIP_LOOKAHEAD_H),
                    long_trip_consumption_frac=float(BEV_LONG_TRIP_CONSUMPTION_frac),
                    very_long_trip_consumption_frac=float(BEV_VERY_LONG_TRIP_CONSUMPTION_frac),
                    long_trip_target_pct=float(SOC_BEV_LONG_TRIP_TARGET_frac),
                    very_long_trip_target_pct=float(SOC_BEV_VERY_LONG_TRIP_TARGET_frac),

                    min_home_ts=int(min_dwell_steps_home),
                    timestep_minutes=int(timestep_minutes),
                    enable_outdoor=bool(ENABLE_BEV_OUTDOOR_RB),
                )

                # (F.2.6) BEV charge is additional electric demand
                demand_sr = demand_sr + p_el_to_bev_kw_sr

                # (F.2.7) save into flow_dfs for Excel/plot
                for key in [
                    "P_el_to_BEV_kw",
                    "SOC_BEV_kWh",
                    "SOC_BEV_%",
                    "BEV_deficit_kWh",
                    "BEV_outdoor_el_kWh",
                    "BEV_outdoor_el_kw",
                    "BEV_shortage_kWh",
                ]:
                    if key not in flow_dfs:
                        flow_dfs[key] = pd.DataFrame(index=timeindex)

                flow_dfs["P_el_to_BEV_kw"].loc[timeindex, household_id] = pd.Series(p_el_to_bev_kw_sr).reindex(timeindex).fillna(0.0).astype(float).values
                flow_dfs["SOC_BEV_kWh"].loc[timeindex, household_id]   = pd.Series(soc_bev_kwh_sr).reindex(timeindex).ffill().fillna(0.0).astype(float).values
                flow_dfs["SOC_BEV_%"].loc[timeindex, household_id]     = pd.Series(soc_bev_pct_sr).reindex(timeindex).ffill().fillna(0.0).astype(float).values
                flow_dfs["BEV_deficit_kWh"].loc[timeindex, household_id] = pd.Series(deficit_bev_sr).reindex(timeindex).fillna(0.0).astype(float).values
                flow_dfs["BEV_outdoor_el_kWh"].loc[timeindex, household_id] = pd.Series(bev_outdoor_el_kwh_sr).reindex(timeindex).fillna(0.0).astype(float).values
                flow_dfs["BEV_outdoor_el_kw"].loc[timeindex, household_id] = pd.Series(bev_outdoor_el_kw_sr).reindex(timeindex).fillna(0.0).astype(float).values
                flow_dfs["BEV_shortage_kWh"].loc[timeindex, household_id] = pd.Series(bev_shortage_kwh_sr).reindex(timeindex).fillna(0.0).astype(float).values


            # -------------------- F.3) PV only if the house has PV, consistent with has_PV ------------
            if household_id in has_PV and household_id in season_pv_df.columns:
                pv_1kwp_sr = season_pv_df[household_id]
            elif household_id in has_PV:
                # if for some reason the column is missing, use the first profile as fallback
                pv_1kwp_sr = season_pv_df.iloc[:, 0]
            else:
                # no PV for this house
                pv_1kwp_sr = pd.Series(0.0, index=timeindex)
            
            #  -------------------- F.4) BESS, only for houses with BESS -----------------------------
            if HAS_BESS_non_zero and (household_id in has_BESS_set):
                capacity_kwh      = BESS_capacity_kwh
                p_charge_max_kw   = BESS_power_kw
                p_discharge_max_kw= BESS_power_kw
            else:
                # no BESS -> capacity and powers 0 -> the function reduces to "PV + grid"
                capacity_kwh      = 0.0
                p_charge_max_kw   = 0.0
                p_discharge_max_kw= 0.0

            out = simulate_bess_series(
                demand_sr          = demand_sr,
                pv_1kwp_sr         = pv_1kwp_sr,
                PV_power_kwp         = PV_power_kwp,        # if it is a dictionary, here use PV_power_kwp[nuc]
                capacity_kwh       = capacity_kwh,
                eta_ch             = eta_BESS,
                eta_dis            = eta_BESS,
                soc0_kwh           = 0.0,
                p_charge_max_kw    = p_charge_max_kw,
                p_discharge_max_kw = p_discharge_max_kw,
                timestep_minutes   = timestep_minutes,
                data_is_power      = data_is_power
                )


            # --- save "core" flows without overwriting protected ones ---
            for fn in flow_names:
                # DO NOT overwrite flows calculated externally, DHW/BEV/econ etc.
                if fn in PROTECTED_KEYS_ACTIVE:
                    continue

                if fn in out.columns:
                    flow_dfs[fn][household_id] = out[fn].reindex(timeindex).fillna(0.0)
                else:
                    # safety: if missing, use a zero series, avoids DataFrame "all scalar values" errors
                    flow_dfs[fn][household_id] = pd.Series(0.0, index=timeindex)

            # -------------------- F.5) Calculate ECONOMICS ----------------------------------------------
            rev_aligned = revenue_sr    
            cost_aligned = cost_sr      

            # Convention consistent with OPT:
            # - cost_aligned > 0
            # - rev_aligned <= 0 if the revenue series is already negative,
            #   as ee_rev_can used in variable_costs in the OPT case.
            #
            # If rev_aligned is already negative:
            #   net cost = import_cost + negative_export_revenue
            #
            # If instead rev_aligned is positive, keep the old logic:
            #   net cost = import_cost - positive_export_revenue
            if float(pd.Series(rev_aligned).max()) <= 1e-10:
                econ_ts = (
                    cost_aligned.values * out["grid_import_kwh"].values
                    + rev_aligned.values * out["grid_export_kwh"].values
                )
            else:
                print(
                    f"[WARN][RB ECON] rev_aligned contains positive values in season {s}. "
                    "Using old convention: import_cost - export_revenue."
                )
                econ_ts = (
                    cost_aligned.values * out["grid_import_kwh"].values
                    - rev_aligned.values * out["grid_export_kwh"].values
                )

            # save the economic series by household inside flow_dfs.
            # Otherwise, the economics_sum_week sheet is not created.
            if "econ_€" not in flow_dfs:
                flow_dfs["econ_€"] = pd.DataFrame(index=timeindex)

            flow_dfs["econ_€"].loc[:, household_id] = pd.Series(
                econ_ts,
                index=timeindex
            ).reindex(timeindex).fillna(0.0).astype(float).values



        

        # -------------------- F.6) calculate AGGREGATES and SAVE Excel / chart ----------------------
        agg_df = pd.DataFrame(index=timeindex)
        all_names = list(flow_names) + list(extra_flow_names)
        
        for fn in all_names:
            df_flow = flow_dfs.get(fn, pd.DataFrame(index=timeindex))
            if df_flow.shape[1] == 0:
                agg_df[fn + "_total"] = 0.0
            else:
                agg_df[fn + "_total"] = df_flow.sum(axis=1, skipna=True)

        # after the existing loop over flow_names, add
        # calculate econ_total if available
        if 'econ_€' in flow_dfs and flow_dfs['econ_€'].shape[1] > 0:
            agg_df['econ_€_total'] = flow_dfs['econ_€'].sum(axis=1, skipna=True)
        else:
            agg_df['econ_€_total'] = 0.0

        # =========================
        #  Sec 4.10c -- DURATION CURVES - GRID NET, by timestep
        #  F.6.1) for each household: net = import - export
        #  F.6.2) sum over all N households -> time series (n_ts,1)
        #  F.6.3) sort to obtain the duration curve, save sheets and chart
        # =========================

        # make sure import/export dataframes are available, otherwise create empty ones with zeros
        df_imp = flow_dfs.get('grid_import_kwh', pd.DataFrame(index=timeindex)).reindex(timeindex).fillna(0.0)
        df_exp = flow_dfs.get('grid_export_kwh', pd.DataFrame(index=timeindex)).reindex(timeindex).fillna(0.0)

        # F.6.1) balance by household: import - export, DataFrame columns = Nucleo_X
        grid_net_per_household_df  = df_imp.subtract(df_exp, fill_value=0.0)

        # kWh/timestep -> kW
        conv_factor = 60.0 / float(timestep_minutes)
        grid_net_per_household_kw_df = grid_net_per_household_df * conv_factor

        # Sum at secondary-substation / MV-grid level
        s_net_mt_total = grid_net_per_household_df.sum(axis=1)
        s_net_mt_total_kw = s_net_mt_total * conv_factor

        # Exchange share for an "average" household
        s_net_mt_avg_kw = s_net_mt_total_kw / float(num_households)

        # duration curves on MV grid
        dur_mt_total = duration_curve_from_series(s_net_mt_total_kw)
        dur_mt_avg   = duration_curve_from_series(s_net_mt_avg_kw)
        
        
        
        
        # duration curves, net LV import-export, by household, kW
        net_dict_rb = {c: pd.to_numeric(grid_net_per_household_kw_df[c], errors='coerce').fillna(0.0) for c in grid_net_per_household_kw_df.columns}
        lv_net_duration_df_rb = build_duration_curves(net_dict_rb)

        # Global limits for the "RB – duration curve percentiles LV" chart
        p95_series_rb = lv_net_duration_df_rb.quantile(0.95, axis=1)
        p05_series_rb = lv_net_duration_df_rb.quantile(0.05, axis=1)


        # base: LV percentiles, no arbitrary caps
        ymax_net_lv_rb_global = max(ymax_net_lv_rb_global, float(p95_series_rb.max()))
        ymin_net_lv_rb_global = min(ymin_net_lv_rb_global, float(p05_series_rb.min()))

        # include MV avg overlay, same unit kW
        dur_mt_avg_for_ylim = duration_curve_from_series(s_net_mt_avg_kw)
        ymax_net_lv_rb_global = max(ymax_net_lv_rb_global, float(dur_mt_avg_for_ylim.max()))
        ymin_net_lv_rb_global = min(ymin_net_lv_rb_global, float(dur_mt_avg_for_ylim.min()))


        # F.6.2) horizontal sum over all households -> time series (n_ts,1) in kWh per timestep
        # total time series, used for Excel export
        df_grid_net_total_kwh = pd.DataFrame(s_net_mt_total, columns=['grid_net_kwh_total'])
        df_grid_net_total_kw  = pd.DataFrame(s_net_mt_total_kw, columns=['grid_net_kw_total'])

        # update global limits
        ymax_here_rb_mt = max(float(dur_mt_total.max()), float(dur_mt_avg.max()))
        ymin_here_rb_mt = min(float(dur_mt_total.min()), float(dur_mt_avg.min()))

        ymax_mt_rb_global = max(ymax_mt_rb_global, ymax_here_rb_mt)
        ymin_mt_rb_global = min(ymin_mt_rb_global, ymin_here_rb_mt)

        
        # ----------------------------------------------------------------------
        # Sec 4.10c -- Cache for second pass, charts
        # ----------------------------------------------------------------------
        # LV: percentile distribution for all households, + average MV series for overlay
        durcurve_rb_cache[current_season] = {
            "df_dur_net_bt": lv_net_duration_df_rb,
            "s_net_mt_avg_kw": s_net_mt_avg_kw,
        }

        # MV: district and average-household duration curves
        durcurve_mt_rb_cache[current_season] = {
            "dur_mt_total": dur_mt_total,
            "dur_mt_avg": dur_mt_avg
        }
        # ----------------------------------------------------------------


        # F.6.3) duration curve, entire neighborhood / MV grid: sort values in kW descending, highest net imports first
        duration_sorted_kw = s_net_mt_total_kw.sort_values(ascending=False)

        df_grid_net_duration = pd.DataFrame({
            'grid_net_kw_total_sorted': duration_sorted_kw.values,
            'original_timestamp': duration_sorted_kw.index
        })

       # also add the sorted kWh version, useful for saving/consistency
        df_grid_net_duration['grid_net_kwh_total_sorted'] = (duration_sorted_kw.values / conv_factor)

 
        # calculate some useful indicators, remain in kWh if summed over timesteps
        total_import_energy_kwh = df_grid_net_total_kwh[df_grid_net_total_kwh['grid_net_kwh_total'] > 0]['grid_net_kwh_total'].sum()
        total_export_energy_kwh = -df_grid_net_total_kwh[df_grid_net_total_kwh['grid_net_kwh_total'] < 0]['grid_net_kwh_total'].sum()

        # save dataframes in dictionaries to use them after the loop, centralized plotting
        grid_net_ts_per_season[current_season] = {
            'kwh': df_grid_net_total_kwh,
            'kw' : df_grid_net_total_kw
        }
        grid_net_detail_per_season[current_season] = grid_net_per_household_df   # detail in kWh/timestep by household
        grid_net_duration_per_season[current_season] = df_grid_net_duration
        
        out_fname = os.path.join(
            rb_cache_audit_folder,
            f"RB_reference_{current_season.replace(' ', '_')}.xlsx",
        )

        os.makedirs(
            rb_cache_audit_folder,
            exist_ok=True,
        )
        
        
        
        # ------------ F.6.4) Create summary Excel ---------------------------------
        with pd.ExcelWriter(out_fname, engine="openpyxl",) as writer:
            for fn, df_flow in flow_dfs.items():
                if df_flow.shape[1] == 0:
                    continue

                df_flow.to_excel(writer,sheet_name=fn[:31],)
            agg_df.to_excel(writer, sheet_name="aggregated_flows",)





            # ----- save ECONOMICS:
            # 1) economics_sum_ts   = total cost aggregated over time
            # 2) economics_sum_week = weekly/seasonal cost by household
            if "econ_€" in flow_dfs and flow_dfs["econ_€"].shape[1] > 0:
                econ_df_detail = flow_dfs["econ_€"].copy()
            else:
                print(
                    f"[WARN][RB ECON] flow_dfs does not contain econ_€ for {current_season}. "
                    "Creating economics_sum_week with zeros to avoid an error in the comparison."
                )
                econ_df_detail = pd.DataFrame(
                    0.0,
                    index=agg_df.index,
                    columns=households_to_run
                )

            # Time series aggregated over all households
            df_economic = pd.DataFrame(
                econ_df_detail.sum(axis=1, skipna=True),
                columns=["econ_€_total"]
            )
            df_economic.to_excel(writer, sheet_name="economics_sum_ts")

            # Total by household
            summary = pd.DataFrame({
                "total_€": econ_df_detail.sum(axis=0).sort_values(ascending=False)
            })
            summary.index.name = 'Household'

            summary.to_excel(writer, sheet_name="economics_sum_week")

            # ---- Flexibility-category information for each household ----
            try:
                df_flex_info = pd.DataFrame({
                    'household': households_to_run,
                    'flex_case': [flex_category_by_c.get(c, 'NA') for c in households_to_run],
                    'flex_label': [flex_category_labels.get(flex_category_by_c.get(c, 'NA'), '') for c in households_to_run]
                })
                df_flex_info.to_excel(writer, sheet_name='flex_categories', index=False)
            except Exception:
                pass



            # ---- save DURATION CURVES / grid net
            # total time series (n_t,1)
            # save total time series, kWh and kW
            try:
                grid_net_ts_per_season[current_season]['kwh'].to_excel(writer, sheet_name='grid_net_ts_kwh'[:31])
            except Exception:
                pass
            try:
                grid_net_ts_per_season[current_season]['kw'].to_excel(writer, sheet_name='grid_net_ts_kw'[:31])
            except Exception:
                pass

            # detail by household, kWh/timestep
            try:
                grid_net_detail_per_season[current_season].to_excel(writer, sheet_name='grid_net_detail'[:31])
            except Exception:
                pass

            # duration curve, in kW + kWh
            try:
                grid_net_duration_per_season[current_season].to_excel(writer, sheet_name='grid_net_duration'[:31], index=False)
            except Exception:
                pass

        # instead of plotting here, store results so they can all be plotted with the same scale
        agg_per_season[current_season] = agg_df
        flows_per_season[current_season] = flow_dfs




        # --------------------------------------
        # Sec 4.10c -- G) optional debugging part
        # --------------------------------------
        
        if DEBUG_RB:
            if "P_el_to_BEV_kw" in flow_dfs and flow_dfs["P_el_to_BEV_kw"].shape[1] > 0:
                mx = float(flow_dfs["P_el_to_BEV_kw"].max().max())
                nz = int((flow_dfs["P_el_to_BEV_kw"].abs() > 1e-6).sum().sum())
                print(f"[RB] {current_season}: BEV P_el max={mx:.3f} nonzero={nz}")
            if "P_el_to_DHW_kw" in flow_dfs and flow_dfs["P_el_to_DHW_kw"].shape[1] > 0:
                mx = float(flow_dfs["P_el_to_DHW_kw"].max().max())
                nz = int((flow_dfs["P_el_to_DHW_kw"].abs() > 1e-6).sum().sum())
                print(f"[RB] {current_season}: DHW P_el max={mx:.3f} nonzero={nz}")



        # --------------------------------------
        # Sec 4.10c -- 7) Save RB profiles for use in the HAGRID model
        # --------------------------------------

        idx = timeindex

        # 1) PV (kW)                                                                # <--- Could also be omitted, commented out, already imported in HAGRID (Opt)
        pv_kw = season_pv_df.reindex(idx).fillna(0.0)

        # 2) Demands (kW) - those "base/cool/heat/cook" already composed, does NOT include BEV charge and DHW charge, it is not demand_sr
        # <--- Could also be omitted, commented out, already imported in HAGRID (Opt)
        dmd_baseCCH_kw = df_dmd_season.reindex(idx).fillna(0.0)

        # 3) BEV demand (kW) and charger (kW), if available
        bev_demand_kw = df_bev_use_all[current_season].reindex(idx).fillna(0.0)
        bev_charge_kw = flow_dfs.get("P_el_to_BEV_kw", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)

        # 3b) Optional BEV outdoor charging, and battery shortage that must be compensated
        bev_outdoor_kw = flow_dfs.get("BEV_outdoor_el_kw", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        bev_outdoor_kwh = flow_dfs.get("BEV_outdoor_el_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        bev_shortage_kwh = flow_dfs.get("BEV_shortage_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)


        # 4) DHW demand (kW) + standard charge + boost
        dhw_demand_kw = all_dhw_df[current_season].reindex(idx).fillna(0.0)
        dhw_charge_kw = flow_dfs.get("P_el_to_DHW_kw", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        dhw_boost_kw = flow_dfs.get("DHW_boost_el_kw", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)

        # 4a) DHW state of charge for storage and additional boost, if required
        dhw_soc_prev_kwh = flow_dfs.get("SOC_DHW_prev_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        dhw_soc_after_loss_kwh = flow_dfs.get("SOC_DHW_after_loss_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        dhw_soc_raw_kwh = flow_dfs.get("SOC_DHW_raw_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        dhw_boost_th_kwh = flow_dfs.get("DHW_boost_th_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        dhw_shortage_th_kwh = flow_dfs.get("DHW_shortage_th_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)



        # 5) BESS charge/discharge, kWh per timestep
        pv_to_bess_input_kwh = flow_dfs.get("pv_to_batt_input_kwh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)           # Energy sent from PV/grid to BESS, before eta_charge
        bess_charge_kwh = flow_dfs.get("energy_added_to_soc_kwh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)             # Energy actually charged into the BESS, after eta_charge
        bess_dis_from_soc_kwh = flow_dfs.get("batt_draw_from_soc_kwh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)        # Energy drawn from the battery, before eta_discharge
        bess_to_load_kwh = flow_dfs.get("batt_delivered_to_load_kwh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)         # Energy actually re-injected into the domestic grid from battery discharge, after eta_discharge


        # 5bis) State of Charge SOC
        bess_soc_kwh = flow_dfs.get("soc_kwh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)

        dhw_soc_kwh = flow_dfs.get("SOC_DHW_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        dhw_soc_pct = flow_dfs.get("SOC_DHW_%", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)

        bev_soc_kwh = flow_dfs.get("SOC_BEV_kWh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        bev_soc_pct = flow_dfs.get("SOC_BEV_%", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)



        # 6a) total required electric demand
        dmd_total_el_kw = (
            dmd_baseCCH_kw
            .add(bev_charge_kw, fill_value=0.0)
            .add(dhw_charge_kw, fill_value=0.0)
            .add(dhw_boost_kw, fill_value=0.0)
        )

        # 6) Grid import/export, kWh per timestep                    # Includes loads for demand, base+cook+cool+heat + DHW and BEV charge!
        grid_import_kwh = flow_dfs.get("grid_import_kwh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)
        grid_export_kwh = flow_dfs.get("grid_export_kwh", pd.DataFrame(index=idx)).reindex(idx).fillna(0.0)

        # 7) Net LV (kW)
        conv_factor = 60.0 / float(timestep_minutes)
        grid_net_kw = (grid_import_kwh - grid_export_kwh) * conv_factor

        rb_profiles_by_season[current_season] = {
            "index": idx,
            "pv_kw": pv_kw,
            "dmd_base_kw": dmd_baseCCH_kw,                  # electric demand including base+cook+cool+heat
            "dmd_total_el_kW": dmd_total_el_kw,             # electric demand including base+cook+cool+heat + BEV charge + DHW charge
            
            "bev_demand_kw": bev_demand_kw,
            "bev_charge_kw": bev_charge_kw,
            "bev_outdoor_kw": bev_outdoor_kw,               # Power from charger_outdoor station
            "bev_outdoor_kwh": bev_outdoor_kwh,             # Electric energy drawn from outdoor chargers during away-and-stationary charging
            "bev_shortage_kwh": bev_shortage_kwh,           # Unserved mobility energy due to insufficient BEV battery energy

            "dhw_demand_kw": dhw_demand_kw,
            "dhw_charge_kw": dhw_charge_kw,
            "pv_to_bess_input_kwh": pv_to_bess_input_kwh,               # Amount from PV/grid sent to the battery, gross of eta_charge
            "dhw_boost_kw": dhw_boost_kw,
            "dhw_soc_prev_kwh": dhw_soc_prev_kwh,                       # DHW SOC gross of losses and imports
            "dhw_soc_after_loss_kwh": dhw_soc_after_loss_kwh,           # DHW SOC net of losses
            "dhw_soc_raw_kwh": dhw_soc_raw_kwh,                         # DHW SOC accounting for actual DHW demand; if <0, external boiler support through DHW boost is needed
            "dhw_boost_th_kwh": dhw_boost_th_kwh,                       # Thermal energy required from DHW boost to meet demand not covered by the boiler
            "dhw_shortage_th_kwh": dhw_shortage_th_kwh,                 # possible shortage if boiler + DHW_boost, with limited power, are not sufficient

            "bess_charge_kwh": bess_charge_kwh,                     # Amount actually charging the battery
            "bess_discharge_from_soc_kwh": bess_dis_from_soc_kwh,   # Amount "leaving" the BESS, gross of eta_discharge
            "bess_to_load_kwh": bess_to_load_kwh,                   # Amount actually returned to the domestic grid
            
            "bess_soc_kwh": bess_soc_kwh,
            "dhw_soc_kwh": dhw_soc_kwh,
            "dhw_soc_pct": dhw_soc_pct,
            "bev_soc_kwh": bev_soc_kwh,
            "bev_soc_pct": bev_soc_pct,


            "grid_import_kwh": grid_import_kwh,
            "grid_export_kwh": grid_export_kwh,
            "grid_net_kw": grid_net_kw,
            "has": has,  # useful to save it together for consistency
        }





    # ------------------------------------------------------------------
    # Sec 4.10c -- After the RB loop by season: normalize global limits
    # ------------------------------------------------------------------

    # --- net LV (RB) ---
    if not np.isfinite(ymax_net_lv_rb_global):
        ymax_net_lv_rb_global = 1.0
    if not np.isfinite(ymin_net_lv_rb_global):
        ymin_net_lv_rb_global = 0.0

    span_lv = ymax_net_lv_rb_global - ymin_net_lv_rb_global
    if span_lv <= 0:
        span_lv = max(1.0, abs(ymax_net_lv_rb_global) if ymax_net_lv_rb_global != 0 else 1.0)

    ymax_net_lv_rb_global += 0.05 * span_lv
    ymin_net_lv_rb_global -= 0.05 * span_lv

    # --- net MV (RB) ---
    if not np.isfinite(ymax_mt_rb_global):
        ymax_mt_rb_global = 1.0
    if not np.isfinite(ymin_mt_rb_global):
        ymin_mt_rb_global = 0.0

    span_mt = ymax_mt_rb_global - ymin_mt_rb_global
    if span_mt <= 0:
        span_mt = max(1.0, abs(ymax_mt_rb_global) if ymax_mt_rb_global != 0 else 1.0)

    ymax_mt_rb_global += 0.05 * span_mt
    ymin_mt_rb_global -= 0.05 * span_mt


    # ============================================================
    # Save RB reference cache
    # ============================================================

    path_rb_profiles_pkl = (
        rb_cache_pickle_path
    )

    os.makedirs(
        rb_cache_folder,
        exist_ok=True,
    )


    # Compressed pickle
    save_compressed_pickle(
        rb_profiles_by_season,
        path_rb_profiles_pkl,
    )


    # Metadata / passport of the RB cache
    rb_cache_meta = {

        "created_at":
            datetime.datetime.now().isoformat(
                timespec="seconds"
            ),

        "experiment_id":
            str(EXPERIMENT_ID),

        "run_revision":
            str(RUN_REVISION),

        "rb_cache_revision":
            str(RB_CACHE_RUN_REVISION),

        "rb_cache_uid":
            str(RB_CACHE_UID),

        "grid_group":
            str(HAGRID_GRID_GROUP),

        "lv_archetype":
            str(LV_ARCHETYPE),

        "n_households":
            int(num_households),

        "selected_region":
            str(selected_region),

        # retained only as human-readable legacy/context information
        "hagrid_context_tag":
            str(HAGRID_CONTEXT_TAG),

        "context_identity":
            _canonical_rb_cache_context_payload(),

        "rb_parameter_signature":
            _rb_cache_parameter_signature(),

        "rb_parameters":
            _rb_cache_parameter_snapshot(),

        "seasons":
            sorted(
                str(s)
                for s
                in rb_profiles_by_season.keys()
            ),

        "profile_keys_by_season": {
            str(season): sorted(
                str(k)
                for k
                in profiles.keys()
            )
            for season, profiles
            in rb_profiles_by_season.items()
        },
    }


    with open(
        rb_cache_meta_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            _json_safe(rb_cache_meta),
            f,
            indent=2,
            ensure_ascii=False,
        )


    print(
        "[OK][RB CACHE] Saved RB reference profiles:"
    )
    print(
        f"  revision : {RB_CACHE_RUN_REVISION}"
    )
    print(
        f"  RB UID   : {RB_CACHE_UID}"
    )
    print(
        f"  profiles : {rb_cache_pickle_path}"
    )
    print(
        f"  metadata : {rb_cache_meta_path}"
    )
    # ---------------------


    print(f"[OK] Saved RB reference profiles in: {path_rb_profiles_pkl}")



    # Get the current time and format it without seconds
    now = datetime.datetime.now()
    current_hour = now.strftime('%Y-%m-%d_%H-%M')
    print(f"Info {current_hour}")
    print("Simulation completed for all seasons. Now building charts using uniform scales.")




# ============================================================
# Sec 4.10.1 -- Load previously saved RB reference profiles
# ============================================================

if (
    NEED_RB_CACHE
    and not RUN_BUILD_RB_REFERENCE
    and not RUN_RULE_BASED
):

    path_rb_profiles_pkl = (
        rb_cache_pickle_path
    )


    print(
        "\n[RB CACHE LOAD CHECK]"
    )

    print("  cwd          :",os.getcwd(),)
    print("  output root  :",os.path.abspath(output_folder),)
    print("  revision     :",RB_CACHE_RUN_REVISION,)
    print("  context UID  :",RB_CACHE_UID,)
    print("  parameter sig:",RB_CACHE_PARAMETER_SIGNATURE,)
    print("  expected file:",os.path.abspath(path_rb_profiles_pkl),    )
    print("  path length  :",len(os.path.abspath(path_rb_profiles_pkl)),)
    print("  exists       :",os.path.isfile(path_rb_profiles_pkl),)


    if not os.path.isfile(
        path_rb_profiles_pkl
    ):

        raise FileNotFoundError(
            "\n[RB CACHE] Required RB reference "
            "profiles were not found.\n\n"
            f"Current RUN_REVISION:\n"
            f"  {RUN_REVISION}\n\n"
            f"Requested RB revision:\n"
            f"  {RB_CACHE_RUN_REVISION}\n\n"
            f"RB context UID:\n"
            f"  {RB_CACHE_UID}\n\n"
            f"Expected file:\n"
            f"  {path_rb_profiles_pkl}\n\n"
            "Run once with "
            "RUN_BUILD_RB_REFERENCE=True "
            "to generate the cache."
        )


    # --------------------------------------------------------
    # Load compressed RB cache
    # --------------------------------------------------------

    rb_profiles_by_season = (
        load_compressed_pickle(
            path_rb_profiles_pkl
        )
    )


    # --------------------------------------------------------
    # Metadata validation
    # --------------------------------------------------------

    if os.path.isfile(
        rb_cache_meta_path
    ):

        with open(
            rb_cache_meta_path,
            "r",
            encoding="utf-8",
        ) as f:

            rb_meta_loaded = (
                json.load(f)
            )


        saved_uid = str(
            rb_meta_loaded.get(
                "rb_cache_uid",
                "",
            )
        )

        if saved_uid != str(
            RB_CACHE_UID
        ):
            raise RuntimeError(
                "[RB CACHE] UID mismatch.\n"
                f"Expected: {RB_CACHE_UID}\n"
                f"Saved:    {saved_uid}"
            )


        saved_revision = str(
            rb_meta_loaded.get(
                "rb_cache_revision",
                "",
            )
        )

        if saved_revision != str(
            RB_CACHE_RUN_REVISION
        ):
            raise RuntimeError(
                "[RB CACHE] Revision mismatch.\n"
                f"Expected: {RB_CACHE_RUN_REVISION}\n"
                f"Saved:    {saved_revision}"
            )


        # When no explicit old revision is requested,
        # verify that current RB parameters still match
        # those used to build the cache.
        if (
            RB_CACHE_RUN_REVISION_OVERRIDE
            is None
        ):

            saved_signature = str(
                rb_meta_loaded.get(
                    "rb_parameter_signature",
                    "",
                )
            )

            current_signature = (
                _rb_cache_parameter_signature()
            )

            if (
                saved_signature
                and saved_signature
                != current_signature
            ):

                raise RuntimeError(
                    "[RB CACHE] Current RB parameters "
                    "do not match the saved cache, "
                    "although RUN_REVISION is unchanged.\n\n"
                    f"Saved signature:   {saved_signature}\n"
                    f"Current signature: {current_signature}\n\n"
                    "Either rebuild the RB cache or "
                    "increment RUN_REVISION."
                )


    # --------------------------------------------------------
    # Season completeness check
    # --------------------------------------------------------

    expected_seasons = set(
        seasons_dict.values()
    )

    cached_seasons = set(
        rb_profiles_by_season.keys()
    )

    missing_seasons = (
        expected_seasons
        - cached_seasons
    )

    if missing_seasons:

        raise RuntimeError(
            "[RB CACHE] Cache is incomplete. "
            "Missing seasons: "
            f"{sorted(missing_seasons)}"
        )


    print(
        "[OK][RB CACHE] Loaded RB reference profiles"
    )
    print(
        f"  current RUN_REVISION : {RUN_REVISION}"
    )
    print(
        f"  RB cache revision    : {RB_CACHE_RUN_REVISION}"
    )
    print(
        f"  RB UID               : {RB_CACHE_UID}"
    )
    print(
        f"  file                 : {rb_cache_pickle_path}"
    )



### 4.10.2 Rule-Based profile diagnostics / validation

In [ ]:
# ============================================================
# Sec 4.10.2 -- Rule-Based profile diagnostics / validation
# ============================================================

if RUN_RB_PROFILE_DIAGNOSTICS:

    rows = []
    shortage_events = []
    soc_percentiles_by_season = {}

    for season, rp in rb_profiles_by_season.items():

        # --------------------------------------------------------
        # Sec 4.10 -- 2.1) EV domestic charging demand
        # --------------------------------------------------------
        ev_kw = (
            rp["bev_charge_kw"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
            .sum(axis=1)
        )

        # --------------------------------------------------------
        # Sec 4.10 -- 2.2) DHW electric demand, including possible boost
        # --------------------------------------------------------
        dhw_kw = (
            rp["dhw_charge_kw"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
            .sum(axis=1)
        )

        if "dhw_boost_kw" in rp:
            dhw_kw = (
                dhw_kw
                + rp["dhw_boost_kw"]
                .apply(pd.to_numeric, errors="coerce")
                .fillna(0.0)
                .sum(axis=1)
            )

        # --------------------------------------------------------
        # Sec 4.10 -- 2.3) Base household demand:
        # base + cooking + cooling + heating
        # --------------------------------------------------------
        base_kw = (
            rp["dmd_base_kw"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
            .sum(axis=1)
        )

        # --------------------------------------------------------
        # Sec 4.10 -- 2.4) BEV shortage / outdoor diagnostics
        # --------------------------------------------------------
        shortage = (
            rp["bev_shortage_kwh"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        outdoor = (
            rp["bev_outdoor_kwh"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        soc_pct = (
            rp["bev_soc_pct"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        n_ev_charging = (
            rp["bev_charge_kw"] > 1e-9
        ).sum(axis=1)

        shortage_by_house = shortage.sum(axis=0)

        ev_plus_dhw_kw = ev_kw + dhw_kw

        total_rb_load_kw = (
            base_kw
            + ev_kw
            + dhw_kw
        )


        t_peak = ev_plus_dhw_kw.idxmax()
        t_peak_total = total_rb_load_kw.idxmax()

        shortage_total_kwh = float(
            shortage.sum().sum()
        )

        rows.append({
            "season": season,

            "EV_charge_t0_kW":
                float(ev_kw.iloc[0]),

            "n_EV_charging_t0":
                int(n_ev_charging.iloc[0]),

            "DHW_plus_boost_t0_kW":
                float(dhw_kw.iloc[0]),

            "EV_plus_DHW_t0_kW":
                float(ev_plus_dhw_kw.iloc[0]),

            "max_EV_charge_kW":
                float(ev_kw.max()),

            "max_EV_plus_DHW_kW":
                float(ev_plus_dhw_kw.max()),

            "peak_timestamp":
                t_peak,

            "BEV_outdoor_total_kWh":
                float(outdoor.sum().sum()),

            "BEV_shortage_total_kWh":
                shortage_total_kwh,

            "BEV_shortage_max_ts_kWh":
                float(shortage.sum(axis=1).max()),

            "BEV_min_SOC_pct":
                float(soc_pct.min().min()),

            "BEV_p05_SOC_pct":
                float(np.nanpercentile(soc_pct.to_numpy(), 5)),

            "BEV_median_SOC_pct":
                float(np.nanmedian(soc_pct.to_numpy())),


            "n_households_with_shortage":
                int(
                    (shortage_by_house > 1e-9).sum()
                ),

            "max_household_shortage_kWh":
                float(shortage_by_house.max()),

            "BEV_mobility_feasible":
                bool(shortage_total_kwh <= 1e-9),


            "base_t0_kW":
                float(base_kw.iloc[0]),

            "total_RB_load_t0_kW":
                float(total_rb_load_kw.iloc[0]),

            "max_total_RB_load_kW":
                float(total_rb_load_kw.max()),

            "total_RB_peak_timestamp":
                t_peak_total,

        })

        # -------------------------------
        # Sec 4.10 -- 2.5) Check available SOC and expected mobility demand
        # -------------------------------
        shortage = rp["bev_shortage_kwh"]
        soc_pct = rp["bev_soc_pct"]
        outdoor = rp["bev_outdoor_kwh"]

        for c in shortage.columns:

            mask = shortage[c] > 1e-9

            for ts in shortage.index[mask]:

                shortage_events.append({
                    "season": season,
                    "household": c,
                    "timestamp": ts,

                    "shortage_kWh":
                        float(shortage.loc[ts, c]),

                    "SOC_pct_end_ts":
                        float(soc_pct.loc[ts, c]),

                    "outdoor_kWh_same_ts":
                        float(outdoor.loc[ts, c]),
                })

        # -------------------------------
        # Sec 4.10 -- 2.6) Graph with SOC and use of outdoor charger (conventional setup/RB)
        # -------------------------------
        soc_pct = (
            rp["bev_soc_pct"]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
        )

        outdoor_kw = (
            rp["bev_outdoor_kw"]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .fillna(0.0)
            .sum(axis=1)
        )

        soc_percentiles = pd.DataFrame(
            {
                "min": soc_pct.min(axis=1),
                "p05": soc_pct.quantile(0.05, axis=1),
                "p10": soc_pct.quantile(0.10, axis=1),
                "p25": soc_pct.quantile(0.25, axis=1),
                "p50": soc_pct.quantile(0.50, axis=1),
                "p75": soc_pct.quantile(0.75, axis=1),
                "p90": soc_pct.quantile(0.90, axis=1),
                "p95": soc_pct.quantile(0.95, axis=1),
                "max": soc_pct.max(axis=1),
                "outdoor_total_kw": outdoor_kw,
            },
            index=soc_pct.index,
        )

        soc_percentiles_by_season[
            season
        ] = soc_percentiles

        fig, ax1 = plt.subplots(
            figsize=(13, 5.5)
        )

        # --------------------------------------------------------
        # Sec 4.10 -- 2.7) SOC distribution "graph"
        # --------------------------------------------------------
        ax1.fill_between(            soc_percentiles.index,
            soc_percentiles["p10"],
            soc_percentiles["p90"],
            alpha=0.20,
            label="SOC p10-p90",
        )

        ax1.plot(            soc_percentiles.index,
            soc_percentiles["p50"],
            linewidth=1.6,
            label="SOC median",
        )

        # Extreme trajectories
        ax1.plot(            soc_percentiles.index,
            soc_percentiles["min"],
            linewidth=1.0,
            linestyle="--",
            label="SOC min",
        )

        ax1.plot(            soc_percentiles.index,
            soc_percentiles["max"],
            linewidth=1.0,
            linestyle="--",
            label="SOC max",
        )

        # Reference SOC thresholds
        ax1.axhline(            5.0,
            linestyle=":",
            linewidth=1.0,
            label="Minimum SOC",
        )

        ax1.axhline(            20.0,
            linestyle=":",
            linewidth=1.0,
            label="Outdoor trigger",
        )

        ax1.axhline(            50.0,
            linestyle=":",
            linewidth=1.0,
            label="Outdoor target",
        )

        ax1.axhline(            80.0,
            linestyle=":",
            linewidth=1.0,
            label="Home target",
        )

        ax1.set_ylabel("BEV SOC [%]")
        ax1.set_ylim(0, 105)
        ax1.grid(
            True,
            linestyle=":",
            alpha=0.3,
        )

        # -----------------
        # Outdoor charging on secondary axis
        # -----------------
        ax2 = ax1.twinx()

        ax2.plot(
            soc_percentiles.index,
            soc_percentiles["outdoor_total_kw"],
            linewidth=1.2,
            label="Outdoor charging",
        )

        ax2.set_ylabel(
            "Outdoor charging power [kW]"
        )

        # -----------------
        # Legend
        # -----------------
        handles1, labels1 = (
            ax1.get_legend_handles_labels()
        )

        handles2, labels2 = (
            ax2.get_legend_handles_labels()
        )

        ax1.legend(
            handles1 + handles2,
            labels1 + labels2,
            loc="best",
            fontsize=8,
        )


        ax1.set_title(
            f"RB BEV SOC and outdoor charging - {season}"
        )

        fig.tight_layout()

        plt.show()

        # -------------------------------




    df_bev_shortage_events = pd.DataFrame(
        shortage_events
    )

    display(df_bev_shortage_events)


    # ============================================================
    # Sec 4.10 -- 2.7)  Detailed temporal windows around BEV shortage events
    # ============================================================

    SHORTAGE_WINDOW_HOURS_BEFORE = 2
    SHORTAGE_WINDOW_HOURS_AFTER = 2

    steps_before = int(
        round(
            SHORTAGE_WINDOW_HOURS_BEFORE
            * 60
            / timestep_minutes
        )
    )

    steps_after = int(
        round(
            SHORTAGE_WINDOW_HOURS_AFTER
            * 60
            / timestep_minutes
        )
    )

    shortage_window_rows = []
    shortage_block_rows = []

    eps = 1e-9


    for season, rp in rb_profiles_by_season.items():

        shortage_df = (
            rp["bev_shortage_kwh"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        soc_pct_df = (
            rp["bev_soc_pct"]
            .apply(pd.to_numeric, errors="coerce")
        )

        mobility_kw_df = (
            rp["bev_demand_kw"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        home_charge_kw_df = (
            rp["bev_charge_kw"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        outdoor_charge_kw_df = (
            rp["bev_outdoor_kw"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        outdoor_charge_kwh_df = (
            rp["bev_outdoor_kwh"]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )


        # --------------------------------------------------------
        # Sec 4.10 -- 2.7.2)  Only households with at least one shortage timestep
        # --------------------------------------------------------
        affected_houses = [
            c
            for c in shortage_df.columns
            if float(shortage_df[c].sum()) > eps
        ]


        for c in affected_houses:

            shortage_c = shortage_df[c]

            mask_shortage = (
                shortage_c > eps
            )


            # ----------------------------------------------------
            # Sec 4.10 -- 2.72b)  Identify contiguous shortage blocks
            # ----------------------------------------------------
            block_group = (
                mask_shortage
                .ne(mask_shortage.shift(fill_value=False))
                .cumsum()
            )

            active_groups = (
                block_group[mask_shortage]
                .unique()
            )


            for block_number, group_id in enumerate(
                active_groups,
                start=1,
            ):

                block_mask = (
                    mask_shortage
                    & (block_group == group_id)
                )

                block_positions = np.flatnonzero(
                    block_mask.to_numpy()
                )

                i_start = int(block_positions[0])
                i_end = int(block_positions[-1])

                block_start = shortage_df.index[i_start]
                block_end = shortage_df.index[i_end]

                # Window around the complete shortage block
                i_window_start = max(
                    0,
                    i_start - steps_before,
                )

                i_window_end = min(
                    len(shortage_df) - 1,
                    i_end + steps_after,
                )


                # ------------------------------------------------
                # # Sec 4.10 -- 2.7.2c)  Summary of the shortage block
                # ------------------------------------------------
                shortage_block_rows.append({

                    "season":
                        season,

                    "household":
                        c,

                    "shortage_block":
                        block_number,

                    "shortage_start":
                        block_start,

                    "shortage_end":
                        block_end,

                    "shortage_duration_min":
                        (
                            len(block_positions)
                            * timestep_minutes
                        ),

                    "shortage_total_kWh":
                        float(
                            shortage_c.iloc[
                                i_start:i_end + 1
                            ].sum()
                        ),

                    "SOC_pct_before_shortage":
                        (
                            float(
                                soc_pct_df[c].iloc[
                                    i_start - 1
                                ]
                            )
                            if i_start > 0
                            else np.nan
                        ),

                    "mobility_demand_peak_kW":
                        float(
                            mobility_kw_df[c].iloc[
                                i_start:i_end + 1
                            ].max()
                        ),
                })


                # ------------------------------------------------
                #    Sec 4.10 -- 2.7.2d)  Detailed timestep table
                # ------------------------------------------------
                for i in range(
                    i_window_start,
                    i_window_end + 1,
                ):

                    ts = shortage_df.index[i]

                    if i < i_start:
                        phase = "before"

                    elif i <= i_end:
                        phase = "SHORTAGE"

                    else:
                        phase = "after"


                    soc_end = (
                        float(soc_pct_df[c].iloc[i])
                        if pd.notna(
                            soc_pct_df[c].iloc[i]
                        )
                        else np.nan
                    )

                    # Previous timestep SOC = best available
                    # approximation of SOC at start of timestep
                    if i > 0:
                        soc_start = (
                            float(
                                soc_pct_df[c].iloc[i - 1]
                            )
                            if pd.notna(
                                soc_pct_df[c].iloc[i - 1]
                            )
                            else np.nan
                        )
                    else:
                        soc_start = (
                            float(initial_SoC_BEV)
                            * 100.0
                        )


                    mobility_kw = float(
                        mobility_kw_df[c].iloc[i]
                    )

                    home_kw = float(
                        home_charge_kw_df[c].iloc[i]
                    )

                    outdoor_kw = float(
                        outdoor_charge_kw_df[c].iloc[i]
                    )

                    outdoor_kwh = float(
                        outdoor_charge_kwh_df[c].iloc[i]
                    )

                    shortage_kwh = float(
                        shortage_df[c].iloc[i]
                    )


                    shortage_window_rows.append({

                        "season":
                            season,

                        "household":
                            c,

                        "shortage_block":
                            block_number,

                        "phase":
                            phase,

                        "timestamp":
                            ts,

                        "SOC_pct_start_ts":
                            soc_start,

                        "SOC_pct_end_ts":
                            soc_end,

                        "mobility_demand_kw":
                            mobility_kw,

                        "is_moving":
                            bool(
                                mobility_kw > eps
                            ),

                        "home_charge_kw":
                            home_kw,

                        "outdoor_charge_kw":
                            outdoor_kw,

                        "outdoor_charge_kWh":
                            outdoor_kwh,

                        "shortage_kWh":
                            shortage_kwh,

                        "is_shortage":
                            bool(
                                shortage_kwh > eps
                            ),
                    })


    df_bev_shortage_blocks = pd.DataFrame(
        shortage_block_rows
    )

    df_bev_shortage_windows = pd.DataFrame(
        shortage_window_rows
    )


    print(
        "\n=== BEV SHORTAGE BLOCK SUMMARY ==="
    )

    display(
        df_bev_shortage_blocks
    )


    print(
        "\n=== TEMPORAL WINDOWS AROUND BEV SHORTAGE ==="
    )

    display(
        df_bev_shortage_windows
    )
    # ---------------------



    # ============================================================
    # Sec 4.10 -- 2.8) Mobility trips responsible for BEV shortage
    # ============================================================

    shortage_trip_rows = []

    eps = 1e-9
    dt_h = float(timestep_minutes) / 60.0

    for _, block in df_bev_shortage_blocks.iterrows():

        season = block["season"]
        c = block["household"]

        shortage_start = pd.Timestamp(
            block["shortage_start"]
        )

        rp = rb_profiles_by_season[season]


        # --------------------------------------------------------
        # # Sec 4.10 -- 2.8.1) Mobility profile
        # --------------------------------------------------------
        mobility_kw = (
            rp["bev_demand_kw"][c]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )


        # --------------------------------------------------------
        # BEV SOC
        # --------------------------------------------------------
        soc_pct = (
            rp["bev_soc_pct"][c]
            .apply(pd.to_numeric, errors="coerce")
        )


        # --------------------------------------------------------
        # Shortage
        # --------------------------------------------------------
        shortage_kwh = (
            rp["bev_shortage_kwh"][c]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )


        # --------------------------------------------------------
        # Home / outdoor charging
        # --------------------------------------------------------
        home_kw = (
            rp["bev_charge_kw"][c]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        outdoor_kw = (
            rp["bev_outdoor_kw"][c]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )


        idx = mobility_kw.index

        i_short = idx.get_loc(
            shortage_start
        )


        # --------------------------------------------------------
        # # Sec 4.10 -- 2.8.2) Go backwards to find the beginning of the continuous
        # mobility block responsible for the shortage
        # --------------------------------------------------------
        i_trip_start = i_short

        while (
            i_trip_start > 0
            and mobility_kw.iloc[
                i_trip_start - 1
            ] > eps
        ):
            i_trip_start -= 1


        # --------------------------------------------------------
        # Go forward to identify the end of the same trip
        # --------------------------------------------------------
        i_trip_end = i_short

        while (
            i_trip_end < len(idx) - 1
            and mobility_kw.iloc[
                i_trip_end + 1
            ] > eps
        ):
            i_trip_end += 1


        trip_start = idx[i_trip_start]
        trip_end = idx[i_trip_end]


        # --------------------------------------------------------
        # # Sec 4.10 -- 2.8.3) SOC available immediately before departure
        # --------------------------------------------------------
        if i_trip_start > 0:

            soc_departure_pct = float(
                soc_pct.iloc[
                    i_trip_start - 1
                ]
            )

        else:

            soc_departure_pct = (
                float(initial_SoC_BEV)
                * 100.0
            )


        # --------------------------------------------------------
        # Total mobility energy requested during the trip
        # --------------------------------------------------------
        trip_mobility_kwh = float(
            mobility_kw.iloc[
                i_trip_start:
                i_trip_end + 1
            ].sum()
            * dt_h
        )


        # Battery energy required to supply that mobility
        trip_battery_required_kwh = (
            trip_mobility_kwh
            / float(
                EV_charge_discharge_efficiency
            )
        )


        # --------------------------------------------------------
        # Battery capacity
        # --------------------------------------------------------
        try:

            cap_bev = float(
                ev_capacity_by_case.loc[c]
            )

        except Exception:

            cap_bev = float(
                default_ev_battery_capacity_kwh
            )


        # --------------------------------------------------------
        # # Sec 4.10 -- 2.8.4) Minimum SOC required at departure
        # --------------------------------------------------------
        required_soc_pct = (
            100.0
            * trip_battery_required_kwh
            / cap_bev
        )


        # Optional reserve, useful only for interpretation
        required_soc_with_10pct_reserve = (
            required_soc_pct
            + 10.0
        )


        # --------------------------------------------------------
        # # Sec 4.10 -- 2.8.5) Last charging event before departure
        # --------------------------------------------------------
        previous_home = (
            home_kw.iloc[:i_trip_start] > eps
        )

        previous_outdoor = (
            outdoor_kw.iloc[:i_trip_start] > eps
        )

        last_home_ts = (
            previous_home[
                previous_home
            ].index[-1]
            if previous_home.any()
            else pd.NaT
        )

        last_outdoor_ts = (
            previous_outdoor[
                previous_outdoor
            ].index[-1]
            if previous_outdoor.any()
            else pd.NaT
        )


        if (
            pd.notna(last_outdoor_ts)
            and (
                pd.isna(last_home_ts)
                or last_outdoor_ts > last_home_ts
            )
        ):
            last_charge_type = "outdoor"
            last_charge_ts = last_outdoor_ts

        elif pd.notna(last_home_ts):

            last_charge_type = "home"
            last_charge_ts = last_home_ts

        else:

            last_charge_type = "none"
            last_charge_ts = pd.NaT


        shortage_trip_rows.append({

            "season":
                season,

            "household":
                c,

            "trip_start":
                trip_start,

            "trip_end":
                trip_end,

            "trip_duration_min":
                int(
                    (
                        i_trip_end
                        - i_trip_start
                        + 1
                    )
                    * timestep_minutes
                ),

            "battery_capacity_kWh":
                cap_bev,

            "SOC_at_departure_pct":
                soc_departure_pct,

            "trip_mobility_kWh":
                trip_mobility_kwh,

            "trip_battery_required_kWh":
                trip_battery_required_kwh,

            "required_SOC_pct":
                required_soc_pct,

            "required_SOC_with_10pct_reserve":
                required_soc_with_10pct_reserve,

            "shortage_total_kWh":
                float(
                    shortage_kwh.iloc[
                        i_trip_start:
                        i_trip_end + 1
                    ].sum()
                ),

            "target_50_sufficient":
                bool(
                    required_soc_pct <= 50.0
                ),

            "target_60_sufficient":
                bool(
                    required_soc_pct <= 60.0
                ),

            "target_70_sufficient":
                bool(
                    required_soc_pct <= 70.0
                ),

            "target_80_sufficient":
                bool(
                    required_soc_pct <= 80.0
                ),

            "full_battery_sufficient":
                bool(
                    required_soc_pct <= 100.0
                ),

            "last_charge_type":
                last_charge_type,

            "last_charge_timestamp":
                last_charge_ts,
        })


    df_bev_shortage_trip_analysis = (
        pd.DataFrame(
            shortage_trip_rows
        )
    )

    print(
        "\n=== BEV TRIPS RESPONSIBLE FOR SHORTAGE ==="
    )

    display(
        df_bev_shortage_trip_analysis
    )
    # ------------------------------------------





    df_rb_profile_diagnostics = pd.DataFrame(rows)

    display(df_rb_profile_diagnostics)

## 4.11 Energy-system model construction and optimization with oemof.solph

In [ ]:
# ------------------------------------------------------------
# Sec 4.11 -- 0) Configurations to solve
# ------------------------------------------------------------
experiment_cost_rows_by_season = []
experiment_cost_rows_total = []
RUNS = SOLVE_RUN_CONFIGS.copy()

N_RUN_CONFIGS = len(RUNS)
N_SEASONS_TO_SOLVE = len(list(seasons_dict.values()))
N_TOTAL_SOLVES = N_RUN_CONFIGS * N_SEASONS_TO_SOLVE

solve_counter = 0

# ------------------------------------------------------------
# Sec 4.11 -- 0.1)  Solve-status registry
# ------------------------------------------------------------
# Pre-populate ALL scheduled run x season combinations as NOT_RUN.
# Each entry will subsequently become:
#   OK
#   FAIL (INFEASIBLE)
#   FAIL (UNBOUNDED)
#   FAIL (INF_OR_UNBD_UNRESOLVED)
#   FAIL (<other solver termination condition>)
# ------------------------------------------------------------

SOLVE_STATUS_REGISTRY = {}

for _cfg in RUNS:
    for _season in seasons_dict.values():

        _key = (
            str(_cfg["run_uid"]),
            str(_season),
        )

        SOLVE_STATUS_REGISTRY[_key] = {
            "run_uid": str(_cfg["run_uid"]),
            "run_label": str(_cfg["run_label"]),
            "case_label": str(_cfg.get("case_label", _cfg["run_label"])),
            "mode": str(_cfg["mode"]),
            "run_revision": str(_cfg.get("run_revision", RUN_REVISION)),

            "control_BESS": str(_cfg["control"]["BESS"]),
            "control_BEV": str(_cfg["control"]["BEV"]),
            "control_DHW": str(_cfg["control"]["DHW"]),

            "season": str(_season),

            "status": "NOT_RUN",

            "termination_condition_initial": None,
            "termination_condition_final": None,
            "solver_status": None,
            "gurobi_status": None,

            "lp_path": None,
            "iis_ilp_path": None,
            "iis_json_path": None,
        }
# -----------------


if N_TOTAL_SOLVES > 0:
    print(
        "\n"
        + "#" * 100
        + f"\n[INFO][4.11c] Optimization problems to solve: {N_TOTAL_SOLVES}"
        + f"\n  Configurations: {N_RUN_CONFIGS}"
        + f"\n  Seasons per configuration: {N_SEASONS_TO_SOLVE}"
        + "\n"
        + "#" * 100
    )
# ----------------------


if len(RUNS) == 0:
    print(
        "[INFO][4.11c] No HAGRID run requested. "
        "Skipping model construction and solve. "
    )
else:

    # 0.0) Definition of a canonical time index on which to build the model structure and optimize, so the ES structure does not need to be rebuilt at each iteration
    # Canonical: same length and same frequency for all seasons
    n_ts = int(number_timesteps)                # 1008
    freq_str = f"{int(timestep_minutes)}min"    # es "10min"

    timeindex_canonico = pd.date_range(
        start="2000-01-01 00:00:00",
        periods=n_ts,
        freq=freq_str
    )

    td_can = pd.to_timedelta(timeindex_canonico.freq)
    dt_hours_can = td_can.total_seconds() / 3600.0
    timeincrement_canonico = [dt_hours_can] * len(timeindex_canonico)

    # ------------------------------



    # 0.1) Pre-loading seasonal time series into a container, only for the technologies available in the different houses
    CAN = defaultdict(dict)   # CAN[season][name] = dictionary by house or series

    for season in seasons_dict.values():
        idx_real = ee_cost_series[season].index

        CAN[season]["ee_cost"] = to_canonical(ee_cost_series[season], idx_real, timeindex_canonico).astype(float)
        CAN[season]["ee_rev"]  = to_canonical(ee_rev_series[season],  idx_real, timeindex_canonico).astype(float)

        CAN[season]["pv"] = {}
        CAN[season]["base"] = {}
        CAN[season]["Cool"] = {}
        CAN[season]["Cook"] = {}
        CAN[season]["Heat"] = {}

        if NEED_RB_PROFILES:
            CAN[season]["bess_charge_kwh"] = {}
            CAN[season]["bess_dis_kwh"] = {}

            # load fixed Rule-Based profiles for the BESS
            rb_prof = rb_profiles_by_season[season]
            for c in has["BESS"]:
                CAN[season]["bess_charge_kwh"][c] = to_canonical(
                    rb_prof["pv_to_bess_input_kwh"][c], idx_real, timeindex_canonico
                ).fillna(0.0).clip(lower=0.0).astype(float)

                CAN[season]["bess_dis_kwh"][c] = to_canonical(
                    rb_prof["bess_to_load_kwh"][c], idx_real, timeindex_canonico
                ).fillna(0.0).clip(lower=0.0).astype(float)
        
        for c in ID:
            CAN[season]["base"][c] = to_canonical(dmd_base_el_fix_dicts[season][c], idx_real, timeindex_canonico).fillna(0.0)
            CAN[season]["Cool"][c] = to_canonical(dmd_Cool_el_fix_dicts[season][c], idx_real, timeindex_canonico).fillna(0.0)
            CAN[season]["Cook"][c] = to_canonical(dmd_Cook_el_fix_dicts[season][c], idx_real, timeindex_canonico).fillna(0.0)
            CAN[season]["Heat"][c] = to_canonical(dmd_Heat_el_fix_dicts[season][c], idx_real, timeindex_canonico).fillna(0.0)

            if c in has['PV']:
                CAN[season]["pv"][c] = to_canonical(pv_fix_dicts[season][c], idx_real, timeindex_canonico).fillna(0.0)

        # 0.1.1) DHW only for houses that have it
        CAN[season]["dhw"] = {}
        for c in has["DHW"]:
            CAN[season]["dhw"][c] = to_canonical(fixed_dhw_electric_demand_dicts[season][c], idx_real, timeindex_canonico).fillna(0.0)


        # 0.1.2) BEV only for houses that have it
        CAN[season]["bev"] = {}
        CAN[season]["occ"] = {}
        CAN[season]["home"] = {}
        CAN[season]["outdoor"] = {}

        season_key = str(season).strip().lower()
        for c in has["BEV"]:
            CAN[season]["bev"][c] = to_canonical(dmd_BEV_el_fix_dicts[season][c], idx_real, timeindex_canonico).fillna(0.0)

            occ_real = max_occupancy_profiles_by_season[season_key][c]

            home_real = build_home_presence_profile(
                occupancy_df=occ_by_season[season],
                idx=idx_real,
                c=c,
                current_season=season,
                member=1,
            )

            # --- debug outdoor/home charging-availability profiles ----
            if DEBUG_CAN_SCAN:
                # check occupancy-index mismatch -> does reindex produce NaN?
                try:
                    tmp_occ = occ_real.reindex(idx_real)
                    nan_occ = float(tmp_occ.isna().mean())
                    if nan_occ > 0.05:  # 5% threshold
                        _scan_print(f"[CAN-CHK] {season_key}/{c} occ_real reindex NaN%={100*nan_occ:.1f}% | "
                                    f"len_occ={len(occ_real)} len_idx={len(idx_real)}")
                        can_scan_rows.append({"season": season_key, "household": c,  "what": "occ_reindex_nan", "value": nan_occ})
                except Exception as e:
                    _scan_print(f"[CAN-CHK] {season_key}/{c} occ_real reindex ERROR: {e}")
                    can_scan_rows.append({"season": season_key, "household": c,  "what": "occ_reindex_err", "value": str(e)})
            # --------------------------

            
            CAN[season]["occ"][c] = (to_canonical(occ_real, idx_real, timeindex_canonico).fillna(0.0).clip(0, 1))
            CAN[season]["home"][c] = to_canonical(home_real, idx_real, timeindex_canonico).fillna(0.0).clip(0, 1)



            # --- debug outdoor/home charging-availability profiles ----
            if DEBUG_CAN_SCAN:
                has_df = (season_key in df_outdoor_by_season)
                has_col = has_df and (c in df_outdoor_by_season[season_key].columns)
                # note: if has_df&has_col = True, you will NOT use the complement; you will use out_real from the dataframe
                if has_df and has_col:
                    branch = "FORCED: (1-occ)*(bev==0)"
                else:
                    branch = "COMPLEMENT(1-occ)"
            # --------------------------



            # ---- outdoor profile available when away from home, occ=0, and not moving, dmd_BEV=0 ----
            eps_move = 1e-9  # tolerance: treat as "moving" if dmd_BEV > eps_move

            # outdoor available when the person is not at home, occ=0, AND not moving, dmd_BEV=0
            mask_not_moving = (CAN[season]["bev"][c] <= eps_move).astype(float)

            CAN[season]["outdoor"][c] = ((1.0 - CAN[season]["home"][c]) * mask_not_moving).clip(0, 1)


            # ---- Energy-balance check, if SOC available + potential BEV charging >= mobility demand + residual SOC ----
            # ---- CHECK "really strange": car stationary, bev==0, but no charging possibility, occ+out==0
            eps = 1e-9

            s_occ = CAN[season]["occ"][c]
            s_out = CAN[season]["outdoor"][c]
            s_bev = CAN[season]["bev"][c]

            avail0 = (s_occ.values + s_out.values) < eps
            not_moving = (s_bev.values <= eps)

            n_weird = int((avail0 & not_moving).sum())
            if n_weird > 0:
                print(f"[CAN-WEIRD] {season_key}/{c} NOT moving but NO charging availability | n={n_weird}")
                    
            

            # --- debug outdoor/home charging-availability profiles -----
            if DEBUG_CAN_SCAN:
                s_occ = CAN[season]["occ"][c]
                s_out = CAN[season]["outdoor"][c]
                s_bev = CAN[season]["bev"][c]

                # 1) is outdoor always zero?
                out_max = float(s_out.max())
                occ_mean = float(s_occ.mean())
                out_mean = float(s_out.mean())

                # 2) are there timesteps with BEV demand but no charging possibility?
                avail = (s_occ.values + s_out.values)
                n_av0 = int((avail < 1e-9).sum())
                n_bad = int(((s_bev.values > 1e-9) & (avail < 1e-9)).sum())

                # report only "suspicious" cases
                # (a) outdoor always zero but not because occupancy is always 1
                if (out_max < 1e-9) and (occ_mean < 0.995):
                    _scan_print(f"[CAN-CHK] {season_key}/{c} OUTDOOR all-zero | branch={branch} | "
                                f"occ_mean={occ_mean:.3f} out_mean={out_mean:.3f} out_max={out_max:.3g}")
                    can_scan_rows.append({"season": season_key, "household": c,  "what": "out_all_zero", "value": out_mean, "branch": branch})

            # ----------------------------------------------------------------



    # --- debug outdoor/home charging-availability profiles -----
    if DEBUG_CAN_SCAN:
        print("\n=== CAN SCAN SUMMARY ===")
        print("Total hits printed:", min(can_scan_hits, CAN_SCAN_PRINT_LIMIT), "/", can_scan_hits)
        if len(can_scan_rows) > 0:
            df_scan = pd.DataFrame(can_scan_rows)
            # sort by most critical cases, high n_bad
            if "value" in df_scan.columns:
                df_scan = df_scan.sort_values(by="value", ascending=False)
            print(df_scan.head(30))
        else:
            print("No suspicious cases found.")
    # ----------------------------------------------------------------


    # for cfg_idx, RUN_CFG in enumerate(RUNS, start=1):
    for cfg_idx, RUN_CFG in enumerate(RUNS, start=1):
        """

            Depending on the flexibility technology to observe, conventional/RB vs smart/OPT
            different components are defined in the optimization problem.

            In the case of conventional technologies, RB: 
                - RB charging profiles for that season
                - build nodes with the same topology, but different components (DHW, 
                BESS, BEV will have fixed profiles to respect)

            In the case of smart technologies, OPT:
                - free to exchange electricity with the grid based on signals
                of price   
        
        """

        # Dictionary used to store computation times for each code block, model construction, solving, print/plot, etc.
        timings = {} 
        t0_run = perf_counter()

        MODE = RUN_CFG["mode"]

        MODE_LABEL = RUN_CFG.get("label",MODE,)

        CONTROL_MODE = RUN_CFG["control"]

        RUN_SHARES = dict(RUN_CFG["shares"])

        RUN_UID = RUN_CFG["run_uid"]

        RUN_LABEL = RUN_CFG["run_label"]


        # ------------------------------------------------------------
        # Run-specific folders
        # ------------------------------------------------------------

        RUN_DIRS = (
            set_hagrid_output_dirs_for_run(RUN_CFG, create_keys=(
                    "xls",
                    "py",
                    "pflow_py",
                ),
            )
        )


        print(f"\n[RUN] {RUN_UID}")
        print(f"[RUN] label      = {RUN_LABEL}")
        print(f"[RUN] experiment = {RUN_CFG['experiment_id']}")
        print(f"[RUN] mode       = {MODE}")
        print(f"[RUN] xls        = {optimization_output_folder}")
        print(f"[RUN] py         = {py_based_output_folder}")
        print(f"[RUN] pflow_py   = {pf_py_based_output_folder}")




        # Potential physical technology endowment of the configuration:
        # reduces global has according to possible configuration-specific shares.
        HAS_POTENTIAL_CFG = _potential_has_for_config_shares(
            has_global=has,
            cfg_shares=RUN_SHARES,
        )

        # Effectively active technology endowment:
        # applies OPT/RB/NO starting from the potential technology endowment of the configuration.
        HAS_RUN = _active_has_for_run(
            has_global=HAS_POTENTIAL_CFG,
            control=CONTROL_MODE,
        )


        # Aliases for the current run
        has_PV_run   = HAS_RUN['PV']
        has_BESS_run = HAS_RUN["BESS"]
        has_BEV_run  = HAS_RUN["BEV"]
        has_V2G_run = HAS_RUN["V2G"]
        has_DHW_run  = HAS_RUN["DHW"]
        has_HP_run   = HAS_RUN["HP"]
        has_COOL_run = HAS_RUN["COOL"]
        has_IND_run  = HAS_RUN["INDUCTION"]

        # ------------------------------------------------------------
        # Sec 4.11 -- 0.x) Run-specific physical DHW parameters
        # ------------------------------------------------------------
        #
        # IMPORTANT:
        # These dictionaries are built directly from the physical
        # household-level dictionaries created in Sec. 4.9.
        #
        # Therefore Sec. 4.11 does NOT depend on the temporary
        # *_eff dictionaries created inside Sec. 4.10.
        #
        # RB and OPT runs use the SAME physical DHW parameters
        # for a given household. Only the control strategy changes.
        # ------------------------------------------------------------

        _dhw_run_set = set(has_DHW_run)


        # ------------------------------------------------------------
        # Consistency check:
        # every household with active DHW in this run must have
        # all physical DHW parameters available from Sec. 4.9.
        # ------------------------------------------------------------

        _missing_dhw_capacity = sorted(
            _dhw_run_set
            - set(capacity_dhw_by_house.keys())
        )

        _missing_dhw_power = sorted(
            _dhw_run_set
            - set(power_limit_dhw_by_house.keys())
        )

        _missing_dhw_loss = sorted(
            _dhw_run_set
            - set(loss_rate_dhw_by_house.keys())
        )

        _missing_dhw_soc0 = sorted(
            _dhw_run_set
            - set(initial_soc_dhw_by_house.keys())
        )

        _missing_dhw_soc0_frac = sorted(
            _dhw_run_set
            - set(initial_soc_dhw_frac_by_house.keys())
        )



        if (
            _missing_dhw_capacity
            or _missing_dhw_power
            or _missing_dhw_loss
            or _missing_dhw_soc0
            or _missing_dhw_soc0_frac
        ):

            raise KeyError(
                "[DHW RUN PARAMETERS] Missing household-specific "
                "DHW physical parameters.\n"
                f"Run: {RUN_LABEL} | MODE={MODE}\n"
                f"Missing capacity:       {_missing_dhw_capacity}\n"
                f"Missing power:          {_missing_dhw_power}\n"
                f"Missing loss:           {_missing_dhw_loss}\n"
                f"Missing SOC0 [kWh]:     {_missing_dhw_soc0}\n"
                f"Missing SOC0 fraction:  {_missing_dhw_soc0_frac}\n"
                "Sec. 4.11 must obtain these parameters from the "
                "global dictionaries created in Sec. 4.9."
            )


        # ------------------------------------------------------------
        # Effective DHW dictionaries for THIS run
        # ------------------------------------------------------------

        capacity_dhw_by_house_eff_run = {
            c: float(
                capacity_dhw_by_house[c]
            )
            for c in _dhw_run_set
        }

        power_limit_dhw_by_house_eff_run = {
            c: float(
                power_limit_dhw_by_house[c]
            )
            for c in _dhw_run_set
        }

        loss_rate_dhw_by_house_eff_run = {
            c: float(
                loss_rate_dhw_by_house[c]
            )
            for c in _dhw_run_set
        }

        initial_soc_dhw_by_house_eff_run = {
            c: float(
                initial_soc_dhw_by_house[c]
            )
            for c in _dhw_run_set
        }


        # Fractional initial SOC is useful for audits and metadata.
        # It is created in Sec. 4.9 for both distributed=True/False.
        initial_soc_dhw_frac_by_house_eff_run = {
            c: float(
                initial_soc_dhw_frac_by_house[c]
            )
            for c in _dhw_run_set
        }



        # ------------------------------------------------------------
        # Strong consistency check:
        # all run-specific DHW dictionaries must describe exactly
        # the same active DHW household population.
        # ------------------------------------------------------------

        assert (
            set(capacity_dhw_by_house_eff_run.keys())
            == _dhw_run_set
        ), (
            "[DHW RUN PARAMETERS] Capacity dictionary does not "
            "match has_DHW_run."
        )

        assert (
            set(power_limit_dhw_by_house_eff_run.keys())
            == _dhw_run_set
        ), (
            "[DHW RUN PARAMETERS] Power dictionary does not "
            "match has_DHW_run."
        )

        assert (
            set(loss_rate_dhw_by_house_eff_run.keys())
            == _dhw_run_set
        ), (
            "[DHW RUN PARAMETERS] Loss-rate dictionary does not "
            "match has_DHW_run."
        )

        assert (
            set(initial_soc_dhw_by_house_eff_run.keys())
            == _dhw_run_set
        ), (
            "[DHW RUN PARAMETERS] Initial-SOC dictionary does not "
            "match has_DHW_run."
        )

        assert (
            set(initial_soc_dhw_frac_by_house_eff_run.keys())
            == _dhw_run_set
        ), (
            "[DHW RUN PARAMETERS] Fractional initial-SOC dictionary "
            "does not match has_DHW_run."
        )






        # ------------------------------------------------------------
        # Run-level sanity check
        # ------------------------------------------------------------

        if len(_dhw_run_set) > 0:

            print(
                "[DHW RUN PARAMETERS] "
                f"{RUN_LABEL} | MODE={MODE} | "
                f"active DHW households={len(_dhw_run_set)}"
            )

            print(
                "  capacity entries :",
                len(capacity_dhw_by_house_eff_run),
            )

            print(
                "  power entries    :",
                len(power_limit_dhw_by_house_eff_run),
            )

            print(
                "  loss entries     :",
                len(loss_rate_dhw_by_house_eff_run),
            )

            print(
                "  initial SOC      :",
                len(initial_soc_dhw_by_house_eff_run),
            )
        # -------------------------------



        # Save configuration information/metadata in a JSON file 
        RUN_META = save_run_metadata(
            dirs=RUN_DIRS,
            run_cfg=RUN_CFG,
            has_potential=HAS_POTENTIAL_CFG,
            has_run=HAS_RUN,
            extra={
                "economic_reporting": {
                    "include_dhw_boost_in_effective_costs": bool(
                        INCLUDE_DHW_BOOST_IN_EFFECTIVE_COSTS
                    ),
                    "cost_boost_DHW_eur_per_kwh": float(
                        cost_boost_DHW
                    ),
                    "cost_outdoor_charger_eur_per_kwh": float(
                        cost_outdoor_charger
                    ),
                }
            },
        )


        manifest_row = (
            build_run_manifest_row(
                run_cfg=RUN_CFG,
                dirs=RUN_DIRS,
                has_potential=(
                    HAS_POTENTIAL_CFG
                ),
                has_run=HAS_RUN,
                status="running",
                completed_seasons=[],
            )
        )


        upsert_hagrid_run_manifest(
            manifest_row
        )


        completed_seasons_current_run = []


        # Effective flexibility category for this run.
        # Needed for exports/charts consistent with possibly disabled BESS/BEV/DHW.
        flex_category_by_c_run = {
            c: classify_flex_case(
                c,
                has_BESS_run,
                has_BEV_run,
                has_DHW_run,
            )
            for c in ID
        }

        df_flex_categories_run = pd.DataFrame({
            "house": ID,
            "mode": MODE,
            "mode_label": MODE_LABEL,

            "share_FV_config": RUN_SHARES['PV'],
            "share_BESS_config": RUN_SHARES["BESS"],
            "share_BEV_config": RUN_SHARES["BEV"],
            "share_DHW_config": RUN_SHARES["DHW"],
            "run_output_tag": RUN_DIRS["tag"],

            "control_BESS": CONTROL_MODE["BESS"],
            "control_BEV": CONTROL_MODE["BEV"],
            "control_DHW": CONTROL_MODE["DHW"],
            "flex_case_run": [flex_category_by_c_run.get(c, "NA") for c in ID],
            "flex_label_run": [
                flex_category_labels.get(flex_category_by_c_run.get(c, "NA"), "")
                for c in ID
            ],
            "has_BESS_run": [c in has_BESS_run for c in ID],
            "has_BEV_run": [c in has_BEV_run for c in ID],
            "has_DHW_run": [c in has_DHW_run for c in ID],
        })



        print("\n\n" + "#" * 80)
        print(f"#### Start solving MODE = {MODE}")
        print(f"#### Label: {MODE_LABEL}")
        print(f"#### Control: BESS={CONTROL_MODE['BESS']} | "
            f"BEV={CONTROL_MODE['BEV']} | DHW={CONTROL_MODE['DHW']}"
            )
        print(f"#### Active tech counts: "
            f"BESS={len(has_BESS_run)} | BEV={len(has_BEV_run)} | DHW={len(has_DHW_run)}"
            )
        print("#" * 80 + "\n")


        # ------------------------------------------------------------
        # Save the exact physical and active technology assignment
        # of this run.
        # ------------------------------------------------------------

        save_compressed_pickle(
            HAS_POTENTIAL_CFG,
            os.path.join(py_based_output_folder, "has_physical.pkl.gz"),
        )

        save_compressed_pickle(
            HAS_RUN,
            os.path.join(py_based_output_folder, "has_active.pkl.gz"),
        )

        rows_has_run = []

        for tech, ids in HAS_RUN.items():
            for house_id in _sort_house_ids(ids):
                rows_has_run.append({
                    "run_uid": RUN_UID,
                    "run_label": RUN_LABEL,
                    "mode": MODE,
                    "tech": tech,
                    "house": house_id,
                    "control": CONTROL_MODE.get(tech, "PHYSICAL"),
                    "share_target": RUN_SHARES.get(tech, np.nan),
                })

        pd.DataFrame(rows_has_run).to_csv(
            os.path.join(py_based_output_folder, "has_active.csv"),
            index=False,
        )




        
        # Track costs for the configuration
        system_cost_rows_mode = []

        # 0.2) Dataset and time-series reading already performed in section 4.7
        


        #--------------------------------
        # Sec 4.11 -- 1) MODEL CONSTRUCTION AND OPTIMIZATION ACROSS SEASONS
        #--------------------------------
        # ----- Optimization loop by season, using pre-loaded profiles -----
        prima_iterazione = True
        pbar_total = len(season)  # Progress-bar iteration count



        for current_season in seasons_dict.values():
            print('-'*40)
            print(f"\n=== construction/solve for season {current_season}  |  MODE={MODE} ===")
            print('-'*40)

            # Mapping of real time profiles and preparation for association with canonical time index
            idx_real = IDX_REAL_BY_SEASON[current_season]
            # idx_real = ee_cost_series[season_corrente].index  # es 2023-02-15 ... (1008)
            assert len(idx_real) == len(timeindex_canonico)
            timeindex = timeindex_canonico
            
            # Loading grid and transformer transmission limits obtained from PF analysis (ie considering possible infeasibilities such as over/under-voltage and saturation)
            PF_LINE_LIMITS_MAP = {}
            if LV_ARCHETYPE in KERBER_BLUEPRINT_FILES and USE_PF_LINE_LIMITS_V1:
                PF_LINE_LIMITS_MAP = _load_pf_line_limits_for_season(
                    current_season,
                    lv_archetype=LV_ARCHETYPE,
                    hagrid_mode= MODE,
                )
                print(f"[INFO][PF LIMITS] season={current_season} -> read {len(PF_LINE_LIMITS_MAP)} PF-calibrated limits")

            PF_INTERFACE_LIMITS_MAP = {}
            if LV_ARCHETYPE in KERBER_BLUEPRINT_FILES and USE_PF_LINE_LIMITS_V1:
                PF_INTERFACE_LIMITS_MAP = _load_pf_interface_limits_for_season(
                    current_season,
                    lv_archetype=LV_ARCHETYPE,
                    hagrid_mode=MODE,
                )


            # Selection of fixed profiles for the current season, RB case
            rb_prof = None

            if _run_uses_any_rb(CONTROL_MODE):
                if current_season not in rb_profiles_by_season:
                    raise KeyError(
                        f"Missing RB profiles for season {current_season}. "
                        "They are required because at least one technology is in RB mode."
                    )

                rb_prof = rb_profiles_by_season[current_season]


            # -------------------------    
            # Sec 4.11 -- 1.1) Read input data, define time index and alignments
            # -------------------------            
            # canonical prices (Series su timeindex_canonico)
            ee_cost_can = CAN[current_season]["ee_cost"]
            ee_rev_can = CAN[current_season]["ee_rev"]

            print(f"[INFO] season={current_season}: model on CANONICAL time index, results reindexed on idx_real")


            # -------------------------    
            # Sec 4.11 -- 1.2) node construction
            # -------------------------
            with timed(f"{MODE}/{current_season} - build nodes", timings):
                noded = {}


                # -------------------------    
                # Sec 4.11 -- 1.2.1) Topology: creation of grid components/buses
                # -------------------------

                # (*) NO TOPOLOGY: single LV bus
                if LV_ARCHETYPE == "single_bus":
                    noded["bus_el_BT"] = solph.buses.Bus(label="bus_el_BT")
                    lv_root_obj = noded["bus_el_BT"]
                    lv_root = "bus_el_BT"


                # (**) OLD TOPOLOGY: single feeder and all houses in series
                elif LV_ARCHETYPE == "legacy_rural_chain":
                    feeder_code = "Da"
                    # root
                    noded[_lv_root_name()] = solph.buses.Bus(label=_lv_root_name())
                    lv_root_obj = noded[_lv_root_name()]
                    lv_root = _lv_root_name()

                    # feeder nodes
                    for k in range(1, num_households + 1):  # 1..N
                        node = _lv_node_name(feeder_code, k)
                        noded[node] = solph.buses.Bus(label=node)

                    # 1) create the "link list", only names/indices for LV-grid components between houses
                    lv_link_recipes = build_lv_link_recipes(num_households, feeder_code=feeder_code)

                    # 2) create the Converters in noded for each link
                    for rec in lv_link_recipes:
                        idx_a = rec["idx_a"]
                        idx_b = rec["idx_b"]
                        lab_forw = rec["lab_forw"]
                        lab_back = rec["lab_back"]

                        a_key = lv_bus_name(idx_a, feeder_code, _lv_root_name, _lv_node_name)
                        b_key = lv_bus_name(idx_b, feeder_code, _lv_root_name, _lv_node_name)

                        # forward: a -> b
                        noded[lab_forw] = solph.components.Converter(
                            label=lab_forw,
                            inputs={noded[a_key]: solph.Flow()},
                            outputs={noded[b_key]: solph.Flow()},
                            conversion_factors={noded[b_key]: lv_efficiency},
                        )

                        # backward: b -> a
                        noded[lab_back] = solph.components.Converter(
                            label=lab_back,
                            inputs={noded[b_key]: solph.Flow()},
                            outputs={noded[a_key]: solph.Flow()},
                            conversion_factors={noded[a_key]: lv_efficiency},
                        )


                # (***) NEW TOPOLOGY: from Kerber LV-grid archetypes, pandapower - rural vs mixed vs urban
                elif LV_ARCHETYPE in KERBER_BLUEPRINT_FILES:
                    if (
                        KERBER_BP is None
                        or not KERBER_BUS_ID_TO_LABEL
                        or not HOUSE_TO_LV_BUS_LABEL
                    ):
                        raise RuntimeError(
                            "LV topology is not initialized. "
                            "Run Sec. 4.7 before Sec. 4.11."
                        )

                    # ------ DEBUG: inspection of pmax along grid sections from the Kerber blueprint ---
                    bp = KERBER_BP  # or: _load_kerber_blueprint(BT_ARCHETYPE)

                    df_lines = pd.DataFrame(bp.get("lines_lv", []))
                    print("keys lines_lv:", df_lines.columns.tolist())
                    print("n_lines:", len(df_lines))

                    if "pmax_kw" in df_lines.columns:
                        s = pd.to_numeric(df_lines["pmax_kw"], errors="coerce")
                        print("\n[pmax_kw] describe:\n", s.describe())
                        print("missing pmax_kw:", int(s.isna().sum()))
                        print("pmax_kw==0:", int((s.fillna(0)==0).sum()))
                    else:
                        print("\nWARNING: the blueprint does NOT contain the pmax_kw column -> you are ALWAYS using the fallback",
                            KERBER_DEFAULT_PMAX_KW)
                    #------------------------------------------------------------

                    # 1) create all LV buses (readable labels)
                    for b in KERBER_BP["lv_buses"]:
                        lbl = KERBER_BUS_ID_TO_LABEL[int(b)]
                        noded[lbl] = solph.buses.Bus(label=lbl)

                    lv_root = LV_ROOT_LABEL
                    lv_root_obj = noded[lv_root]

                    # 2) create bidirectional line links with capacity, proxy for active power
                    for e in KERBER_BP["lines_lv"]:
                        fb = int(e["from_bus"])
                        tb = int(e["to_bus"])
                        a_key = KERBER_BUS_ID_TO_LABEL[fb]
                        b_key = KERBER_BUS_ID_TO_LABEL[tb]

                        # cap_kw = float(e.get("pmax_kw") or KERBER_DEFAULT_PMAX_KW)
                        cap_kw_raw = e.get("pmax_kw")
                        # cap_kw_send = float(cap_kw_raw) if cap_kw_raw is not None else KERBER_DEFAULT_PMAX_KW
                        cap_kw_send = _get_kerber_line_p_limit_kw(e, PF_LINE_LIMITS_MAP)            # <---- consider the new possible transmission constraint obtained from PF analysis


                        eta_line_raw = e.get("eta_ref")
                        eta_line = float(eta_line_raw) if eta_line_raw is not None else KERBER_LINE_ETA
                        if not np.isfinite(eta_line) or eta_line <= 0.0:
                            eta_line = KERBER_LINE_ETA
                        eta_line = min(eta_line, 1.0)

                        # output-side capacity consistent with the sending-side thermal limit
                        cap_kw_recv = cap_kw_send * eta_line


                        a_s = _short_lv_bus_label(a_key)    # es: n001 / root / h001
                        b_s = _short_lv_bus_label(b_key)
                        # forward: a -> b
                        lab_forw = f"line_forw_{a_s}_to_{b_s}"
                        # backward: b -> a
                        lab_back = f"line_back_{b_s}_to_{a_s}"


                        # forward a -> b
                        noded[lab_forw] = solph.components.Converter(
                            label=lab_forw,
                            inputs={noded[a_key]: solph.Flow()},
                            outputs={noded[b_key]: solph.Flow(
                                nominal_capacity=cap_kw_recv, 
                                variable_costs=KERBER_LINE_COST,
                            )},
                            conversion_factors={noded[b_key]: eta_line},
                            
                        )

                        # backward b -> a
                        noded[lab_back] = solph.components.Converter(
                            label=lab_back,
                            inputs={noded[b_key]: solph.Flow()},
                            outputs={noded[a_key]: solph.Flow(
                                nominal_capacity=cap_kw_recv,
                                variable_costs=KERBER_LINE_COST,
                            )},
                            conversion_factors={noded[a_key]: eta_line},
                        )
                else:
                    raise ValueError(
                        f"[BT] BT_ARCHETYPE='{LV_ARCHETYPE}' not valid. "
                        f"Use: single_bus | legacy_rural_chain | {list(KERBER_BLUEPRINT_FILES.keys())}"
                    )


                # ------------------------------------------------------------
                # Sec 4.11 -- 1.2.2) House loop, generic components for all houses
                # ------------------------------------------------------------
                for c in ID:
                    # Extract household index i from identifier c_i
                    i = int(c.split("_")[1])


                    # ============
                    #  Sec 4.11 -- Buses
                    # ============ 
                    noded[c + '_bus_el'] = solph.buses.Bus(label=c + '_bus_el')                               # Domestic-customer electric bus
                    noded[c + '_bus_DHW_use'] = solph.buses.Bus(label=c + '_bus_DHW_use')                     # DHW demand bus, after electric boiler
                    noded[c + '_bus_el_charge_BEV'] = solph.buses.Bus(label=c + '_bus_el_charge_BEV')         # Electric bus to charge the BEV battery
                    noded[c + '_bus_el_discharge_BEV'] = solph.buses.Bus(label=c + '_bus_el_discharge_BEV')   # Electric bus obtained by discharging the BEV battery
                    # # Bus to add if technologies providing these services are also included
                    # noded[c + '_bus_heat'] = solph.buses.Bus(label=c + '_bus_heat')                         # Space-heating bus 
                    # noded[c + '_bus_cool'] = solph.buses.Bus(label=c + '_bus_cool')                         # Space-cooling bus (currently treated as electric cooling demand, NO air-conditioning technology)
                    # noded[c + '_bus_DHW'] = solph.buses.Bus(label=c + '_bus_DHW')                           # DHW bus, before electric boiler


                    # ============ 
                    #  Sec 4.11 -- PV node
                    # ============ 
                    if c in has_PV_run:
                        # P_nom per 1 kWp, from profile, * PV_power_kwp, kWp installed per house
                        P_nom_pv_1kwp = pv_pnom_perkwp_dicts[current_season][c]    # kW per 1 kWp
                        P_nom_pv_tot  = P_nom_pv_1kwp * PV_power_kwp                        # kW for 3 kWp

                        fix_pv = CAN[current_season]["pv"][c]


                        # ----
                        #  If PV curtailment is enabled for excess generation
                        # ----
                        if ENABLE_PV_CURTAILMENT:
                            # intermediate bus for energy produced by PV
                            noded[c + '_bus_el_PV'] = solph.buses.Bus(label=c + '_bus_el_PV')         # Electric bus produced by PV (available to customer/grid or to the sink/excess "curtailment" if not absorbed)


                            # Source PV
                            noded[c + '_PV'] = solph.components.Source (label= c + '_PV',
                                outputs={noded[c + '_bus_el_PV']: solph.Flow(
                                    nominal_capacity = P_nom_pv_tot,                               # Total installed PV capacity for the household [kW].
                                    fix = fix_pv,
                                    )}, 
                                )

                            # "Fictitious" component/inverter to transfer electricity from PV to the house
                            noded[c + '_PV_to_house'] = solph.components.Converter(
                                label=c + '_PV_to_house',
                                inputs={noded[c + '_bus_el_PV']: solph.Flow()},
                                outputs={noded[c + '_bus_el']: solph.Flow()},
                                conversion_factors={noded[c + '_bus_el']: 1.0},
                            )

                            # "Fictitious" sink to track possible PV curtailment
                            noded[c + '_PV_curtail'] = solph.components.Sink(
                                label=c + '_PV_curtail',                                        
                                inputs={noded[c + '_bus_el_PV']: solph.Flow( 
                                    variable_costs = PV_curtail_cost,
                                )}
                            )

                        
                        # ---- 
                        # Case without curtailment
                        # ----
                        else:
                            noded[c + '_PV'] = solph.components.Source (label= c + '_PV',
                                outputs={noded[c + '_bus_el']: solph.Flow(
                                    nominal_capacity = P_nom_pv_tot,                               # Total installed PV capacity for the household [kW].
                                    fix = fix_pv,
                                    )}, 
                                )
                        
                            
                    #------------------------------------------                
                    
                    # ============ 
                    # Sec 4.11 -- Exchange between HOUSE and LV grid
                    # ============ 
                    lv_bus_key = _get_lv_bus_key_for_house(
                        c, i,
                        LV_ARCHETYPE=LV_ARCHETYPE,
                        num_households=num_households,
                        KERBER_BLUEPRINT_FILES=KERBER_BLUEPRINT_FILES,
                        HOUSE_TO_LV_BUS_LABEL=HOUSE_TO_LV_BUS_LABEL,
                        LV_ROOT_LABEL=LV_ROOT_LABEL,
                        _lv_bus_for_house=_lv_bus_for_house
                    )

                    noded[c + '_to_the_grid_BT'] = solph.components.Converter(
                        label=c + '_to_the_grid_BT',
                        inputs={noded[c + '_bus_el']: solph.Flow()},
                        outputs={noded[lv_bus_key]: solph.Flow(
                            variable_costs = ee_rev_can
                            )},
                        conversion_factors={noded[lv_bus_key]: eta_grid}
                    )

                    noded['from_the_grid_BT_to_' + c] = solph.components.Converter(
                        label='from_the_grid_BT_to_' + c,
                        inputs={noded[lv_bus_key]: solph.Flow(
                            variable_costs = ee_cost_can
                            )},
                        outputs={noded[c + '_bus_el']: solph.Flow()},
                        conversion_factors={noded[c + '_bus_el']: eta_grid}
                    )
                    #------------------------------------------


                    # ============ 
                    #  Sec 4.11 -- BESS
                    # ============ 
                    if c in has_BESS_run:

                        # --- BESS: OPT case ---
                        if _tech_is_opt(CONTROL_MODE, "BESS"):
                            noded[c + '_BESS_Opt'] = solph.components.GenericStorage(
                                label=c + '_BESS_Opt',
                                inputs={noded[c + '_bus_el']: solph.Flow(nominal_capacity = BESS_power_kw)},            
                                outputs={noded[c + '_bus_el']: solph.Flow(nominal_capacity = BESS_power_kw)},           
                                loss_rate=BESS_loss_rate,
                                initial_storage_level= initial_SoC_BESS,
                                balanced=True,                                                                  # imposes SoC_final == SoC_initial
                                max_storage_level=1,
                                min_storage_level=0,
                                nominal_storage_capacity= BESS_capacity_kwh,                                        
                                inflow_conversion_factor=eta_BESS,
                                outflow_conversion_factor=eta_BESS,
                            )



                        # --- BESS: RB case ---
                        elif _tech_is_rb(CONTROL_MODE, "BESS"):
                            # a) Load RB profiles (kWh/timestep)
                            dt_hours = float(timestep_minutes) / 60.0
                            s_charge_kwh = CAN[current_season]["bess_charge_kwh"][c].copy()
                            s_dis_kwh    = CAN[current_season]["bess_dis_kwh"][c].copy()


                            # numerical safety
                            s_charge_kwh = s_charge_kwh.clip(lower=0.0)
                            s_dis_kwh    = s_dis_kwh.clip(lower=0.0)

                            # b) # Two separate fictitious components are used to prevent simultaneous BESS charging and discharging.
                            mask_overlap = (s_charge_kwh > 1e-9) & (s_dis_kwh > 1e-9)

                            if mask_overlap.any():
                                charge_ge_dis = s_charge_kwh >= s_dis_kwh

                                # where charge >= discharge -> zero discharge
                                s_dis_kwh.loc[mask_overlap & charge_ge_dis] = 0.0

                                # where discharge > charge -> zero charge
                                s_charge_kwh.loc[mask_overlap & (~charge_ge_dis)] = 0.0

                            # c) convert flows to kW
                            s_charge_kw = s_charge_kwh / dt_hours
                            s_dis_kw    = s_dis_kwh    / dt_hours

                            # d) normalize fixed profile (0,1)
                            # ---- BESS charge, fictitious Sink
                            P_nom_charge = float(s_charge_kw.max())
                            if P_nom_charge > 0:
                                fix_charge = (s_charge_kw / P_nom_charge).values
                            else:
                                fix_charge = np.zeros(len(timeindex))
                                P_nom_charge = 0.0

                            # ---- BESS discharge, fictitious Source
                            P_nom_dis = float(s_dis_kw.max())
                            if P_nom_dis > 0:
                                fix_dis = (s_dis_kw / P_nom_dis).values
                            else:
                                fix_dis = np.zeros(len(timeindex))
                                P_nom_dis = 0.0

                            # e) "Fictitious" Sink to reproduce the "BESS charge" effect          <--- Storage with fixed flows could cause infeasibility errors
                            noded[c + '_BESS_charge_RB'] = solph.components.Sink(
                                label=c + '_BESS_charge_RB',                                        
                                inputs={noded[c + '_bus_el']: solph.Flow( 
                                    nominal_capacity=P_nom_charge,
                                    fix=fix_charge
                                )}
                            )

                            # f) "Fictitious" Source to reproduce the "BESS discharge" effect     <--- Storage with fixed flows could cause infeasibility errors
                            noded[c + '_BESS_discharge_RB'] = solph.components.Source(
                                label=c + '_BESS_discharge_RB',                                       
                                outputs={noded[c + '_bus_el']: solph.Flow( 
                                    nominal_capacity=P_nom_dis,
                                    fix=fix_dis
                                )}
                            )
                    #------------------------------------------


                    # ============ 
                    #  Sec 4.11 -- DHW boiler storage
                    # ============ 
                    if c in has_DHW_run:
                        # Physical parameters of the DHW unit associated with
                        # this household in the CURRENT run.
                        #
                        # Same values are used for RB and OPT:
                        # only the control representation changes downstream.
                        cap_kWh_DHW_c = float(capacity_dhw_by_house_eff_run[c])
                        dhw_power_kw_c = float(power_limit_dhw_by_house_eff_run[c])
                        DHW_loss_rate_c = float(loss_rate_dhw_by_house_eff_run[c])

                        dhw_nominal_power = (dhw_nominal_power_dicts[current_season][c])
                        dhw_fixed = (CAN[current_season]["dhw"][c])


                        # --- DHW: OPT case ---
                        if _tech_is_opt(
                            CONTROL_MODE,
                            "DHW",
                        ):

                            noded[c + '_DHW_storage'] = solph.components.GenericStorage(
                                label=c + '_DHW_storage',
                                inputs={noded[c + '_bus_el']: solph.Flow(nominal_capacity=dhw_power_kw_c)},
                                outputs={noded[c + '_bus_DHW_use']: solph.Flow()},
                                loss_rate=DHW_loss_rate_c,
                                initial_storage_level=None,
                                balanced=True,                                           # imposes SoC_final == SoC_initial
                                max_storage_level=1,
                                min_storage_level= min_storage_level_ESWH,
                                nominal_storage_capacity=(cap_kWh_DHW_c),
                                inflow_conversion_factor=eta_DHW,
                                outflow_conversion_factor=1,                                # electric boiler, electricity enters and thermal energy exits (input efficiency 80%; there is no discharge efficiency because the output is heat going directly to DHW use)
                            )

                            # b) DHW thermal demand
                            noded[c + '_dmd_th_DHW'] = solph.components.Sink(
                                label=c + '_dmd_th_DHW',                                        # always check that in "Module 03" always calculates DHW thermal demand (eff boiler=1 li)
                                inputs={noded[c + '_bus_DHW_use']: solph.Flow( 
                                    nominal_capacity = dhw_nominal_power,
                                    fix           = dhw_fixed
                                )}
                            )

                            # c) Additional resistance to provide boiler boost (case Converter)
                            noded[c + '_DHW_boost'] = solph.components.Converter(
                                label= c + '_DHW_boost',
                                inputs={noded[c + '_bus_el']: solph.Flow(
                                        nominal_capacity = P_boost_DHW,                 # additional kW of "boost" resistance to cover DHW demand peaks
                                        variable_costs= cost_boost_DHW                  # additional cost to discourage use of this technology
                                )},
                                outputs={noded[c + '_bus_DHW_use']: solph.Flow()},
                                conversion_factors={noded[c + '_bus_DHW_use']: eta_DHW}
                            )


                        # --- DHW: RB case ---
                        elif _tech_is_rb(CONTROL_MODE, "DHW"):
                            # Import fixed profiles (case RB)
                            s = to_canonical(rb_prof["dhw_charge_kw"][c], idx_real, timeindex_canonico).fillna(0.0).astype(float)
                            P_nom = float(s.max()) if float(s.max()) > 0 else 0.0
                            fix = (s / P_nom).values if P_nom > 0 else np.zeros(len(timeindex))

                            # "Fictitious" component replicating electricity absorption by the boiler to meet DHW demand
                            noded[c + "_DHW_charge_RB_sink"] = solph.components.Sink(
                                label=c + "_DHW_charge_RB_sink",
                                inputs={noded[c + "_bus_el"]: solph.Flow(
                                    nominal_capacity=P_nom, 
                                    fix=fix
                                    )}
                            )

                            # "Additional fictitious" component replicating possible need for additional resistance (DHW_boost similare) where there is DHW demand but the boiler is discharged
                            if ENABLE_DHW_BOOST_RB:
                                if "dhw_boost_kw" in rb_prof and c in rb_prof["dhw_boost_kw"].columns:
                                    s_boost = to_canonical(rb_prof["dhw_boost_kw"][c], idx_real, timeindex_canonico).fillna(0.0).astype(float)
                                else:
                                    s_boost = pd.Series(0.0, index=timeindex_canonico)

                                P_nom_boost = float(P_boost_DHW) if float(P_boost_DHW) > 0 else float(s_boost.max())
                                if P_nom_boost > 0:
                                    fix_boost = (s_boost / P_nom_boost).clip(lower=0.0, upper=1.0).values
                                else:
                                    fix_boost = np.zeros(len(timeindex))

                                noded[c + "_DHW_boost"] = solph.components.Sink(    # Same name as in the OPT case
                                    label=c + "_DHW_boost",                         # Same name as in the OPT case, even though here it is a Sink and not a converter: needed to facilitate cost association to individual households, Sec. 4.13
                                    inputs={noded[c + "_bus_el"]: solph.Flow(
                                        nominal_capacity=P_nom_boost,
                                        fix=fix_boost,
                                        variable_costs=cost_boost_DHW       # Same cost, shortage cost, as in the OPT case for "DHW_boost"
                                    )}
                                )
                        
                    #------------------------------------------


                    # ============ 
                    # Sec 4.11 -- BEV electric vehicle
                    # ============ 
                    if c in has_BEV_run:
                        # BEV capacity for this household, in kWh, taken from Excel
                        bev_cap = ev_capacity_by_case.loc[c]
                        P_nom_bev = dmd_BEV_pnom_dicts[current_season][c]
                        fix_bev = CAN[current_season]["bev"][c]

                        # if for some reason it is NaN or <=0, you may decide to skip or use a default
                        if pd.isna(bev_cap) or bev_cap <= 0:
                            # e.g. skip BEV storage creation for this house
                            continue


                        # --- BEV: OPT case ---
                        if _tech_is_opt(CONTROL_MODE, "BEV"):
                            # a) Car battery
                            noded[c + '_BEV_storage'] = solph.components.GenericStorage(   
                                label=c + '_BEV_storage',
                                inputs={noded[c + '_bus_el_charge_BEV']: solph.Flow()},                                     
                                outputs={noded[c + '_bus_el_discharge_BEV']: solph.Flow()},                                 
                                loss_rate=BEV_loss_rate, 
                                initial_storage_level= None,    # initial_SoC_BEV,
                                balanced=True,                                                                             # for BEVs, cars may leave home and travel at the end, therefore not being able to "recharge before the last trip"
                                max_storage_level=1,
                                min_storage_level= min_storage_level_BEV,                                                   # Impose a min SoC, e.g. 10% (cars must always have a minimum charge (always, even during trips))
                                nominal_storage_capacity=float(bev_cap), 
                                inflow_conversion_factor=EV_charge_discharge_efficiency, 
                                outflow_conversion_factor=EV_charge_discharge_efficiency, 
                            )

                            # b) Domestic and outdoor BEV charging-availability profiles. --------
                            season_key = str(current_season).strip().lower()
                            
                            if season_key not in max_occupancy_profiles_by_season:
                                raise KeyError(
                                    f"profiles_max_occ_by_season does not contain the season '{season_key}'. "
                                    f"Available keys: {list(max_occupancy_profiles_by_season.keys())[:10]}"
                                )

                            max_profile     = CAN[current_season]["occ"][c].values
                            
                            # ---- OUTDOOR PROFILE, complementary: 1 = away from home -> possible outdoor charging
                            outdoor_profile = CAN[current_season]["outdoor"][c].values
                            #-------------------------

                            # c) Home charger
                            noded[c + '_charger'] = solph.components.Converter(
                                label= c + '_charger',
                                inputs={noded[c + '_bus_el']: solph.Flow()},      
                                    outputs={noded[c + '_bus_el_charge_BEV']: solph.Flow(
                                    nominal_capacity = home_charging_power_kw,
                                    max = max_profile,                                                  # Can charge the BEV at home only when occupancy_df is actually 1
                                )},
                                conversion_factors={noded[c + '_bus_el_charge_BEV']: eta_charger},
                            )


                            # d) BEV electric demand 
                            noded[c + '_dmd_elc_BEV'] = solph.components.Sink(
                                label=c + '_dmd_elc_BEV',                                               # <----- WARNING must be electric mobility demand
                                inputs={noded[c + '_bus_el_discharge_BEV']: solph.Flow( 
                                    nominal_capacity = P_nom_bev,
                                    fix           = fix_bev
                                )}
                            )

                            # e) V2Grid option (if enabled)
                            if c in has_V2G_run:
                                noded[c + '_V2G'] = solph.components.Converter(
                                    label= c + '_V2G',
                                    inputs={noded[c + '_bus_el_discharge_BEV']: solph.Flow(
                                        nominal_capacity= home_charging_power_kw,                                    # Can vary charging power and re-export to the domestic grid
                                        max = max_profile,                                                # Can do V2G, like charging the BEV, only when at home (with occupancy_df=1)
                                    )},
                                    outputs={noded[c + '_bus_el']: solph.Flow()},
                                    conversion_factors={noded[c + '_bus_el']: eta_V2G},
                                )


                            # f) Additional source used to represent outdoor EV charging. (e.g. work, shopping center, etc.)
                            noded[c + '_charger_outdoor'] = solph.components.Source (
                                label = f'{c}_charger_outdoor',
                                outputs={noded[c + '_bus_el_charge_BEV']: solph.Flow(
                                                        nominal_capacity = P_outdoor_charger,             # Charging power (overall) Maximum outdoor BEV charging power [kW] (constraint maybe not to impose because it is a "shortage")
                                                        variable_costs= cost_outdoor_charger,               # additional cost to charge the BEV when away from home 
                                                        max = outdoor_profile,                              # defines the profile in which these storage components can be used (can charge when away from home, therefore when Occ = 0 )
                                                        )}, 
                                                )

                        # --- BEV: RB case ---
                        elif _tech_is_rb(CONTROL_MODE, "BEV"):
                            # Import fixed profiles (case RB)
                            s = to_canonical(rb_prof["bev_charge_kw"][c], idx_real, timeindex_canonico).fillna(0.0).astype(float)
                            P_nom = float(s.max()) if float(s.max()) > 0 else 0.0
                            fix = (s / P_nom).values if P_nom > 0 else np.zeros(len(timeindex))

                            # "Fictitious" Sink to represent non-smart electric absorption to charge the BEV
                            noded[c + "_BEV_charge_RB_sink"] = solph.components.Sink(
                                label=c + "_BEV_charge_RB_sink",
                                inputs={noded[c + "_bus_el"]: solph.Flow(
                                    nominal_capacity=P_nom,
                                        fix=fix
                                        )}
                            )

                            # Add an additional "charger_outdoor" component also for RB cases
                            if ENABLE_BEV_OUTDOOR_RB:
                                if "bev_outdoor_kw" in rb_prof and c in rb_prof["bev_outdoor_kw"].columns:
                                    s_outdoor = to_canonical(rb_prof["bev_outdoor_kw"][c], idx_real, timeindex_canonico).fillna(0.0).astype(float)
                                else:
                                    s_outdoor = pd.Series(0.0, index=timeindex_canonico)

                                P_nom_outdoor = float(P_outdoor_charger) if float(P_outdoor_charger) > 0 else float(s_outdoor.max())
                                if P_nom_outdoor > 0:
                                    fix_outdoor = (s_outdoor / P_nom_outdoor).clip(lower=0.0, upper=1.0).values
                                else:
                                    fix_outdoor = np.zeros(len(timeindex))

                                # dummy bus to account for kWh and costs without altering the domestic electricity balance
                                noded[c + "_bus_el_outdoor_RB"] = solph.Bus(label=c + "_bus_el_outdoor_RB")

                                noded[c + "_charger_outdoor"] = solph.components.Source(        # Same name as in the OPT case: needed to facilitate cost association to individual households, Sec. 4.13
                                    label=c + "_charger_outdoor",                               # Same name as in the OPT case: needed to facilitate cost association to individual households, Sec. 4.13
                                    outputs={noded[c + "_bus_el_outdoor_RB"]: solph.Flow(
                                        nominal_capacity=P_nom_outdoor,
                                        fix=fix_outdoor,
                                        variable_costs=cost_outdoor_charger
                                    )}
                                )

                                noded[c + "_outdoor_use_RB_sink"] = solph.components.Sink(
                                    label=c + "_outdoor_use_RB_sink",
                                    inputs={noded[c + "_bus_el_outdoor_RB"]: solph.Flow(
                                        nominal_capacity=P_nom_outdoor,
                                        fix=fix_outdoor
                                    )}
                                )

                    #------------------------------------------



                    # ============ 
                    #   Sec 4.11 -- Sink
                    # ============ 
                    # a) BASE electric demand (appliances)
                    P_nom_base = dmd_base_pnom_dicts[current_season][c]
                    fix_base = CAN[current_season]["base"][c]

                    noded[c + '_dmd_elc_base'] = solph.components.Sink(
                        label=c + '_dmd_elc_base',
                        inputs={noded[c + '_bus_el']: solph.Flow(
                            nominal_capacity = P_nom_base,
                            fix           = fix_base
                        )}
                    )

                    # b) COOKING electric demand (exogenous induction) 
                    if c in has_IND_run:
                        P_nom_cook = dmd_Cook_pnom_dicts[current_season][c]
                        fix_cook = CAN[current_season]["Cook"][c]

                        noded[c + '_dmd_elc_Cook'] = solph.components.Sink(
                            label=c + '_dmd_elc_Cook',  # <----- WARNING whether electric bus or something else
                            inputs={noded[c + '_bus_el']: solph.Flow(
                                nominal_capacity = P_nom_cook,
                                fix           = fix_cook
                            )}
                        )


                    # c) COOLING electric demand (exogenous through air conditioners) 
                    if c in has_COOL_run:
                        P_nom_cool = dmd_Cool_pnom_dicts[current_season][c]
                        fix_cool = CAN[current_season]["Cool"][c]
                        
                        noded[c + '_dmd_elc_Cool'] = solph.components.Sink(
                            label=c + '_dmd_elc_Cool',
                            inputs={noded[c + '_bus_el']: solph.Flow(
                                nominal_capacity = P_nom_cool,
                                fix           = fix_cool
                            
                            )}
                        )


                    # d) HEATING electric demand (exogenous through HP) 
                    if c in has_HP_run:
                        P_nom_heat = dmd_Heat_pnom_dicts[current_season][c]
                        fix_heat = CAN[current_season]["Heat"][c]  
                        
                        noded[c + '_dmd_elc_Heat'] = solph.components.Sink(
                            label=c + '_dmd_elc_Heat',  
                            inputs={noded[c + '_bus_el']: solph.Flow(
                            nominal_capacity = P_nom_heat,
                            fix           = fix_heat
                            )}
                        )
                #------------------------------------------


                # ============ 
                # Sec 4.11 -- Export to and import from MV grid
                # ============

                # Definition/acquisition of static limits defined on the MV transformer from PF analysis
                cap_imp_mt_kw = _get_pf_interface_limit_kw(
                    PF_INTERFACE_LIMITS_MAP,
                    direction="import",
                    fallback_kw=secondary_substation_active_power_kw,
                )

                cap_exp_mt_kw = _get_pf_interface_limit_kw(
                    PF_INTERFACE_LIMITS_MAP,
                    direction="export",
                    fallback_kw=secondary_substation_active_power_kw,
                )
                
                
                
                noded['excess_to_grid_MT'] = solph.components.Sink(
                    label='excess_to_grid_MT',
                    inputs={lv_root_obj: solph.Flow(
                        nominal_capacity=cap_exp_mt_kw,                     # PF analysis may impose transformer-loading constraints both for import from MV and export
                    )}
                )

                noded['supply_from_grid_MT'] = solph.components.Source(
                    label='supply_from_grid_MT',
                    outputs={lv_root_obj: solph.Flow(
                        # nominal_capacity = pot_cabina_secondaria
                        nominal_capacity=cap_imp_mt_kw,                     # PF analysis may impose transformer-loading constraints both for import from MV and export
                        # max = ,
                        )}
                )
                #------------------------------------------

            
            # ---------------------------
            # Sec 4.11 -- 1.3. Energy-system construction
            # ---------------------------
            with timed(f"{MODE}/{current_season} - build ES", timings):
                # es = solph.EnergySystem(timeindex=timeindex)
                es = solph.EnergySystem(timeindex=timeindex_canonico)
                es.add(*noded.values())


                # # ---------------------------
                # # 1.4. Print nodes (only at the first iteration)
                # # ---------------------------
                # if prima_iterazione:
                #     print("-" * 40)
                #     print("The following objects have been created from the Excel sheet:")
                #     for node in es.nodes:
                #         oobj = str(type(node)).replace("<class 'oemof.solph.", "").replace("'>", "")
                #         print(oobj + ":", node.label)
                #     prima_iterazione = False
                #     print("-" * 40)


            # ===========================
            # Sec 4.11 -- 1.5. Create the Pyomo/Solph model
            # ===========================
            with timed(f"{MODE}/{current_season} - build Pyomo model", timings):
                # calculate timeincrement in hours and create the model (pass it explicitly)
                td = pd.to_timedelta(timeindex.freq)
                dt_hours = td.total_seconds() / 3600.0   # es. 10 min → 0.166666...
                
                om = solph.Model(es, timeincrement=timeincrement_canonico)

                print("-"*20)
                logging.info('OPTIMIZATION')
            


       
            # ===========================
            # Sec 4.11 -- 1.6) Solve optimization problem with Gurobi
            # ===========================

            with timed(f"{MODE}/{current_season} - solve", timings):

                opt = po.SolverFactory("gurobi")
                opt.options["Method"] = 2
                res = opt.solve(om, tee=True)

                tc = res.solver.termination_condition
                print(">>> termination_condition:", tc)

                lp_path = None
                iis_ilp_path = None
                iis_json_path = None
                gurobi_status = None

                # --------------------------------------------------------
                # Classify solve
                # --------------------------------------------------------
                if tc == TerminationCondition.optimal:
                    solve_status_label = "OK"

                elif tc == TerminationCondition.infeasible:
                    solve_status_label = "FAIL (INFEASIBLE)"

                elif tc == TerminationCondition.unbounded:
                    solve_status_label = "FAIL (UNBOUNDED)"

                elif tc == TerminationCondition.infeasibleOrUnbounded:
                    solve_status_label = "FAIL (INF_OR_UNBD)"

                else:
                    solve_status_label = f"FAIL ({str(tc).upper()})"


                # --------------------------------------------------------
                # Non-optimal solve: save LP, but DO NOT re-solve here
                # --------------------------------------------------------
                if tc != TerminationCondition.optimal:

                    print(
                        f"[NON-OPTIMAL SOLVE] {RUN_LABEL} | "
                        f"{current_season} | {solve_status_label}"
                    )

                    if DEBUG_INFEASIBLE_ON_SOLVE:

                        lp_path = os.path.join(
                            optimization_output_folder,
                            f"debug_infeasible_{current_season}.lp",
                        )

                        om.write(
                            lp_path,
                            io_options={"symbolic_solver_labels": True},
                        )

                        print("[DEBUG] wrote LP:", lp_path)


                    debug_info = {
                        "run_uid": RUN_UID,
                        "run_label": RUN_LABEL,
                        "case_label": RUN_CFG.get("case_label", RUN_LABEL),
                        "season": current_season,
                        "mode": MODE,
                        "termination_condition": str(tc),
                        "solve_status": solve_status_label,
                        "lp_path": lp_path,
                        "iis_ilp_path": None,
                        "iis_json_path": None,
                        "gurobi_status": None,
                    }

                    debug_key = (str(RUN_UID), str(current_season))

                    INFEASIBLE_DEBUG_REGISTRY[debug_key] = debug_info
                    LAST_INFEASIBLE_KEY = debug_key
                    LAST_INFEASIBLE_SEASON = current_season


                # --------------------------------------------------------
                # Register status for final Sec. 4.11 table
                # --------------------------------------------------------
                status_key = (str(RUN_UID), str(current_season))

                SOLVE_STATUS_REGISTRY[status_key].update({
                    "status": solve_status_label,
                    "termination_condition_initial": str(tc),
                    "termination_condition_final": str(tc),
                    "solver_status": str(res.solver.status),
                    "gurobi_status": None,
                    "lp_path": lp_path,
                    "iis_ilp_path": None,
                    "iis_json_path": None,
                })

            # ------------------------------------------------------------
            # Skip post-processing for non-optimal solves
            # ------------------------------------------------------------

            if tc != TerminationCondition.optimal:

                print(
                    f"[SKIP POSTPROCESS] {RUN_LABEL} | "
                    f"{current_season} | {solve_status_label}"
                )

                del om, es, noded
                gc.collect()

                continue



            with timed(f"{MODE}/{current_season} - postprocess", timings):
                #----------------------------
                # Sec 4.11 -- 1.7 Result extraction (exchanged flows) -- POST-PROCESS
                #----------------------------
    
                # Process results
                results = solph.processing.results(om)

                # 1.7.0) Build cost information for the observed configuration
                (
                df_system_cost_ts,
                df_system_cost_summary, 
                df_objective_terms_ts,
                df_objective_terms_by_flow,
                df_objective_terms_summary,
                df_objective_reconciliation,
                ) = build_opt_system_cost_outputs(
                    om=om,
                    results=results,
                    lv_root=lv_root,
                    idx_real=idx_real,
                    timeindex_canonico=timeindex_canonico,
                    current_season=current_season,
                    MODE=MODE,
                    ee_cost_series=ee_cost_series,
                    ee_rev_series=ee_rev_series,
                    timestep_minutes=timestep_minutes,
                    LV_ARCHETYPE=LV_ARCHETYPE,
                    num_households=num_households,
                    selected_region=selected_region,
                    USE_PF_LINE_LIMITS_V1=USE_PF_LINE_LIMITS_V1,
                    PF_LINE_LIMITS_MAP=PF_LINE_LIMITS_MAP,
                    PF_INTERFACE_LIMITS_MAP=PF_INTERFACE_LIMITS_MAP,
                    cap_imp_mt_kw=cap_imp_mt_kw if "cap_imp_mt_kw" in locals() else None,
                    cap_exp_mt_kw=cap_exp_mt_kw if "cap_exp_mt_kw" in locals() else None,
                    cost_boost_dhw_eur_per_kwh=cost_boost_DHW,
                    cost_outdoor_charger_eur_per_kwh=cost_outdoor_charger,
                    include_dhw_boost_in_effective_costs=INCLUDE_DHW_BOOST_IN_EFFECTIVE_COSTS,
                    shortage_cost_eur_per_kwh=0.0,
                )


                df_system_cost_summary["mode_label"] = MODE_LABEL

                df_system_cost_summary["share_FV_config"] = RUN_SHARES['PV']
                df_system_cost_summary["share_BESS_config"] = RUN_SHARES["BESS"]
                df_system_cost_summary["share_BEV_config"] = RUN_SHARES["BEV"]
                df_system_cost_summary["share_DHW_config"] = RUN_SHARES["DHW"]
                df_system_cost_summary["run_output_tag"] = RUN_DIRS["tag"]

                df_system_cost_summary["control_BESS"] = CONTROL_MODE["BESS"]
                df_system_cost_summary["control_BEV"] = CONTROL_MODE["BEV"]
                df_system_cost_summary["control_DHW"] = CONTROL_MODE["DHW"]

                df_system_cost_summary["active_n_BESS"] = len(has_BESS_run)
                df_system_cost_summary["active_n_BEV"] = len(has_BEV_run)
                df_system_cost_summary["active_n_DHW"] = len(has_DHW_run)


                # ------------------------------------------------------------
                # Household objective-cost allocation
                #
                # IMPORTANT:
                # this dataframe is required for economic calculations.
                # SAVE_OBJECTIVE_TERMS_BY_HOUSE_XLSX controls only its export,
                # not whether it is calculated.
                # ------------------------------------------------------------

                if (
                    df_objective_terms_by_flow is not None
                    and not df_objective_terms_by_flow.empty
                ):

                    df_objective_terms_by_house = (
                        df_objective_terms_by_flow
                        .dropna(subset=["house_id"])
                        .groupby(
                            [
                                "house_id",
                                "category",
                                "cost_group",
                                "description",
                            ],
                            as_index=False,
                        )
                        .agg(
                            term_total_eur=("term_total_eur", "sum"),
                            term_abs_total_eur=("term_abs_total_eur", "sum"),
                            flow_total_kwh=("flow_total_kwh", "sum"),
                            n_flows=("from_label", "count"),
                        )
                        .sort_values(
                            [
                                "house_id",
                                "category",
                                "cost_group",
                                "description",
                            ]
                        )
                        .reset_index(drop=True)
                    )

                else:

                    df_objective_terms_by_house = pd.DataFrame()














                # ------------------------------------------------------------
                # objective_terms_ts / summary - clean version for export
                if FILTER_OBJECTIVE_TERMS_TS_FOR_EXPORT:
                    objective_terms_ts_cols_export = [
                        "bt_energy_import_settlement_eur",
                        "bt_energy_export_settlement_eur",
                        "dhw_boost_cost_eur",
                        "bev_outdoor_charge_cost_eur",
                        "pv_curtail_cost_eur",
                        "objective_variable_cost_terms_eur",
                    ]
                    df_objective_terms_ts_export = (
                        df_objective_terms_ts
                        .reindex(columns=objective_terms_ts_cols_export, fill_value=0.0)
                        .copy()
                    )
                else:
                    df_objective_terms_ts_export = df_objective_terms_ts.copy()

                if FILTER_OBJECTIVE_TERMS_SUMMARY_FOR_EXPORT:
                    categories_export = [
                        "bt_energy_import_settlement_eur",
                        "bt_energy_export_settlement_eur",
                        "dhw_boost_cost_eur",
                        "bev_outdoor_charge_cost_eur",
                        "pv_curtail_cost_eur",
                    ]
                    df_objective_terms_summary_export = (
                        df_objective_terms_summary[
                            df_objective_terms_summary["category"].isin(categories_export)
                        ]
                        .sort_values(["cost_group", "category"])
                        .reset_index(drop=True)
                    )
                else:
                    df_objective_terms_summary_export = df_objective_terms_summary.copy()
                # ------------------------------------------------------------



                system_cost_rows_mode.extend(df_system_cost_summary.to_dict(orient="records"))

                # 1.7.1) Flows on LV grid (all buses + root)
                flow_grid = {bus_label: collect_bus_sequences_from_results(results, bus_label, timeindex_canonico) for bus_label in LV_BUS_LABELS_EXPORT}
                flow_grid_df = pd.concat(flow_grid, axis=1) if flow_grid else pd.DataFrame(index=timeindex_canonico)
                flow_grid_root = flow_grid.get(lv_root, pd.DataFrame(index=timeindex_canonico))

                # 1.7.2) Domestic bus_el flows for each house
                flow_el = {c: collect_bus_sequences_from_results(results, f"{c}_bus_el", timeindex_canonico) for c in ID}
                flow_el_df = pd.concat(flow_el, axis=1) if flow_el else pd.DataFrame(index=timeindex_canonico)


                # 1.7.2bis) Flows on the PV-curtailment bus (only households with PV and only if active)
                if ENABLE_PV_CURTAILMENT:
                    flow_PV_curtail = {
                        c: collect_bus_sequences_from_results(results, f"{c}_bus_el_PV", timeindex_canonico)
                        for c in has_PV_run
                    }
                    flow_PV_curtail_df = pd.concat(flow_PV_curtail, axis=1) if flow_PV_curtail else pd.DataFrame(index=timeindex_canonico)
                else:
                    flow_PV_curtail_df = pd.DataFrame(index=timeindex_canonico)





                # ------------- MIXED VERSION - Export comparable DHW/BEV flows -------------------------
                # 1.7.3) Input demand profiles consistent with the active run configuration

                dhw_input_profile = {
                    c: _fixed_input_series_kw(
                        fixed_dhw_electric_demand_dicts,
                        dhw_nominal_power_dicts,
                        current_season,
                        c,
                        idx_real,
                        timeindex_canonico,
                    )
                    for c in has_DHW_run
                }
                dhw_input_profile_df = (
                    pd.concat(dhw_input_profile, axis=1)
                    if dhw_input_profile
                    else pd.DataFrame(index=timeindex_canonico)
                )

                bev_input_profile = {
                    c: _fixed_input_series_kw(
                        dmd_BEV_el_fix_dicts,
                        dmd_BEV_pnom_dicts,
                        current_season,
                        c,
                        idx_real,
                        timeindex_canonico,
                    )
                    for c in has_BEV_run
                }
                bev_input_profile_df = (
                    pd.concat(bev_input_profile, axis=1)
                    if bev_input_profile
                    else pd.DataFrame(index=timeindex_canonico)
                )

                # 1.7.4) Comparable DHW flows ----------------------------------------------
                if _tech_is_opt(CONTROL_MODE, "DHW"):
                    # DHW OPT case: real DHW_use bus
                    flow_DHW_use = {
                        c: collect_bus_sequences_from_results(
                            results,
                            f"{c}_bus_DHW_use",
                            timeindex_canonico,
                        )
                        for c in has_DHW_run
                    }

                    flow_DHW_use_df = (
                        pd.concat(flow_DHW_use, axis=1)
                        if flow_DHW_use
                        else pd.DataFrame(index=timeindex_canonico)
                    )

                elif _tech_is_rb(CONTROL_MODE, "DHW"):
                    # DHW RB case: there is no real operational bus_DHW_use as in OPT.
                    # Export a comparable dataframe with final demand + RB electric input.
                    dhw_cols = {}

                    for c in has_DHW_run:
                        s_dmd = dhw_input_profile.get(
                            c,
                            pd.Series(0.0, index=timeindex_canonico),
                        )

                        s_charge = _rb_profile_series_kw(
                            rb_prof,
                            "dhw_charge_kw",
                            c,
                            idx_real,
                            timeindex_canonico,
                        )

                        s_boost = _rb_profile_series_kw(
                            rb_prof,
                            "dhw_boost_kw",
                            c,
                            idx_real,
                            timeindex_canonico,
                        )

                        s_total = s_charge.add(s_boost, fill_value=0.0)

                        dhw_cols[(c, "input_dmd_th_DHW_kw", "flow")] = s_dmd
                        dhw_cols[(c, "el_DHW_charge_RB_sink_kw", "flow")] = s_charge
                        dhw_cols[(c, "el_DHW_boost_RB_kw", "flow")] = s_boost
                        dhw_cols[(c, "DHW_total_el_RB_kw", "flow")] = s_total

                    flow_DHW_use_df = pd.DataFrame(dhw_cols, index=timeindex_canonico)

                else:
                    # DHW = NO
                    flow_DHW_use_df = pd.DataFrame(index=timeindex_canonico)

                # 1.7.5) Comparable BEV flows ----------------------------------------------
                if _tech_is_opt(CONTROL_MODE, "BEV"):
                    # BEV OPT case: real BEV charge/discharge buses
                    flow_charge_el_BEV = {
                        c: collect_bus_sequences_from_results(
                            results,
                            f"{c}_bus_el_charge_BEV",
                            timeindex_canonico,
                        )
                        for c in has_BEV_run
                    }

                    flow_charge_el_BEV_df = (
                        pd.concat(flow_charge_el_BEV, axis=1)
                        if flow_charge_el_BEV
                        else pd.DataFrame(index=timeindex_canonico)
                    )

                    flow_discharge_el_BEV = {
                        c: collect_bus_sequences_from_results(
                            results,
                            f"{c}_bus_el_discharge_BEV",
                            timeindex_canonico,
                        )
                        for c in has_BEV_run
                    }

                    flow_discharge_el_BEV_df = (
                        pd.concat(flow_discharge_el_BEV, axis=1)
                        if flow_discharge_el_BEV
                        else pd.DataFrame(index=timeindex_canonico)
                    )

                elif _tech_is_rb(CONTROL_MODE, "BEV"):
                    # BEV RB case: comparable export with input mobility demand,
                    # RB home charging, RB outdoor charging and shortage.
                    bev_charge_cols = {}
                    bev_discharge_cols = {}

                    dt_h = float(timestep_minutes) / 60.0

                    for c in has_BEV_run:
                        s_bev_dmd = bev_input_profile.get(
                            c,
                            pd.Series(0.0, index=timeindex_canonico),
                        )

                        s_home_charge = _rb_profile_series_kw(
                            rb_prof,
                            "bev_charge_kw",
                            c,
                            idx_real,
                            timeindex_canonico,
                        )

                        s_outdoor = _rb_profile_series_kw(
                            rb_prof,
                            "bev_outdoor_kw",
                            c,
                            idx_real,
                            timeindex_canonico,
                        )

                        s_total_charge = s_home_charge.add(s_outdoor, fill_value=0.0)

                        if (
                            "bev_shortage_kwh" in rb_prof
                            and isinstance(rb_prof["bev_shortage_kwh"], pd.DataFrame)
                            and c in rb_prof["bev_shortage_kwh"].columns
                        ):
                            s_shortage_kwh = to_canonical(
                                rb_prof["bev_shortage_kwh"][c],
                                idx_real,
                                timeindex_canonico,
                            ).fillna(0.0).astype(float)

                            s_shortage_kw = s_shortage_kwh / dt_h if dt_h > 0 else s_shortage_kwh * 0.0
                        else:
                            s_shortage_kw = pd.Series(
                                0.0,
                                index=timeindex_canonico,
                                dtype=float,
                            )

                        bev_charge_cols[(c, "input_dmd_elc_BEV_kw", "flow")] = s_bev_dmd
                        bev_charge_cols[(c, "BEV_charge_home_RB_kw", "flow")] = s_home_charge
                        bev_charge_cols[(c, "BEV_charge_outdoor_RB_kw", "flow")] = s_outdoor
                        bev_charge_cols[(c, "BEV_charge_total_RB_kw", "flow")] = s_total_charge

                        bev_discharge_cols[(c, "input_dmd_elc_BEV_kw", "flow")] = s_bev_dmd
                        bev_discharge_cols[(c, "BEV_shortage_RB_kw", "flow")] = s_shortage_kw

                    flow_charge_el_BEV_df = pd.DataFrame(bev_charge_cols, index=timeindex_canonico)
                    flow_discharge_el_BEV_df = pd.DataFrame(bev_discharge_cols, index=timeindex_canonico)

                else:
                    # BEV = NO
                    flow_charge_el_BEV_df = pd.DataFrame(index=timeindex_canonico)
                    flow_discharge_el_BEV_df = pd.DataFrame(index=timeindex_canonico)



                # 1.7.5) Check that all DataFrames contain n_ts timesteps
                for _df_name in [
                        "flow_el_df", 
                        "flow_PV_curtail_df",
                        "flow_grid_df", 
                        "flow_grid_root",
                        "flow_DHW_use_df", 
                        "flow_charge_el_BEV_df",
                        "flow_discharge_el_BEV_df",
                        # "flow_heat_df",
                        "dhw_input_profile_df",
                        "bev_input_profile_df",
                        ]:
                    _df = locals()[_df_name]
                    if len(_df.index) != len(timeindex_canonico):
                        # Realign the series and fill missing values with zero.
                        locals()[_df_name] = _df.reindex(timeindex_canonico).fillna(0.0)


                #----------------------------
                # Sec 4.11 -- 1.8 Reindex results on the real datetime of the season
                #----------------------------
                flow_el_df              = set_real_index(flow_el_df, idx_real)
                flow_PV_curtail_df      = set_real_index(flow_PV_curtail_df, idx_real)
                flow_grid_df            = set_real_index(flow_grid_df, idx_real)
                flow_grid_root          = set_real_index(flow_grid_root, idx_real)
                flow_DHW_use_df          = set_real_index(flow_DHW_use_df, idx_real)
                flow_charge_el_BEV_df    = set_real_index(flow_charge_el_BEV_df, idx_real)
                flow_discharge_el_BEV_df = set_real_index(flow_discharge_el_BEV_df, idx_real)
                # flow_heat_df            = set_real_index(flow_heat_df, idx_real)
                dhw_input_profile_df      = set_real_index(dhw_input_profile_df, idx_real)
                bev_input_profile_df      = set_real_index(bev_input_profile_df, idx_real)


                # ------------------------------------------------------------
                # Sec 4.11 -- 1.8a) Build storage flows for mixed SOC calculation
                # ------------------------------------------------------------
                # These dataframes are used ONLY for SOC calculation.

                # -- DHW --
                if flow_DHW_use_df is not None and not flow_DHW_use_df.empty:
                    dhw_house_flows_df = pd.concat([flow_el_df, flow_DHW_use_df], axis=1)
                else:
                    print(
                        f"[WARN][SOC DHW][{MODE}/{current_season}] "
                        "flow_DHW_use_df is empty: using only flow_el_df for DHW SOC calculation."
                    )
                    dhw_house_flows_df = flow_el_df.copy()

                # -- BEV --
                _bev_soc_parts = []

                if flow_charge_el_BEV_df is not None and not flow_charge_el_BEV_df.empty:
                    _bev_soc_parts.append(flow_charge_el_BEV_df)

                if flow_discharge_el_BEV_df is not None and not flow_discharge_el_BEV_df.empty:
                    _bev_soc_parts.append(flow_discharge_el_BEV_df)

                if len(_bev_soc_parts) > 0:
                    ev_house_flows_df = pd.concat(_bev_soc_parts, axis=1)
                else:
                    print(
                        f"[WARN][SOC BEV][{MODE}/{current_season}] "
                        "No BEV flow available: creating an empty dataframe."
                    )
                    ev_house_flows_df = pd.DataFrame(index=idx_real)

                # Normalize columns to flow-tuple format.
                dhw_house_flows_df = convert_busseq_to_flowtuple_df(dhw_house_flows_df)
                ev_house_flows_df = convert_busseq_to_flowtuple_df(ev_house_flows_df)


                # ------------------------------------------------------------
                # Sec 4.11 -- 1.8b) DHW parameters of the current run
                # ------------------------------------------------------------
                #
                # capacity_dhw_by_house_eff_run,
                # power_limit_dhw_by_house_eff_run,
                # loss_rate_dhw_by_house_eff_run and
                # initial_soc_dhw_by_house_eff_run
                #
                # were already created immediately after HAS_RUN,
                # before model construction.
                #
                # They are intentionally reused here for SOC extraction
                # and output generation, so the exact same physical
                # parameters are used during model construction and export.
                # ------------------------------------------------------------

                # ------------------------------------------------------------
                # Sec 4.11 -- 1.8c) Build LV-line dataframe
                # ------------------------------------------------------------
                # Needed for both:
                # - pickle flow_grid_lines_*.pkl.gz
                # - export tecnico net_lines_*.xlsx
                # - charts 4.12

                if LV_BUS_LABELS_EXPORT == LV_BUS_LABELS_ALL:
                    _src_for_lines = flow_grid_df
                else:
                    # If bt_bus_labels contains only the root and POD buses,
                    # rebuild the complete LV-bus list to include all Kerber lines.
                    flow_grid_all = {
                        b: collect_bus_sequences_from_results(results, b, timeindex_canonico)
                        for b in LV_BUS_LABELS_ALL
                    }

                    flow_grid_all_df = (
                        pd.concat(flow_grid_all, axis=1)
                        if flow_grid_all else pd.DataFrame(index=timeindex_canonico)
                    )

                    flow_grid_all_df = flow_grid_all_df.reindex(timeindex_canonico).fillna(0.0)
                    flow_grid_all_df = set_real_index(flow_grid_all_df, idx_real)

                    _src_for_lines = flow_grid_all_df

                _line_kw = ("line_forw", "line_back", "bt_link_forw", "bt_link_back")

                line_cols = [
                    c for c in _src_for_lines.columns
                    if any(k in str(c) for k in _line_kw)
                ]

                flow_grid_lines_only_df = (
                    _src_for_lines.loc[:, line_cols].copy()
                    if len(line_cols) > 0
                    else _src_for_lines.iloc[:, 0:0].copy()
                )

                try:
                    flow_grid_lines_only_df = convert_busseq_to_flowtuple_df(flow_grid_lines_only_df)
                except Exception as e:
                    print(f"[WARN] convert_busseq_to_flowtuple_df failed for flow_grid_lines_only_df: {e}")

                _tmp = flow_grid_lines_only_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
                _max_abs = float(_tmp.abs().to_numpy().max()) if _tmp.size else 0.0
                _n_nz = int((_tmp.abs() > 1e-9).sum().sum()) if _tmp.size else 0

                print(
                    f"[CHECK] flow_grid_lines_only_df: "
                    f"max_abs={_max_abs} | nonzero_cells={_n_nz} | shape={flow_grid_lines_only_df.shape}"
                )



                # ------------------------------------------------------------
                # Sec 4.11 -- 1.8d) Export technical post-processing of the run
                # ------------------------------------------------------------
                postproc_outputs = export_postproc_xlsx_for_season(
                    flow_el_df=flow_el_df,
                    flow_grid_root=flow_grid_root,
                    season=current_season,
                    out_dir=optimization_output_folder,
                    mode=MODE,
                    mode_label=MODE_LABEL,
                )




                # ------------------------------------------------------------
                # Sec 4.11 -- 1.8e) Save mixed OPT/RB SOC
                # ------------------------------------------------------------

                # ---- BESS SOC ------------------------------------------------
                if len(has_BESS_run) > 0:

                    if _tech_is_opt(CONTROL_MODE, "BESS"):

                        soc_kwh = pd.DataFrame(
                            index=idx_real
                        )

                        soc_pct = pd.DataFrame(
                            index=idx_real
                        )

                        boundary_rows = []


                        for c in sorted(has_BESS_run):

                            storage_label = (
                                f"{c}_BESS_Opt"
                            )

                            storage_node = noded[
                                storage_label
                            ]

                            cap_kwh = float(
                                BESS_capacity_kwh
                            )


                            (
                                soc_c_kwh,
                                soc_c_pct,
                                boundary_info,
                            ) = extract_opt_storage_soc(
                                results=results,
                                storage_node=storage_node,
                                capacity_kwh=cap_kwh,
                                idx_real=idx_real,
                                storage_label=storage_label,

                                # BESS has a FIXED initial level
                                initial_storage_level=initial_SoC_BESS,
                            )


                            # Keep the old Excel column naming convention
                            soc_kwh[
                                f"{c}_SOC_kWh"
                            ] = soc_c_kwh.values

                            soc_pct[
                                f"{c}_SOC_pct"
                            ] = soc_c_pct.values


                            boundary_info[
                                "house"
                            ] = c

                            boundary_rows.append(
                                boundary_info
                            )


                        boundary_df = pd.DataFrame(
                            boundary_rows
                        )


                        save_soc_excel(
                            soc_kwh=soc_kwh,
                            soc_pct=soc_pct,
                            output_path=soc_output_path(
                                "BESS",
                                current_season,
                            ),
                            meta={
                                "mode": MODE,
                                "tech": "BESS",
                                "control": CONTROL_MODE["BESS"],
                                "season": current_season,
                                "soc_source": "oemof_storage_content",
                            },
                            boundary_df=boundary_df,
                        )



                    elif _tech_is_rb(CONTROL_MODE, "BESS"):
                        soc_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "bess_soc_kwh",
                            idx_real,
                            columns=has_BESS_run,
                        )
                        soc_pct = (soc_kwh / float(BESS_capacity_kwh) * 100.0) if not soc_kwh.empty else pd.DataFrame(index=idx_real)

                        save_soc_excel(
                            soc_kwh=soc_kwh,
                            soc_pct=soc_pct,
                            output_path=soc_output_path("BESS", current_season),
                            meta={
                                "mode": MODE,
                                "tech": "BESS",
                                "control": CONTROL_MODE["BESS"],
                                "season": current_season,
                            },
                        )


                # ---- DHW SOC -------------------------------------------------
                if len(has_DHW_run) > 0:

                    if _tech_is_opt(CONTROL_MODE, "DHW"):

                        soc_kwh = pd.DataFrame(
                            index=idx_real
                        )

                        soc_pct = pd.DataFrame(
                            index=idx_real
                        )

                        boundary_rows = []


                        for c in sorted(has_DHW_run):

                            storage_label = (
                                f"{c}_DHW_storage"
                            )

                            storage_node = noded[
                                storage_label
                            ]

                            cap_kwh = float(
                                capacity_dhw_by_house_eff_run[c]
                            )


                            (
                                soc_c_kwh,
                                soc_c_pct,
                                boundary_info,
                            ) = extract_opt_storage_soc(
                                results=results,
                                storage_node=storage_node,
                                capacity_kwh=cap_kwh,
                                idx_real=idx_real,
                                storage_label=storage_label,

                                # IMPORTANT:
                                # DHW initial storage is optimized by oemof
                                initial_storage_level=None,
                            )


                            soc_kwh[
                                f"{c}_SOC_DHW_kWh"
                            ] = soc_c_kwh.values

                            soc_pct[
                                f"{c}_SOC_DHW_pct"
                            ] = soc_c_pct.values


                            boundary_info[
                                "house"
                            ] = c

                            boundary_rows.append(
                                boundary_info
                            )


                        boundary_df = pd.DataFrame(
                            boundary_rows
                        )


                        save_soc_excel(
                            soc_kwh=soc_kwh,
                            soc_pct=soc_pct,
                            output_path=soc_output_path(
                                "DHW",
                                current_season,
                            ),
                            meta={
                                "mode": MODE,
                                "tech": "DHW",
                                "control": CONTROL_MODE["DHW"],
                                "season": current_season,
                                "soc_source": "oemof_storage_content",
                            },
                            boundary_df=boundary_df,
                        )


                    elif _tech_is_rb(CONTROL_MODE, "DHW"):

                        # --------------------------------------------------------
                        # Standard post-charge SOC
                        # --------------------------------------------------------

                        soc_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_soc_kwh",
                            idx_real,
                            columns=has_DHW_run,
                        )

                        soc_pct = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_soc_pct",
                            idx_real,
                            columns=has_DHW_run,
                        )


                        # --------------------------------------------------------
                        # RB intermediate storage states
                        #
                        # Timeline within timestep:
                        #
                        # SOC_prev
                        #   -> losses
                        # SOC_after_loss
                        #   -> DHW draw
                        # SOC_raw
                        #   -> clipping at zero
                        # SOC_after_draw
                        #   -> resistance charging
                        # SOC_kWh (= post-charge state)
                        #
                        # These are diagnostics only.
                        # The RB control is NOT modified.
                        # --------------------------------------------------------

                        rb_soc_prev_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_soc_prev_kwh",
                            idx_real,
                            columns=has_DHW_run,
                        )

                        rb_soc_after_loss_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_soc_after_loss_kwh",
                            idx_real,
                            columns=has_DHW_run,
                        )

                        rb_soc_raw_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_soc_raw_kwh",
                            idx_real,
                            columns=has_DHW_run,
                        )

                        # Physical storage content immediately after DHW draw.
                        # SOC_raw may be negative if demand exceeds available
                        # energy; physical tank SOC cannot be negative.
                        rb_soc_after_draw_kwh = (
                            rb_soc_raw_kwh.clip(lower=0.0)
                        )

                        rb_soc_after_draw_pct = pd.DataFrame(
                            index=rb_soc_after_draw_kwh.index
                        )

                        for c in rb_soc_after_draw_kwh.columns:

                            cap_c = float(
                                capacity_dhw_by_house_eff_run.get(
                                    c,
                                    capacity_dhw_by_house.get(
                                        c,
                                        cap_kWh_DHW,
                                    ),
                                )
                            )

                            rb_soc_after_draw_pct[c] = (
                                100.0
                                * rb_soc_after_draw_kwh[c]
                                / cap_c
                                if cap_c > 0.0
                                else np.nan
                            )



                        # --------------------------------------------------------
                        # Actual DHW draw during timestep [kWh_th]
                        # --------------------------------------------------------

                        rb_dhw_draw_kw = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_demand_kw",
                            idx_real,
                            columns=has_DHW_run,
                        )

                        rb_dhw_draw_kwh = (
                            rb_dhw_draw_kw
                            * float(timestep_minutes)
                            / 60.0
                        )


                        # --------------------------------------------------------
                        # Boiler charging power
                        # --------------------------------------------------------

                        rb_dhw_charge_kw = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_charge_kw",
                            idx_real,
                            columns=has_DHW_run,
                        )


                        # --------------------------------------------------------
                        # Backup / shortage
                        # --------------------------------------------------------

                        rb_dhw_boost_th_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_boost_th_kwh",
                            idx_real,
                            columns=has_DHW_run,
                        )

                        rb_dhw_shortage_th_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "dhw_shortage_th_kwh",
                            idx_real,
                            columns=has_DHW_run,
                        )


                        # --------------------------------------------------------
                        # Export
                        # --------------------------------------------------------

                        save_soc_excel(
                            soc_kwh=soc_kwh,
                            soc_pct=soc_pct,

                            output_path=soc_output_path(
                                "DHW",
                                current_season,
                            ),

                            meta={
                                "mode": MODE,
                                "tech": "DHW",
                                "control": CONTROL_MODE["DHW"],
                                "season": current_season,

                                "rb_diagnostic_states_exported": True,

                                "rb_soc_timing": (
                                    "post-charge SOC in SOC_kWh/SOC_pct; "
                                    "intermediate states in dedicated sheets"
                                ),
                            },

                            extra_sheets={
                                "SOC_prev_kWh":
                                    rb_soc_prev_kwh,

                                "SOC_after_loss_kWh":
                                    rb_soc_after_loss_kwh,

                                "SOC_raw_kWh":
                                    rb_soc_raw_kwh,

                                "SOC_after_draw_kWh":
                                    rb_soc_after_draw_kwh,

                                "SOC_after_draw_pct":
                                    rb_soc_after_draw_pct,                              

                                "DHW_draw_kWh":
                                    rb_dhw_draw_kwh,

                                "DHW_charge_kW":
                                    rb_dhw_charge_kw,

                                "DHW_boost_th_kWh":
                                    rb_dhw_boost_th_kwh,

                                "DHW_shortage_th_kWh":
                                    rb_dhw_shortage_th_kwh,
                            },
                        )





                # ---- BEV SOC -------------------------------------------------
                if len(has_BEV_run) > 0:

                    if _tech_is_opt(CONTROL_MODE, "DHW"):

                        soc_kwh = pd.DataFrame(
                            index=idx_real
                        )

                        soc_pct = pd.DataFrame(
                            index=idx_real
                        )

                        boundary_rows = []


                        for c in sorted(has_DHW_run):

                            storage_label = (
                                f"{c}_DHW_storage"
                            )

                            storage_node = noded[
                                storage_label
                            ]

                            cap_kwh = float(
                                capacity_dhw_by_house_eff_run[c]
                            )


                            (
                                soc_c_kwh,
                                soc_c_pct,
                                boundary_info,
                            ) = extract_opt_storage_soc(
                                results=results,
                                storage_node=storage_node,
                                capacity_kwh=cap_kwh,
                                idx_real=idx_real,
                                storage_label=storage_label,

                                # IMPORTANT:
                                # DHW initial storage is optimized by oemof
                                initial_storage_level=None,
                            )


                            soc_kwh[
                                f"{c}_SOC_DHW_kWh"
                            ] = soc_c_kwh.values

                            soc_pct[
                                f"{c}_SOC_DHW_pct"
                            ] = soc_c_pct.values


                            boundary_info[
                                "house"
                            ] = c

                            boundary_rows.append(
                                boundary_info
                            )


                        boundary_df = pd.DataFrame(
                            boundary_rows
                        )


                        save_soc_excel(
                            soc_kwh=soc_kwh,
                            soc_pct=soc_pct,
                            output_path=soc_output_path(
                                "DHW",
                                current_season,
                            ),
                            meta={
                                "mode": MODE,
                                "tech": "DHW",
                                "control": CONTROL_MODE["DHW"],
                                "season": current_season,
                                "soc_source": "oemof_storage_content",
                            },
                            boundary_df=boundary_df,
                        )





                    elif _tech_is_rb(CONTROL_MODE, "BEV"):
                        soc_kwh = rb_soc_df_from_profile(
                            rb_prof,
                            "bev_soc_kwh",
                            idx_real,
                            columns=has_BEV_run,
                        )
                        soc_pct = rb_soc_df_from_profile(
                            rb_prof,
                            "bev_soc_pct",
                            idx_real,
                            columns=has_BEV_run,
                        )

                        save_soc_excel(
                            soc_kwh=soc_kwh,
                            soc_pct=soc_pct,
                            output_path=soc_output_path("BEV", current_season),
                            meta={
                                "mode": MODE,
                                "tech": "BEV",
                                "control": CONTROL_MODE["BEV"],
                                "season": current_season,
                            },
                        )


                # ------------------------------------------------------------
                # Sec 4.11 -- 1.8f) Technical export of LV lines
                # ------------------------------------------------------------
                kerber_line_recipes = build_kerber_line_recipes_for_current_limits(PF_LINE_LIMITS_MAP)

                net_line_outputs = build_net_lines_outputs(
                    lv_grid_line_results_df=flow_grid_lines_only_df,
                    kerber_line_recipes=kerber_line_recipes,
                )

                if net_line_outputs:
                    out_xlsx_lines = os.path.join(
                        optimization_output_folder,
                        f"net_lines_{current_season}.xlsx"
                    )

                    sheets_lines = {
                        "net_lines_ts_signed": net_line_outputs["net_lines_ts_signed"],
                        "net_lines_ts_for_dur": net_line_outputs["net_lines_ts_for_dur"],
                        "durcurve_net_lines": net_line_outputs["durcurve_net_lines"],
                        "lines_meta": net_line_outputs["lines_meta"],
                        "sat_lines_ts_pct": net_line_outputs["sat_lines_ts_pct"],
                        "durcurve_sat_lines_pct": net_line_outputs["durcurve_sat_lines_pct"],
                    }

                    write_many_sheets(out_xlsx_lines, sheets_lines)

                    print(
                        f"[INFO][NET LINES] Saved {out_xlsx_lines} "
                        f"| missing_forw={net_line_outputs['missing_forw']} "
                        f"| missing_back={net_line_outputs['missing_back']}"
                    )











                #----------------------------
                # Sec 4.11 -- 1.8g) Calculate cost dataframe for individual households in the different configurations
                #----------------------------
                total_costs_per_household_df = (
                    build_total_costs_per_household_from_el_flows(
                        flow_el_df=flow_el_df,
                        df_objective_terms_by_house=df_objective_terms_by_house,
                        house_ids=ID,
                        ee_cost_real=ee_cost_series[current_season],
                        ee_rev_real=ee_rev_series[current_season],
                        timestep_minutes=timestep_minutes,
                        values_are_power_kw=True,
                        include_dhw_boost_in_effective_costs=(
                            INCLUDE_DHW_BOOST_IN_EFFECTIVE_COSTS
                        ),
                    )
                )


                # ------------------------------------------------------------
                # Sec 4.11 -- 1.8h) Household economic-allocation consistency check
                # ------------------------------------------------------------

                ECON_HH_ALLOC_TOL_EUR = 1e-8

                _household_system_cost_columns = [
                    "dhw_boost_cost_eur",
                    "dhw_boost_cost_effective_eur",
                    "dhw_boost_cost_excluded_eur",
                    "bev_outdoor_charge_cost_eur",
                    "pv_curtail_cost_eur",
                ]

                _household_allocation_failures = []

                for _cost_col in _household_system_cost_columns:

                    if (
                        _cost_col
                        not in total_costs_per_household_df.columns
                    ):
                        raise KeyError(
                            "Missing household economic column: "
                            f"{_cost_col}"
                        )

                    if (
                        _cost_col
                        not in df_system_cost_ts.columns
                    ):
                        raise KeyError(
                            "Missing system economic column: "
                            f"{_cost_col}"
                        )

                    _hh_total_eur = float(
                        pd.to_numeric(
                            total_costs_per_household_df[_cost_col],
                            errors="coerce",
                        )
                        .fillna(0.0)
                        .sum()
                    )

                    _system_total_eur = float(
                        pd.to_numeric(
                            df_system_cost_ts[_cost_col],
                            errors="coerce",
                        )
                        .fillna(0.0)
                        .sum()
                    )

                    _residual_eur = (
                        _hh_total_eur
                        - _system_total_eur
                    )

                    if (
                        abs(_residual_eur)
                        > ECON_HH_ALLOC_TOL_EUR
                    ):
                        _household_allocation_failures.append(
                            {
                                "cost_component": _cost_col,
                                "household_total_eur":
                                    _hh_total_eur,
                                "system_total_eur":
                                    _system_total_eur,
                                "residual_eur":
                                    _residual_eur,
                            }
                        )


                _hh_identity_residual = (
                    total_costs_per_household_df[
                        "simulation_total_eur"
                    ]
                    - total_costs_per_household_df[
                        "effective_total_eur"
                    ]
                    - total_costs_per_household_df[
                        "dhw_boost_cost_excluded_eur"
                    ]
                )

                _max_hh_identity_residual_eur = float(
                    _hh_identity_residual.abs().max()
                )

                if (
                    _max_hh_identity_residual_eur
                    > ECON_HH_ALLOC_TOL_EUR
                ):
                    _household_allocation_failures.append(
                        {
                            "cost_component":
                                "household_effective_identity",
                            "household_total_eur": np.nan,
                            "system_total_eur": np.nan,
                            "residual_eur":
                                _max_hh_identity_residual_eur,
                        }
                    )


                if _household_allocation_failures:

                    _df_household_allocation_failures = (
                        pd.DataFrame(
                            _household_allocation_failures
                        )
                    )

                    print(
                        "\n[ERROR] Household economic allocation "
                        "does not reconcile with system costs:"
                    )

                    display(
                        _df_household_allocation_failures
                    )

                    raise AssertionError(
                        "Household economic-cost allocation failed. "
                        "Check objective_terms_by_flow / "
                        "objective_terms_by_house."
                    )


                print(
                    "[PASS] Household economic allocation reconciles "
                    "with system costs."
                )


            #----------------------------
            # Sec 4.11 -- 1.9 Save results, in Xlsx and Npy files
            #----------------------------
            with timed(f"{MODE}/{current_season} - export", timings):
                print('-'*40)
                print(f"\n=== Optimization by season {current_season} completed: saving === MODE: {MODE}")
                print('-'*40)
                
                # Name of the file to save
                file_name = f"ris_{current_season}.xlsx"
                output_file = os.path.join(optimization_output_folder, file_name)

                os.makedirs(os.path.dirname(output_file), exist_ok=True)
                
                with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                    # 1.9.1) Result sheets
                    flow_el_df.to_excel(writer, sheet_name='Flow Electric', index=True, startrow=1)
                    flow_PV_curtail_df.to_excel(writer, sheet_name='Flow PV curtail', index=True)
                    flow_grid_df.to_excel(writer, sheet_name='Flow Grid', index=True)
                    # flow_heat_df.to_excel(writer, sheet_name='Flow Heat', index=True)                     # Currently not included
                    flow_DHW_use_df.to_excel(writer, sheet_name='Flow DHW_use', index=True)
                    flow_charge_el_BEV_df.to_excel(writer, sheet_name='Flow BEV charge', index=True)
                    flow_discharge_el_BEV_df.to_excel(writer, sheet_name='Flow BEV discharge', index=True)

                    dhw_input_profile_df.to_excel(writer, sheet_name='DHW input demand', index=True)
                    bev_input_profile_df.to_excel(writer, sheet_name='BEV input demand', index=True)

                    # Cost sheets
                    df_system_cost_ts.to_excel(writer, sheet_name='system_cost_ts', index=True)
                    df_system_cost_summary.to_excel(writer, sheet_name='system_cost_summary', index=False)
                    df_objective_terms_ts.to_excel(writer, sheet_name='objective_terms_ts', index=True)
                    df_objective_terms_summary.to_excel(writer, sheet_name='objective_terms_summary', index=False)
                    total_costs_per_household_df.to_excel(writer, sheet_name='total_costs_per_household', index=False)

                    if SAVE_OBJECTIVE_TERMS_BY_FLOW_XLSX:
                        df_objective_terms_by_flow.to_excel(writer, sheet_name='objective_terms_by_flow', index=False)

                    if SAVE_OBJECTIVE_TERMS_BY_HOUSE_XLSX and not df_objective_terms_by_house.empty:
                        df_objective_terms_by_house.to_excel(writer, sheet_name='objective_terms_by_house', index=False)
                    df_objective_reconciliation.to_excel(writer, sheet_name='objective_reconciliation', index=False)

                    df_flex_categories_run.to_excel(writer, sheet_name="active_flex_categories", index=False)



                    # Add information sheet
                    info_dict = {
                        'Parametro': [
                            'Smart or rule-based case',
                            'Grid archetype used',
                            'Number of houses',
                            'Configuration output tag',
                            'Controllo BESS',
                            'Controllo BEV',
                            'Controllo DHW',
                            'Share HP [%]', 
                            'Air-conditioner share [%]',
                            'PV share [%]',
                            'Season',
                            'Geographic area',
                            'Induction share [%]',
                            'Flexibility share with BESS, across all households [%]',
                            'Flexibility share with electric boilers (DHW) [%]',
                            'Flexibility share with BEV [%]',
                            'BEVs also with V2G [%]',

                        ],
                        'Valore': [
                            MODE,
                            LV_ARCHETYPE,
                            num_households, 
                            RUN_DIRS["tag"],
                            CONTROL_MODE["BESS"],
                            CONTROL_MODE["BEV"],
                            CONTROL_MODE["DHW"],
                            RUN_SHARES["HP"],
                            RUN_SHARES["COOL"],  
                            RUN_SHARES['PV'], 
                            current_season,
                            selected_region, 
                            RUN_SHARES["INDUCTION"],
                            RUN_SHARES["BESS"],
                            RUN_SHARES["DHW"],
                            RUN_SHARES["BEV"],
                            RUN_SHARES["V2G"],
                        ]
                    }
                    df_info = pd.DataFrame(info_dict)
                    df_info.to_excel(writer, sheet_name='Configuration information', index=False)
                
                
                # Save compressed dataframes in .gpkl format
                save_compressed_pickle(flow_el_df, flow_pickle_path("flow_el_df", current_season))
                save_compressed_pickle(flow_grid_df, flow_pickle_path("flow_grid_df", current_season))
                # save_compressed_pickle(flow_heat_df,  flow_pickle_path("flow_heat_df", season_corrente))
                save_compressed_pickle(flow_grid_root, flow_pickle_path("flow_grid_root", current_season))

                # PV curtailment flows
                save_compressed_pickle(flow_PV_curtail_df, flow_pickle_path("flow_PV_curtail_df", current_season,),)

                save_compressed_pickle(flow_DHW_use_df, flow_pickle_path("flow_DHW_use_df", current_season))
                save_compressed_pickle(flow_charge_el_BEV_df, flow_pickle_path("flow_charge_el_BEV_df", current_season))
                save_compressed_pickle(flow_discharge_el_BEV_df, flow_pickle_path("flow_discharge_el_BEV_df", current_season))
                save_compressed_pickle(flow_grid_lines_only_df, flow_pickle_path("flow_grid_lines_only_df", current_season))
                                
                

                # 1.9.2) Save output results as input for Power-Flow analysis
                idx_real = IDX_REAL_BY_SEASON[current_season]
                
                p_net, meta_pf = export_pf_inputs_house_lv(
                    results=results,
                    noded=noded,
                    LV_ARCHETYPE=LV_ARCHETYPE,
                    num_households=num_households,
                    KERBER_BLUEPRINT_FILES=KERBER_BLUEPRINT_FILES,
                    HOUSE_TO_LV_BUS_LABEL=HOUSE_TO_LV_BUS_LABEL,
                    LV_ROOT_LABEL=LV_ROOT_LABEL,
                    _lv_bus_for_house=_lv_bus_for_house,
                    out_dir=pf_py_based_output_folder,
                    season_tag=current_season,
                    mode_tag=MODE,
                    timeindex_canonico=timeindex_canonico,
                    to_mw_factor=1 / 1000,
                    idx_real=idx_real,
                    export_component_profiles=EXPORT_PF_COMPONENT_PROFILES,
                    export_assumed_q=EXPORT_ASSUMED_Q_FOR_PF,
                    cosphi_by_component=PF_EXPORT_COSPHI_BY_COMPONENT,
                    balance_tolerance_kw=PF_EXPORT_BALANCE_TOLERANCE_KW,
                    eta_grid=eta_grid,
                    validate_reconstruction=VALIDATE_PF_EXPORT_RECONSTRUCTION,
                    validation_strict=PF_EXPORT_VALIDATION_STRICT,
                    save_validation_details=SAVE_PF_EXPORT_VALIDATION_DETAILS,
                    run_context=RUN_CFG,
                )


                if current_season not in completed_seasons_current_run:
                    completed_seasons_current_run.append(current_season)

                update_hagrid_run_manifest_status(
                    RUN_UID,
                    status="partial",
                    completed_seasons=completed_seasons_current_run,
                )


                # Saving economic information
                if len(system_cost_rows_mode) > 0:
                    df_mode_costs = pd.DataFrame(system_cost_rows_mode)



                    total_row = pd.DataFrame([{
                        "mode": MODE,
                        "season": "ALL",
                        "bt_archetype": LV_ARCHETYPE,
                        "region": selected_region,
                        "n_houses": num_households,

                        "timesteps": _sum_if_present(df_mode_costs, "timesteps"),
                        "dt_minutes": _first_if_present(df_mode_costs, "dt_minutes"),
                        "use_pf_limits": _first_if_present(df_mode_costs, "use_pf_limits"),
                        "mt_import_cap_kw": _max_if_present(df_mode_costs, "mt_import_cap_kw"),
                        "mt_export_cap_kw": _max_if_present(df_mode_costs, "mt_export_cap_kw"),

                        # Scambi LV
                        "bt_import_total_kwh": _sum_if_present(df_mode_costs, "bt_import_total_kwh"),
                        "bt_export_total_kwh": _sum_if_present(df_mode_costs, "bt_export_total_kwh"),
                        "bt_purchase_cost_eur": _sum_if_present(df_mode_costs, "bt_purchase_cost_eur"),
                        "bt_sale_revenues_eur": _sum_if_present(df_mode_costs, "bt_sale_revenues_eur"),
                        "bt_energy_net_settlement_eur": _sum_if_present(df_mode_costs, "bt_energy_net_settlement_eur"),

                        # DHW boost
                        "dhw_boost_total_kwh": _sum_if_present(df_mode_costs, "dhw_boost_total_kwh"),
                        "dhw_boost_active_timesteps": _sum_if_present(df_mode_costs, "dhw_boost_active_timesteps"),
                        "dhw_boost_cost_eur": _sum_if_present(df_mode_costs, "dhw_boost_cost_eur"),
                        "n_houses_dhw_boost_active": _sum_if_present(df_mode_costs, "n_houses_dhw_boost_active"),
                        "n_edges_dhw_boost_matched": _sum_if_present(df_mode_costs, "n_edges_dhw_boost_matched"),

                        "include_dhw_boost_in_effective_costs": (_first_if_present(df_mode_costs,"include_dhw_boost_in_effective_costs",)),
                        "dhw_boost_cost_effective_eur": (_sum_if_present(df_mode_costs,"dhw_boost_cost_effective_eur",)),
                        "dhw_boost_cost_excluded_eur": (_sum_if_present(df_mode_costs, "dhw_boost_cost_excluded_eur",)),

                        # BEV outdoor
                        "bev_outdoor_total_kwh": _sum_if_present(df_mode_costs, "bev_outdoor_total_kwh"),
                        "bev_outdoor_active_timesteps": _sum_if_present(df_mode_costs, "bev_outdoor_active_timesteps"),
                        "bev_outdoor_charge_cost_eur": _sum_if_present(df_mode_costs, "bev_outdoor_charge_cost_eur"),
                        "n_houses_bev_outdoor_active": _sum_if_present(df_mode_costs, "n_houses_bev_outdoor_active"),
                        "n_edges_bev_outdoor_matched": _sum_if_present(df_mode_costs, "n_edges_bev_outdoor_matched"),

                        # PV curtailment
                        "pv_curtail_total_kwh": _sum_if_present(df_mode_costs, "pv_curtail_total_kwh"),
                        "pv_curtail_active_timesteps": _sum_if_present(df_mode_costs, "pv_curtail_active_timesteps"),
                        "pv_curtail_cost_eur": _sum_if_present(df_mode_costs, "pv_curtail_cost_eur"),
                        "n_houses_pv_curtail_active": _sum_if_present(df_mode_costs, "n_houses_pv_curtail_active"),
                        "n_edges_pv_curtail_matched": _sum_if_present(df_mode_costs, "n_edges_pv_curtail_matched"),

                        # Tecnic
                        # Total system cost business
                        "total_system_cost_business_eur": _sum_if_present(df_mode_costs, "total_system_cost_business_eur"),

                        "total_system_cost_effective_eur": (_sum_if_present(df_mode_costs,"total_system_cost_effective_eur",)),
                        "total_system_cost_simulation_eur": (_sum_if_present(df_mode_costs,"total_system_cost_simulation_eur",)),

                        "total_local_demand_kwh": _sum_if_present(df_mode_costs, "total_local_demand_kwh"),
                        "mt_import_total_kwh": _sum_if_present(df_mode_costs, "mt_import_total_kwh"),
                        "local_self_consumption_kwh": _sum_if_present(df_mode_costs, "local_self_consumption_kwh"),

                        # Final electric-demand breakdown by component.
                        "demand_base_kwh": _sum_if_present(                        df_mode_costs, "demand_base_kwh"                    ),
                        "demand_cooling_kwh": _sum_if_present(                        df_mode_costs, "demand_cooling_kwh"                    ),
                        "demand_heating_kwh": _sum_if_present(df_mode_costs, "demand_heating_kwh"),
                        "demand_cooking_kwh": _sum_if_present(df_mode_costs, "demand_cooking_kwh"),
                        "demand_ev_charging_kwh": _sum_if_present(df_mode_costs, "demand_ev_charging_kwh"),
                        "demand_dhw_storage_kwh": _sum_if_present(df_mode_costs, "demand_dhw_storage_kwh"),
                        "demand_dhw_total_kwh": _sum_if_present(df_mode_costs, "demand_dhw_total_kwh"),
                        "demand_components_total_kwh": _sum_if_present(df_mode_costs, "demand_components_total_kwh"),
                        "demand_components_gap_kwh": _sum_if_present(df_mode_costs, "demand_components_gap_kwh"),

                        "n_edges_demand_base_matched": _max_if_present(df_mode_costs, "n_edges_demand_base_matched"),
                        "n_edges_demand_cooling_matched": _max_if_present(df_mode_costs, "n_edges_demand_cooling_matched"),
                        "n_edges_demand_heating_matched": _max_if_present(df_mode_costs, "n_edges_demand_heating_matched"),
                        "n_edges_demand_cooking_matched": _max_if_present(df_mode_costs, "n_edges_demand_cooking_matched"),
                        "n_edges_demand_ev_charging_matched": _max_if_present(df_mode_costs, "n_edges_demand_ev_charging_matched"),
                        "n_edges_demand_dhw_storage_matched": _max_if_present(df_mode_costs, "n_edges_demand_dhw_storage_matched"),

                        "mt_import_share_of_demand": (
                            _sum_if_present(df_mode_costs, "mt_import_total_kwh")
                            / _sum_if_present(df_mode_costs, "total_local_demand_kwh")
                            if _sum_if_present(df_mode_costs, "total_local_demand_kwh") > 1e-12
                            else np.nan
                        ),
                        "local_self_consumption_share_of_demand": (
                            _sum_if_present(df_mode_costs, "local_self_consumption_kwh")
                            / _sum_if_present(df_mode_costs, "total_local_demand_kwh")
                            if _sum_if_present(df_mode_costs, "total_local_demand_kwh") > 1e-12
                            else np.nan
                        ),
                        "n_edges_total_local_demand_matched": _max_if_present(
                            df_mode_costs, "n_edges_total_local_demand_matched"
                        ),
                        "n_edges_mt_import_matched": _max_if_present(
                            df_mode_costs, "n_edges_mt_import_matched"
                        ),


                        # Kerber / extra objective terms
                        "kerber_line_transfer_total_kwh": _sum_if_present(df_mode_costs, "kerber_line_transfer_total_kwh"),
                        "kerber_line_transfer_cost_eur": _sum_if_present(df_mode_costs, "kerber_line_transfer_cost_eur"),
                        "n_edges_kerber_line_matched": _max_if_present(df_mode_costs, "n_edges_kerber_line_matched"),
                        "other_variable_cost_terms_eur": _sum_if_present(df_mode_costs, "other_variable_cost_terms_eur"),
                        "shortage_cost_eur": _sum_if_present(df_mode_costs, "shortage_cost_eur"),
                        "objective_extra_variable_cost_terms_eur": _sum_if_present(df_mode_costs, "objective_extra_variable_cost_terms_eur"),

                        # Reconstructed v. objective
                        "total_system_cost_including_objective_variable_terms_eur": _sum_if_present(df_mode_costs, "total_system_cost_including_objective_variable_terms_eur"),
                        "objective_variable_cost_terms_eur": _sum_if_present(df_mode_costs, "objective_variable_cost_terms_eur"),
                        "terms_not_captured_by_variable_cost_breakdown_eur": _sum_if_present(df_mode_costs, "terms_not_captured_by_variable_cost_breakdown_eur"),
                        "objective_gap_vs_variable_cost_breakdown_eur": _sum_if_present(df_mode_costs, "objective_gap_vs_variable_cost_breakdown_eur"),
                        "objective_gap_vs_reconstructed_total_eur": _sum_if_present(df_mode_costs, "objective_gap_vs_reconstructed_total_eur"),
                        "objective_match_ok": bool(df_mode_costs["objective_match_ok"].fillna(False).all()) if "objective_match_ok" in df_mode_costs.columns else False,
                        "total_system_cost_including_all_objective_terms_eur": _sum_if_present(df_mode_costs, "total_system_cost_including_all_objective_terms_eur"),
                        "objective_total_eur": _sum_if_present(df_mode_costs, "objective_total_eur"),

                    }])


                    # ------------------------------------------------
                    # Information for the experiment-level
                    # economic summary.
                    # ------------------------------------------------

                    experiment_identity = {
                        "experiment_id": RUN_CFG["experiment_id"],
                        "run_revision": RUN_CFG["run_revision"],
                        "run_uid": RUN_UID,
                        "run_label": RUN_LABEL,
                        "electrification_id": RUN_CFG["electrification_id"],
                        "case_label": RUN_CFG.get("case_label"),
                        "mode": MODE,

                        "share_PV_target": RUN_SHARES["PV"],
                        "share_BESS_target": RUN_SHARES["BESS"],
                        "share_BEV_target": RUN_SHARES["BEV"],
                        "share_V2G_target_of_BEV": RUN_SHARES["V2G"],
                        "share_DHW_target": RUN_SHARES["DHW"],
                        "share_HP_target": RUN_SHARES["HP"],
                        "share_COOL_target": RUN_SHARES["COOL"],
                        "share_INDUCTION_target": RUN_SHARES["INDUCTION"],
                    }

                    scenario_meta = RUN_CFG.get("scenario_meta", {}) or {}

                    experiment_identity["share_BESS_target_of_PV"] = (
                        scenario_meta.get(
                            "BESS_share_of_PV_target",
                            np.nan,
                        )
                    )


                    df_mode_costs_exp = df_mode_costs.copy()

                    for col, value in experiment_identity.items():
                        df_mode_costs_exp[col] = value

                    experiment_cost_rows_by_season.extend(
                        df_mode_costs_exp.to_dict(
                            orient="records"
                        )
                    )


                    total_row_exp = total_row.copy()

                    for col, value in experiment_identity.items():
                        total_row_exp[col] = value

                    experiment_cost_rows_total.extend(
                        total_row_exp.to_dict(
                            orient="records"
                        )
                    )






                    out_mode_cost = os.path.join(
                        optimization_output_folder,
                        f"system_cost_summary_{MODE}_{num_households}_nuc_{selected_region}.xlsx"
                    )

                    with pd.ExcelWriter(out_mode_cost, engine="openpyxl") as writer:
                        df_mode_costs.to_excel(writer, sheet_name="by_season", index=False)
                        total_row.to_excel(writer, sheet_name="total", index=False)

                    print(f"[INFO] Saved system cost summary: {out_mode_cost}")


            # ------------------ memory cleanup -------------------------
            del om, es, noded #, results
            gc.collect()
            # ------------------------------------------------------------



        print("\n=== TIMINGS SUMMARY (sorted) ===")
        for k, v in sorted(timings.items(), key=lambda kv: kv[1], reverse=True):
            print(f"{v:10.2f} s  {k}")

        total_run_seconds = perf_counter() - t0_run

        save_timings_txt(
            timings=timings,                    # the dictionary used by with timed(...)
            total_run_seconds=total_run_seconds,
            out_dir=optimization_output_folder,    # same directory as the Excel files
            filename="timings_summary.txt"
        )


        # ============================================================
        # End of the seasonal loop for the current RUN_CFG
        # ============================================================
        expected_seasons = list(seasons_dict.values())

        run_completed = (
            set(completed_seasons_current_run)
            == set(expected_seasons)
        )

        update_hagrid_run_manifest_status(
            RUN_UID,
            status="completed" if run_completed else "partial",
            completed_seasons=completed_seasons_current_run,
        )

        # # Condition to exit the loop if the result is infeasible
        # if ABORT_ALL_RUNS:
        #     break




    # ========================================================
    # Sec 4.11 -- FINAL SOLVE STATUS SUMMARY
    # ========================================================
    if SOLVE_STATUS_REGISTRY:

        # 1) Detailed table: one row per run x season
        df_solve_status = pd.DataFrame(SOLVE_STATUS_REGISTRY.values())

        run_order = {str(cfg["run_uid"]): i for i, cfg in enumerate(RUNS)}
        season_cols = [str(s) for s in seasons_dict.values()]
        season_order = {s: i for i, s in enumerate(season_cols)}

        df_solve_status["_run_order"] = (
            df_solve_status["run_uid"].astype(str).map(run_order)
        )
        df_solve_status["_season_order"] = (
            df_solve_status["season"].astype(str).map(season_order)
        )

        df_solve_status = (
            df_solve_status
            .sort_values(["_run_order", "_season_order"])
            .drop(columns=["_run_order", "_season_order"])
            .reset_index(drop=True)
        )

        # 2) Compact status matrix
        df_solve_status_matrix = (
            df_solve_status
            .pivot(
                index=["case_label", "run_label", "run_uid", "mode"],
                columns="season",
                values="status",
            )
            .reindex(columns=season_cols)
        )

        # 3) Counts
        status = df_solve_status["status"].astype(str)

        n_total = len(status)
        n_ok = int((status == "OK").sum())
        n_fail = int(status.str.startswith("FAIL").sum())
        n_not_run = int((status == "NOT_RUN").sum())

        # 4) Failed solves only
        failed = df_solve_status.loc[
            status.str.startswith("FAIL"),
            [
                "case_label",
                "run_label",
                "run_uid",
                "mode",
                "season",
                "status",
                "termination_condition_initial",
                "termination_condition_final",
                "solver_status",
                "gurobi_status",
                "lp_path",
                "iis_ilp_path",
                "iis_json_path",
            ],
        ].copy()

        # 5) Print
        print("\n" + "=" * 120)
        print("[SEC. 4.11] FINAL SOLVE STATUS SUMMARY")
        print("=" * 120)

        print(df_solve_status_matrix.to_string())

        print(
            f"\n[SOLVE SUMMARY] total={n_total} | "
            f"OK={n_ok} | FAIL={n_fail} | NOT_RUN={n_not_run}"
        )

        if n_fail == 0 and n_not_run == 0:
            print(
                "[OK][SEC. 4.11] "
                "All scheduled configuration/season solves completed successfully."
            )
        else:
            print(
                "[WARN][SEC. 4.11] "
                "Some scheduled optimization problems were not solved successfully."
            )

            if not failed.empty:
                print("\n[FAILED SOLVES - DETAILS]")
                print(failed.to_string(index=False))

        # 6) Save
        solve_status_output_dir = build_experiment_revision_output_dir(
            experiment_id=EXPERIMENT_ID,
            run_revision=RUN_REVISION,
        )

        os.makedirs(solve_status_output_dir, exist_ok=True)

        solve_status_xlsx_path = os.path.join(
            solve_status_output_dir,
            "solve_status_summary.xlsx",
        )

        solve_status_csv_path = os.path.join(
            solve_status_output_dir,
            "solve_status_by_run_season.csv",
        )

        with pd.ExcelWriter(
            solve_status_xlsx_path,
            engine="openpyxl",
        ) as writer:

            df_solve_status.to_excel(
                writer,
                sheet_name="by_run_season",
                index=False,
            )

            df_solve_status_matrix.to_excel(
                writer,
                sheet_name="status_matrix",
            )

            failed.to_excel(
                writer,
                sheet_name="failed_solves",
                index=False,
            )

        df_solve_status.to_csv(
            solve_status_csv_path,
            index=False,
        )

        print("\n[OK][SOLVE STATUS] Saved:")
        print(" ", solve_status_xlsx_path)
        print(" ", solve_status_csv_path)
    
    # ========================================================
    # Sec 4.11 -- 1.x) EXPERIMENT-LEVEL ECONOMIC SUMMARY
    # ========================================================

    if (
        WRITE_EXPERIMENT_ECONOMIC_SUMMARY
        and experiment_cost_rows_by_season
    ):

        experiment_revision_output_dir = (
            build_experiment_revision_output_dir(
                experiment_id=EXPERIMENT_ID,
                run_revision=RUN_REVISION,
            )
        )

        experiment_summary_path = os.path.join(
            experiment_revision_output_dir,
            "economic_summary.xlsx",
        )

        df_exp_by_season = pd.DataFrame(
            experiment_cost_rows_by_season
        )

        df_exp_total = pd.DataFrame(
            experiment_cost_rows_total
        )


        # Corresponding manifest rows.
        manifest_df = load_hagrid_run_manifest()

        if not manifest_df.empty:

            manifest_exp = manifest_df.loc[
                (
                    manifest_df["experiment_id"].astype(str)
                    == str(EXPERIMENT_ID)
                )
                &
                (
                    manifest_df["run_revision"].astype(str)
                    == str(RUN_REVISION)
                )
            ].copy()

        else:
            manifest_exp = pd.DataFrame()


        with pd.ExcelWriter(
            experiment_summary_path,
            engine="openpyxl",
        ) as writer:

            df_exp_by_season.to_excel(
                writer,
                sheet_name="by_run_season",
                index=False,
            )

            df_exp_total.to_excel(
                writer,
                sheet_name="by_run_total",
                index=False,
            )

            if not manifest_exp.empty:
                manifest_exp.to_excel(
                    writer,
                    sheet_name="run_index",
                    index=False,
                )


        print(
            "[INFO] Saved experiment economic summary:\n"
            f"  {experiment_summary_path}"
        )



    #--------------------------------------
    # Sec 4.11 -- 2) SAVE "has" indices assigning technologies/storage to houses
    #--------------------------------------

    has = {
        "PV": has_PV,
        "BESS": has_BESS,
        "BEV": has_BEV,
        "V2G": has_V2G,
        "DHW": has_el_boiler,
        "HP": has_HP,
        "COOL": has_cooler,
        "INDUCTION": has_induction,
    }
    save_compressed_pickle(has, path_has_tech)

    # Print summary of "has" indices
    print_has_summary(has, ID=ID, max_show=50, title="HAS_TECH used by the mixed configurations (RUN_SOLVE_MIXED_SCENARIOS=True)")

    # Also save a CSV
    # after OPT, when `has` is available
    out_csv = has_global_used_path
    rows = []
    for k in ['PV',"BESS","BEV","V2G","DHW","HP","COOL","INDUCTION"]:
        for c in sorted(set(has.get(k, set()))):
            rows.append({"tech": k, "house": c})
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[DEBUG OPT] saved: {out_csv}")

   ## 4.12 Mixed-configuration charts

In [ ]:
# -----------------------------------------------------------------------------
#               Sec 4.12 -- 0) GLOBAL LIMITS, y-limits, to standardize charts across seasons
# -----------------------------------------------------------------------------

# Define the variable used to choose the maximum represented on the chart axes
# for generation-demand charts, both Opt and RB, and for duration curves
ymax_gen_dmd_global = None
ymax_net_lv_global = -np.inf
ymin_net_lv_global =  np.inf
ymax_mt_global = -np.inf
ymin_mt_global =  np.inf

if RUN_PLOT_SCENARIO_GRAPHS:

    # -----------
    #   Sec 4.12 -- Loop to print/plot charts for each analyzed configuration
    
    PLOT_RUN_CONFIGS = build_plot_run_configs_from_manifest()

    if not PLOT_RUN_CONFIGS:
        print(
            "[INFO][4.12] No completed saved run matches "
            "the manifest plot filters."
        )

    for RUN_CFG_PLOT in PLOT_RUN_CONFIGS:

        RUN_UID_PLOT = RUN_CFG_PLOT["run_uid"]
        RUN_LABEL_PLOT = RUN_CFG_PLOT["run_label"]
        MODE_PLOT = RUN_CFG_PLOT["mode"]
        MODE_LABEL_PLOT = RUN_CFG_PLOT.get("mode_label", MODE_PLOT)

        CONTROL_MODE_PLOT = {
            "BESS": RUN_CFG_PLOT["control_BESS"],
            "BEV": RUN_CFG_PLOT["control_BEV"],
            "DHW": RUN_CFG_PLOT["control_DHW"],
        }

        RUN_SHARES_PLOT = {
            tech: float(RUN_CFG_PLOT[f"share_{tech}_target"])
            for tech in RUN_SHARE_KEYS
        }

        # Use exactly the saved directories of this run.
        activate_hagrid_dirs_from_manifest_row(
            RUN_CFG_PLOT,
            create_pics=True,
        )

        # Do NOT reconstruct HAS_RUN from the current notebook settings.
        has_active_path = os.path.join(
            py_based_output_folder,
            "has_active.pkl.gz",
        )

        if not os.path.isfile(has_active_path):
            raise FileNotFoundError(
                f"Saved HAS_RUN not found for {RUN_UID_PLOT}:\n"
                f"{has_active_path}"
            )

        HAS_RUN_PLOT = load_compressed_pickle(has_active_path)

        has_PV_plot = set(HAS_RUN_PLOT.get("PV", set()))
        has_BESS_plot = set(HAS_RUN_PLOT.get("BESS", set()))
        has_BEV_plot = set(HAS_RUN_PLOT.get("BEV", set()))
        has_DHW_plot = set(HAS_RUN_PLOT.get("DHW", set()))
        has_HP_plot = set(HAS_RUN_PLOT.get("HP", set()))
        has_COOL_plot = set(HAS_RUN_PLOT.get("COOL", set()))
        has_IND_plot = set(HAS_RUN_PLOT.get("INDUCTION", set()))

        print("\n" + "=" * 100)
        print(f"[PLOT] run_uid = {RUN_UID_PLOT}")
        print(f"[PLOT] label   = {RUN_LABEL_PLOT}")
        print(f"[PLOT] input   = {py_based_output_folder}")
        print(f"[PLOT] output  = {optimization_graph_output_folder}")
        print("=" * 100)



        # -----------------------------------------------------------------------------
        #               Sec 4.12 -- 0.bis) Build CONTAINER for LV-GRID CONGESTION information
        # -----------------------------------------------------------------------------

        kerber_line_recipes = []
        if LV_ARCHETYPE in KERBER_BLUEPRINT_FILES:
            for e in KERBER_BP["lines_lv"]:
                fb = int(e["from_bus"])
                tb = int(e["to_bus"])
                a_key = KERBER_BUS_ID_TO_LABEL[fb]
                b_key = KERBER_BUS_ID_TO_LABEL[tb]
                a_s = _short_lv_bus_label(a_key)   # e.g. n001 / root / h001
                b_s = _short_lv_bus_label(b_key)

                # type of grid segment considered, backbone vs. final connection to customer
                role_a = KERBER_BUS_ROLE.get(a_key, "unknown")
                role_b = KERBER_BUS_ROLE.get(b_key, "unknown")

                # nominal capacity, kW, from the Kerber blueprint
                # pmax_kw = float(e.get("pmax_kw") or KERBER_DEFAULT_PMAX_KW)
                pmax_kw = _get_kerber_line_p_limit_kw(e, PF_LINE_LIMITS_MAP)

                # customer branch if it touches a "house" bus, h00Y; otherwise backbone
                kind = "branch_house" if ("house" in (role_a, role_b) or a_s.startswith("h") or b_s.startswith("h")) else "backbone"

                kerber_line_recipes.append({
                    "a_key": a_key, "b_key": b_key,
                    "a_s": a_s, "b_s": b_s,
                    "lab_forw": f"line_forw_{a_s}_to_{b_s}",
                    "lab_back": f"line_back_{b_s}_to_{a_s}",
                    "role_a": role_a, "role_b": role_b,
                    "pmax_kw": pmax_kw,
                    "kind": kind,
                })
        




        # ---------------------------------------------------------------------------------------------------------
        #               Sec 4.12 -- 1)  DEFINE COLORS/AREAS/DICTIONARIES FOR CHART CONSTRUCTION
        # --------------------------------------------------------------------------------------------------------- 
        # Recall the structure of the elements, columns, areas, colors, to include in the generation-demand chart,
        # used both in the Opt case and in the RB case
        area_columns, line_columns, generation_column_colors, demand_column_colors, column_names = \
            init_gen_dmd_plot_maps(FLEX_CATEGORIES, make_gradient)



    # ---------------------------------------------------------------------------------------------------------
    #               Sec 4.12 -- 2)  FIRST PASS, by season: load data, build plot df, update chart limits
    # ----------------------------------------------------------------------------------------------------------
    # if RUN_GRAFICI_CONFIGS:

        # --------------------------------------------------
        #           Sec 4.12 -- 2.0) initialize cache and global maxima
        # --------------------------------------------------

        # global maximum for Y-axis, Generation/Demand, calculated on the cumulative stack
        ymax_gen_dmd_global = 0.0
        
        # here we save the dataframes already prepared for plotting
        df_opt_plot_data = {}
        ymax_tmp = 0.0

        # here I save information for duration curves
        durcurve_cache = {}
        durcurve_mt_cache = {}  

        # cache for LV import duration curves, percentiles, and LV line congestion, Kerber
        lv_import_duration_curve_cache = {}
        durcurve_lv_lines_cache  = {}

        # global limits, standardization of Y axes across seasons, by chart category
        ymax_import_lv_global = -np.inf
        ymin_import_lv_global =  np.inf

        ymax_net_lines_global = -np.inf
        ymin_net_lines_global =  np.inf

        ymax_sat_branch_global   = -np.inf
        ymax_sat_backbone_global = -np.inf



        # ----------------------------------  
        #         1_old)  DEFINE COLORS/AREAS/DICTIONARIES FOR CHART CONSTRUCTION, previously done here
        # ----------------------------------  
        # Moved earlier into a function
        # -----------------------------------





        # ---------------------------------------------------------------------------------------------------------
        #               Sec 4.12 -- 2.1)  LOOP OVER SEASONS, import data and chart construction, Opt case
        # ---------------------------------------------------------------------------------------------------------  
        
        seasons_to_plot = globals().get("PLOT_ONLY_SEASONS", None)

        if seasons_to_plot is None:
            seasons_to_plot = seasons
        else:
            seasons_to_plot = [s for s in seasons_to_plot if s in seasons]

        for selected_season in tqdm(seasons_to_plot, desc="Seasons, weeks"):
            
            # -------------------------------------
            #           Sec 4.12 -- 2.1.1) Input/Output files
            # -------------------------------------
            # Input pickle files for this configuration and season
            filename_pov_case = resolve_flow_pickle_path("flow_el_df",selected_season, mode=MODE_PLOT, base_dir=py_based_output_folder,)
            lv_grid_pov_filename = resolve_flow_pickle_path("flow_grid_df",selected_season,mode=MODE_PLOT,base_dir=py_based_output_folder,)
            lv_grid_root_pov_filename = resolve_flow_pickle_path("flow_grid_root",selected_season,mode=MODE_PLOT,base_dir=py_based_output_folder,)
            file_dhw_use = resolve_flow_pickle_path("flow_DHW_use_df",selected_season,mode=MODE_PLOT,base_dir=py_based_output_folder,required=False,)
            file_bev_charge = resolve_flow_pickle_path("flow_charge_el_BEV_df",selected_season,mode=MODE_PLOT,base_dir=py_based_output_folder,required=False,)
            file_bev_dis = resolve_flow_pickle_path("flow_discharge_el_BEV_df",selected_season,mode=MODE_PLOT,base_dir=py_based_output_folder,required=False,)
            lv_grid_line_file = resolve_flow_pickle_path("flow_grid_lines_only_df",selected_season,mode=MODE_PLOT,base_dir=py_based_output_folder,required=False,)

            # Extract the file name without path and extension
            file_name = os.path.basename(filename_pov_case).replace('.pkl.gz', '')
                 
                    
            # Load files for the current configuration
            df_raw_case = load_compressed_pickle(filename_pov_case)
            lv_grid_results_df = load_compressed_pickle(lv_grid_pov_filename)               # Includes all buses along the feeder
            lv_grid_root_results_df = load_compressed_pickle(lv_grid_root_pov_filename)     # Includes only the bus from secondary substation to feeder start

            df_dhw_use = (load_compressed_pickle(file_dhw_use) if file_dhw_use else None)
            df_bev_charge = (load_compressed_pickle(file_bev_charge) if file_bev_charge else None)
            df_bev_dis = (load_compressed_pickle(file_bev_dis) if file_bev_dis else None)
            lv_grid_line_results_df = (load_compressed_pickle(lv_grid_line_file) if lv_grid_line_file else pd.DataFrame(index=df_raw_case.index))




            # ---------------------------
            #   Sec 4.12 -- 2.1.1bis) DEBUG: check whether there are bidirectional flows along the LV grid
            # ---------------------------
            # check: how many times forward and backward are both > 0, possible loops/artifacts
            eps = 1e-6
            pairs = []
            for r in kerber_line_recipes:
                cf = (r["lab_forw"], r["b_key"], "flow")
                cb = (r["lab_back"], r["a_key"], "flow")
                if cf in lv_grid_line_results_df.columns and cb in lv_grid_line_results_df.columns:
                    forw = pd.to_numeric(lv_grid_line_results_df[cf], errors="coerce").fillna(0.0)
                    back = pd.to_numeric(lv_grid_line_results_df[cb], errors="coerce").fillna(0.0)
                    both = int(((forw > eps) & (back > eps)).sum())
                    pairs.append((r["lab_forw"], r["lab_back"], both))

            pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
            print("Top simultaneous forward/backward flows, possible loops:", pairs[:10])
            # -----------------------------------------------------------------------



            # -------------------------------------
            #       Sec 4.12 -- 2.1.2) storage flows: DHW and BEV, pre-merge
            # -------------------------------------

            # --- flows for DHW storage: charge is in df_raw_case, discharge in df_dhw_use
            if df_dhw_use is not None:
                dhw_house_flows_df = pd.concat([df_raw_case, df_dhw_use], axis=1)
            else:
                # fallback: only electric side, no DHW discharge -> soc decreases only because of losses
                print(f"[WARN DHW] No df_dhw_use for season {selected_season}, using only df_raw_case.")
                dhw_house_flows_df = df_raw_case

            # --- flows for BEV storage: charge + discharge
            dfs_bev = []
            if df_bev_charge is not None:
                dfs_bev.append(df_bev_charge)
            if df_bev_dis is not None:
                dfs_bev.append(df_bev_dis)

            if len(dfs_bev) > 0:
                ev_house_flows_df = pd.concat(dfs_bev, axis=1)
            else:
                ev_house_flows_df = None
                print(f"[WARN BEV] No BEV flow for season {selected_season}.")


            # -------------------------------------------------------------
            #       Sec 4.12 -- 2.1.3) identify technologies actually available in the plotted configuration, from file
            # -------------------------------------------------------------

            has_BESS_from_pkl = infer_has_bess_from_flows(df_raw_case)
            has_DHW_from_pkl  = infer_has_dhw_from_flows(dhw_house_flows_df)
            has_BEV_from_pkl  = infer_has_bev_from_flows(df_bev_charge, df_bev_dis)

            # In mixed mode, some RB flows do not have the same tags as OPT storage flows.
            # Therefore, the main source remains HAS_RUN_PLOT.
            # Inference from pickle is only for debug/sanity check.
            print(f"[INFO][{MODE_PLOT}] BESS active from config: {len(has_BESS_plot)} | inferred from pkl: {len(has_BESS_from_pkl)}")
            print(f"[INFO][{MODE_PLOT}] DHW  active from config: {len(has_DHW_plot)} | inferred from pkl: {len(has_DHW_from_pkl)}")
            print(f"[INFO][{MODE_PLOT}] BEV  active from config: {len(has_BEV_plot)} | inferred from pkl: {len(has_BEV_from_pkl)}")


            # -------------------------------------------------------------
            #       Sec 4.12 -- 2.1.4) DHW: capacities/powers ONLY for houses that actually have DHW - BEV
            # -------------------------------------------------------------
            
            # DHW
            capacity_dhw_by_house_eff = {c: capacity_dhw_by_house[c]            for c in has_DHW_plot                if c in capacity_dhw_by_house}
            power_limit_dhw_by_house_eff = {c: power_limit_dhw_by_house[c]      for c in has_DHW_plot                if c in power_limit_dhw_by_house}
            loss_rate_dhw_by_house_eff = {c: loss_rate_dhw_by_house[c]          for c in has_DHW_plot                if c in loss_rate_dhw_by_house}
            initial_soc_dhw_by_house_eff = {c: initial_soc_dhw_by_house[c]      for c in has_DHW_plot                if c in initial_soc_dhw_by_house}
            
            
            # BEV
            bev_cap_eff = ev_capacity_by_case.where(ev_capacity_by_case.index.isin(has_BEV_plot), np.nan)

            capacity_bev_by_house = {c: float(bev_cap_eff.loc[c])
                for c in has_BEV_plot                if (c in bev_cap_eff.index)
                and pd.notna(bev_cap_eff.loc[c])   and (bev_cap_eff.loc[c] > 0)}


            initial_soc_bev_by_house = {
                c: initial_SoC_BEV * capacity_bev_by_house[c]
                for c in capacity_bev_by_house
            }


            # -------------------------------------------------------------
            #       Sec 4.12 -- 2.1.5) normalize/flatten columns
            # -------------------------------------------------------------
            df_result_el           = convert_busseq_to_flowtuple_df(df_raw_case)
            lv_grid_results_df      = convert_busseq_to_flowtuple_df(lv_grid_results_df)
            lv_grid_root_results_df = convert_busseq_to_flowtuple_df(lv_grid_root_results_df)

            dhw_house_flows_df = convert_busseq_to_flowtuple_df(dhw_house_flows_df)
            if ev_house_flows_df is not None:
                ev_house_flows_df = convert_busseq_to_flowtuple_df(ev_house_flows_df)

            lv_grid_line_results_df = convert_busseq_to_flowtuple_df(lv_grid_line_results_df)
            # --- debug
            _tmp = lv_grid_line_results_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
            print(f"[CHECK] df_result_rete_BT_lines max abs, post-load = {float(_tmp.abs().to_numpy().max()) if _tmp.size else 0.0}")
            # -------------

            # -------------------------------------------------------------
            #       Sec 4.12 -- 2.1.6) Read net flows on LV lines
            # -------------------------------------------------------------
            if plt_load_duration_LV_grid_lines:
                try:
                    net_line_outputs_plot = load_net_lines_outputs_for_plot(
                        season=selected_season,
                        base_dir=optimization_output_folder,
                    )

                    df_net_lines_ts_signed = net_line_outputs_plot["net_lines_ts_signed"]
                    df_net_lines_ts_for_dur = net_line_outputs_plot["net_lines_ts_for_dur"]
                    df_dur_net_lines = net_line_outputs_plot["durcurve_net_lines"]
                    df_lines_meta = net_line_outputs_plot["lines_meta"]
                    df_sat_lines_ts_pct = net_line_outputs_plot["sat_lines_ts_pct"]
                    df_dur_sat_lines = net_line_outputs_plot["durcurve_sat_lines_pct"]

                    if "kind" in df_lines_meta.columns:
                        s_kind = df_lines_meta["kind"]
                    else:
                        s_kind = pd.Series(index=df_dur_sat_lines.columns, dtype=object)

                    cols_branch = [
                        c for c in df_dur_sat_lines.columns
                        if s_kind.get(c) == "branch_house"
                    ]

                    cols_backbone = [
                        c for c in df_dur_sat_lines.columns
                        if s_kind.get(c) == "backbone"
                    ]

                    pct = (10, 20, 30, 40, 50, 60, 70, 80, 90)

                    durcurve_lv_lines_cache[selected_season] = {
                        "df_dur_net_lines": df_dur_net_lines,
                        "df_dur_sat_lines": df_dur_sat_lines,
                        "cols_branch": cols_branch,
                        "cols_backbone": cols_backbone,
                        "pct": pct,
                    }

                    _arr_lines = df_dur_net_lines.to_numpy(dtype=float)
                    _arr_lines = _arr_lines[np.isfinite(_arr_lines)]

                    if _arr_lines.size:
                        ymax_net_lines_global = max(ymax_net_lines_global, float(_arr_lines.max()))
                        ymin_net_lines_global = min(ymin_net_lines_global, float(_arr_lines.min()))

                    if df_dur_sat_lines is not None and not df_dur_sat_lines.empty:
                        if cols_branch:
                            _arr_sb = df_dur_sat_lines[cols_branch].to_numpy(dtype=float)
                            _arr_sb = _arr_sb[np.isfinite(_arr_sb)]
                            if _arr_sb.size:
                                ymax_sat_branch_global = max(ymax_sat_branch_global, float(_arr_sb.max()))

                        if cols_backbone:
                            _arr_sbb = df_dur_sat_lines[cols_backbone].to_numpy(dtype=float)
                            _arr_sbb = _arr_sbb[np.isfinite(_arr_sbb)]
                            if _arr_sbb.size:
                                ymax_sat_backbone_global = max(ymax_sat_backbone_global, float(_arr_sbb.max()))

                except FileNotFoundError as e:
                    print(f"[WARN][4.12][NET LINES] {e}")


            # --------------------------------------------
            #       Sec 4.12 -- 2.2) DURATION CURVES for LV IMPORT flows, Opt case -> houses
            #      and calculation of DHW BOOST usage
            # --------------------------------------------
            dt_hours = _dt_hours_from_index(df_raw_case.index, timestep_minutes_fallback=timestep_minutes)
            
            lv_import_dict = extract_lv_import_series_by_house(df_result_el, ID)
            lv_export_dict = extract_lv_export_series_by_house(df_result_el, ID)
            net_lv_dict    = compute_net_lv_by_house(lv_import_dict, lv_export_dict)


            # Duration curve of domestic import only from LV grid and duration curve of domestic NET flow, import-feed-in, from LV grid
            lv_import_duration_df = build_duration_curves(lv_import_dict)    if lv_import_dict else pd.DataFrame()
            lv_net_duration_df = build_duration_curves(net_lv_dict)          if net_lv_dict    else pd.DataFrame()       

            # number of weekly timesteps, 10'
            n_ts_week = len(next(iter(lv_import_dict.values()))) if lv_import_dict else 1008

            # Two-pass cache + global limits for LV IMPORT duration curves, percentiles
            if (lv_import_duration_df is not None) and (not lv_import_duration_df.empty):
                lv_import_duration_curve_cache[selected_season] = {
                    "df_dur_import_bt": lv_import_duration_df,
                    "dt_hours": dt_hours,
                }
                _arr_imp = lv_import_duration_df.to_numpy(dtype=float)
                _arr_imp = _arr_imp[np.isfinite(_arr_imp)]
                if _arr_imp.size:
                    ymax_import_lv_global = max(ymax_import_lv_global, float(_arr_imp.max()))
                    ymin_import_lv_global = min(ymin_import_lv_global, float(_arr_imp.min()))




            # --------------------------------------------
            #       Sec 4.12 -- 2.2.1) Calculate flows exchanged with MV grid
    
            # LV root bus to MV
            lv_root = LV_ROOT_LABEL if LV_ROOT_LABEL is not None else ("bus_el_BT" if LV_ARCHETYPE == "single_bus" else "bus_el_BT_0")

            col_imp_mv = ('supply_from_grid_MT', lv_root, 'flow')
            col_exp_mv = (lv_root, 'excess_to_grid_MT', 'flow')

            s_imp_mv = pd.to_numeric(lv_grid_root_results_df[col_imp_mv], errors='coerce').fillna(0.0) if col_imp_mv in lv_grid_root_results_df.columns else pd.Series(0.0, index=lv_grid_root_results_df.index)
            s_exp_mv = pd.to_numeric(lv_grid_root_results_df[col_exp_mv], errors='coerce').fillna(0.0) if col_exp_mv in lv_grid_root_results_df.columns else pd.Series(0.0, index=lv_grid_root_results_df.index)

            s_net_mv = s_imp_mv - s_exp_mv                              # <--- total demand requested by the neighborhood from MV
            s_net_mv_avg = s_net_mv / float(num_households)         # <--- demand requested by an "average" household, total demand from MV grid / no. customers in the neighborhood; useful to see simultaneity

            # duration curve, neighborhood
            dur_mt = duration_curve_from_series(s_net_mv)

            # duration curve, average household
            dur_mt_avg = duration_curve_from_series(s_net_mv_avg)

            ymax_mt_global = max(ymax_mt_global, float(max(dur_mt.max(), dur_mt_avg.max())))
            ymin_mt_global = min(ymin_mt_global, float(min(dur_mt.min(), dur_mt_avg.min())))

            # save for second pass
            durcurve_mt_cache[selected_season] = {"dur_mt": dur_mt, "dur_mt_avg": dur_mt_avg}



            # --------------------------------------------
            #       Sec 4.12 -- 2.2.2) Duration curve of summed imports from LV grid to houses
            #           simultaneity required by customers, LV "gross avg": sum of LV imports of all houses / n_households


            # If there is no import_bt_dict, fallback to a zero series consistent with the index
            if lv_import_dict:
                # DataFrame with columns = houses, c_1, c_2, ..., index = timesteps
                all_lv_import_df = pd.DataFrame(lv_import_dict)

                # Total LV import series, kW, by timestep: sum over all houses
                total_lv_import = all_lv_import_df.sum(axis=1).fillna(0.0)   # kW

            else:
                # fallback: zero series with consistent index
                total_lv_import = pd.Series(0.0, index=df_result_el.index)

            # Total duration curve, sorted descending, as Series indexed 0..N-1
            s_import_duration_total = pd.Series(np.sort(total_lv_import.values)[::-1]).reset_index(drop=True)

            # Duration curve PER HOUSEHOLD: divide by number of households
            s_import_mv_avg = s_import_duration_total / float(num_households)

            # cache for second pass: save df_dur_net_bt, net MV avg overlay and LV import, Low Voltage LV, avg overlay
            durcurve_cache[selected_season] = {
                "df_dur_net_bt": lv_net_duration_df,
                "s_net_mv_avg": s_net_mv_avg,
                "s_import_mv_avg": s_import_mv_avg,
                "dt_hours": dt_hours,
            }
        

            # --------------------------------------------
            #      Sec 4.12 -- 2.2.3) update global limits, chart Y-axis, LV duration curves

            if (lv_net_duration_df is not None) and (not lv_net_duration_df.empty):
                p95_series = lv_net_duration_df.quantile(0.95, axis=1)  # index=rank
                p05_series = lv_net_duration_df.quantile(0.05, axis=1)

                # MV avg overlay in duration
                dur_mv_avg = duration_curve_from_series(s_net_mv_avg)
                dur_import_mv_avg = duration_curve_from_series(s_import_mv_avg)

                ymax_here = max(float(p95_series.max()), float(dur_mv_avg.max()), float(dur_import_mv_avg.max()))
                ymin_here = min(float(p05_series.min()), float(dur_mv_avg.min()), float(dur_import_mv_avg.min()))

                ymax_net_lv_global = max(ymax_net_lv_global, ymax_here)
                ymin_net_lv_global = min(ymin_net_lv_global, ymin_here)





            # --------------------------------------------
            #       Sec 4.12 -- 2.4)  HOUSE CLASSIFICATION BY FLEXIBILITY
            # --------------------------------------------

            categories_to_aggregate = {
                "case_0": [],
                "case_B": [],
                "case_V": [],
                "case_H": [],
                "case_BV": [],
                "case_BH": [],
                "case_VH": [],
                "case_BVH": [],
            }

            for c in ID:
                cat = classify_flex_case(
                    c,
                    has_BESS_plot,
                    has_BEV_plot,
                    has_DHW_plot,
                )
                categories_to_aggregate[cat].append(c)

            case_0   = categories_to_aggregate["case_0"]
            case_B   = categories_to_aggregate["case_B"]
            case_V   = categories_to_aggregate["case_V"]
            case_H   = categories_to_aggregate["case_H"]
            case_BV  = categories_to_aggregate["case_BV"]
            case_BH  = categories_to_aggregate["case_BH"]
            case_VH  = categories_to_aggregate["case_VH"]
            case_BVH = categories_to_aggregate["case_BVH"]
            

            # DataFrame of sums on the house side and LV side, for this season
            sums_df = pd.DataFrame(index=df_result_el.index)
            lv_grid_sums_df = pd.DataFrame(index=df_result_el.index)

            # Define the start and end of the period: the whole week contained in the dataframe
            period_start = lv_grid_sums_df.index.min()
            period_end   = lv_grid_sums_df.index.max()


            # --------------------------------------------
            #       Sec 4.12 -- 2.4.1) HOUSE SIDE: aggregations
            for prefisso, case_list in categories_to_aggregate.items():
                if not case_list:
                    continue

                # Base demand
                house_flows_sum(df_result_el, case_list,'bus_el', 'dmd_elc_base',(f'{prefisso}_bus_el', f'{prefisso}_dmd_elc_base'),sums_df)

                # Cooling demand
                house_flows_sum(df_result_el, case_list,'bus_el', 'dmd_elc_Cool',(f'{prefisso}_bus_el', f'{prefisso}_dmd_elc_Cool'),sums_df)

                # Heating demand
                house_flows_sum(df_result_el, case_list,'bus_el', 'dmd_elc_Heat',(f'{prefisso}_bus_el', f'{prefisso}_dmd_elc_Heat'),sums_df)

                # Cooking demand
                house_flows_sum(df_result_el, case_list,'bus_el', 'dmd_elc_Cook',(f'{prefisso}_bus_el', f'{prefisso}_dmd_elc_Cook'),sums_df)


                # PV generation, both cases with curtailment "c_i_PV_to_house -> c_i_bus_el" and without curtailment "c_i_PV -> c_i_bus_el"
                house_flows_from_pv_sum(df_result_el, case_list, (f'{prefisso}_PV', f'{prefisso}_bus_el'), sums_df)

                # BESS, charge and discharge
                house_flows_sum(df_result_el, case_list,'bus_el', 'BESS',(f'{prefisso}_bus_el', f'{prefisso}_BESS'),sums_df)
                house_flows_sum(df_result_el, case_list,'BESS', 'bus_el',(f'{prefisso}_BESS', f'{prefisso}_bus_el'),sums_df)

                # BEV: charge and V2G
                house_flows_sum(df_result_el, case_list,'bus_el', 'charger',(f'{prefisso}_bus_el', f'{prefisso}_charger'),sums_df)
                house_flows_sum(df_result_el, case_list,'V2G', 'bus_el',(f'{prefisso}_V2G', f'{prefisso}_bus_el'),sums_df)

                # DHW storage
                house_flows_sum(df_result_el, case_list,'bus_el', 'DHW_storage',(f'{prefisso}_bus_el', f'{prefisso}_DHW_storage'),sums_df)

                # Additional DHW with "boost"
                house_flows_sum(df_result_el, case_list,'bus_el', 'DHW_boost',(f'{prefisso}_bus_el', f'{prefisso}_DHW_boost'),sums_df)


                # Exchanges with LV grid
                house_flows_sum(df_result_el, case_list,'bus_el', 'to_the_grid_BT',(f'{prefisso}_bus_el', f'{prefisso}_to_the_grid_BT'),sums_df)
                house_flows_sum(df_result_el, case_list,'from_the_grid_BT_to', 'bus_el',(f'from_the_grid_BT_to_{prefisso}', f'{prefisso}_bus_el'),sums_df)


            # --------------------------------------------
            #       Sec 4.12 -- 2.4.2) LV GRID SIDE: aggregations 

            for prefisso, case_list in categories_to_aggregate.items():
                if not case_list:
                    continue

                # 2.4.2a) WITHDRAWALS: BT_segment -> from_the_grid_BT_to_<c_i>
                cols_import = []
                for lv in lv_bus_labels:
                    for c in case_list:
                        cols_import.append((lv, f"from_the_grid_BT_to_{c}", "flow"))

                s_import = _sum_cols_safe(lv_grid_results_df, cols_import)

                # save with a "logical" key on bt_root, so the LV grid appears as a single bus in the charts
                lv_grid_sums_df[(lv_root, f"from_the_grid_BT_to_{prefisso}", "flow")] = s_import

                # 2.4.2b) FEED-IN: <c_i>_to_the_grid_BT -> BT_segment
                cols_export = []
                for lv in lv_bus_labels:
                    for c in case_list:
                        cols_export.append((f"{c}_to_the_grid_BT", lv, "flow"))

                s_export = _sum_cols_safe(lv_grid_results_df, cols_export)

                lv_grid_sums_df[(f"{prefisso}_to_the_grid_BT", lv_root, "flow")] = s_export


            # --------------------------------------------
            #       Sec 4.12 -- 2.4.3) HOUSE SIDE: flattening, aggregation and name assignment for chart mapping  

            flattened_sums_df = sums_df.copy()
            flattened_sums_df.columns = [(c[0][0], c[0][1], c[1]) for c in flattened_sums_df.columns]
            lv_grid_apparent_sums_df = lv_grid_sums_df.copy() 

            lv_column_names = {}
            for pref, descr in flex_category_labels.items():
                lv_column_names[(lv_root, f'from_the_grid_BT_to_{pref}', 'flow')]   = f'Withdrawal from LV grid to {descr}'
                lv_column_names[(f'{pref}_to_the_grid_BT', lv_root, 'flow')]        = f'Electricity feed-in to LV grid from {descr}'
            
            lv_column_names[(lv_root, 'excess_to_grid_MT', 'flow')]     = 'Excess to MV grid'          
            lv_column_names[('supply_from_grid_MT', lv_root, 'flow')]   = 'Import from MV grid'



            # --------------------------------------------
            #       Sec 4.12 -- 2.4.4) BUILD AGGREGATED COLUMNS ON THE DEMAND SIDE 
            # Sum base / cool / heat / cook demands across all household cases

            demand_flows_agg = [
                ('dmd_elc_base', 'ALL_dmd_elc_base'),
                ('dmd_elc_Cool', 'ALL_dmd_elc_Cool'),
                ('dmd_elc_Heat', 'ALL_dmd_elc_Heat'),
                ('dmd_elc_Cook', 'ALL_dmd_elc_Cook'),
            ]

            for suffix, agg_name in demand_flows_agg:
                columns_to_sum = [
                    (f'{pref}_bus_el', f'{pref}_{suffix}', 'flow')
                    for pref in FLEX_CATEGORIES
                    if (f'{pref}_bus_el', f'{pref}_{suffix}', 'flow') in flattened_sums_df.columns
                ]
                if columns_to_sum:
                    flattened_sums_df[('ALL_bus_el', agg_name, 'flow')] = \
                        flattened_sums_df[columns_to_sum].sum(axis=1)


            # --------------------------------------------
            #       Sec 4.12 -- 2.4.5) Also add import/export with MV directly from df_result_grid_BT

            col_imp_mv = ('supply_from_grid_MT', lv_root, 'flow')
            col_exp_mv = (lv_root, 'excess_to_grid_MT', 'flow')

            if col_imp_mv in lv_grid_root_results_df.columns:
                lv_grid_sums_df[(col_imp_mv[0], col_imp_mv[1], 'flow')] = pd.to_numeric(lv_grid_root_results_df[col_imp_mv], errors='coerce').fillna(0.0)
            else:
                lv_grid_sums_df[(col_imp_mv[0], col_imp_mv[1], 'flow')] = 0.0

            if col_exp_mv in lv_grid_root_results_df.columns:
                lv_grid_sums_df[(col_exp_mv[0], col_exp_mv[1], 'flow')] = pd.to_numeric(lv_grid_root_results_df[col_exp_mv], errors='coerce').fillna(0.0)
            else:
                lv_grid_sums_df[(col_exp_mv[0], col_exp_mv[1], 'flow')] = 0.0


            # --------------------------------------------
            #       Sec 4.12 -- 2.5) WEEKLY SUMMARY, weekly sums, + construction of df_chart_case
            # --------------------------------------------

            house_sums = flattened_sums_df.sum()
            lv_sums = lv_grid_apparent_sums_df.sum()

            if 'weekly_sums_all_seasons' not in globals():
                weekly_sums_all_seasons = pd.DataFrame()

            season_summary = pd.DataFrame({
                'Season': [selected_season] * (len(house_sums) + len(lv_sums)),
                'Type': ['Case'] * len(house_sums) + ['Rete LV'] * len(lv_sums),
                'Column': list(house_sums.index) + list(lv_sums.index),
                'Weekly_sum [kWh/week]': list(house_sums) + list(lv_sums),
                'PV_households': [None] * (len(house_sums) + len(lv_sums)),
                'BESS_households': [None] * (len(house_sums) + len(lv_sums)),
                'BESS_share [%]': [None] * (len(house_sums) + len(lv_sums))
            })

            # Rename columns with column_names / lv_column_names map
            renamed_map = {**column_names, **lv_column_names}
            season_summary['Display_name'] = season_summary['Column'].map(renamed_map)

            # Add to the global DataFrame
            weekly_sums_all_seasons = pd.concat(
                [weekly_sums_all_seasons, season_summary],
                ignore_index=True
            )

            # dataframe for gen/dmd charts, whole week
            start = period_start
            end = period_end
            print(f"Generating chart for period {start} - {end}")
            
            scenario_plot_df = flattened_sums_df[
                (flattened_sums_df.index >= start) &
                (flattened_sums_df.index <= end)
            ]



            # --------------------------------------------
            #       Sec 4.12 -- 2.5.1) Add area including DHW_boost, always, even if all zero

            col_boost_all = ('ALL_bus_el', 'ALL_DHW_boost', 'flow')
            if col_boost_all not in line_columns:
                line_columns.append(col_boost_all)

            # Add area including DHW_boost
            boost_cols = [
                col for col in scenario_plot_df.columns
                if isinstance(col, tuple)
                and len(col) == 3
                and col[2] == 'flow'
                and str(col[1]).endswith('_DHW_boost')   # target = *_DHW_boost
                and str(col[0]).endswith('_bus_el')      # source = *_bus_el
            ]
            
            if boost_cols:
                scenario_plot_df[col_boost_all] = scenario_plot_df[boost_cols].sum(axis=1)
            else:
                # still create the column, all zero, so it can appear in the legend
                scenario_plot_df[col_boost_all] = 0.0



            # --------------------------------------------
            #       Sec 4.12 -- 2.5.2) Select only columns that actually exist 

            present_area_columns = [col for col in area_columns
                                        if col in scenario_plot_df.columns]
            present_line_columns = [col for col in line_columns
                                        if col in scenario_plot_df.columns]

            if (not present_area_columns) or (not present_line_columns):
                print(f"[WARN] No gen/dmd column for season {selected_season} -> skip gen/dmd chart.")
                continue



            # ---------------------------------------------------
            #       Sec 4.12 -- 2.5.3) Update global maximum for chart Y-axis
            # ---------------------------------------------------

            # Sum of GENERATION contributions for each timestep
            gen_stack = scenario_plot_df[present_area_columns].sum(axis=1)

            # Sum of DEMAND contributions for each timestep
            dom_stack = scenario_plot_df[present_line_columns].sum(axis=1)

            # maximum "stack height" for this season
            max_here = max(gen_stack.max(skipna=True), dom_stack.max(skipna=True))

            if pd.notna(max_here):
                ymax_gen_dmd_global = max(ymax_gen_dmd_global, float(max_here))


            # Save what is needed for the second pass, only seasons with valid data
            df_opt_plot_data[selected_season] = scenario_plot_df



            # Legend colors and labels, only for actually available columns
            colors_gen = [generation_column_colors.get(col, 'gray') for col in present_area_columns]
            colors_dom = [demand_column_colors.get(col, 'gray') for col in present_line_columns]
            labels_gen = [column_names.get(col, str(col)) for col in present_area_columns]
            labels_dom = [column_names.get(col, str(col)) for col in present_line_columns]
            edgecolors_gen = [darken_color(c, 0.6) for c in colors_gen]
            edgecolors_dom = [darken_color(c, 0.6) for c in colors_dom]
            #-------------------------------------------------------------------------------------------


            # -----------------------------
            #       Sec 4.12 -- 3) PRINT SOC PERCENTILE CHARTS
            # -----------------------------
            soc_techs_to_plot = globals().get(
                "PLOT_ONLY_SOC_TECHS",
                ["BESS", "DHW", "BEV"],
            )

            for tech_soc in soc_techs_to_plot:

                tech_soc = str(tech_soc).upper()

                # If the technology is NO in the configuration, skip directly.
                if _tech_is_no(CONTROL_MODE_PLOT, tech_soc):
                    print(
                        f"[INFO][SOC PLOT] Skip {tech_soc} for {MODE_PLOT}: "
                        f"technology absent in the configuration."
                    )
                    continue

                plot_soc_percentiles_from_xlsx(
                    tech=tech_soc,
                    season=selected_season,
                    out_dir_graph=optimization_graph_output_folder,
                    mode=MODE_PLOT,
                    mode_label=MODE_LABEL_PLOT,
                    base_dir_soc=optimization_output_folder,
                    control_mode=CONTROL_MODE_PLOT.get(tech_soc, "OPT"),
                )




        # --------------------------------------------
        #       Sec 4.12 -- 4) At the end of RUN_GRAFICI_CONFIGS: normalize global limits/maxima
        # --------------------------------------------
        # gen/dmd
        ymax_gen_dmd_global = 1.0 if (ymax_gen_dmd_global is None or ymax_gen_dmd_global <= 0) else 1.05 * ymax_gen_dmd_global

        # LV curves
        if not np.isfinite(ymax_net_lv_global): ymax_net_lv_global = 1.0
        if not np.isfinite(ymin_net_lv_global): ymin_net_lv_global = 0.0
        span = ymax_net_lv_global - ymin_net_lv_global
        span = span if span > 0 else max(1.0, abs(ymax_net_lv_global) if ymax_net_lv_global != 0 else 1.0)
        ymax_net_lv_global += 0.05 * span
        ymin_net_lv_global -= 0.05 * span

        # MV curves
        if not np.isfinite(ymax_mt_global): ymax_mt_global = 1.0
        if not np.isfinite(ymin_mt_global): ymin_mt_global = 0.0
        span = ymax_mt_global - ymin_mt_global
        span = span if span > 0 else max(1.0, abs(ymax_mt_global))
        ymax_mt_global += 0.05 * span
        ymin_mt_global -= 0.05 * span

        # LV IMPORT curves, percentiles
        if not np.isfinite(ymax_import_lv_global): ymax_import_lv_global = 1.0
        if not np.isfinite(ymin_import_lv_global): ymin_import_lv_global = 0.0
        span = ymax_import_lv_global - ymin_import_lv_global
        span = span if span > 0 else max(1.0, abs(ymax_import_lv_global))
        ymax_import_lv_global += 0.05 * span
        ymin_import_lv_global = max(0.0, ymin_import_lv_global - 0.05 * span)

        # LV line curves, net flows on segments
        if not np.isfinite(ymax_net_lines_global): ymax_net_lines_global = 1.0
        if not np.isfinite(ymin_net_lines_global): ymin_net_lines_global = 0.0
        span = ymax_net_lines_global - ymin_net_lines_global
        span = span if span > 0 else max(1.0, max(abs(ymax_net_lines_global), abs(ymin_net_lines_global)))
        ymax_net_lines_global += 0.05 * span
        ymin_net_lines_global -= 0.05 * span
        if USE_ABS_FOR_LINES_DURCURVE:
            ymin_net_lines_global = max(0.0, ymin_net_lines_global)

        # LV line saturation [%], branch/backbone
        if not np.isfinite(ymax_sat_branch_global): ymax_sat_branch_global = 100.0
        if not np.isfinite(ymax_sat_backbone_global): ymax_sat_backbone_global = 100.0
        ymax_sat_branch_global   = min(100.0, 1.05 * ymax_sat_branch_global)   if ymax_sat_branch_global   > 0 else 100.0
        ymax_sat_backbone_global = min(100.0, 1.05 * ymax_sat_backbone_global) if ymax_sat_backbone_global > 0 else 100.0




        
        # ------------------------------------------------
        #       Sec 4.12 -- 5) SECOND PASS: print/plot GENERATION/DEMAND CHARTS, OPTIMIZATION
        #       USING ymax_gen_dmd_global FOR ALL SEASONS
        # ------------------------------------------------
        for selected_season, scenario_plot_df in df_opt_plot_data.items():

            # rebuild the list of available columns from the saved dataframes
            present_area_columns = [col for col in area_columns
                                        if col in scenario_plot_df.columns]
            present_line_columns = [col for col in line_columns
                                        if col in scenario_plot_df.columns]

            if not present_area_columns or not present_line_columns:
                continue


            # Make sure not to insert impossible columns, e.g. El to DHW for houses with only B+V
            present_area_columns  = filter_cols_structural(present_area_columns, FLEX_CATEGORIES)
            present_line_columns = filter_cols_structural(present_line_columns, FLEX_CATEGORIES)

            # Legend colors and labels, only for actually available columns
            colors_gen = [generation_column_colors.get(col, 'gray') for col in present_area_columns]
            colors_dom = [demand_column_colors.get(col, 'gray') for col in present_line_columns]
            labels_gen = [column_names.get(col, str(col)) for col in present_area_columns]
            labels_dom = [column_names.get(col, str(col)) for col in present_line_columns]
            edgecolors_gen = [darken_color(c, 0.6) for c in colors_gen]
            edgecolors_dom = [darken_color(c, 0.6) for c in colors_dom]

            max_leg_cols = 2
            n_items_gen = len(labels_gen)
            n_items_dom = len(labels_dom)

            rows_gen = math.ceil(n_items_gen / max_leg_cols) if n_items_gen > 0 else 1
            rows_dom = math.ceil(n_items_dom / max_leg_cols) if n_items_dom > 0 else 1

            fig = plt.figure(figsize=(15, 7))

            gs = gridspec.GridSpec(
                2, 2,
                width_ratios=[4.7, 1.3],
                height_ratios=[1.0, 1.0],
                wspace=0.03,
                hspace=0.15
            )

            ax1 = fig.add_subplot(gs[0, 0])                 # Generation
            ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)     # Demand
            ax_leg1 = fig.add_subplot(gs[0, 1])             # Generation legend
            ax_leg2 = fig.add_subplot(gs[1, 1])             # Demand legend



            # ------------------------------------------------
            #       Sec 4.12 -- 5.1) PLOT GENERATION

            polys_gen = ax1.stackplot(
                scenario_plot_df.index,
                [scenario_plot_df[col] for col in present_area_columns],
                labels=labels_gen,
                colors=colors_gen,
                linewidth=0.3,
                alpha=0.85
            )
            for poly, c in zip(polys_gen, colors_gen):
                poly.set_edgecolor(darken_color(c, 0.55))
                poly.set_linewidth(0.3)

            ax1.set_title("Generation", fontsize=11)
            ax1.set_ylabel("Power [kW]", fontsize=9)
            ax1.tick_params(axis='x', labelbottom=False)


            # ------------------------------------------------
            #       Sec 4.12 -- 5.2) PLOT DEMAND

            polys_dom = ax2.stackplot(
                scenario_plot_df.index,
                [scenario_plot_df[col] for col in present_line_columns],
                labels=labels_dom,
                colors=colors_dom,
                linewidth=0.3,
                alpha=0.85
            )
            for poly, c in zip(polys_dom, colors_dom):
                poly.set_edgecolor(darken_color(c, 0.55))
                poly.set_linewidth(0.3)

            ax2.set_title("Demand", fontsize=11)
            ax2.set_ylabel("Power [kW]", fontsize=9)

            # Common limit across all seasons
            ax1.set_ylim(0, ymax_gen_dmd_global)
            ax2.set_ylim(0, ymax_gen_dmd_global)


            # ------------------------------------------------
            #       Sec 4.12 -- 5.3) legend 
            
            handles1, labels1 = ax1.get_legend_handles_labels()
            ncol1 = auto_ncol_legend(labels1, fig, max_cols=max_leg_cols,
                                    min_cols=1, char_px=7)
            ax_leg1.axis("off")
            leg1 = ax_leg1.legend(
                handles1, labels1,
                loc="upper left",
                bbox_to_anchor=(0.0, 1.0),
                ncol=ncol1,
                frameon=True,
                fontsize=7.5,
                title= "Legend – energy flows for generation\n"
                    "Flexibility options:\n(B = battery, V = el vehicle, W = hot water tank)\n ",
                title_fontsize=8,
                columnspacing=0.4,
                handlelength=0.9,
                handletextpad=0.25,
                borderpad=0.15,
                labelspacing=0.16
            )
            leg1.get_title().set_fontweight("bold")

            handles2, labels2 = ax2.get_legend_handles_labels()
            ncol2 = auto_ncol_legend(labels2, fig, max_cols=max_leg_cols,
                                    min_cols=1, char_px=7)
            ax_leg2.axis("off")
            leg2 = ax_leg2.legend(
                handles2, labels2,
                loc="upper left",
                bbox_to_anchor=(0.0, 1.0),
                ncol=ncol2,
                frameon=True,
                fontsize=7.5,
                title= "Legend – energy flows for demand\n"
                    "Flexibility options:\n(B = battery, V = el vehicle, W = hot water tank)\n ",
                title_fontsize=8,
                columnspacing=0.4,
                handlelength=0.9,
                handletextpad=0.25,
                borderpad=0.15,
                labelspacing=0.16
            )
            leg2.get_title().set_fontweight("bold")

            hours = pd.date_range(
                scenario_plot_df.index.min(),
                scenario_plot_df.index.max(),
                freq="1h"
            )
            ax2.set_xticks(hours)
            xticklabels = []
            for h in hours:
                if h.hour == 0:
                    xticklabels.append(h.strftime("%d-%b"))
                elif h.hour in [6, 12, 18]:
                    xticklabels.append(h.strftime("%H:%M"))
                else:
                    xticklabels.append("")
            ax2.set_xticklabels(xticklabels, rotation=45)
            ax2.tick_params(axis='x', pad=1)
            ax2.set_xlabel("Timestep", fontsize=9, labelpad=5)

            fig.suptitle(
                f"Generation v. Demand – {season_english_dict[selected_season]}",
                fontsize=13,
                fontweight="bold",
                y=0.98
            )

            fig.subplots_adjust(
                left=0.07,
                right=0.98,
                top=0.88,
                bottom=0.10
            )

            file_path = os.path.join(
                optimization_graph_output_folder,
                f'generation_demand_profiles_{num_households}_households_{selected_region}_{selected_season}.png'
            )
            fig.savefig(file_path, dpi=300, bbox_inches="tight", pad_inches=0.1)
            plt.close(fig)




        # ------------------------------------------------
        #       Sec 4.12 -- 6) SECOND PASS: reprint DURATION-CURVE CHARTS for MV-LV grid
        #       USING ymax_net_bt_global FOR ALL SEASONS
        # ------------------------------------------------
        for selected_season, pack in durcurve_cache.items():
            lv_net_duration_df = pack["df_dur_net_bt"]
            s_net_mv_avg = pack["s_net_mv_avg"]
            s_import_mv_avg = pack.get("s_import_mv_avg")

            # build overlays, include import if available
            overlays = {}
            overlays_style = {}

            if s_net_mv_avg is not None:
                overlays["MV grid - Net average family withdraw"] = s_net_mv_avg
                overlays_style["MV grid - Net average family withdraw"] = {
                    "color": "tab:orange",
                    "linestyle": "--",
                    "linewidth": 2.0,
                    "zorder": 12,
                }

            if s_import_mv_avg is not None:
                overlays["LV grid - Gross average family withdraw"] = s_import_mv_avg
                overlays_style["LV grid - Gross average family withdraw"] = {
                    "color": "tab:green",
                    # "linestyle": "-.",
                    "linewidth": 1.2,
                    "zorder": 13,
                }



            out_png_pct_net = os.path.join(
                optimization_graph_output_folder,
                f"{num_households}_nuc_{selected_region}_durcurve_netBT_PCT_with_MVavg_{selected_season}.png"
            )

            plot_duration_curve_percentile_bands_with_overlays(
                duration_df=lv_net_duration_df,
                percentiles=(5,10,20,30,40,50,60,70,80,90,95),
                title=f"Duration curve percentiles \nLV net withdrawal (import-export)\n + MV avg net withdrawal (Season: {season_english_dict[selected_season]})",
                xlabel="Sorted timesteps",
                ylabel="Power [kW]",
                overlays=overlays,
                overlays_style=overlays_style,
                cmap_name="Blues",
                ylim=(ymin_net_lv_global, ymax_net_lv_global),     
                savepath=out_png_pct_net
            )


        # ------------------------------------------------
        #       Sec 4.12 -- 6.b) SECOND PASS: netBT "ALL houses"
        # ------------------------------------------------
        dt_hours = pack.get("dt_hours", None)

        # if dt_hours is missing, retrieve it safely, fallback: 10 minutes
        if dt_hours is None:
            dt_hours = 10.0 / 60.0

        out_png_all = os.path.join(
            optimization_graph_output_folder,
            f"{num_households}_nuc_{selected_region}_durcurve_netBT_{selected_season}.png"
        )

        # duration curve for all households, one curve per column, with uniform Y-axis across seasons
        plot_duration_curves(
            duration_df=lv_net_duration_df,
            dt_hours=dt_hours,
            season=selected_season,
            out_png=out_png_all,
            ylim=(ymin_net_lv_global, ymax_net_lv_global),
        )




        # ------------------------------------------------
        #       Sec 4.12 -- 7) SECOND PASS: reprint DURATION-CURVE CHARTS for MV grid only
        # ------------------------------------------------
        for selected_season, pack in durcurve_mt_cache.items():
            dur_mt = pack["dur_mt"]

            out_png_mt = os.path.join(
                optimization_graph_output_folder,
                f"{num_households}_nuc_{selected_region}_durcurve_netMT_{selected_season}.png"
            )

            plt.figure(figsize=(8, 4.5))
            plt.plot(
                np.arange(len(dur_mt)),
                dur_mt.values,
                linewidth=2.2,
                color="tab:blue",
                label="Net MV withdrawal (district)"
            )

            plt.axhline(0, color="k", linewidth=0.8, alpha=0.6)
            plt.ylim(ymin_mt_global, ymax_mt_global)   # <<< HERE

            plt.title(
                f"Duration curve – Net withdrawal from MV grid\n"
                f"(District total, Season: {season_english_dict[selected_season]})"
            )
            plt.xlabel("Sorted timesteps")
            plt.ylabel("Power [kW]")
            plt.grid(True, alpha=0.3)
            plt.legend()

            plt.savefig(out_png_mt, dpi=300, bbox_inches="tight")
            plt.close()


        # ------------------------------------------------
        # Sec 4.12 -- 8) SECOND PASS: reprint LV IMPORT, percentiles, with uniform Y-axis
        # ------------------------------------------------
        for selected_season, pack in lv_import_duration_curve_cache.items():
            lv_import_duration_df = pack.get("df_dur_import_bt")
            dt_hours = pack.get("dt_hours")
            if (lv_import_duration_df is None) or lv_import_duration_df.empty:
                continue

            out_png_pct = os.path.join(
                optimization_graph_output_folder,
                f"{num_households}_nuc_{selected_region}_durcurve_importBT_PCT_{selected_season}.png"
            )

            plot_duration_curve_percentile_bands(
                duration_df=lv_import_duration_df,
                dt_hours=dt_hours,
                season=selected_season,
                out_png=out_png_pct,
                x_mode="rank",
                cmap_name="Purples",
                ylim=(0.0, ymax_import_lv_global),
            )


        # ------------------------------------------------
        # Sec 4.12 -- 9) SECOND PASS: reprint LV line congestion, Kerber, with uniform Y axes
        # ------------------------------------------------
        for selected_season, pack in (durcurve_lv_lines_cache.items() if plt_load_duration_LV_grid_lines else []):

            df_dur_net_lines = pack.get("df_dur_net_lines")
            df_dur_sat_lines = pack.get("df_dur_sat_lines")
            cols_branch = pack.get("cols_branch", [])
            cols_backbone = pack.get("cols_backbone", [])
            pct = pack.get("pct", (10,20,30,40,50,60,70,80,90))

            if (df_dur_net_lines is None) or df_dur_net_lines.empty:
                continue

            # 9.1) SAT branch
            if (df_dur_sat_lines is not None) and (not df_dur_sat_lines.empty) and cols_branch:
                out_png_sat_branch = os.path.join(
                    optimization_graph_output_folder,
                    f"{num_households}_nuc_{selected_region}_durcurve_SAT_lines_BRANCH_{selected_season}.png"
                )
                plot_duration_curve_percentile_bands_with_overlays(
                    duration_df=df_dur_sat_lines[cols_branch],
                    percentiles=pct,
                    title=f"LV lines saturation – branches to households ({season_english_dict[selected_season]})",
                    xlabel="Sorted timesteps",
                    ylabel="Line saturation (P/P_max) [%]",
                    overlays=None,
                    cmap_name="Oranges",
                    ylim=(0.0, ymax_sat_branch_global),
                    savepath=out_png_sat_branch,
                )

            # 9.2) SAT backbone
            if (df_dur_sat_lines is not None) and (not df_dur_sat_lines.empty) and cols_backbone:
                out_png_sat_backbone = os.path.join(
                    optimization_graph_output_folder,
                    f"{num_households}_nuc_{selected_region}_durcurve_SAT_lines_BACKBONE_{selected_season}.png"
                )
                plot_duration_curve_percentile_bands_with_overlays(
                    duration_df=df_dur_sat_lines[cols_backbone],
                    percentiles=pct,
                    title=f"LV lines saturation – backbone segments ({season_english_dict[selected_season]})",
                    xlabel="Sorted timesteps",
                    ylabel="Line saturation (P/P_max) [%]",
                    overlays=None,
                    cmap_name="Reds",
                    ylim=(0.0, ymax_sat_backbone_global),
                    savepath=out_png_sat_backbone,
                )

            # 9.3) NET line percentiles
            out_png_lines_pct = os.path.join(
                optimization_graph_output_folder,
                f"{num_households}_nuc_{selected_region}_durcurve_NET_grid_lines_pct_{selected_season}.png"
            )

            plot_duration_curve_percentile_bands_with_overlays(
                duration_df=df_dur_net_lines,
                percentiles=(10,20,30,40,50,60,70,80,90),
                title=f"LV network – net flows on LV line segments (both branch and backbone) ({season_english_dict[selected_season]})",
                xlabel="Sorted timesteps",
                ylabel="|Net line flow| [kW]" if USE_ABS_FOR_LINES_DURCURVE else "Net line flow [kW]",
                overlays=None,
                cmap_name="Greys",
                ylim=(ymin_net_lines_global, ymax_net_lines_global),
                savepath=out_png_lines_pct,
            )

            # 9.4) NET lines ALL curves
            out_png_lines_all = os.path.join(
                optimization_graph_output_folder,
                f"{num_households}_nuc_{selected_region}_durcurve_NET_grid_lines_ALL_{selected_season}.png"
            )

            fig, ax = plt.subplots(figsize=(10, 5.2), dpi=160)
            x = np.arange(len(df_dur_net_lines))

            for c in df_dur_net_lines.columns:
                ax.plot(x, df_dur_net_lines[c].values, linewidth=0.7, alpha=0.12)

            ax.plot(x, df_dur_net_lines.median(axis=1).values, linewidth=1.4, label="Median across segments")

            ax.set_title(f"LV network – duration curves of ALL line segments ({season_english_dict[selected_season]})")
            ax.set_xlabel("Sorted timesteps")
            ax.set_ylabel("|Net line flow| [kW]" if USE_ABS_FOR_LINES_DURCURVE else "Net line flow [kW]")
            ax.set_ylim(ymin_net_lines_global, ymax_net_lines_global)
            ax.grid(True, alpha=0.25)
            ax.legend(loc="best", fontsize=9)

            fig.tight_layout()
            fig.savefig(out_png_lines_all, dpi=250, bbox_inches="tight")
            plt.close(fig)

   ## 4.13 Multi-simulation comparison charts

In [ ]:
# ------------------------------------------------------------
# Sec 4.13 -- Multi-run comparison Case 0-7
# ------------------------------------------------------------
# Ordered workflow:
#
#   PRE-CHECK   verify the exact set of case selected
#
#   PASS 1     load/rebuild df_plot for each season
#              + optionally save audit/diagnostic tables
#   PASS 1.bis calculate common Y limits across seasons
#   PASS 2     print/plot the requested charts
#
# The safety analyses from the former commented block are kept as optional:
#   - MULTIRUN_EXPORT_MARKET_TS
#   - MULTIRUN_SAVE_ADDITIONAL_COST_AUDIT
#   - MULTIRUN_SAVE_SELF_SUFFICIENCY_TABLE
#   - MULTIRUN_RUN_SELF_SUFFICIENCY_AUDIT
#   - MULTIRUN_PLOT_SYSTEM_COST_DURATION_CURVES



if RUN_COMPARE_MULTIRUN:

    # ============================================================
    # PRE-CHECK FOR SEC. 4.13
    # Verify the exact set of paper cases selected from the manifest
    # ============================================================

    EXPECTED_MULTIRUN_CASES = [
        "Base case",
        "Case 1",
        "Case 2",
        "Case 3",
        "Case 4",
        "Case 5",
        "Case 6",
        "Case 7",
    ]

    _multirun_preview_filters = dict(
        MULTIRUN_MANIFEST_FILTERS or {}
    )

    _multirun_preview_filters.setdefault(
        "run_revision",
        [MULTIRUN_RUN_REVISION],
    )

    df_multirun_preview = select_hagrid_manifest_runs(
        experiment_id=MULTIRUN_EXPERIMENT_ID,
        filters=_multirun_preview_filters,
        allowed_statuses=("completed",),
    )

    preview_cols = [
        c for c in [
            "case_label",
            "run_label",
            "mode",
            "run_revision",
            "status",
            "run_uid",
        ]
        if c in df_multirun_preview.columns
    ]

    print("\n" + "=" * 90)
    print("[MULTIRUN PRE-CHECK] Runs selected for Sec. 4.13")
    print("=" * 90)

    print(
        df_multirun_preview[
            preview_cols
        ].to_string(index=False)
    )

    actual_multirun_cases = (
        df_multirun_preview["case_label"]
        .astype(str)
        .tolist()
    )

    missing_multirun_cases = [
        case
        for case in EXPECTED_MULTIRUN_CASES
        if case not in actual_multirun_cases
    ]

    if missing_multirun_cases:
        raise ValueError(
            "[MULTIRUN PRE-CHECK] Missing paper cases:\n"
            f"{missing_multirun_cases}\n\n"
            f"Selected cases:\n{actual_multirun_cases}\n\n"
            f"MULTIRUN_RUN_REVISION = {MULTIRUN_RUN_REVISION}"
        )

    print(
        "\n[OK][MULTIRUN PRE-CHECK] "
        "All 8 paper cases are available."
    )



    if len(MULTIRUN_GRAPHS_TO_PLOT) == 0:
        print("[INFO][4.13] RUN_COMPARE_MULTIRUN=True but no chart requested.")
    else:
        print("\n" + "=" * 100)
        print("[INFO][4.13] Starting multi-run comparison")
        print(f"  Requested charts: {MULTIRUN_GRAPHS_TO_PLOT}")
        print(f"  Use cache if complete: {MULTIRUN_USE_CACHE_IF_COMPLETE}")
        print(f"  Rebuild from pickle if needed: {MULTIRUN_REBUILD_FROM_PICKLE_IF_NEEDED}")
        print("  Optional outputs:")
        print(f"    market TS/KPI:          {MULTIRUN_EXPORT_MARKET_TS}")
        print(f"    additional cost audit:  {MULTIRUN_SAVE_ADDITIONAL_COST_AUDIT}")
        print(f"    self-suff. table:       {MULTIRUN_SAVE_SELF_SUFFICIENCY_TABLE}")
        print(f"    self-suff. audit:       {MULTIRUN_RUN_SELF_SUFFICIENCY_AUDIT}")
        print(f"    cost duration curves:   {MULTIRUN_PLOT_SYSTEM_COST_DURATION_CURVES}")
        print("=" * 100 + "\n")

    MULTIRUN_SEASON_CACHE = {}


    # ------------------------------------------------------------
    # Experiment/revision-specific output directory
    # ------------------------------------------------------------

    if MULTIRUN_OUTPUT_TAG_OVERRIDE is not None:
        MULTIRUN_SELECTION_TAG = _safe_fs_tag(
            MULTIRUN_OUTPUT_TAG_OVERRIDE
        )
    else:
        MULTIRUN_SELECTION_TAG = (
            _manifest_filters_folder_tag(
                MULTIRUN_MANIFEST_FILTERS
            )
        )

    comparison_output_folder = (
        build_compare_output_dir(
            experiment_id=(
                MULTIRUN_EXPERIMENT_ID
            ),
            run_revision=(
                MULTIRUN_RUN_REVISION
            ),
            comparison_type="multirun",
            comparison_tag=(
                MULTIRUN_SELECTION_TAG
            ),
        )
    )

    print(
        "[INFO][MULTIRUN] Output directory:\n"
        f"  {comparison_output_folder}"
    )



    # ============================================================
    # PASS 1: load or rebuild data for all seasons
    # ============================================================
    for selected_season in seasons:

        print("\n" + "-" * 100)
        print(f"[INFO][MULTIRUN][PASS 1] Season: {selected_season}")
        print("-" * 100)

        # Season-specific tag retained in filenames to avoid collisions.
        compare_tag = _safe_fs_tag(
            selected_season
        )

        out_cost_xlsx = _safe_output_path(folder=comparison_output_folder,
            descriptive_filename=(f'cost_percentiles_vs_avg_household_weekly_system_cost_eur_{compare_tag}.xlsx'),
            compact_filename=(f'multirun_summary_{selected_season}.xlsx'))

        MULTIRUN_COST_SCENARIOS = build_multirun_cost_scenarios_for_season(
            selected_season
        )

        try:
            df_multirun_cost_plot, df_multirun_cost_skipped, data_source_used = (
                _get_or_build_multirun_plot_data(
                    cache_xlsx=out_cost_xlsx,
                    scenarios=MULTIRUN_COST_SCENARIOS,
                    graphs_to_plot=MULTIRUN_GRAPHS_TO_PLOT,
                    output_folder=output_folder,
                    num_households=num_households,
                    selected_region=selected_region,
                    PV_share=pv_share_to_compare,
                    PV_power_kwp=PV_power_kwp,
                    preferred_total_cost_col=TOTAL_COST_COL_FOR_MULTIRUN_PLOT,
                    use_cache_if_complete=MULTIRUN_USE_CACHE_IF_COMPLETE,
                    rebuild_from_pickle_if_needed=MULTIRUN_REBUILD_FROM_PICKLE_IF_NEEDED,
                    save_updated_cache=MULTIRUN_SAVE_UPDATED_CACHE,
                )
            )

        except Exception as e:
            print(f'[ERROR][MULTIRUN] Could not build data for {selected_season}.')
            print(f'{type(e).__name__}: {e}')
            print(f'[DEBUG][MULTIRUN] Cache path: {os.path.abspath(out_cost_xlsx)}')
            print(f'[DEBUG][MULTIRUN] Cache path length: {len(os.path.abspath(out_cost_xlsx))} characters')
            print(f'[DEBUG][MULTIRUN] Parent directory exists: {os.path.isdir(os.path.dirname(os.path.abspath(out_cost_xlsx)))}')

            continue

        if df_multirun_cost_plot is None or df_multirun_cost_plot.empty:
            print(f"[WARN][MULTIRUN] No valid data for {selected_season}. Skip.")
            continue



        # ------------------------------------------------------------
        # PASS 1.1: household-level costs and paired differences
        # ------------------------------------------------------------
        df_multirun_delta_plot = pd.DataFrame()
        df_household_cost_detail = pd.DataFrame()
        df_household_costs_wide = pd.DataFrame()
        df_household_deltas_wide = pd.DataFrame()
        df_household_delta_skipped = pd.DataFrame()

        need_household_delta_data = (
            RUN_PLOT_MULTIRUN_COST_DELTAS
            or RUN_SAVE_MULTIRUN_HOUSEHOLD_COST_DETAILS
        )

        if need_household_delta_data:

            try:
                (
                    df_multirun_delta_plot,
                    df_household_cost_detail,
                    df_household_costs_wide,
                    df_household_deltas_wide,
                    df_household_delta_skipped,
                ) = build_multirun_household_cost_delta_summary(
                    scenarios=MULTIRUN_COST_SCENARIOS,
                    reference_case_label=MULTIRUN_DELTA_REFERENCE_CASE_LABEL,
                    pair_key=MULTIRUN_DELTA_PAIR_KEY,
                    zero_tolerance_eur=MULTIRUN_DELTA_ZERO_TOL_EUR,
                    skip_missing=True,
                )

                if RUN_SAVE_MULTIRUN_HOUSEHOLD_COST_DETAILS:

                    out_delta_xlsx = _safe_output_path(
                        folder=comparison_output_folder,

                        descriptive_filename=(
                            'household_cost_deltas_vs_'
                            f'{_safe_fs_tag(MULTIRUN_DELTA_REFERENCE_CASE_LABEL)}_'
                            f'{compare_tag}.xlsx'
                        ),

                        compact_filename=(
                            f'household_cost_deltas_{selected_season}.xlsx'
                        ),
                    )

                    with pd.ExcelWriter(
                        out_delta_xlsx,
                        engine="openpyxl",
                    ) as writer:

                        df_multirun_delta_plot.to_excel(
                            writer,
                            sheet_name="delta_percentiles",
                            index=False,
                        )

                        df_household_cost_detail.to_excel(
                            writer,
                            sheet_name="household_detail",
                            index=False,
                        )

                        df_household_costs_wide.to_excel(
                            writer,
                            sheet_name="costs_by_household",
                            index=False,
                        )

                        df_household_deltas_wide.to_excel(
                            writer,
                            sheet_name="deltas_by_household",
                            index=False,
                        )

                        if (
                            df_household_delta_skipped is not None
                            and not df_household_delta_skipped.empty
                        ):
                            df_household_delta_skipped.to_excel(
                                writer,
                                sheet_name="skipped",
                                index=False,
                            )

                    print(
                        "[OK][HOUSEHOLD DELTA] Saved household-level "
                        f"comparison: {out_delta_xlsx}"
                    )

            except Exception as error:
                print(
                    "[WARN][HOUSEHOLD DELTA] Could not build paired "
                    f"household comparison for {selected_season}: "
                    f"{type(error).__name__}: {error}"
                )



        # ------------------------------------------------------------
        # PASS 1.2: market cost time series and gross MV import/export KPI
        # ------------------------------------------------------------
        df_market_ts_wide = pd.DataFrame()
        df_market_summary = pd.DataFrame()
        df_market_skipped = pd.DataFrame()

        need_market_ts = (
            MULTIRUN_EXPORT_MARKET_TS
            or (
                MULTIRUN_PLOT_SYSTEM_COST_DURATION_CURVES
                and MULTIRUN_BUILD_MARKET_TS_FOR_DURATION_CURVES
            )
        )

        if need_market_ts:
            try:
                df_market_ts_wide, df_market_summary, df_market_skipped = (
                    build_multirun_market_cost_timeseries_dataset(
                        scenarios=MULTIRUN_COST_SCENARIOS,
                        PV_share=pv_share_to_compare,
                        PV_power_kwp=PV_power_kwp,
                        columns_to_export=MULTIRUN_MARKET_TS_COLUMNS,
                    )
                )

                # Bring some synthetic KPI also into df_multirun_cost_plot.
                # This is useful because the main table remains self-contained.
                if df_market_summary is not None and not df_market_summary.empty:
                    cols_market_merge = [
                        "x_order",
                        "gross_mv_import_share_of_demand",
                        "gross_mv_export_share_of_demand",
                        "gross_mv_exchange_share_of_demand",
                        "market_mt_purchase_cost_eur",
                        "market_mt_sale_revenues_eur",
                        "market_mt_net_cost_eur",
                        "average_market_mt_net_cost_eur_per_timestep",
                        "average_market_mt_net_cost_eur_per_kwh_demand",
                        "average_total_system_cost_eur_per_timestep",
                    ]

                    cols_market_merge = [
                        c for c in cols_market_merge
                        if c in df_market_summary.columns
                    ]

                    if len(cols_market_merge) > 1:
                        # Avoid duplicates if df_plot comes from an already enriched cache.
                        cols_to_drop = [
                            c for c in cols_market_merge
                            if c != "x_order" and c in df_multirun_cost_plot.columns
                        ]

                        if cols_to_drop:
                            df_multirun_cost_plot = df_multirun_cost_plot.drop(
                                columns=cols_to_drop
                            )

                        df_multirun_cost_plot = df_multirun_cost_plot.merge(
                            df_market_summary[cols_market_merge],
                            on="x_order",
                            how="left",
                        )

                if MULTIRUN_EXPORT_MARKET_TS:
                    out_market_xlsx = _safe_output_path(
                        folder=comparison_output_folder,

                        descriptive_filename=(
                            f'market_cost_timeseries_{compare_tag}.xlsx'
                        ),

                        compact_filename=(
                            f'market_cost_ts_{selected_season}.xlsx'
                        ),
                    )

                    with pd.ExcelWriter(out_market_xlsx, engine="openpyxl") as writer:
                        if df_market_ts_wide is not None and not df_market_ts_wide.empty:
                            df_market_ts_wide.to_excel(
                                writer,
                                sheet_name="market_ts_wide",
                                index=True,
                            )

                        if df_market_summary is not None and not df_market_summary.empty:
                            df_market_summary.to_excel(
                                writer,
                                sheet_name="market_summary",
                                index=False,
                            )

                        if df_market_skipped is not None and not df_market_skipped.empty:
                            df_market_skipped.to_excel(
                                writer,
                                sheet_name="skipped",
                                index=False,
                            )

                    print(f"[OK] Saved market cost time series/KPI: {out_market_xlsx}")

            except Exception as e:
                print(
                    f"[WARN][MULTIRUN][MARKET TS] Error during export/rebuild of "
                    f"market TS for {selected_season}: {type(e).__name__}: {e}"
                )

        # ------------------------------------------------------------
        # PASS 1.3: audit of additional-cost weight
        # ------------------------------------------------------------
        df_additional_cost_detail = pd.DataFrame()
        df_additional_cost_summary = pd.DataFrame()
        df_additional_cost_skipped = pd.DataFrame()

        if MULTIRUN_SAVE_ADDITIONAL_COST_AUDIT:
            try:
                (
                    df_additional_cost_detail,
                    df_additional_cost_summary,
                    df_additional_cost_skipped,
                ) = build_multirun_additional_cost_weight_audit(
                    scenarios=MULTIRUN_COST_SCENARIOS,
                    PV_share=pv_share_to_compare,
                    PV_power_kwp=PV_power_kwp,
                )

                out_additional_cost_xlsx = _safe_output_path(
                    folder=comparison_output_folder,
                    descriptive_filename=(
                        f'additional_cost_weight_audit_{compare_tag}.xlsx'),
                    compact_filename=(
                        f'additional_cost_audit_{selected_season}.xlsx'),
                )

                with pd.ExcelWriter(out_additional_cost_xlsx, engine="openpyxl") as writer:
                    if (
                        df_additional_cost_summary is not None
                        and not df_additional_cost_summary.empty
                    ):
                        df_additional_cost_summary.to_excel(
                            writer,
                            sheet_name="summary",
                            index=False,
                        )

                    if (
                        df_additional_cost_detail is not None
                        and not df_additional_cost_detail.empty
                    ):
                        df_additional_cost_detail.to_excel(
                            writer,
                            sheet_name="detail",
                            index=False,
                        )

                    if (
                        df_additional_cost_skipped is not None
                        and not df_additional_cost_skipped.empty
                    ):
                        df_additional_cost_skipped.to_excel(
                            writer,
                            sheet_name="skipped",
                            index=False,
                        )

                print(f"[OK] Saved additional-cost audit: {out_additional_cost_xlsx}")

            except Exception as e:
                print(
                    f"[WARN][MULTIRUN][ADDITIONAL COST AUDIT] Audit error "
                    f"for {selected_season}: {type(e).__name__}: {e}"
                )

        # ------------------------------------------------------------
        # PASS 1.4: save self-sufficiency / demand coverage table
        # ------------------------------------------------------------
        out_self_suff_xlsx = _safe_output_path(
            folder=comparison_output_folder,

            descriptive_filename=(
                f'self_sufficiency_{compare_tag}.xlsx'
            ),

            compact_filename=(
                f'self_sufficiency_{selected_season}.xlsx'
            ),
        )

        if MULTIRUN_SAVE_SELF_SUFFICIENCY_TABLE:
            try:
                with pd.ExcelWriter(out_self_suff_xlsx, engine="openpyxl") as writer:
                    df_multirun_cost_plot.to_excel(
                        writer,
                        sheet_name="plot_data",
                        index=False,
                    )

                    if (
                        df_multirun_cost_skipped is not None
                        and not df_multirun_cost_skipped.empty
                    ):
                        df_multirun_cost_skipped.to_excel(
                            writer,
                            sheet_name="skipped",
                            index=False,
                        )

                    if df_market_summary is not None and not df_market_summary.empty:
                        df_market_summary.to_excel(
                            writer,
                            sheet_name="market_summary",
                            index=False,
                        )

                    metadata_df = pd.DataFrame([{
                        "compare_tag": compare_tag,
                        "season": selected_season,
                        "data_source_used": data_source_used,
                        "market_ts_built": bool(
                            df_market_ts_wide is not None
                            and not df_market_ts_wide.empty
                        ),
                        "additional_cost_audit": bool(MULTIRUN_SAVE_ADDITIONAL_COST_AUDIT),
                        "self_sufficiency_audit": bool(MULTIRUN_RUN_SELF_SUFFICIENCY_AUDIT),
                    }])

                    metadata_df.to_excel(writer, sheet_name="metadata", index=False)

                print(f"[OK] Saved multi-run self-sufficiency table: {out_self_suff_xlsx}")

            except Exception as e:
                print(
                    f"[WARN][MULTIRUN][SELF SUFF TABLE] Saving error "
                    f"for {selected_season}: {type(e).__name__}: {e}"
                )

        # ------------------------------------------------------------
        # PASS 1.5: self-sufficiency audit corrected for Delta SoC
        # ------------------------------------------------------------
        df_self_suff_audit = pd.DataFrame()

        if MULTIRUN_RUN_SELF_SUFFICIENCY_AUDIT:
            try:
                out_self_suff_audit_xlsx = _safe_output_path(
                    folder=comparison_output_folder,

                    descriptive_filename=(
                        'self_sufficiency_soc_corrected_audit_'
                        f'{compare_tag}.xlsx'
                    ),

                    compact_filename=(
                        f'self_suff_soc_audit_{selected_season}.xlsx'
                    ),
                )

                df_self_suff_audit = print_multirun_self_sufficiency_audit(
                    df_plot=df_multirun_cost_plot,
                    out_xlsx=out_self_suff_audit_xlsx,
                    gap_warn_pct_demand=MULTIRUN_SELF_SUFF_GAP_WARN_PCT_DEMAND,
                    gap_warn_abs_kwh=MULTIRUN_SELF_SUFF_GAP_WARN_ABS_KWH,
                )

            except Exception as e:
                print(
                    f"[WARN][MULTIRUN][SELF SUFF AUDIT] Audit error "
                    f"for {selected_season}: {type(e).__name__}: {e}"
                )

        # ------------------------------------------------------------
        # PASS 1.6: save the main cache again if it has been enriched
        # ------------------------------------------------------------
        # _get_or_build_multirun_plot_data already saves the cache when it rebuilds.
        # However, here df_plot may have been enriched with market_summary.
        # Therefore, save it again, but only if requested.
        if MULTIRUN_SAVE_UPDATED_CACHE:
            try:
                with pd.ExcelWriter(out_cost_xlsx, engine="openpyxl") as writer:
                    df_multirun_cost_plot.to_excel(
                        writer,
                        sheet_name="plot_data",
                        index=False,
                    )

                    if (
                        df_multirun_cost_skipped is not None
                        and not df_multirun_cost_skipped.empty
                    ):
                        df_multirun_cost_skipped.to_excel(
                            writer,
                            sheet_name="skipped",
                            index=False,
                        )

                    if df_market_summary is not None and not df_market_summary.empty:
                        df_market_summary.to_excel(
                            writer,
                            sheet_name="market_summary",
                            index=False,
                        )

                    if (
                        df_additional_cost_summary is not None
                        and not df_additional_cost_summary.empty
                    ):
                        df_additional_cost_summary.to_excel(
                            writer,
                            sheet_name="additional_cost_summary",
                            index=False,
                        )

                    metadata_df = pd.DataFrame([{
                        "cache_created_from": data_source_used,
                        "cache_policy": "use_cache_if_complete_else_rebuild_pickle",
                        "graphs_to_plot": ", ".join(MULTIRUN_GRAPHS_TO_PLOT),
                        "preferred_total_cost_col": TOTAL_COST_COL_FOR_MULTIRUN_PLOT,
                        "compare_tag": compare_tag,
                        "season": selected_season,
                        "market_ts_exported": bool(MULTIRUN_EXPORT_MARKET_TS),
                        "market_ts_built_for_duration": bool(
                            MULTIRUN_PLOT_SYSTEM_COST_DURATION_CURVES
                            and MULTIRUN_BUILD_MARKET_TS_FOR_DURATION_CURVES
                        ),
                        "additional_cost_audit": bool(MULTIRUN_SAVE_ADDITIONAL_COST_AUDIT),
                        "self_sufficiency_table": bool(MULTIRUN_SAVE_SELF_SUFFICIENCY_TABLE),
                        "self_sufficiency_audit": bool(MULTIRUN_RUN_SELF_SUFFICIENCY_AUDIT),
                    }])

                    metadata_df.to_excel(writer, sheet_name="metadata", index=False)

                print(f"[OK][MULTIRUN] Saved main cache/table: {out_cost_xlsx}")

            except Exception as e:
                print(
                    f"[WARN][MULTIRUN][CACHE SAVE] Cache saving error "
                    f"for {selected_season}: {type(e).__name__}: {e}"
                )

        # ------------------------------------------------------------
        # PASS 1.7: save everything in the in-memory cache for PASS 2
        # ------------------------------------------------------------
        MULTIRUN_SEASON_CACHE[selected_season] = {
            "compare_tag": compare_tag,
            "comparison_output_folder": comparison_output_folder,

            "df_plot": df_multirun_cost_plot,
            "df_skipped": df_multirun_cost_skipped,

            "df_market_ts_wide": df_market_ts_wide,
            "df_market_summary": df_market_summary,
            "df_market_skipped": df_market_skipped,
            "df_additional_cost_summary": df_additional_cost_summary,
            "df_self_suff_audit": df_self_suff_audit,
            "data_source_used": data_source_used,

            "df_delta_plot": df_multirun_delta_plot,
            "df_household_cost_detail": df_household_cost_detail,
            "df_household_costs_wide": df_household_costs_wide,
            "df_household_deltas_wide": df_household_deltas_wide,
            "df_household_delta_skipped": df_household_delta_skipped,

        }

    # ============================================================
    # PASS 1.bis: calculate common limits, only if requested
    # ============================================================
    if MULTIRUN_USE_COMMON_YLIMS and len(MULTIRUN_SEASON_CACHE) > 0:
        df_plot_by_season = {
            season: item["df_plot"]
            for season, item in MULTIRUN_SEASON_CACHE.items()
            if item.get("df_plot") is not None and not item["df_plot"].empty
        }

        common_y_limits = _compute_multirun_common_y_limits(df_plot_by_season)

        df_delta_by_season = {
            season: item.get("df_delta_plot")
            for season, item in MULTIRUN_SEASON_CACHE.items()
            if (
                item.get("df_delta_plot") is not None
                and not item["df_delta_plot"].empty
            )
        }

        common_delta_ylim = _compute_multirun_delta_common_ylim(
            df_delta_by_season=df_delta_by_season,
            symmetric=MULTIRUN_DELTA_SYMMETRIC_YLIM,
        )


        print("\n[INFO][MULTIRUN] Common Y-axis limits calculated:")
        for k, v in common_y_limits.items():
            print(f"  {k}: {v}")

    else:
        common_y_limits = {
            "cost_y1_ylim": None,
            "cost_y2_ylim": None,
            "demand_y1_ylim": None,
            "demand_y2_ylim": None,
        }

        common_delta_ylim = None




    # ============================================================
    # PASS 2: print/plot only the requested charts
    # ============================================================
    for selected_season, item in MULTIRUN_SEASON_CACHE.items():

        print("\n" + "-" * 100)
        print(f"[INFO][MULTIRUN][PASS 2] Plot season: {selected_season}")
        print(f"  Data source: {item.get('data_source_used')}")
        print("-" * 100)

        compare_tag = item["compare_tag"]
        comparison_output_folder = item["comparison_output_folder"]
        df_multirun_cost_plot = item["df_plot"]

        if df_multirun_cost_plot is None or df_multirun_cost_plot.empty:
            print(f"[WARN][MULTIRUN] Empty df_plot for {selected_season}. Skip.")
            continue

        # ------------------------------------------------------------
        # PASS 2.1: demand coverage chart
        # ------------------------------------------------------------
        if RUN_PLOT_MULTIRUN_DEMAND_COVERAGE:

            out_self_suff_demand_png = _safe_output_path(
                folder=comparison_output_folder,

                descriptive_filename=(
                    f'self_sufficiency_total_cost_{compare_tag}.png'
                ),

                compact_filename=(
                    f'demand_coverage_{selected_season}.png'
                ),
            )

            plot_multirun_self_sufficiency_and_demand_components(
                df_plot=df_multirun_cost_plot,
                title=(
                    "Demand coverage and demand components by scenario\n"
                    f"Season: {season_english_dict[selected_season]}"
                ),
                out_png=out_self_suff_demand_png,
                rotate_xticks=0,
                y1_label="Energy [kWh/week]",
                y2_label="Effective system operating cost [EUR/week]",
                demand_tolerance_abs_kwh=10.0,
                demand_tolerance_rel=5e-4,
                demand_mismatch_action=MULTIRUN_DEMAND_MISMATCH_ACTION,
                print_checks=MULTIRUN_PRINT_DEMAND_COVERAGE_CHECKS,
                demand_reference_if_within_tolerance="max",
                y1_ylim=common_y_limits.get("demand_y1_ylim"),
                y2_ylim=common_y_limits.get("demand_y2_ylim"),
                legend_mode=globals().get(
                    "MULTIRUN_DEMAND_LEGEND_MODE",
                    "right_outside",
                ),
                show_title=SHOW_MULTIRUN_FIGURE_TITLE,
                save_pdf=SAVE_JOURNAL_PDF,
            )

            print(f"[OK] Saved demand coverage chart: {out_self_suff_demand_png}")

        # ------------------------------------------------------------
        # PASS 2.2: cost percentiles + household average system cost chart
        # ------------------------------------------------------------
        if RUN_PLOT_MULTIRUN_COST_PERCENTILES:

            out_multirun_png = _safe_output_path(
                folder=comparison_output_folder,

                descriptive_filename=(
                    'cost_percentiles_vs_'
                    'avg_household_weekly_system_cost_eur_'
                    f'{compare_tag}.png'
                ),

                compact_filename=(
                    f'cost_percentiles_{selected_season}.png'
                ),
            )

            plot_multirun_cost_percentiles_vs_household_avg_cost(
                df_plot=df_multirun_cost_plot,
                title=(
                    "Household cost percentiles by scenario + "
                    "mean weekly system cost per household\n"
                    f"Season: {season_english_dict[selected_season]}"
                ),
                out_png=out_multirun_png,
                y1_label="Effective household weekly cost\npercentiles [EUR]",
                y2_label="Mean effective weekly system cost\nper household [EUR]",
                y1_ylim=common_y_limits.get("cost_y1_ylim"),
                y2_ylim=common_y_limits.get("cost_y2_ylim"),
                legend_mode=globals().get(
                    "MULTIRUN_COST_LEGEND_MODE",
                    "upper_right_inside",
                ),
                show_title=SHOW_MULTIRUN_FIGURE_TITLE,
                save_pdf=SAVE_JOURNAL_PDF,
            )

            print(f"[OK] Saved cost percentiles chart: {out_multirun_png}")



        # ------------------------------------------------------------
        # PASS 2.3: paired household-cost differences vs base case chart
        # ------------------------------------------------------------
        if RUN_PLOT_MULTIRUN_COST_DELTAS:

            df_multirun_delta_plot = item.get(
                "df_delta_plot",
                pd.DataFrame(),
            )

            if (
                df_multirun_delta_plot is None
                or df_multirun_delta_plot.empty
            ):
                print(
                    f"[WARN][HOUSEHOLD DELTA] No paired-delta data "
                    f"for {selected_season}. Skip chart."
                )

            else:
                out_delta_png = _safe_output_path(
                    folder=comparison_output_folder,

                    descriptive_filename=(
                        'household_cost_delta_percentiles_vs_'
                        f'{_safe_fs_tag(MULTIRUN_DELTA_REFERENCE_CASE_LABEL)}_'
                        f'{compare_tag}.png'
                    ),

                    compact_filename=(
                        f'cost_deltas_{selected_season}.png'
                    ),
                )

                plot_multirun_household_cost_delta_percentiles(
                    df_delta_plot=df_multirun_delta_plot,
                    title=(
                        "Paired household-cost differences relative "
                        "to the base configuration\n"
                        f"Season: {season_english_dict[selected_season]}"
                    ),
                    out_png=out_delta_png,
                    y1_label=(
                        "Effective household weekly cost difference\n"
                        "vs base case [EUR]"
                    ),
                    y2_label="Penalized households [%]",
                    y1_ylim=common_delta_ylim,
                    legend_mode=globals().get(
                        "MULTIRUN_COST_LEGEND_MODE",
                        "upper_right_inside",
                    ),
                    show_title=SHOW_MULTIRUN_FIGURE_TITLE,
                    save_pdf=SAVE_JOURNAL_PDF,
                )









        # ------------------------------------------------------------
        # PASS 2.4: duration curves of cost per timestep
        # ------------------------------------------------------------
        if MULTIRUN_PLOT_SYSTEM_COST_DURATION_CURVES:

            df_market_ts_wide = item.get("df_market_ts_wide", pd.DataFrame())

            if df_market_ts_wide is None or df_market_ts_wide.empty:
                print(
                    "[WARN][DURATION CURVE] Empty df_market_ts_wide. "
                    "Enable MULTIRUN_EXPORT_MARKET_TS=True or "
                    "MULTIRUN_BUILD_MARKET_TS_FOR_DURATION_CURVES=True."
                )

            else:
                duration_metric = globals().get(
                    "MULTIRUN_DURATION_CURVE_COST_COLUMN",
                    "total_system_cost_per_timestep_eur",
                )

                out_duration_png = _safe_output_path(
                    folder=comparison_output_folder,

                    descriptive_filename=(
                        f'duration_curve_{duration_metric}_'
                        f'{compare_tag}.png'
                    ),

                    compact_filename=(
                        f'duration_curve_{selected_season}.png'
                    ),
                )

                out_duration_xlsx = _safe_output_path(
                    folder=comparison_output_folder,

                    descriptive_filename=(
                        f'duration_curve_{duration_metric}_'
                        f'{compare_tag}.xlsx'
                    ),

                    compact_filename=(
                        f'duration_curve_{selected_season}.xlsx'
                    ),
                )

                df_duration_curve = plot_multirun_system_cost_duration_curves(
                    df_market_ts_wide=df_market_ts_wide,
                    title=(
                        "Duration curves of system cost per timestep by scenario\n"
                        f"Season: {season_english_dict[selected_season]}"
                    ),
                    metric_col=duration_metric,
                    out_png=out_duration_png,
                    out_xlsx=out_duration_xlsx,
                    legend_mode=globals().get(
                        "MULTIRUN_COST_LEGEND_MODE",
                        "upper_right_inside",
                    ),
                )

                print(f"[OK] Saved system-cost duration curve: {out_duration_png}")

else:
    print("[INFO][4.13] RUN_COMPARE_MULTIRUN=False. Skipping multi-run comparison.")




# ============================================================
# Sec 4.13.1 -- Economic savings table from saved runs
# ============================================================

if RUN_BUILD_ECONOMIC_SAVINGS_TABLE:

    print(
        "\n"
        + "=" * 100
    )

    print(
        "[ECON SAVINGS] "
        "Reconstructing savings from saved optimization outputs"
    )

    print(
        "=" * 100
    )


    ECON_SAVINGS_TABLES = (
        build_economic_savings_from_saved_runs(
            seasons_to_use=list(
                seasons_dict.values()
            ),

            reference_case_label=(
                ECON_SAVINGS_REFERENCE_CASE
            ),

            annualization_factor=(
                ECON_SAVINGS_WEEKS_PER_SEASON
            ),

            percentile_levels=(
                ECON_SAVINGS_PERCENTILES
            ),

            preferred_total_cost_col=(
                ECON_SAVINGS_SYSTEM_COST_COLUMN
            ),

            strict_household_pairing=(
                ECON_SAVINGS_STRICT_HOUSEHOLD_PAIRING
            ),
        )
    )


    experiment_revision_output_dir = (
        build_experiment_revision_output_dir(
            experiment_id=(
                MULTIRUN_EXPERIMENT_ID
            ),
            run_revision=(
                MULTIRUN_RUN_REVISION
            ),
        )
    )

    os.makedirs(
        experiment_revision_output_dir,
        exist_ok=True,
    )


    # --------------------------------------------------------
    # Standalone detailed workbook
    # --------------------------------------------------------

    economic_savings_path = os.path.join(
        experiment_revision_output_dir,
        "economic_savings_summary.xlsx",
    )

    write_economic_savings_tables(
        output_path=economic_savings_path,
        tables=ECON_SAVINGS_TABLES,
        append=False,
    )

    print(
        "[OK][ECON SAVINGS] Saved:\n"
        f"  {economic_savings_path}"
    )


    # --------------------------------------------------------
    # Also append to the experiment-level economic_summary.xlsx
    # --------------------------------------------------------

    if (
        ECON_SAVINGS_APPEND_TO_EXPERIMENT_SUMMARY
    ):

        experiment_summary_path = (
            os.path.join(
                experiment_revision_output_dir,
                "economic_summary.xlsx",
            )
        )

        write_economic_savings_tables(
            output_path=(
                experiment_summary_path
            ),
            tables=(
                ECON_SAVINGS_TABLES
            ),
            append=True,
        )

        print(
            "[OK][ECON SAVINGS] "
            "Savings sheets appended/replaced in:\n"
            f"  {experiment_summary_path}"
        )


    print(
        "\n[PAPER TABLE]"
    )

    print(
        ECON_SAVINGS_TABLES[
            "paper_table"
        ].to_string()
    )

## 4.14 Plot comparison charts between 2 configurations

In [ ]:
if RUN_COMPARE_DHW_SOC_TWO_CONFIGS:
    print("\n[SOC COMPARE] Source experiment:")
    print(f"  experiment = {SOC_COMPARE_EXPERIMENT_ID}")
    print(f"  revision   = {SOC_COMPARE_RUN_REVISION}")


    # --------------------------------------------------------
    # Resolve physical/electrification scenario for SOC compare
    # --------------------------------------------------------

    if SOC_COMPARE_ELECTRIFICATION_ID_OVERRIDE is not None:

        SOC_COMPARE_ELECTRIFICATION_ID = str(
            SOC_COMPARE_ELECTRIFICATION_ID_OVERRIDE
        )

    else:

        _soc_active_electrification_ids = list(
            ACTIVE_ELECTRIFICATION_SCENARIO_IDS
        )

        if len(_soc_active_electrification_ids) != 1:

            raise ValueError(
                "[SOC COMPARE] More than one physical/electrification "
                "scenario is active.\n"
                f"Available active IDs: "
                f"{_soc_active_electrification_ids}\n\n"
                "Set explicitly, for example:\n"
                'SOC_COMPARE_ELECTRIFICATION_ID_OVERRIDE = "HP020"'
            )

        SOC_COMPARE_ELECTRIFICATION_ID = (
            _soc_active_electrification_ids[0]
        )


    SOC_COMPARE_SELECTOR_A_RESOLVED = {
        **SOC_COMPARE_SELECTOR_A,
        "electrification_id":
            SOC_COMPARE_ELECTRIFICATION_ID,
    }

    SOC_COMPARE_SELECTOR_B_RESOLVED = {
        **SOC_COMPARE_SELECTOR_B,
        "electrification_id":
            SOC_COMPARE_ELECTRIFICATION_ID,
    }


    print(
        "[SOC COMPARE] Physical scenario = "
        f"{SOC_COMPARE_ELECTRIFICATION_ID}"
    )



    RUN_SOC_A = resolve_unique_manifest_run(
        experiment_id=SOC_COMPARE_EXPERIMENT_ID,
        selector=SOC_COMPARE_SELECTOR_A_RESOLVED,
        run_revision=SOC_COMPARE_RUN_REVISION,
    )

    RUN_SOC_B = resolve_unique_manifest_run(
        experiment_id=SOC_COMPARE_EXPERIMENT_ID,
        selector=SOC_COMPARE_SELECTOR_B_RESOLVED,
        run_revision=SOC_COMPARE_RUN_REVISION,
    )


    soc_comparison_output_folder = (
        _soc_compare_output_dir(
            experiment_id=(
                SOC_COMPARE_EXPERIMENT_ID
            ),
            run_revision=(
                SOC_COMPARE_RUN_REVISION
            ),
            run_a=RUN_SOC_A,
            run_b=RUN_SOC_B,
        )
    )

    print(
        "[INFO][SOC COMPARE] Output directory:\n"
        f"  {soc_comparison_output_folder}"
    )

    soc_compare_seasons = SOC_COMPARE_SEASONS

    if soc_compare_seasons is None:
        soc_compare_seasons = seasons
    else:
        soc_compare_seasons = [
            s
            for s in soc_compare_seasons
            if s in seasons
        ]

    # --------------------------------------------------------
    # Same two runs, repeated over the requested seasons.
    # --------------------------------------------------------

    for selected_season in soc_compare_seasons:

        print("\n" + "=" * 100)
        print(
            "[SOC COMPARE] DHW - "
            f"{RUN_SOC_A['run_label']} vs "
            f"{RUN_SOC_B['run_label']} - "
            f"season: {selected_season}"
        )
        print("=" * 100)

        # DHW SOC + PV
        plot_compare_dhw_soc_two_configs_with_secondary(
            season=selected_season,
            run_a=RUN_SOC_A,
            run_b=RUN_SOC_B,
            secondary_kind="pv",
            out_dir=soc_comparison_output_folder,
            pv_source=SOC_COMPARE_PV_SOURCE,
            single_house=SOC_COMPARE_SINGLE_PV_HOUSE,
            save_xlsx=False,
            show_title=SHOW_SOC_COMPARE_TITLE,
            save_pdf=SAVE_JOURNAL_PDF,
        )

        # DHW SOC + purchase price
        plot_compare_dhw_soc_two_configs_with_secondary(
            season=selected_season,
            run_a=RUN_SOC_A,
            run_b=RUN_SOC_B,
            secondary_kind="price",
            out_dir=soc_comparison_output_folder,
            save_xlsx=False,
            show_title=SHOW_SOC_COMPARE_TITLE,
            save_pdf=SAVE_JOURNAL_PDF,
        )

 ## 4.15 Debug infeasibility

In [ ]:
# -----------------------------------------------------------------------------
#       Sec 4.15 -- 1) -- INFEASIBLE RUN ANALYSIS (LP / IIS) — activated with RUN_DEBUG_INFEASIBLE
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Sec 4.15 -- 1) SECOND-STAGE DIAGNOSTIC OF A FAILED RUN
# -----------------------------------------------------------------------------

if RUN_DEBUG_INFEASIBLE:

    season, lp_path, iis_json_path = auto_pick_paths()

    print("\n" + "=" * 90)
    print("[SEC. 4.15] SECOND-STAGE FAILURE DIAGNOSTIC")
    print("=" * 90)
    print("Selected season:", season)
    print("LP:", lp_path)

    if lp_path is None or not Path(lp_path).exists():

        print(
            "[ERROR] LP not found. "
            "Set DEBUG_SELECT_RUN_UID / DEBUG_SELECT_SEASON "
            "or DEBUG_LP_PATH."
        )

    else:

        # -----------------------------------------------------
        # Optional LP preview
        # -----------------------------------------------------
        if PRINT_LP_PREVIEW:
            preview_text_file(
                lp_path,
                head=LP_PREVIEW_HEAD,
                tail=LP_PREVIEW_TAIL,
            )


        # -----------------------------------------------------
        # Re-solve LP only now, for diagnostic purposes
        # -----------------------------------------------------
        print(
            "\n[DIAGNOSTIC] Re-solving failed LP with "
            "DualReductions=0..."
        )

        m = gp.read(lp_path)

        m.Params.DualReductions = 0
        m.Params.InfUnbdInfo = 1

        m.optimize()

        gurobi_status = int(m.status)

        print(
            "[DIAGNOSTIC] Gurobi status:",
            gurobi_status,
        )


        # =====================================================
        # INFEASIBLE
        # =====================================================
        if m.status == gp.GRB.INFEASIBLE:

            print(
                "[DIAGNOSTIC] Model confirmed INFEASIBLE."
            )

            if COMPUTE_IIS_WITH_GUROBI:

                print(
                    "[IIS] Computing irreducible "
                    "inconsistent subsystem..."
                )

                m.computeIIS()

                iis_ilp_path = os.path.join(
                    os.path.dirname(lp_path),
                    f"iis_{season}.ilp",
                )

                iis_json_path = os.path.join(
                    os.path.dirname(lp_path),
                    f"iis_{season}.json",
                )

                m.write(iis_ilp_path)


                iis_constr = [
                    c.ConstrName
                    for c in m.getConstrs()
                    if getattr(c, "IISConstr", 0)
                ]

                iis_vars_lb = [
                    v.VarName
                    for v in m.getVars()
                    if getattr(v, "IISLB", 0)
                ]

                iis_vars_ub = [
                    v.VarName
                    for v in m.getVars()
                    if getattr(v, "IISUB", 0)
                ]


                if SAVE_IIS_JSON:

                    payload = {
                        "season": season,
                        "diagnostic_status": "INFEASIBLE",
                        "gurobi_status": gurobi_status,
                        "lp_path": lp_path,
                        "iis_ilp_path": iis_ilp_path,
                        "constraints": iis_constr,
                        "vars_lb": iis_vars_lb,
                        "vars_ub": iis_vars_ub,
                    }

                    with open(
                        iis_json_path,
                        "w",
                        encoding="utf-8",
                    ) as f:

                        json.dump(
                            payload,
                            f,
                            indent=2,
                            ensure_ascii=False,
                        )

                    print(
                        "[IIS] JSON saved:",
                        iis_json_path,
                    )

                print(
                    f"[IIS] constraints involved: "
                    f"{len(iis_constr)}"
                )


        # =====================================================
        # UNBOUNDED
        # =====================================================
        elif m.status == gp.GRB.UNBOUNDED:

            print(
                "[DIAGNOSTIC] Model confirmed UNBOUNDED."
            )
            print(
                "[DIAGNOSTIC] IIS is not applicable "
                "to an unbounded problem."
            )


        # =====================================================
        # STILL AMBIGUOUS
        # =====================================================
        elif m.status == gp.GRB.INF_OR_UNBD:

            print(
                "[WARN] Model still reported INF_OR_UNBD "
                "after DualReductions=0."
            )


        # =====================================================
        # OTHER STATUS
        # =====================================================
        else:

            print(
                "[WARN] Unexpected diagnostic status:",
                m.status,
            )


    # If the IIS JSON exists, immediately print/plot the short list
    if iis_json_path is not None and Path(iis_json_path).exists():
        iis = load_iis_json(iis_json_path)
        c_list = iis.get("constraints", [])

        lb_list = iis.get("vars_lb", [])
        ub_list = iis.get("vars_ub", [])

        print("\n[IIS] variable lower bounds:")
        for name in lb_list[:100]:
            print("  -", name)

        print("\n[IIS] variable upper bounds:")
        for name in ub_list[:100]:
            print("  -", name)






        print("\n================ IIS SHORT-LIST ================\n")
        print("n_constraints:", len(c_list))
        summarize_names(c_list, top=30)

        print("\n[IIS] first 50 constraints:")
        for name in c_list[:50]:
            print("  -", name)

        print("\nTo see the formula of an IIS constraint in the LP:")
        print("  lp_grep(lp_path, r\"^<CONSTRAINT_NAME>:\\s\", max_hits=1, context=6)")
    else:
        print("\n[IIS] No iis_*.json found, or SAVE_IIS_JSON=False / IIS not computed.")


   ## Conclusion

In [ ]:
# Get the end time
end_time = datetime.datetime.now()
print(f"End time : {end_time}")

# Calculate execution time
execution_time = end_time - start_time
print(f"Execution time : {execution_time}")